# 07 unified FDS Core signal threshold research — RV21D repaired v2

**Cleaned notebook status:** final signal threshold research notebook for the locked `unified_fds_no_min_return` stack.

This notebook starts from the repaired RV21D signal-base stage and carries the research through the final locked signal stack:

```text
Core → Secondary → Tertiary Front
```

Final locked stack:

```text
Stack:     core_secondary_tertiary_signal_stack_locked_v1
Decision:  core_secondary_tertiary_signal_stack_locked_001
Model:     unified_fds_no_min_return
```

Important contract:

- `forecast_vol_pct` is a model/denominator diagnostic, not the signal volatility-floor threshold.
- The signal volatility-floor variable is `rv21d_vol_pct`.
- `rv21d_vol_pct` is a trailing 21-trading-day realized-volatility input and is date-level, not tenor-specific.
- Cell 22R-FIX is the canonical repaired combined comparison for Tertiary Front. Use Cell 22R only as the Tertiary Front selection/grid source; use Cell 22R-FIX for final combined metrics.
- Cell 23R writes the final locked Core + Secondary + Tertiary Front artifacts.

Cleaned-delivery changes:

- Cleared all execution outputs and execution counts.
- Removed the empty trailing cell.
- Added this updated notebook overview.
- Preserved research and lock cells through Cell 23R.

Cell map:

1. **Cell 0R** — setup and corrected RV21D threshold contract.
2. **Cell 1R2** — discover/join RV21D source by trade date.
3. **Cell 2R** — build repaired signal base panel with `rv21d_vol_pct`.
4. **Cell 3R** — validate repaired signal base panel.
5. **Cell 4R** — Core RV21D reference baseline.
6. **Cell 5R** — Front RV21D weakness diagnostic and candidate design.
7. **Cell 6R** — Front RV21D candidate sweep.
8. **Cell 7R** — confirm Front ex-9D candidate.
9. **Cell 8R** — Middle RV21D weakness diagnostic and candidate design.
10. **Cell 9R** — 18D bucket reassignment test.
11. **Cell 10R** — DTE-level smooth parameter sweep.
12. **Cell 11R** — lock repaired Core bucket map and parameters.
13. **Cell 12R** — Secondary Back sweep, Core-gated.
14. **Cell 13R** — Secondary Middle sweep, Core-gated, with Back shortlist.
15. **Cell 14R / 14R-FIX** — Secondary Front sweep and repaired combined stack assignment.
16. **Cell 15R** — independent Secondary bucket recalibration.
17. **Cell 16R-LITE** — Secondary Middle/Back finalist grid, Front excluded.
18. **Cell 19R** — Secondary frequency unlock review.
19. **Cell 20R** — Secondary frequency unlock confirmation sweep.
20. **Cell 21R** — write higher-frequency Secondary lock artifacts.
21. **Cell 22R** — Tertiary Front fit review and selected-panel source.
22. **Cell 22R-FIX** — repaired Tertiary Front combined comparison.
23. **Cell 23R** — write final Core + Secondary + Tertiary Front lock artifacts.


In [ ]:

# ======================================================================================
# Cell 0R — Setup and corrected RV21D threshold contract
# ======================================================================================
#
# Repaired notebook:
#   07_unified_fds_core_signal_threshold_research_v1_RV21D_repaired_v2.ipynb
#
# Why this repair exists:
#   The prior notebook incorrectly used forecast_vol_pct as the Core volatility-floor
#   threshold parameter. That was wrong.
#
# Correct contract:
#   - forecast_vol_pct remains a model denominator / forecast diagnostic field.
#   - Core entry vol-floor threshold uses trailing 21-trading-day realized volatility:
#
#         rv21d_vol_pct > rv21d_vol_pct_min
#
#   - rv21d_vol_pct is date-level / underlying-level, not tenor-specific.
#   - The 21D realized-vol window applies to all DTEs.
#   - The threshold level can differ by bucket: Front / Middle / Back.
#
# Scope of this repaired notebook:
#   Cells 0R–3R only rebuild and validate the clean signal base panel.
#   Contaminated threshold cells from the old notebook are intentionally removed.
#
# Not in scope yet:
#   - No Core threshold sweep.
#   - No candidate optimization.
#   - No Secondary.
#   - No production lock.
#   - No sizing changes.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import math
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

pd.set_option("display.max_columns", 260)
pd.set_option("display.width", 460)

# --------------------------------------------------------------------------------------
# Project paths
# --------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

SOURCE_BRANCH = "vrp_front_middle_corsi_forecast_repair_v1"
NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

SOURCE_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / SOURCE_BRANCH

OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
OUT_CHART_DIR = OUT_AUDIT_DIR / "charts"

OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CHART_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# --------------------------------------------------------------------------------------
# Locked model / tenor contract
# --------------------------------------------------------------------------------------

LOCKED_MODEL_SPEC = "unified_fds_no_min_return"
EXISTING_REFERENCE_SPEC = "existing_har_downside_v1"

ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]
FRONT_TENORS = [9, 12, 15]
MIDDLE_TENORS = [18, 21, 24]
BACK_TENORS = [27, 30, 33]

TENOR_BUCKET_MAP = {
    9: "Front",
    12: "Front",
    15: "Front",
    18: "Middle",
    21: "Middle",
    24: "Middle",
    27: "Back",
    30: "Back",
    33: "Back",
}

BUCKET_ORDER_MAP = {
    "Front": 1,
    "Middle": 2,
    "Back": 3,
}

# --------------------------------------------------------------------------------------
# Correct threshold contract
# --------------------------------------------------------------------------------------
#
# Do NOT use forecast_vol_pct as a threshold floor.
# Use rv21d_vol_pct_min.
#
# These are legacy/reference levels only. They are not final production locks for
# Front/Middle. Back remains the locked anchor from the prior Core Back decision.
# --------------------------------------------------------------------------------------

LOCKED_BACK_CORE_THRESHOLDS = {
    "model_vrp_log": 0.70,
    "model_vrp_z_3m": 0.70,
    "model_vrp_z_1y": 0.70,
    "RSI14_max": 70.0,
    "rv21d_vol_pct_min": 8.5,
}

LEGACY_CORE_REFERENCE_THRESHOLDS = {
    "Front": {
        "model_vrp_log": 0.60,
        "model_vrp_z_3m": 0.55,
        "model_vrp_z_1y": 0.65,
        "RSI14_max": 70.0,
        "rv21d_vol_pct_min": 8.5,
    },
    "Middle": {
        "model_vrp_log": 0.65,
        "model_vrp_z_3m": 0.75,
        "model_vrp_z_1y": 0.65,
        "RSI14_max": 68.0,
        "rv21d_vol_pct_min": 8.5,
    },
    "Back": LOCKED_BACK_CORE_THRESHOLDS,
}

THRESHOLD_PARAMETER_COLUMNS = [
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14_max",
    "rv21d_vol_pct_min",
]

FORBIDDEN_THRESHOLD_PARAMETER_COLUMNS = [
    "forecast_vol_pct_min",
    "candidate_forecast_vol_pct_min",
]

# --------------------------------------------------------------------------------------
# RV21D source discovery config
# --------------------------------------------------------------------------------------
# Preferred: existing date-level RV21D / 21-trading-day realized-vol column.
# Fallback: compute RV21D from an SPX daily close file, if no explicit date-level source
# is found. The computed definition is rolling 21 trading-day log-return std, annualized
# by sqrt(252), expressed in percent-vol units.
# --------------------------------------------------------------------------------------

RV21D_WINDOW_TRADING_DAYS = 21
RV21D_ANNUALIZATION_DAYS = 252
RV21D_TENOR_INVARIANCE_TOLERANCE = 1e-10
RV21D_PLAUSIBLE_MIN_PCT = 2.0
RV21D_PLAUSIBLE_MAX_PCT = 150.0
RV21D_REQUIRED_JOIN_COVERAGE = 0.90

ALLOW_COMPUTE_RV21D_FROM_SPX_DAILY_IF_EXPLICIT_SOURCE_MISSING = True

DATE_COLUMN_CANDIDATES = [
    "trade_date",
    "date",
    "snapshot_date",
    "asof_date",
    "obs_date",
    "observation_date",
    "timestamp",
    "datetime",
]

SPX_CLOSE_COLUMN_CANDIDATES = [
    "spx_close",
    "SPX_close",
    "spx",
    "SPX",
    "close",
    "Close",
    "px_last",
    "PX_LAST",
    "last",
    "settle",
    "adj_close",
    "adjusted_close",
]

# Direct vol candidates. These may be in percent or decimal units; Cell 1R2 standardizes.
RV21D_DIRECT_CANDIDATE_COLUMNS = [
    "rv21d_vol_pct",
    "RV21D",
    "rv21d",
    "rv_21d",
    "rv21",
    "rv_21",
    "rv21d_pct",
    "rv_21d_pct",
    "rv21d_vol",
    "rv_21d_vol",
    "realized_vol_21d_pct",
    "realized_vol_21d",
    "realized_vol_21",
    "realized_volatility_21d",
    "realized_volatility_21",
    "realized_vol_21_trading_day_pct",
    "realized_vol_21_trading_day",
    "trailing_21d_realized_vol_pct",
    "trailing_21d_realized_vol",
    "trailing_21_trading_day_realized_vol_pct",
    "hist_vol_21d_pct",
    "hist_vol_21d",
    "historical_vol_21d_pct",
    "historical_vol_21d",
    "spx_realized_vol_21d_pct",
    "spx_realized_vol_21d",
    "spx_rv21d_vol_pct",
    "spx_rv21d_vol",
]

# Date-level feature candidates that may be logged realized-vol/std fields.
# If selected, conversion is explicit and audited.
RV21D_LOG_STD_CANDIDATE_COLUMNS = [
    "log_rv_std_21d",
    "log_realized_vol_21d",
    "log_realized_volatility_21d",
    "log_rv21d",
    "log_rv_21d",
]

RV21D_VARIANCE_CANDIDATE_COLUMNS = [
    "realized_variance_21d",
    "realized_var_21d",
    "rv21d_variance",
    "rv_21d_variance",
    "trailing_21d_realized_variance",
    "trailing_realized_variance_21d",
]

# Explicit source search list. Cell 1R2 audits every matching file and chooses the first
# usable candidate by priority and locked-date coverage.
RV21D_SOURCE_SEARCH_SPECS = [
    {
        "source_label": "locked_denominator_panel_same_file",
        "directory": SOURCE_PROCESSED_DIR,
        "pattern": "07A_unified_fds_no_min_return_oos_forecast_panel_*.parquet",
        "max_files": 1,
    },
    {
        "source_label": "production_feature_panel_candidate",
        "directory": PROJECT_ROOT / "data" / "processed" / "staging",
        "pattern": "production_feature_panel_v0_1_candidate_*.parquet",
        "max_files": 3,
    },
    {
        "source_label": "corsi_30d_extended_feature_target_panel",
        "directory": PROJECT_ROOT / "data" / "processed" / "vrp_30d_corsi_v1",
        "pattern": "02B_30d_corsi_extended_feature_target_panel_*.parquet",
        "max_files": 3,
    },
    {
        "source_label": "core_bucket_signal_base_panel_legacy",
        "directory": PROJECT_ROOT / "data" / "processed" / "vrp_core_bucket_parameters_v1",
        "pattern": "02C_cross_tenor_core_signal_base_panel_*.parquet",
        "max_files": 3,
    },
    {
        "source_label": "core_bucket_denominator_panel_legacy",
        "directory": PROJECT_ROOT / "data" / "processed" / "vrp_core_bucket_parameters_v1",
        "pattern": "02B_cross_tenor_har_downside_denominator_panel_*.parquet",
        "max_files": 3,
    },
    {
        "source_label": "external_spx_daily_parquet",
        "directory": PROJECT_ROOT / "data" / "external",
        "pattern": "*spx*daily*.parquet",
        "max_files": 5,
    },
    {
        "source_label": "external_spx_daily_csv",
        "directory": PROJECT_ROOT / "data" / "external",
        "pattern": "*spx*daily*.csv",
        "max_files": 5,
    },
    {
        "source_label": "raw_spx_daily_parquet",
        "directory": PROJECT_ROOT / "data" / "raw",
        "pattern": "*spx*daily*.parquet",
        "max_files": 5,
    },
    {
        "source_label": "raw_spx_daily_csv",
        "directory": PROJECT_ROOT / "data" / "raw",
        "pattern": "*spx*daily*.csv",
        "max_files": 5,
    },
]

TARGET_VAR_COL = "target_realized_variance"
TARGET_LOG_COL = "target_log_variance"

ZSCORE_WINDOWS = {
    "model_vrp_z_3m": {
        "window": 63,
        "min_periods": 63,
    },
    "model_vrp_z_1y": {
        "window": 252,
        "min_periods": 252,
    },
}

EPS = 1e-12

print("=" * 100)
print("Cell 0R — Setup and corrected RV21D threshold contract")
print("=" * 100)
print(f"Source branch:       {SOURCE_BRANCH}")
print(f"New branch:          {NEW_BRANCH}")
print(f"Locked model spec:   {LOCKED_MODEL_SPEC}")
print(f"All tenors:          {ALL_TENORS}")
print(f"Output processed:    {OUT_PROCESSED_DIR}")
print(f"Output audit:        {OUT_AUDIT_DIR}")
print()
print("Correct threshold vol-floor contract:")
print("  rv21d_vol_pct > rv21d_vol_pct_min")
print()
print("forecast_vol_pct is retained only as a model/forecast diagnostic, not as a threshold floor.")
print()
print("RV21D source discovery will search explicit feature panels first, then compute from SPX daily close if needed.")
print()
print("Primary cross-tenor evaluation metric remains:")
print("  normalized_pnl_per_day = normalized_pnl_dollars / tenor")
print()
print("CELL 0R COMPLETE")


In [ ]:

# ======================================================================================
# Cell 1R2 — Load locked panel, discover RV21D source, and join date-level RV21D
# ======================================================================================
#
# Objective:
#   The locked denominator panel does not necessarily contain the entry-filter RV21D input.
#   This cell searches external/date-level feature sources, standardizes RV21D to pct-vol
#   units, validates date-level behavior, and joins it to the locked model panel by date.
#
# Preferred source:
#   Existing date-level 21-trading-day realized-vol column.
#
# Fallback source:
#   SPX daily close file, with RV21D computed as:
#       rolling_std(log_return, 21 trading days) * sqrt(252) * 100
#
# Outputs:
#   01R_locked_panel_with_rv21d_source_*.parquet
#   01R_rv21d_source_discovery_*.csv
#   01R_rv21d_join_validation_*.csv
#   01R_rv21d_source_manifest_*.json
#
# No threshold sweep in this cell.
# ======================================================================================


def latest_files(directory: Path, pattern: str, max_files: int = 1) -> list[Path]:
    if not directory.exists():
        return []
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return matches[:max_files]


def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = latest_files(directory, pattern, max_files=1)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported source file type: {path}")


def find_first_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    lower_map = {str(c).lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    return None


def normalize_date_column(df: pd.DataFrame, date_col: str) -> pd.Series:
    return pd.to_datetime(df[date_col], errors="coerce").dt.normalize()


def column_priority(col: str) -> tuple[int, str]:
    """Return priority and candidate kind for possible RV21D columns."""
    c = str(col).lower()
    compact = re.sub(r"[^a-z0-9]+", "_", c).strip("_")

    exact_direct = {x.lower() for x in RV21D_DIRECT_CANDIDATE_COLUMNS}
    exact_log_std = {x.lower() for x in RV21D_LOG_STD_CANDIDATE_COLUMNS}
    exact_variance = {x.lower() for x in RV21D_VARIANCE_CANDIDATE_COLUMNS}

    if c in exact_direct:
        return 0, "direct_vol_candidate"
    if c in exact_log_std:
        return 50, "log_std_candidate"
    if c in exact_variance:
        return 100, "variance_candidate"

    has_21 = bool(re.search(r"(^|_)21(d|day|td|trading)?(_|$)", compact)) or "21d" in compact or "21_day" in compact
    mentions_realized = any(token in compact for token in ["rv", "realized", "hist", "historical", "trailing"])
    mentions_vol = any(token in compact for token in ["vol", "std", "sigma"])
    mentions_var = any(token in compact for token in ["variance", "var"])
    mentions_log = compact.startswith("log_") or "_log_" in compact
    bad_context = any(token in compact for token in ["forward", "target", "future", "forecast", "pred", "candidate_forecast", "implied", "vix"])

    if has_21 and mentions_realized and mentions_vol and not bad_context:
        if mentions_log:
            return 150, "log_std_candidate"
        return 200, "pattern_direct_vol_candidate"
    if has_21 and mentions_realized and mentions_var and not bad_context:
        return 250, "pattern_variance_candidate"

    return 1_000_000, "not_rv21d_candidate"


def standardize_rv21d_candidate(raw: pd.Series, source_col: str, candidate_kind: str) -> tuple[pd.Series | None, str, str, dict]:
    """
    Standardize a candidate RV21D series into percentage-vol units.

    Returns:
        standardized_series, conversion_method, unit_status, diagnostics

    unit_status:
        auto_usable
        not_usable
    """
    s = pd.to_numeric(raw, errors="coerce")
    non_null = int(s.notna().sum())
    raw_median = float(s.median()) if non_null else np.nan
    raw_min = float(s.min()) if non_null else np.nan
    raw_max = float(s.max()) if non_null else np.nan

    diag = {
        "raw_non_null": non_null,
        "raw_median": raw_median,
        "raw_min": raw_min,
        "raw_max": raw_max,
    }

    if non_null == 0:
        return None, "no_non_null_values", "not_usable", diag

    col_lower = str(source_col).lower()

    def plausible(x: pd.Series) -> bool:
        xx = pd.to_numeric(x, errors="coerce").dropna()
        if xx.empty:
            return False
        med = float(xx.median())
        q01 = float(xx.quantile(0.01))
        q99 = float(xx.quantile(0.99))
        return (
            RV21D_PLAUSIBLE_MIN_PCT <= med <= RV21D_PLAUSIBLE_MAX_PCT
            and q01 > 0
            and q99 <= RV21D_PLAUSIBLE_MAX_PCT * 2
        )

    # Explicit logged realized-vol/std feature. Common in model feature panels.
    if candidate_kind == "log_std_candidate" or col_lower.startswith("log_"):
        exp_times_100 = np.exp(s) * 100.0
        exp_sqrt252_times_100 = np.exp(s) * math.sqrt(RV21D_ANNUALIZATION_DAYS) * 100.0

        diag.update({
            "exp_times_100_median": float(exp_times_100.median()) if exp_times_100.notna().any() else np.nan,
            "exp_sqrt252_times_100_median": float(exp_sqrt252_times_100.median()) if exp_sqrt252_times_100.notna().any() else np.nan,
        })

        if plausible(exp_times_100):
            return exp_times_100, "log_feature_exp_times_100", "auto_usable", diag
        if plausible(exp_sqrt252_times_100):
            return exp_sqrt252_times_100, "log_feature_exp_sqrt252_times_100", "auto_usable", diag
        return None, "log_feature_no_plausible_pct_conversion", "not_usable", diag

    # Variance candidate. Assume annualized variance unless values force otherwise.
    if candidate_kind == "variance_candidate" or "variance" in col_lower or re.search(r"(^|_)var($|_)", col_lower):
        ann_var_to_vol_pct = np.sqrt(s.clip(lower=0)) * 100.0
        daily_var_to_ann_vol_pct = np.sqrt(s.clip(lower=0) * RV21D_ANNUALIZATION_DAYS) * 100.0

        diag.update({
            "ann_var_to_vol_pct_median": float(ann_var_to_vol_pct.median()) if ann_var_to_vol_pct.notna().any() else np.nan,
            "daily_var_to_ann_vol_pct_median": float(daily_var_to_ann_vol_pct.median()) if daily_var_to_ann_vol_pct.notna().any() else np.nan,
        })

        if plausible(ann_var_to_vol_pct):
            return ann_var_to_vol_pct, "annualized_variance_sqrt_times_100", "auto_usable", diag
        if plausible(daily_var_to_ann_vol_pct):
            return daily_var_to_ann_vol_pct, "daily_variance_sqrt252_sqrt_times_100", "auto_usable", diag
        return None, "variance_no_plausible_pct_conversion", "not_usable", diag

    # Direct vol candidate. Could already be percent vol or decimal annualized vol.
    if pd.isna(raw_median):
        return None, "direct_no_non_null_values", "not_usable", diag

    direct_pct = s.astype("float64")
    decimal_to_pct = s * 100.0

    diag.update({
        "direct_pct_median": float(direct_pct.median()) if direct_pct.notna().any() else np.nan,
        "decimal_to_pct_median": float(decimal_to_pct.median()) if decimal_to_pct.notna().any() else np.nan,
    })

    # Prefer already-percent if plausible. Otherwise treat as decimal annualized vol.
    if plausible(direct_pct):
        return direct_pct, "already_pct_vol", "auto_usable", diag
    if plausible(decimal_to_pct):
        return decimal_to_pct, "decimal_vol_times_100", "auto_usable", diag

    return None, "direct_no_plausible_pct_conversion", "not_usable", diag


def date_level_from_source(df: pd.DataFrame, date_col: str, rv_series: pd.Series) -> tuple[pd.DataFrame, dict]:
    tmp = pd.DataFrame({
        "trade_date": normalize_date_column(df, date_col),
        "rv21d_vol_pct": pd.to_numeric(rv_series, errors="coerce"),
    }).dropna(subset=["trade_date"])

    by_date = (
        tmp.groupby("trade_date", as_index=False)
        .agg(
            source_rows=("rv21d_vol_pct", "size"),
            non_null_rows=("rv21d_vol_pct", lambda x: int(pd.Series(x).notna().sum())),
            rv21d_min=("rv21d_vol_pct", "min"),
            rv21d_max=("rv21d_vol_pct", "max"),
            rv21d_median=("rv21d_vol_pct", "median"),
        )
    )
    by_date["rv21d_spread_within_date"] = by_date["rv21d_max"] - by_date["rv21d_min"]
    bad_dates = by_date[
        by_date["non_null_rows"].gt(1)
        & by_date["rv21d_spread_within_date"].abs().gt(RV21D_TENOR_INVARIANCE_TOLERANCE)
    ]

    out = by_date[["trade_date", "rv21d_median"]].rename(columns={"rv21d_median": "rv21d_vol_pct"})

    info = {
        "date_rows": int(len(out)),
        "bad_date_level_rows": int(len(bad_dates)),
        "date_min": out["trade_date"].min(),
        "date_max": out["trade_date"].max(),
        "rv21d_median": float(out["rv21d_vol_pct"].median()) if out["rv21d_vol_pct"].notna().any() else np.nan,
        "rv21d_min": float(out["rv21d_vol_pct"].min()) if out["rv21d_vol_pct"].notna().any() else np.nan,
        "rv21d_max": float(out["rv21d_vol_pct"].max()) if out["rv21d_vol_pct"].notna().any() else np.nan,
    }

    return out, info


def compute_rv21d_from_close_source(df: pd.DataFrame, date_col: str, close_col: str) -> tuple[pd.DataFrame, dict]:
    tmp = pd.DataFrame({
        "trade_date": normalize_date_column(df, date_col),
        "spx_close": pd.to_numeric(df[close_col], errors="coerce"),
    }).dropna(subset=["trade_date", "spx_close"])

    tmp = tmp.sort_values("trade_date").drop_duplicates("trade_date", keep="last").reset_index(drop=True)
    tmp["spx_log_return"] = np.log(tmp["spx_close"] / tmp["spx_close"].shift(1))
    tmp["rv21d_vol_pct"] = (
        tmp["spx_log_return"]
        .rolling(RV21D_WINDOW_TRADING_DAYS, min_periods=RV21D_WINDOW_TRADING_DAYS)
        .std(ddof=1)
        * math.sqrt(RV21D_ANNUALIZATION_DAYS)
        * 100.0
    )

    rv = tmp[["trade_date", "rv21d_vol_pct"]].copy()

    info = {
        "date_rows": int(len(rv)),
        "date_min": rv["trade_date"].min(),
        "date_max": rv["trade_date"].max(),
        "rv21d_non_null_rows": int(rv["rv21d_vol_pct"].notna().sum()),
        "rv21d_median": float(rv["rv21d_vol_pct"].median()) if rv["rv21d_vol_pct"].notna().any() else np.nan,
        "rv21d_min": float(rv["rv21d_vol_pct"].min()) if rv["rv21d_vol_pct"].notna().any() else np.nan,
        "rv21d_max": float(rv["rv21d_vol_pct"].max()) if rv["rv21d_vol_pct"].notna().any() else np.nan,
    }

    return rv, info


# --------------------------------------------------------------------------------------
# Load locked denominator panel
# --------------------------------------------------------------------------------------

locked_panel_path = latest_file(
    SOURCE_PROCESSED_DIR,
    "07A_unified_fds_no_min_return_oos_forecast_panel_*.parquet",
    required=True,
)

full_panel = pd.read_parquet(locked_panel_path)

if "trade_date" not in full_panel.columns:
    if "date" in full_panel.columns:
        full_panel["trade_date"] = full_panel["date"]
    else:
        raise KeyError("Locked panel must contain trade_date or date.")

full_panel["trade_date"] = pd.to_datetime(full_panel["trade_date"], errors="coerce").dt.normalize()
full_panel["last_forward_rv_date"] = pd.to_datetime(full_panel["last_forward_rv_date"], errors="coerce").dt.normalize()
full_panel["tenor"] = pd.to_numeric(full_panel["tenor"], errors="coerce").astype("Int64")

locked_rows = full_panel[
    full_panel["model_spec"].eq(LOCKED_MODEL_SPEC)
    & full_panel["tenor"].isin(ALL_TENORS)
].copy()

locked_rows["tenor"] = locked_rows["tenor"].astype(int)
locked_dates = pd.Series(sorted(locked_rows["trade_date"].dropna().unique()))
locked_date_count = int(len(locked_dates))

required_locked_cols = [
    "trade_date",
    "tenor",
    "model_spec",
    "last_forward_rv_date",
    "implied_variance",
    "forecast_variance_candidate",
    TARGET_VAR_COL,
    TARGET_LOG_COL,
    "normalized_pnl_dollars",
    "win",
    "RSI14",
]
missing_locked_cols = [c for c in required_locked_cols if c not in locked_rows.columns]
if missing_locked_cols:
    raise KeyError(f"Locked panel missing required columns: {missing_locked_cols}")

# --------------------------------------------------------------------------------------
# Build candidate source file list
# --------------------------------------------------------------------------------------

source_files = []
seen_paths = set()

for spec in RV21D_SOURCE_SEARCH_SPECS:
    for path in latest_files(spec["directory"], spec["pattern"], max_files=spec.get("max_files", 1)):
        resolved = str(path.resolve()) if path.exists() else str(path)
        if resolved in seen_paths:
            continue
        seen_paths.add(resolved)
        source_files.append({
            "source_label": spec["source_label"],
            "path": path,
        })

# --------------------------------------------------------------------------------------
# Inspect each source file for explicit RV21D candidates or SPX-close fallback
# --------------------------------------------------------------------------------------

audit_rows = []
candidate_payloads = []

for file_info in source_files:
    source_label = file_info["source_label"]
    path = file_info["path"]

    try:
        df = read_table(path)
    except Exception as exc:
        audit_rows.append({
            "source_label": source_label,
            "path": str(path),
            "status": "READ_FAIL",
            "detail": repr(exc),
        })
        continue

    date_col = find_first_column(df, DATE_COLUMN_CANDIDATES)
    if date_col is None:
        audit_rows.append({
            "source_label": source_label,
            "path": str(path),
            "status": "NO_DATE_COLUMN",
            "detail": f"columns={list(df.columns)}",
        })
        continue

    source_dates = normalize_date_column(df, date_col)
    source_date_set = set(source_dates.dropna().unique())
    locked_date_set = set(locked_dates.dropna().unique())
    date_overlap = len(source_date_set & locked_date_set)
    join_coverage = date_overlap / locked_date_count if locked_date_count else 0.0

    # Column candidates in this source.
    rv_cols = []
    for col in df.columns:
        if col == date_col:
            continue
        priority, kind = column_priority(col)
        if kind != "not_rv21d_candidate":
            rv_cols.append((priority, kind, col))

    rv_cols = sorted(rv_cols, key=lambda x: (x[0], str(x[2]).lower()))

    for priority, kind, col in rv_cols:
        standardized, conversion_method, unit_status, diag = standardize_rv21d_candidate(
            df[col],
            source_col=col,
            candidate_kind=kind,
        )

        if standardized is not None:
            by_date, date_info = date_level_from_source(df, date_col, standardized)
            rv_date_set = set(by_date.loc[by_date["rv21d_vol_pct"].notna(), "trade_date"].dropna().unique())
            rv_overlap = len(rv_date_set & locked_date_set)
            rv_join_coverage = rv_overlap / locked_date_count if locked_date_count else 0.0
        else:
            by_date = pd.DataFrame(columns=["trade_date", "rv21d_vol_pct"])
            date_info = {}
            rv_join_coverage = 0.0

        usable = (
            unit_status == "auto_usable"
            and date_info.get("bad_date_level_rows", 0) == 0
            and rv_join_coverage >= RV21D_REQUIRED_JOIN_COVERAGE
            and date_info.get("date_rows", 0) > 0
        )

        audit_row = {
            "source_label": source_label,
            "path": str(path),
            "status": "CANDIDATE_COLUMN",
            "date_col": date_col,
            "candidate_col": col,
            "candidate_kind": kind,
            "priority": priority,
            "conversion_method": conversion_method,
            "unit_status": unit_status,
            "source_rows": int(len(df)),
            "source_date_count": int(len(source_date_set)),
            "locked_date_count": locked_date_count,
            "raw_date_overlap": int(date_overlap),
            "raw_join_coverage": float(join_coverage),
            "rv_join_coverage": float(rv_join_coverage),
            "usable": bool(usable),
        }
        audit_row.update({k: v for k, v in diag.items()})
        audit_row.update({f"date_info_{k}": v for k, v in date_info.items()})
        audit_rows.append(audit_row)

        if usable:
            candidate_payloads.append({
                "priority": priority,
                "source_label": source_label,
                "path": path,
                "date_col": date_col,
                "candidate_col": col,
                "candidate_kind": kind,
                "conversion_method": conversion_method,
                "join_coverage": rv_join_coverage,
                "rv21d_by_date": by_date.copy(),
            })

    # SPX close fallback candidate from the same file.
    close_col = find_first_column(df, SPX_CLOSE_COLUMN_CANDIDATES)
    if close_col is not None and ALLOW_COMPUTE_RV21D_FROM_SPX_DAILY_IF_EXPLICIT_SOURCE_MISSING:
        try:
            computed_by_date, close_info = compute_rv21d_from_close_source(df, date_col, close_col)
            computed_date_set = set(computed_by_date.loc[computed_by_date["rv21d_vol_pct"].notna(), "trade_date"].dropna().unique())
            computed_overlap = len(computed_date_set & locked_date_set)
            computed_join_coverage = computed_overlap / locked_date_count if locked_date_count else 0.0

            median_ok = (
                pd.notna(close_info.get("rv21d_median", np.nan))
                and RV21D_PLAUSIBLE_MIN_PCT <= close_info["rv21d_median"] <= RV21D_PLAUSIBLE_MAX_PCT
            )
            usable = computed_join_coverage >= RV21D_REQUIRED_JOIN_COVERAGE and median_ok

            audit_rows.append({
                "source_label": source_label,
                "path": str(path),
                "status": "COMPUTED_FROM_SPX_CLOSE_CANDIDATE",
                "date_col": date_col,
                "candidate_col": close_col,
                "candidate_kind": "computed_from_spx_close",
                "priority": 500,
                "conversion_method": "rolling_21_trading_day_log_return_std_sqrt252_times_100",
                "unit_status": "auto_usable" if usable else "not_usable",
                "source_rows": int(len(df)),
                "source_date_count": int(len(source_date_set)),
                "locked_date_count": locked_date_count,
                "raw_date_overlap": int(date_overlap),
                "raw_join_coverage": float(join_coverage),
                "rv_join_coverage": float(computed_join_coverage),
                "usable": bool(usable),
                **{f"date_info_{k}": v for k, v in close_info.items()},
            })

            if usable:
                candidate_payloads.append({
                    "priority": 500,
                    "source_label": source_label,
                    "path": path,
                    "date_col": date_col,
                    "candidate_col": close_col,
                    "candidate_kind": "computed_from_spx_close",
                    "conversion_method": "rolling_21_trading_day_log_return_std_sqrt252_times_100",
                    "join_coverage": computed_join_coverage,
                    "rv21d_by_date": computed_by_date.copy(),
                })
        except Exception as exc:
            audit_rows.append({
                "source_label": source_label,
                "path": str(path),
                "status": "COMPUTE_FROM_CLOSE_FAIL",
                "date_col": date_col,
                "candidate_col": close_col,
                "detail": repr(exc),
            })

# --------------------------------------------------------------------------------------
# Select best source, join, validate
# --------------------------------------------------------------------------------------

audit = pd.DataFrame(audit_rows)
rv21d_discovery_path = OUT_AUDIT_DIR / f"01R_rv21d_source_discovery_{TIMESTAMP}.csv"
audit.to_csv(rv21d_discovery_path, index=False)

if not candidate_payloads:
    print("=" * 100)
    print("Cell 1R2 — RV21D source discovery failed")
    print("=" * 100)
    print(f"Locked source panel: {locked_panel_path}")
    print(f"Audit saved:         {rv21d_discovery_path}")
    print()
    print("No usable date-level RV21D source was found in the configured source files.")
    print("Review the discovery audit, add the correct SPX/RV21D source, then rerun.")
    print()
    print("Files inspected:")
    display(pd.DataFrame(source_files))
    print()
    print("Discovery audit preview:")
    display(audit.head(100))
    raise RuntimeError("Missing usable RV21D source. Cannot proceed with threshold repair.")

# Prefer lower priority, then higher join coverage, then newest file mtime.
candidate_payloads = sorted(
    candidate_payloads,
    key=lambda x: (
        x["priority"],
        -x["join_coverage"],
        -x["path"].stat().st_mtime if x["path"].exists() else 0,
    ),
)
selected = candidate_payloads[0]

rv21d_source_label = selected["source_label"]
rv21d_source_path = selected["path"]
rv21d_source_col = selected["candidate_col"]
rv21d_source_kind = selected["candidate_kind"]
rv21d_conversion_method = selected["conversion_method"]
rv21d_join_coverage = float(selected["join_coverage"])
rv21d_by_date_source = selected["rv21d_by_date"].copy()

rv21d_by_date_source = rv21d_by_date_source.dropna(subset=["trade_date"]).drop_duplicates("trade_date", keep="last")

locked_rows = locked_rows.merge(
    rv21d_by_date_source.rename(columns={"rv21d_vol_pct": "rv21d_vol_pct"}),
    on="trade_date",
    how="left",
    validate="many_to_one",
)

locked_rows["rv21d_source_label"] = rv21d_source_label
locked_rows["rv21d_source_path"] = str(rv21d_source_path)
locked_rows["rv21d_source_col"] = rv21d_source_col
locked_rows["rv21d_source_kind"] = rv21d_source_kind
locked_rows["rv21d_conversion_method"] = rv21d_conversion_method
locked_rows["rv21d_source_raw_value"] = locked_rows["rv21d_vol_pct"]

# Date-level validation after join across the 9-tenor grid.
rv21d_join_by_date = (
    locked_rows.groupby("trade_date", as_index=False)
    .agg(
        rows=("tenor", "size"),
        tenor_count=("tenor", lambda x: int(pd.Series(x).nunique())),
        rv21d_non_null_rows=("rv21d_vol_pct", lambda x: int(pd.Series(x).notna().sum())),
        rv21d_min=("rv21d_vol_pct", "min"),
        rv21d_max=("rv21d_vol_pct", "max"),
    )
)
rv21d_join_by_date["rv21d_spread_across_tenors"] = rv21d_join_by_date["rv21d_max"] - rv21d_join_by_date["rv21d_min"]

bad_rv21d_date_level = rv21d_join_by_date[
    rv21d_join_by_date["rv21d_non_null_rows"].gt(1)
    & rv21d_join_by_date["rv21d_spread_across_tenors"].abs().gt(RV21D_TENOR_INVARIANCE_TOLERANCE)
]

rv21d_missing_dates = rv21d_join_by_date[rv21d_join_by_date["rv21d_non_null_rows"].eq(0)]
joined_coverage = 1.0 - (len(rv21d_missing_dates) / len(rv21d_join_by_date) if len(rv21d_join_by_date) else 1.0)

join_validation_rows = []

def add_join_validation(check: str, status: str, detail: str):
    join_validation_rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })

add_join_validation(
    "locked_panel_loaded",
    "PASS" if len(locked_rows) > 0 else "FAIL",
    f"rows={len(locked_rows):,}; path={locked_panel_path}",
)

add_join_validation(
    "selected_rv21d_source_exists",
    "PASS" if rv21d_source_path.exists() else "FAIL",
    f"source_label={rv21d_source_label}; path={rv21d_source_path}",
)

add_join_validation(
    "rv21d_join_coverage",
    "PASS" if joined_coverage >= RV21D_REQUIRED_JOIN_COVERAGE else "FAIL",
    f"coverage={joined_coverage:.4f}; required={RV21D_REQUIRED_JOIN_COVERAGE:.4f}; missing_dates={len(rv21d_missing_dates):,}",
)

add_join_validation(
    "rv21d_date_level_not_tenor_specific",
    "PASS" if bad_rv21d_date_level.empty else "FAIL",
    f"bad_dates={len(bad_rv21d_date_level):,}; tolerance={RV21D_TENOR_INVARIANCE_TOLERANCE}",
)

rv21d_present = locked_rows["rv21d_vol_pct"].dropna()
rv21d_median = float(rv21d_present.median()) if len(rv21d_present) else np.nan
rv21d_min = float(rv21d_present.min()) if len(rv21d_present) else np.nan
rv21d_max = float(rv21d_present.max()) if len(rv21d_present) else np.nan

add_join_validation(
    "rv21d_level_plausible",
    "PASS" if len(rv21d_present) and RV21D_PLAUSIBLE_MIN_PCT <= rv21d_median <= RV21D_PLAUSIBLE_MAX_PCT else "FAIL",
    f"median={rv21d_median:.6f}; min={rv21d_min:.6f}; max={rv21d_max:.6f}; unit=pct_vol",
)

add_join_validation(
    "forecast_vol_not_used_as_rv21d_source",
    "PASS" if "forecast" not in str(rv21d_source_col).lower() else "FAIL",
    f"rv21d_source_col={rv21d_source_col}",
)

join_validation = pd.DataFrame(join_validation_rows)
join_validation_path = OUT_AUDIT_DIR / f"01R_rv21d_join_validation_{TIMESTAMP}.csv"
rv21d_join_by_date_path = OUT_AUDIT_DIR / f"01R_rv21d_join_by_date_{TIMESTAMP}.csv"
locked_with_rv21d_path = OUT_PROCESSED_DIR / (
    f"01R_locked_panel_with_rv21d_source_{locked_rows['trade_date'].min():%Y%m%d}_"
    f"{locked_rows['trade_date'].max():%Y%m%d}_{TIMESTAMP}.parquet"
)
manifest_path = OUT_AUDIT_DIR / f"01R_rv21d_source_manifest_{TIMESTAMP}.json"

join_validation.to_csv(join_validation_path, index=False)
rv21d_join_by_date.to_csv(rv21d_join_by_date_path, index=False)
locked_rows.to_parquet(locked_with_rv21d_path, index=False)

manifest = {
    "cell": "Cell 1R2 — Load locked panel, discover RV21D source, and join date-level RV21D",
    "timestamp": TIMESTAMP,
    "locked_panel_path": str(locked_panel_path),
    "locked_rows": int(len(locked_rows)),
    "locked_date_count": int(locked_date_count),
    "rv21d_source_label": rv21d_source_label,
    "rv21d_source_path": str(rv21d_source_path),
    "rv21d_source_col": rv21d_source_col,
    "rv21d_source_kind": rv21d_source_kind,
    "rv21d_conversion_method": rv21d_conversion_method,
    "rv21d_join_coverage": rv21d_join_coverage,
    "joined_coverage": joined_coverage,
    "rv21d_median": rv21d_median,
    "rv21d_min": rv21d_min,
    "rv21d_max": rv21d_max,
    "locked_with_rv21d_path": str(locked_with_rv21d_path),
    "rv21d_discovery_path": str(rv21d_discovery_path),
    "join_validation_path": str(join_validation_path),
    "rv21d_join_by_date_path": str(rv21d_join_by_date_path),
    "hard_constraints": [
        "RV21D is date-level, not tenor-specific.",
        "Core vol-floor threshold must use rv21d_vol_pct > rv21d_vol_pct_min.",
        "forecast_vol_pct remains model diagnostic only.",
        "No threshold sweep in Cell 1R2.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

failures = join_validation[join_validation["status"].eq("FAIL")]
if not failures.empty:
    display(join_validation)
    raise RuntimeError("Cell 1R2 RV21D join validation failed. Review validation output above.")

print("=" * 100)
print("Cell 1R2 — Loaded locked panel and joined RV21D source")
print("=" * 100)
print(f"Locked panel:              {locked_panel_path}")
print(f"Locked rows:               {len(locked_rows):,}")
print(f"Locked date range:         {locked_rows['trade_date'].min().date()} to {locked_rows['trade_date'].max().date()}")
print(f"Selected RV21D source:     {rv21d_source_label}")
print(f"RV21D source path:         {rv21d_source_path}")
print(f"RV21D source column:       {rv21d_source_col}")
print(f"RV21D source kind:         {rv21d_source_kind}")
print(f"RV21D conversion:          {rv21d_conversion_method}")
print(f"Joined coverage:           {joined_coverage:.2%}")
print(f"RV21D median/min/max:      {rv21d_median:.4f} / {rv21d_min:.4f} / {rv21d_max:.4f}")
print(f"Discovery audit:           {rv21d_discovery_path}")
print(f"Join validation:           {join_validation_path}")
print(f"Joined panel saved:        {locked_with_rv21d_path}")
print()
print("Join validation:")
display(join_validation)
print()
print("Selected-source rows from discovery audit:")
selected_audit = audit[
    audit.get("usable", pd.Series(False, index=audit.index)).fillna(False)
].sort_values(["priority", "rv_join_coverage"], ascending=[True, False])
display(selected_audit.head(20))
print()
print("RV21D joined by date sample:")
display(rv21d_join_by_date.head(10))

print("\nCELL 1R2 COMPLETE")


In [ ]:

# ======================================================================================
# Cell 2R — Build repaired signal base panel with joined standardized RV21D
# ======================================================================================
#
# Objective:
#   Build one clean signal row per trade_date × tenor using only:
#       unified_fds_no_min_return
#
# Key fields created / retained:
#   - implied_vol_pct
#   - forecast_vol_pct                 # model denominator / diagnostic only
#   - rv21d_vol_pct                    # correct Core vol-floor input
#   - forward_realized_vol_pct_raw
#   - forward_realized_complete
#   - forward_realized_vol_pct_completed
#   - model_vrp_log
#   - model_vrp_z_3m, prior-only by tenor
#   - model_vrp_z_1y, prior-only by tenor
#   - normalized_pnl_per_day = normalized_pnl_dollars / tenor
#
# Critical repair:
#   legacy_reference_core_pass uses rv21d_vol_pct, not forecast_vol_pct.
# ======================================================================================

def annualized_vol_pct_from_variance(x: pd.Series) -> pd.Series:
    return np.sqrt(pd.to_numeric(x, errors="coerce").clip(lower=0)) * 100.0


def standardize_rv21d_vol_pct(raw_values: pd.Series, source_col: str, source_kind: str) -> tuple[pd.Series, str]:
    """
    Convert a discovered RV21D source column to percentage-vol units.

    Rules:
      - variance candidate: sqrt(var) * 100
      - vol/pct candidate:
            median between 1.5 and 200 -> already pct vol
            median between 0 and 1.5  -> decimal vol, multiply by 100

    The conversion method is saved into the manifest.
    """
    s = pd.to_numeric(raw_values, errors="coerce")
    median = float(s.dropna().median()) if s.notna().any() else np.nan
    col_lower = source_col.lower()

    if source_kind == "variance_candidate" or "variance" in col_lower or re.search(r"(^|_)var($|_)", col_lower):
        return annualized_vol_pct_from_variance(s), "variance_to_pct_vol_sqrt_times_100"

    if pd.isna(median):
        return s.astype("float64"), "no_non_null_values_no_conversion"

    if 0 <= median <= 1.5:
        return s * 100.0, "decimal_vol_to_pct_vol_times_100"

    return s.astype("float64"), "already_pct_vol"


def compute_prior_z_by_tenor(
    df: pd.DataFrame,
    value_col: str,
    window: int,
    min_periods: int,
) -> pd.Series:
    """
    Prior-only rolling z-score by tenor.

    For each row:
        z_t = (x_t - mean(x_{t-window} ... x_{t-1})) / std(x_{t-window} ... x_{t-1})

    This intentionally excludes the current row from the rolling mean/std.
    """
    out = pd.Series(np.nan, index=df.index, dtype="float64")

    for tenor, g in df.sort_values(["tenor", "trade_date"]).groupby("tenor"):
        shifted = g[value_col].shift(1)
        roll_mean = shifted.rolling(window=window, min_periods=min_periods).mean()
        roll_std = shifted.rolling(window=window, min_periods=min_periods).std(ddof=1)
        z = (g[value_col] - roll_mean) / roll_std.replace(0.0, np.nan)
        out.loc[g.index] = z

    return out


# Start with locked model only.
signal = locked_rows.copy()

signal = signal.sort_values(["trade_date", "tenor"]).reset_index(drop=True)

# Defensive uniqueness check before building one-row-per-date-tenor panel.
dup_keys = signal.duplicated(["trade_date", "tenor"], keep=False)
if dup_keys.any():
    duplicate_sample = signal.loc[dup_keys, ["trade_date", "tenor", "model_spec"]].head(20)
    display(duplicate_sample)
    raise RuntimeError("Locked model panel has duplicate trade_date × tenor rows.")

signal["tenor"] = signal["tenor"].astype(int)
signal["tenor_bucket"] = signal["tenor"].map(TENOR_BUCKET_MAP)
signal["tenor_bucket_order"] = signal["tenor_bucket"].map(BUCKET_ORDER_MAP)

# Core vol / variance fields.
signal["implied_variance"] = pd.to_numeric(signal["implied_variance"], errors="coerce")
signal["forecast_variance"] = pd.to_numeric(signal["forecast_variance_candidate"], errors="coerce")
signal["target_realized_variance"] = pd.to_numeric(signal[TARGET_VAR_COL], errors="coerce")

signal["implied_vol_pct"] = annualized_vol_pct_from_variance(signal["implied_variance"])
signal["forecast_vol_pct"] = annualized_vol_pct_from_variance(signal["forecast_variance"])
signal["forward_realized_vol_pct_raw"] = annualized_vol_pct_from_variance(signal["target_realized_variance"])

# Correct threshold vol-floor input, joined in Cell 1R2.
if "rv21d_vol_pct" not in signal.columns:
    raise RuntimeError("Cell 2R requires rv21d_vol_pct from Cell 1R2. Rerun Cell 1R2.")

signal["rv21d_vol_pct"] = pd.to_numeric(signal["rv21d_vol_pct"], errors="coerce")

# Keep explicit source lineage on every row.
for _lineage_col, _lineage_value in {
    "rv21d_source_label": rv21d_source_label,
    "rv21d_source_path": str(rv21d_source_path),
    "rv21d_source_col": rv21d_source_col,
    "rv21d_source_kind": rv21d_source_kind,
    "rv21d_conversion_method": rv21d_conversion_method,
}.items():
    signal[_lineage_col] = _lineage_value

signal["rv21d_source_raw_value"] = signal["rv21d_vol_pct"]

# Max available trade date from the locked panel.
max_available_trade_date = signal["trade_date"].max()

signal["forward_realized_complete"] = (
    signal["last_forward_rv_date"].notna()
    & (signal["last_forward_rv_date"] <= max_available_trade_date)
)

signal["target_realized_variance_completed"] = np.where(
    signal["forward_realized_complete"],
    signal["target_realized_variance"],
    np.nan,
)

signal["forward_realized_vol_pct_completed"] = np.where(
    signal["forward_realized_complete"],
    signal["forward_realized_vol_pct_raw"],
    np.nan,
)

# VRP fields.
signal["model_vrp_log"] = np.log(signal["implied_variance"] / signal["forecast_variance"].clip(lower=EPS))
signal["model_vrp_ratio"] = signal["implied_variance"] / signal["forecast_variance"].clip(lower=EPS)

signal["model_vrp_z_3m"] = compute_prior_z_by_tenor(
    signal,
    value_col="model_vrp_log",
    window=ZSCORE_WINDOWS["model_vrp_z_3m"]["window"],
    min_periods=ZSCORE_WINDOWS["model_vrp_z_3m"]["min_periods"],
)

signal["model_vrp_z_1y"] = compute_prior_z_by_tenor(
    signal,
    value_col="model_vrp_log",
    window=ZSCORE_WINDOWS["model_vrp_z_1y"]["window"],
    min_periods=ZSCORE_WINDOWS["model_vrp_z_1y"]["min_periods"],
)

# P&L normalization.
signal["normalized_pnl_dollars"] = pd.to_numeric(signal["normalized_pnl_dollars"], errors="coerce")
signal["normalized_pnl_per_day"] = signal["normalized_pnl_dollars"] / signal["tenor"].replace(0, np.nan)

# Retain useful trade outcome fields.
signal["win"] = pd.to_numeric(signal["win"], errors="coerce")
signal["RSI14"] = pd.to_numeric(signal["RSI14"], errors="coerce")

# Legacy/reference thresholds, now repaired to use RV21D.
signal["legacy_reference_model_vrp_log_threshold"] = signal["tenor_bucket"].map(
    {k: v["model_vrp_log"] for k, v in LEGACY_CORE_REFERENCE_THRESHOLDS.items()}
)
signal["legacy_reference_model_vrp_z_3m_threshold"] = signal["tenor_bucket"].map(
    {k: v["model_vrp_z_3m"] for k, v in LEGACY_CORE_REFERENCE_THRESHOLDS.items()}
)
signal["legacy_reference_model_vrp_z_1y_threshold"] = signal["tenor_bucket"].map(
    {k: v["model_vrp_z_1y"] for k, v in LEGACY_CORE_REFERENCE_THRESHOLDS.items()}
)
signal["legacy_reference_RSI14_max"] = signal["tenor_bucket"].map(
    {k: v["RSI14_max"] for k, v in LEGACY_CORE_REFERENCE_THRESHOLDS.items()}
)
signal["legacy_reference_rv21d_vol_pct_min"] = signal["tenor_bucket"].map(
    {k: v["rv21d_vol_pct_min"] for k, v in LEGACY_CORE_REFERENCE_THRESHOLDS.items()}
)

signal["legacy_reference_core_pass"] = (
    (signal["model_vrp_log"] > signal["legacy_reference_model_vrp_log_threshold"])
    & (signal["model_vrp_z_3m"] > signal["legacy_reference_model_vrp_z_3m_threshold"])
    & (signal["model_vrp_z_1y"] > signal["legacy_reference_model_vrp_z_1y_threshold"])
    & (signal["RSI14"] < signal["legacy_reference_RSI14_max"])
    & (signal["rv21d_vol_pct"] > signal["legacy_reference_rv21d_vol_pct_min"])
)

# Explicitly fail if any old threshold-floor naming survives as a threshold label.
forbidden_signal_cols = [c for c in signal.columns if c.startswith("legacy_reference_forecast_vol") or c.endswith("forecast_vol_pct_min")]
if forbidden_signal_cols:
    raise RuntimeError(
        "Old forecast-vol threshold column names survived in the repaired signal panel: "
        f"{forbidden_signal_cols}"
    )

# Keep a clean, explicit column order where possible.
preferred_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "last_forward_rv_date",
    "forward_realized_complete",
    "implied_variance",
    "forecast_variance",
    "target_realized_variance",
    "target_realized_variance_completed",
    "implied_vol_pct",
    "forecast_vol_pct",
    "rv21d_vol_pct",
    "rv21d_source_col",
    "rv21d_source_kind",
    "rv21d_conversion_method",
    "rv21d_source_path",
    "rv21d_source_raw_value",
    "forward_realized_vol_pct_raw",
    "forward_realized_vol_pct_completed",
    "model_vrp_log",
    "model_vrp_ratio",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
    "legacy_reference_model_vrp_log_threshold",
    "legacy_reference_model_vrp_z_3m_threshold",
    "legacy_reference_model_vrp_z_1y_threshold",
    "legacy_reference_RSI14_max",
    "legacy_reference_rv21d_vol_pct_min",
    "legacy_reference_core_pass",
]

other_cols = [c for c in signal.columns if c not in preferred_cols]
signal = signal[preferred_cols + other_cols].copy()

# Save repaired base panel.
base_panel_path = OUT_PROCESSED_DIR / (
    f"01R_unified_fds_signal_base_panel_with_rv21d_{signal['trade_date'].min():%Y%m%d}_"
    f"{signal['trade_date'].max():%Y%m%d}_{TIMESTAMP}.parquet"
)

signal.to_parquet(base_panel_path, index=False)

# Save summary audit.
summary_by_tenor = (
    signal.groupby(["tenor", "tenor_bucket"], as_index=False)
    .agg(
        rows=("trade_date", "size"),
        date_min=("trade_date", "min"),
        date_max=("trade_date", "max"),
        completed_forward_rows=("forward_realized_complete", "sum"),
        avg_implied_vol_pct=("implied_vol_pct", "mean"),
        avg_forecast_vol_pct=("forecast_vol_pct", "mean"),
        avg_rv21d_vol_pct=("rv21d_vol_pct", "mean"),
        min_rv21d_vol_pct=("rv21d_vol_pct", "min"),
        max_rv21d_vol_pct=("rv21d_vol_pct", "max"),
        avg_model_vrp_log=("model_vrp_log", "mean"),
        z_3m_available_rows=("model_vrp_z_3m", lambda x: int(x.notna().sum())),
        z_1y_available_rows=("model_vrp_z_1y", lambda x: int(x.notna().sum())),
        avg_pnl_dollars=("normalized_pnl_dollars", "mean"),
        avg_pnl_per_day=("normalized_pnl_per_day", "mean"),
        median_pnl_per_day=("normalized_pnl_per_day", "median"),
    )
)

summary_by_bucket = (
    signal.groupby(["tenor_bucket", "tenor_bucket_order"], as_index=False)
    .agg(
        rows=("trade_date", "size"),
        completed_forward_rows=("forward_realized_complete", "sum"),
        avg_implied_vol_pct=("implied_vol_pct", "mean"),
        avg_forecast_vol_pct=("forecast_vol_pct", "mean"),
        avg_rv21d_vol_pct=("rv21d_vol_pct", "mean"),
        min_rv21d_vol_pct=("rv21d_vol_pct", "min"),
        max_rv21d_vol_pct=("rv21d_vol_pct", "max"),
        avg_model_vrp_log=("model_vrp_log", "mean"),
        z_3m_available_rows=("model_vrp_z_3m", lambda x: int(x.notna().sum())),
        z_1y_available_rows=("model_vrp_z_1y", lambda x: int(x.notna().sum())),
        avg_pnl_dollars=("normalized_pnl_dollars", "mean"),
        avg_pnl_per_day=("normalized_pnl_per_day", "mean"),
        median_pnl_per_day=("normalized_pnl_per_day", "median"),
    )
    .sort_values("tenor_bucket_order")
)

rv21d_by_date = (
    signal.groupby("trade_date", as_index=False)
    .agg(
        rows=("tenor", "size"),
        tenors=("tenor", lambda x: sorted(pd.Series(x).dropna().astype(int).unique().tolist())),
        rv21d_min=("rv21d_vol_pct", "min"),
        rv21d_max=("rv21d_vol_pct", "max"),
        rv21d_non_null_rows=("rv21d_vol_pct", lambda x: int(pd.Series(x).notna().sum())),
    )
)
rv21d_by_date["rv21d_spread_across_tenors"] = rv21d_by_date["rv21d_max"] - rv21d_by_date["rv21d_min"]

summary_by_tenor_path = OUT_AUDIT_DIR / f"01R_signal_base_summary_by_tenor_{TIMESTAMP}.csv"
summary_by_bucket_path = OUT_AUDIT_DIR / f"01R_signal_base_summary_by_bucket_{TIMESTAMP}.csv"
rv21d_by_date_path = OUT_AUDIT_DIR / f"01R_signal_base_rv21d_by_date_{TIMESTAMP}.csv"

summary_by_tenor.to_csv(summary_by_tenor_path, index=False)
summary_by_bucket.to_csv(summary_by_bucket_path, index=False)
rv21d_by_date.to_csv(rv21d_by_date_path, index=False)

print("=" * 100)
print("Cell 2R — Built repaired unified signal base panel with RV21D")
print("=" * 100)
print(f"Rows:                         {len(signal):,}")
print(f"Date range:                   {signal['trade_date'].min().date()} to {signal['trade_date'].max().date()}")
print(f"Max available trade date:      {max_available_trade_date.date()}")
print(f"Completed forward rows:        {int(signal['forward_realized_complete'].sum()):,} / {len(signal):,}")
print(f"Suppressed incomplete rows:    {int((~signal['forward_realized_complete']).sum()):,}")
print(f"RV21D source column:           {rv21d_source_col}")
print(f"RV21D source kind:             {rv21d_source_kind}")
print(f"RV21D conversion method:       {rv21d_conversion_method}")
print(f"Base panel saved:              {base_panel_path}")
print(f"Summary by tenor:              {summary_by_tenor_path}")
print(f"Summary by bucket:             {summary_by_bucket_path}")
print(f"RV21D by date audit:           {rv21d_by_date_path}")
print()
print("Summary by bucket:")
display(summary_by_bucket)
print()
print("Summary by tenor:")
display(summary_by_tenor)
print()
print("RV21D by date sample:")
display(rv21d_by_date.head(10))

print("\nCELL 2R COMPLETE")


In [ ]:

# ======================================================================================
# Cell 3R — Validate repaired signal base panel with RV21D threshold contract
# ======================================================================================
#
# Objective:
#   Validate that the repaired signal base panel is suitable for Core threshold research.
#
# Checks:
#   - one locked model only
#   - one row per trade_date × tenor
#   - all exact tenors present
#   - bucket mapping correct
#   - forecast_vol_pct retained only as diagnostic
#   - rv21d_vol_pct exists and is the only vol-floor threshold variable
#   - rv21d_vol_pct is date-level / not tenor-specific
#   - prior-only z-scores exist after warmup
#   - normalized_pnl_per_day created correctly
#   - incomplete forward realized targets are flagged
#   - Back locked threshold reference uses rv21d_vol_pct_min
#
# No threshold sweep in this cell.
# ======================================================================================

validation_rows = []


def add_validation(check: str, status: str, detail: str):
    validation_rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


# --------------------------------------------------------------------------------------
# Basic structure
# --------------------------------------------------------------------------------------

models_found = sorted(signal["model_spec"].dropna().unique().tolist())
tenors_found = sorted(signal["tenor"].dropna().astype(int).unique().tolist())
missing_tenors = sorted(set(ALL_TENORS) - set(tenors_found))
extra_tenors = sorted(set(tenors_found) - set(ALL_TENORS))

duplicate_key_count = int(signal.duplicated(["trade_date", "tenor"]).sum())

add_validation(
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    "one_row_per_trade_date_tenor",
    "PASS" if duplicate_key_count == 0 else "FAIL",
    f"duplicate_key_count={duplicate_key_count}",
)

add_validation(
    "all_exact_tenors_present",
    "PASS" if not missing_tenors and not extra_tenors else "FAIL",
    f"missing={missing_tenors}; extra={extra_tenors}",
)

bucket_bad = signal[
    signal["tenor_bucket"] != signal["tenor"].map(TENOR_BUCKET_MAP)
]

add_validation(
    "bucket_mapping",
    "PASS" if bucket_bad.empty else "FAIL",
    f"bad_rows={len(bucket_bad):,}",
)

# --------------------------------------------------------------------------------------
# Variance / vol sanity
# --------------------------------------------------------------------------------------

bad_variance_rows = signal[
    (~np.isfinite(signal["implied_variance"]))
    | (~np.isfinite(signal["forecast_variance"]))
    | (signal["implied_variance"] <= 0)
    | (signal["forecast_variance"] <= 0)
]

add_validation(
    "positive_implied_and_forecast_variance",
    "PASS" if bad_variance_rows.empty else "FAIL",
    f"bad_rows={len(bad_variance_rows):,}",
)

bad_vrp_rows = signal[
    ~np.isfinite(signal["model_vrp_log"])
]

add_validation(
    "model_vrp_log_finite",
    "PASS" if bad_vrp_rows.empty else "FAIL",
    f"bad_rows={len(bad_vrp_rows):,}",
)

# --------------------------------------------------------------------------------------
# RV21D threshold contract validation
# --------------------------------------------------------------------------------------

rv21d_available_rows = int(signal["rv21d_vol_pct"].notna().sum())
rv21d_positive_bad = signal[
    signal["rv21d_vol_pct"].notna()
    & (~np.isfinite(signal["rv21d_vol_pct"]) | (signal["rv21d_vol_pct"] <= 0))
]

add_validation(
    "rv21d_vol_pct_available",
    "PASS" if rv21d_available_rows > 0 else "FAIL",
    f"available_rows={rv21d_available_rows:,}; total_rows={len(signal):,}; source_col={rv21d_source_col}",
)

add_validation(
    "rv21d_vol_pct_positive_when_present",
    "PASS" if rv21d_positive_bad.empty else "FAIL",
    f"bad_rows={len(rv21d_positive_bad):,}",
)

rv21d_date_level = (
    signal.groupby("trade_date", as_index=False)
    .agg(
        rows=("tenor", "size"),
        tenor_count=("tenor", lambda x: int(pd.Series(x).nunique())),
        rv21d_min=("rv21d_vol_pct", "min"),
        rv21d_max=("rv21d_vol_pct", "max"),
        rv21d_non_null_rows=("rv21d_vol_pct", lambda x: int(pd.Series(x).notna().sum())),
    )
)
rv21d_date_level["rv21d_spread_across_tenors"] = rv21d_date_level["rv21d_max"] - rv21d_date_level["rv21d_min"]

bad_rv21d_date_level = rv21d_date_level[
    rv21d_date_level["rv21d_non_null_rows"].gt(1)
    & rv21d_date_level["rv21d_spread_across_tenors"].abs().gt(RV21D_TENOR_INVARIANCE_TOLERANCE)
]

add_validation(
    "rv21d_date_level_not_tenor_specific",
    "PASS" if bad_rv21d_date_level.empty else "FAIL",
    f"bad_dates={len(bad_rv21d_date_level):,}; tolerance={RV21D_TENOR_INVARIANCE_TOLERANCE}",
)

rv21d_missing_by_tenor = (
    signal.groupby("tenor", as_index=False)
    .agg(
        rows=("trade_date", "size"),
        rv21d_non_null_rows=("rv21d_vol_pct", lambda x: int(pd.Series(x).notna().sum())),
        rv21d_missing_rows=("rv21d_vol_pct", lambda x: int(pd.Series(x).isna().sum())),
    )
)
rv21d_missing_tenors = rv21d_missing_by_tenor[rv21d_missing_by_tenor["rv21d_non_null_rows"].eq(0)]

add_validation(
    "rv21d_available_for_all_tenors",
    "PASS" if rv21d_missing_tenors.empty else "FAIL",
    rv21d_missing_by_tenor.to_dict("records").__repr__(),
)

old_forecast_floor_cols = [
    c for c in signal.columns
    if c.startswith("legacy_reference_forecast_vol") or c.endswith("forecast_vol_pct_min")
]

add_validation(
    "no_forecast_vol_floor_threshold_columns",
    "PASS" if not old_forecast_floor_cols else "FAIL",
    f"old_forecast_floor_cols={old_forecast_floor_cols}",
)

rv21d_threshold_cols_present = [c for c in signal.columns if c.endswith("rv21d_vol_pct_min")]

add_validation(
    "rv21d_threshold_floor_columns_present",
    "PASS" if "legacy_reference_rv21d_vol_pct_min" in rv21d_threshold_cols_present else "FAIL",
    f"rv21d_threshold_cols_present={rv21d_threshold_cols_present}",
)

# forecast_vol_pct should exist, but only as a diagnostic.
add_validation(
    "forecast_vol_pct_retained_as_diagnostic",
    "PASS" if "forecast_vol_pct" in signal.columns else "FAIL",
    "forecast_vol_pct exists but is not used as a threshold floor in repaired Cells 0R-3R.",
)

# --------------------------------------------------------------------------------------
# Z-score availability and prior-only warmup
# --------------------------------------------------------------------------------------

z_summary = (
    signal.groupby("tenor", as_index=False)
    .agg(
        rows=("trade_date", "size"),
        z_3m_available_rows=("model_vrp_z_3m", lambda x: int(x.notna().sum())),
        z_1y_available_rows=("model_vrp_z_1y", lambda x: int(x.notna().sum())),
        first_date=("trade_date", "min"),
        first_z_3m_date=("trade_date", lambda x: signal.loc[x.index[signal.loc[x.index, "model_vrp_z_3m"].notna()], "trade_date"].min()),
        first_z_1y_date=("trade_date", lambda x: signal.loc[x.index[signal.loc[x.index, "model_vrp_z_1y"].notna()], "trade_date"].min()),
    )
)

z_3m_all_have_values = bool((z_summary["z_3m_available_rows"] > 0).all())
z_1y_all_have_values = bool((z_summary["z_1y_available_rows"] > 0).all())

add_validation(
    "z_3m_available_after_warmup",
    "PASS" if z_3m_all_have_values else "FAIL",
    z_summary[["tenor", "z_3m_available_rows", "first_z_3m_date"]].to_dict("records").__repr__(),
)

add_validation(
    "z_1y_available_after_warmup",
    "PASS" if z_1y_all_have_values else "FAIL",
    z_summary[["tenor", "z_1y_available_rows", "first_z_1y_date"]].to_dict("records").__repr__(),
)

prior_only_rows = []

for tenor, g in signal.sort_values(["tenor", "trade_date"]).groupby("tenor"):
    g = g.reset_index(drop=True).copy()
    first_z3_idx = g["model_vrp_z_3m"].first_valid_index()
    first_z1_idx = g["model_vrp_z_1y"].first_valid_index()

    prior_only_rows.append({
        "tenor": int(tenor),
        "first_z3_idx": first_z3_idx,
        "first_z1_idx": first_z1_idx,
        "z3_prior_only_ok": first_z3_idx is not None and first_z3_idx >= ZSCORE_WINDOWS["model_vrp_z_3m"]["min_periods"],
        "z1_prior_only_ok": first_z1_idx is not None and first_z1_idx >= ZSCORE_WINDOWS["model_vrp_z_1y"]["min_periods"],
    })

prior_only_check = pd.DataFrame(prior_only_rows)

add_validation(
    "zscore_prior_only_warmup_check",
    "PASS" if prior_only_check["z3_prior_only_ok"].all() and prior_only_check["z1_prior_only_ok"].all() else "FAIL",
    prior_only_check.to_dict("records").__repr__(),
)

# --------------------------------------------------------------------------------------
# P&L per day validation
# --------------------------------------------------------------------------------------

pnl_check = signal.copy()
pnl_check["expected_pnl_per_day"] = pnl_check["normalized_pnl_dollars"] / pnl_check["tenor"]
pnl_check["pnl_per_day_diff"] = pnl_check["normalized_pnl_per_day"] - pnl_check["expected_pnl_per_day"]

bad_pnl_per_day = pnl_check[
    pnl_check["normalized_pnl_dollars"].notna()
    & pnl_check["normalized_pnl_per_day"].notna()
    & (pnl_check["pnl_per_day_diff"].abs() > 1e-8)
]

add_validation(
    "normalized_pnl_per_day_created",
    "PASS" if bad_pnl_per_day.empty else "FAIL",
    f"bad_rows={len(bad_pnl_per_day):,}; formula=normalized_pnl_dollars / tenor",
)

pnl_per_day_available = int(signal["normalized_pnl_per_day"].notna().sum())

add_validation(
    "normalized_pnl_per_day_available",
    "PASS" if pnl_per_day_available > 0 else "FAIL",
    f"available_rows={pnl_per_day_available:,}",
)

# --------------------------------------------------------------------------------------
# Forward realized completion guard
# --------------------------------------------------------------------------------------

max_available_trade_date = signal["trade_date"].max()

completion_check_bad = signal[
    signal["last_forward_rv_date"].notna()
    & (
        signal["forward_realized_complete"]
        != (signal["last_forward_rv_date"] <= max_available_trade_date)
    )
]

add_validation(
    "forward_realized_completion_guard",
    "PASS" if completion_check_bad.empty else "FAIL",
    f"bad_rows={len(completion_check_bad):,}; max_available_trade_date={max_available_trade_date.date()}",
)

latest_rows = signal[signal["trade_date"] == max_available_trade_date].copy()
latest_completed_count = int(latest_rows["forward_realized_complete"].sum())

add_validation(
    "latest_forward_completion_count",
    "PASS",
    f"latest_date={max_available_trade_date.date()}; completed_forward_tenors={latest_completed_count}/9",
)

# --------------------------------------------------------------------------------------
# Back locked threshold reference
# --------------------------------------------------------------------------------------

back_ref = LEGACY_CORE_REFERENCE_THRESHOLDS["Back"]

back_reference_ok = (
    back_ref["model_vrp_log"] == LOCKED_BACK_CORE_THRESHOLDS["model_vrp_log"]
    and back_ref["model_vrp_z_3m"] == LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_3m"]
    and back_ref["model_vrp_z_1y"] == LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_1y"]
    and back_ref["RSI14_max"] == LOCKED_BACK_CORE_THRESHOLDS["RSI14_max"]
    and back_ref["rv21d_vol_pct_min"] == LOCKED_BACK_CORE_THRESHOLDS["rv21d_vol_pct_min"]
    and "forecast_vol_pct_min" not in back_ref
)

add_validation(
    "back_locked_threshold_reference_uses_rv21d",
    "PASS" if back_reference_ok else "FAIL",
    f"Back reference={back_ref}",
)

# --------------------------------------------------------------------------------------
# Save validation / manifest
# --------------------------------------------------------------------------------------

validation = pd.DataFrame(validation_rows)

validation_path = OUT_AUDIT_DIR / f"01R_signal_base_validation_{TIMESTAMP}.csv"
z_summary_path = OUT_AUDIT_DIR / f"01R_signal_base_zscore_summary_{TIMESTAMP}.csv"
prior_only_check_path = OUT_AUDIT_DIR / f"01R_signal_base_prior_only_zscore_check_{TIMESTAMP}.csv"
rv21d_date_level_path = OUT_AUDIT_DIR / f"01R_signal_base_rv21d_date_level_validation_{TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"01R_signal_base_manifest_{TIMESTAMP}.json"

validation.to_csv(validation_path, index=False)
z_summary.to_csv(z_summary_path, index=False)
prior_only_check.to_csv(prior_only_check_path, index=False)
rv21d_date_level.to_csv(rv21d_date_level_path, index=False)

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1_RV21D_repaired_v2.ipynb",
    "cell_stage": "Cells 0R-3R — repaired locked signal base panel with RV21D threshold contract",
    "timestamp": TIMESTAMP,
    "locked_panel_path": str(locked_panel_path),
    "locked_with_rv21d_path": str(locked_with_rv21d_path),
    "locked_model_spec": LOCKED_MODEL_SPEC,
    "all_tenors": ALL_TENORS,
    "new_branch": NEW_BRANCH,
    "base_panel_path": str(base_panel_path),
    "summary_by_tenor_path": str(summary_by_tenor_path),
    "summary_by_bucket_path": str(summary_by_bucket_path),
    "rv21d_by_date_path": str(rv21d_by_date_path),
    "rv21d_source_label": rv21d_source_label,
    "rv21d_source_path": str(rv21d_source_path),
    "rv21d_source_col": rv21d_source_col,
    "rv21d_source_kind": rv21d_source_kind,
    "rv21d_conversion_method": rv21d_conversion_method,
    "validation_path": str(validation_path),
    "z_summary_path": str(z_summary_path),
    "prior_only_check_path": str(prior_only_check_path),
    "rv21d_date_level_path": str(rv21d_date_level_path),
    "threshold_vol_floor_contract": "rv21d_vol_pct > rv21d_vol_pct_min",
    "forecast_vol_pct_role": "model denominator / forecast diagnostic only, not threshold floor",
    "primary_cross_tenor_trade_quality_metric": "normalized_pnl_per_day",
    "normalized_pnl_per_day_formula": "normalized_pnl_dollars / tenor",
    "zscore_method": {
        "model_vrp_z_3m": "prior-only rolling z-score of model_vrp_log by tenor, 63-row window, shifted one row",
        "model_vrp_z_1y": "prior-only rolling z-score of model_vrp_log by tenor, 252-row window, shifted one row",
    },
    "locked_back_core_thresholds": LOCKED_BACK_CORE_THRESHOLDS,
    "legacy_core_reference_thresholds": LEGACY_CORE_REFERENCE_THRESHOLDS,
    "notes": [
        "No threshold sweep performed.",
        "No Secondary research performed.",
        "No sizing changes performed.",
        "Cells 4+ from the prior notebook are intentionally not carried forward because they used forecast_vol_pct as a vol-floor threshold.",
        "forecast_vol_pct remains in the panel for diagnostics only.",
        "rv21d_vol_pct is validated as date-level / not tenor-specific.",
        "Total P&L retained, but average P&L per day should be used for cross-tenor trade quality comparisons.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

failures = validation[validation["status"] == "FAIL"]

print("=" * 100)
print("Cell 3R — Validate repaired signal base panel with RV21D threshold contract")
print("=" * 100)
print(f"Signal base panel:         {base_panel_path}")
print(f"Validation file:           {validation_path}")
print(f"Z-score summary:           {z_summary_path}")
print(f"Prior-only check:          {prior_only_check_path}")
print(f"RV21D date-level check:    {rv21d_date_level_path}")
print(f"Manifest:                  {manifest_path}")
print()
print("Validation:")
display(validation)

print()
print("RV21D date-level validation summary:")
display(rv21d_date_level.describe(include="all"))

print()
print("Z-score summary:")
display(z_summary)

print()
print("Prior-only z-score warmup check:")
display(prior_only_check)

print()
print("Latest date rows:")
display(
    latest_rows[
        [
            "trade_date",
            "tenor",
            "tenor_bucket",
            "implied_vol_pct",
            "forecast_vol_pct",
            "rv21d_vol_pct",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "forward_realized_complete",
            "last_forward_rv_date",
            "normalized_pnl_dollars",
            "normalized_pnl_per_day",
        ]
    ].sort_values("tenor")
)

if not failures.empty:
    raise RuntimeError("Cell 3R validation failed. Review validation output above.")

print("\nCELL 3R PASSED")


In [ ]:
# ======================================================================================
# Cell 4R — Core RV21D reference baseline
# ======================================================================================
#
# Objective:
#   Rebuild a clean Core reference baseline using the corrected volatility-floor contract:
#
#       rv21d_vol_pct > rv21d_vol_pct_min
#
#   NOT:
#
#       forecast_vol_pct > forecast_vol_pct_min
#
# Hard constraints:
#   1. Back bucket parameters remain locked.
#   2. Within each bucket, 3m and 1y z-score thresholds are equal.
#   3. Middle parameter levels must be between/equal Front and Back.
#   4. No threshold sweep.
#   5. No Secondary.
#   6. No sizing changes.
#   7. No production lock.
#
# Selection universes:
#   1. all_qualified_trades
#   2. one_trade_per_bucket_per_date_best_rank
#   3. one_trade_per_date_best_rank
#   4. thirty_day_anchor_only
#
# Primary metric:
#   avg_pnl_per_day
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import time
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 280)
pd.set_option("display.width", 460)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL4R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

if "ALL_TENORS" not in globals():
    ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

if "TENOR_BUCKET_MAP" not in globals():
    TENOR_BUCKET_MAP = {
        9: "Front",
        12: "Front",
        15: "Front",
        18: "Middle",
        21: "Middle",
        24: "Middle",
        27: "Back",
        30: "Back",
        33: "Back",
    }

if "BUCKET_ORDER_MAP" not in globals():
    BUCKET_ORDER_MAP = {
        "Front": 1,
        "Middle": 2,
        "Back": 3,
    }

BUCKET_CENTER_TENOR = {
    "Front": 12,
    "Middle": 21,
    "Back": 30,
}

REFERENCE_CANDIDATE_ID = "core_rv21d_reference_0001"
REFERENCE_CANDIDATE_FAMILY = "core_rv21d_reference"
REFERENCE_CANDIDATE_SUBFAMILY = "constrained_reference_baseline"

SELECTION_UNIVERSES = [
    "all_qualified_trades",
    "one_trade_per_bucket_per_date_best_rank",
    "one_trade_per_date_best_rank",
    "thirty_day_anchor_only",
]

PRIMARY_SELECTION_UNIVERSE = "one_trade_per_bucket_per_date_best_rank"
PRIMARY_RANKING_METRIC = "avg_pnl_per_day"

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def clean_float(x, default=np.nan):
    try:
        if x is None:
            return float(default)
        if pd.isna(x):
            return float(default)
        return float(x)
    except Exception:
        return float(default)


def first_existing_value(d, keys, default=np.nan):
    if not isinstance(d, dict):
        return default
    for k in keys:
        if k in d and not pd.isna(d[k]):
            return d[k]
    return default


def normalize_threshold_dict(raw, fallback):
    """
    Normalize any legacy/current threshold dict to the corrected RV21D contract.

    Output keys:
        model_vrp_log
        model_vrp_z
        model_vrp_z_3m
        model_vrp_z_1y
        RSI14_max
        rv21d_vol_pct_min

    Notes:
        - If separate 3m/1y z thresholds are found, use max(z3m, z1y)
          so the equal-z version does not loosen either old z condition.
        - If an old forecast_vol_pct_min key exists in legacy configs, treat the
          numeric level as the intended RV21D floor but do not preserve the old name.
    """
    if raw is None:
        raw = {}

    model_vrp_log = clean_float(
        first_existing_value(
            raw,
            ["model_vrp_log", "vrp_log", "model_vrp_log_min"],
            fallback["model_vrp_log"],
        ),
        fallback["model_vrp_log"],
    )

    z_direct = first_existing_value(
        raw,
        ["model_vrp_z", "bucket_z", "z", "model_vrp_z_min"],
        np.nan,
    )
    z3 = first_existing_value(
        raw,
        ["model_vrp_z_3m", "z_3m", "model_vrp_z_3m_min"],
        np.nan,
    )
    z1 = first_existing_value(
        raw,
        ["model_vrp_z_1y", "z_1y", "model_vrp_z_1y_min"],
        np.nan,
    )

    if not pd.isna(z_direct):
        model_vrp_z = clean_float(z_direct, fallback["model_vrp_z"])
    else:
        z_candidates = [clean_float(v, np.nan) for v in [z3, z1]]
        z_candidates = [v for v in z_candidates if not pd.isna(v)]
        model_vrp_z = max(z_candidates) if z_candidates else fallback["model_vrp_z"]

    RSI14_max = clean_float(
        first_existing_value(
            raw,
            ["RSI14_max", "rsi_max", "RSI_cap", "rsi_cap"],
            fallback["RSI14_max"],
        ),
        fallback["RSI14_max"],
    )

    rv21d_vol_pct_min = clean_float(
        first_existing_value(
            raw,
            [
                "rv21d_vol_pct_min",
                "rv21d_min",
                "rv21d_vol_min",
                "rv21d_floor",
                "realized_vol_21d_pct_min",
                "rv21d_realized_vol_pct_min",

                # Legacy key name. Numeric value is treated as old RV21D floor level,
                # but the corrected output does not use forecast_vol as a threshold.
                "forecast_vol_pct_min",
            ],
            fallback["rv21d_vol_pct_min"],
        ),
        fallback["rv21d_vol_pct_min"],
    )

    out = {
        "model_vrp_log": float(model_vrp_log),
        "model_vrp_z": float(model_vrp_z),
        "model_vrp_z_3m": float(model_vrp_z),
        "model_vrp_z_1y": float(model_vrp_z),
        "RSI14_max": float(RSI14_max),
        "rv21d_vol_pct_min": float(rv21d_vol_pct_min),
    }

    return out


def between_or_equal(x, a, b, tol=1e-12):
    lo = min(float(a), float(b)) - tol
    hi = max(float(a), float(b)) + tol
    return lo <= float(x) <= hi


def clip_between(x, a, b):
    lo = min(float(a), float(b))
    hi = max(float(a), float(b))
    return float(min(max(float(x), lo), hi))


def constrain_middle_between_front_back(middle_raw, front, back):
    """
    Force default/reference Middle to obey hard constraint:
        Middle parameter levels between/equal Front and Back.

    If the user defines explicit globals that violate the constraint, validation
    will catch it. This function is used only for constructing the default
    reference configuration.
    """
    m = middle_raw.copy()

    for param in ["model_vrp_log", "model_vrp_z", "RSI14_max", "rv21d_vol_pct_min"]:
        m[param] = clip_between(m[param], front[param], back[param])

    m["model_vrp_z_3m"] = m["model_vrp_z"]
    m["model_vrp_z_1y"] = m["model_vrp_z"]

    return m


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def add_selection_ranks(df):
    d = df.copy()

    if d.empty:
        return d

    d["bucket_center_tenor"] = d["tenor_bucket"].map(BUCKET_CENTER_TENOR)
    d["distance_to_bucket_center"] = (d["tenor"] - d["bucket_center_tenor"]).abs()

    bucket_group = ["trade_date", "tenor_bucket"]

    d["rank_z_1y_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_bucket_date"] = d[
        [
            "rank_z_1y_in_bucket_date",
            "rank_z_3m_in_bucket_date",
            "rank_vrp_log_in_bucket_date",
        ]
    ].mean(axis=1)

    d["rank_z_1y_in_date"] = (
        d.groupby("trade_date")["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_date"] = (
        d.groupby("trade_date")["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_date"] = (
        d.groupby("trade_date")["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_date"] = d[
        [
            "rank_z_1y_in_date",
            "rank_z_3m_in_date",
            "rank_vrp_log_in_date",
        ]
    ].mean(axis=1)

    return d


def select_trade_universes(qualified):
    if qualified.empty:
        return {
            "all_qualified_trades": qualified.copy(),
            "one_trade_per_bucket_per_date_best_rank": qualified.copy(),
            "one_trade_per_date_best_rank": qualified.copy(),
            "thirty_day_anchor_only": qualified.copy(),
        }

    q = add_selection_ranks(qualified)

    one_trade_per_bucket_date = (
        q.sort_values(
            [
                "trade_date",
                "tenor_bucket_order",
                "avg_signal_rank_in_bucket_date",
                "distance_to_bucket_center",
                "rank_z_1y_in_bucket_date",
                "rank_z_3m_in_bucket_date",
                "rank_vrp_log_in_bucket_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date", "tenor_bucket"], as_index=False)
        .head(1)
        .copy()
    )

    one_trade_per_date = (
        q.sort_values(
            [
                "trade_date",
                "avg_signal_rank_in_date",
                "rank_z_1y_in_date",
                "rank_z_3m_in_date",
                "rank_vrp_log_in_date",
                "distance_to_bucket_center",
                "tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    thirty_day_anchor = q[q["tenor"].eq(30)].copy()

    return {
        "all_qualified_trades": q.copy(),
        "one_trade_per_bucket_per_date_best_rank": one_trade_per_bucket_date,
        "one_trade_per_date_best_rank": one_trade_per_date,
        "thirty_day_anchor_only": thirty_day_anchor,
    }


def summarize_trade_set(df, group_cols, selection_universe):
    d = df.copy()

    if d.empty:
        if len(group_cols) == 0:
            return pd.DataFrame([{
                "candidate_id": REFERENCE_CANDIDATE_ID,
                "selection_universe": selection_universe,
                "trades": 0,
                "unique_trade_dates": 0,
                "date_min": pd.NaT,
                "date_max": pd.NaT,
                "tenor_min": np.nan,
                "tenor_max": np.nan,
                "avg_tenor": np.nan,
                "win_rate": np.nan,
                "total_pnl": 0.0,
                "avg_pnl": np.nan,
                "median_pnl": np.nan,
                "worst_trade": np.nan,
                "best_trade": np.nan,
                "profit_factor": np.nan,
                "selected_trade_drawdown": np.nan,
                "worst_5_trade_sum": np.nan,
                "worst_10_trade_sum": np.nan,
                "worst_20_trade_sum": np.nan,
                "trades_le_neg_50k": 0,
                "trades_le_neg_100k": 0,
                "trades_le_neg_150k": 0,
                "total_pnl_per_day_sum": 0.0,
                "avg_pnl_per_day": np.nan,
                "median_pnl_per_day": np.nan,
                "worst_trade_pnl_per_day": np.nan,
                "best_trade_pnl_per_day": np.nan,
                "profit_factor_pnl_per_day": np.nan,
                "pnl_per_day_drawdown": np.nan,
                "worst_5_trade_pnl_per_day_sum": np.nan,
                "worst_10_trade_pnl_per_day_sum": np.nan,
                "worst_20_trade_pnl_per_day_sum": np.nan,
                "trades_2025": 0,
                "pnl_2025": 0.0,
                "avg_pnl_per_day_2025": np.nan,
                "worst_trade_2025": np.nan,
                "trades_2026": 0,
                "pnl_2026": 0.0,
                "avg_pnl_per_day_2026": np.nan,
                "worst_trade_2026": np.nan,
            }])
        return pd.DataFrame()

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    for c in [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "forecast_vol_pct",
        "rv21d_vol_pct",
    ]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return summarize_trade_set(d, group_cols, selection_universe)

    d = d.sort_values(["trade_date", "tenor"]).copy()

    rows = []

    if len(group_cols) == 0:
        grouped = [((), d)]
    else:
        grouped = d.groupby(group_cols, dropna=False, observed=False)

    for keys, g in grouped:
        if len(group_cols) == 0:
            keys = ()
        elif not isinstance(keys, tuple):
            keys = (keys,)

        row = {
            "candidate_id": REFERENCE_CANDIDATE_ID,
            "selection_universe": selection_universe,
        }

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": int(g["trade_date"].nunique()),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        for signal_col in ["rv21d_vol_pct", "forecast_vol_pct", "RSI14", "model_vrp_log", "model_vrp_z_3m", "model_vrp_z_1y"]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        rows.append(row)

    return pd.DataFrame(rows)


def attach_rule_columns(summary_df, rules_wide):
    if summary_df.empty:
        return summary_df

    return summary_df.merge(
        rules_wide,
        on="candidate_id",
        how="left",
        validate="many_to_one",
    )


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


# ======================================================================================
# 2. Load repaired signal base panel
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01R_unified_fds_signal_base_panel_with_rv21d_*.parquet",
    required=True,
)

base = pd.read_parquet(base_panel_path)

base["trade_date"] = pd.to_datetime(base["trade_date"], errors="coerce").dt.normalize()

required_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "implied_variance",
    "forecast_variance",
    "implied_vol_pct",
    "forecast_vol_pct",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

missing_required_cols = [c for c in required_cols if c not in base.columns]

if missing_required_cols:
    raise KeyError(f"Cell 4R missing required repaired base columns: {missing_required_cols}")

for c in [
    "tenor",
    "tenor_bucket_order",
    "implied_variance",
    "forecast_variance",
    "implied_vol_pct",
    "forecast_vol_pct",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    base[c] = pd.to_numeric(base[c], errors="coerce")

base["tenor"] = base["tenor"].astype(int)

# ======================================================================================
# 3. Build constrained reference rule
# ======================================================================================

legacy_fallback = {
    "Front": {
        "model_vrp_log": 0.60,
        "model_vrp_z": 0.65,
        "model_vrp_z_3m": 0.65,
        "model_vrp_z_1y": 0.65,
        "RSI14_max": 70.0,
        "rv21d_vol_pct_min": 8.5,
    },
    "Middle": {
        "model_vrp_log": 0.65,
        "model_vrp_z": 0.75,
        "model_vrp_z_3m": 0.75,
        "model_vrp_z_1y": 0.75,
        "RSI14_max": 68.0,
        "rv21d_vol_pct_min": 8.5,
    },
    "Back": {
        "model_vrp_log": 0.70,
        "model_vrp_z": 0.75,
        "model_vrp_z_3m": 0.75,
        "model_vrp_z_1y": 0.75,
        "RSI14_max": 66.0,
        "rv21d_vol_pct_min": 8.5,
    },
}

legacy_global = globals().get("LEGACY_CORE_REFERENCE_THRESHOLDS", {})
locked_back_global = globals().get("LOCKED_BACK_CORE_THRESHOLDS", legacy_fallback["Back"])

front_raw = legacy_global.get("Front", legacy_fallback["Front"]) if isinstance(legacy_global, dict) else legacy_fallback["Front"]
middle_raw = legacy_global.get("Middle", legacy_fallback["Middle"]) if isinstance(legacy_global, dict) else legacy_fallback["Middle"]

front_rule = normalize_threshold_dict(front_raw, legacy_fallback["Front"])
back_rule = normalize_threshold_dict(locked_back_global, legacy_fallback["Back"])
middle_desired = normalize_threshold_dict(middle_raw, legacy_fallback["Middle"])
middle_rule = constrain_middle_between_front_back(middle_desired, front_rule, back_rule)

rules_wide = pd.DataFrame([{
    "candidate_id": REFERENCE_CANDIDATE_ID,
    "candidate_family": REFERENCE_CANDIDATE_FAMILY,
    "candidate_subfamily": REFERENCE_CANDIDATE_SUBFAMILY,
    "candidate_description": (
        "Constrained Core RV21D reference baseline. Back locked; z_3m and z_1y equal "
        "within each bucket; Middle bounded between/equal Front and Back."
    ),
    "locked_model_spec": LOCKED_MODEL_SPEC,

    "front_model_vrp_log": front_rule["model_vrp_log"],
    "front_model_vrp_z": front_rule["model_vrp_z"],
    "front_model_vrp_z_3m": front_rule["model_vrp_z_3m"],
    "front_model_vrp_z_1y": front_rule["model_vrp_z_1y"],
    "front_RSI14_max": front_rule["RSI14_max"],
    "front_rv21d_vol_pct_min": front_rule["rv21d_vol_pct_min"],

    "middle_model_vrp_log": middle_rule["model_vrp_log"],
    "middle_model_vrp_z": middle_rule["model_vrp_z"],
    "middle_model_vrp_z_3m": middle_rule["model_vrp_z_3m"],
    "middle_model_vrp_z_1y": middle_rule["model_vrp_z_1y"],
    "middle_RSI14_max": middle_rule["RSI14_max"],
    "middle_rv21d_vol_pct_min": middle_rule["rv21d_vol_pct_min"],

    "back_model_vrp_log": back_rule["model_vrp_log"],
    "back_model_vrp_z": back_rule["model_vrp_z"],
    "back_model_vrp_z_3m": back_rule["model_vrp_z_3m"],
    "back_model_vrp_z_1y": back_rule["model_vrp_z_1y"],
    "back_RSI14_max": back_rule["RSI14_max"],
    "back_rv21d_vol_pct_min": back_rule["rv21d_vol_pct_min"],

    "back_locked": True,
    "middle_between_front_back": all([
        between_or_equal(middle_rule["model_vrp_log"], front_rule["model_vrp_log"], back_rule["model_vrp_log"]),
        between_or_equal(middle_rule["model_vrp_z"], front_rule["model_vrp_z"], back_rule["model_vrp_z"]),
        between_or_equal(middle_rule["RSI14_max"], front_rule["RSI14_max"], back_rule["RSI14_max"]),
        between_or_equal(middle_rule["rv21d_vol_pct_min"], front_rule["rv21d_vol_pct_min"], back_rule["rv21d_vol_pct_min"]),
    ]),
    "uses_rv21d_threshold_contract": True,
    "forecast_vol_pct_threshold_used": False,
}])

long_rows = []

bucket_rule_map = {
    "Front": front_rule,
    "Middle": middle_rule,
    "Back": back_rule,
}

for tenor in ALL_TENORS:
    bucket = TENOR_BUCKET_MAP[int(tenor)]
    rule = bucket_rule_map[bucket]

    long_rows.append({
        "candidate_id": REFERENCE_CANDIDATE_ID,
        "candidate_family": REFERENCE_CANDIDATE_FAMILY,
        "candidate_subfamily": REFERENCE_CANDIDATE_SUBFAMILY,
        "locked_model_spec": LOCKED_MODEL_SPEC,
        "tenor": int(tenor),
        "tenor_bucket": bucket,
        "tenor_bucket_order": BUCKET_ORDER_MAP[bucket],
        "rule_scope": bucket.lower(),
        "include_tenor": True,

        "model_vrp_log": rule["model_vrp_log"],
        "model_vrp_z": rule["model_vrp_z"],
        "model_vrp_z_3m": rule["model_vrp_z_3m"],
        "model_vrp_z_1y": rule["model_vrp_z_1y"],
        "RSI14_max": rule["RSI14_max"],
        "rv21d_vol_pct_min": rule["rv21d_vol_pct_min"],

        "is_front": bucket == "Front",
        "is_middle": bucket == "Middle",
        "is_back_locked": bucket == "Back",
    })

rules_long = pd.DataFrame(long_rows).sort_values("tenor").reset_index(drop=True)

# ======================================================================================
# 4. Apply reference rule
# ======================================================================================

threshold_join_cols = [
    "tenor",
    "candidate_id",
    "candidate_family",
    "candidate_subfamily",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14_max",
    "rv21d_vol_pct_min",
]

thresholds = rules_long[threshold_join_cols].rename(columns={
    "model_vrp_log": "threshold_model_vrp_log",
    "model_vrp_z_3m": "threshold_model_vrp_z_3m",
    "model_vrp_z_1y": "threshold_model_vrp_z_1y",
    "RSI14_max": "threshold_RSI14_max",
    "rv21d_vol_pct_min": "threshold_rv21d_vol_pct_min",
})

panel = base.merge(
    thresholds,
    on="tenor",
    how="left",
    validate="many_to_one",
)

required_signal_available = (
    panel["model_vrp_log"].notna()
    & panel["model_vrp_z_3m"].notna()
    & panel["model_vrp_z_1y"].notna()
    & panel["RSI14"].notna()
    & panel["rv21d_vol_pct"].notna()
    & panel["threshold_model_vrp_log"].notna()
    & panel["threshold_model_vrp_z_3m"].notna()
    & panel["threshold_model_vrp_z_1y"].notna()
    & panel["threshold_RSI14_max"].notna()
    & panel["threshold_rv21d_vol_pct_min"].notna()
)

outcome_available = (
    panel["normalized_pnl_dollars"].notna()
    & panel["normalized_pnl_per_day"].notna()
)

pass_mask = (
    required_signal_available
    & (panel["model_vrp_log"] > panel["threshold_model_vrp_log"])
    & (panel["model_vrp_z_3m"] > panel["threshold_model_vrp_z_3m"])
    & (panel["model_vrp_z_1y"] > panel["threshold_model_vrp_z_1y"])
    & (panel["RSI14"] < panel["threshold_RSI14_max"])
    & (panel["rv21d_vol_pct"] > panel["threshold_rv21d_vol_pct_min"])
)

panel["core_rv21d_reference_pass"] = pass_mask

qualified_all_dates = panel[pass_mask].copy()
qualified_complete = panel[pass_mask & outcome_available].copy()

universes = select_trade_universes(qualified_complete)

selected_trade_panel = pd.concat(
    [df.assign(selection_universe=name) for name, df in universes.items()],
    ignore_index=True,
)

# ======================================================================================
# 5. Summaries
# ======================================================================================

summary_frames = []
bucket_frames = []
tenor_frames = []
year_frames = []

for universe_name, df_u in universes.items():
    s = summarize_trade_set(
        df_u,
        group_cols=[],
        selection_universe=universe_name,
    )
    summary_frames.append(s)

    b = summarize_trade_set(
        df_u,
        group_cols=["tenor_bucket", "tenor_bucket_order"],
        selection_universe=universe_name,
    )
    bucket_frames.append(b)

    t = summarize_trade_set(
        df_u,
        group_cols=["tenor", "tenor_bucket", "tenor_bucket_order"],
        selection_universe=universe_name,
    )
    tenor_frames.append(t)

    if not df_u.empty:
        y = df_u.copy()
        y["year"] = y["trade_date"].dt.year.astype(int)

        yy = summarize_trade_set(
            y,
            group_cols=["year"],
            selection_universe=universe_name,
        )
        year_frames.append(yy)

summary = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
by_bucket = pd.concat(bucket_frames, ignore_index=True) if bucket_frames else pd.DataFrame()
by_tenor = pd.concat(tenor_frames, ignore_index=True) if tenor_frames else pd.DataFrame()
by_year = pd.concat(year_frames, ignore_index=True) if year_frames else pd.DataFrame()

summary = attach_rule_columns(summary, rules_wide)
by_bucket = attach_rule_columns(by_bucket, rules_wide)
by_tenor = attach_rule_columns(by_tenor, rules_wide)
by_year = attach_rule_columns(by_year, rules_wide)

summary = summary.sort_values(
    ["selection_universe", "avg_pnl_per_day"],
    ascending=[True, False],
).reset_index(drop=True)

if not by_bucket.empty:
    by_bucket = by_bucket.sort_values(
        ["selection_universe", "tenor_bucket_order"],
    ).reset_index(drop=True)

if not by_tenor.empty:
    by_tenor = by_tenor.sort_values(
        ["selection_universe", "tenor"],
    ).reset_index(drop=True)

if not by_year.empty:
    by_year = by_year.sort_values(
        ["selection_universe", "year"],
    ).reset_index(drop=True)

# Pass diagnostics.
pass_diagnostics = pd.DataFrame([{
    "candidate_id": REFERENCE_CANDIDATE_ID,
    "candidate_family": REFERENCE_CANDIDATE_FAMILY,
    "candidate_subfamily": REFERENCE_CANDIDATE_SUBFAMILY,
    "base_rows": int(len(base)),
    "required_signal_available_rows": int(required_signal_available.sum()),
    "outcome_available_rows": int(outcome_available.sum()),
    "pass_rows_all_dates": int(pass_mask.sum()),
    "pass_rows_with_outcomes": int((pass_mask & outcome_available).sum()),
    "qualified_all_dates_rows": int(len(qualified_all_dates)),
    "qualified_complete_rows": int(len(qualified_complete)),
    "selected_trade_panel_rows": int(len(selected_trade_panel)),
}])

for bucket in ["Front", "Middle", "Back"]:
    bucket_mask = panel["tenor_bucket"].eq(bucket)
    pass_diagnostics[f"{bucket}_pass_rows_all_dates"] = int((pass_mask & bucket_mask).sum())
    pass_diagnostics[f"{bucket}_pass_rows_with_outcomes"] = int((pass_mask & outcome_available & bucket_mask).sum())

for tenor in ALL_TENORS:
    tenor_mask = panel["tenor"].eq(tenor)
    pass_diagnostics[f"tenor_{tenor}_pass_rows_all_dates"] = int((pass_mask & tenor_mask).sum())
    pass_diagnostics[f"tenor_{tenor}_pass_rows_with_outcomes"] = int((pass_mask & outcome_available & tenor_mask).sum())

# ======================================================================================
# 6. Save outputs
# ======================================================================================

rules_wide_path = OUT_AUDIT_DIR / f"04R_core_rv21d_reference_rules_wide_{CELL4R_TIMESTAMP}.csv"
rules_long_path = OUT_AUDIT_DIR / f"04R_core_rv21d_reference_rules_long_{CELL4R_TIMESTAMP}.csv"
summary_path = OUT_AUDIT_DIR / f"04R_core_rv21d_reference_summary_{CELL4R_TIMESTAMP}.csv"
by_bucket_path = OUT_AUDIT_DIR / f"04R_core_rv21d_reference_by_bucket_{CELL4R_TIMESTAMP}.csv"
by_tenor_path = OUT_AUDIT_DIR / f"04R_core_rv21d_reference_by_tenor_{CELL4R_TIMESTAMP}.csv"
by_year_path = OUT_AUDIT_DIR / f"04R_core_rv21d_reference_by_year_{CELL4R_TIMESTAMP}.csv"
pass_diagnostics_path = OUT_AUDIT_DIR / f"04R_core_rv21d_reference_pass_diagnostics_{CELL4R_TIMESTAMP}.csv"
selected_trade_panel_path = OUT_PROCESSED_DIR / f"04R_core_rv21d_reference_selected_trade_panel_{CELL4R_TIMESTAMP}.parquet"
validation_path = OUT_AUDIT_DIR / f"04R_core_rv21d_reference_validation_{CELL4R_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"04R_core_rv21d_reference_manifest_{CELL4R_TIMESTAMP}.json"

rules_wide.to_csv(rules_wide_path, index=False)
rules_long.to_csv(rules_long_path, index=False)
summary.to_csv(summary_path, index=False)
by_bucket.to_csv(by_bucket_path, index=False)
by_tenor.to_csv(by_tenor_path, index=False)
by_year.to_csv(by_year_path, index=False)
pass_diagnostics.to_csv(pass_diagnostics_path, index=False)
selected_trade_panel.to_parquet(selected_trade_panel_path, index=False)

# ======================================================================================
# 7. Validation
# ======================================================================================

validation_rows = []

models_found = sorted(base["model_spec"].dropna().unique().tolist())

duplicate_keys = (
    base.groupby(["trade_date", "tenor"], dropna=False)
    .size()
    .reset_index(name="rows")
)
duplicate_key_count = int((duplicate_keys["rows"] > 1).sum())

tenors_found = sorted(base["tenor"].dropna().astype(int).unique().tolist())
missing_tenors = sorted(set(ALL_TENORS) - set(tenors_found))
extra_tenors = sorted(set(tenors_found) - set(ALL_TENORS))

bad_bucket_rows = base[
    base["tenor"].map(TENOR_BUCKET_MAP).ne(base["tenor_bucket"])
]

rv21d_date_level = (
    base.groupby("trade_date")["rv21d_vol_pct"]
    .nunique(dropna=True)
    .reset_index(name="rv21d_unique_non_null_values")
)
bad_rv21d_dates = rv21d_date_level[
    rv21d_date_level["rv21d_unique_non_null_values"] > 1
]

old_forecast_threshold_cols = [
    c for c in list(rules_wide.columns) + list(rules_long.columns)
    if "forecast_vol_pct_min" in c
]

bad_z_equality = rules_long[
    rules_long["model_vrp_z_3m"].ne(rules_long["model_vrp_z_1y"])
]

bad_middle_between_rows = []

for param_wide, front_col, middle_col, back_col in [
    ("model_vrp_log", "front_model_vrp_log", "middle_model_vrp_log", "back_model_vrp_log"),
    ("model_vrp_z", "front_model_vrp_z", "middle_model_vrp_z", "back_model_vrp_z"),
    ("RSI14_max", "front_RSI14_max", "middle_RSI14_max", "back_RSI14_max"),
    ("rv21d_vol_pct_min", "front_rv21d_vol_pct_min", "middle_rv21d_vol_pct_min", "back_rv21d_vol_pct_min"),
]:
    f = float(rules_wide[front_col].iloc[0])
    m = float(rules_wide[middle_col].iloc[0])
    b = float(rules_wide[back_col].iloc[0])

    if not between_or_equal(m, f, b):
        bad_middle_between_rows.append({
            "parameter": param_wide,
            "front": f,
            "middle": m,
            "back": b,
        })

bad_middle_between = pd.DataFrame(bad_middle_between_rows)

bad_back_lock = []
for param, col, expected_value in [
    ("model_vrp_log", "back_model_vrp_log", back_rule["model_vrp_log"]),
    ("model_vrp_z", "back_model_vrp_z", back_rule["model_vrp_z"]),
    ("RSI14_max", "back_RSI14_max", back_rule["RSI14_max"]),
    ("rv21d_vol_pct_min", "back_rv21d_vol_pct_min", back_rule["rv21d_vol_pct_min"]),
]:
    observed = float(rules_wide[col].iloc[0])
    if abs(observed - float(expected_value)) > 1e-12:
        bad_back_lock.append({
            "parameter": param,
            "observed": observed,
            "expected": float(expected_value),
        })

bad_back_lock = pd.DataFrame(bad_back_lock)

bad_normalized_pnl = base[
    base["normalized_pnl_dollars"].notna()
    & base["normalized_pnl_per_day"].notna()
    & (
        (
            base["normalized_pnl_dollars"] / base["tenor"].replace(0, np.nan)
        )
        - base["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
]

expected_summary_rows = len(SELECTION_UNIVERSES)
summary_universes_found = sorted(summary["selection_universe"].dropna().unique().tolist())

add_validation(
    validation_rows,
    "repaired_signal_panel_loaded",
    "PASS" if len(base) > 0 else "FAIL",
    f"rows={len(base):,}; path={base_panel_path}",
)

add_validation(
    validation_rows,
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    validation_rows,
    "one_row_per_trade_date_tenor",
    "PASS" if duplicate_key_count == 0 else "FAIL",
    f"duplicate_key_count={duplicate_key_count:,}",
)

add_validation(
    validation_rows,
    "all_exact_tenors_present",
    "PASS" if not missing_tenors and not extra_tenors else "FAIL",
    f"missing={missing_tenors}; extra={extra_tenors}",
)

add_validation(
    validation_rows,
    "bucket_mapping",
    "PASS" if bad_bucket_rows.empty else "FAIL",
    f"bad_rows={len(bad_bucket_rows):,}",
)

add_validation(
    validation_rows,
    "rv21d_vol_pct_available",
    "PASS" if base["rv21d_vol_pct"].notna().sum() > 0 else "FAIL",
    f"available_rows={int(base['rv21d_vol_pct'].notna().sum()):,}; total_rows={len(base):,}",
)

add_validation(
    validation_rows,
    "rv21d_date_level_not_tenor_specific",
    "PASS" if bad_rv21d_dates.empty else "FAIL",
    f"bad_dates={len(bad_rv21d_dates):,}",
)

add_validation(
    validation_rows,
    "uses_rv21d_threshold_contract",
    "PASS" if "rv21d_vol_pct_min" in rules_long.columns else "FAIL",
    "rule field is rv21d_vol_pct_min",
)

add_validation(
    validation_rows,
    "no_forecast_vol_threshold_columns",
    "PASS" if not old_forecast_threshold_cols else "FAIL",
    f"old_forecast_threshold_cols={old_forecast_threshold_cols}",
)

add_validation(
    validation_rows,
    "forecast_vol_pct_retained_as_diagnostic",
    "PASS" if "forecast_vol_pct" in base.columns else "FAIL",
    "forecast_vol_pct remains in base panel only as model/diagnostic field",
)

add_validation(
    validation_rows,
    "back_locked",
    "PASS" if bad_back_lock.empty else "FAIL",
    f"bad_back_lock_rows={len(bad_back_lock):,}",
)

add_validation(
    validation_rows,
    "z_3m_equals_z_1y_within_bucket_rules",
    "PASS" if bad_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_z_equality):,}",
)

add_validation(
    validation_rows,
    "middle_between_or_equal_front_back",
    "PASS" if bad_middle_between.empty else "FAIL",
    f"bad_params={bad_middle_between.to_dict('records') if not bad_middle_between.empty else []}",
)

add_validation(
    validation_rows,
    "normalized_pnl_per_day_formula",
    "PASS" if bad_normalized_pnl.empty else "FAIL",
    f"bad_rows={len(bad_normalized_pnl):,}",
)

add_validation(
    validation_rows,
    "summary_rows_complete",
    "PASS" if len(summary) == expected_summary_rows else "FAIL",
    f"summary_rows={len(summary):,}; expected={expected_summary_rows:,}",
)

add_validation(
    validation_rows,
    "all_selection_universes_evaluated",
    "PASS" if summary_universes_found == sorted(SELECTION_UNIVERSES) else "FAIL",
    f"found={summary_universes_found}",
)

add_validation(
    validation_rows,
    "selected_trade_panel_saved",
    "PASS" if len(selected_trade_panel) > 0 else "WARN",
    f"rows={len(selected_trade_panel):,}; path={selected_trade_panel_path}",
)

add_validation(
    validation_rows,
    "no_threshold_sweep",
    "PASS",
    "One constrained reference configuration only.",
)

add_validation(
    validation_rows,
    "no_secondary",
    "PASS",
    "Core only.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed or added.",
)

add_validation(
    validation_rows,
    "no_production_lock",
    "PASS",
    "No production threshold lock performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"].eq("FAIL")]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1_RV21D_repaired_v2.ipynb",
    "cell": "Cell 4R — Core RV21D reference baseline",
    "timestamp": CELL4R_TIMESTAMP,
    "base_panel_path": str(base_panel_path),
    "candidate_id": REFERENCE_CANDIDATE_ID,
    "candidate_family": REFERENCE_CANDIDATE_FAMILY,
    "selection_universes": SELECTION_UNIVERSES,
    "primary_selection_universe": PRIMARY_SELECTION_UNIVERSE,
    "primary_ranking_metric": PRIMARY_RANKING_METRIC,
    "rules_wide_path": str(rules_wide_path),
    "rules_long_path": str(rules_long_path),
    "summary_path": str(summary_path),
    "by_bucket_path": str(by_bucket_path),
    "by_tenor_path": str(by_tenor_path),
    "by_year_path": str(by_year_path),
    "pass_diagnostics_path": str(pass_diagnostics_path),
    "selected_trade_panel_path": str(selected_trade_panel_path),
    "validation_path": str(validation_path),
    "threshold_contract": "rv21d_vol_pct > rv21d_vol_pct_min",
    "forecast_vol_pct_role": "model denominator / forecast diagnostic only, not threshold floor",
    "front_rule": front_rule,
    "middle_rule": middle_rule,
    "back_rule": back_rule,
    "hard_constraints": [
        "Back bucket parameters remain locked.",
        "3m and 1y z-score thresholds are equal within each bucket.",
        "Middle parameter levels are between/equal Front and Back.",
        "No threshold sweep.",
        "No Secondary.",
        "No sizing changes.",
        "No production lock.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 4R validation failed. Review validation output above.")

# ======================================================================================
# 8. Display review
# ======================================================================================

print("=" * 100)
print("Cell 4R — Core RV21D reference baseline")
print("=" * 100)
print(f"Signal base panel:                 {base_panel_path}")
print(f"Candidate ID:                      {REFERENCE_CANDIDATE_ID}")
print(f"Base rows:                         {len(base):,}")
print(f"Qualified complete rows:           {len(qualified_complete):,}")
print(f"Selected trade panel rows:         {len(selected_trade_panel):,}")
print(f"Threshold contract:                rv21d_vol_pct > rv21d_vol_pct_min")
print(f"Primary metric:                    {PRIMARY_RANKING_METRIC}")
print("Back locked:                       True")
print("3m/1y z equal within bucket:        True")
print("Middle between Front/Back:         True")
print("No threshold sweep:                True")
print("No Secondary:                      True")
print("No sizing changes:                 True")
print("No production lock:                True")

print()
print("Validation:")
display(validation)

print()
print("Reference rules — wide:")
display(rules_wide)

print()
print("Reference rules — long:")
display(rules_long)

print()
print("Pass diagnostics:")
display(pass_diagnostics)

display_cols = [
    "selection_universe",
    "trades",
    "unique_trade_dates",
    "avg_tenor",
    "win_rate",
    "total_pnl",
    "profit_factor",
    "avg_pnl_per_day",
    "median_pnl_per_day",
    "worst_trade_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "trades_le_neg_100k",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
    "front_model_vrp_log",
    "front_model_vrp_z",
    "front_RSI14_max",
    "front_rv21d_vol_pct_min",
    "middle_model_vrp_log",
    "middle_model_vrp_z",
    "middle_RSI14_max",
    "middle_rv21d_vol_pct_min",
    "back_model_vrp_log",
    "back_model_vrp_z",
    "back_RSI14_max",
    "back_rv21d_vol_pct_min",
]

print()
print("Core RV21D reference summary:")
display(summary[[c for c in display_cols if c in summary.columns]])

print()
print("By-bucket detail:")
display(by_bucket)

print()
print("By-tenor detail:")
display(by_tenor)

print()
print("By-year detail:")
display(by_year)

print()
print("Saved outputs:")
for p in [
    rules_wide_path,
    rules_long_path,
    summary_path,
    by_bucket_path,
    by_tenor_path,
    by_year_path,
    pass_diagnostics_path,
    selected_trade_panel_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 4R PASSED")

In [ ]:
# ======================================================================================
# Cell 5R — Front RV21D weakness diagnostic and candidate design
# ======================================================================================
#
# Objective:
#   Diagnose Front weakness under the repaired RV21D threshold contract and create a
#   constrained Front candidate grid for the next sweep.
#
# Correct volatility-floor contract:
#       rv21d_vol_pct > rv21d_vol_pct_min
#
# NOT:
#       forecast_vol_pct > forecast_vol_pct_min
#
# Hard constraints:
#   1. Back bucket parameters remain locked.
#   2. Within each bucket, 3m and 1y z-score thresholds are equal.
#   3. Middle parameter levels must be between/equal Front and Back.
#   4. forecast_vol_pct remains diagnostic only.
#   5. No threshold sweep in this cell.
#   6. No final lock.
#   7. No Secondary.
#   8. No sizing changes.
#   9. No production lock.
#
# Candidate design only:
#   0001 current Front reference
#   0002 tighter Front model_vrp_log to 0.65
#   0003 tighter Front model_vrp_log to 0.70
#   0004 tighter Front z to 0.70
#   0005 tighter Front RSI cap to 68
#   0006 higher Front RV21D floor to 10.0
#   0007 higher Front RV21D floor to 12.0
#   0008 exclude 9D
#   0009 9D only RV21D floor to 12.0
#   0010 9D only RV21D floor to 14.5
#
# This cell does NOT evaluate the new candidates.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 500)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL5R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

if "ALL_TENORS" not in globals():
    ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

if "TENOR_BUCKET_MAP" not in globals():
    TENOR_BUCKET_MAP = {
        9: "Front",
        12: "Front",
        15: "Front",
        18: "Middle",
        21: "Middle",
        24: "Middle",
        27: "Back",
        30: "Back",
        33: "Back",
    }

if "BUCKET_ORDER_MAP" not in globals():
    BUCKET_ORDER_MAP = {
        "Front": 1,
        "Middle": 2,
        "Back": 3,
    }

PRIMARY_SELECTION_UNIVERSE = "one_trade_per_bucket_per_date_best_rank"
ALL_QUALIFIED_UNIVERSE = "all_qualified_trades"
REFERENCE_CANDIDATE_ID = "core_rv21d_reference_0001"

FRONT_CANDIDATE_FAMILY = "core_front_rv21d_candidate_design"

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def to_float(x, default=np.nan):
    try:
        if x is None:
            return float(default)
        if pd.isna(x):
            return float(default)
        return float(x)
    except Exception:
        return float(default)


def rule_dict(model_vrp_log, model_vrp_z, RSI14_max, rv21d_vol_pct_min):
    return {
        "model_vrp_log": float(model_vrp_log),
        "model_vrp_z": float(model_vrp_z),
        "model_vrp_z_3m": float(model_vrp_z),
        "model_vrp_z_1y": float(model_vrp_z),
        "RSI14_max": float(RSI14_max),
        "rv21d_vol_pct_min": float(rv21d_vol_pct_min),
    }


def clone_rule(rule):
    return {
        "model_vrp_log": float(rule["model_vrp_log"]),
        "model_vrp_z": float(rule["model_vrp_z"]),
        "model_vrp_z_3m": float(rule["model_vrp_z"]),
        "model_vrp_z_1y": float(rule["model_vrp_z"]),
        "RSI14_max": float(rule["RSI14_max"]),
        "rv21d_vol_pct_min": float(rule["rv21d_vol_pct_min"]),
    }


def update_rule(rule, **kwargs):
    out = clone_rule(rule)

    for k, v in kwargs.items():
        if k in ["model_vrp_z", "model_vrp_z_3m", "model_vrp_z_1y"]:
            out["model_vrp_z"] = float(v)
            out["model_vrp_z_3m"] = float(v)
            out["model_vrp_z_1y"] = float(v)
        elif k in out:
            out[k] = float(v)
        else:
            raise KeyError(f"Unexpected rule key: {k}")

    out["model_vrp_z_3m"] = float(out["model_vrp_z"])
    out["model_vrp_z_1y"] = float(out["model_vrp_z"])

    return out


def between_or_equal(x, a, b, tol=1e-12):
    lo = min(float(a), float(b)) - tol
    hi = max(float(a), float(b)) + tol
    return lo <= float(x) <= hi


def clip_between(x, a, b):
    lo = min(float(a), float(b))
    hi = max(float(a), float(b))
    return float(min(max(float(x), lo), hi))


def constrain_middle_between_front_back(middle_rule, front_rule_for_middle_constraint, back_rule):
    """
    Middle must remain between/equal Front and Back.

    For bucket-wide Front candidates, front_rule_for_middle_constraint is the
    modified Front rule.

    For 9D-specific candidates, front_rule_for_middle_constraint is the 12D/15D
    Front rule, because the bucket-level Front rule for 12D/15D is unchanged.
    """
    m = clone_rule(middle_rule)

    for param in ["model_vrp_log", "model_vrp_z", "RSI14_max", "rv21d_vol_pct_min"]:
        m[param] = clip_between(
            m[param],
            front_rule_for_middle_constraint[param],
            back_rule[param],
        )

    m["model_vrp_z_3m"] = m["model_vrp_z"]
    m["model_vrp_z_1y"] = m["model_vrp_z"]

    return m


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def summarize_trades(df, group_cols, diagnostic_label):
    d = df.copy()

    if d.empty:
        return pd.DataFrame()

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    numeric_cols = [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "rv21d_vol_pct",
        "forecast_vol_pct",
        "RSI14",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "front_min_z",
    ]

    for c in numeric_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame()

    d = d.sort_values(["trade_date", "tenor"]).copy()

    if len(group_cols) == 0:
        grouped = [((), d)]
    else:
        grouped = d.groupby(group_cols, dropna=False, observed=False)

    rows = []

    for keys, g in grouped:
        if len(group_cols) == 0:
            keys = ()
        elif not isinstance(keys, tuple):
            keys = (keys,)

        row = {
            "diagnostic_label": diagnostic_label,
        }

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": int(g["trade_date"].nunique()),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        for signal_col in [
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "front_min_z",
        ]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        rows.append(row)

    return pd.DataFrame(rows)


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


# ======================================================================================
# 2. Load repaired baseline and Cell 4R reference outputs
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01R_unified_fds_signal_base_panel_with_rv21d_*.parquet",
    required=True,
)

selected_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "04R_core_rv21d_reference_selected_trade_panel_*.parquet",
    required=True,
)

rules_wide_path = latest_file(
    OUT_AUDIT_DIR,
    "04R_core_rv21d_reference_rules_wide_*.csv",
    required=True,
)

rules_long_path = latest_file(
    OUT_AUDIT_DIR,
    "04R_core_rv21d_reference_rules_long_*.csv",
    required=True,
)

cell4_by_bucket_path = latest_file(
    OUT_AUDIT_DIR,
    "04R_core_rv21d_reference_by_bucket_*.csv",
    required=True,
)

cell4_by_tenor_path = latest_file(
    OUT_AUDIT_DIR,
    "04R_core_rv21d_reference_by_tenor_*.csv",
    required=True,
)

cell4_by_year_path = latest_file(
    OUT_AUDIT_DIR,
    "04R_core_rv21d_reference_by_year_*.csv",
    required=True,
)

base = pd.read_parquet(base_panel_path)
selected = pd.read_parquet(selected_panel_path)
rules_wide_ref = pd.read_csv(rules_wide_path)
rules_long_ref = pd.read_csv(rules_long_path)
cell4_by_bucket = pd.read_csv(cell4_by_bucket_path)
cell4_by_tenor = pd.read_csv(cell4_by_tenor_path)
cell4_by_year = pd.read_csv(cell4_by_year_path)

for df in [base, selected]:
    df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce").dt.normalize()

    for c in [
        "tenor",
        "tenor_bucket_order",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "rv21d_vol_pct",
        "forecast_vol_pct",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
    ]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

selected_ref = selected[selected["candidate_id"].eq(REFERENCE_CANDIDATE_ID)].copy()

front_all_qualified = selected_ref[
    selected_ref["selection_universe"].eq(ALL_QUALIFIED_UNIVERSE)
    & selected_ref["tenor_bucket"].eq("Front")
].copy()

front_primary = selected_ref[
    selected_ref["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
    & selected_ref["tenor_bucket"].eq("Front")
].copy()

for df in [front_all_qualified, front_primary]:
    df["front_min_z"] = df[["model_vrp_z_3m", "model_vrp_z_1y"]].min(axis=1)

# ======================================================================================
# 3. Front diagnostics under current Cell 4R reference
# ======================================================================================

front_by_tenor = pd.concat(
    [
        summarize_trades(
            front_all_qualified,
            group_cols=["selection_universe", "tenor", "tenor_bucket", "tenor_bucket_order"],
            diagnostic_label="front_all_qualified_by_tenor",
        ),
        summarize_trades(
            front_primary,
            group_cols=["selection_universe", "tenor", "tenor_bucket", "tenor_bucket_order"],
            diagnostic_label="front_primary_by_tenor",
        ),
    ],
    ignore_index=True,
)

front_all_year = front_all_qualified.copy()
front_all_year["year"] = front_all_year["trade_date"].dt.year.astype(int)

front_primary_year = front_primary.copy()
front_primary_year["year"] = front_primary_year["trade_date"].dt.year.astype(int)

front_by_year = pd.concat(
    [
        summarize_trades(
            front_all_year,
            group_cols=["selection_universe", "year"],
            diagnostic_label="front_all_qualified_by_year",
        ),
        summarize_trades(
            front_primary_year,
            group_cols=["selection_universe", "year"],
            diagnostic_label="front_primary_by_year",
        ),
    ],
    ignore_index=True,
)

front_by_tenor_year = pd.concat(
    [
        summarize_trades(
            front_all_year,
            group_cols=["selection_universe", "tenor", "year"],
            diagnostic_label="front_all_qualified_by_tenor_year",
        ),
        summarize_trades(
            front_primary_year,
            group_cols=["selection_universe", "tenor", "year"],
            diagnostic_label="front_primary_by_tenor_year",
        ),
    ],
    ignore_index=True,
)

front_state = pd.concat(
    [
        front_all_qualified.assign(state_source="all_qualified"),
        front_primary.assign(state_source="primary_selected"),
    ],
    ignore_index=True,
)

front_state["front_min_z"] = front_state[["model_vrp_z_3m", "model_vrp_z_1y"]].min(axis=1)

front_state["rv21d_state"] = pd.cut(
    front_state["rv21d_vol_pct"],
    bins=[-np.inf, 8.5, 10.0, 12.0, 14.5, 16.0, 18.0, 20.0, np.inf],
    labels=["<=8.5", "8.5-10", "10-12", "12-14.5", "14.5-16", "16-18", "18-20", ">20"],
    include_lowest=True,
)

front_state["RSI14_state"] = pd.cut(
    front_state["RSI14"],
    bins=[-np.inf, 55.0, 60.0, 64.0, 66.0, 68.0, 70.0, np.inf],
    labels=["<=55", "55-60", "60-64", "64-66", "66-68", "68-70", ">70"],
    include_lowest=True,
)

front_state["model_vrp_log_state"] = pd.cut(
    front_state["model_vrp_log"],
    bins=[-np.inf, 0.60, 0.65, 0.70, 0.75, 0.80, 0.90, np.inf],
    labels=["<=0.60", "0.60-0.65", "0.65-0.70", "0.70-0.75", "0.75-0.80", "0.80-0.90", ">0.90"],
    include_lowest=True,
)

front_state["front_min_z_state"] = pd.cut(
    front_state["front_min_z"],
    bins=[-np.inf, 0.65, 0.70, 0.80, 1.00, 1.25, 1.50, np.inf],
    labels=["<=0.65", "0.65-0.70", "0.70-0.80", "0.80-1.00", "1.00-1.25", "1.25-1.50", ">1.50"],
    include_lowest=True,
)

front_state_diagnostics = pd.concat(
    [
        summarize_trades(
            front_state,
            group_cols=["state_source", "rv21d_state"],
            diagnostic_label="front_by_rv21d_state",
        ),
        summarize_trades(
            front_state,
            group_cols=["state_source", "RSI14_state"],
            diagnostic_label="front_by_RSI14_state",
        ),
        summarize_trades(
            front_state,
            group_cols=["state_source", "model_vrp_log_state"],
            diagnostic_label="front_by_model_vrp_log_state",
        ),
        summarize_trades(
            front_state,
            group_cols=["state_source", "front_min_z_state"],
            diagnostic_label="front_by_min_z_state",
        ),
    ],
    ignore_index=True,
)

front_2026_detail = front_primary[
    front_primary["trade_date"].dt.year.eq(2026)
].copy()

front_2026_detail = front_2026_detail[
    [
        c for c in [
            "candidate_id",
            "selection_universe",
            "trade_date",
            "tenor",
            "tenor_bucket",
            "normalized_pnl_dollars",
            "normalized_pnl_per_day",
            "win",
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "front_min_z",
            "implied_vol_pct",
            "model_vrp_ratio",
        ]
        if c in front_2026_detail.columns
    ]
].sort_values(["trade_date", "tenor"]).reset_index(drop=True)

front_2026_worst = (
    front_2026_detail
    .sort_values("normalized_pnl_per_day")
    .head(30)
    .reset_index(drop=True)
)

# ======================================================================================
# 4. Build Front candidate design
# ======================================================================================

rw = rules_wide_ref.iloc[0].to_dict()

front_current = rule_dict(
    rw["front_model_vrp_log"],
    rw["front_model_vrp_z"],
    rw["front_RSI14_max"],
    rw["front_rv21d_vol_pct_min"],
)

middle_current = rule_dict(
    rw["middle_model_vrp_log"],
    rw["middle_model_vrp_z"],
    rw["middle_RSI14_max"],
    rw["middle_rv21d_vol_pct_min"],
)

back_locked = rule_dict(
    rw["back_model_vrp_log"],
    rw["back_model_vrp_z"],
    rw["back_RSI14_max"],
    rw["back_rv21d_vol_pct_min"],
)

candidate_specs = [
    {
        "candidate_id": "core_front_rv21d_0001",
        "candidate_subfamily": "current_front_reference",
        "candidate_description": "Current Front reference from Cell 4R.",
        "front9_rule": clone_rule(front_current),
        "front1215_rule": clone_rule(front_current),
        "include_9d": True,
    },
    {
        "candidate_id": "core_front_rv21d_0002",
        "candidate_subfamily": "front_log_0_65",
        "candidate_description": "Tighten bucket-wide Front model_vrp_log threshold to 0.65.",
        "front9_rule": update_rule(front_current, model_vrp_log=0.65),
        "front1215_rule": update_rule(front_current, model_vrp_log=0.65),
        "include_9d": True,
    },
    {
        "candidate_id": "core_front_rv21d_0003",
        "candidate_subfamily": "front_log_0_70",
        "candidate_description": "Tighten bucket-wide Front model_vrp_log threshold to 0.70.",
        "front9_rule": update_rule(front_current, model_vrp_log=0.70),
        "front1215_rule": update_rule(front_current, model_vrp_log=0.70),
        "include_9d": True,
    },
    {
        "candidate_id": "core_front_rv21d_0004",
        "candidate_subfamily": "front_z_0_70",
        "candidate_description": "Tighten bucket-wide Front 3m/1y z threshold to 0.70.",
        "front9_rule": update_rule(front_current, model_vrp_z=0.70),
        "front1215_rule": update_rule(front_current, model_vrp_z=0.70),
        "include_9d": True,
    },
    {
        "candidate_id": "core_front_rv21d_0005",
        "candidate_subfamily": "front_RSI14_68",
        "candidate_description": "Tighten bucket-wide Front RSI14 cap to 68.",
        "front9_rule": update_rule(front_current, RSI14_max=68.0),
        "front1215_rule": update_rule(front_current, RSI14_max=68.0),
        "include_9d": True,
    },
    {
        "candidate_id": "core_front_rv21d_0006",
        "candidate_subfamily": "front_rv21d_10",
        "candidate_description": "Raise bucket-wide Front RV21D floor to 10.0.",
        "front9_rule": update_rule(front_current, rv21d_vol_pct_min=10.0),
        "front1215_rule": update_rule(front_current, rv21d_vol_pct_min=10.0),
        "include_9d": True,
    },
    {
        "candidate_id": "core_front_rv21d_0007",
        "candidate_subfamily": "front_rv21d_12",
        "candidate_description": "Raise bucket-wide Front RV21D floor to 12.0.",
        "front9_rule": update_rule(front_current, rv21d_vol_pct_min=12.0),
        "front1215_rule": update_rule(front_current, rv21d_vol_pct_min=12.0),
        "include_9d": True,
    },
    {
        "candidate_id": "core_front_rv21d_0008",
        "candidate_subfamily": "front_ex_9d",
        "candidate_description": "Exclude 9D from Front; keep 12D/15D Front reference unchanged.",
        "front9_rule": clone_rule(front_current),
        "front1215_rule": clone_rule(front_current),
        "include_9d": False,
    },
    {
        "candidate_id": "core_front_rv21d_0009",
        "candidate_subfamily": "front_9d_rv21d_12",
        "candidate_description": "Allow 9D only with RV21D floor 12.0; keep 12D/15D Front reference unchanged.",
        "front9_rule": update_rule(front_current, rv21d_vol_pct_min=12.0),
        "front1215_rule": clone_rule(front_current),
        "include_9d": True,
    },
    {
        "candidate_id": "core_front_rv21d_0010",
        "candidate_subfamily": "front_9d_rv21d_14_5",
        "candidate_description": "Allow 9D only with RV21D floor 14.5; keep 12D/15D Front reference unchanged.",
        "front9_rule": update_rule(front_current, rv21d_vol_pct_min=14.5),
        "front1215_rule": clone_rule(front_current),
        "include_9d": True,
    },
]

wide_rows = []
long_rows = []

for spec in candidate_specs:
    front9_rule = clone_rule(spec["front9_rule"])
    front1215_rule = clone_rule(spec["front1215_rule"])

    # Middle constrained against the bucket-level 12D/15D Front rule and locked Back.
    middle_rule = constrain_middle_between_front_back(
        middle_current,
        front1215_rule,
        back_locked,
    )

    wide_rows.append({
        "candidate_id": spec["candidate_id"],
        "candidate_family": FRONT_CANDIDATE_FAMILY,
        "candidate_subfamily": spec["candidate_subfamily"],
        "candidate_description": spec["candidate_description"],
        "locked_model_spec": LOCKED_MODEL_SPEC,

        "front_9d_include": bool(spec["include_9d"]),
        "front_9d_model_vrp_log": front9_rule["model_vrp_log"],
        "front_9d_model_vrp_z": front9_rule["model_vrp_z"],
        "front_9d_model_vrp_z_3m": front9_rule["model_vrp_z_3m"],
        "front_9d_model_vrp_z_1y": front9_rule["model_vrp_z_1y"],
        "front_9d_RSI14_max": front9_rule["RSI14_max"],
        "front_9d_rv21d_vol_pct_min": front9_rule["rv21d_vol_pct_min"],

        "front_12_15_include": True,
        "front_12_15_model_vrp_log": front1215_rule["model_vrp_log"],
        "front_12_15_model_vrp_z": front1215_rule["model_vrp_z"],
        "front_12_15_model_vrp_z_3m": front1215_rule["model_vrp_z_3m"],
        "front_12_15_model_vrp_z_1y": front1215_rule["model_vrp_z_1y"],
        "front_12_15_RSI14_max": front1215_rule["RSI14_max"],
        "front_12_15_rv21d_vol_pct_min": front1215_rule["rv21d_vol_pct_min"],

        "middle_model_vrp_log": middle_rule["model_vrp_log"],
        "middle_model_vrp_z": middle_rule["model_vrp_z"],
        "middle_model_vrp_z_3m": middle_rule["model_vrp_z_3m"],
        "middle_model_vrp_z_1y": middle_rule["model_vrp_z_1y"],
        "middle_RSI14_max": middle_rule["RSI14_max"],
        "middle_rv21d_vol_pct_min": middle_rule["rv21d_vol_pct_min"],

        "back_model_vrp_log": back_locked["model_vrp_log"],
        "back_model_vrp_z": back_locked["model_vrp_z"],
        "back_model_vrp_z_3m": back_locked["model_vrp_z_3m"],
        "back_model_vrp_z_1y": back_locked["model_vrp_z_1y"],
        "back_RSI14_max": back_locked["RSI14_max"],
        "back_rv21d_vol_pct_min": back_locked["rv21d_vol_pct_min"],

        "back_locked": True,
        "middle_between_front_12_15_and_back": all([
            between_or_equal(middle_rule["model_vrp_log"], front1215_rule["model_vrp_log"], back_locked["model_vrp_log"]),
            between_or_equal(middle_rule["model_vrp_z"], front1215_rule["model_vrp_z"], back_locked["model_vrp_z"]),
            between_or_equal(middle_rule["RSI14_max"], front1215_rule["RSI14_max"], back_locked["RSI14_max"]),
            between_or_equal(middle_rule["rv21d_vol_pct_min"], front1215_rule["rv21d_vol_pct_min"], back_locked["rv21d_vol_pct_min"]),
        ]),
        "uses_rv21d_threshold_contract": True,
        "forecast_vol_pct_threshold_used": False,
        "design_only": True,
    })

    for tenor in ALL_TENORS:
        bucket = TENOR_BUCKET_MAP[int(tenor)]

        if tenor == 9:
            rule = front9_rule
            include_tenor = bool(spec["include_9d"])
            rule_scope = "front_9d"
            rule_type = "front_candidate_9d_specific"
        elif tenor in [12, 15]:
            rule = front1215_rule
            include_tenor = True
            rule_scope = "front_12_15"
            rule_type = "front_candidate_12_15"
        elif bucket == "Middle":
            rule = middle_rule
            include_tenor = True
            rule_scope = "middle"
            rule_type = "middle_constrained_between_front_and_back"
        elif bucket == "Back":
            rule = back_locked
            include_tenor = True
            rule_scope = "back"
            rule_type = "back_locked"
        else:
            raise ValueError(f"Unexpected tenor/bucket: tenor={tenor}, bucket={bucket}")

        long_rows.append({
            "candidate_id": spec["candidate_id"],
            "candidate_family": FRONT_CANDIDATE_FAMILY,
            "candidate_subfamily": spec["candidate_subfamily"],
            "candidate_description": spec["candidate_description"],
            "locked_model_spec": LOCKED_MODEL_SPEC,
            "tenor": int(tenor),
            "tenor_bucket": bucket,
            "tenor_bucket_order": BUCKET_ORDER_MAP[bucket],
            "rule_scope": rule_scope,
            "rule_type": rule_type,
            "include_tenor": bool(include_tenor),

            "model_vrp_log": rule["model_vrp_log"],
            "model_vrp_z": rule["model_vrp_z"],
            "model_vrp_z_3m": rule["model_vrp_z_3m"],
            "model_vrp_z_1y": rule["model_vrp_z_1y"],
            "RSI14_max": rule["RSI14_max"],
            "rv21d_vol_pct_min": rule["rv21d_vol_pct_min"],

            "is_front": bucket == "Front",
            "is_front_9d": tenor == 9,
            "is_front_12_15": tenor in [12, 15],
            "is_middle": bucket == "Middle",
            "is_back_locked": bucket == "Back",
            "design_only": True,
        })

candidate_grid_wide = pd.DataFrame(wide_rows)
candidate_grid_long = (
    pd.DataFrame(long_rows)
    .sort_values(["candidate_id", "tenor"])
    .reset_index(drop=True)
)

# ======================================================================================
# 5. Save outputs
# ======================================================================================

front_by_tenor_path = OUT_AUDIT_DIR / f"05R_front_rv21d_weakness_by_tenor_{CELL5R_TIMESTAMP}.csv"
front_by_year_path = OUT_AUDIT_DIR / f"05R_front_rv21d_weakness_by_year_{CELL5R_TIMESTAMP}.csv"
front_by_tenor_year_path = OUT_AUDIT_DIR / f"05R_front_rv21d_weakness_by_tenor_year_{CELL5R_TIMESTAMP}.csv"
front_state_diagnostics_path = OUT_AUDIT_DIR / f"05R_front_rv21d_weakness_state_diagnostics_{CELL5R_TIMESTAMP}.csv"
front_2026_detail_path = OUT_AUDIT_DIR / f"05R_front_rv21d_weakness_2026_trade_detail_{CELL5R_TIMESTAMP}.csv"
front_2026_worst_path = OUT_AUDIT_DIR / f"05R_front_rv21d_weakness_2026_worst_trades_{CELL5R_TIMESTAMP}.csv"

candidate_grid_wide_csv_path = OUT_AUDIT_DIR / f"05R_core_front_rv21d_candidate_grid_wide_{CELL5R_TIMESTAMP}.csv"
candidate_grid_wide_parquet_path = OUT_PROCESSED_DIR / f"05R_core_front_rv21d_candidate_grid_wide_{CELL5R_TIMESTAMP}.parquet"
candidate_grid_long_csv_path = OUT_AUDIT_DIR / f"05R_core_front_rv21d_candidate_grid_long_{CELL5R_TIMESTAMP}.csv"
candidate_grid_long_parquet_path = OUT_PROCESSED_DIR / f"05R_core_front_rv21d_candidate_grid_long_{CELL5R_TIMESTAMP}.parquet"
validation_path = OUT_AUDIT_DIR / f"05R_core_front_rv21d_candidate_grid_validation_{CELL5R_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"05R_core_front_rv21d_candidate_grid_manifest_{CELL5R_TIMESTAMP}.json"

front_by_tenor.to_csv(front_by_tenor_path, index=False)
front_by_year.to_csv(front_by_year_path, index=False)
front_by_tenor_year.to_csv(front_by_tenor_year_path, index=False)
front_state_diagnostics.to_csv(front_state_diagnostics_path, index=False)
front_2026_detail.to_csv(front_2026_detail_path, index=False)
front_2026_worst.to_csv(front_2026_worst_path, index=False)

candidate_grid_wide.to_csv(candidate_grid_wide_csv_path, index=False)
candidate_grid_wide.to_parquet(candidate_grid_wide_parquet_path, index=False)
candidate_grid_long.to_csv(candidate_grid_long_csv_path, index=False)
candidate_grid_long.to_parquet(candidate_grid_long_parquet_path, index=False)

# ======================================================================================
# 6. Validation
# ======================================================================================

validation_rows = []

required_base_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
]

missing_base_cols = [c for c in required_base_cols if c not in base.columns]

expected_candidate_count = 10
expected_long_rows = expected_candidate_count * len(ALL_TENORS)

old_forecast_threshold_cols = [
    c for c in list(candidate_grid_wide.columns) + list(candidate_grid_long.columns)
    if "forecast_vol_pct_min" in c
]

bad_z_equality = candidate_grid_long[
    candidate_grid_long["model_vrp_z_3m"].ne(candidate_grid_long["model_vrp_z_1y"])
]

bad_back_lock = candidate_grid_long[
    candidate_grid_long["is_back_locked"]
    & (
        candidate_grid_long["model_vrp_log"].ne(back_locked["model_vrp_log"])
        | candidate_grid_long["model_vrp_z"].ne(back_locked["model_vrp_z"])
        | candidate_grid_long["RSI14_max"].ne(back_locked["RSI14_max"])
        | candidate_grid_long["rv21d_vol_pct_min"].ne(back_locked["rv21d_vol_pct_min"])
    )
]

bad_middle_between = candidate_grid_wide[
    ~candidate_grid_wide["middle_between_front_12_15_and_back"].astype(bool)
]

unexpected_trade_cols = [
    c for c in [
        "trades",
        "total_pnl",
        "avg_pnl_per_day",
        "profit_factor",
        "win_rate",
        "pnl_per_day_drawdown",
    ]
    if c in candidate_grid_wide.columns or c in candidate_grid_long.columns
]

front9_exclusion_rows = candidate_grid_long[
    candidate_grid_long["candidate_id"].eq("core_front_rv21d_0008")
    & candidate_grid_long["tenor"].eq(9)
]

add_validation(
    validation_rows,
    "base_panel_loaded",
    "PASS" if len(base) > 0 else "FAIL",
    f"rows={len(base):,}; path={base_panel_path}",
)

add_validation(
    validation_rows,
    "cell4_selected_panel_loaded",
    "PASS" if len(selected) > 0 else "FAIL",
    f"rows={len(selected):,}; path={selected_panel_path}",
)

add_validation(
    validation_rows,
    "required_base_columns_available",
    "PASS" if not missing_base_cols else "FAIL",
    f"missing={missing_base_cols}",
)

add_validation(
    validation_rows,
    "current_front_diagnostics_created",
    "PASS" if len(front_by_tenor) > 0 and len(front_by_year) > 0 else "FAIL",
    f"front_by_tenor_rows={len(front_by_tenor):,}; front_by_year_rows={len(front_by_year):,}",
)

add_validation(
    validation_rows,
    "front_2026_detail_created",
    "PASS" if len(front_2026_detail) > 0 else "WARN",
    f"rows={len(front_2026_detail):,}",
)

add_validation(
    validation_rows,
    "candidate_design_count",
    "PASS" if len(candidate_grid_wide) == expected_candidate_count else "FAIL",
    f"candidate_count={len(candidate_grid_wide):,}; expected={expected_candidate_count:,}",
)

add_validation(
    validation_rows,
    "long_grid_all_tenors_per_candidate",
    "PASS" if len(candidate_grid_long) == expected_long_rows else "FAIL",
    f"long_rows={len(candidate_grid_long):,}; expected={expected_long_rows:,}",
)

add_validation(
    validation_rows,
    "uses_rv21d_threshold_contract",
    "PASS" if "rv21d_vol_pct_min" in candidate_grid_long.columns else "FAIL",
    "rule field is rv21d_vol_pct_min",
)

add_validation(
    validation_rows,
    "no_forecast_vol_threshold_columns",
    "PASS" if not old_forecast_threshold_cols else "FAIL",
    f"old_forecast_threshold_cols={old_forecast_threshold_cols}",
)

add_validation(
    validation_rows,
    "forecast_vol_pct_diagnostic_only",
    "PASS" if "forecast_vol_pct" in base.columns else "FAIL",
    "forecast_vol_pct retained only in source/diagnostic data, not candidate threshold grid",
)

add_validation(
    validation_rows,
    "z_3m_equals_z_1y_within_bucket_rules",
    "PASS" if bad_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_z_equality):,}",
)

add_validation(
    validation_rows,
    "back_locked_all_candidates",
    "PASS" if bad_back_lock.empty else "FAIL",
    f"bad_rows={len(bad_back_lock):,}",
)

add_validation(
    validation_rows,
    "middle_between_front_12_15_and_back",
    "PASS" if bad_middle_between.empty else "FAIL",
    f"bad_rows={len(bad_middle_between):,}",
)

add_validation(
    validation_rows,
    "front_ex_9d_candidate_present",
    "PASS" if len(front9_exclusion_rows) == 1 and bool(front9_exclusion_rows["include_tenor"].iloc[0]) is False else "FAIL",
    f"rows={front9_exclusion_rows[['candidate_id', 'tenor', 'include_tenor']].to_dict('records') if len(front9_exclusion_rows) else []}",
)

add_validation(
    validation_rows,
    "no_trade_metrics_in_candidate_design",
    "PASS" if not unexpected_trade_cols else "FAIL",
    f"unexpected_trade_cols={unexpected_trade_cols}",
)

add_validation(
    validation_rows,
    "no_candidate_sweep",
    "PASS",
    "Cell 5R creates diagnostics and candidate design only.",
)

add_validation(
    validation_rows,
    "no_final_lock",
    "PASS",
    "No final Front rule selected.",
)

add_validation(
    validation_rows,
    "no_secondary",
    "PASS",
    "Core only.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed or added.",
)

add_validation(
    validation_rows,
    "no_production_lock",
    "PASS",
    "No production threshold lock performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"].eq("FAIL")]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1_RV21D_repaired_v2.ipynb",
    "cell": "Cell 5R — Front RV21D weakness diagnostic and candidate design",
    "timestamp": CELL5R_TIMESTAMP,
    "base_panel_path": str(base_panel_path),
    "selected_panel_path": str(selected_panel_path),
    "rules_wide_path": str(rules_wide_path),
    "rules_long_path": str(rules_long_path),
    "cell4_by_bucket_path": str(cell4_by_bucket_path),
    "cell4_by_tenor_path": str(cell4_by_tenor_path),
    "cell4_by_year_path": str(cell4_by_year_path),
    "front_by_tenor_path": str(front_by_tenor_path),
    "front_by_year_path": str(front_by_year_path),
    "front_by_tenor_year_path": str(front_by_tenor_year_path),
    "front_state_diagnostics_path": str(front_state_diagnostics_path),
    "front_2026_detail_path": str(front_2026_detail_path),
    "front_2026_worst_path": str(front_2026_worst_path),
    "candidate_grid_wide_csv_path": str(candidate_grid_wide_csv_path),
    "candidate_grid_wide_parquet_path": str(candidate_grid_wide_parquet_path),
    "candidate_grid_long_csv_path": str(candidate_grid_long_csv_path),
    "candidate_grid_long_parquet_path": str(candidate_grid_long_parquet_path),
    "validation_path": str(validation_path),
    "candidate_family": FRONT_CANDIDATE_FAMILY,
    "candidate_count": int(len(candidate_grid_wide)),
    "threshold_contract": "rv21d_vol_pct > rv21d_vol_pct_min",
    "forecast_vol_pct_role": "model denominator / diagnostic only, not threshold floor",
    "front_current_rule": front_current,
    "middle_current_rule": middle_current,
    "back_locked_rule": back_locked,
    "candidate_ids": candidate_grid_wide["candidate_id"].tolist(),
    "hard_constraints": [
        "Back bucket parameters remain locked.",
        "3m and 1y z-score thresholds are equal within each bucket.",
        "Middle parameter levels are between/equal Front and Back.",
        "forecast_vol_pct is diagnostic only.",
        "No threshold sweep.",
        "No final lock.",
        "No Secondary.",
        "No sizing changes.",
        "No production lock.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 5R validation failed. Review validation output above.")

# ======================================================================================
# 7. Display review
# ======================================================================================

print("=" * 100)
print("Cell 5R — Front RV21D weakness diagnostic and candidate design")
print("=" * 100)
print(f"Base panel:                         {base_panel_path}")
print(f"Cell 4R selected panel:             {selected_panel_path}")
print(f"Reference candidate:                {REFERENCE_CANDIDATE_ID}")
print(f"Front all-qualified rows reviewed:  {len(front_all_qualified):,}")
print(f"Front primary rows reviewed:        {len(front_primary):,}")
print(f"Front 2026 primary rows reviewed:   {len(front_2026_detail):,}")
print(f"Candidate count:                    {len(candidate_grid_wide):,}")
print(f"Candidate long rows:                {len(candidate_grid_long):,}")
print("Threshold contract:                 rv21d_vol_pct > rv21d_vol_pct_min")
print("Back locked:                        True")
print("3m/1y z equal within bucket:         True")
print("Middle between Front/Back:          True")
print("No candidate sweep:                 True")
print("No final lock:                      True")
print("No Secondary:                       True")
print("No sizing changes:                  True")
print("No production lock:                 True")

print()
print("Validation:")
display(validation)

print()
print("Front diagnostics by tenor:")
display(front_by_tenor)

print()
print("Front diagnostics by year:")
display(front_by_year)

print()
print("Front diagnostics by tenor/year:")
display(front_by_tenor_year)

print()
print("Front state diagnostics:")
display(front_state_diagnostics)

print()
print("Front 2026 primary selected trade detail:")
display(front_2026_detail)

print()
print("Front 2026 worst primary selected trades:")
display(front_2026_worst)

print()
print("Front candidate grid — wide:")
display(candidate_grid_wide)

print()
print("Front candidate grid — long:")
display(candidate_grid_long)

print()
print("Saved outputs:")
for p in [
    front_by_tenor_path,
    front_by_year_path,
    front_by_tenor_year_path,
    front_state_diagnostics_path,
    front_2026_detail_path,
    front_2026_worst_path,
    candidate_grid_wide_csv_path,
    candidate_grid_wide_parquet_path,
    candidate_grid_long_csv_path,
    candidate_grid_long_parquet_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 5R PASSED")

In [ ]:
# ======================================================================================
# Cell 6R — Front RV21D candidate sweep
# ======================================================================================
#
# Objective:
#   Evaluate the 10 Front candidate designs created in Cell 5R.
#
# Correct volatility-floor contract:
#       rv21d_vol_pct > rv21d_vol_pct_min
#
# NOT:
#       forecast_vol_pct > forecast_vol_pct_min
#
# Inputs:
#   01R_unified_fds_signal_base_panel_with_rv21d_*.parquet
#   05R_core_front_rv21d_candidate_grid_wide_*.parquet
#   05R_core_front_rv21d_candidate_grid_long_*.parquet
#
# Selection universes:
#   1. all_qualified_trades
#   2. one_trade_per_bucket_per_date_best_rank
#   3. one_trade_per_date_best_rank
#
# Primary universe:
#   one_trade_per_bucket_per_date_best_rank
#
# Primary metric:
#   avg_pnl_per_day
#
# Hard constraints:
#   1. Back bucket parameters remain locked.
#   2. Within each bucket, 3m and 1y z-score thresholds are equal.
#   3. Middle parameter levels must be between/equal Front and Back.
#   4. forecast_vol_pct remains diagnostic only.
#   5. No final lock.
#   6. No Secondary.
#   7. No sizing changes.
#   8. No production lock.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 320)
pd.set_option("display.width", 520)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL6R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

if "ALL_TENORS" not in globals():
    ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

if "TENOR_BUCKET_MAP" not in globals():
    TENOR_BUCKET_MAP = {
        9: "Front",
        12: "Front",
        15: "Front",
        18: "Middle",
        21: "Middle",
        24: "Middle",
        27: "Back",
        30: "Back",
        33: "Back",
    }

if "BUCKET_ORDER_MAP" not in globals():
    BUCKET_ORDER_MAP = {
        "Front": 1,
        "Middle": 2,
        "Back": 3,
    }

BUCKET_CENTER_TENOR = {
    "Front": 12,
    "Middle": 21,
    "Back": 30,
}

SELECTION_UNIVERSES = [
    "all_qualified_trades",
    "one_trade_per_bucket_per_date_best_rank",
    "one_trade_per_date_best_rank",
]

PRIMARY_SELECTION_UNIVERSE = "one_trade_per_bucket_per_date_best_rank"
PRIMARY_RANKING_METRIC = "avg_pnl_per_day"
REFERENCE_CANDIDATE_ID = "core_front_rv21d_0001"

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def between_or_equal(x, a, b, tol=1e-12):
    lo = min(float(a), float(b)) - tol
    hi = max(float(a), float(b)) + tol
    return lo <= float(x) <= hi


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def add_selection_ranks(df):
    d = df.copy()

    if d.empty:
        return d

    d["bucket_center_tenor"] = d["tenor_bucket"].map(BUCKET_CENTER_TENOR)
    d["distance_to_bucket_center"] = (d["tenor"] - d["bucket_center_tenor"]).abs()

    bucket_group = ["candidate_id", "trade_date", "tenor_bucket"]

    d["rank_z_1y_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_bucket_date"] = d[
        [
            "rank_z_1y_in_bucket_date",
            "rank_z_3m_in_bucket_date",
            "rank_vrp_log_in_bucket_date",
        ]
    ].mean(axis=1)

    date_group = ["candidate_id", "trade_date"]

    d["rank_z_1y_in_date"] = (
        d.groupby(date_group)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_date"] = (
        d.groupby(date_group)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_date"] = (
        d.groupby(date_group)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_date"] = d[
        [
            "rank_z_1y_in_date",
            "rank_z_3m_in_date",
            "rank_vrp_log_in_date",
        ]
    ].mean(axis=1)

    return d


def select_trade_universes(qualified):
    if qualified.empty:
        return {
            "all_qualified_trades": qualified.copy(),
            "one_trade_per_bucket_per_date_best_rank": qualified.copy(),
            "one_trade_per_date_best_rank": qualified.copy(),
        }

    q = add_selection_ranks(qualified)

    one_trade_per_bucket_date = (
        q.sort_values(
            [
                "candidate_id",
                "trade_date",
                "tenor_bucket_order",
                "avg_signal_rank_in_bucket_date",
                "distance_to_bucket_center",
                "rank_z_1y_in_bucket_date",
                "rank_z_3m_in_bucket_date",
                "rank_vrp_log_in_bucket_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True, True],
        )
        .groupby(["candidate_id", "trade_date", "tenor_bucket"], as_index=False)
        .head(1)
        .copy()
    )

    one_trade_per_date = (
        q.sort_values(
            [
                "candidate_id",
                "trade_date",
                "avg_signal_rank_in_date",
                "rank_z_1y_in_date",
                "rank_z_3m_in_date",
                "rank_vrp_log_in_date",
                "distance_to_bucket_center",
                "tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True, True],
        )
        .groupby(["candidate_id", "trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return {
        "all_qualified_trades": q.copy(),
        "one_trade_per_bucket_per_date_best_rank": one_trade_per_bucket_date,
        "one_trade_per_date_best_rank": one_trade_per_date,
    }


def summarize_trade_set(df, group_cols):
    d = df.copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    numeric_cols = [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "forecast_vol_pct",
        "rv21d_vol_pct",
        "threshold_model_vrp_log",
        "threshold_model_vrp_z_3m",
        "threshold_model_vrp_z_1y",
        "threshold_RSI14_max",
        "threshold_rv21d_vol_pct_min",
    ]

    for c in numeric_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d = d.sort_values(["candidate_id", "selection_universe", "trade_date", "tenor"]).copy()

    rows = []

    grouped = d.groupby(group_cols, dropna=False, observed=False)

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {}

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": int(g["trade_date"].nunique()),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        for signal_col in [
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "threshold_model_vrp_log",
            "threshold_model_vrp_z_3m",
            "threshold_model_vrp_z_1y",
            "threshold_RSI14_max",
            "threshold_rv21d_vol_pct_min",
        ]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        rows.append(row)

    return pd.DataFrame(rows)


def attach_candidate_design(summary_df, candidate_grid_wide):
    if summary_df.empty:
        return summary_df

    return summary_df.merge(
        candidate_grid_wide,
        on="candidate_id",
        how="left",
        validate="many_to_one",
    )


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


# ======================================================================================
# 2. Load inputs
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01R_unified_fds_signal_base_panel_with_rv21d_*.parquet",
    required=True,
)

candidate_grid_wide_path = latest_file(
    OUT_PROCESSED_DIR,
    "05R_core_front_rv21d_candidate_grid_wide_*.parquet",
    required=True,
)

candidate_grid_long_path = latest_file(
    OUT_PROCESSED_DIR,
    "05R_core_front_rv21d_candidate_grid_long_*.parquet",
    required=True,
)

base = pd.read_parquet(base_panel_path)
candidate_grid_wide = pd.read_parquet(candidate_grid_wide_path)
candidate_grid_long = pd.read_parquet(candidate_grid_long_path)

base["trade_date"] = pd.to_datetime(base["trade_date"], errors="coerce").dt.normalize()

for c in [
    "tenor",
    "tenor_bucket_order",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")

for c in [
    "tenor",
    "tenor_bucket_order",
    "model_vrp_log",
    "model_vrp_z",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14_max",
    "rv21d_vol_pct_min",
]:
    if c in candidate_grid_long.columns:
        candidate_grid_long[c] = pd.to_numeric(candidate_grid_long[c], errors="coerce")

candidate_grid_long["include_tenor"] = candidate_grid_long["include_tenor"].astype(bool)

# ======================================================================================
# 3. Validate input schemas before applying rules
# ======================================================================================

required_base_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

required_grid_long_cols = [
    "candidate_id",
    "candidate_family",
    "candidate_subfamily",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "include_tenor",
    "model_vrp_log",
    "model_vrp_z",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14_max",
    "rv21d_vol_pct_min",
    "is_back_locked",
    "is_middle",
    "design_only",
]

required_grid_wide_cols = [
    "candidate_id",
    "candidate_family",
    "candidate_subfamily",
    "front_9d_include",
    "front_9d_model_vrp_log",
    "front_9d_model_vrp_z",
    "front_9d_RSI14_max",
    "front_9d_rv21d_vol_pct_min",
    "front_12_15_model_vrp_log",
    "front_12_15_model_vrp_z",
    "front_12_15_RSI14_max",
    "front_12_15_rv21d_vol_pct_min",
    "middle_model_vrp_log",
    "middle_model_vrp_z",
    "middle_RSI14_max",
    "middle_rv21d_vol_pct_min",
    "back_model_vrp_log",
    "back_model_vrp_z",
    "back_RSI14_max",
    "back_rv21d_vol_pct_min",
    "back_locked",
    "middle_between_front_12_15_and_back",
    "uses_rv21d_threshold_contract",
    "forecast_vol_pct_threshold_used",
    "design_only",
]

missing_base_cols = [c for c in required_base_cols if c not in base.columns]
missing_grid_long_cols = [c for c in required_grid_long_cols if c not in candidate_grid_long.columns]
missing_grid_wide_cols = [c for c in required_grid_wide_cols if c not in candidate_grid_wide.columns]

if missing_base_cols:
    raise KeyError(f"Cell 6R missing base columns: {missing_base_cols}")

if missing_grid_long_cols:
    raise KeyError(f"Cell 6R missing candidate_grid_long columns: {missing_grid_long_cols}")

if missing_grid_wide_cols:
    raise KeyError(f"Cell 6R missing candidate_grid_wide columns: {missing_grid_wide_cols}")

# ======================================================================================
# 4. Apply candidate rules
# ======================================================================================

threshold_cols = [
    "candidate_id",
    "candidate_family",
    "candidate_subfamily",
    "candidate_description",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "rule_scope",
    "rule_type",
    "include_tenor",
    "model_vrp_log",
    "model_vrp_z",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14_max",
    "rv21d_vol_pct_min",
    "is_front",
    "is_front_9d",
    "is_front_12_15",
    "is_middle",
    "is_back_locked",
]

thresholds = candidate_grid_long[threshold_cols].rename(columns={
    "model_vrp_log": "threshold_model_vrp_log",
    "model_vrp_z": "threshold_model_vrp_z",
    "model_vrp_z_3m": "threshold_model_vrp_z_3m",
    "model_vrp_z_1y": "threshold_model_vrp_z_1y",
    "RSI14_max": "threshold_RSI14_max",
    "rv21d_vol_pct_min": "threshold_rv21d_vol_pct_min",
})

panel = base.merge(
    thresholds,
    on=["tenor", "tenor_bucket", "tenor_bucket_order"],
    how="inner",
)

required_signal_available = (
    panel["include_tenor"].astype(bool)
    & panel["model_vrp_log"].notna()
    & panel["model_vrp_z_3m"].notna()
    & panel["model_vrp_z_1y"].notna()
    & panel["RSI14"].notna()
    & panel["rv21d_vol_pct"].notna()
    & panel["threshold_model_vrp_log"].notna()
    & panel["threshold_model_vrp_z_3m"].notna()
    & panel["threshold_model_vrp_z_1y"].notna()
    & panel["threshold_RSI14_max"].notna()
    & panel["threshold_rv21d_vol_pct_min"].notna()
)

outcome_available = (
    panel["normalized_pnl_dollars"].notna()
    & panel["normalized_pnl_per_day"].notna()
)

pass_mask = (
    required_signal_available
    & (panel["model_vrp_log"] > panel["threshold_model_vrp_log"])
    & (panel["model_vrp_z_3m"] > panel["threshold_model_vrp_z_3m"])
    & (panel["model_vrp_z_1y"] > panel["threshold_model_vrp_z_1y"])
    & (panel["RSI14"] < panel["threshold_RSI14_max"])
    & (panel["rv21d_vol_pct"] > panel["threshold_rv21d_vol_pct_min"])
)

panel["candidate_pass"] = pass_mask

qualified_all_dates = panel[pass_mask].copy()
qualified_complete = panel[pass_mask & outcome_available].copy()

universes = select_trade_universes(qualified_complete)

selected_trade_panel = pd.concat(
    [df.assign(selection_universe=name) for name, df in universes.items()],
    ignore_index=True,
)

# ======================================================================================
# 5. Summaries
# ======================================================================================

summary_frames = []
bucket_frames = []
tenor_frames = []
year_frames = []
tenor_year_frames = []

for universe_name, df_u in universes.items():
    df_u = df_u.assign(selection_universe=universe_name).copy()

    summary_frames.append(
        summarize_trade_set(
            df_u,
            group_cols=["candidate_id", "selection_universe"],
        )
    )

    bucket_frames.append(
        summarize_trade_set(
            df_u,
            group_cols=["candidate_id", "selection_universe", "tenor_bucket", "tenor_bucket_order"],
        )
    )

    tenor_frames.append(
        summarize_trade_set(
            df_u,
            group_cols=["candidate_id", "selection_universe", "tenor", "tenor_bucket", "tenor_bucket_order"],
        )
    )

    y = df_u.copy()
    y["year"] = y["trade_date"].dt.year.astype(int)

    year_frames.append(
        summarize_trade_set(
            y,
            group_cols=["candidate_id", "selection_universe", "year"],
        )
    )

    tenor_year_frames.append(
        summarize_trade_set(
            y,
            group_cols=["candidate_id", "selection_universe", "tenor", "tenor_bucket", "year"],
        )
    )

summary = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
by_bucket = pd.concat(bucket_frames, ignore_index=True) if bucket_frames else pd.DataFrame()
by_tenor = pd.concat(tenor_frames, ignore_index=True) if tenor_frames else pd.DataFrame()
by_year = pd.concat(year_frames, ignore_index=True) if year_frames else pd.DataFrame()
by_tenor_year = pd.concat(tenor_year_frames, ignore_index=True) if tenor_year_frames else pd.DataFrame()

summary = attach_candidate_design(summary, candidate_grid_wide)
by_bucket = attach_candidate_design(by_bucket, candidate_grid_wide)
by_tenor = attach_candidate_design(by_tenor, candidate_grid_wide)
by_year = attach_candidate_design(by_year, candidate_grid_wide)
by_tenor_year = attach_candidate_design(by_tenor_year, candidate_grid_wide)

summary = summary.sort_values(
    ["selection_universe", "avg_pnl_per_day"],
    ascending=[True, False],
).reset_index(drop=True)

if not by_bucket.empty:
    by_bucket = by_bucket.sort_values(
        ["selection_universe", "candidate_id", "tenor_bucket_order"],
    ).reset_index(drop=True)

if not by_tenor.empty:
    by_tenor = by_tenor.sort_values(
        ["selection_universe", "candidate_id", "tenor"],
    ).reset_index(drop=True)

if not by_year.empty:
    by_year = by_year.sort_values(
        ["selection_universe", "candidate_id", "year"],
    ).reset_index(drop=True)

if not by_tenor_year.empty:
    by_tenor_year = by_tenor_year.sort_values(
        ["selection_universe", "candidate_id", "tenor", "year"],
    ).reset_index(drop=True)

# ======================================================================================
# 6. Build critical front diagnostics into summary
# ======================================================================================

primary_summary = summary[
    summary["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

primary_bucket = by_bucket[
    by_bucket["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

primary_tenor = by_tenor[
    by_tenor["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

def extract_bucket_metric(bucket_name, metric, output_name):
    x = primary_bucket[primary_bucket["tenor_bucket"].eq(bucket_name)][
        ["candidate_id", metric]
    ].rename(columns={metric: output_name})
    return x

def extract_tenor_metric(tenor, metric, output_name):
    x = primary_tenor[primary_tenor["tenor"].eq(tenor)][
        ["candidate_id", metric]
    ].rename(columns={metric: output_name})
    return x

critical_summary = primary_summary.copy()

critical_merges = [
    extract_bucket_metric("Front", "trades", "front_trades"),
    extract_bucket_metric("Front", "avg_pnl_per_day", "front_avg_pnl_per_day"),
    extract_bucket_metric("Front", "avg_pnl_per_day_2025", "front_avg_pnl_per_day_2025"),
    extract_bucket_metric("Front", "avg_pnl_per_day_2026", "front_avg_pnl_per_day_2026"),
    extract_bucket_metric("Front", "profit_factor_pnl_per_day", "front_profit_factor_pnl_per_day"),
    extract_bucket_metric("Front", "pnl_per_day_drawdown", "front_pnl_per_day_drawdown"),
    extract_bucket_metric("Front", "worst_20_trade_pnl_per_day_sum", "front_worst_20_trade_pnl_per_day_sum"),

    extract_tenor_metric(9, "trades", "tenor_9_trades"),
    extract_tenor_metric(9, "avg_pnl_per_day", "tenor_9_avg_pnl_per_day"),
    extract_tenor_metric(9, "avg_pnl_per_day_2026", "tenor_9_avg_pnl_per_day_2026"),

    extract_tenor_metric(12, "trades", "tenor_12_trades"),
    extract_tenor_metric(12, "avg_pnl_per_day", "tenor_12_avg_pnl_per_day"),
    extract_tenor_metric(12, "avg_pnl_per_day_2026", "tenor_12_avg_pnl_per_day_2026"),

    extract_tenor_metric(15, "trades", "tenor_15_trades"),
    extract_tenor_metric(15, "avg_pnl_per_day", "tenor_15_avg_pnl_per_day"),
    extract_tenor_metric(15, "avg_pnl_per_day_2026", "tenor_15_avg_pnl_per_day_2026"),
]

for m in critical_merges:
    critical_summary = critical_summary.merge(
        m,
        on="candidate_id",
        how="left",
        validate="one_to_one",
    )

critical_summary = critical_summary.sort_values(
    "avg_pnl_per_day",
    ascending=False,
).reset_index(drop=True)

# Candidate deltas vs current reference.
reference_row = critical_summary[
    critical_summary["candidate_id"].eq(REFERENCE_CANDIDATE_ID)
].copy()

if reference_row.empty:
    candidate_comparison = critical_summary.copy()
else:
    ref = reference_row.iloc[0].to_dict()

    candidate_comparison = critical_summary.copy()

    delta_cols = [
        "trades",
        "total_pnl",
        "profit_factor",
        "avg_pnl_per_day",
        "avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",
        "front_trades",
        "front_avg_pnl_per_day",
        "front_avg_pnl_per_day_2025",
        "front_avg_pnl_per_day_2026",
        "front_profit_factor_pnl_per_day",
        "front_pnl_per_day_drawdown",
        "front_worst_20_trade_pnl_per_day_sum",
        "tenor_9_trades",
        "tenor_9_avg_pnl_per_day",
        "tenor_9_avg_pnl_per_day_2026",
        "tenor_12_trades",
        "tenor_12_avg_pnl_per_day",
        "tenor_12_avg_pnl_per_day_2026",
        "tenor_15_trades",
        "tenor_15_avg_pnl_per_day",
        "tenor_15_avg_pnl_per_day_2026",
    ]

    for c in delta_cols:
        if c in candidate_comparison.columns and c in ref:
            candidate_comparison[f"delta_{c}_vs_current"] = candidate_comparison[c] - ref[c]

    candidate_comparison = candidate_comparison.sort_values(
        "avg_pnl_per_day",
        ascending=False,
    ).reset_index(drop=True)

# Pass diagnostics.
pass_diagnostics_rows = []

candidate_ids = sorted(candidate_grid_wide["candidate_id"].unique().tolist())

for candidate_id in candidate_ids:
    p = panel[panel["candidate_id"].eq(candidate_id)].copy()
    q = qualified_complete[qualified_complete["candidate_id"].eq(candidate_id)].copy()

    row = {
        "candidate_id": candidate_id,
        "candidate_subfamily": candidate_grid_wide.loc[
            candidate_grid_wide["candidate_id"].eq(candidate_id),
            "candidate_subfamily",
        ].iloc[0],
        "panel_rows": int(len(p)),
        "pass_rows_all_dates": int(p["candidate_pass"].sum()),
        "pass_rows_with_outcomes": int(len(q)),
    }

    for bucket in ["Front", "Middle", "Back"]:
        row[f"{bucket}_pass_rows_with_outcomes"] = int(
            len(q[q["tenor_bucket"].eq(bucket)])
        )

    for tenor in ALL_TENORS:
        row[f"tenor_{tenor}_pass_rows_with_outcomes"] = int(
            len(q[q["tenor"].eq(tenor)])
        )

    pass_diagnostics_rows.append(row)

pass_diagnostics = pd.DataFrame(pass_diagnostics_rows)

# ======================================================================================
# 7. Save outputs
# ======================================================================================

summary_path = OUT_AUDIT_DIR / f"06R_front_rv21d_candidate_sweep_summary_{CELL6R_TIMESTAMP}.csv"
critical_summary_path = OUT_AUDIT_DIR / f"06R_front_rv21d_candidate_sweep_critical_summary_{CELL6R_TIMESTAMP}.csv"
candidate_comparison_path = OUT_AUDIT_DIR / f"06R_front_rv21d_candidate_sweep_vs_current_{CELL6R_TIMESTAMP}.csv"
by_bucket_path = OUT_AUDIT_DIR / f"06R_front_rv21d_candidate_sweep_by_bucket_{CELL6R_TIMESTAMP}.csv"
by_tenor_path = OUT_AUDIT_DIR / f"06R_front_rv21d_candidate_sweep_by_tenor_{CELL6R_TIMESTAMP}.csv"
by_year_path = OUT_AUDIT_DIR / f"06R_front_rv21d_candidate_sweep_by_year_{CELL6R_TIMESTAMP}.csv"
by_tenor_year_path = OUT_AUDIT_DIR / f"06R_front_rv21d_candidate_sweep_by_tenor_year_{CELL6R_TIMESTAMP}.csv"
pass_diagnostics_path = OUT_AUDIT_DIR / f"06R_front_rv21d_candidate_sweep_pass_diagnostics_{CELL6R_TIMESTAMP}.csv"
selected_trade_panel_path = OUT_PROCESSED_DIR / f"06R_front_rv21d_candidate_sweep_selected_trade_panel_{CELL6R_TIMESTAMP}.parquet"
selected_trade_panel_csv_path = OUT_AUDIT_DIR / f"06R_front_rv21d_candidate_sweep_selected_trade_panel_{CELL6R_TIMESTAMP}.csv"
validation_path = OUT_AUDIT_DIR / f"06R_front_rv21d_candidate_sweep_validation_{CELL6R_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"06R_front_rv21d_candidate_sweep_manifest_{CELL6R_TIMESTAMP}.json"

summary.to_csv(summary_path, index=False)
critical_summary.to_csv(critical_summary_path, index=False)
candidate_comparison.to_csv(candidate_comparison_path, index=False)
by_bucket.to_csv(by_bucket_path, index=False)
by_tenor.to_csv(by_tenor_path, index=False)
by_year.to_csv(by_year_path, index=False)
by_tenor_year.to_csv(by_tenor_year_path, index=False)
pass_diagnostics.to_csv(pass_diagnostics_path, index=False)
selected_trade_panel.to_parquet(selected_trade_panel_path, index=False)
selected_trade_panel.to_csv(selected_trade_panel_csv_path, index=False)

# ======================================================================================
# 8. Validation
# ======================================================================================

validation_rows = []

models_found = sorted(base["model_spec"].dropna().unique().tolist())
candidate_count = int(candidate_grid_wide["candidate_id"].nunique())
long_rows_expected = candidate_count * len(ALL_TENORS)

old_forecast_threshold_cols = [
    c for c in list(candidate_grid_wide.columns) + list(candidate_grid_long.columns)
    if "forecast_vol_pct_min" in c
]

bad_z_equality = candidate_grid_long[
    candidate_grid_long["model_vrp_z_3m"].ne(candidate_grid_long["model_vrp_z_1y"])
]

back_rules = candidate_grid_long[candidate_grid_long["is_back_locked"].astype(bool)].copy()

back_reference = (
    back_rules[
        [
            "model_vrp_log",
            "model_vrp_z",
            "RSI14_max",
            "rv21d_vol_pct_min",
        ]
    ]
    .drop_duplicates()
)

bad_back_locked = len(back_reference) != 1

bad_middle_between_records = []

for _, row in candidate_grid_wide.iterrows():
    cid = row["candidate_id"]

    checks = [
        (
            "model_vrp_log",
            row["front_12_15_model_vrp_log"],
            row["middle_model_vrp_log"],
            row["back_model_vrp_log"],
        ),
        (
            "model_vrp_z",
            row["front_12_15_model_vrp_z"],
            row["middle_model_vrp_z"],
            row["back_model_vrp_z"],
        ),
        (
            "RSI14_max",
            row["front_12_15_RSI14_max"],
            row["middle_RSI14_max"],
            row["back_RSI14_max"],
        ),
        (
            "rv21d_vol_pct_min",
            row["front_12_15_rv21d_vol_pct_min"],
            row["middle_rv21d_vol_pct_min"],
            row["back_rv21d_vol_pct_min"],
        ),
    ]

    for param, front_val, middle_val, back_val in checks:
        if not between_or_equal(middle_val, front_val, back_val):
            bad_middle_between_records.append({
                "candidate_id": cid,
                "parameter": param,
                "front_12_15": front_val,
                "middle": middle_val,
                "back": back_val,
            })

bad_middle_between = pd.DataFrame(bad_middle_between_records)

bad_design_only = candidate_grid_wide[
    ~candidate_grid_wide["design_only"].astype(bool)
]

bad_forecast_threshold_flag = candidate_grid_wide[
    candidate_grid_wide["forecast_vol_pct_threshold_used"].astype(bool)
]

bad_rv21d_contract_flag = candidate_grid_wide[
    ~candidate_grid_wide["uses_rv21d_threshold_contract"].astype(bool)
]

summary_universes_found = sorted(summary["selection_universe"].dropna().unique().tolist())
summary_candidates_found = sorted(summary["candidate_id"].dropna().unique().tolist())

expected_summary_rows = candidate_count * len(SELECTION_UNIVERSES)

bad_normalized_pnl = base[
    base["normalized_pnl_dollars"].notna()
    & base["normalized_pnl_per_day"].notna()
    & (
        (
            base["normalized_pnl_dollars"] / base["tenor"].replace(0, np.nan)
        )
        - base["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
]

add_validation(
    validation_rows,
    "base_panel_loaded",
    "PASS" if len(base) > 0 else "FAIL",
    f"rows={len(base):,}; path={base_panel_path}",
)

add_validation(
    validation_rows,
    "candidate_grid_wide_loaded",
    "PASS" if len(candidate_grid_wide) > 0 else "FAIL",
    f"rows={len(candidate_grid_wide):,}; path={candidate_grid_wide_path}",
)

add_validation(
    validation_rows,
    "candidate_grid_long_loaded",
    "PASS" if len(candidate_grid_long) > 0 else "FAIL",
    f"rows={len(candidate_grid_long):,}; path={candidate_grid_long_path}",
)

add_validation(
    validation_rows,
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    validation_rows,
    "candidate_count",
    "PASS" if candidate_count == 10 else "FAIL",
    f"candidate_count={candidate_count:,}; expected=10",
)

add_validation(
    validation_rows,
    "long_grid_all_tenors_per_candidate",
    "PASS" if len(candidate_grid_long) == long_rows_expected else "FAIL",
    f"long_rows={len(candidate_grid_long):,}; expected={long_rows_expected:,}",
)

add_validation(
    validation_rows,
    "panel_rule_application_created",
    "PASS" if len(panel) > 0 else "FAIL",
    f"panel_rows={len(panel):,}; qualified_complete_rows={len(qualified_complete):,}",
)

add_validation(
    validation_rows,
    "uses_rv21d_threshold_contract",
    "PASS" if "threshold_rv21d_vol_pct_min" in panel.columns and bad_rv21d_contract_flag.empty else "FAIL",
    f"bad_rv21d_contract_rows={len(bad_rv21d_contract_flag):,}",
)

add_validation(
    validation_rows,
    "no_forecast_vol_threshold_columns",
    "PASS" if not old_forecast_threshold_cols else "FAIL",
    f"old_forecast_threshold_cols={old_forecast_threshold_cols}",
)

add_validation(
    validation_rows,
    "forecast_vol_pct_diagnostic_only",
    "PASS" if bad_forecast_threshold_flag.empty else "FAIL",
    f"bad_forecast_threshold_flag_rows={len(bad_forecast_threshold_flag):,}",
)

add_validation(
    validation_rows,
    "z_3m_equals_z_1y_within_bucket_rules",
    "PASS" if bad_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_z_equality):,}",
)

add_validation(
    validation_rows,
    "back_locked_all_candidates",
    "PASS" if not bad_back_locked else "FAIL",
    f"distinct_back_rule_rows={len(back_reference):,}; back_reference={back_reference.to_dict('records')}",
)

add_validation(
    validation_rows,
    "middle_between_front_12_15_and_back",
    "PASS" if bad_middle_between.empty else "FAIL",
    f"bad_rows={len(bad_middle_between):,}; examples={bad_middle_between.head(10).to_dict('records') if not bad_middle_between.empty else []}",
)

add_validation(
    validation_rows,
    "candidate_grid_design_only",
    "PASS" if bad_design_only.empty else "FAIL",
    f"bad_rows={len(bad_design_only):,}",
)

add_validation(
    validation_rows,
    "normalized_pnl_per_day_formula",
    "PASS" if bad_normalized_pnl.empty else "FAIL",
    f"bad_rows={len(bad_normalized_pnl):,}",
)

add_validation(
    validation_rows,
    "all_selection_universes_evaluated",
    "PASS" if summary_universes_found == sorted(SELECTION_UNIVERSES) else "FAIL",
    f"found={summary_universes_found}",
)

add_validation(
    validation_rows,
    "summary_rows_complete",
    "PASS" if len(summary) == expected_summary_rows else "FAIL",
    f"summary_rows={len(summary):,}; expected={expected_summary_rows:,}",
)

add_validation(
    validation_rows,
    "all_candidates_evaluated",
    "PASS" if summary_candidates_found == sorted(candidate_grid_wide["candidate_id"].tolist()) else "FAIL",
    f"found_count={len(summary_candidates_found):,}; expected_count={candidate_count:,}",
)

add_validation(
    validation_rows,
    "critical_summary_created",
    "PASS" if len(critical_summary) == candidate_count else "FAIL",
    f"rows={len(critical_summary):,}; expected={candidate_count:,}",
)

add_validation(
    validation_rows,
    "candidate_comparison_created",
    "PASS" if len(candidate_comparison) == candidate_count else "FAIL",
    f"rows={len(candidate_comparison):,}; expected={candidate_count:,}; reference={REFERENCE_CANDIDATE_ID}",
)

add_validation(
    validation_rows,
    "selected_trade_panel_saved",
    "PASS" if len(selected_trade_panel) > 0 else "WARN",
    f"rows={len(selected_trade_panel):,}; path={selected_trade_panel_path}",
)

add_validation(
    validation_rows,
    "no_final_lock",
    "PASS",
    "Candidate sweep only; no final Front rule selected.",
)

add_validation(
    validation_rows,
    "no_secondary",
    "PASS",
    "Core only.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed or added.",
)

add_validation(
    validation_rows,
    "no_production_lock",
    "PASS",
    "No production threshold lock performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"].eq("FAIL")]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1_RV21D_repaired_v2.ipynb",
    "cell": "Cell 6R — Front RV21D candidate sweep",
    "timestamp": CELL6R_TIMESTAMP,
    "base_panel_path": str(base_panel_path),
    "candidate_grid_wide_path": str(candidate_grid_wide_path),
    "candidate_grid_long_path": str(candidate_grid_long_path),
    "candidate_count": int(candidate_count),
    "candidate_ids": candidate_grid_wide["candidate_id"].tolist(),
    "selection_universes": SELECTION_UNIVERSES,
    "primary_selection_universe": PRIMARY_SELECTION_UNIVERSE,
    "primary_ranking_metric": PRIMARY_RANKING_METRIC,
    "summary_path": str(summary_path),
    "critical_summary_path": str(critical_summary_path),
    "candidate_comparison_path": str(candidate_comparison_path),
    "by_bucket_path": str(by_bucket_path),
    "by_tenor_path": str(by_tenor_path),
    "by_year_path": str(by_year_path),
    "by_tenor_year_path": str(by_tenor_year_path),
    "pass_diagnostics_path": str(pass_diagnostics_path),
    "selected_trade_panel_path": str(selected_trade_panel_path),
    "selected_trade_panel_csv_path": str(selected_trade_panel_csv_path),
    "validation_path": str(validation_path),
    "threshold_contract": "rv21d_vol_pct > rv21d_vol_pct_min",
    "forecast_vol_pct_role": "model denominator / diagnostic only, not threshold floor",
    "hard_constraints": [
        "Back bucket parameters remain locked.",
        "3m and 1y z-score thresholds are equal within each bucket.",
        "Middle parameter levels are between/equal Front and Back.",
        "forecast_vol_pct is diagnostic only.",
        "No final lock.",
        "No Secondary.",
        "No sizing changes.",
        "No production lock.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 6R validation failed. Review validation output above.")

# ======================================================================================
# 9. Display review
# ======================================================================================

print("=" * 100)
print("Cell 6R — Front RV21D candidate sweep")
print("=" * 100)
print(f"Base panel:                         {base_panel_path}")
print(f"Candidate grid wide:                {candidate_grid_wide_path}")
print(f"Candidate grid long:                {candidate_grid_long_path}")
print(f"Candidate count:                    {candidate_count:,}")
print(f"Qualified complete rows:            {len(qualified_complete):,}")
print(f"Selected trade panel rows:          {len(selected_trade_panel):,}")
print(f"Primary universe:                   {PRIMARY_SELECTION_UNIVERSE}")
print(f"Primary metric:                     {PRIMARY_RANKING_METRIC}")
print("Threshold contract:                 rv21d_vol_pct > rv21d_vol_pct_min")
print("Back locked:                        True")
print("3m/1y z equal within bucket:         True")
print("Middle between Front/Back:          True")
print("No final lock:                      True")
print("No Secondary:                       True")
print("No sizing changes:                  True")
print("No production lock:                 True")

print()
print("Validation:")
display(validation)

critical_display_cols = [
    "candidate_id",
    "candidate_subfamily",
    "trades",
    "unique_trade_dates",
    "total_pnl",
    "profit_factor",
    "avg_pnl_per_day",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "front_trades",
    "front_avg_pnl_per_day",
    "front_avg_pnl_per_day_2025",
    "front_avg_pnl_per_day_2026",
    "front_profit_factor_pnl_per_day",
    "front_pnl_per_day_drawdown",
    "front_worst_20_trade_pnl_per_day_sum",
    "tenor_9_trades",
    "tenor_9_avg_pnl_per_day",
    "tenor_9_avg_pnl_per_day_2026",
    "tenor_12_trades",
    "tenor_12_avg_pnl_per_day",
    "tenor_12_avg_pnl_per_day_2026",
    "tenor_15_trades",
    "tenor_15_avg_pnl_per_day",
    "tenor_15_avg_pnl_per_day_2026",
    "front_9d_include",
    "front_9d_rv21d_vol_pct_min",
    "front_12_15_rv21d_vol_pct_min",
    "front_12_15_model_vrp_log",
    "front_12_15_model_vrp_z",
    "front_12_15_RSI14_max",
]

print()
print("Critical summary — primary universe sorted by avg P&L/day:")
display(critical_summary[[c for c in critical_display_cols if c in critical_summary.columns]])

comparison_display_cols = [
    "candidate_id",
    "candidate_subfamily",
    "avg_pnl_per_day",
    "delta_avg_pnl_per_day_vs_current",
    "total_pnl",
    "delta_total_pnl_vs_current",
    "profit_factor",
    "delta_profit_factor_vs_current",
    "avg_pnl_per_day_2026",
    "delta_avg_pnl_per_day_2026_vs_current",
    "pnl_per_day_drawdown",
    "delta_pnl_per_day_drawdown_vs_current",
    "worst_20_trade_pnl_per_day_sum",
    "delta_worst_20_trade_pnl_per_day_sum_vs_current",
    "front_avg_pnl_per_day",
    "delta_front_avg_pnl_per_day_vs_current",
    "front_avg_pnl_per_day_2026",
    "delta_front_avg_pnl_per_day_2026_vs_current",
    "tenor_9_avg_pnl_per_day",
    "delta_tenor_9_avg_pnl_per_day_vs_current",
    "tenor_9_avg_pnl_per_day_2026",
    "delta_tenor_9_avg_pnl_per_day_2026_vs_current",
    "tenor_12_avg_pnl_per_day",
    "delta_tenor_12_avg_pnl_per_day_vs_current",
    "tenor_12_avg_pnl_per_day_2026",
    "delta_tenor_12_avg_pnl_per_day_2026_vs_current",
    "tenor_15_avg_pnl_per_day",
    "delta_tenor_15_avg_pnl_per_day_vs_current",
    "tenor_15_avg_pnl_per_day_2026",
    "delta_tenor_15_avg_pnl_per_day_2026_vs_current",
]

print()
print("Candidate comparison vs current Front reference — primary universe:")
display(candidate_comparison[[c for c in comparison_display_cols if c in candidate_comparison.columns]])

print()
print("Pass diagnostics:")
display(pass_diagnostics)

print()
print("By-bucket detail:")
display(by_bucket)

print()
print("By-tenor detail:")
display(by_tenor)

print()
print("By-year detail:")
display(by_year)

print()
print("Saved outputs:")
for p in [
    summary_path,
    critical_summary_path,
    candidate_comparison_path,
    by_bucket_path,
    by_tenor_path,
    by_year_path,
    by_tenor_year_path,
    pass_diagnostics_path,
    selected_trade_panel_path,
    selected_trade_panel_csv_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 6R PASSED")

In [ ]:
# ======================================================================================
# Cell 7R — Confirm Front ex-9D candidate
# ======================================================================================
#
# Objective:
#   Confirm / attribute the improvement from the best Cell 6R Front candidate:
#
#       core_front_rv21d_0008 — front_ex_9d
#
#   Compare it directly against:
#
#       core_front_rv21d_0001 — current_front_reference
#
# Correct volatility-floor contract:
#       rv21d_vol_pct > rv21d_vol_pct_min
#
# NOT:
#       forecast_vol_pct > forecast_vol_pct_min
#
# What this cell does:
#   1. Compare current vs ex-9D across selection universes.
#   2. Compare current vs ex-9D by bucket, tenor, year, and 2026 stress window.
#   3. Identify dates where current selected 9D and ex-9D selected 12D/15D instead.
#   4. Identify dates where excluding 9D helped or hurt.
#   5. Confirm Back remains unchanged in the primary bucket/date universe.
#
# What this cell does NOT do:
#   1. No new candidates.
#   2. No final lock.
#   3. No Secondary.
#   4. No sizing changes.
#   5. No production lock.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 340)
pd.set_option("display.width", 560)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL7R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

CURRENT_CANDIDATE_ID = "core_front_rv21d_0001"
EX9D_CANDIDATE_ID = "core_front_rv21d_0008"

CANDIDATES_TO_COMPARE = [
    CURRENT_CANDIDATE_ID,
    EX9D_CANDIDATE_ID,
]

PRIMARY_SELECTION_UNIVERSE = "one_trade_per_bucket_per_date_best_rank"

SELECTION_UNIVERSES = [
    "all_qualified_trades",
    "one_trade_per_bucket_per_date_best_rank",
    "one_trade_per_date_best_rank",
]

STRESS_START = pd.Timestamp("2026-02-01")
STRESS_END = pd.Timestamp("2026-04-30")
STRESS_LABEL = "2026_front_stress_window_20260201_20260430"

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def summarize_trade_set(df, group_cols):
    d = df.copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    numeric_cols = [
        "tenor",
        "tenor_bucket_order",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "rv21d_vol_pct",
        "forecast_vol_pct",
        "threshold_model_vrp_log",
        "threshold_model_vrp_z_3m",
        "threshold_model_vrp_z_1y",
        "threshold_RSI14_max",
        "threshold_rv21d_vol_pct_min",
    ]

    for c in numeric_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d = d.sort_values(["candidate_id", "selection_universe", "trade_date", "tenor"]).copy()

    grouped = d.groupby(group_cols, dropna=False, observed=False)

    rows = []

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {}

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": int(g["trade_date"].nunique()),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        for signal_col in [
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "threshold_model_vrp_log",
            "threshold_model_vrp_z_3m",
            "threshold_model_vrp_z_1y",
            "threshold_RSI14_max",
            "threshold_rv21d_vol_pct_min",
        ]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        rows.append(row)

    return pd.DataFrame(rows)


def build_candidate_delta(left_current, right_ex9d, join_cols, label):
    """
    Build row-level or date-level deltas:
        ex9D - current

    Missing side is treated as zero P&L for delta attribution.
    """
    cur = left_current.copy()
    ex = right_ex9d.copy()

    cur_cols_keep = join_cols + [
        "candidate_id",
        "candidate_subfamily",
        "tenor",
        "tenor_bucket",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "rv21d_vol_pct",
        "forecast_vol_pct",
        "RSI14",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
    ]

    ex_cols_keep = cur_cols_keep.copy()

    cur_cols_keep = [c for c in cur_cols_keep if c in cur.columns]
    ex_cols_keep = [c for c in ex_cols_keep if c in ex.columns]

    cur = cur[cur_cols_keep].copy()
    ex = ex[ex_cols_keep].copy()

    cur = cur.rename(columns={
        c: f"current_{c}" for c in cur.columns if c not in join_cols
    })

    ex = ex.rename(columns={
        c: f"ex9d_{c}" for c in ex.columns if c not in join_cols
    })

    paired = cur.merge(
        ex,
        on=join_cols,
        how="outer",
        validate="one_to_one",
    )

    for c in [
        "current_normalized_pnl_dollars",
        "current_normalized_pnl_per_day",
        "ex9d_normalized_pnl_dollars",
        "ex9d_normalized_pnl_per_day",
    ]:
        if c not in paired.columns:
            paired[c] = 0.0
        paired[c] = pd.to_numeric(paired[c], errors="coerce").fillna(0.0)

    paired["delta_pnl_dollars_ex9d_minus_current"] = (
        paired["ex9d_normalized_pnl_dollars"]
        - paired["current_normalized_pnl_dollars"]
    )

    paired["delta_pnl_per_day_ex9d_minus_current"] = (
        paired["ex9d_normalized_pnl_per_day"]
        - paired["current_normalized_pnl_per_day"]
    )

    paired["delta_label"] = label

    return paired


def summarize_delta_set(df, group_cols):
    d = df.copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    for c in [
        "current_normalized_pnl_dollars",
        "current_normalized_pnl_per_day",
        "ex9d_normalized_pnl_dollars",
        "ex9d_normalized_pnl_per_day",
        "delta_pnl_dollars_ex9d_minus_current",
        "delta_pnl_per_day_ex9d_minus_current",
    ]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0)

    grouped = d.groupby(group_cols, dropna=False, observed=False)

    rows = []

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {}

        for col, key in zip(group_cols, keys):
            row[col] = key

        delta_day = g["delta_pnl_per_day_ex9d_minus_current"]
        delta_pnl = g["delta_pnl_dollars_ex9d_minus_current"]

        row.update({
            "rows": int(len(g)),
            "helped_rows": int((delta_day > 0).sum()),
            "hurt_rows": int((delta_day < 0).sum()),
            "flat_rows": int((delta_day == 0).sum()),
            "total_delta_pnl_dollars": float(delta_pnl.sum()),
            "total_delta_pnl_per_day": float(delta_day.sum()),
            "avg_delta_pnl_per_day": float(delta_day.mean()) if len(delta_day) else np.nan,
            "median_delta_pnl_per_day": float(delta_day.median()) if len(delta_day) else np.nan,
            "best_delta_pnl_per_day": float(delta_day.max()) if len(delta_day) else np.nan,
            "worst_delta_pnl_per_day": float(delta_day.min()) if len(delta_day) else np.nan,
            "delta_pnl_per_day_drawdown": max_drawdown_from_pnl(delta_day),
            "worst_5_delta_pnl_per_day_sum": worst_rolling_sum(delta_day, 5),
            "worst_10_delta_pnl_per_day_sum": worst_rolling_sum(delta_day, 10),
            "worst_20_delta_pnl_per_day_sum": worst_rolling_sum(delta_day, 20),
        })

        rows.append(row)

    return pd.DataFrame(rows)


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


# ======================================================================================
# 2. Load Cell 6R outputs
# ======================================================================================

selected_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "06R_front_rv21d_candidate_sweep_selected_trade_panel_*.parquet",
    required=True,
)

critical_summary_path = latest_file(
    OUT_AUDIT_DIR,
    "06R_front_rv21d_candidate_sweep_critical_summary_*.csv",
    required=True,
)

candidate_comparison_path = latest_file(
    OUT_AUDIT_DIR,
    "06R_front_rv21d_candidate_sweep_vs_current_*.csv",
    required=True,
)

by_bucket_source_path = latest_file(
    OUT_AUDIT_DIR,
    "06R_front_rv21d_candidate_sweep_by_bucket_*.csv",
    required=True,
)

by_tenor_source_path = latest_file(
    OUT_AUDIT_DIR,
    "06R_front_rv21d_candidate_sweep_by_tenor_*.csv",
    required=True,
)

by_year_source_path = latest_file(
    OUT_AUDIT_DIR,
    "06R_front_rv21d_candidate_sweep_by_year_*.csv",
    required=True,
)

by_tenor_year_source_path = latest_file(
    OUT_AUDIT_DIR,
    "06R_front_rv21d_candidate_sweep_by_tenor_year_*.csv",
    required=True,
)

selected = pd.read_parquet(selected_panel_path)
critical_summary_source = pd.read_csv(critical_summary_path)
candidate_comparison_source = pd.read_csv(candidate_comparison_path)
by_bucket_source = pd.read_csv(by_bucket_source_path)
by_tenor_source = pd.read_csv(by_tenor_source_path)
by_year_source = pd.read_csv(by_year_source_path)
by_tenor_year_source = pd.read_csv(by_tenor_year_source_path)

selected["trade_date"] = pd.to_datetime(selected["trade_date"], errors="coerce").dt.normalize()

for c in [
    "tenor",
    "tenor_bucket_order",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "RSI14",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "threshold_model_vrp_log",
    "threshold_model_vrp_z_3m",
    "threshold_model_vrp_z_1y",
    "threshold_RSI14_max",
    "threshold_rv21d_vol_pct_min",
]:
    if c in selected.columns:
        selected[c] = pd.to_numeric(selected[c], errors="coerce")

compare_selected = selected[
    selected["candidate_id"].isin(CANDIDATES_TO_COMPARE)
].copy()

current_selected = compare_selected[
    compare_selected["candidate_id"].eq(CURRENT_CANDIDATE_ID)
].copy()

ex9d_selected = compare_selected[
    compare_selected["candidate_id"].eq(EX9D_CANDIDATE_ID)
].copy()

# ======================================================================================
# 3. Rebuild clean comparison summaries from selected panel
# ======================================================================================

summary = summarize_trade_set(
    compare_selected,
    group_cols=["candidate_id", "candidate_subfamily", "selection_universe"],
)

by_bucket = summarize_trade_set(
    compare_selected,
    group_cols=[
        "candidate_id",
        "candidate_subfamily",
        "selection_universe",
        "tenor_bucket",
        "tenor_bucket_order",
    ],
)

by_tenor = summarize_trade_set(
    compare_selected,
    group_cols=[
        "candidate_id",
        "candidate_subfamily",
        "selection_universe",
        "tenor",
        "tenor_bucket",
        "tenor_bucket_order",
    ],
)

selected_year = compare_selected.copy()
selected_year["year"] = selected_year["trade_date"].dt.year.astype(int)

by_year = summarize_trade_set(
    selected_year,
    group_cols=["candidate_id", "candidate_subfamily", "selection_universe", "year"],
)

by_tenor_year = summarize_trade_set(
    selected_year,
    group_cols=[
        "candidate_id",
        "candidate_subfamily",
        "selection_universe",
        "tenor",
        "tenor_bucket",
        "year",
    ],
)

stress_selected = compare_selected[
    compare_selected["trade_date"].between(STRESS_START, STRESS_END)
].copy()

stress_selected["stress_window"] = STRESS_LABEL

stress_summary = summarize_trade_set(
    stress_selected,
    group_cols=["candidate_id", "candidate_subfamily", "selection_universe", "stress_window"],
)

stress_by_bucket = summarize_trade_set(
    stress_selected,
    group_cols=[
        "candidate_id",
        "candidate_subfamily",
        "selection_universe",
        "stress_window",
        "tenor_bucket",
        "tenor_bucket_order",
    ],
)

stress_by_tenor = summarize_trade_set(
    stress_selected,
    group_cols=[
        "candidate_id",
        "candidate_subfamily",
        "selection_universe",
        "stress_window",
        "tenor",
        "tenor_bucket",
        "tenor_bucket_order",
    ],
)

# ======================================================================================
# 4. Build date-level candidate deltas by selection universe
# ======================================================================================

date_delta_rows = []

for universe in SELECTION_UNIVERSES:
    cur_u = current_selected[
        current_selected["selection_universe"].eq(universe)
    ].copy()

    ex_u = ex9d_selected[
        ex9d_selected["selection_universe"].eq(universe)
    ].copy()

    cur_date = (
        cur_u.groupby("trade_date", as_index=False)
        .agg(
            current_normalized_pnl_dollars=("normalized_pnl_dollars", "sum"),
            current_normalized_pnl_per_day=("normalized_pnl_per_day", "sum"),
            current_trade_count=("tenor", "size"),
            current_front_trade_count=("tenor_bucket", lambda s: int((s == "Front").sum())),
            current_9d_trade_count=("tenor", lambda s: int((s == 9).sum())),
            current_min_tenor=("tenor", "min"),
            current_max_tenor=("tenor", "max"),
        )
    )

    ex_date = (
        ex_u.groupby("trade_date", as_index=False)
        .agg(
            ex9d_normalized_pnl_dollars=("normalized_pnl_dollars", "sum"),
            ex9d_normalized_pnl_per_day=("normalized_pnl_per_day", "sum"),
            ex9d_trade_count=("tenor", "size"),
            ex9d_front_trade_count=("tenor_bucket", lambda s: int((s == "Front").sum())),
            ex9d_9d_trade_count=("tenor", lambda s: int((s == 9).sum())),
            ex9d_min_tenor=("tenor", "min"),
            ex9d_max_tenor=("tenor", "max"),
        )
    )

    paired = cur_date.merge(
        ex_date,
        on="trade_date",
        how="outer",
        validate="one_to_one",
    )

    for c in [
        "current_normalized_pnl_dollars",
        "current_normalized_pnl_per_day",
        "current_trade_count",
        "current_front_trade_count",
        "current_9d_trade_count",
        "ex9d_normalized_pnl_dollars",
        "ex9d_normalized_pnl_per_day",
        "ex9d_trade_count",
        "ex9d_front_trade_count",
        "ex9d_9d_trade_count",
    ]:
        paired[c] = pd.to_numeric(paired[c], errors="coerce").fillna(0.0)

    paired["delta_pnl_dollars_ex9d_minus_current"] = (
        paired["ex9d_normalized_pnl_dollars"]
        - paired["current_normalized_pnl_dollars"]
    )

    paired["delta_pnl_per_day_ex9d_minus_current"] = (
        paired["ex9d_normalized_pnl_per_day"]
        - paired["current_normalized_pnl_per_day"]
    )

    paired["selection_universe"] = universe
    paired["stress_window_flag"] = paired["trade_date"].between(STRESS_START, STRESS_END)

    date_delta_rows.append(paired)

date_delta = pd.concat(date_delta_rows, ignore_index=True)

date_delta_summary = summarize_delta_set(
    date_delta,
    group_cols=["selection_universe"],
)

stress_date_delta_summary = summarize_delta_set(
    date_delta[date_delta["stress_window_flag"]],
    group_cols=["selection_universe"],
)

# ======================================================================================
# 5. Primary Front substitution attribution
# ======================================================================================

primary_front = compare_selected[
    compare_selected["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
    & compare_selected["tenor_bucket"].eq("Front")
].copy()

current_front = primary_front[
    primary_front["candidate_id"].eq(CURRENT_CANDIDATE_ID)
].copy()

ex9d_front = primary_front[
    primary_front["candidate_id"].eq(EX9D_CANDIDATE_ID)
].copy()

front_pair_cols = [
    "trade_date",
]

front_substitution = build_candidate_delta(
    current_front,
    ex9d_front,
    join_cols=front_pair_cols,
    label="primary_front_date_level_ex9d_minus_current",
)

front_substitution["current_has_front_trade"] = front_substitution["current_tenor"].notna()
front_substitution["ex9d_has_front_trade"] = front_substitution["ex9d_tenor"].notna()

front_substitution["current_selected_9d"] = front_substitution["current_tenor"].eq(9)
front_substitution["current_selected_12_15"] = front_substitution["current_tenor"].isin([12, 15])
front_substitution["ex9d_selected_9d"] = front_substitution["ex9d_tenor"].eq(9)
front_substitution["ex9d_selected_12_15"] = front_substitution["ex9d_tenor"].isin([12, 15])

conditions = [
    front_substitution["current_selected_9d"] & front_substitution["ex9d_selected_12_15"],
    front_substitution["current_selected_9d"] & (~front_substitution["ex9d_has_front_trade"]),
    front_substitution["current_selected_9d"] & front_substitution["ex9d_has_front_trade"] & (~front_substitution["ex9d_selected_12_15"]),
    front_substitution["current_selected_12_15"] & front_substitution["ex9d_selected_12_15"] & front_substitution["current_tenor"].eq(front_substitution["ex9d_tenor"]),
    front_substitution["current_selected_12_15"] & front_substitution["ex9d_selected_12_15"] & (~front_substitution["current_tenor"].eq(front_substitution["ex9d_tenor"])),
    (~front_substitution["current_has_front_trade"]) & front_substitution["ex9d_has_front_trade"],
]

choices = [
    "current_9d_to_ex9d_12_15",
    "current_9d_to_no_front_trade",
    "current_9d_to_other_front",
    "current_12_15_same_front_tenor",
    "current_12_15_different_front_tenor",
    "no_current_front_to_ex9d_front",
]

front_substitution["substitution_type"] = np.select(
    conditions,
    choices,
    default="other_or_no_change",
)

front_substitution["help_hurt_flat"] = np.select(
    [
        front_substitution["delta_pnl_per_day_ex9d_minus_current"] > 0,
        front_substitution["delta_pnl_per_day_ex9d_minus_current"] < 0,
    ],
    [
        "helped",
        "hurt",
    ],
    default="flat",
)

front_substitution["stress_window_flag"] = front_substitution["trade_date"].between(STRESS_START, STRESS_END)

front_substitution = front_substitution.sort_values(
    ["trade_date"]
).reset_index(drop=True)

front_substitution_summary = summarize_delta_set(
    front_substitution,
    group_cols=["substitution_type"],
)

front_substitution_stress_summary = summarize_delta_set(
    front_substitution[front_substitution["stress_window_flag"]],
    group_cols=["substitution_type"],
)

front_substitution_help_hurt = summarize_delta_set(
    front_substitution,
    group_cols=["help_hurt_flat"],
)

front_substitution_helped = (
    front_substitution[
        front_substitution["delta_pnl_per_day_ex9d_minus_current"] > 0
    ]
    .sort_values("delta_pnl_per_day_ex9d_minus_current", ascending=False)
    .reset_index(drop=True)
)

front_substitution_hurt = (
    front_substitution[
        front_substitution["delta_pnl_per_day_ex9d_minus_current"] < 0
    ]
    .sort_values("delta_pnl_per_day_ex9d_minus_current", ascending=True)
    .reset_index(drop=True)
)

front_current_9d_substitution = front_substitution[
    front_substitution["current_selected_9d"]
].copy()

front_current_9d_to_12_15 = front_substitution[
    front_substitution["current_selected_9d"]
    & front_substitution["ex9d_selected_12_15"]
].copy()

front_current_9d_to_no_trade = front_substitution[
    front_substitution["current_selected_9d"]
    & (~front_substitution["ex9d_has_front_trade"])
].copy()

# ======================================================================================
# 6. Confirm Back invariance in primary bucket/date universe
# ======================================================================================

primary_back = compare_selected[
    compare_selected["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
    & compare_selected["tenor_bucket"].eq("Back")
].copy()

current_back = primary_back[
    primary_back["candidate_id"].eq(CURRENT_CANDIDATE_ID)
].copy()

ex9d_back = primary_back[
    primary_back["candidate_id"].eq(EX9D_CANDIDATE_ID)
].copy()

back_compare_cols = [
    "trade_date",
    "tenor",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
]

cur_back = current_back[[c for c in back_compare_cols if c in current_back.columns]].copy()
ex_back = ex9d_back[[c for c in back_compare_cols if c in ex9d_back.columns]].copy()

cur_back = cur_back.rename(columns={c: f"current_{c}" for c in cur_back.columns if c != "trade_date"})
ex_back = ex_back.rename(columns={c: f"ex9d_{c}" for c in ex_back.columns if c != "trade_date"})

back_invariance = cur_back.merge(
    ex_back,
    on="trade_date",
    how="outer",
    validate="one_to_one",
)

for base_col in [
    "tenor",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
]:
    cur_col = f"current_{base_col}"
    ex_col = f"ex9d_{base_col}"

    if cur_col in back_invariance.columns and ex_col in back_invariance.columns:
        back_invariance[f"{base_col}_same"] = (
            pd.to_numeric(back_invariance[cur_col], errors="coerce")
            .fillna(-999999999.0)
            .eq(
                pd.to_numeric(back_invariance[ex_col], errors="coerce")
                .fillna(-999999999.0)
            )
        )

same_cols = [c for c in back_invariance.columns if c.endswith("_same")]

if same_cols:
    back_invariance["all_checked_fields_same"] = back_invariance[same_cols].all(axis=1)
else:
    back_invariance["all_checked_fields_same"] = False

# ======================================================================================
# 7. Save outputs
# ======================================================================================

summary_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_summary_{CELL7R_TIMESTAMP}.csv"
by_bucket_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_by_bucket_{CELL7R_TIMESTAMP}.csv"
by_tenor_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_by_tenor_{CELL7R_TIMESTAMP}.csv"
by_year_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_by_year_{CELL7R_TIMESTAMP}.csv"
by_tenor_year_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_by_tenor_year_{CELL7R_TIMESTAMP}.csv"
stress_summary_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_stress_summary_{CELL7R_TIMESTAMP}.csv"
stress_by_bucket_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_stress_by_bucket_{CELL7R_TIMESTAMP}.csv"
stress_by_tenor_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_stress_by_tenor_{CELL7R_TIMESTAMP}.csv"

date_delta_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_date_delta_{CELL7R_TIMESTAMP}.csv"
date_delta_summary_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_date_delta_summary_{CELL7R_TIMESTAMP}.csv"
stress_date_delta_summary_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_stress_date_delta_summary_{CELL7R_TIMESTAMP}.csv"

front_substitution_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_front_substitution_dates_{CELL7R_TIMESTAMP}.csv"
front_substitution_summary_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_front_substitution_summary_{CELL7R_TIMESTAMP}.csv"
front_substitution_stress_summary_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_front_substitution_stress_summary_{CELL7R_TIMESTAMP}.csv"
front_substitution_help_hurt_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_front_help_hurt_summary_{CELL7R_TIMESTAMP}.csv"
front_substitution_helped_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_front_helped_dates_{CELL7R_TIMESTAMP}.csv"
front_substitution_hurt_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_front_hurt_dates_{CELL7R_TIMESTAMP}.csv"
front_current_9d_substitution_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_current_9d_substitution_dates_{CELL7R_TIMESTAMP}.csv"
front_current_9d_to_12_15_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_current_9d_to_12_15_dates_{CELL7R_TIMESTAMP}.csv"
front_current_9d_to_no_trade_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_current_9d_to_no_front_trade_dates_{CELL7R_TIMESTAMP}.csv"

back_invariance_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_back_invariance_{CELL7R_TIMESTAMP}.csv"
validation_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_validation_{CELL7R_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"07R_front_ex9d_confirmation_manifest_{CELL7R_TIMESTAMP}.json"

summary.to_csv(summary_path, index=False)
by_bucket.to_csv(by_bucket_path, index=False)
by_tenor.to_csv(by_tenor_path, index=False)
by_year.to_csv(by_year_path, index=False)
by_tenor_year.to_csv(by_tenor_year_path, index=False)
stress_summary.to_csv(stress_summary_path, index=False)
stress_by_bucket.to_csv(stress_by_bucket_path, index=False)
stress_by_tenor.to_csv(stress_by_tenor_path, index=False)

date_delta.to_csv(date_delta_path, index=False)
date_delta_summary.to_csv(date_delta_summary_path, index=False)
stress_date_delta_summary.to_csv(stress_date_delta_summary_path, index=False)

front_substitution.to_csv(front_substitution_path, index=False)
front_substitution_summary.to_csv(front_substitution_summary_path, index=False)
front_substitution_stress_summary.to_csv(front_substitution_stress_summary_path, index=False)
front_substitution_help_hurt.to_csv(front_substitution_help_hurt_path, index=False)
front_substitution_helped.to_csv(front_substitution_helped_path, index=False)
front_substitution_hurt.to_csv(front_substitution_hurt_path, index=False)
front_current_9d_substitution.to_csv(front_current_9d_substitution_path, index=False)
front_current_9d_to_12_15.to_csv(front_current_9d_to_12_15_path, index=False)
front_current_9d_to_no_trade.to_csv(front_current_9d_to_no_trade_path, index=False)

back_invariance.to_csv(back_invariance_path, index=False)

# ======================================================================================
# 8. Validation
# ======================================================================================

validation_rows = []

candidates_found = sorted(compare_selected["candidate_id"].dropna().unique().tolist())
universes_found = sorted(compare_selected["selection_universe"].dropna().unique().tolist())

old_forecast_threshold_cols = [
    c for c in compare_selected.columns
    if "forecast_vol_pct_min" in c
]

current_primary_front_9d_count = int(
    current_front["tenor"].eq(9).sum()
)

ex9d_primary_front_9d_count = int(
    ex9d_front["tenor"].eq(9).sum()
)

current_primary_front_rows = int(len(current_front))
ex9d_primary_front_rows = int(len(ex9d_front))

back_bad_rows = int((~back_invariance["all_checked_fields_same"]).sum()) if len(back_invariance) else 0

current_9d_sub_count = int(len(front_current_9d_substitution))
current_9d_to_12_15_count = int(len(front_current_9d_to_12_15))
current_9d_to_no_trade_count = int(len(front_current_9d_to_no_trade))

add_validation(
    validation_rows,
    "cell6_selected_panel_loaded",
    "PASS" if len(selected) > 0 else "FAIL",
    f"rows={len(selected):,}; path={selected_panel_path}",
)

add_validation(
    validation_rows,
    "only_current_and_ex9d_compared",
    "PASS" if candidates_found == sorted(CANDIDATES_TO_COMPARE) else "FAIL",
    f"candidates_found={candidates_found}",
)

add_validation(
    validation_rows,
    "all_selection_universes_present",
    "PASS" if universes_found == sorted(SELECTION_UNIVERSES) else "FAIL",
    f"universes_found={universes_found}",
)

add_validation(
    validation_rows,
    "no_forecast_vol_threshold_columns",
    "PASS" if not old_forecast_threshold_cols else "FAIL",
    f"old_forecast_threshold_cols={old_forecast_threshold_cols}",
)

add_validation(
    validation_rows,
    "rv21d_threshold_column_present",
    "PASS" if "threshold_rv21d_vol_pct_min" in compare_selected.columns else "FAIL",
    "requires threshold_rv21d_vol_pct_min from repaired candidate sweep",
)

add_validation(
    validation_rows,
    "primary_current_has_9d_front_trades",
    "PASS" if current_primary_front_9d_count > 0 else "FAIL",
    f"current_primary_front_9d_count={current_primary_front_9d_count:,}",
)

add_validation(
    validation_rows,
    "primary_ex9d_has_no_9d_front_trades",
    "PASS" if ex9d_primary_front_9d_count == 0 else "FAIL",
    f"ex9d_primary_front_9d_count={ex9d_primary_front_9d_count:,}",
)

add_validation(
    validation_rows,
    "primary_front_rows_created",
    "PASS" if current_primary_front_rows > 0 and ex9d_primary_front_rows > 0 else "FAIL",
    f"current_primary_front_rows={current_primary_front_rows:,}; ex9d_primary_front_rows={ex9d_primary_front_rows:,}",
)

add_validation(
    validation_rows,
    "front_substitution_dates_created",
    "PASS" if len(front_substitution) > 0 else "FAIL",
    f"rows={len(front_substitution):,}",
)

add_validation(
    validation_rows,
    "current_9d_substitution_dates_identified",
    "PASS" if current_9d_sub_count > 0 else "FAIL",
    f"current_9d_substitution_rows={current_9d_sub_count:,}; to_12_15={current_9d_to_12_15_count:,}; to_no_front={current_9d_to_no_trade_count:,}",
)

add_validation(
    validation_rows,
    "date_delta_created",
    "PASS" if len(date_delta) > 0 else "FAIL",
    f"rows={len(date_delta):,}",
)

add_validation(
    validation_rows,
    "stress_window_created",
    "PASS" if len(stress_selected) > 0 else "FAIL",
    f"stress_rows={len(stress_selected):,}; start={STRESS_START.date()}; end={STRESS_END.date()}",
)

add_validation(
    validation_rows,
    "back_invariance_primary_bucket_universe",
    "PASS" if back_bad_rows == 0 else "FAIL",
    f"bad_rows={back_bad_rows:,}; checked_rows={len(back_invariance):,}",
)

add_validation(
    validation_rows,
    "no_new_candidates",
    "PASS",
    f"Only compared {CURRENT_CANDIDATE_ID} vs {EX9D_CANDIDATE_ID}.",
)

add_validation(
    validation_rows,
    "no_final_lock",
    "PASS",
    "Confirmation only; no final Front rule selected.",
)

add_validation(
    validation_rows,
    "no_secondary",
    "PASS",
    "Core only.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed or added.",
)

add_validation(
    validation_rows,
    "no_production_lock",
    "PASS",
    "No production threshold lock performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"].eq("FAIL")]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1_RV21D_repaired_v2.ipynb",
    "cell": "Cell 7R — Confirm Front ex-9D candidate",
    "timestamp": CELL7R_TIMESTAMP,
    "selected_panel_path": str(selected_panel_path),
    "critical_summary_path": str(critical_summary_path),
    "candidate_comparison_path": str(candidate_comparison_path),
    "by_bucket_source_path": str(by_bucket_source_path),
    "by_tenor_source_path": str(by_tenor_source_path),
    "by_year_source_path": str(by_year_source_path),
    "by_tenor_year_source_path": str(by_tenor_year_source_path),
    "current_candidate_id": CURRENT_CANDIDATE_ID,
    "ex9d_candidate_id": EX9D_CANDIDATE_ID,
    "primary_selection_universe": PRIMARY_SELECTION_UNIVERSE,
    "stress_start": str(STRESS_START.date()),
    "stress_end": str(STRESS_END.date()),
    "stress_label": STRESS_LABEL,
    "summary_path": str(summary_path),
    "by_bucket_path": str(by_bucket_path),
    "by_tenor_path": str(by_tenor_path),
    "by_year_path": str(by_year_path),
    "by_tenor_year_path": str(by_tenor_year_path),
    "stress_summary_path": str(stress_summary_path),
    "stress_by_bucket_path": str(stress_by_bucket_path),
    "stress_by_tenor_path": str(stress_by_tenor_path),
    "date_delta_path": str(date_delta_path),
    "date_delta_summary_path": str(date_delta_summary_path),
    "stress_date_delta_summary_path": str(stress_date_delta_summary_path),
    "front_substitution_path": str(front_substitution_path),
    "front_substitution_summary_path": str(front_substitution_summary_path),
    "front_substitution_stress_summary_path": str(front_substitution_stress_summary_path),
    "front_substitution_help_hurt_path": str(front_substitution_help_hurt_path),
    "front_substitution_helped_path": str(front_substitution_helped_path),
    "front_substitution_hurt_path": str(front_substitution_hurt_path),
    "front_current_9d_substitution_path": str(front_current_9d_substitution_path),
    "front_current_9d_to_12_15_path": str(front_current_9d_to_12_15_path),
    "front_current_9d_to_no_trade_path": str(front_current_9d_to_no_trade_path),
    "back_invariance_path": str(back_invariance_path),
    "validation_path": str(validation_path),
    "threshold_contract": "rv21d_vol_pct > rv21d_vol_pct_min",
    "forecast_vol_pct_role": "model denominator / diagnostic only, not threshold floor",
    "constraints": [
        "Back bucket parameters remain locked.",
        "3m and 1y z-score thresholds are equal within bucket rules.",
        "Middle parameter levels remain between/equal Front and Back.",
        "No new candidates.",
        "No final lock.",
        "No Secondary.",
        "No sizing changes.",
        "No production lock.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 7R validation failed. Review validation output above.")

# ======================================================================================
# 9. Display review
# ======================================================================================

print("=" * 100)
print("Cell 7R — Confirm Front ex-9D candidate")
print("=" * 100)
print(f"Selected panel:                     {selected_panel_path}")
print(f"Current candidate:                  {CURRENT_CANDIDATE_ID}")
print(f"Ex-9D candidate:                    {EX9D_CANDIDATE_ID}")
print(f"Primary universe:                   {PRIMARY_SELECTION_UNIVERSE}")
print(f"Stress window:                      {STRESS_START.date()} to {STRESS_END.date()}")
print(f"Current primary Front rows:         {current_primary_front_rows:,}")
print(f"Ex-9D primary Front rows:           {ex9d_primary_front_rows:,}")
print(f"Current primary Front 9D rows:      {current_primary_front_9d_count:,}")
print(f"Ex-9D primary Front 9D rows:        {ex9d_primary_front_9d_count:,}")
print(f"Current 9D substitution rows:       {current_9d_sub_count:,}")
print(f"Current 9D -> ex 12D/15D rows:      {current_9d_to_12_15_count:,}")
print(f"Current 9D -> no Front rows:        {current_9d_to_no_trade_count:,}")
print(f"Back invariance bad rows:           {back_bad_rows:,}")
print("Threshold contract:                 rv21d_vol_pct > rv21d_vol_pct_min")
print("No new candidates:                  True")
print("No final lock:                      True")
print("No Secondary:                       True")
print("No sizing changes:                  True")
print("No production lock:                 True")

print()
print("Validation:")
display(validation)

summary_display_cols = [
    "candidate_id",
    "candidate_subfamily",
    "selection_universe",
    "trades",
    "unique_trade_dates",
    "total_pnl",
    "profit_factor",
    "avg_pnl_per_day",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
]

print()
print("Current vs ex-9D summary:")
display(summary[[c for c in summary_display_cols if c in summary.columns]])

print()
print("Current vs ex-9D by bucket:")
display(by_bucket)

print()
print("Current vs ex-9D by tenor:")
display(by_tenor)

print()
print("Current vs ex-9D by year:")
display(by_year)

print()
print("Stress-window summary:")
display(stress_summary)

print()
print("Stress-window by bucket:")
display(stress_by_bucket)

print()
print("Stress-window by tenor:")
display(stress_by_tenor)

print()
print("Date-level delta summary by selection universe:")
display(date_delta_summary)

print()
print("Stress date-level delta summary by selection universe:")
display(stress_date_delta_summary)

print()
print("Primary Front substitution summary:")
display(front_substitution_summary)

print()
print("Primary Front stress substitution summary:")
display(front_substitution_stress_summary)

print()
print("Primary Front help/hurt summary:")
display(front_substitution_help_hurt)

print()
print("Dates where current selected 9D and ex-9D selected 12D/15D instead:")
display(front_current_9d_to_12_15)

print()
print("Dates where current selected 9D and ex-9D selected no Front trade:")
display(front_current_9d_to_no_trade)

print()
print("Top Front dates helped by ex-9D:")
display(front_substitution_helped.head(30))

print()
print("Top Front dates hurt by ex-9D:")
display(front_substitution_hurt.head(30))

print()
print("Back invariance check:")
display(back_invariance)

print()
print("Saved outputs:")
for p in [
    summary_path,
    by_bucket_path,
    by_tenor_path,
    by_year_path,
    by_tenor_year_path,
    stress_summary_path,
    stress_by_bucket_path,
    stress_by_tenor_path,
    date_delta_path,
    date_delta_summary_path,
    stress_date_delta_summary_path,
    front_substitution_path,
    front_substitution_summary_path,
    front_substitution_stress_summary_path,
    front_substitution_help_hurt_path,
    front_substitution_helped_path,
    front_substitution_hurt_path,
    front_current_9d_substitution_path,
    front_current_9d_to_12_15_path,
    front_current_9d_to_no_trade_path,
    back_invariance_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 7R PASSED")

In [ ]:
# ======================================================================================
# Cell 8R — Middle RV21D weakness diagnostic and candidate design
# ======================================================================================
#
# Objective:
#   Diagnose Middle weakness under the repaired RV21D threshold contract while carrying
#   forward the leading Front candidate:
#
#       Front active candidate = core_front_rv21d_0008 — front_ex_9d
#
# Correct volatility-floor contract:
#       rv21d_vol_pct > rv21d_vol_pct_min
#
# NOT:
#       forecast_vol_pct > forecast_vol_pct_min
#
# Hard constraints:
#   1. Front active candidate is ex-9D.
#   2. Back bucket parameters remain locked.
#   3. Within each bucket, 3m and 1y z-score thresholds are equal.
#   4. Middle parameter levels must be between/equal active Front and locked Back.
#   5. forecast_vol_pct remains diagnostic only.
#   6. No threshold sweep in this cell.
#   7. No final lock.
#   8. No Secondary.
#   9. No sizing changes.
#   10. No production lock.
#
# Constraint implication:
#   Active Front 12D/15D:
#       model_vrp_log = 0.60
#       model_vrp_z   = 0.65
#       RSI14_max     = 70
#       rv21d floor   = 8.5
#
#   Locked Back:
#       model_vrp_log = 0.70
#       model_vrp_z   = 0.70
#       RSI14_max     = 70
#       rv21d floor   = 8.5
#
#   Therefore valid Middle ranges are:
#       model_vrp_log:        0.60 to 0.70
#       model_vrp_z:          0.65 to 0.70
#       RSI14_max:            exactly 70
#       rv21d_vol_pct_min:    exactly 8.5
#
# Valid design-only candidates:
#   0001 current Middle reference
#   0002 bucket-wide Middle model_vrp_log to 0.70
#   0003 exclude 18D
#   0004 18D-only model_vrp_log to 0.70
#
# This cell does NOT evaluate the new candidates.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 340)
pd.set_option("display.width", 560)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL8R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

if "ALL_TENORS" not in globals():
    ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

if "TENOR_BUCKET_MAP" not in globals():
    TENOR_BUCKET_MAP = {
        9: "Front",
        12: "Front",
        15: "Front",
        18: "Middle",
        21: "Middle",
        24: "Middle",
        27: "Back",
        30: "Back",
        33: "Back",
    }

if "BUCKET_ORDER_MAP" not in globals():
    BUCKET_ORDER_MAP = {
        "Front": 1,
        "Middle": 2,
        "Back": 3,
    }

ACTIVE_FRONT_CANDIDATE_ID = "core_front_rv21d_0008"
ACTIVE_FRONT_LABEL = "front_ex_9d"

MIDDLE_CANDIDATE_FAMILY = "core_middle_rv21d_candidate_design"

PRIMARY_SELECTION_UNIVERSE = "one_trade_per_bucket_per_date_best_rank"
ALL_QUALIFIED_UNIVERSE = "all_qualified_trades"

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def rule_from_row(row):
    return {
        "model_vrp_log": float(row["model_vrp_log"]),
        "model_vrp_z": float(row["model_vrp_z"]),
        "model_vrp_z_3m": float(row["model_vrp_z"]),
        "model_vrp_z_1y": float(row["model_vrp_z"]),
        "RSI14_max": float(row["RSI14_max"]),
        "rv21d_vol_pct_min": float(row["rv21d_vol_pct_min"]),
    }


def clone_rule(rule):
    return {
        "model_vrp_log": float(rule["model_vrp_log"]),
        "model_vrp_z": float(rule["model_vrp_z"]),
        "model_vrp_z_3m": float(rule["model_vrp_z"]),
        "model_vrp_z_1y": float(rule["model_vrp_z"]),
        "RSI14_max": float(rule["RSI14_max"]),
        "rv21d_vol_pct_min": float(rule["rv21d_vol_pct_min"]),
    }


def update_rule(rule, **kwargs):
    out = clone_rule(rule)

    for k, v in kwargs.items():
        if k in ["model_vrp_z", "model_vrp_z_3m", "model_vrp_z_1y"]:
            out["model_vrp_z"] = float(v)
            out["model_vrp_z_3m"] = float(v)
            out["model_vrp_z_1y"] = float(v)
        elif k in out:
            out[k] = float(v)
        else:
            raise KeyError(f"Unexpected rule key: {k}")

    out["model_vrp_z_3m"] = float(out["model_vrp_z"])
    out["model_vrp_z_1y"] = float(out["model_vrp_z"])

    return out


def between_or_equal(x, a, b, tol=1e-12):
    lo = min(float(a), float(b)) - tol
    hi = max(float(a), float(b)) + tol
    return lo <= float(x) <= hi


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def summarize_trades(df, group_cols, diagnostic_label):
    d = df.copy()

    if d.empty:
        return pd.DataFrame()

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    numeric_cols = [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "rv21d_vol_pct",
        "forecast_vol_pct",
        "RSI14",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "middle_min_z",
    ]

    for c in numeric_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame()

    d = d.sort_values(["trade_date", "tenor"]).copy()

    if len(group_cols) == 0:
        grouped = [((), d)]
    else:
        grouped = d.groupby(group_cols, dropna=False, observed=False)

    rows = []

    for keys, g in grouped:
        if len(group_cols) == 0:
            keys = ()
        elif not isinstance(keys, tuple):
            keys = (keys,)

        row = {
            "diagnostic_label": diagnostic_label,
        }

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": int(g["trade_date"].nunique()),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        for signal_col in [
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "middle_min_z",
        ]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        rows.append(row)

    return pd.DataFrame(rows)


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


# ======================================================================================
# 2. Load active selected panel and prior candidate grid
# ======================================================================================

selected_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "06R_front_rv21d_candidate_sweep_selected_trade_panel_*.parquet",
    required=True,
)

front_candidate_grid_wide_path = latest_file(
    OUT_PROCESSED_DIR,
    "05R_core_front_rv21d_candidate_grid_wide_*.parquet",
    required=True,
)

front_candidate_grid_long_path = latest_file(
    OUT_PROCESSED_DIR,
    "05R_core_front_rv21d_candidate_grid_long_*.parquet",
    required=True,
)

selected = pd.read_parquet(selected_panel_path)
front_grid_wide = pd.read_parquet(front_candidate_grid_wide_path)
front_grid_long = pd.read_parquet(front_candidate_grid_long_path)

selected["trade_date"] = pd.to_datetime(selected["trade_date"], errors="coerce").dt.normalize()

for c in [
    "tenor",
    "tenor_bucket_order",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    if c in selected.columns:
        selected[c] = pd.to_numeric(selected[c], errors="coerce")

for c in [
    "tenor",
    "tenor_bucket_order",
    "model_vrp_log",
    "model_vrp_z",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14_max",
    "rv21d_vol_pct_min",
]:
    if c in front_grid_long.columns:
        front_grid_long[c] = pd.to_numeric(front_grid_long[c], errors="coerce")

front_grid_long["include_tenor"] = front_grid_long["include_tenor"].astype(bool)

active_selected = selected[
    selected["candidate_id"].eq(ACTIVE_FRONT_CANDIDATE_ID)
].copy()

middle_all_qualified = active_selected[
    active_selected["selection_universe"].eq(ALL_QUALIFIED_UNIVERSE)
    & active_selected["tenor_bucket"].eq("Middle")
].copy()

middle_primary = active_selected[
    active_selected["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
    & active_selected["tenor_bucket"].eq("Middle")
].copy()

for df in [middle_all_qualified, middle_primary]:
    df["middle_min_z"] = df[["model_vrp_z_3m", "model_vrp_z_1y"]].min(axis=1)

# ======================================================================================
# 3. Middle diagnostics under active Front ex-9D candidate
# ======================================================================================

middle_by_tenor = pd.concat(
    [
        summarize_trades(
            middle_all_qualified,
            group_cols=["selection_universe", "tenor", "tenor_bucket", "tenor_bucket_order"],
            diagnostic_label="middle_all_qualified_by_tenor",
        ),
        summarize_trades(
            middle_primary,
            group_cols=["selection_universe", "tenor", "tenor_bucket", "tenor_bucket_order"],
            diagnostic_label="middle_primary_by_tenor",
        ),
    ],
    ignore_index=True,
)

middle_all_year = middle_all_qualified.copy()
middle_all_year["year"] = middle_all_year["trade_date"].dt.year.astype(int)

middle_primary_year = middle_primary.copy()
middle_primary_year["year"] = middle_primary_year["trade_date"].dt.year.astype(int)

middle_by_year = pd.concat(
    [
        summarize_trades(
            middle_all_year,
            group_cols=["selection_universe", "year"],
            diagnostic_label="middle_all_qualified_by_year",
        ),
        summarize_trades(
            middle_primary_year,
            group_cols=["selection_universe", "year"],
            diagnostic_label="middle_primary_by_year",
        ),
    ],
    ignore_index=True,
)

middle_by_tenor_year = pd.concat(
    [
        summarize_trades(
            middle_all_year,
            group_cols=["selection_universe", "tenor", "year"],
            diagnostic_label="middle_all_qualified_by_tenor_year",
        ),
        summarize_trades(
            middle_primary_year,
            group_cols=["selection_universe", "tenor", "year"],
            diagnostic_label="middle_primary_by_tenor_year",
        ),
    ],
    ignore_index=True,
)

middle_state = pd.concat(
    [
        middle_all_qualified.assign(state_source="all_qualified"),
        middle_primary.assign(state_source="primary_selected"),
    ],
    ignore_index=True,
)

middle_state["middle_min_z"] = middle_state[["model_vrp_z_3m", "model_vrp_z_1y"]].min(axis=1)

middle_state["rv21d_state"] = pd.cut(
    middle_state["rv21d_vol_pct"],
    bins=[-np.inf, 8.5, 10.0, 12.0, 14.5, 16.0, 18.0, 20.0, np.inf],
    labels=["<=8.5", "8.5-10", "10-12", "12-14.5", "14.5-16", "16-18", "18-20", ">20"],
    include_lowest=True,
)

middle_state["RSI14_state"] = pd.cut(
    middle_state["RSI14"],
    bins=[-np.inf, 55.0, 60.0, 64.0, 66.0, 68.0, 70.0, np.inf],
    labels=["<=55", "55-60", "60-64", "64-66", "66-68", "68-70", ">70"],
    include_lowest=True,
)

middle_state["model_vrp_log_state"] = pd.cut(
    middle_state["model_vrp_log"],
    bins=[-np.inf, 0.65, 0.70, 0.75, 0.80, 0.90, np.inf],
    labels=["<=0.65", "0.65-0.70", "0.70-0.75", "0.75-0.80", "0.80-0.90", ">0.90"],
    include_lowest=True,
)

middle_state["middle_min_z_state"] = pd.cut(
    middle_state["middle_min_z"],
    bins=[-np.inf, 0.70, 0.80, 1.00, 1.25, 1.50, np.inf],
    labels=["<=0.70", "0.70-0.80", "0.80-1.00", "1.00-1.25", "1.25-1.50", ">1.50"],
    include_lowest=True,
)

middle_state_diagnostics = pd.concat(
    [
        summarize_trades(
            middle_state,
            group_cols=["state_source", "rv21d_state"],
            diagnostic_label="middle_by_rv21d_state",
        ),
        summarize_trades(
            middle_state,
            group_cols=["state_source", "RSI14_state"],
            diagnostic_label="middle_by_RSI14_state",
        ),
        summarize_trades(
            middle_state,
            group_cols=["state_source", "model_vrp_log_state"],
            diagnostic_label="middle_by_model_vrp_log_state",
        ),
        summarize_trades(
            middle_state,
            group_cols=["state_source", "middle_min_z_state"],
            diagnostic_label="middle_by_min_z_state",
        ),
    ],
    ignore_index=True,
)

middle_2026_detail = middle_primary[
    middle_primary["trade_date"].dt.year.eq(2026)
].copy()

middle_2026_detail = middle_2026_detail[
    [
        c for c in [
            "candidate_id",
            "selection_universe",
            "trade_date",
            "tenor",
            "tenor_bucket",
            "normalized_pnl_dollars",
            "normalized_pnl_per_day",
            "win",
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "middle_min_z",
            "implied_vol_pct",
            "model_vrp_ratio",
        ]
        if c in middle_2026_detail.columns
    ]
].sort_values(["trade_date", "tenor"]).reset_index(drop=True)

middle_2026_worst = (
    middle_2026_detail
    .sort_values("normalized_pnl_per_day")
    .head(30)
    .reset_index(drop=True)
)

# ======================================================================================
# 4. Build constrained Middle candidate design
# ======================================================================================

active_long = front_grid_long[
    front_grid_long["candidate_id"].eq(ACTIVE_FRONT_CANDIDATE_ID)
].copy()

if active_long.empty:
    raise RuntimeError(f"Could not find active Front candidate {ACTIVE_FRONT_CANDIDATE_ID} in Cell 5R grid.")

active_front9_row = active_long[active_long["tenor"].eq(9)].iloc[0]
active_front12_row = active_long[active_long["tenor"].eq(12)].iloc[0]
active_middle18_row = active_long[active_long["tenor"].eq(18)].iloc[0]
active_middle21_row = active_long[active_long["tenor"].eq(21)].iloc[0]
active_back30_row = active_long[active_long["tenor"].eq(30)].iloc[0]

front9_rule = rule_from_row(active_front9_row)
front1215_rule = rule_from_row(active_front12_row)
middle_current_rule = rule_from_row(active_middle21_row)
back_locked_rule = rule_from_row(active_back30_row)

def valid_middle_rule(rule):
    checks = [
        between_or_equal(rule["model_vrp_log"], front1215_rule["model_vrp_log"], back_locked_rule["model_vrp_log"]),
        between_or_equal(rule["model_vrp_z"], front1215_rule["model_vrp_z"], back_locked_rule["model_vrp_z"]),
        between_or_equal(rule["RSI14_max"], front1215_rule["RSI14_max"], back_locked_rule["RSI14_max"]),
        between_or_equal(rule["rv21d_vol_pct_min"], front1215_rule["rv21d_vol_pct_min"], back_locked_rule["rv21d_vol_pct_min"]),
    ]
    return bool(all(checks))

candidate_specs = [
    {
        "candidate_id": "core_middle_rv21d_0001",
        "candidate_subfamily": "current_middle_reference",
        "candidate_description": "Current Middle reference carried forward with active Front ex-9D and locked Back.",
        "middle18_rule": clone_rule(middle_current_rule),
        "middle2124_rule": clone_rule(middle_current_rule),
        "include_18d": True,
    },
    {
        "candidate_id": "core_middle_rv21d_0002",
        "candidate_subfamily": "middle_log_0_70",
        "candidate_description": "Tighten bucket-wide Middle model_vrp_log threshold to 0.70; remains within Front/Back range.",
        "middle18_rule": update_rule(middle_current_rule, model_vrp_log=0.70),
        "middle2124_rule": update_rule(middle_current_rule, model_vrp_log=0.70),
        "include_18d": True,
    },
    {
        "candidate_id": "core_middle_rv21d_0003",
        "candidate_subfamily": "middle_ex_18d",
        "candidate_description": "Exclude 18D from Middle; keep 21D/24D Middle reference unchanged.",
        "middle18_rule": clone_rule(middle_current_rule),
        "middle2124_rule": clone_rule(middle_current_rule),
        "include_18d": False,
    },
    {
        "candidate_id": "core_middle_rv21d_0004",
        "candidate_subfamily": "middle_18d_log_0_70",
        "candidate_description": "Tighten only 18D model_vrp_log threshold to 0.70; keep 21D/24D Middle reference unchanged.",
        "middle18_rule": update_rule(middle_current_rule, model_vrp_log=0.70),
        "middle2124_rule": clone_rule(middle_current_rule),
        "include_18d": True,
    },
]

wide_rows = []
long_rows = []

for spec in candidate_specs:
    middle18_rule = clone_rule(spec["middle18_rule"])
    middle2124_rule = clone_rule(spec["middle2124_rule"])

    if not valid_middle_rule(middle18_rule):
        raise RuntimeError(f"Invalid Middle 18D rule for {spec['candidate_id']}: outside Front/Back bounds.")

    if not valid_middle_rule(middle2124_rule):
        raise RuntimeError(f"Invalid Middle 21D/24D rule for {spec['candidate_id']}: outside Front/Back bounds.")

    wide_rows.append({
        "candidate_id": spec["candidate_id"],
        "candidate_family": MIDDLE_CANDIDATE_FAMILY,
        "candidate_subfamily": spec["candidate_subfamily"],
        "candidate_description": spec["candidate_description"],
        "locked_model_spec": LOCKED_MODEL_SPEC,

        "active_front_candidate_id": ACTIVE_FRONT_CANDIDATE_ID,
        "active_front_label": ACTIVE_FRONT_LABEL,

        "front_9d_include": bool(active_front9_row["include_tenor"]),
        "front_9d_model_vrp_log": front9_rule["model_vrp_log"],
        "front_9d_model_vrp_z": front9_rule["model_vrp_z"],
        "front_9d_model_vrp_z_3m": front9_rule["model_vrp_z_3m"],
        "front_9d_model_vrp_z_1y": front9_rule["model_vrp_z_1y"],
        "front_9d_RSI14_max": front9_rule["RSI14_max"],
        "front_9d_rv21d_vol_pct_min": front9_rule["rv21d_vol_pct_min"],

        "front_12_15_include": True,
        "front_12_15_model_vrp_log": front1215_rule["model_vrp_log"],
        "front_12_15_model_vrp_z": front1215_rule["model_vrp_z"],
        "front_12_15_model_vrp_z_3m": front1215_rule["model_vrp_z_3m"],
        "front_12_15_model_vrp_z_1y": front1215_rule["model_vrp_z_1y"],
        "front_12_15_RSI14_max": front1215_rule["RSI14_max"],
        "front_12_15_rv21d_vol_pct_min": front1215_rule["rv21d_vol_pct_min"],

        "middle_18d_include": bool(spec["include_18d"]),
        "middle_18d_model_vrp_log": middle18_rule["model_vrp_log"],
        "middle_18d_model_vrp_z": middle18_rule["model_vrp_z"],
        "middle_18d_model_vrp_z_3m": middle18_rule["model_vrp_z_3m"],
        "middle_18d_model_vrp_z_1y": middle18_rule["model_vrp_z_1y"],
        "middle_18d_RSI14_max": middle18_rule["RSI14_max"],
        "middle_18d_rv21d_vol_pct_min": middle18_rule["rv21d_vol_pct_min"],

        "middle_21_24_include": True,
        "middle_21_24_model_vrp_log": middle2124_rule["model_vrp_log"],
        "middle_21_24_model_vrp_z": middle2124_rule["model_vrp_z"],
        "middle_21_24_model_vrp_z_3m": middle2124_rule["model_vrp_z_3m"],
        "middle_21_24_model_vrp_z_1y": middle2124_rule["model_vrp_z_1y"],
        "middle_21_24_RSI14_max": middle2124_rule["RSI14_max"],
        "middle_21_24_rv21d_vol_pct_min": middle2124_rule["rv21d_vol_pct_min"],

        "back_model_vrp_log": back_locked_rule["model_vrp_log"],
        "back_model_vrp_z": back_locked_rule["model_vrp_z"],
        "back_model_vrp_z_3m": back_locked_rule["model_vrp_z_3m"],
        "back_model_vrp_z_1y": back_locked_rule["model_vrp_z_1y"],
        "back_RSI14_max": back_locked_rule["RSI14_max"],
        "back_rv21d_vol_pct_min": back_locked_rule["rv21d_vol_pct_min"],

        "back_locked": True,
        "middle_between_front_12_15_and_back": bool(valid_middle_rule(middle18_rule) and valid_middle_rule(middle2124_rule)),
        "uses_rv21d_threshold_contract": True,
        "forecast_vol_pct_threshold_used": False,
        "design_only": True,
    })

    for tenor in ALL_TENORS:
        bucket = TENOR_BUCKET_MAP[int(tenor)]

        if tenor == 9:
            rule = front9_rule
            include_tenor = bool(active_front9_row["include_tenor"])
            rule_scope = "front_9d"
            rule_type = "active_front_ex9d_9d_excluded"
        elif tenor in [12, 15]:
            rule = front1215_rule
            include_tenor = True
            rule_scope = "front_12_15"
            rule_type = "active_front_ex9d_12_15"
        elif tenor == 18:
            rule = middle18_rule
            include_tenor = bool(spec["include_18d"])
            rule_scope = "middle_18d"
            rule_type = "middle_candidate_18d_specific"
        elif tenor in [21, 24]:
            rule = middle2124_rule
            include_tenor = True
            rule_scope = "middle_21_24"
            rule_type = "middle_candidate_21_24"
        elif bucket == "Back":
            rule = back_locked_rule
            include_tenor = True
            rule_scope = "back"
            rule_type = "back_locked"
        else:
            raise ValueError(f"Unexpected tenor/bucket: tenor={tenor}, bucket={bucket}")

        long_rows.append({
            "candidate_id": spec["candidate_id"],
            "candidate_family": MIDDLE_CANDIDATE_FAMILY,
            "candidate_subfamily": spec["candidate_subfamily"],
            "candidate_description": spec["candidate_description"],
            "locked_model_spec": LOCKED_MODEL_SPEC,
            "active_front_candidate_id": ACTIVE_FRONT_CANDIDATE_ID,
            "active_front_label": ACTIVE_FRONT_LABEL,
            "tenor": int(tenor),
            "tenor_bucket": bucket,
            "tenor_bucket_order": BUCKET_ORDER_MAP[bucket],
            "rule_scope": rule_scope,
            "rule_type": rule_type,
            "include_tenor": bool(include_tenor),

            "model_vrp_log": rule["model_vrp_log"],
            "model_vrp_z": rule["model_vrp_z"],
            "model_vrp_z_3m": rule["model_vrp_z_3m"],
            "model_vrp_z_1y": rule["model_vrp_z_1y"],
            "RSI14_max": rule["RSI14_max"],
            "rv21d_vol_pct_min": rule["rv21d_vol_pct_min"],

            "is_front_active": bucket == "Front",
            "is_front_9d_excluded": tenor == 9 and not bool(include_tenor),
            "is_middle": bucket == "Middle",
            "is_middle_18d": tenor == 18,
            "is_middle_21_24": tenor in [21, 24],
            "is_back_locked": bucket == "Back",
            "design_only": True,
        })

candidate_grid_wide = pd.DataFrame(wide_rows)
candidate_grid_long = (
    pd.DataFrame(long_rows)
    .sort_values(["candidate_id", "tenor"])
    .reset_index(drop=True)
)

# ======================================================================================
# 5. Save outputs
# ======================================================================================

middle_by_tenor_path = OUT_AUDIT_DIR / f"08R_middle_rv21d_weakness_by_tenor_{CELL8R_TIMESTAMP}.csv"
middle_by_year_path = OUT_AUDIT_DIR / f"08R_middle_rv21d_weakness_by_year_{CELL8R_TIMESTAMP}.csv"
middle_by_tenor_year_path = OUT_AUDIT_DIR / f"08R_middle_rv21d_weakness_by_tenor_year_{CELL8R_TIMESTAMP}.csv"
middle_state_diagnostics_path = OUT_AUDIT_DIR / f"08R_middle_rv21d_weakness_state_diagnostics_{CELL8R_TIMESTAMP}.csv"
middle_2026_detail_path = OUT_AUDIT_DIR / f"08R_middle_rv21d_weakness_2026_trade_detail_{CELL8R_TIMESTAMP}.csv"
middle_2026_worst_path = OUT_AUDIT_DIR / f"08R_middle_rv21d_weakness_2026_worst_trades_{CELL8R_TIMESTAMP}.csv"

candidate_grid_wide_csv_path = OUT_AUDIT_DIR / f"08R_core_middle_rv21d_candidate_grid_wide_{CELL8R_TIMESTAMP}.csv"
candidate_grid_wide_parquet_path = OUT_PROCESSED_DIR / f"08R_core_middle_rv21d_candidate_grid_wide_{CELL8R_TIMESTAMP}.parquet"
candidate_grid_long_csv_path = OUT_AUDIT_DIR / f"08R_core_middle_rv21d_candidate_grid_long_{CELL8R_TIMESTAMP}.csv"
candidate_grid_long_parquet_path = OUT_PROCESSED_DIR / f"08R_core_middle_rv21d_candidate_grid_long_{CELL8R_TIMESTAMP}.parquet"
validation_path = OUT_AUDIT_DIR / f"08R_core_middle_rv21d_candidate_grid_validation_{CELL8R_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"08R_core_middle_rv21d_candidate_grid_manifest_{CELL8R_TIMESTAMP}.json"

middle_by_tenor.to_csv(middle_by_tenor_path, index=False)
middle_by_year.to_csv(middle_by_year_path, index=False)
middle_by_tenor_year.to_csv(middle_by_tenor_year_path, index=False)
middle_state_diagnostics.to_csv(middle_state_diagnostics_path, index=False)
middle_2026_detail.to_csv(middle_2026_detail_path, index=False)
middle_2026_worst.to_csv(middle_2026_worst_path, index=False)

candidate_grid_wide.to_csv(candidate_grid_wide_csv_path, index=False)
candidate_grid_wide.to_parquet(candidate_grid_wide_parquet_path, index=False)
candidate_grid_long.to_csv(candidate_grid_long_csv_path, index=False)
candidate_grid_long.to_parquet(candidate_grid_long_parquet_path, index=False)

# ======================================================================================
# 6. Validation
# ======================================================================================

validation_rows = []

required_selected_cols = [
    "candidate_id",
    "selection_universe",
    "trade_date",
    "tenor",
    "tenor_bucket",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
]

missing_selected_cols = [c for c in required_selected_cols if c not in selected.columns]

expected_candidate_count = 4
expected_long_rows = expected_candidate_count * len(ALL_TENORS)

old_forecast_threshold_cols = [
    c for c in list(candidate_grid_wide.columns) + list(candidate_grid_long.columns)
    if "forecast_vol_pct_min" in c
]

bad_z_equality = candidate_grid_long[
    candidate_grid_long["model_vrp_z_3m"].ne(candidate_grid_long["model_vrp_z_1y"])
]

bad_front9_include = candidate_grid_long[
    candidate_grid_long["tenor"].eq(9)
    & candidate_grid_long["include_tenor"].astype(bool)
]

back_rules = candidate_grid_long[candidate_grid_long["is_back_locked"].astype(bool)].copy()

back_reference = (
    back_rules[
        [
            "model_vrp_log",
            "model_vrp_z",
            "RSI14_max",
            "rv21d_vol_pct_min",
        ]
    ]
    .drop_duplicates()
)

bad_back_locked = len(back_reference) != 1

bad_middle_between_rows = []

for _, row in candidate_grid_long[candidate_grid_long["is_middle"].astype(bool)].iterrows():
    for param in ["model_vrp_log", "model_vrp_z", "RSI14_max", "rv21d_vol_pct_min"]:
        if not between_or_equal(row[param], front1215_rule[param], back_locked_rule[param]):
            bad_middle_between_rows.append({
                "candidate_id": row["candidate_id"],
                "tenor": int(row["tenor"]),
                "parameter": param,
                "front_12_15": front1215_rule[param],
                "middle": row[param],
                "back": back_locked_rule[param],
            })

bad_middle_between = pd.DataFrame(bad_middle_between_rows)

unexpected_trade_cols = [
    c for c in [
        "trades",
        "total_pnl",
        "avg_pnl_per_day",
        "profit_factor",
        "win_rate",
        "pnl_per_day_drawdown",
    ]
    if c in candidate_grid_wide.columns or c in candidate_grid_long.columns
]

add_validation(
    validation_rows,
    "cell6_selected_panel_loaded",
    "PASS" if len(selected) > 0 else "FAIL",
    f"rows={len(selected):,}; path={selected_panel_path}",
)

add_validation(
    validation_rows,
    "active_front_candidate_available",
    "PASS" if len(active_selected) > 0 else "FAIL",
    f"active_front_candidate_id={ACTIVE_FRONT_CANDIDATE_ID}; rows={len(active_selected):,}",
)

add_validation(
    validation_rows,
    "required_selected_columns_available",
    "PASS" if not missing_selected_cols else "FAIL",
    f"missing={missing_selected_cols}",
)

add_validation(
    validation_rows,
    "middle_diagnostics_created",
    "PASS" if len(middle_by_tenor) > 0 and len(middle_by_year) > 0 else "FAIL",
    f"middle_by_tenor_rows={len(middle_by_tenor):,}; middle_by_year_rows={len(middle_by_year):,}",
)

add_validation(
    validation_rows,
    "middle_2026_detail_created",
    "PASS" if len(middle_2026_detail) > 0 else "WARN",
    f"rows={len(middle_2026_detail):,}",
)

add_validation(
    validation_rows,
    "candidate_design_count",
    "PASS" if len(candidate_grid_wide) == expected_candidate_count else "FAIL",
    f"candidate_count={len(candidate_grid_wide):,}; expected={expected_candidate_count:,}",
)

add_validation(
    validation_rows,
    "long_grid_all_tenors_per_candidate",
    "PASS" if len(candidate_grid_long) == expected_long_rows else "FAIL",
    f"long_rows={len(candidate_grid_long):,}; expected={expected_long_rows:,}",
)

add_validation(
    validation_rows,
    "active_front_ex9d_carried_forward",
    "PASS" if bad_front9_include.empty else "FAIL",
    f"bad_front_9d_include_rows={len(bad_front9_include):,}",
)

add_validation(
    validation_rows,
    "uses_rv21d_threshold_contract",
    "PASS" if "rv21d_vol_pct_min" in candidate_grid_long.columns else "FAIL",
    "rule field is rv21d_vol_pct_min",
)

add_validation(
    validation_rows,
    "no_forecast_vol_threshold_columns",
    "PASS" if not old_forecast_threshold_cols else "FAIL",
    f"old_forecast_threshold_cols={old_forecast_threshold_cols}",
)

add_validation(
    validation_rows,
    "forecast_vol_pct_diagnostic_only",
    "PASS" if "forecast_vol_pct" in selected.columns else "FAIL",
    "forecast_vol_pct retained only in source/diagnostic data, not candidate threshold grid",
)

add_validation(
    validation_rows,
    "z_3m_equals_z_1y_within_bucket_rules",
    "PASS" if bad_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_z_equality):,}",
)

add_validation(
    validation_rows,
    "back_locked_all_candidates",
    "PASS" if not bad_back_locked else "FAIL",
    f"distinct_back_rule_rows={len(back_reference):,}; back_reference={back_reference.to_dict('records')}",
)

add_validation(
    validation_rows,
    "middle_between_front_12_15_and_back",
    "PASS" if bad_middle_between.empty else "FAIL",
    f"bad_rows={len(bad_middle_between):,}; examples={bad_middle_between.head(10).to_dict('records') if not bad_middle_between.empty else []}",
)

add_validation(
    validation_rows,
    "no_invalid_middle_z_tightening",
    "PASS" if candidate_grid_long[candidate_grid_long["is_middle"].astype(bool)]["model_vrp_z"].max() <= back_locked_rule["model_vrp_z"] else "FAIL",
    f"middle_z_max={candidate_grid_long[candidate_grid_long['is_middle'].astype(bool)]['model_vrp_z'].max()}; back_z={back_locked_rule['model_vrp_z']}",
)

add_validation(
    validation_rows,
    "no_invalid_middle_RSI_tightening",
    "PASS" if candidate_grid_long[candidate_grid_long["is_middle"].astype(bool)]["RSI14_max"].min() >= min(front1215_rule["RSI14_max"], back_locked_rule["RSI14_max"]) else "FAIL",
    f"middle_RSI_min={candidate_grid_long[candidate_grid_long['is_middle'].astype(bool)]['RSI14_max'].min()}; front_RSI={front1215_rule['RSI14_max']}; back_RSI={back_locked_rule['RSI14_max']}",
)

add_validation(
    validation_rows,
    "no_invalid_middle_rv21d_floor_tightening",
    "PASS" if candidate_grid_long[candidate_grid_long["is_middle"].astype(bool)]["rv21d_vol_pct_min"].max() <= back_locked_rule["rv21d_vol_pct_min"] else "FAIL",
    f"middle_rv21d_max={candidate_grid_long[candidate_grid_long['is_middle'].astype(bool)]['rv21d_vol_pct_min'].max()}; back_rv21d={back_locked_rule['rv21d_vol_pct_min']}",
)

add_validation(
    validation_rows,
    "no_trade_metrics_in_candidate_design",
    "PASS" if not unexpected_trade_cols else "FAIL",
    f"unexpected_trade_cols={unexpected_trade_cols}",
)

add_validation(
    validation_rows,
    "no_candidate_sweep",
    "PASS",
    "Cell 8R creates diagnostics and candidate design only.",
)

add_validation(
    validation_rows,
    "no_final_lock",
    "PASS",
    "No final Middle rule selected.",
)

add_validation(
    validation_rows,
    "no_secondary",
    "PASS",
    "Core only.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed or added.",
)

add_validation(
    validation_rows,
    "no_production_lock",
    "PASS",
    "No production threshold lock performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"].eq("FAIL")]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1_RV21D_repaired_v2.ipynb",
    "cell": "Cell 8R — Middle RV21D weakness diagnostic and candidate design",
    "timestamp": CELL8R_TIMESTAMP,
    "selected_panel_path": str(selected_panel_path),
    "front_candidate_grid_wide_path": str(front_candidate_grid_wide_path),
    "front_candidate_grid_long_path": str(front_candidate_grid_long_path),
    "active_front_candidate_id": ACTIVE_FRONT_CANDIDATE_ID,
    "active_front_label": ACTIVE_FRONT_LABEL,
    "middle_by_tenor_path": str(middle_by_tenor_path),
    "middle_by_year_path": str(middle_by_year_path),
    "middle_by_tenor_year_path": str(middle_by_tenor_year_path),
    "middle_state_diagnostics_path": str(middle_state_diagnostics_path),
    "middle_2026_detail_path": str(middle_2026_detail_path),
    "middle_2026_worst_path": str(middle_2026_worst_path),
    "candidate_grid_wide_csv_path": str(candidate_grid_wide_csv_path),
    "candidate_grid_wide_parquet_path": str(candidate_grid_wide_parquet_path),
    "candidate_grid_long_csv_path": str(candidate_grid_long_csv_path),
    "candidate_grid_long_parquet_path": str(candidate_grid_long_parquet_path),
    "validation_path": str(validation_path),
    "candidate_family": MIDDLE_CANDIDATE_FAMILY,
    "candidate_count": int(len(candidate_grid_wide)),
    "candidate_ids": candidate_grid_wide["candidate_id"].tolist(),
    "threshold_contract": "rv21d_vol_pct > rv21d_vol_pct_min",
    "forecast_vol_pct_role": "model denominator / diagnostic only, not threshold floor",
    "front_12_15_rule": front1215_rule,
    "middle_current_rule": middle_current_rule,
    "back_locked_rule": back_locked_rule,
    "valid_middle_parameter_ranges": {
        "model_vrp_log": [front1215_rule["model_vrp_log"], back_locked_rule["model_vrp_log"]],
        "model_vrp_z": [front1215_rule["model_vrp_z"], back_locked_rule["model_vrp_z"]],
        "RSI14_max": [front1215_rule["RSI14_max"], back_locked_rule["RSI14_max"]],
        "rv21d_vol_pct_min": [front1215_rule["rv21d_vol_pct_min"], back_locked_rule["rv21d_vol_pct_min"]],
    },
    "excluded_invalid_candidate_ideas": [
        "Middle z above locked Back z is not allowed.",
        "Middle RSI cap below active Front/locked Back RSI cap is not allowed.",
        "Middle RV21D floor above active Front/locked Back RV21D floor is not allowed.",
    ],
    "hard_constraints": [
        "Active Front candidate is ex-9D.",
        "Back bucket parameters remain locked.",
        "3m and 1y z-score thresholds are equal within each bucket.",
        "Middle parameter levels are between/equal active Front and locked Back.",
        "forecast_vol_pct is diagnostic only.",
        "No threshold sweep.",
        "No final lock.",
        "No Secondary.",
        "No sizing changes.",
        "No production lock.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 8R validation failed. Review validation output above.")

# ======================================================================================
# 7. Display review
# ======================================================================================

print("=" * 100)
print("Cell 8R — Middle RV21D weakness diagnostic and candidate design")
print("=" * 100)
print(f"Selected panel:                     {selected_panel_path}")
print(f"Active Front candidate:             {ACTIVE_FRONT_CANDIDATE_ID} / {ACTIVE_FRONT_LABEL}")
print(f"Middle all-qualified rows reviewed: {len(middle_all_qualified):,}")
print(f"Middle primary rows reviewed:       {len(middle_primary):,}")
print(f"Middle 2026 primary rows reviewed:  {len(middle_2026_detail):,}")
print(f"Candidate count:                    {len(candidate_grid_wide):,}")
print(f"Candidate long rows:                {len(candidate_grid_long):,}")
print("Threshold contract:                 rv21d_vol_pct > rv21d_vol_pct_min")
print("Active Front ex-9D carried forward: True")
print("Back locked:                        True")
print("3m/1y z equal within bucket:         True")
print("Middle between Front/Back:          True")
print("No candidate sweep:                 True")
print("No final lock:                      True")
print("No Secondary:                       True")
print("No sizing changes:                  True")
print("No production lock:                 True")

print()
print("Valid Middle parameter ranges under hard constraints:")
display(pd.DataFrame([
    {
        "parameter": "model_vrp_log",
        "front_12_15": front1215_rule["model_vrp_log"],
        "current_middle": middle_current_rule["model_vrp_log"],
        "locked_back": back_locked_rule["model_vrp_log"],
        "valid_action": "Can tighten to 0.70.",
    },
    {
        "parameter": "model_vrp_z",
        "front_12_15": front1215_rule["model_vrp_z"],
        "current_middle": middle_current_rule["model_vrp_z"],
        "locked_back": back_locked_rule["model_vrp_z"],
        "valid_action": "Already at Back ceiling; cannot tighten.",
    },
    {
        "parameter": "RSI14_max",
        "front_12_15": front1215_rule["RSI14_max"],
        "current_middle": middle_current_rule["RSI14_max"],
        "locked_back": back_locked_rule["RSI14_max"],
        "valid_action": "Must remain 70.",
    },
    {
        "parameter": "rv21d_vol_pct_min",
        "front_12_15": front1215_rule["rv21d_vol_pct_min"],
        "current_middle": middle_current_rule["rv21d_vol_pct_min"],
        "locked_back": back_locked_rule["rv21d_vol_pct_min"],
        "valid_action": "Must remain 8.5.",
    },
]))

print()
print("Validation:")
display(validation)

print()
print("Middle diagnostics by tenor:")
display(middle_by_tenor)

print()
print("Middle diagnostics by year:")
display(middle_by_year)

print()
print("Middle diagnostics by tenor/year:")
display(middle_by_tenor_year)

print()
print("Middle state diagnostics:")
display(middle_state_diagnostics)

print()
print("Middle 2026 primary selected trade detail:")
display(middle_2026_detail)

print()
print("Middle 2026 worst primary selected trades:")
display(middle_2026_worst)

print()
print("Middle candidate grid — wide:")
display(candidate_grid_wide)

print()
print("Middle candidate grid — long:")
display(candidate_grid_long)

print()
print("Saved outputs:")
for p in [
    middle_by_tenor_path,
    middle_by_year_path,
    middle_by_tenor_year_path,
    middle_state_diagnostics_path,
    middle_2026_detail_path,
    middle_2026_worst_path,
    candidate_grid_wide_csv_path,
    candidate_grid_wide_parquet_path,
    candidate_grid_long_csv_path,
    candidate_grid_long_parquet_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 8R PASSED")

In [ ]:
# ======================================================================================
# Cell 9R — 18D bucket reassignment test
# ======================================================================================
#
# Objective:
#   Test whether 18D belongs in Front instead of Middle.
#
# Compare:
#
#   core_bucket_reassign_0001 — current active structure
#       Front:   12D / 15D
#       Middle:  18D / 21D / 24D
#       Back:    27D / 30D / 33D
#       9D excluded
#
#   core_bucket_reassign_0002 — move 18D into Front
#       Front:   12D / 15D / 18D
#       Middle:  21D / 24D
#       Back:    27D / 30D / 33D
#       9D excluded
#
# Correct volatility-floor contract:
#       rv21d_vol_pct > rv21d_vol_pct_min
#
# NOT:
#       forecast_vol_pct > forecast_vol_pct_min
#
# Hard constraints:
#   1. 9D remains excluded.
#   2. Front ex-9D remains active.
#   3. Back bucket parameters remain locked.
#   4. Within each effective bucket, 3m and 1y z-score thresholds are equal.
#   5. RV21D is the only vol-floor threshold.
#   6. forecast_vol_pct remains diagnostic only.
#   7. No final lock.
#   8. No Secondary.
#   9. No sizing changes.
#   10. No production lock.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 360)
pd.set_option("display.width", 600)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL9R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

ORIGINAL_TENOR_BUCKET_MAP = {
    9: "Front",
    12: "Front",
    15: "Front",
    18: "Middle",
    21: "Middle",
    24: "Middle",
    27: "Back",
    30: "Back",
    33: "Back",
}

BUCKET_ORDER_MAP = {
    "Front": 1,
    "Middle": 2,
    "Back": 3,
}

CURRENT_CANDIDATE_ID = "core_bucket_reassign_0001"
MOVE18_CANDIDATE_ID = "core_bucket_reassign_0002"

CANDIDATE_IDS = [
    CURRENT_CANDIDATE_ID,
    MOVE18_CANDIDATE_ID,
]

SELECTION_UNIVERSES = [
    "all_qualified_trades",
    "one_trade_per_bucket_per_date_best_rank",
    "one_trade_per_date_best_rank",
]

PRIMARY_SELECTION_UNIVERSE = "one_trade_per_bucket_per_date_best_rank"
PRIMARY_RANKING_METRIC = "avg_pnl_per_day"

# Candidate-specific tie-breaker bucket centers.
# Current candidate keeps prior ranking convention from earlier cells.
CANDIDATE_BUCKET_CENTER = {
    CURRENT_CANDIDATE_ID: {
        "Front": 12,
        "Middle": 21,
        "Back": 30,
    },
    MOVE18_CANDIDATE_ID: {
        "Front": 15,
        "Middle": 21,
        "Back": 30,
    },
}

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def extract_rule(grid_long, candidate_id, tenor):
    row = grid_long[
        grid_long["candidate_id"].eq(candidate_id)
        & grid_long["tenor"].eq(tenor)
    ]

    if row.empty:
        raise RuntimeError(f"No rule found for candidate_id={candidate_id}, tenor={tenor}")

    r = row.iloc[0]

    return {
        "model_vrp_log": float(r["model_vrp_log"]),
        "model_vrp_z": float(r["model_vrp_z"]),
        "model_vrp_z_3m": float(r["model_vrp_z_3m"]),
        "model_vrp_z_1y": float(r["model_vrp_z_1y"]),
        "RSI14_max": float(r["RSI14_max"]),
        "rv21d_vol_pct_min": float(r["rv21d_vol_pct_min"]),
    }


def clone_rule(rule):
    out = {
        "model_vrp_log": float(rule["model_vrp_log"]),
        "model_vrp_z": float(rule["model_vrp_z"]),
        "model_vrp_z_3m": float(rule["model_vrp_z"]),
        "model_vrp_z_1y": float(rule["model_vrp_z"]),
        "RSI14_max": float(rule["RSI14_max"]),
        "rv21d_vol_pct_min": float(rule["rv21d_vol_pct_min"]),
    }
    return out


def between_or_equal(x, a, b, tol=1e-12):
    lo = min(float(a), float(b)) - tol
    hi = max(float(a), float(b)) + tol
    return lo <= float(x) <= hi


def add_selection_ranks(df):
    d = df.copy()

    if d.empty:
        return d

    d["bucket_center_tenor"] = d.apply(
        lambda r: CANDIDATE_BUCKET_CENTER[r["candidate_id"]][r["effective_tenor_bucket"]],
        axis=1,
    )

    d["distance_to_bucket_center"] = (d["tenor"] - d["bucket_center_tenor"]).abs()

    bucket_group = ["candidate_id", "trade_date", "effective_tenor_bucket"]

    d["rank_z_1y_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_bucket_date"] = d[
        [
            "rank_z_1y_in_bucket_date",
            "rank_z_3m_in_bucket_date",
            "rank_vrp_log_in_bucket_date",
        ]
    ].mean(axis=1)

    date_group = ["candidate_id", "trade_date"]

    d["rank_z_1y_in_date"] = (
        d.groupby(date_group)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_date"] = (
        d.groupby(date_group)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_date"] = (
        d.groupby(date_group)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_date"] = d[
        [
            "rank_z_1y_in_date",
            "rank_z_3m_in_date",
            "rank_vrp_log_in_date",
        ]
    ].mean(axis=1)

    return d


def select_trade_universes(qualified):
    if qualified.empty:
        return {
            "all_qualified_trades": qualified.copy(),
            "one_trade_per_bucket_per_date_best_rank": qualified.copy(),
            "one_trade_per_date_best_rank": qualified.copy(),
        }

    q = add_selection_ranks(qualified)

    one_trade_per_bucket_date = (
        q.sort_values(
            [
                "candidate_id",
                "trade_date",
                "effective_tenor_bucket_order",
                "avg_signal_rank_in_bucket_date",
                "distance_to_bucket_center",
                "rank_z_1y_in_bucket_date",
                "rank_z_3m_in_bucket_date",
                "rank_vrp_log_in_bucket_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True, True],
        )
        .groupby(["candidate_id", "trade_date", "effective_tenor_bucket"], as_index=False)
        .head(1)
        .copy()
    )

    one_trade_per_date = (
        q.sort_values(
            [
                "candidate_id",
                "trade_date",
                "avg_signal_rank_in_date",
                "rank_z_1y_in_date",
                "rank_z_3m_in_date",
                "rank_vrp_log_in_date",
                "distance_to_bucket_center",
                "effective_tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True, True],
        )
        .groupby(["candidate_id", "trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return {
        "all_qualified_trades": q.copy(),
        "one_trade_per_bucket_per_date_best_rank": one_trade_per_bucket_date,
        "one_trade_per_date_best_rank": one_trade_per_date,
    }


def summarize_trade_set(df, group_cols):
    d = df.copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    numeric_cols = [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "forecast_vol_pct",
        "rv21d_vol_pct",
        "threshold_model_vrp_log",
        "threshold_model_vrp_z_3m",
        "threshold_model_vrp_z_1y",
        "threshold_RSI14_max",
        "threshold_rv21d_vol_pct_min",
    ]

    for c in numeric_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d = d.sort_values(["candidate_id", "selection_universe", "trade_date", "tenor"]).copy()

    grouped = d.groupby(group_cols, dropna=False, observed=False)

    rows = []

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {}

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": int(g["trade_date"].nunique()),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        for signal_col in [
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "threshold_model_vrp_log",
            "threshold_model_vrp_z_3m",
            "threshold_model_vrp_z_1y",
            "threshold_RSI14_max",
            "threshold_rv21d_vol_pct_min",
        ]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        rows.append(row)

    return pd.DataFrame(rows)


def attach_candidate_design(summary_df, candidate_grid_wide):
    if summary_df.empty:
        return summary_df

    return summary_df.merge(
        candidate_grid_wide,
        on="candidate_id",
        how="left",
        validate="many_to_one",
    )


def pair_bucket_selection(selected_primary, bucket_name):
    cur = selected_primary[
        selected_primary["candidate_id"].eq(CURRENT_CANDIDATE_ID)
        & selected_primary["effective_tenor_bucket"].eq(bucket_name)
    ].copy()

    mv = selected_primary[
        selected_primary["candidate_id"].eq(MOVE18_CANDIDATE_ID)
        & selected_primary["effective_tenor_bucket"].eq(bucket_name)
    ].copy()

    keep_cols = [
        "trade_date",
        "tenor",
        "original_tenor_bucket",
        "effective_tenor_bucket",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "rv21d_vol_pct",
        "forecast_vol_pct",
        "RSI14",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
    ]

    cur = cur[[c for c in keep_cols if c in cur.columns]].rename(
        columns={c: f"current_{c}" for c in keep_cols if c != "trade_date" and c in cur.columns}
    )

    mv = mv[[c for c in keep_cols if c in mv.columns]].rename(
        columns={c: f"move18_{c}" for c in keep_cols if c != "trade_date" and c in mv.columns}
    )

    paired = cur.merge(
        mv,
        on="trade_date",
        how="outer",
        validate="one_to_one",
    )

    for c in [
        "current_normalized_pnl_dollars",
        "current_normalized_pnl_per_day",
        "move18_normalized_pnl_dollars",
        "move18_normalized_pnl_per_day",
    ]:
        if c not in paired.columns:
            paired[c] = 0.0
        paired[c] = pd.to_numeric(paired[c], errors="coerce").fillna(0.0)

    paired["delta_pnl_dollars_move18_minus_current"] = (
        paired["move18_normalized_pnl_dollars"]
        - paired["current_normalized_pnl_dollars"]
    )

    paired["delta_pnl_per_day_move18_minus_current"] = (
        paired["move18_normalized_pnl_per_day"]
        - paired["current_normalized_pnl_per_day"]
    )

    paired["effective_bucket_compared"] = bucket_name

    return paired.sort_values("trade_date").reset_index(drop=True)


def summarize_delta(df, group_cols):
    d = df.copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    for c in [
        "delta_pnl_dollars_move18_minus_current",
        "delta_pnl_per_day_move18_minus_current",
    ]:
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0)

    rows = []

    grouped = d.groupby(group_cols, dropna=False, observed=False)

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {}

        for col, key in zip(group_cols, keys):
            row[col] = key

        delta_day = g["delta_pnl_per_day_move18_minus_current"]
        delta_pnl = g["delta_pnl_dollars_move18_minus_current"]

        row.update({
            "rows": int(len(g)),
            "helped_rows": int((delta_day > 0).sum()),
            "hurt_rows": int((delta_day < 0).sum()),
            "flat_rows": int((delta_day == 0).sum()),
            "total_delta_pnl_dollars": float(delta_pnl.sum()),
            "total_delta_pnl_per_day": float(delta_day.sum()),
            "avg_delta_pnl_per_day": float(delta_day.mean()),
            "median_delta_pnl_per_day": float(delta_day.median()),
            "best_delta_pnl_per_day": float(delta_day.max()),
            "worst_delta_pnl_per_day": float(delta_day.min()),
            "delta_pnl_per_day_drawdown": max_drawdown_from_pnl(delta_day),
            "worst_5_delta_pnl_per_day_sum": worst_rolling_sum(delta_day, 5),
            "worst_10_delta_pnl_per_day_sum": worst_rolling_sum(delta_day, 10),
            "worst_20_delta_pnl_per_day_sum": worst_rolling_sum(delta_day, 20),
        })

        rows.append(row)

    return pd.DataFrame(rows)


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


# ======================================================================================
# 2. Load inputs
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01R_unified_fds_signal_base_panel_with_rv21d_*.parquet",
    required=True,
)

middle_grid_wide_path = latest_file(
    OUT_PROCESSED_DIR,
    "08R_core_middle_rv21d_candidate_grid_wide_*.parquet",
    required=True,
)

middle_grid_long_path = latest_file(
    OUT_PROCESSED_DIR,
    "08R_core_middle_rv21d_candidate_grid_long_*.parquet",
    required=True,
)

base = pd.read_parquet(base_panel_path)
middle_grid_wide = pd.read_parquet(middle_grid_wide_path)
middle_grid_long = pd.read_parquet(middle_grid_long_path)

base["trade_date"] = pd.to_datetime(base["trade_date"], errors="coerce").dt.normalize()

for c in [
    "tenor",
    "tenor_bucket_order",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")

for c in [
    "tenor",
    "tenor_bucket_order",
    "model_vrp_log",
    "model_vrp_z",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14_max",
    "rv21d_vol_pct_min",
]:
    if c in middle_grid_long.columns:
        middle_grid_long[c] = pd.to_numeric(middle_grid_long[c], errors="coerce")

middle_grid_long["include_tenor"] = middle_grid_long["include_tenor"].astype(bool)

# Use the current Middle reference from Cell 8R as the active current structure.
ACTIVE_MIDDLE_REFERENCE_ID = "core_middle_rv21d_0001"

active_grid = middle_grid_long[
    middle_grid_long["candidate_id"].eq(ACTIVE_MIDDLE_REFERENCE_ID)
].copy()

if active_grid.empty:
    raise RuntimeError(f"Could not find {ACTIVE_MIDDLE_REFERENCE_ID} in Cell 8R grid.")

# ======================================================================================
# 3. Build bucket reassignment candidate grid
# ======================================================================================

front12_rule = extract_rule(middle_grid_long, ACTIVE_MIDDLE_REFERENCE_ID, 12)
middle18_rule = extract_rule(middle_grid_long, ACTIVE_MIDDLE_REFERENCE_ID, 18)
middle21_rule = extract_rule(middle_grid_long, ACTIVE_MIDDLE_REFERENCE_ID, 21)
back30_rule = extract_rule(middle_grid_long, ACTIVE_MIDDLE_REFERENCE_ID, 30)

wide_rows = []
long_rows = []

candidate_specs = [
    {
        "candidate_id": CURRENT_CANDIDATE_ID,
        "candidate_family": "core_18d_bucket_reassignment_test",
        "candidate_subfamily": "current_active_structure",
        "candidate_description": "Current active structure: Front 12/15, Middle 18/21/24, Back 27/30/33, 9D excluded.",
        "front_tenors": [12, 15],
        "middle_tenors": [18, 21, 24],
        "back_tenors": [27, 30, 33],
        "move_18d_to_front": False,
    },
    {
        "candidate_id": MOVE18_CANDIDATE_ID,
        "candidate_family": "core_18d_bucket_reassignment_test",
        "candidate_subfamily": "move_18d_to_front",
        "candidate_description": "Move 18D into Front: Front 12/15/18, Middle 21/24, Back 27/30/33, 9D excluded.",
        "front_tenors": [12, 15, 18],
        "middle_tenors": [21, 24],
        "back_tenors": [27, 30, 33],
        "move_18d_to_front": True,
    },
]

for spec in candidate_specs:
    cid = spec["candidate_id"]

    for tenor in ALL_TENORS:
        original_bucket = ORIGINAL_TENOR_BUCKET_MAP[tenor]

        if tenor == 9:
            effective_bucket = "Front"
            effective_bucket_order = BUCKET_ORDER_MAP[effective_bucket]
            rule = front12_rule
            include_tenor = False
            rule_scope = "front_9d"
            rule_type = "front_ex9d_9d_excluded"
        elif tenor in [12, 15]:
            effective_bucket = "Front"
            effective_bucket_order = BUCKET_ORDER_MAP[effective_bucket]
            rule = front12_rule
            include_tenor = True
            rule_scope = "front"
            rule_type = "active_front_12_15"
        elif tenor == 18 and spec["move_18d_to_front"]:
            effective_bucket = "Front"
            effective_bucket_order = BUCKET_ORDER_MAP[effective_bucket]
            rule = front12_rule
            include_tenor = True
            rule_scope = "front"
            rule_type = "18d_moved_to_front_uses_front_rule"
        elif tenor == 18 and not spec["move_18d_to_front"]:
            effective_bucket = "Middle"
            effective_bucket_order = BUCKET_ORDER_MAP[effective_bucket]
            rule = middle18_rule
            include_tenor = True
            rule_scope = "middle"
            rule_type = "current_18d_middle_uses_middle_rule"
        elif tenor in [21, 24]:
            effective_bucket = "Middle"
            effective_bucket_order = BUCKET_ORDER_MAP[effective_bucket]
            rule = middle21_rule
            include_tenor = True
            rule_scope = "middle"
            rule_type = "middle_21_24"
        elif tenor in [27, 30, 33]:
            effective_bucket = "Back"
            effective_bucket_order = BUCKET_ORDER_MAP[effective_bucket]
            rule = back30_rule
            include_tenor = True
            rule_scope = "back"
            rule_type = "back_locked"
        else:
            raise ValueError(f"Unexpected tenor: {tenor}")

        long_rows.append({
            "candidate_id": cid,
            "candidate_family": spec["candidate_family"],
            "candidate_subfamily": spec["candidate_subfamily"],
            "candidate_description": spec["candidate_description"],
            "locked_model_spec": LOCKED_MODEL_SPEC,

            "tenor": int(tenor),
            "original_tenor_bucket": original_bucket,
            "original_tenor_bucket_order": BUCKET_ORDER_MAP[original_bucket],
            "effective_tenor_bucket": effective_bucket,
            "effective_tenor_bucket_order": effective_bucket_order,

            "rule_scope": rule_scope,
            "rule_type": rule_type,
            "include_tenor": bool(include_tenor),

            "model_vrp_log": float(rule["model_vrp_log"]),
            "model_vrp_z": float(rule["model_vrp_z"]),
            "model_vrp_z_3m": float(rule["model_vrp_z"]),
            "model_vrp_z_1y": float(rule["model_vrp_z"]),
            "RSI14_max": float(rule["RSI14_max"]),
            "rv21d_vol_pct_min": float(rule["rv21d_vol_pct_min"]),

            "is_9d_excluded": tenor == 9 and not bool(include_tenor),
            "is_18d": tenor == 18,
            "is_18d_moved_to_front": tenor == 18 and spec["move_18d_to_front"],
            "is_front_effective": effective_bucket == "Front",
            "is_middle_effective": effective_bucket == "Middle",
            "is_back_locked": effective_bucket == "Back",
            "uses_rv21d_threshold_contract": True,
            "forecast_vol_pct_threshold_used": False,
            "design_only": False,
            "final_lock": False,
        })

    wide_rows.append({
        "candidate_id": cid,
        "candidate_family": spec["candidate_family"],
        "candidate_subfamily": spec["candidate_subfamily"],
        "candidate_description": spec["candidate_description"],
        "locked_model_spec": LOCKED_MODEL_SPEC,

        "front_tenors": ",".join(map(str, spec["front_tenors"])),
        "middle_tenors": ",".join(map(str, spec["middle_tenors"])),
        "back_tenors": ",".join(map(str, spec["back_tenors"])),

        "include_9d": False,
        "move_18d_to_front": bool(spec["move_18d_to_front"]),

        "front_model_vrp_log": front12_rule["model_vrp_log"],
        "front_model_vrp_z": front12_rule["model_vrp_z"],
        "front_model_vrp_z_3m": front12_rule["model_vrp_z"],
        "front_model_vrp_z_1y": front12_rule["model_vrp_z"],
        "front_RSI14_max": front12_rule["RSI14_max"],
        "front_rv21d_vol_pct_min": front12_rule["rv21d_vol_pct_min"],

        "middle_model_vrp_log": middle21_rule["model_vrp_log"],
        "middle_model_vrp_z": middle21_rule["model_vrp_z"],
        "middle_model_vrp_z_3m": middle21_rule["model_vrp_z"],
        "middle_model_vrp_z_1y": middle21_rule["model_vrp_z"],
        "middle_RSI14_max": middle21_rule["RSI14_max"],
        "middle_rv21d_vol_pct_min": middle21_rule["rv21d_vol_pct_min"],

        "back_model_vrp_log": back30_rule["model_vrp_log"],
        "back_model_vrp_z": back30_rule["model_vrp_z"],
        "back_model_vrp_z_3m": back30_rule["model_vrp_z"],
        "back_model_vrp_z_1y": back30_rule["model_vrp_z"],
        "back_RSI14_max": back30_rule["RSI14_max"],
        "back_rv21d_vol_pct_min": back30_rule["rv21d_vol_pct_min"],

        "back_locked": True,
        "uses_rv21d_threshold_contract": True,
        "forecast_vol_pct_threshold_used": False,
        "final_lock": False,
    })

candidate_grid_long = pd.DataFrame(long_rows).sort_values(["candidate_id", "tenor"]).reset_index(drop=True)
candidate_grid_wide = pd.DataFrame(wide_rows).sort_values("candidate_id").reset_index(drop=True)

# ======================================================================================
# 4. Apply candidate rules
# ======================================================================================

thresholds = candidate_grid_long.rename(columns={
    "model_vrp_log": "threshold_model_vrp_log",
    "model_vrp_z": "threshold_model_vrp_z",
    "model_vrp_z_3m": "threshold_model_vrp_z_3m",
    "model_vrp_z_1y": "threshold_model_vrp_z_1y",
    "RSI14_max": "threshold_RSI14_max",
    "rv21d_vol_pct_min": "threshold_rv21d_vol_pct_min",
})

threshold_cols = [
    "candidate_id",
    "candidate_family",
    "candidate_subfamily",
    "candidate_description",
    "tenor",
    "original_tenor_bucket",
    "original_tenor_bucket_order",
    "effective_tenor_bucket",
    "effective_tenor_bucket_order",
    "rule_scope",
    "rule_type",
    "include_tenor",
    "threshold_model_vrp_log",
    "threshold_model_vrp_z",
    "threshold_model_vrp_z_3m",
    "threshold_model_vrp_z_1y",
    "threshold_RSI14_max",
    "threshold_rv21d_vol_pct_min",
    "is_9d_excluded",
    "is_18d",
    "is_18d_moved_to_front",
    "is_front_effective",
    "is_middle_effective",
    "is_back_locked",
]

base_for_join = base.copy()

base_for_join = base_for_join.rename(columns={
    "tenor_bucket": "source_tenor_bucket",
    "tenor_bucket_order": "source_tenor_bucket_order",
})

panel = base_for_join.merge(
    thresholds[threshold_cols],
    on="tenor",
    how="inner",
    validate="many_to_many",
)

required_signal_available = (
    panel["include_tenor"].astype(bool)
    & panel["model_vrp_log"].notna()
    & panel["model_vrp_z_3m"].notna()
    & panel["model_vrp_z_1y"].notna()
    & panel["RSI14"].notna()
    & panel["rv21d_vol_pct"].notna()
    & panel["threshold_model_vrp_log"].notna()
    & panel["threshold_model_vrp_z_3m"].notna()
    & panel["threshold_model_vrp_z_1y"].notna()
    & panel["threshold_RSI14_max"].notna()
    & panel["threshold_rv21d_vol_pct_min"].notna()
)

outcome_available = (
    panel["normalized_pnl_dollars"].notna()
    & panel["normalized_pnl_per_day"].notna()
)

pass_mask = (
    required_signal_available
    & (panel["model_vrp_log"] > panel["threshold_model_vrp_log"])
    & (panel["model_vrp_z_3m"] > panel["threshold_model_vrp_z_3m"])
    & (panel["model_vrp_z_1y"] > panel["threshold_model_vrp_z_1y"])
    & (panel["RSI14"] < panel["threshold_RSI14_max"])
    & (panel["rv21d_vol_pct"] > panel["threshold_rv21d_vol_pct_min"])
)

panel["candidate_pass"] = pass_mask

qualified_all_dates = panel[pass_mask].copy()
qualified_complete = panel[pass_mask & outcome_available].copy()

universes = select_trade_universes(qualified_complete)

selected_trade_panel = pd.concat(
    [df.assign(selection_universe=name) for name, df in universes.items()],
    ignore_index=True,
)

# ======================================================================================
# 5. Summaries
# ======================================================================================

summary_frames = []
bucket_frames = []
tenor_frames = []
year_frames = []
tenor_year_frames = []

for universe_name, df_u in universes.items():
    df_u = df_u.assign(selection_universe=universe_name).copy()

    summary_frames.append(
        summarize_trade_set(
            df_u,
            group_cols=["candidate_id", "candidate_subfamily", "selection_universe"],
        )
    )

    bucket_frames.append(
        summarize_trade_set(
            df_u,
            group_cols=[
                "candidate_id",
                "candidate_subfamily",
                "selection_universe",
                "effective_tenor_bucket",
                "effective_tenor_bucket_order",
            ],
        )
    )

    tenor_frames.append(
        summarize_trade_set(
            df_u,
            group_cols=[
                "candidate_id",
                "candidate_subfamily",
                "selection_universe",
                "tenor",
                "original_tenor_bucket",
                "effective_tenor_bucket",
                "effective_tenor_bucket_order",
            ],
        )
    )

    y = df_u.copy()
    y["year"] = y["trade_date"].dt.year.astype(int)

    year_frames.append(
        summarize_trade_set(
            y,
            group_cols=["candidate_id", "candidate_subfamily", "selection_universe", "year"],
        )
    )

    tenor_year_frames.append(
        summarize_trade_set(
            y,
            group_cols=[
                "candidate_id",
                "candidate_subfamily",
                "selection_universe",
                "tenor",
                "original_tenor_bucket",
                "effective_tenor_bucket",
                "year",
            ],
        )
    )

summary = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
by_bucket = pd.concat(bucket_frames, ignore_index=True) if bucket_frames else pd.DataFrame()
by_tenor = pd.concat(tenor_frames, ignore_index=True) if tenor_frames else pd.DataFrame()
by_year = pd.concat(year_frames, ignore_index=True) if year_frames else pd.DataFrame()
by_tenor_year = pd.concat(tenor_year_frames, ignore_index=True) if tenor_year_frames else pd.DataFrame()

summary = attach_candidate_design(summary, candidate_grid_wide)
by_bucket = attach_candidate_design(by_bucket, candidate_grid_wide)
by_tenor = attach_candidate_design(by_tenor, candidate_grid_wide)
by_year = attach_candidate_design(by_year, candidate_grid_wide)
by_tenor_year = attach_candidate_design(by_tenor_year, candidate_grid_wide)

summary = summary.sort_values(
    ["selection_universe", "avg_pnl_per_day"],
    ascending=[True, False],
).reset_index(drop=True)

by_bucket = by_bucket.sort_values(
    ["selection_universe", "candidate_id", "effective_tenor_bucket_order"],
).reset_index(drop=True)

by_tenor = by_tenor.sort_values(
    ["selection_universe", "candidate_id", "tenor"],
).reset_index(drop=True)

by_year = by_year.sort_values(
    ["selection_universe", "candidate_id", "year"],
).reset_index(drop=True)

by_tenor_year = by_tenor_year.sort_values(
    ["selection_universe", "candidate_id", "tenor", "year"],
).reset_index(drop=True)

# ======================================================================================
# 6. Critical summary and deltas
# ======================================================================================

primary_summary = summary[
    summary["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

primary_bucket = by_bucket[
    by_bucket["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

primary_tenor = by_tenor[
    by_tenor["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

def extract_bucket_metric(bucket_name, metric, output_name):
    return (
        primary_bucket[primary_bucket["effective_tenor_bucket"].eq(bucket_name)][["candidate_id", metric]]
        .rename(columns={metric: output_name})
    )


def extract_tenor_metric(tenor, metric, output_name):
    return (
        primary_tenor[primary_tenor["tenor"].eq(tenor)][["candidate_id", metric]]
        .rename(columns={metric: output_name})
    )


critical_summary = primary_summary.copy()

critical_merges = [
    extract_bucket_metric("Front", "trades", "front_trades"),
    extract_bucket_metric("Front", "avg_pnl_per_day", "front_avg_pnl_per_day"),
    extract_bucket_metric("Front", "avg_pnl_per_day_2025", "front_avg_pnl_per_day_2025"),
    extract_bucket_metric("Front", "avg_pnl_per_day_2026", "front_avg_pnl_per_day_2026"),
    extract_bucket_metric("Front", "profit_factor_pnl_per_day", "front_profit_factor_pnl_per_day"),
    extract_bucket_metric("Front", "pnl_per_day_drawdown", "front_pnl_per_day_drawdown"),
    extract_bucket_metric("Front", "worst_20_trade_pnl_per_day_sum", "front_worst_20_trade_pnl_per_day_sum"),

    extract_bucket_metric("Middle", "trades", "middle_trades"),
    extract_bucket_metric("Middle", "avg_pnl_per_day", "middle_avg_pnl_per_day"),
    extract_bucket_metric("Middle", "avg_pnl_per_day_2025", "middle_avg_pnl_per_day_2025"),
    extract_bucket_metric("Middle", "avg_pnl_per_day_2026", "middle_avg_pnl_per_day_2026"),
    extract_bucket_metric("Middle", "profit_factor_pnl_per_day", "middle_profit_factor_pnl_per_day"),
    extract_bucket_metric("Middle", "pnl_per_day_drawdown", "middle_pnl_per_day_drawdown"),
    extract_bucket_metric("Middle", "worst_20_trade_pnl_per_day_sum", "middle_worst_20_trade_pnl_per_day_sum"),

    extract_tenor_metric(12, "trades", "tenor_12_trades"),
    extract_tenor_metric(12, "avg_pnl_per_day", "tenor_12_avg_pnl_per_day"),
    extract_tenor_metric(12, "avg_pnl_per_day_2026", "tenor_12_avg_pnl_per_day_2026"),

    extract_tenor_metric(15, "trades", "tenor_15_trades"),
    extract_tenor_metric(15, "avg_pnl_per_day", "tenor_15_avg_pnl_per_day"),
    extract_tenor_metric(15, "avg_pnl_per_day_2026", "tenor_15_avg_pnl_per_day_2026"),

    extract_tenor_metric(18, "trades", "tenor_18_trades"),
    extract_tenor_metric(18, "avg_pnl_per_day", "tenor_18_avg_pnl_per_day"),
    extract_tenor_metric(18, "avg_pnl_per_day_2026", "tenor_18_avg_pnl_per_day_2026"),

    extract_tenor_metric(21, "trades", "tenor_21_trades"),
    extract_tenor_metric(21, "avg_pnl_per_day", "tenor_21_avg_pnl_per_day"),
    extract_tenor_metric(21, "avg_pnl_per_day_2026", "tenor_21_avg_pnl_per_day_2026"),

    extract_tenor_metric(24, "trades", "tenor_24_trades"),
    extract_tenor_metric(24, "avg_pnl_per_day", "tenor_24_avg_pnl_per_day"),
    extract_tenor_metric(24, "avg_pnl_per_day_2026", "tenor_24_avg_pnl_per_day_2026"),
]

for m in critical_merges:
    critical_summary = critical_summary.merge(
        m,
        on="candidate_id",
        how="left",
        validate="one_to_one",
    )

reference_row = critical_summary[
    critical_summary["candidate_id"].eq(CURRENT_CANDIDATE_ID)
].copy()

candidate_comparison = critical_summary.copy()

if not reference_row.empty:
    ref = reference_row.iloc[0].to_dict()

    delta_cols = [
        "trades",
        "total_pnl",
        "profit_factor",
        "avg_pnl_per_day",
        "avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",
        "front_trades",
        "front_avg_pnl_per_day",
        "front_avg_pnl_per_day_2025",
        "front_avg_pnl_per_day_2026",
        "front_profit_factor_pnl_per_day",
        "front_pnl_per_day_drawdown",
        "middle_trades",
        "middle_avg_pnl_per_day",
        "middle_avg_pnl_per_day_2025",
        "middle_avg_pnl_per_day_2026",
        "middle_profit_factor_pnl_per_day",
        "middle_pnl_per_day_drawdown",
        "tenor_18_trades",
        "tenor_18_avg_pnl_per_day",
        "tenor_18_avg_pnl_per_day_2026",
        "tenor_21_avg_pnl_per_day",
        "tenor_21_avg_pnl_per_day_2026",
        "tenor_24_avg_pnl_per_day",
        "tenor_24_avg_pnl_per_day_2026",
    ]

    for c in delta_cols:
        if c in candidate_comparison.columns and c in ref:
            candidate_comparison[f"delta_{c}_vs_current"] = candidate_comparison[c] - ref[c]

candidate_comparison = candidate_comparison.sort_values(
    "avg_pnl_per_day",
    ascending=False,
).reset_index(drop=True)

# ======================================================================================
# 7. Reassignment attribution
# ======================================================================================

primary_selected = selected_trade_panel[
    selected_trade_panel["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

front_pair = pair_bucket_selection(primary_selected, "Front")
middle_pair = pair_bucket_selection(primary_selected, "Middle")

front_pair["move18_selected_18d"] = front_pair["move18_tenor"].eq(18)
front_pair["current_selected_12_15"] = front_pair["current_tenor"].isin([12, 15])
front_pair["current_no_front"] = front_pair["current_tenor"].isna()

front_pair["front_substitution_type"] = np.select(
    [
        front_pair["move18_selected_18d"] & front_pair["current_selected_12_15"],
        front_pair["move18_selected_18d"] & front_pair["current_no_front"],
        (~front_pair["move18_selected_18d"]) & front_pair["current_selected_12_15"] & front_pair["move18_tenor"].eq(front_pair["current_tenor"]),
        (~front_pair["move18_selected_18d"]) & front_pair["current_selected_12_15"] & (~front_pair["move18_tenor"].eq(front_pair["current_tenor"])),
    ],
    [
        "move18_18d_replaces_current_12_15",
        "move18_18d_replaces_no_current_front",
        "same_12_15_front_tenor",
        "different_12_15_front_tenor",
    ],
    default="other_or_no_change",
)

middle_pair["current_selected_18d"] = middle_pair["current_tenor"].eq(18)
middle_pair["move18_selected_21_24"] = middle_pair["move18_tenor"].isin([21, 24])
middle_pair["move18_no_middle"] = middle_pair["move18_tenor"].isna()

middle_pair["middle_substitution_type"] = np.select(
    [
        middle_pair["current_selected_18d"] & middle_pair["move18_selected_21_24"],
        middle_pair["current_selected_18d"] & middle_pair["move18_no_middle"],
        (~middle_pair["current_selected_18d"]) & middle_pair["move18_selected_21_24"] & middle_pair["move18_tenor"].eq(middle_pair["current_tenor"]),
        (~middle_pair["current_selected_18d"]) & middle_pair["move18_selected_21_24"] & (~middle_pair["move18_tenor"].eq(middle_pair["current_tenor"])),
    ],
    [
        "current_18d_middle_to_move18_21_24",
        "current_18d_middle_to_no_middle",
        "same_21_24_middle_tenor",
        "different_21_24_middle_tenor",
    ],
    default="other_or_no_change",
)

front_substitution_summary = summarize_delta(
    front_pair,
    group_cols=["front_substitution_type"],
)

middle_substitution_summary = summarize_delta(
    middle_pair,
    group_cols=["middle_substitution_type"],
)

front_18d_replaces_12_15 = front_pair[
    front_pair["front_substitution_type"].eq("move18_18d_replaces_current_12_15")
].sort_values("delta_pnl_per_day_move18_minus_current", ascending=False).reset_index(drop=True)

front_18d_replaces_no_front = front_pair[
    front_pair["front_substitution_type"].eq("move18_18d_replaces_no_current_front")
].sort_values("delta_pnl_per_day_move18_minus_current", ascending=False).reset_index(drop=True)

middle_18d_to_21_24 = middle_pair[
    middle_pair["middle_substitution_type"].eq("current_18d_middle_to_move18_21_24")
].sort_values("delta_pnl_per_day_move18_minus_current", ascending=False).reset_index(drop=True)

middle_18d_to_no_middle = middle_pair[
    middle_pair["middle_substitution_type"].eq("current_18d_middle_to_no_middle")
].sort_values("delta_pnl_per_day_move18_minus_current", ascending=False).reset_index(drop=True)

# 18D selected trade detail under both candidates.
selected_18d_detail = primary_selected[
    primary_selected["tenor"].eq(18)
].sort_values(["candidate_id", "trade_date"]).reset_index(drop=True)

# Pass diagnostics.
pass_diagnostics_rows = []

for cid in CANDIDATE_IDS:
    p = panel[panel["candidate_id"].eq(cid)].copy()
    q = qualified_complete[qualified_complete["candidate_id"].eq(cid)].copy()

    row = {
        "candidate_id": cid,
        "candidate_subfamily": candidate_grid_wide.loc[candidate_grid_wide["candidate_id"].eq(cid), "candidate_subfamily"].iloc[0],
        "panel_rows": int(len(p)),
        "pass_rows_all_dates": int(p["candidate_pass"].sum()),
        "pass_rows_with_outcomes": int(len(q)),
    }

    for bucket in ["Front", "Middle", "Back"]:
        row[f"{bucket}_pass_rows_with_outcomes"] = int(
            len(q[q["effective_tenor_bucket"].eq(bucket)])
        )

    for tenor in ALL_TENORS:
        row[f"tenor_{tenor}_pass_rows_with_outcomes"] = int(
            len(q[q["tenor"].eq(tenor)])
        )

    pass_diagnostics_rows.append(row)

pass_diagnostics = pd.DataFrame(pass_diagnostics_rows)

# ======================================================================================
# 8. Save outputs
# ======================================================================================

candidate_grid_wide_csv_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_grid_wide_{CELL9R_TIMESTAMP}.csv"
candidate_grid_wide_parquet_path = OUT_PROCESSED_DIR / f"09R_18d_bucket_reassignment_grid_wide_{CELL9R_TIMESTAMP}.parquet"
candidate_grid_long_csv_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_grid_long_{CELL9R_TIMESTAMP}.csv"
candidate_grid_long_parquet_path = OUT_PROCESSED_DIR / f"09R_18d_bucket_reassignment_grid_long_{CELL9R_TIMESTAMP}.parquet"

summary_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_summary_{CELL9R_TIMESTAMP}.csv"
critical_summary_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_critical_summary_{CELL9R_TIMESTAMP}.csv"
candidate_comparison_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_vs_current_{CELL9R_TIMESTAMP}.csv"
by_bucket_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_by_effective_bucket_{CELL9R_TIMESTAMP}.csv"
by_tenor_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_by_tenor_{CELL9R_TIMESTAMP}.csv"
by_year_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_by_year_{CELL9R_TIMESTAMP}.csv"
by_tenor_year_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_by_tenor_year_{CELL9R_TIMESTAMP}.csv"
pass_diagnostics_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_pass_diagnostics_{CELL9R_TIMESTAMP}.csv"

front_pair_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_front_pair_dates_{CELL9R_TIMESTAMP}.csv"
middle_pair_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_middle_pair_dates_{CELL9R_TIMESTAMP}.csv"
front_substitution_summary_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_front_substitution_summary_{CELL9R_TIMESTAMP}.csv"
middle_substitution_summary_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_middle_substitution_summary_{CELL9R_TIMESTAMP}.csv"
front_18d_replaces_12_15_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_front_18d_replaces_12_15_{CELL9R_TIMESTAMP}.csv"
front_18d_replaces_no_front_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_front_18d_replaces_no_front_{CELL9R_TIMESTAMP}.csv"
middle_18d_to_21_24_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_middle_18d_to_21_24_{CELL9R_TIMESTAMP}.csv"
middle_18d_to_no_middle_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_middle_18d_to_no_middle_{CELL9R_TIMESTAMP}.csv"
selected_18d_detail_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_selected_18d_detail_{CELL9R_TIMESTAMP}.csv"

selected_trade_panel_path = OUT_PROCESSED_DIR / f"09R_18d_bucket_reassignment_selected_trade_panel_{CELL9R_TIMESTAMP}.parquet"
selected_trade_panel_csv_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_selected_trade_panel_{CELL9R_TIMESTAMP}.csv"

validation_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_validation_{CELL9R_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"09R_18d_bucket_reassignment_manifest_{CELL9R_TIMESTAMP}.json"

candidate_grid_wide.to_csv(candidate_grid_wide_csv_path, index=False)
candidate_grid_wide.to_parquet(candidate_grid_wide_parquet_path, index=False)
candidate_grid_long.to_csv(candidate_grid_long_csv_path, index=False)
candidate_grid_long.to_parquet(candidate_grid_long_parquet_path, index=False)

summary.to_csv(summary_path, index=False)
critical_summary.to_csv(critical_summary_path, index=False)
candidate_comparison.to_csv(candidate_comparison_path, index=False)
by_bucket.to_csv(by_bucket_path, index=False)
by_tenor.to_csv(by_tenor_path, index=False)
by_year.to_csv(by_year_path, index=False)
by_tenor_year.to_csv(by_tenor_year_path, index=False)
pass_diagnostics.to_csv(pass_diagnostics_path, index=False)

front_pair.to_csv(front_pair_path, index=False)
middle_pair.to_csv(middle_pair_path, index=False)
front_substitution_summary.to_csv(front_substitution_summary_path, index=False)
middle_substitution_summary.to_csv(middle_substitution_summary_path, index=False)
front_18d_replaces_12_15.to_csv(front_18d_replaces_12_15_path, index=False)
front_18d_replaces_no_front.to_csv(front_18d_replaces_no_front_path, index=False)
middle_18d_to_21_24.to_csv(middle_18d_to_21_24_path, index=False)
middle_18d_to_no_middle.to_csv(middle_18d_to_no_middle_path, index=False)
selected_18d_detail.to_csv(selected_18d_detail_path, index=False)

selected_trade_panel.to_parquet(selected_trade_panel_path, index=False)
selected_trade_panel.to_csv(selected_trade_panel_csv_path, index=False)

# ======================================================================================
# 9. Validation
# ======================================================================================

validation_rows = []

required_base_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

missing_base_cols = [c for c in required_base_cols if c not in base.columns]

models_found = sorted(base["model_spec"].dropna().unique().tolist())

candidate_count = int(candidate_grid_wide["candidate_id"].nunique())
expected_long_rows = candidate_count * len(ALL_TENORS)

old_forecast_threshold_cols = [
    c for c in list(candidate_grid_wide.columns) + list(candidate_grid_long.columns)
    if "forecast_vol_pct_min" in c
]

bad_forecast_threshold_flag = candidate_grid_long[
    candidate_grid_long["forecast_vol_pct_threshold_used"].astype(bool)
]

bad_rv21d_contract_flag = candidate_grid_long[
    ~candidate_grid_long["uses_rv21d_threshold_contract"].astype(bool)
]

bad_z_equality = candidate_grid_long[
    candidate_grid_long["model_vrp_z_3m"].ne(candidate_grid_long["model_vrp_z_1y"])
]

bad_9d_include = candidate_grid_long[
    candidate_grid_long["tenor"].eq(9)
    & candidate_grid_long["include_tenor"].astype(bool)
]

move18_18d_row = candidate_grid_long[
    candidate_grid_long["candidate_id"].eq(MOVE18_CANDIDATE_ID)
    & candidate_grid_long["tenor"].eq(18)
]

current_18d_row = candidate_grid_long[
    candidate_grid_long["candidate_id"].eq(CURRENT_CANDIDATE_ID)
    & candidate_grid_long["tenor"].eq(18)
]

bad_move18_bucket = not (
    len(move18_18d_row) == 1
    and move18_18d_row["effective_tenor_bucket"].iloc[0] == "Front"
    and bool(move18_18d_row["include_tenor"].iloc[0])
)

bad_current_18d_bucket = not (
    len(current_18d_row) == 1
    and current_18d_row["effective_tenor_bucket"].iloc[0] == "Middle"
    and bool(current_18d_row["include_tenor"].iloc[0])
)

back_rules = candidate_grid_long[candidate_grid_long["is_back_locked"].astype(bool)].copy()

back_reference = (
    back_rules[
        [
            "model_vrp_log",
            "model_vrp_z",
            "RSI14_max",
            "rv21d_vol_pct_min",
        ]
    ]
    .drop_duplicates()
)

bad_back_locked = len(back_reference) != 1

summary_universes_found = sorted(summary["selection_universe"].dropna().unique().tolist())
summary_candidates_found = sorted(summary["candidate_id"].dropna().unique().tolist())
expected_summary_rows = candidate_count * len(SELECTION_UNIVERSES)

bad_normalized_pnl = base[
    base["normalized_pnl_dollars"].notna()
    & base["normalized_pnl_per_day"].notna()
    & (
        (
            base["normalized_pnl_dollars"] / base["tenor"].replace(0, np.nan)
        )
        - base["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
]

add_validation(
    validation_rows,
    "base_panel_loaded",
    "PASS" if len(base) > 0 else "FAIL",
    f"rows={len(base):,}; path={base_panel_path}",
)

add_validation(
    validation_rows,
    "required_base_columns_available",
    "PASS" if not missing_base_cols else "FAIL",
    f"missing={missing_base_cols}",
)

add_validation(
    validation_rows,
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    validation_rows,
    "cell8_middle_grid_loaded",
    "PASS" if len(middle_grid_long) > 0 else "FAIL",
    f"rows={len(middle_grid_long):,}; path={middle_grid_long_path}",
)

add_validation(
    validation_rows,
    "candidate_count",
    "PASS" if candidate_count == 2 else "FAIL",
    f"candidate_count={candidate_count:,}; expected=2",
)

add_validation(
    validation_rows,
    "long_grid_all_tenors_per_candidate",
    "PASS" if len(candidate_grid_long) == expected_long_rows else "FAIL",
    f"long_rows={len(candidate_grid_long):,}; expected={expected_long_rows:,}",
)

add_validation(
    validation_rows,
    "current_18d_remains_middle",
    "PASS" if not bad_current_18d_bucket else "FAIL",
    f"current_18d_row={current_18d_row.to_dict('records')}",
)

add_validation(
    validation_rows,
    "move18_18d_is_front",
    "PASS" if not bad_move18_bucket else "FAIL",
    f"move18_18d_row={move18_18d_row.to_dict('records')}",
)

add_validation(
    validation_rows,
    "nine_day_excluded_both_candidates",
    "PASS" if bad_9d_include.empty else "FAIL",
    f"bad_rows={len(bad_9d_include):,}",
)

add_validation(
    validation_rows,
    "uses_rv21d_threshold_contract",
    "PASS" if "threshold_rv21d_vol_pct_min" in panel.columns and bad_rv21d_contract_flag.empty else "FAIL",
    f"bad_rv21d_contract_rows={len(bad_rv21d_contract_flag):,}",
)

add_validation(
    validation_rows,
    "no_forecast_vol_threshold_columns",
    "PASS" if not old_forecast_threshold_cols else "FAIL",
    f"old_forecast_threshold_cols={old_forecast_threshold_cols}",
)

add_validation(
    validation_rows,
    "forecast_vol_pct_diagnostic_only",
    "PASS" if bad_forecast_threshold_flag.empty else "FAIL",
    f"bad_forecast_threshold_flag_rows={len(bad_forecast_threshold_flag):,}",
)

add_validation(
    validation_rows,
    "z_3m_equals_z_1y_within_effective_bucket_rules",
    "PASS" if bad_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_z_equality):,}",
)

add_validation(
    validation_rows,
    "back_locked_all_candidates",
    "PASS" if not bad_back_locked else "FAIL",
    f"distinct_back_rule_rows={len(back_reference):,}; back_reference={back_reference.to_dict('records')}",
)

add_validation(
    validation_rows,
    "panel_rule_application_created",
    "PASS" if len(panel) > 0 and len(qualified_complete) > 0 else "FAIL",
    f"panel_rows={len(panel):,}; qualified_complete_rows={len(qualified_complete):,}",
)

add_validation(
    validation_rows,
    "all_selection_universes_evaluated",
    "PASS" if summary_universes_found == sorted(SELECTION_UNIVERSES) else "FAIL",
    f"found={summary_universes_found}",
)

add_validation(
    validation_rows,
    "summary_rows_complete",
    "PASS" if len(summary) == expected_summary_rows else "FAIL",
    f"summary_rows={len(summary):,}; expected={expected_summary_rows:,}",
)

add_validation(
    validation_rows,
    "all_candidates_evaluated",
    "PASS" if summary_candidates_found == sorted(CANDIDATE_IDS) else "FAIL",
    f"found={summary_candidates_found}",
)

add_validation(
    validation_rows,
    "critical_summary_created",
    "PASS" if len(critical_summary) == candidate_count else "FAIL",
    f"rows={len(critical_summary):,}; expected={candidate_count:,}",
)

add_validation(
    validation_rows,
    "candidate_comparison_created",
    "PASS" if len(candidate_comparison) == candidate_count else "FAIL",
    f"rows={len(candidate_comparison):,}; expected={candidate_count:,}",
)

add_validation(
    validation_rows,
    "front_reassignment_pairing_created",
    "PASS" if len(front_pair) > 0 else "FAIL",
    f"rows={len(front_pair):,}",
)

add_validation(
    validation_rows,
    "middle_reassignment_pairing_created",
    "PASS" if len(middle_pair) > 0 else "FAIL",
    f"rows={len(middle_pair):,}",
)

add_validation(
    validation_rows,
    "normalized_pnl_per_day_formula",
    "PASS" if bad_normalized_pnl.empty else "FAIL",
    f"bad_rows={len(bad_normalized_pnl):,}",
)

add_validation(
    validation_rows,
    "no_final_lock",
    "PASS",
    "Bucket reassignment test only; no final rule selected.",
)

add_validation(
    validation_rows,
    "no_secondary",
    "PASS",
    "Core only.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed or added.",
)

add_validation(
    validation_rows,
    "no_production_lock",
    "PASS",
    "No production threshold lock performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"].eq("FAIL")]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1_RV21D_repaired_v2.ipynb",
    "cell": "Cell 9R — 18D bucket reassignment test",
    "timestamp": CELL9R_TIMESTAMP,
    "base_panel_path": str(base_panel_path),
    "middle_grid_wide_path": str(middle_grid_wide_path),
    "middle_grid_long_path": str(middle_grid_long_path),
    "candidate_ids": CANDIDATE_IDS,
    "primary_selection_universe": PRIMARY_SELECTION_UNIVERSE,
    "primary_ranking_metric": PRIMARY_RANKING_METRIC,
    "candidate_grid_wide_csv_path": str(candidate_grid_wide_csv_path),
    "candidate_grid_wide_parquet_path": str(candidate_grid_wide_parquet_path),
    "candidate_grid_long_csv_path": str(candidate_grid_long_csv_path),
    "candidate_grid_long_parquet_path": str(candidate_grid_long_parquet_path),
    "summary_path": str(summary_path),
    "critical_summary_path": str(critical_summary_path),
    "candidate_comparison_path": str(candidate_comparison_path),
    "by_bucket_path": str(by_bucket_path),
    "by_tenor_path": str(by_tenor_path),
    "by_year_path": str(by_year_path),
    "by_tenor_year_path": str(by_tenor_year_path),
    "pass_diagnostics_path": str(pass_diagnostics_path),
    "front_pair_path": str(front_pair_path),
    "middle_pair_path": str(middle_pair_path),
    "front_substitution_summary_path": str(front_substitution_summary_path),
    "middle_substitution_summary_path": str(middle_substitution_summary_path),
    "front_18d_replaces_12_15_path": str(front_18d_replaces_12_15_path),
    "front_18d_replaces_no_front_path": str(front_18d_replaces_no_front_path),
    "middle_18d_to_21_24_path": str(middle_18d_to_21_24_path),
    "middle_18d_to_no_middle_path": str(middle_18d_to_no_middle_path),
    "selected_18d_detail_path": str(selected_18d_detail_path),
    "selected_trade_panel_path": str(selected_trade_panel_path),
    "selected_trade_panel_csv_path": str(selected_trade_panel_csv_path),
    "validation_path": str(validation_path),
    "threshold_contract": "rv21d_vol_pct > rv21d_vol_pct_min",
    "forecast_vol_pct_role": "model denominator / diagnostic only, not threshold floor",
    "constraints": [
        "9D remains excluded.",
        "Front ex-9D remains active.",
        "18D is moved into Front only for the reassignment candidate.",
        "Back bucket parameters remain locked.",
        "3m and 1y z-score thresholds are equal within effective bucket rules.",
        "RV21D is the only vol-floor threshold.",
        "forecast_vol_pct is diagnostic only.",
        "No final lock.",
        "No Secondary.",
        "No sizing changes.",
        "No production lock.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 9R validation failed. Review validation output above.")

# ======================================================================================
# 10. Display review
# ======================================================================================

print("=" * 100)
print("Cell 9R — 18D bucket reassignment test")
print("=" * 100)
print(f"Base panel:                         {base_panel_path}")
print(f"Cell 8R grid long:                  {middle_grid_long_path}")
print(f"Candidate count:                    {candidate_count:,}")
print(f"Qualified complete rows:            {len(qualified_complete):,}")
print(f"Selected trade panel rows:          {len(selected_trade_panel):,}")
print(f"Primary universe:                   {PRIMARY_SELECTION_UNIVERSE}")
print(f"Primary metric:                     {PRIMARY_RANKING_METRIC}")
print("Current candidate:                  Front 12/15, Middle 18/21/24, Back 27/30/33, 9D excluded")
print("Move-18 candidate:                  Front 12/15/18, Middle 21/24, Back 27/30/33, 9D excluded")
print("Threshold contract:                 rv21d_vol_pct > rv21d_vol_pct_min")
print("Back locked:                        True")
print("3m/1y z equal within bucket:         True")
print("No final lock:                      True")
print("No Secondary:                       True")
print("No sizing changes:                  True")
print("No production lock:                 True")

print()
print("Validation:")
display(validation)

print()
print("Candidate grid — wide:")
display(candidate_grid_wide)

print()
print("Candidate grid — long:")
display(candidate_grid_long)

critical_display_cols = [
    "candidate_id",
    "candidate_subfamily",
    "trades",
    "unique_trade_dates",
    "total_pnl",
    "profit_factor",
    "avg_pnl_per_day",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "front_trades",
    "front_avg_pnl_per_day",
    "front_avg_pnl_per_day_2025",
    "front_avg_pnl_per_day_2026",
    "front_profit_factor_pnl_per_day",
    "middle_trades",
    "middle_avg_pnl_per_day",
    "middle_avg_pnl_per_day_2025",
    "middle_avg_pnl_per_day_2026",
    "middle_profit_factor_pnl_per_day",
    "tenor_12_trades",
    "tenor_12_avg_pnl_per_day",
    "tenor_12_avg_pnl_per_day_2026",
    "tenor_15_trades",
    "tenor_15_avg_pnl_per_day",
    "tenor_15_avg_pnl_per_day_2026",
    "tenor_18_trades",
    "tenor_18_avg_pnl_per_day",
    "tenor_18_avg_pnl_per_day_2026",
    "tenor_21_trades",
    "tenor_21_avg_pnl_per_day",
    "tenor_21_avg_pnl_per_day_2026",
    "tenor_24_trades",
    "tenor_24_avg_pnl_per_day",
    "tenor_24_avg_pnl_per_day_2026",
]

print()
print("Critical summary — primary universe:")
display(critical_summary[[c for c in critical_display_cols if c in critical_summary.columns]])

comparison_display_cols = [
    "candidate_id",
    "candidate_subfamily",
    "avg_pnl_per_day",
    "delta_avg_pnl_per_day_vs_current",
    "total_pnl",
    "delta_total_pnl_vs_current",
    "profit_factor",
    "delta_profit_factor_vs_current",
    "avg_pnl_per_day_2026",
    "delta_avg_pnl_per_day_2026_vs_current",
    "pnl_per_day_drawdown",
    "delta_pnl_per_day_drawdown_vs_current",
    "worst_20_trade_pnl_per_day_sum",
    "delta_worst_20_trade_pnl_per_day_sum_vs_current",
    "front_avg_pnl_per_day",
    "delta_front_avg_pnl_per_day_vs_current",
    "front_avg_pnl_per_day_2026",
    "delta_front_avg_pnl_per_day_2026_vs_current",
    "middle_avg_pnl_per_day",
    "delta_middle_avg_pnl_per_day_vs_current",
    "middle_avg_pnl_per_day_2026",
    "delta_middle_avg_pnl_per_day_2026_vs_current",
    "tenor_18_trades",
    "delta_tenor_18_trades_vs_current",
    "tenor_18_avg_pnl_per_day",
    "delta_tenor_18_avg_pnl_per_day_vs_current",
    "tenor_18_avg_pnl_per_day_2026",
    "delta_tenor_18_avg_pnl_per_day_2026_vs_current",
]

print()
print("Candidate comparison vs current — primary universe:")
display(candidate_comparison[[c for c in comparison_display_cols if c in candidate_comparison.columns]])

print()
print("By effective bucket:")
display(by_bucket)

print()
print("By tenor:")
display(by_tenor)

print()
print("By year:")
display(by_year)

print()
print("By tenor/year:")
display(by_tenor_year)

print()
print("Pass diagnostics:")
display(pass_diagnostics)

print()
print("Front substitution summary:")
display(front_substitution_summary)

print()
print("Middle substitution summary:")
display(middle_substitution_summary)

print()
print("Dates where moved-18D Front selects 18D instead of current 12D/15D:")
display(front_18d_replaces_12_15)

print()
print("Dates where moved-18D Front selects 18D and current has no Front trade:")
display(front_18d_replaces_no_front)

print()
print("Dates where current Middle selected 18D and moved structure selects 21D/24D:")
display(middle_18d_to_21_24)

print()
print("Dates where current Middle selected 18D and moved structure has no Middle trade:")
display(middle_18d_to_no_middle)

print()
print("Selected 18D detail under both candidates:")
display(selected_18d_detail)

print()
print("Saved outputs:")
for p in [
    candidate_grid_wide_csv_path,
    candidate_grid_wide_parquet_path,
    candidate_grid_long_csv_path,
    candidate_grid_long_parquet_path,
    summary_path,
    critical_summary_path,
    candidate_comparison_path,
    by_bucket_path,
    by_tenor_path,
    by_year_path,
    by_tenor_year_path,
    pass_diagnostics_path,
    front_pair_path,
    middle_pair_path,
    front_substitution_summary_path,
    middle_substitution_summary_path,
    front_18d_replaces_12_15_path,
    front_18d_replaces_no_front_path,
    middle_18d_to_21_24_path,
    middle_18d_to_no_middle_path,
    selected_18d_detail_path,
    selected_trade_panel_path,
    selected_trade_panel_csv_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 9R PASSED")

In [ ]:
# ======================================================================================
# Cell 10R — DTE-level smooth parameter sweep
# ======================================================================================
#
# Objective:
#   Test whether constrained DTE-level parameter smoothing improves on the locked Core
#   bucket structure.
#
# Benchmark:
#   core_dte_smooth_0001 — locked bucket benchmark
#
#       Excluded:
#           9D
#
#       Front:
#           12D / 15D / 18D
#           model_vrp_log > 0.60
#           model_vrp_z_3m > 0.65
#           model_vrp_z_1y > 0.65
#           RSI14 < 70
#           rv21d_vol_pct > 8.5
#
#       Middle:
#           21D / 24D
#           model_vrp_log > 0.65
#           model_vrp_z_3m > 0.70
#           model_vrp_z_1y > 0.70
#           RSI14 < 70
#           rv21d_vol_pct > 8.5
#
#       Back:
#           27D / 30D / 33D
#           model_vrp_log > 0.70
#           model_vrp_z_3m > 0.70
#           model_vrp_z_1y > 0.70
#           RSI14 < 70
#           rv21d_vol_pct > 8.5
#
# Candidate families:
#   0001 — locked bucket benchmark
#   0002 — smooth VRP log curve, mild
#   0003 — smooth VRP log curve, stricter transition
#   0004 — smooth z curve, mild
#   0005 — smooth VRP log + smooth z
#   0006 — 18D slightly stricter only
#   0007 — 24D slightly stricter only
#
# Correct volatility-floor contract:
#       rv21d_vol_pct > rv21d_vol_pct_min
#
# NOT:
#       forecast_vol_pct > forecast_vol_pct_min
#
# Hard constraints:
#   1. 9D remains excluded.
#   2. 18D remains in Front.
#   3. Effective bucket map remains:
#          Front  = 12D / 15D / 18D
#          Middle = 21D / 24D
#          Back   = 27D / 30D / 33D
#   4. Back endpoint remains anchored:
#          27D / 30D / 33D log = 0.70
#          27D / 30D / 33D z   = 0.70
#   5. z3m and z1y are equal at each DTE.
#   6. RSI14 cap remains 70 for all DTEs.
#   7. RV21D floor remains 8.5 for all DTEs.
#   8. DTE curves must be monotone nondecreasing with DTE.
#   9. Each interior DTE parameter must be between/equal neighboring DTEs.
#   10. No Secondary.
#   11. No sizing changes.
#   12. No production lock.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 380)
pd.set_option("display.width", 640)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL10R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]
INCLUDED_CORE_TENORS = [12, 15, 18, 21, 24, 27, 30, 33]
EXCLUDED_CORE_TENORS = [9]

ORIGINAL_TENOR_BUCKET_MAP = {
    9: "Front",
    12: "Front",
    15: "Front",
    18: "Middle",
    21: "Middle",
    24: "Middle",
    27: "Back",
    30: "Back",
    33: "Back",
}

EFFECTIVE_TENOR_BUCKET_MAP = {
    9: "Front",
    12: "Front",
    15: "Front",
    18: "Front",
    21: "Middle",
    24: "Middle",
    27: "Back",
    30: "Back",
    33: "Back",
}

BUCKET_ORDER_MAP = {
    "Front": 1,
    "Middle": 2,
    "Back": 3,
}

BUCKET_CENTER = {
    "Front": 15,
    "Middle": 21,
    "Back": 30,
}

SELECTION_UNIVERSES = [
    "all_qualified_trades",
    "one_trade_per_bucket_per_date_best_rank",
    "one_trade_per_date_best_rank",
]

PRIMARY_SELECTION_UNIVERSE = "one_trade_per_bucket_per_date_best_rank"
PRIMARY_RANKING_METRIC = "avg_pnl_per_day"
BENCHMARK_CANDIDATE_ID = "core_dte_smooth_0001"

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def between_or_equal(x, a, b, tol=1e-12):
    lo = min(float(a), float(b)) - tol
    hi = max(float(a), float(b)) + tol
    return lo <= float(x) <= hi


def is_monotone_nondecreasing(values, tol=1e-12):
    vals = [float(v) for v in values]
    return all(vals[i] <= vals[i + 1] + tol for i in range(len(vals) - 1))


def interior_between_neighbors(values, tol=1e-12):
    vals = [float(v) for v in values]
    if len(vals) <= 2:
        return True

    for i in range(1, len(vals) - 1):
        if not between_or_equal(vals[i], vals[i - 1], vals[i + 1], tol=tol):
            return False

    return True


def add_selection_ranks(df):
    d = df.copy()

    if d.empty:
        return d

    d["bucket_center_tenor"] = d["effective_tenor_bucket"].map(BUCKET_CENTER)
    d["distance_to_bucket_center"] = (d["tenor"] - d["bucket_center_tenor"]).abs()

    bucket_group = ["candidate_id", "trade_date", "effective_tenor_bucket"]

    d["rank_z_1y_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_bucket_date"] = d[
        [
            "rank_z_1y_in_bucket_date",
            "rank_z_3m_in_bucket_date",
            "rank_vrp_log_in_bucket_date",
        ]
    ].mean(axis=1)

    date_group = ["candidate_id", "trade_date"]

    d["rank_z_1y_in_date"] = (
        d.groupby(date_group)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_date"] = (
        d.groupby(date_group)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_date"] = (
        d.groupby(date_group)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_date"] = d[
        [
            "rank_z_1y_in_date",
            "rank_z_3m_in_date",
            "rank_vrp_log_in_date",
        ]
    ].mean(axis=1)

    return d


def select_trade_universes(qualified):
    if qualified.empty:
        return {
            "all_qualified_trades": qualified.copy(),
            "one_trade_per_bucket_per_date_best_rank": qualified.copy(),
            "one_trade_per_date_best_rank": qualified.copy(),
        }

    q = add_selection_ranks(qualified)

    one_trade_per_bucket_date = (
        q.sort_values(
            [
                "candidate_id",
                "trade_date",
                "effective_tenor_bucket_order",
                "avg_signal_rank_in_bucket_date",
                "distance_to_bucket_center",
                "rank_z_1y_in_bucket_date",
                "rank_z_3m_in_bucket_date",
                "rank_vrp_log_in_bucket_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True, True],
        )
        .groupby(["candidate_id", "trade_date", "effective_tenor_bucket"], as_index=False)
        .head(1)
        .copy()
    )

    one_trade_per_date = (
        q.sort_values(
            [
                "candidate_id",
                "trade_date",
                "avg_signal_rank_in_date",
                "rank_z_1y_in_date",
                "rank_z_3m_in_date",
                "rank_vrp_log_in_date",
                "distance_to_bucket_center",
                "effective_tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True, True],
        )
        .groupby(["candidate_id", "trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return {
        "all_qualified_trades": q.copy(),
        "one_trade_per_bucket_per_date_best_rank": one_trade_per_bucket_date,
        "one_trade_per_date_best_rank": one_trade_per_date,
    }


def summarize_trade_set(df, group_cols):
    d = df.copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    numeric_cols = [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "forecast_vol_pct",
        "rv21d_vol_pct",
        "threshold_model_vrp_log",
        "threshold_model_vrp_z_3m",
        "threshold_model_vrp_z_1y",
        "threshold_RSI14_max",
        "threshold_rv21d_vol_pct_min",
    ]

    for c in numeric_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d = d.sort_values(["candidate_id", "selection_universe", "trade_date", "tenor"]).copy()

    grouped = d.groupby(group_cols, dropna=False, observed=False)

    rows = []

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {}

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": int(g["trade_date"].nunique()),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        for signal_col in [
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "threshold_model_vrp_log",
            "threshold_model_vrp_z_3m",
            "threshold_model_vrp_z_1y",
            "threshold_RSI14_max",
            "threshold_rv21d_vol_pct_min",
        ]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        rows.append(row)

    return pd.DataFrame(rows)


def attach_candidate_design(summary_df, candidate_grid_wide):
    if summary_df.empty:
        return summary_df

    return summary_df.merge(
        candidate_grid_wide,
        on="candidate_id",
        how="left",
        validate="many_to_one",
    )


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


# ======================================================================================
# 2. Load inputs
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01R_unified_fds_signal_base_panel_with_rv21d_*.parquet",
    required=True,
)

cell9_grid_long_path = latest_file(
    OUT_PROCESSED_DIR,
    "09R_18d_bucket_reassignment_grid_long_*.parquet",
    required=True,
)

cell9_grid_wide_path = latest_file(
    OUT_PROCESSED_DIR,
    "09R_18d_bucket_reassignment_grid_wide_*.parquet",
    required=True,
)

base = pd.read_parquet(base_panel_path)
cell9_grid_long = pd.read_parquet(cell9_grid_long_path)
cell9_grid_wide = pd.read_parquet(cell9_grid_wide_path)

base["trade_date"] = pd.to_datetime(base["trade_date"], errors="coerce").dt.normalize()

for c in [
    "tenor",
    "tenor_bucket_order",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")

for c in [
    "tenor",
    "model_vrp_log",
    "model_vrp_z",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14_max",
    "rv21d_vol_pct_min",
]:
    if c in cell9_grid_long.columns:
        cell9_grid_long[c] = pd.to_numeric(cell9_grid_long[c], errors="coerce")

# ======================================================================================
# 3. Build constrained DTE-level candidate grid
# ======================================================================================

candidate_specs = [
    {
        "candidate_id": "core_dte_smooth_0001",
        "candidate_subfamily": "locked_bucket_benchmark",
        "candidate_description": "Locked bucket benchmark: 9D excluded; 12/15/18 Front, 21/24 Middle, 27/30/33 Back.",
        "vrp_log_by_tenor": {
            12: 0.600,
            15: 0.600,
            18: 0.600,
            21: 0.650,
            24: 0.650,
            27: 0.700,
            30: 0.700,
            33: 0.700,
        },
        "z_by_tenor": {
            12: 0.650,
            15: 0.650,
            18: 0.650,
            21: 0.700,
            24: 0.700,
            27: 0.700,
            30: 0.700,
            33: 0.700,
        },
    },
    {
        "candidate_id": "core_dte_smooth_0002",
        "candidate_subfamily": "smooth_vrp_log_mild",
        "candidate_description": "Smooth VRP log only: 18D=0.625 and 24D=0.675; z unchanged.",
        "vrp_log_by_tenor": {
            12: 0.600,
            15: 0.600,
            18: 0.625,
            21: 0.650,
            24: 0.675,
            27: 0.700,
            30: 0.700,
            33: 0.700,
        },
        "z_by_tenor": {
            12: 0.650,
            15: 0.650,
            18: 0.650,
            21: 0.700,
            24: 0.700,
            27: 0.700,
            30: 0.700,
            33: 0.700,
        },
    },
    {
        "candidate_id": "core_dte_smooth_0003",
        "candidate_subfamily": "smooth_vrp_log_stricter_transition",
        "candidate_description": "Stricter smooth VRP log transition: 15D=0.625, 18D=0.650, 24D=0.675; z unchanged.",
        "vrp_log_by_tenor": {
            12: 0.600,
            15: 0.625,
            18: 0.650,
            21: 0.650,
            24: 0.675,
            27: 0.700,
            30: 0.700,
            33: 0.700,
        },
        "z_by_tenor": {
            12: 0.650,
            15: 0.650,
            18: 0.650,
            21: 0.700,
            24: 0.700,
            27: 0.700,
            30: 0.700,
            33: 0.700,
        },
    },
    {
        "candidate_id": "core_dte_smooth_0004",
        "candidate_subfamily": "smooth_z_mild",
        "candidate_description": "Smooth z only: 18D z=0.675; VRP log unchanged.",
        "vrp_log_by_tenor": {
            12: 0.600,
            15: 0.600,
            18: 0.600,
            21: 0.650,
            24: 0.650,
            27: 0.700,
            30: 0.700,
            33: 0.700,
        },
        "z_by_tenor": {
            12: 0.650,
            15: 0.650,
            18: 0.675,
            21: 0.700,
            24: 0.700,
            27: 0.700,
            30: 0.700,
            33: 0.700,
        },
    },
    {
        "candidate_id": "core_dte_smooth_0005",
        "candidate_subfamily": "smooth_vrp_log_and_z",
        "candidate_description": "Smooth VRP log mild plus smooth 18D z=0.675.",
        "vrp_log_by_tenor": {
            12: 0.600,
            15: 0.600,
            18: 0.625,
            21: 0.650,
            24: 0.675,
            27: 0.700,
            30: 0.700,
            33: 0.700,
        },
        "z_by_tenor": {
            12: 0.650,
            15: 0.650,
            18: 0.675,
            21: 0.700,
            24: 0.700,
            27: 0.700,
            30: 0.700,
            33: 0.700,
        },
    },
    {
        "candidate_id": "core_dte_smooth_0006",
        "candidate_subfamily": "eighteen_d_log_0_625_only",
        "candidate_description": "Only tighten 18D VRP log from 0.600 to 0.625; all else benchmark.",
        "vrp_log_by_tenor": {
            12: 0.600,
            15: 0.600,
            18: 0.625,
            21: 0.650,
            24: 0.650,
            27: 0.700,
            30: 0.700,
            33: 0.700,
        },
        "z_by_tenor": {
            12: 0.650,
            15: 0.650,
            18: 0.650,
            21: 0.700,
            24: 0.700,
            27: 0.700,
            30: 0.700,
            33: 0.700,
        },
    },
    {
        "candidate_id": "core_dte_smooth_0007",
        "candidate_subfamily": "twentyfour_d_log_0_675_only",
        "candidate_description": "Only tighten 24D VRP log from 0.650 to 0.675; all else benchmark.",
        "vrp_log_by_tenor": {
            12: 0.600,
            15: 0.600,
            18: 0.600,
            21: 0.650,
            24: 0.675,
            27: 0.700,
            30: 0.700,
            33: 0.700,
        },
        "z_by_tenor": {
            12: 0.650,
            15: 0.650,
            18: 0.650,
            21: 0.700,
            24: 0.700,
            27: 0.700,
            30: 0.700,
            33: 0.700,
        },
    },
]

wide_rows = []
long_rows = []

for spec in candidate_specs:
    cid = spec["candidate_id"]
    vrp_curve = spec["vrp_log_by_tenor"]
    z_curve = spec["z_by_tenor"]

    vrp_values = [vrp_curve[t] for t in INCLUDED_CORE_TENORS]
    z_values = [z_curve[t] for t in INCLUDED_CORE_TENORS]

    if not is_monotone_nondecreasing(vrp_values):
        raise RuntimeError(f"{cid} VRP log curve is not monotone nondecreasing: {vrp_curve}")

    if not is_monotone_nondecreasing(z_values):
        raise RuntimeError(f"{cid} z curve is not monotone nondecreasing: {z_curve}")

    if not interior_between_neighbors(vrp_values):
        raise RuntimeError(f"{cid} VRP log curve violates neighbor-between constraint: {vrp_curve}")

    if not interior_between_neighbors(z_values):
        raise RuntimeError(f"{cid} z curve violates neighbor-between constraint: {z_curve}")

    if not all(abs(vrp_curve[t] - 0.700) < 1e-12 for t in [27, 30, 33]):
        raise RuntimeError(f"{cid} Back VRP log endpoint not anchored at 0.70.")

    if not all(abs(z_curve[t] - 0.700) < 1e-12 for t in [27, 30, 33]):
        raise RuntimeError(f"{cid} Back z endpoint not anchored at 0.70.")

    wide_row = {
        "candidate_id": cid,
        "candidate_family": "core_dte_smooth_parameter_sweep",
        "candidate_subfamily": spec["candidate_subfamily"],
        "candidate_description": spec["candidate_description"],
        "locked_model_spec": LOCKED_MODEL_SPEC,
        "excluded_tenors": "9",
        "front_tenors": "12,15,18",
        "middle_tenors": "21,24",
        "back_tenors": "27,30,33",
        "include_9d": False,
        "eighteen_d_in_front": True,
        "RSI14_max_all": 70.0,
        "rv21d_vol_pct_min_all": 8.5,
        "z3m_equals_z1y": True,
        "vrp_log_monotone_nondecreasing": True,
        "z_monotone_nondecreasing": True,
        "interior_between_neighbors": True,
        "back_endpoint_anchored": True,
        "uses_rv21d_threshold_contract": True,
        "forecast_vol_pct_threshold_used": False,
        "final_lock": False,
    }

    for tenor in INCLUDED_CORE_TENORS:
        wide_row[f"tenor_{tenor}_model_vrp_log"] = float(vrp_curve[tenor])
        wide_row[f"tenor_{tenor}_model_vrp_z"] = float(z_curve[tenor])
        wide_row[f"tenor_{tenor}_RSI14_max"] = 70.0
        wide_row[f"tenor_{tenor}_rv21d_vol_pct_min"] = 8.5

    wide_rows.append(wide_row)

    for tenor in ALL_TENORS:
        original_bucket = ORIGINAL_TENOR_BUCKET_MAP[tenor]
        effective_bucket = EFFECTIVE_TENOR_BUCKET_MAP[tenor]
        include_tenor = tenor != 9

        if tenor == 9:
            model_vrp_log = float(vrp_curve[12])
            model_vrp_z = float(z_curve[12])
            rule_type = "excluded_9d_uses_front_reference_for_schema_only"
        else:
            model_vrp_log = float(vrp_curve[tenor])
            model_vrp_z = float(z_curve[tenor])
            rule_type = "dte_specific_smooth_rule"

        long_rows.append({
            "candidate_id": cid,
            "candidate_family": "core_dte_smooth_parameter_sweep",
            "candidate_subfamily": spec["candidate_subfamily"],
            "candidate_description": spec["candidate_description"],
            "locked_model_spec": LOCKED_MODEL_SPEC,

            "tenor": int(tenor),
            "original_tenor_bucket": original_bucket,
            "original_tenor_bucket_order": BUCKET_ORDER_MAP[original_bucket],
            "effective_tenor_bucket": effective_bucket,
            "effective_tenor_bucket_order": BUCKET_ORDER_MAP[effective_bucket],

            "include_tenor": bool(include_tenor),
            "rule_type": rule_type,

            "model_vrp_log": model_vrp_log,
            "model_vrp_z": model_vrp_z,
            "model_vrp_z_3m": model_vrp_z,
            "model_vrp_z_1y": model_vrp_z,
            "RSI14_max": 70.0,
            "rv21d_vol_pct_min": 8.5,

            "is_9d_excluded": tenor == 9,
            "is_18d": tenor == 18,
            "is_18d_in_front": tenor == 18 and effective_bucket == "Front",
            "is_front_effective": effective_bucket == "Front",
            "is_middle_effective": effective_bucket == "Middle",
            "is_back_effective": effective_bucket == "Back",
            "is_back_endpoint": tenor in [27, 30, 33],
            "uses_rv21d_threshold_contract": True,
            "forecast_vol_pct_threshold_used": False,
            "final_lock": False,
        })

candidate_grid_wide = pd.DataFrame(wide_rows).sort_values("candidate_id").reset_index(drop=True)
candidate_grid_long = pd.DataFrame(long_rows).sort_values(["candidate_id", "tenor"]).reset_index(drop=True)

# ======================================================================================
# 4. Apply candidate rules
# ======================================================================================

thresholds = candidate_grid_long.rename(columns={
    "model_vrp_log": "threshold_model_vrp_log",
    "model_vrp_z": "threshold_model_vrp_z",
    "model_vrp_z_3m": "threshold_model_vrp_z_3m",
    "model_vrp_z_1y": "threshold_model_vrp_z_1y",
    "RSI14_max": "threshold_RSI14_max",
    "rv21d_vol_pct_min": "threshold_rv21d_vol_pct_min",
})

threshold_cols = [
    "candidate_id",
    "candidate_family",
    "candidate_subfamily",
    "candidate_description",
    "tenor",
    "original_tenor_bucket",
    "original_tenor_bucket_order",
    "effective_tenor_bucket",
    "effective_tenor_bucket_order",
    "include_tenor",
    "rule_type",
    "threshold_model_vrp_log",
    "threshold_model_vrp_z",
    "threshold_model_vrp_z_3m",
    "threshold_model_vrp_z_1y",
    "threshold_RSI14_max",
    "threshold_rv21d_vol_pct_min",
    "is_9d_excluded",
    "is_18d",
    "is_18d_in_front",
    "is_front_effective",
    "is_middle_effective",
    "is_back_effective",
]

base_for_join = base.copy()

base_for_join = base_for_join.rename(columns={
    "tenor_bucket": "source_tenor_bucket",
    "tenor_bucket_order": "source_tenor_bucket_order",
})

panel = base_for_join.merge(
    thresholds[threshold_cols],
    on="tenor",
    how="inner",
    validate="many_to_many",
)

required_signal_available = (
    panel["include_tenor"].astype(bool)
    & panel["model_vrp_log"].notna()
    & panel["model_vrp_z_3m"].notna()
    & panel["model_vrp_z_1y"].notna()
    & panel["RSI14"].notna()
    & panel["rv21d_vol_pct"].notna()
    & panel["threshold_model_vrp_log"].notna()
    & panel["threshold_model_vrp_z_3m"].notna()
    & panel["threshold_model_vrp_z_1y"].notna()
    & panel["threshold_RSI14_max"].notna()
    & panel["threshold_rv21d_vol_pct_min"].notna()
)

outcome_available = (
    panel["normalized_pnl_dollars"].notna()
    & panel["normalized_pnl_per_day"].notna()
)

pass_mask = (
    required_signal_available
    & (panel["model_vrp_log"] > panel["threshold_model_vrp_log"])
    & (panel["model_vrp_z_3m"] > panel["threshold_model_vrp_z_3m"])
    & (panel["model_vrp_z_1y"] > panel["threshold_model_vrp_z_1y"])
    & (panel["RSI14"] < panel["threshold_RSI14_max"])
    & (panel["rv21d_vol_pct"] > panel["threshold_rv21d_vol_pct_min"])
)

panel["candidate_pass"] = pass_mask

qualified_all_dates = panel[pass_mask].copy()
qualified_complete = panel[pass_mask & outcome_available].copy()

universes = select_trade_universes(qualified_complete)

selected_trade_panel = pd.concat(
    [df.assign(selection_universe=name) for name, df in universes.items()],
    ignore_index=True,
)

# ======================================================================================
# 5. Summaries
# ======================================================================================

summary_frames = []
bucket_frames = []
tenor_frames = []
year_frames = []
tenor_year_frames = []

for universe_name, df_u in universes.items():
    df_u = df_u.assign(selection_universe=universe_name).copy()

    summary_frames.append(
        summarize_trade_set(
            df_u,
            group_cols=["candidate_id", "candidate_subfamily", "selection_universe"],
        )
    )

    bucket_frames.append(
        summarize_trade_set(
            df_u,
            group_cols=[
                "candidate_id",
                "candidate_subfamily",
                "selection_universe",
                "effective_tenor_bucket",
                "effective_tenor_bucket_order",
            ],
        )
    )

    tenor_frames.append(
        summarize_trade_set(
            df_u,
            group_cols=[
                "candidate_id",
                "candidate_subfamily",
                "selection_universe",
                "tenor",
                "original_tenor_bucket",
                "effective_tenor_bucket",
                "effective_tenor_bucket_order",
            ],
        )
    )

    y = df_u.copy()
    y["year"] = y["trade_date"].dt.year.astype(int)

    year_frames.append(
        summarize_trade_set(
            y,
            group_cols=["candidate_id", "candidate_subfamily", "selection_universe", "year"],
        )
    )

    tenor_year_frames.append(
        summarize_trade_set(
            y,
            group_cols=[
                "candidate_id",
                "candidate_subfamily",
                "selection_universe",
                "tenor",
                "original_tenor_bucket",
                "effective_tenor_bucket",
                "year",
            ],
        )
    )

summary = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
by_bucket = pd.concat(bucket_frames, ignore_index=True) if bucket_frames else pd.DataFrame()
by_tenor = pd.concat(tenor_frames, ignore_index=True) if tenor_frames else pd.DataFrame()
by_year = pd.concat(year_frames, ignore_index=True) if year_frames else pd.DataFrame()
by_tenor_year = pd.concat(tenor_year_frames, ignore_index=True) if tenor_year_frames else pd.DataFrame()

summary = attach_candidate_design(summary, candidate_grid_wide)
by_bucket = attach_candidate_design(by_bucket, candidate_grid_wide)
by_tenor = attach_candidate_design(by_tenor, candidate_grid_wide)
by_year = attach_candidate_design(by_year, candidate_grid_wide)
by_tenor_year = attach_candidate_design(by_tenor_year, candidate_grid_wide)

summary = summary.sort_values(
    ["selection_universe", "avg_pnl_per_day"],
    ascending=[True, False],
).reset_index(drop=True)

by_bucket = by_bucket.sort_values(
    ["selection_universe", "candidate_id", "effective_tenor_bucket_order"],
).reset_index(drop=True)

by_tenor = by_tenor.sort_values(
    ["selection_universe", "candidate_id", "tenor"],
).reset_index(drop=True)

by_year = by_year.sort_values(
    ["selection_universe", "candidate_id", "year"],
).reset_index(drop=True)

by_tenor_year = by_tenor_year.sort_values(
    ["selection_universe", "candidate_id", "tenor", "year"],
).reset_index(drop=True)

# ======================================================================================
# 6. Critical primary summary and benchmark deltas
# ======================================================================================

primary_summary = summary[
    summary["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

primary_bucket = by_bucket[
    by_bucket["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

primary_tenor = by_tenor[
    by_tenor["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

def extract_bucket_metric(bucket_name, metric, output_name):
    return (
        primary_bucket[primary_bucket["effective_tenor_bucket"].eq(bucket_name)][["candidate_id", metric]]
        .rename(columns={metric: output_name})
    )


def extract_tenor_metric(tenor, metric, output_name):
    return (
        primary_tenor[primary_tenor["tenor"].eq(tenor)][["candidate_id", metric]]
        .rename(columns={metric: output_name})
    )


critical_summary = primary_summary.copy()

critical_merges = [
    extract_bucket_metric("Front", "trades", "front_trades"),
    extract_bucket_metric("Front", "avg_pnl_per_day", "front_avg_pnl_per_day"),
    extract_bucket_metric("Front", "avg_pnl_per_day_2025", "front_avg_pnl_per_day_2025"),
    extract_bucket_metric("Front", "avg_pnl_per_day_2026", "front_avg_pnl_per_day_2026"),
    extract_bucket_metric("Front", "profit_factor_pnl_per_day", "front_profit_factor_pnl_per_day"),
    extract_bucket_metric("Front", "pnl_per_day_drawdown", "front_pnl_per_day_drawdown"),
    extract_bucket_metric("Front", "worst_20_trade_pnl_per_day_sum", "front_worst_20_trade_pnl_per_day_sum"),

    extract_bucket_metric("Middle", "trades", "middle_trades"),
    extract_bucket_metric("Middle", "avg_pnl_per_day", "middle_avg_pnl_per_day"),
    extract_bucket_metric("Middle", "avg_pnl_per_day_2025", "middle_avg_pnl_per_day_2025"),
    extract_bucket_metric("Middle", "avg_pnl_per_day_2026", "middle_avg_pnl_per_day_2026"),
    extract_bucket_metric("Middle", "profit_factor_pnl_per_day", "middle_profit_factor_pnl_per_day"),
    extract_bucket_metric("Middle", "pnl_per_day_drawdown", "middle_pnl_per_day_drawdown"),
    extract_bucket_metric("Middle", "worst_20_trade_pnl_per_day_sum", "middle_worst_20_trade_pnl_per_day_sum"),

    extract_bucket_metric("Back", "trades", "back_trades"),
    extract_bucket_metric("Back", "avg_pnl_per_day", "back_avg_pnl_per_day"),
    extract_bucket_metric("Back", "avg_pnl_per_day_2025", "back_avg_pnl_per_day_2025"),
    extract_bucket_metric("Back", "avg_pnl_per_day_2026", "back_avg_pnl_per_day_2026"),
    extract_bucket_metric("Back", "profit_factor_pnl_per_day", "back_profit_factor_pnl_per_day"),

    extract_tenor_metric(12, "trades", "tenor_12_trades"),
    extract_tenor_metric(12, "avg_pnl_per_day", "tenor_12_avg_pnl_per_day"),
    extract_tenor_metric(12, "avg_pnl_per_day_2026", "tenor_12_avg_pnl_per_day_2026"),

    extract_tenor_metric(15, "trades", "tenor_15_trades"),
    extract_tenor_metric(15, "avg_pnl_per_day", "tenor_15_avg_pnl_per_day"),
    extract_tenor_metric(15, "avg_pnl_per_day_2026", "tenor_15_avg_pnl_per_day_2026"),

    extract_tenor_metric(18, "trades", "tenor_18_trades"),
    extract_tenor_metric(18, "avg_pnl_per_day", "tenor_18_avg_pnl_per_day"),
    extract_tenor_metric(18, "avg_pnl_per_day_2026", "tenor_18_avg_pnl_per_day_2026"),

    extract_tenor_metric(21, "trades", "tenor_21_trades"),
    extract_tenor_metric(21, "avg_pnl_per_day", "tenor_21_avg_pnl_per_day"),
    extract_tenor_metric(21, "avg_pnl_per_day_2026", "tenor_21_avg_pnl_per_day_2026"),

    extract_tenor_metric(24, "trades", "tenor_24_trades"),
    extract_tenor_metric(24, "avg_pnl_per_day", "tenor_24_avg_pnl_per_day"),
    extract_tenor_metric(24, "avg_pnl_per_day_2026", "tenor_24_avg_pnl_per_day_2026"),

    extract_tenor_metric(27, "trades", "tenor_27_trades"),
    extract_tenor_metric(27, "avg_pnl_per_day", "tenor_27_avg_pnl_per_day"),
    extract_tenor_metric(27, "avg_pnl_per_day_2026", "tenor_27_avg_pnl_per_day_2026"),

    extract_tenor_metric(30, "trades", "tenor_30_trades"),
    extract_tenor_metric(30, "avg_pnl_per_day", "tenor_30_avg_pnl_per_day"),
    extract_tenor_metric(30, "avg_pnl_per_day_2026", "tenor_30_avg_pnl_per_day_2026"),

    extract_tenor_metric(33, "trades", "tenor_33_trades"),
    extract_tenor_metric(33, "avg_pnl_per_day", "tenor_33_avg_pnl_per_day"),
    extract_tenor_metric(33, "avg_pnl_per_day_2026", "tenor_33_avg_pnl_per_day_2026"),
]

for m in critical_merges:
    critical_summary = critical_summary.merge(
        m,
        on="candidate_id",
        how="left",
        validate="one_to_one",
    )

benchmark_row = critical_summary[
    critical_summary["candidate_id"].eq(BENCHMARK_CANDIDATE_ID)
].copy()

candidate_comparison = critical_summary.copy()

if not benchmark_row.empty:
    ref = benchmark_row.iloc[0].to_dict()

    delta_cols = [
        "trades",
        "total_pnl",
        "profit_factor",
        "avg_pnl_per_day",
        "avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",

        "front_trades",
        "front_avg_pnl_per_day",
        "front_avg_pnl_per_day_2025",
        "front_avg_pnl_per_day_2026",
        "front_profit_factor_pnl_per_day",
        "front_pnl_per_day_drawdown",
        "front_worst_20_trade_pnl_per_day_sum",

        "middle_trades",
        "middle_avg_pnl_per_day",
        "middle_avg_pnl_per_day_2025",
        "middle_avg_pnl_per_day_2026",
        "middle_profit_factor_pnl_per_day",
        "middle_pnl_per_day_drawdown",
        "middle_worst_20_trade_pnl_per_day_sum",

        "back_trades",
        "back_avg_pnl_per_day",
        "back_avg_pnl_per_day_2025",
        "back_avg_pnl_per_day_2026",
        "back_profit_factor_pnl_per_day",

        "tenor_12_trades",
        "tenor_12_avg_pnl_per_day",
        "tenor_12_avg_pnl_per_day_2026",
        "tenor_15_trades",
        "tenor_15_avg_pnl_per_day",
        "tenor_15_avg_pnl_per_day_2026",
        "tenor_18_trades",
        "tenor_18_avg_pnl_per_day",
        "tenor_18_avg_pnl_per_day_2026",
        "tenor_21_trades",
        "tenor_21_avg_pnl_per_day",
        "tenor_21_avg_pnl_per_day_2026",
        "tenor_24_trades",
        "tenor_24_avg_pnl_per_day",
        "tenor_24_avg_pnl_per_day_2026",
        "tenor_27_trades",
        "tenor_27_avg_pnl_per_day",
        "tenor_27_avg_pnl_per_day_2026",
        "tenor_30_trades",
        "tenor_30_avg_pnl_per_day",
        "tenor_30_avg_pnl_per_day_2026",
        "tenor_33_trades",
        "tenor_33_avg_pnl_per_day",
        "tenor_33_avg_pnl_per_day_2026",
    ]

    for c in delta_cols:
        if c in candidate_comparison.columns and c in ref:
            candidate_comparison[f"delta_{c}_vs_benchmark"] = candidate_comparison[c] - ref[c]

candidate_comparison = candidate_comparison.sort_values(
    "avg_pnl_per_day",
    ascending=False,
).reset_index(drop=True)

# ======================================================================================
# 7. Pass diagnostics
# ======================================================================================

pass_diagnostics_rows = []

for cid in candidate_grid_wide["candidate_id"].tolist():
    p = panel[panel["candidate_id"].eq(cid)].copy()
    q = qualified_complete[qualified_complete["candidate_id"].eq(cid)].copy()

    row = {
        "candidate_id": cid,
        "candidate_subfamily": candidate_grid_wide.loc[candidate_grid_wide["candidate_id"].eq(cid), "candidate_subfamily"].iloc[0],
        "panel_rows": int(len(p)),
        "pass_rows_all_dates": int(p["candidate_pass"].sum()),
        "pass_rows_with_outcomes": int(len(q)),
    }

    for bucket in ["Front", "Middle", "Back"]:
        row[f"{bucket}_pass_rows_with_outcomes"] = int(
            len(q[q["effective_tenor_bucket"].eq(bucket)])
        )

    for tenor in ALL_TENORS:
        row[f"tenor_{tenor}_pass_rows_with_outcomes"] = int(
            len(q[q["tenor"].eq(tenor)])
        )

    pass_diagnostics_rows.append(row)

pass_diagnostics = pd.DataFrame(pass_diagnostics_rows)

# ======================================================================================
# 8. Save outputs
# ======================================================================================

candidate_grid_wide_csv_path = OUT_AUDIT_DIR / f"10R_dte_smooth_parameter_grid_wide_{CELL10R_TIMESTAMP}.csv"
candidate_grid_wide_parquet_path = OUT_PROCESSED_DIR / f"10R_dte_smooth_parameter_grid_wide_{CELL10R_TIMESTAMP}.parquet"
candidate_grid_long_csv_path = OUT_AUDIT_DIR / f"10R_dte_smooth_parameter_grid_long_{CELL10R_TIMESTAMP}.csv"
candidate_grid_long_parquet_path = OUT_PROCESSED_DIR / f"10R_dte_smooth_parameter_grid_long_{CELL10R_TIMESTAMP}.parquet"

summary_path = OUT_AUDIT_DIR / f"10R_dte_smooth_parameter_sweep_summary_{CELL10R_TIMESTAMP}.csv"
critical_summary_path = OUT_AUDIT_DIR / f"10R_dte_smooth_parameter_sweep_critical_summary_{CELL10R_TIMESTAMP}.csv"
candidate_comparison_path = OUT_AUDIT_DIR / f"10R_dte_smooth_parameter_sweep_vs_benchmark_{CELL10R_TIMESTAMP}.csv"
by_bucket_path = OUT_AUDIT_DIR / f"10R_dte_smooth_parameter_sweep_by_effective_bucket_{CELL10R_TIMESTAMP}.csv"
by_tenor_path = OUT_AUDIT_DIR / f"10R_dte_smooth_parameter_sweep_by_tenor_{CELL10R_TIMESTAMP}.csv"
by_year_path = OUT_AUDIT_DIR / f"10R_dte_smooth_parameter_sweep_by_year_{CELL10R_TIMESTAMP}.csv"
by_tenor_year_path = OUT_AUDIT_DIR / f"10R_dte_smooth_parameter_sweep_by_tenor_year_{CELL10R_TIMESTAMP}.csv"
pass_diagnostics_path = OUT_AUDIT_DIR / f"10R_dte_smooth_parameter_sweep_pass_diagnostics_{CELL10R_TIMESTAMP}.csv"

selected_trade_panel_path = OUT_PROCESSED_DIR / f"10R_dte_smooth_parameter_sweep_selected_trade_panel_{CELL10R_TIMESTAMP}.parquet"
selected_trade_panel_csv_path = OUT_AUDIT_DIR / f"10R_dte_smooth_parameter_sweep_selected_trade_panel_{CELL10R_TIMESTAMP}.csv"

validation_path = OUT_AUDIT_DIR / f"10R_dte_smooth_parameter_sweep_validation_{CELL10R_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"10R_dte_smooth_parameter_sweep_manifest_{CELL10R_TIMESTAMP}.json"

candidate_grid_wide.to_csv(candidate_grid_wide_csv_path, index=False)
candidate_grid_wide.to_parquet(candidate_grid_wide_parquet_path, index=False)
candidate_grid_long.to_csv(candidate_grid_long_csv_path, index=False)
candidate_grid_long.to_parquet(candidate_grid_long_parquet_path, index=False)

summary.to_csv(summary_path, index=False)
critical_summary.to_csv(critical_summary_path, index=False)
candidate_comparison.to_csv(candidate_comparison_path, index=False)
by_bucket.to_csv(by_bucket_path, index=False)
by_tenor.to_csv(by_tenor_path, index=False)
by_year.to_csv(by_year_path, index=False)
by_tenor_year.to_csv(by_tenor_year_path, index=False)
pass_diagnostics.to_csv(pass_diagnostics_path, index=False)

selected_trade_panel.to_parquet(selected_trade_panel_path, index=False)
selected_trade_panel.to_csv(selected_trade_panel_csv_path, index=False)

# ======================================================================================
# 9. Validation
# ======================================================================================

validation_rows = []

required_base_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

missing_base_cols = [c for c in required_base_cols if c not in base.columns]

models_found = sorted(base["model_spec"].dropna().unique().tolist())

candidate_count = int(candidate_grid_wide["candidate_id"].nunique())
expected_long_rows = candidate_count * len(ALL_TENORS)
expected_summary_rows = candidate_count * len(SELECTION_UNIVERSES)

old_forecast_threshold_cols = [
    c for c in list(candidate_grid_wide.columns) + list(candidate_grid_long.columns)
    if "forecast_vol_pct_min" in c
]

bad_forecast_threshold_flag = candidate_grid_long[
    candidate_grid_long["forecast_vol_pct_threshold_used"].astype(bool)
]

bad_rv21d_contract_flag = candidate_grid_long[
    ~candidate_grid_long["uses_rv21d_threshold_contract"].astype(bool)
]

bad_z_equality = candidate_grid_long[
    candidate_grid_long["model_vrp_z_3m"].ne(candidate_grid_long["model_vrp_z_1y"])
]

bad_9d_include = candidate_grid_long[
    candidate_grid_long["tenor"].eq(9)
    & candidate_grid_long["include_tenor"].astype(bool)
]

bad_18d_bucket = candidate_grid_long[
    candidate_grid_long["tenor"].eq(18)
    & ~candidate_grid_long["effective_tenor_bucket"].eq("Front")
]

bad_back_anchor = candidate_grid_long[
    candidate_grid_long["tenor"].isin([27, 30, 33])
    & (
        candidate_grid_long["model_vrp_log"].ne(0.70)
        | candidate_grid_long["model_vrp_z"].ne(0.70)
    )
]

bad_rsi = candidate_grid_long[
    candidate_grid_long["RSI14_max"].ne(70.0)
]

bad_rv21d = candidate_grid_long[
    candidate_grid_long["rv21d_vol_pct_min"].ne(8.5)
]

curve_validation_rows = []

for cid, g in candidate_grid_long[
    candidate_grid_long["tenor"].isin(INCLUDED_CORE_TENORS)
].groupby("candidate_id"):
    g = g.sort_values("tenor")
    vrp_vals = g["model_vrp_log"].tolist()
    z_vals = g["model_vrp_z"].tolist()

    curve_validation_rows.append({
        "candidate_id": cid,
        "vrp_log_monotone_nondecreasing": is_monotone_nondecreasing(vrp_vals),
        "z_monotone_nondecreasing": is_monotone_nondecreasing(z_vals),
        "vrp_log_interior_between_neighbors": interior_between_neighbors(vrp_vals),
        "z_interior_between_neighbors": interior_between_neighbors(z_vals),
    })

curve_validation = pd.DataFrame(curve_validation_rows)

bad_curve_rows = curve_validation[
    ~(
        curve_validation["vrp_log_monotone_nondecreasing"]
        & curve_validation["z_monotone_nondecreasing"]
        & curve_validation["vrp_log_interior_between_neighbors"]
        & curve_validation["z_interior_between_neighbors"]
    )
]

summary_universes_found = sorted(summary["selection_universe"].dropna().unique().tolist())
summary_candidates_found = sorted(summary["candidate_id"].dropna().unique().tolist())

bad_normalized_pnl = base[
    base["normalized_pnl_dollars"].notna()
    & base["normalized_pnl_per_day"].notna()
    & (
        (
            base["normalized_pnl_dollars"] / base["tenor"].replace(0, np.nan)
        )
        - base["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
]

add_validation(
    validation_rows,
    "base_panel_loaded",
    "PASS" if len(base) > 0 else "FAIL",
    f"rows={len(base):,}; path={base_panel_path}",
)

add_validation(
    validation_rows,
    "required_base_columns_available",
    "PASS" if not missing_base_cols else "FAIL",
    f"missing={missing_base_cols}",
)

add_validation(
    validation_rows,
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    validation_rows,
    "cell9_grid_loaded",
    "PASS" if len(cell9_grid_long) > 0 else "FAIL",
    f"grid_long_rows={len(cell9_grid_long):,}; path={cell9_grid_long_path}",
)

add_validation(
    validation_rows,
    "candidate_count",
    "PASS" if candidate_count == 7 else "FAIL",
    f"candidate_count={candidate_count:,}; expected=7",
)

add_validation(
    validation_rows,
    "long_grid_all_tenors_per_candidate",
    "PASS" if len(candidate_grid_long) == expected_long_rows else "FAIL",
    f"long_rows={len(candidate_grid_long):,}; expected={expected_long_rows:,}",
)

add_validation(
    validation_rows,
    "nine_day_excluded_all_candidates",
    "PASS" if bad_9d_include.empty else "FAIL",
    f"bad_rows={len(bad_9d_include):,}",
)

add_validation(
    validation_rows,
    "eighteen_day_stays_front",
    "PASS" if bad_18d_bucket.empty else "FAIL",
    f"bad_rows={len(bad_18d_bucket):,}",
)

add_validation(
    validation_rows,
    "back_endpoint_anchored",
    "PASS" if bad_back_anchor.empty else "FAIL",
    f"bad_rows={len(bad_back_anchor):,}",
)

add_validation(
    validation_rows,
    "RSI_fixed_70_all_DTEs",
    "PASS" if bad_rsi.empty else "FAIL",
    f"bad_rows={len(bad_rsi):,}",
)

add_validation(
    validation_rows,
    "rv21d_floor_fixed_8_5_all_DTEs",
    "PASS" if bad_rv21d.empty else "FAIL",
    f"bad_rows={len(bad_rv21d):,}",
)

add_validation(
    validation_rows,
    "uses_rv21d_threshold_contract",
    "PASS" if "threshold_rv21d_vol_pct_min" in panel.columns and bad_rv21d_contract_flag.empty else "FAIL",
    f"bad_rv21d_contract_rows={len(bad_rv21d_contract_flag):,}",
)

add_validation(
    validation_rows,
    "no_forecast_vol_threshold_columns",
    "PASS" if not old_forecast_threshold_cols else "FAIL",
    f"old_forecast_threshold_cols={old_forecast_threshold_cols}",
)

add_validation(
    validation_rows,
    "forecast_vol_pct_diagnostic_only",
    "PASS" if bad_forecast_threshold_flag.empty else "FAIL",
    f"bad_forecast_threshold_flag_rows={len(bad_forecast_threshold_flag):,}",
)

add_validation(
    validation_rows,
    "z_3m_equals_z_1y_at_each_DTE",
    "PASS" if bad_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_z_equality):,}",
)

add_validation(
    validation_rows,
    "curves_monotone_and_neighbor_constrained",
    "PASS" if bad_curve_rows.empty else "FAIL",
    f"bad_rows={len(bad_curve_rows):,}; details={bad_curve_rows.to_dict('records') if not bad_curve_rows.empty else []}",
)

add_validation(
    validation_rows,
    "panel_rule_application_created",
    "PASS" if len(panel) > 0 and len(qualified_complete) > 0 else "FAIL",
    f"panel_rows={len(panel):,}; qualified_complete_rows={len(qualified_complete):,}",
)

add_validation(
    validation_rows,
    "all_selection_universes_evaluated",
    "PASS" if summary_universes_found == sorted(SELECTION_UNIVERSES) else "FAIL",
    f"found={summary_universes_found}",
)

add_validation(
    validation_rows,
    "all_candidates_evaluated",
    "PASS" if summary_candidates_found == sorted(candidate_grid_wide["candidate_id"].tolist()) else "FAIL",
    f"found={summary_candidates_found}",
)

add_validation(
    validation_rows,
    "summary_rows_complete",
    "PASS" if len(summary) == expected_summary_rows else "FAIL",
    f"summary_rows={len(summary):,}; expected={expected_summary_rows:,}",
)

add_validation(
    validation_rows,
    "benchmark_present",
    "PASS" if BENCHMARK_CANDIDATE_ID in candidate_grid_wide["candidate_id"].tolist() else "FAIL",
    f"benchmark_candidate_id={BENCHMARK_CANDIDATE_ID}",
)

add_validation(
    validation_rows,
    "critical_summary_created",
    "PASS" if len(critical_summary) == candidate_count else "FAIL",
    f"rows={len(critical_summary):,}; expected={candidate_count:,}",
)

add_validation(
    validation_rows,
    "candidate_comparison_created",
    "PASS" if len(candidate_comparison) == candidate_count else "FAIL",
    f"rows={len(candidate_comparison):,}; expected={candidate_count:,}",
)

add_validation(
    validation_rows,
    "normalized_pnl_per_day_formula",
    "PASS" if bad_normalized_pnl.empty else "FAIL",
    f"bad_rows={len(bad_normalized_pnl):,}",
)

add_validation(
    validation_rows,
    "no_secondary",
    "PASS",
    "Core only.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed or added.",
)

add_validation(
    validation_rows,
    "no_production_lock",
    "PASS",
    "No production threshold lock performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"].eq("FAIL")]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1_RV21D_repaired_v2.ipynb",
    "cell": "Cell 10R — DTE-level smooth parameter sweep",
    "timestamp": CELL10R_TIMESTAMP,
    "base_panel_path": str(base_panel_path),
    "cell9_grid_long_path": str(cell9_grid_long_path),
    "cell9_grid_wide_path": str(cell9_grid_wide_path),
    "candidate_ids": candidate_grid_wide["candidate_id"].tolist(),
    "benchmark_candidate_id": BENCHMARK_CANDIDATE_ID,
    "primary_selection_universe": PRIMARY_SELECTION_UNIVERSE,
    "primary_ranking_metric": PRIMARY_RANKING_METRIC,
    "effective_bucket_map": EFFECTIVE_TENOR_BUCKET_MAP,
    "candidate_grid_wide_csv_path": str(candidate_grid_wide_csv_path),
    "candidate_grid_wide_parquet_path": str(candidate_grid_wide_parquet_path),
    "candidate_grid_long_csv_path": str(candidate_grid_long_csv_path),
    "candidate_grid_long_parquet_path": str(candidate_grid_long_parquet_path),
    "summary_path": str(summary_path),
    "critical_summary_path": str(critical_summary_path),
    "candidate_comparison_path": str(candidate_comparison_path),
    "by_bucket_path": str(by_bucket_path),
    "by_tenor_path": str(by_tenor_path),
    "by_year_path": str(by_year_path),
    "by_tenor_year_path": str(by_tenor_year_path),
    "pass_diagnostics_path": str(pass_diagnostics_path),
    "selected_trade_panel_path": str(selected_trade_panel_path),
    "selected_trade_panel_csv_path": str(selected_trade_panel_csv_path),
    "validation_path": str(validation_path),
    "manifest_path": str(manifest_path),
    "threshold_contract": "rv21d_vol_pct > rv21d_vol_pct_min",
    "forecast_vol_pct_role": "model denominator / diagnostic only, not threshold floor",
    "constraints": [
        "9D remains excluded.",
        "18D remains in Front.",
        "Effective bucket map is Front 12/15/18, Middle 21/24, Back 27/30/33.",
        "Back endpoint remains anchored at log=0.70 and z=0.70.",
        "z3m and z1y are equal at each DTE.",
        "RSI14 cap remains 70 for all DTEs.",
        "RV21D floor remains 8.5 for all DTEs.",
        "DTE curves are monotone nondecreasing.",
        "Interior DTE parameters are between/equal neighboring DTEs.",
        "No Secondary.",
        "No sizing changes.",
        "No production lock.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 10R validation failed. Review validation output above.")

# ======================================================================================
# 10. Display review
# ======================================================================================

print("=" * 100)
print("Cell 10R — DTE-level smooth parameter sweep")
print("=" * 100)
print(f"Base panel:                         {base_panel_path}")
print(f"Cell 9R grid long:                  {cell9_grid_long_path}")
print(f"Candidate count:                    {candidate_count:,}")
print(f"Qualified complete rows:            {len(qualified_complete):,}")
print(f"Selected trade panel rows:          {len(selected_trade_panel):,}")
print(f"Primary universe:                   {PRIMARY_SELECTION_UNIVERSE}")
print(f"Primary metric:                     {PRIMARY_RANKING_METRIC}")
print("Effective bucket map:               Front 12/15/18; Middle 21/24; Back 27/30/33; 9D excluded")
print("Threshold contract:                 rv21d_vol_pct > rv21d_vol_pct_min")
print("RSI fixed:                          70")
print("RV21D floor fixed:                  8.5")
print("Back endpoint anchored:             True")
print("z3m/z1y equal at each DTE:           True")
print("No Secondary:                       True")
print("No sizing changes:                  True")
print("No production lock:                 True")

print()
print("Validation:")
display(validation)

print()
print("Curve validation:")
display(curve_validation)

print()
print("Candidate grid — wide:")
display(candidate_grid_wide)

print()
print("Candidate grid — long:")
display(candidate_grid_long)

critical_display_cols = [
    "candidate_id",
    "candidate_subfamily",
    "trades",
    "unique_trade_dates",
    "total_pnl",
    "profit_factor",
    "avg_pnl_per_day",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "front_trades",
    "front_avg_pnl_per_day",
    "front_avg_pnl_per_day_2025",
    "front_avg_pnl_per_day_2026",
    "front_profit_factor_pnl_per_day",
    "middle_trades",
    "middle_avg_pnl_per_day",
    "middle_avg_pnl_per_day_2025",
    "middle_avg_pnl_per_day_2026",
    "middle_profit_factor_pnl_per_day",
    "back_trades",
    "back_avg_pnl_per_day",
    "back_avg_pnl_per_day_2025",
    "back_avg_pnl_per_day_2026",
    "back_profit_factor_pnl_per_day",
    "tenor_12_trades",
    "tenor_12_avg_pnl_per_day",
    "tenor_12_avg_pnl_per_day_2026",
    "tenor_15_trades",
    "tenor_15_avg_pnl_per_day",
    "tenor_15_avg_pnl_per_day_2026",
    "tenor_18_trades",
    "tenor_18_avg_pnl_per_day",
    "tenor_18_avg_pnl_per_day_2026",
    "tenor_21_trades",
    "tenor_21_avg_pnl_per_day",
    "tenor_21_avg_pnl_per_day_2026",
    "tenor_24_trades",
    "tenor_24_avg_pnl_per_day",
    "tenor_24_avg_pnl_per_day_2026",
]

print()
print("Critical summary — primary universe:")
display(critical_summary[[c for c in critical_display_cols if c in critical_summary.columns]])

comparison_display_cols = [
    "candidate_id",
    "candidate_subfamily",
    "avg_pnl_per_day",
    "delta_avg_pnl_per_day_vs_benchmark",
    "total_pnl",
    "delta_total_pnl_vs_benchmark",
    "profit_factor",
    "delta_profit_factor_vs_benchmark",
    "avg_pnl_per_day_2026",
    "delta_avg_pnl_per_day_2026_vs_benchmark",
    "pnl_per_day_drawdown",
    "delta_pnl_per_day_drawdown_vs_benchmark",
    "worst_20_trade_pnl_per_day_sum",
    "delta_worst_20_trade_pnl_per_day_sum_vs_benchmark",
    "front_avg_pnl_per_day",
    "delta_front_avg_pnl_per_day_vs_benchmark",
    "front_avg_pnl_per_day_2026",
    "delta_front_avg_pnl_per_day_2026_vs_benchmark",
    "middle_avg_pnl_per_day",
    "delta_middle_avg_pnl_per_day_vs_benchmark",
    "middle_avg_pnl_per_day_2026",
    "delta_middle_avg_pnl_per_day_2026_vs_benchmark",
    "tenor_18_trades",
    "delta_tenor_18_trades_vs_benchmark",
    "tenor_18_avg_pnl_per_day",
    "delta_tenor_18_avg_pnl_per_day_vs_benchmark",
    "tenor_18_avg_pnl_per_day_2026",
    "delta_tenor_18_avg_pnl_per_day_2026_vs_benchmark",
    "tenor_24_trades",
    "delta_tenor_24_trades_vs_benchmark",
    "tenor_24_avg_pnl_per_day",
    "delta_tenor_24_avg_pnl_per_day_vs_benchmark",
    "tenor_24_avg_pnl_per_day_2026",
    "delta_tenor_24_avg_pnl_per_day_2026_vs_benchmark",
]

print()
print("Candidate comparison vs DTE benchmark — primary universe:")
display(candidate_comparison[[c for c in comparison_display_cols if c in candidate_comparison.columns]])

print()
print("By effective bucket:")
display(by_bucket)

print()
print("By tenor:")
display(by_tenor)

print()
print("By year:")
display(by_year)

print()
print("By tenor/year:")
display(by_tenor_year)

print()
print("Pass diagnostics:")
display(pass_diagnostics)

print()
print("Saved outputs:")
for p in [
    candidate_grid_wide_csv_path,
    candidate_grid_wide_parquet_path,
    candidate_grid_long_csv_path,
    candidate_grid_long_parquet_path,
    summary_path,
    critical_summary_path,
    candidate_comparison_path,
    by_bucket_path,
    by_tenor_path,
    by_year_path,
    by_tenor_year_path,
    pass_diagnostics_path,
    selected_trade_panel_path,
    selected_trade_panel_csv_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 10R PASSED")

In [ ]:
# ======================================================================================
# Cell 11R — Lock repaired Core bucket map and parameters
# ======================================================================================
#
# Objective:
#   Formal lock artifact for Core repaired v1.
#
# Decision:
#   Use the Cell 10R bucket benchmark, not DTE-specific smoothing.
#
# Locked Core repaired v1:
#
#   Excluded:
#       9D
#
#   Core Front:
#       tenors: 12D / 15D / 18D
#       model_vrp_log > 0.60
#       model_vrp_z_3m > 0.65
#       model_vrp_z_1y > 0.65
#       RSI14 < 70
#       rv21d_vol_pct > 8.5
#
#   Core Middle:
#       tenors: 21D / 24D
#       model_vrp_log > 0.65
#       model_vrp_z_3m > 0.70
#       model_vrp_z_1y > 0.70
#       RSI14 < 70
#       rv21d_vol_pct > 8.5
#
#   Core Back:
#       tenors: 27D / 30D / 33D
#       model_vrp_log > 0.70
#       model_vrp_z_3m > 0.70
#       model_vrp_z_1y > 0.70
#       RSI14 < 70
#       rv21d_vol_pct > 8.5
#
# Correct volatility-floor contract:
#       rv21d_vol_pct > rv21d_vol_pct_min
#
# NOT:
#       forecast_vol_pct > forecast_vol_pct_min
#
# This cell:
#   1. Writes locked Core rules wide and long.
#   2. Re-applies locked rules to the repaired base panel.
#   3. Builds selected trade panels for all standard universes.
#   4. Writes summary, by bucket, by tenor, by year, by tenor/year.
#   5. Writes validation and manifest.
#
# This cell does NOT:
#   1. No Secondary.
#   2. No sizing changes.
#   3. No production/intraday implementation.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 380)
pd.set_option("display.width", 640)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL11R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

LOCKED_CORE_VERSION = "core_repaired_v1"
LOCKED_CORE_DECISION_ID = "core_repaired_v1_locked_001"
LOCKED_CORE_SOURCE_CANDIDATE_ID = "core_dte_smooth_0001"
LOCKED_CORE_SOURCE_CANDIDATE_SUBFAMILY = "locked_bucket_benchmark"

ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

ORIGINAL_TENOR_BUCKET_MAP = {
    9: "Front",
    12: "Front",
    15: "Front",
    18: "Middle",
    21: "Middle",
    24: "Middle",
    27: "Back",
    30: "Back",
    33: "Back",
}

LOCKED_EFFECTIVE_TENOR_BUCKET_MAP = {
    9: "Excluded",
    12: "Front",
    15: "Front",
    18: "Front",
    21: "Middle",
    24: "Middle",
    27: "Back",
    30: "Back",
    33: "Back",
}

BUCKET_ORDER_MAP = {
    "Excluded": 0,
    "Front": 1,
    "Middle": 2,
    "Back": 3,
}

BUCKET_CENTER = {
    "Front": 15,
    "Middle": 21,
    "Back": 30,
}

SELECTION_UNIVERSES = [
    "all_qualified_trades",
    "one_trade_per_bucket_per_date_best_rank",
    "one_trade_per_date_best_rank",
]

PRIMARY_SELECTION_UNIVERSE = "one_trade_per_bucket_per_date_best_rank"
PRIMARY_RANKING_METRIC = "avg_pnl_per_day"

LOCKED_RULES_BY_TENOR = {
    9:  {"include_tenor": False, "effective_bucket": "Excluded", "model_vrp_log": 0.60, "model_vrp_z": 0.65, "RSI14_max": 70.0, "rv21d_vol_pct_min": 8.5},

    12: {"include_tenor": True,  "effective_bucket": "Front",   "model_vrp_log": 0.60, "model_vrp_z": 0.65, "RSI14_max": 70.0, "rv21d_vol_pct_min": 8.5},
    15: {"include_tenor": True,  "effective_bucket": "Front",   "model_vrp_log": 0.60, "model_vrp_z": 0.65, "RSI14_max": 70.0, "rv21d_vol_pct_min": 8.5},
    18: {"include_tenor": True,  "effective_bucket": "Front",   "model_vrp_log": 0.60, "model_vrp_z": 0.65, "RSI14_max": 70.0, "rv21d_vol_pct_min": 8.5},

    21: {"include_tenor": True,  "effective_bucket": "Middle",  "model_vrp_log": 0.65, "model_vrp_z": 0.70, "RSI14_max": 70.0, "rv21d_vol_pct_min": 8.5},
    24: {"include_tenor": True,  "effective_bucket": "Middle",  "model_vrp_log": 0.65, "model_vrp_z": 0.70, "RSI14_max": 70.0, "rv21d_vol_pct_min": 8.5},

    27: {"include_tenor": True,  "effective_bucket": "Back",    "model_vrp_log": 0.70, "model_vrp_z": 0.70, "RSI14_max": 70.0, "rv21d_vol_pct_min": 8.5},
    30: {"include_tenor": True,  "effective_bucket": "Back",    "model_vrp_log": 0.70, "model_vrp_z": 0.70, "RSI14_max": 70.0, "rv21d_vol_pct_min": 8.5},
    33: {"include_tenor": True,  "effective_bucket": "Back",    "model_vrp_log": 0.70, "model_vrp_z": 0.70, "RSI14_max": 70.0, "rv21d_vol_pct_min": 8.5},
}

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def add_selection_ranks(df):
    d = df.copy()

    if d.empty:
        return d

    d["bucket_center_tenor"] = d["effective_tenor_bucket"].map(BUCKET_CENTER)
    d["distance_to_bucket_center"] = (d["tenor"] - d["bucket_center_tenor"]).abs()

    bucket_group = ["trade_date", "effective_tenor_bucket"]

    d["rank_z_1y_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_bucket_date"] = d[
        [
            "rank_z_1y_in_bucket_date",
            "rank_z_3m_in_bucket_date",
            "rank_vrp_log_in_bucket_date",
        ]
    ].mean(axis=1)

    date_group = ["trade_date"]

    d["rank_z_1y_in_date"] = (
        d.groupby(date_group)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_date"] = (
        d.groupby(date_group)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_date"] = (
        d.groupby(date_group)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_date"] = d[
        [
            "rank_z_1y_in_date",
            "rank_z_3m_in_date",
            "rank_vrp_log_in_date",
        ]
    ].mean(axis=1)

    return d


def select_trade_universes(qualified):
    if qualified.empty:
        return {
            "all_qualified_trades": qualified.copy(),
            "one_trade_per_bucket_per_date_best_rank": qualified.copy(),
            "one_trade_per_date_best_rank": qualified.copy(),
        }

    q = add_selection_ranks(qualified)

    one_trade_per_bucket_date = (
        q.sort_values(
            [
                "trade_date",
                "effective_tenor_bucket_order",
                "avg_signal_rank_in_bucket_date",
                "distance_to_bucket_center",
                "rank_z_1y_in_bucket_date",
                "rank_z_3m_in_bucket_date",
                "rank_vrp_log_in_bucket_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date", "effective_tenor_bucket"], as_index=False)
        .head(1)
        .copy()
    )

    one_trade_per_date = (
        q.sort_values(
            [
                "trade_date",
                "avg_signal_rank_in_date",
                "rank_z_1y_in_date",
                "rank_z_3m_in_date",
                "rank_vrp_log_in_date",
                "distance_to_bucket_center",
                "effective_tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return {
        "all_qualified_trades": q.copy(),
        "one_trade_per_bucket_per_date_best_rank": one_trade_per_bucket_date,
        "one_trade_per_date_best_rank": one_trade_per_date,
    }


def summarize_trade_set(df, group_cols):
    d = df.copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    numeric_cols = [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "forecast_vol_pct",
        "rv21d_vol_pct",
        "threshold_model_vrp_log",
        "threshold_model_vrp_z_3m",
        "threshold_model_vrp_z_1y",
        "threshold_RSI14_max",
        "threshold_rv21d_vol_pct_min",
    ]

    for c in numeric_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d = d.sort_values(["selection_universe", "trade_date", "tenor"]).copy()

    if group_cols:
        grouped = d.groupby(group_cols, dropna=False, observed=False)
    else:
        grouped = [((), d)]

    rows = []

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {}

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        row.update({
            "locked_core_version": LOCKED_CORE_VERSION,
            "locked_core_decision_id": LOCKED_CORE_DECISION_ID,

            "trades": int(len(g)),
            "unique_trade_dates": int(g["trade_date"].nunique()),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        for signal_col in [
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "threshold_model_vrp_log",
            "threshold_model_vrp_z_3m",
            "threshold_model_vrp_z_1y",
            "threshold_RSI14_max",
            "threshold_rv21d_vol_pct_min",
        ]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        rows.append(row)

    return pd.DataFrame(rows)


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


# ======================================================================================
# 2. Load repaired base panel
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01R_unified_fds_signal_base_panel_with_rv21d_*.parquet",
    required=True,
)

base = pd.read_parquet(base_panel_path)
base["trade_date"] = pd.to_datetime(base["trade_date"], errors="coerce").dt.normalize()

for c in [
    "tenor",
    "tenor_bucket_order",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")

# ======================================================================================
# 3. Build locked rules artifacts
# ======================================================================================

locked_rules_long_rows = []

for tenor in ALL_TENORS:
    rule = LOCKED_RULES_BY_TENOR[tenor]
    original_bucket = ORIGINAL_TENOR_BUCKET_MAP[tenor]
    effective_bucket = rule["effective_bucket"]

    locked_rules_long_rows.append({
        "locked_core_version": LOCKED_CORE_VERSION,
        "locked_core_decision_id": LOCKED_CORE_DECISION_ID,
        "locked_model_spec": LOCKED_MODEL_SPEC,
        "source_research_candidate_id": LOCKED_CORE_SOURCE_CANDIDATE_ID,
        "source_research_candidate_subfamily": LOCKED_CORE_SOURCE_CANDIDATE_SUBFAMILY,

        "tenor": int(tenor),
        "original_tenor_bucket": original_bucket,
        "original_tenor_bucket_order": BUCKET_ORDER_MAP[original_bucket],
        "effective_tenor_bucket": effective_bucket,
        "effective_tenor_bucket_order": BUCKET_ORDER_MAP[effective_bucket],

        "include_tenor": bool(rule["include_tenor"]),

        "model_vrp_log_min": float(rule["model_vrp_log"]),
        "model_vrp_z_min": float(rule["model_vrp_z"]),
        "model_vrp_z_3m_min": float(rule["model_vrp_z"]),
        "model_vrp_z_1y_min": float(rule["model_vrp_z"]),
        "RSI14_max": float(rule["RSI14_max"]),
        "rv21d_vol_pct_min": float(rule["rv21d_vol_pct_min"]),

        "uses_rv21d_threshold_contract": True,
        "forecast_vol_pct_threshold_used": False,
        "forecast_vol_pct_role": "model denominator / diagnostic only",
        "core_rule_locked": True,
        "secondary_rule_locked": False,
        "sizing_locked_here": False,
        "production_locked_here": False,
    })

locked_rules_long = (
    pd.DataFrame(locked_rules_long_rows)
    .sort_values("tenor")
    .reset_index(drop=True)
)

locked_rules_wide = pd.DataFrame([{
    "locked_core_version": LOCKED_CORE_VERSION,
    "locked_core_decision_id": LOCKED_CORE_DECISION_ID,
    "locked_model_spec": LOCKED_MODEL_SPEC,
    "source_research_candidate_id": LOCKED_CORE_SOURCE_CANDIDATE_ID,
    "source_research_candidate_subfamily": LOCKED_CORE_SOURCE_CANDIDATE_SUBFAMILY,

    "excluded_tenors": "9",
    "front_tenors": "12,15,18",
    "middle_tenors": "21,24",
    "back_tenors": "27,30,33",

    "front_model_vrp_log_min": 0.60,
    "front_model_vrp_z_3m_min": 0.65,
    "front_model_vrp_z_1y_min": 0.65,
    "front_RSI14_max": 70.0,
    "front_rv21d_vol_pct_min": 8.5,

    "middle_model_vrp_log_min": 0.65,
    "middle_model_vrp_z_3m_min": 0.70,
    "middle_model_vrp_z_1y_min": 0.70,
    "middle_RSI14_max": 70.0,
    "middle_rv21d_vol_pct_min": 8.5,

    "back_model_vrp_log_min": 0.70,
    "back_model_vrp_z_3m_min": 0.70,
    "back_model_vrp_z_1y_min": 0.70,
    "back_RSI14_max": 70.0,
    "back_rv21d_vol_pct_min": 8.5,

    "nine_day_excluded": True,
    "eighteen_day_in_front": True,
    "z3m_equals_z1y_within_each_effective_bucket": True,
    "rv21d_threshold_contract": "rv21d_vol_pct > rv21d_vol_pct_min",
    "forecast_vol_pct_threshold_used": False,
    "forecast_vol_pct_role": "model denominator / diagnostic only",
    "dte_specific_smoothing_used": False,
    "core_rule_locked": True,
    "secondary_rule_locked": False,
    "sizing_locked_here": False,
    "production_locked_here": False,
}])

# ======================================================================================
# 4. Apply locked rules
# ======================================================================================

thresholds = locked_rules_long.rename(columns={
    "model_vrp_log_min": "threshold_model_vrp_log",
    "model_vrp_z_min": "threshold_model_vrp_z",
    "model_vrp_z_3m_min": "threshold_model_vrp_z_3m",
    "model_vrp_z_1y_min": "threshold_model_vrp_z_1y",
    "RSI14_max": "threshold_RSI14_max",
    "rv21d_vol_pct_min": "threshold_rv21d_vol_pct_min",
})

threshold_cols = [
    "tenor",
    "original_tenor_bucket",
    "original_tenor_bucket_order",
    "effective_tenor_bucket",
    "effective_tenor_bucket_order",
    "include_tenor",
    "threshold_model_vrp_log",
    "threshold_model_vrp_z",
    "threshold_model_vrp_z_3m",
    "threshold_model_vrp_z_1y",
    "threshold_RSI14_max",
    "threshold_rv21d_vol_pct_min",
]

base_for_join = base.copy()

base_for_join = base_for_join.rename(columns={
    "tenor_bucket": "source_tenor_bucket",
    "tenor_bucket_order": "source_tenor_bucket_order",
})

panel = base_for_join.merge(
    thresholds[threshold_cols],
    on="tenor",
    how="inner",
    validate="many_to_one",
)

panel["locked_core_version"] = LOCKED_CORE_VERSION
panel["locked_core_decision_id"] = LOCKED_CORE_DECISION_ID
panel["locked_model_spec"] = LOCKED_MODEL_SPEC

required_signal_available = (
    panel["include_tenor"].astype(bool)
    & panel["model_vrp_log"].notna()
    & panel["model_vrp_z_3m"].notna()
    & panel["model_vrp_z_1y"].notna()
    & panel["RSI14"].notna()
    & panel["rv21d_vol_pct"].notna()
    & panel["threshold_model_vrp_log"].notna()
    & panel["threshold_model_vrp_z_3m"].notna()
    & panel["threshold_model_vrp_z_1y"].notna()
    & panel["threshold_RSI14_max"].notna()
    & panel["threshold_rv21d_vol_pct_min"].notna()
)

outcome_available = (
    panel["normalized_pnl_dollars"].notna()
    & panel["normalized_pnl_per_day"].notna()
)

pass_mask = (
    required_signal_available
    & (panel["model_vrp_log"] > panel["threshold_model_vrp_log"])
    & (panel["model_vrp_z_3m"] > panel["threshold_model_vrp_z_3m"])
    & (panel["model_vrp_z_1y"] > panel["threshold_model_vrp_z_1y"])
    & (panel["RSI14"] < panel["threshold_RSI14_max"])
    & (panel["rv21d_vol_pct"] > panel["threshold_rv21d_vol_pct_min"])
)

panel["core_repaired_v1_pass"] = pass_mask
panel["candidate_pass"] = pass_mask

qualified_all_dates = panel[pass_mask].copy()
qualified_complete = panel[pass_mask & outcome_available].copy()

universes = select_trade_universes(qualified_complete)

selected_trade_panel = pd.concat(
    [df.assign(selection_universe=name) for name, df in universes.items()],
    ignore_index=True,
)

selected_trade_panel["locked_core_version"] = LOCKED_CORE_VERSION
selected_trade_panel["locked_core_decision_id"] = LOCKED_CORE_DECISION_ID

# ======================================================================================
# 5. Summaries
# ======================================================================================

summary_frames = []
bucket_frames = []
tenor_frames = []
year_frames = []
tenor_year_frames = []

for universe_name, df_u in universes.items():
    df_u = df_u.assign(selection_universe=universe_name).copy()

    summary_frames.append(
        summarize_trade_set(
            df_u,
            group_cols=["selection_universe"],
        )
    )

    bucket_frames.append(
        summarize_trade_set(
            df_u,
            group_cols=[
                "selection_universe",
                "effective_tenor_bucket",
                "effective_tenor_bucket_order",
            ],
        )
    )

    tenor_frames.append(
        summarize_trade_set(
            df_u,
            group_cols=[
                "selection_universe",
                "tenor",
                "original_tenor_bucket",
                "effective_tenor_bucket",
                "effective_tenor_bucket_order",
            ],
        )
    )

    y = df_u.copy()
    y["year"] = y["trade_date"].dt.year.astype(int)

    year_frames.append(
        summarize_trade_set(
            y,
            group_cols=["selection_universe", "year"],
        )
    )

    tenor_year_frames.append(
        summarize_trade_set(
            y,
            group_cols=[
                "selection_universe",
                "tenor",
                "original_tenor_bucket",
                "effective_tenor_bucket",
                "year",
            ],
        )
    )

summary = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
by_bucket = pd.concat(bucket_frames, ignore_index=True) if bucket_frames else pd.DataFrame()
by_tenor = pd.concat(tenor_frames, ignore_index=True) if tenor_frames else pd.DataFrame()
by_year = pd.concat(year_frames, ignore_index=True) if year_frames else pd.DataFrame()
by_tenor_year = pd.concat(tenor_year_frames, ignore_index=True) if tenor_year_frames else pd.DataFrame()

summary = summary.sort_values(["selection_universe"]).reset_index(drop=True)
by_bucket = by_bucket.sort_values(["selection_universe", "effective_tenor_bucket_order"]).reset_index(drop=True)
by_tenor = by_tenor.sort_values(["selection_universe", "tenor"]).reset_index(drop=True)
by_year = by_year.sort_values(["selection_universe", "year"]).reset_index(drop=True)
by_tenor_year = by_tenor_year.sort_values(["selection_universe", "tenor", "year"]).reset_index(drop=True)

primary_summary = summary[
    summary["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

primary_bucket = by_bucket[
    by_bucket["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

primary_tenor = by_tenor[
    by_tenor["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

# Pass diagnostics.
pass_diagnostics_rows = [{
    "locked_core_version": LOCKED_CORE_VERSION,
    "locked_core_decision_id": LOCKED_CORE_DECISION_ID,
    "panel_rows": int(len(panel)),
    "pass_rows_all_dates": int(panel["core_repaired_v1_pass"].sum()),
    "pass_rows_with_outcomes": int(len(qualified_complete)),
}]

for bucket in ["Front", "Middle", "Back"]:
    pass_diagnostics_rows[0][f"{bucket}_pass_rows_with_outcomes"] = int(
        len(qualified_complete[qualified_complete["effective_tenor_bucket"].eq(bucket)])
    )

for tenor in ALL_TENORS:
    pass_diagnostics_rows[0][f"tenor_{tenor}_pass_rows_with_outcomes"] = int(
        len(qualified_complete[qualified_complete["tenor"].eq(tenor)])
    )

pass_diagnostics = pd.DataFrame(pass_diagnostics_rows)

# ======================================================================================
# 6. Save outputs
# ======================================================================================

locked_rules_wide_csv_path = OUT_AUDIT_DIR / f"11R_core_repaired_v1_locked_rules_wide_{CELL11R_TIMESTAMP}.csv"
locked_rules_wide_parquet_path = OUT_PROCESSED_DIR / f"11R_core_repaired_v1_locked_rules_wide_{CELL11R_TIMESTAMP}.parquet"
locked_rules_long_csv_path = OUT_AUDIT_DIR / f"11R_core_repaired_v1_locked_rules_long_{CELL11R_TIMESTAMP}.csv"
locked_rules_long_parquet_path = OUT_PROCESSED_DIR / f"11R_core_repaired_v1_locked_rules_long_{CELL11R_TIMESTAMP}.parquet"

locked_panel_path = OUT_PROCESSED_DIR / f"11R_core_repaired_v1_locked_panel_{CELL11R_TIMESTAMP}.parquet"
locked_panel_csv_path = OUT_AUDIT_DIR / f"11R_core_repaired_v1_locked_panel_{CELL11R_TIMESTAMP}.csv"

selected_trade_panel_path = OUT_PROCESSED_DIR / f"11R_core_repaired_v1_selected_trade_panel_{CELL11R_TIMESTAMP}.parquet"
selected_trade_panel_csv_path = OUT_AUDIT_DIR / f"11R_core_repaired_v1_selected_trade_panel_{CELL11R_TIMESTAMP}.csv"

summary_path = OUT_AUDIT_DIR / f"11R_core_repaired_v1_summary_{CELL11R_TIMESTAMP}.csv"
by_bucket_path = OUT_AUDIT_DIR / f"11R_core_repaired_v1_by_effective_bucket_{CELL11R_TIMESTAMP}.csv"
by_tenor_path = OUT_AUDIT_DIR / f"11R_core_repaired_v1_by_tenor_{CELL11R_TIMESTAMP}.csv"
by_year_path = OUT_AUDIT_DIR / f"11R_core_repaired_v1_by_year_{CELL11R_TIMESTAMP}.csv"
by_tenor_year_path = OUT_AUDIT_DIR / f"11R_core_repaired_v1_by_tenor_year_{CELL11R_TIMESTAMP}.csv"
pass_diagnostics_path = OUT_AUDIT_DIR / f"11R_core_repaired_v1_pass_diagnostics_{CELL11R_TIMESTAMP}.csv"

validation_path = OUT_AUDIT_DIR / f"11R_core_repaired_v1_validation_{CELL11R_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"11R_core_repaired_v1_manifest_{CELL11R_TIMESTAMP}.json"

locked_rules_wide.to_csv(locked_rules_wide_csv_path, index=False)
locked_rules_wide.to_parquet(locked_rules_wide_parquet_path, index=False)
locked_rules_long.to_csv(locked_rules_long_csv_path, index=False)
locked_rules_long.to_parquet(locked_rules_long_parquet_path, index=False)

panel.to_parquet(locked_panel_path, index=False)
panel.to_csv(locked_panel_csv_path, index=False)

selected_trade_panel.to_parquet(selected_trade_panel_path, index=False)
selected_trade_panel.to_csv(selected_trade_panel_csv_path, index=False)

summary.to_csv(summary_path, index=False)
by_bucket.to_csv(by_bucket_path, index=False)
by_tenor.to_csv(by_tenor_path, index=False)
by_year.to_csv(by_year_path, index=False)
by_tenor_year.to_csv(by_tenor_year_path, index=False)
pass_diagnostics.to_csv(pass_diagnostics_path, index=False)

# ======================================================================================
# 7. Validation
# ======================================================================================

validation_rows = []

required_base_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

missing_base_cols = [c for c in required_base_cols if c not in base.columns]
models_found = sorted(base["model_spec"].dropna().unique().tolist()) if "model_spec" in base.columns else []

bad_9d_include = locked_rules_long[
    locked_rules_long["tenor"].eq(9)
    & locked_rules_long["include_tenor"].astype(bool)
]

bad_18d_bucket = locked_rules_long[
    locked_rules_long["tenor"].eq(18)
    & ~locked_rules_long["effective_tenor_bucket"].eq("Front")
]

bad_z_equality = locked_rules_long[
    locked_rules_long["model_vrp_z_3m_min"].ne(locked_rules_long["model_vrp_z_1y_min"])
]

bad_forecast_threshold_flag = locked_rules_long[
    locked_rules_long["forecast_vol_pct_threshold_used"].astype(bool)
]

old_forecast_threshold_cols = [
    c for c in list(locked_rules_wide.columns) + list(locked_rules_long.columns)
    if "forecast_vol_pct_min" in c
]

bad_rv21d_contract = locked_rules_long[
    ~locked_rules_long["uses_rv21d_threshold_contract"].astype(bool)
]

bad_rsi = locked_rules_long[
    locked_rules_long["RSI14_max"].ne(70.0)
]

bad_rv21d = locked_rules_long[
    locked_rules_long["rv21d_vol_pct_min"].ne(8.5)
]

bad_back = locked_rules_long[
    locked_rules_long["tenor"].isin([27, 30, 33])
    & (
        locked_rules_long["model_vrp_log_min"].ne(0.70)
        | locked_rules_long["model_vrp_z_3m_min"].ne(0.70)
        | locked_rules_long["model_vrp_z_1y_min"].ne(0.70)
    )
]

bad_normalized_pnl = base[
    base["normalized_pnl_dollars"].notna()
    & base["normalized_pnl_per_day"].notna()
    & (
        (
            base["normalized_pnl_dollars"] / base["tenor"].replace(0, np.nan)
        )
        - base["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
]

universes_found = sorted(selected_trade_panel["selection_universe"].dropna().unique().tolist())

primary_has_rows = len(primary_summary) == 1
primary_trade_count = int(primary_summary["trades"].iloc[0]) if primary_has_rows else 0
primary_avg_day = float(primary_summary["avg_pnl_per_day"].iloc[0]) if primary_has_rows else np.nan

add_validation(
    validation_rows,
    "base_panel_loaded",
    "PASS" if len(base) > 0 else "FAIL",
    f"rows={len(base):,}; path={base_panel_path}",
)

add_validation(
    validation_rows,
    "required_base_columns_available",
    "PASS" if not missing_base_cols else "FAIL",
    f"missing={missing_base_cols}",
)

add_validation(
    validation_rows,
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    validation_rows,
    "locked_rules_wide_created",
    "PASS" if len(locked_rules_wide) == 1 else "FAIL",
    f"rows={len(locked_rules_wide):,}",
)

add_validation(
    validation_rows,
    "locked_rules_long_all_tenors",
    "PASS" if len(locked_rules_long) == len(ALL_TENORS) else "FAIL",
    f"rows={len(locked_rules_long):,}; expected={len(ALL_TENORS):,}",
)

add_validation(
    validation_rows,
    "nine_day_excluded",
    "PASS" if bad_9d_include.empty else "FAIL",
    f"bad_rows={len(bad_9d_include):,}",
)

add_validation(
    validation_rows,
    "eighteen_day_in_front",
    "PASS" if bad_18d_bucket.empty else "FAIL",
    f"bad_rows={len(bad_18d_bucket):,}",
)

add_validation(
    validation_rows,
    "back_locked",
    "PASS" if bad_back.empty else "FAIL",
    f"bad_rows={len(bad_back):,}",
)

add_validation(
    validation_rows,
    "z3m_equals_z1y",
    "PASS" if bad_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_z_equality):,}",
)

add_validation(
    validation_rows,
    "RSI_fixed_70",
    "PASS" if bad_rsi.empty else "FAIL",
    f"bad_rows={len(bad_rsi):,}",
)

add_validation(
    validation_rows,
    "rv21d_floor_fixed_8_5",
    "PASS" if bad_rv21d.empty else "FAIL",
    f"bad_rows={len(bad_rv21d):,}",
)

add_validation(
    validation_rows,
    "uses_rv21d_threshold_contract",
    "PASS" if bad_rv21d_contract.empty else "FAIL",
    f"bad_rows={len(bad_rv21d_contract):,}",
)

add_validation(
    validation_rows,
    "no_forecast_vol_threshold_columns",
    "PASS" if not old_forecast_threshold_cols else "FAIL",
    f"old_forecast_threshold_cols={old_forecast_threshold_cols}",
)

add_validation(
    validation_rows,
    "forecast_vol_pct_diagnostic_only",
    "PASS" if bad_forecast_threshold_flag.empty else "FAIL",
    f"bad_rows={len(bad_forecast_threshold_flag):,}",
)

add_validation(
    validation_rows,
    "panel_rule_application_created",
    "PASS" if len(panel) > 0 and len(qualified_complete) > 0 else "FAIL",
    f"panel_rows={len(panel):,}; qualified_complete_rows={len(qualified_complete):,}",
)

add_validation(
    validation_rows,
    "all_selection_universes_created",
    "PASS" if universes_found == sorted(SELECTION_UNIVERSES) else "FAIL",
    f"found={universes_found}",
)

add_validation(
    validation_rows,
    "primary_summary_created",
    "PASS" if primary_has_rows and primary_trade_count > 0 else "FAIL",
    f"primary_rows={len(primary_summary):,}; primary_trades={primary_trade_count:,}; primary_avg_pnl_per_day={primary_avg_day}",
)

add_validation(
    validation_rows,
    "normalized_pnl_per_day_formula",
    "PASS" if bad_normalized_pnl.empty else "FAIL",
    f"bad_rows={len(bad_normalized_pnl):,}",
)

add_validation(
    validation_rows,
    "no_dte_specific_smoothing",
    "PASS",
    "Locked bucket benchmark only; no DTE-specific smoothing used.",
)

add_validation(
    validation_rows,
    "no_secondary",
    "PASS",
    "Core only.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed or added.",
)

add_validation(
    validation_rows,
    "no_production_lock",
    "PASS",
    "No production/intraday implementation performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"].eq("FAIL")]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1_RV21D_repaired_v2.ipynb",
    "cell": "Cell 11R — Lock repaired Core bucket map and parameters",
    "timestamp": CELL11R_TIMESTAMP,

    "locked_core_version": LOCKED_CORE_VERSION,
    "locked_core_decision_id": LOCKED_CORE_DECISION_ID,
    "locked_model_spec": LOCKED_MODEL_SPEC,
    "source_research_candidate_id": LOCKED_CORE_SOURCE_CANDIDATE_ID,
    "source_research_candidate_subfamily": LOCKED_CORE_SOURCE_CANDIDATE_SUBFAMILY,

    "base_panel_path": str(base_panel_path),

    "locked_rules_wide_csv_path": str(locked_rules_wide_csv_path),
    "locked_rules_wide_parquet_path": str(locked_rules_wide_parquet_path),
    "locked_rules_long_csv_path": str(locked_rules_long_csv_path),
    "locked_rules_long_parquet_path": str(locked_rules_long_parquet_path),

    "locked_panel_path": str(locked_panel_path),
    "locked_panel_csv_path": str(locked_panel_csv_path),

    "selected_trade_panel_path": str(selected_trade_panel_path),
    "selected_trade_panel_csv_path": str(selected_trade_panel_csv_path),

    "summary_path": str(summary_path),
    "by_bucket_path": str(by_bucket_path),
    "by_tenor_path": str(by_tenor_path),
    "by_year_path": str(by_year_path),
    "by_tenor_year_path": str(by_tenor_year_path),
    "pass_diagnostics_path": str(pass_diagnostics_path),
    "validation_path": str(validation_path),
    "manifest_path": str(manifest_path),

    "primary_selection_universe": PRIMARY_SELECTION_UNIVERSE,
    "primary_ranking_metric": PRIMARY_RANKING_METRIC,

    "effective_bucket_map": LOCKED_EFFECTIVE_TENOR_BUCKET_MAP,
    "locked_rules_by_tenor": LOCKED_RULES_BY_TENOR,

    "threshold_contract": "rv21d_vol_pct > rv21d_vol_pct_min",
    "forecast_vol_pct_role": "model denominator / diagnostic only",

    "locked_core_rules": {
        "excluded": {
            "tenors": [9],
        },
        "front": {
            "tenors": [12, 15, 18],
            "model_vrp_log_min": 0.60,
            "model_vrp_z_3m_min": 0.65,
            "model_vrp_z_1y_min": 0.65,
            "RSI14_max": 70.0,
            "rv21d_vol_pct_min": 8.5,
        },
        "middle": {
            "tenors": [21, 24],
            "model_vrp_log_min": 0.65,
            "model_vrp_z_3m_min": 0.70,
            "model_vrp_z_1y_min": 0.70,
            "RSI14_max": 70.0,
            "rv21d_vol_pct_min": 8.5,
        },
        "back": {
            "tenors": [27, 30, 33],
            "model_vrp_log_min": 0.70,
            "model_vrp_z_3m_min": 0.70,
            "model_vrp_z_1y_min": 0.70,
            "RSI14_max": 70.0,
            "rv21d_vol_pct_min": 8.5,
        },
    },

    "constraints": [
        "9D excluded from Core.",
        "18D moved into Front.",
        "Core Front = 12D/15D/18D.",
        "Core Middle = 21D/24D.",
        "Core Back = 27D/30D/33D.",
        "Back parameters locked.",
        "z3m and z1y equal within each effective bucket.",
        "RSI14 cap fixed at 70.",
        "RV21D floor fixed at 8.5.",
        "RV21D is the only vol-floor threshold.",
        "forecast_vol_pct is diagnostic only.",
        "No DTE-specific smoothing.",
        "No Secondary.",
        "No sizing changes.",
        "No production/intraday implementation.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 11R validation failed. Review validation output above.")

# ======================================================================================
# 8. Display review
# ======================================================================================

print("=" * 100)
print("Cell 11R — Lock repaired Core bucket map and parameters")
print("=" * 100)
print(f"Locked core version:                {LOCKED_CORE_VERSION}")
print(f"Locked decision ID:                 {LOCKED_CORE_DECISION_ID}")
print(f"Base panel:                         {base_panel_path}")
print(f"Panel rows:                         {len(panel):,}")
print(f"Qualified complete rows:            {len(qualified_complete):,}")
print(f"Selected trade panel rows:          {len(selected_trade_panel):,}")
print(f"Primary universe:                   {PRIMARY_SELECTION_UNIVERSE}")
print(f"Primary metric:                     {PRIMARY_RANKING_METRIC}")
print("Effective bucket map:               Excluded 9D; Front 12/15/18; Middle 21/24; Back 27/30/33")
print("Threshold contract:                 rv21d_vol_pct > rv21d_vol_pct_min")
print("DTE-specific smoothing used:         False")
print("Secondary locked here:              False")
print("Sizing locked here:                 False")
print("Production locked here:             False")

print()
print("Validation:")
display(validation)

print()
print("Locked rules — wide:")
display(locked_rules_wide)

print()
print("Locked rules — long:")
display(locked_rules_long)

print()
print("Primary summary:")
display(primary_summary)

print()
print("Primary by effective bucket:")
display(primary_bucket)

print()
print("Primary by tenor:")
display(primary_tenor)

print()
print("All-universe summary:")
display(summary)

print()
print("By year:")
display(by_year)

print()
print("By tenor/year:")
display(by_tenor_year)

print()
print("Pass diagnostics:")
display(pass_diagnostics)

print()
print("Saved outputs:")
for p in [
    locked_rules_wide_csv_path,
    locked_rules_wide_parquet_path,
    locked_rules_long_csv_path,
    locked_rules_long_parquet_path,
    locked_panel_path,
    locked_panel_csv_path,
    selected_trade_panel_path,
    selected_trade_panel_csv_path,
    summary_path,
    by_bucket_path,
    by_tenor_path,
    by_year_path,
    by_tenor_year_path,
    pass_diagnostics_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 11R PASSED — CORE REPAIRED V1 LOCKED")

In [ ]:
# ======================================================================================
# Cell 12R — Secondary Back sweep, Core-gated
# ======================================================================================
#
# Objective:
#   Sweep Secondary Back parameters as an incremental layer behind locked Core repaired v1.
#
# Locked Core:
#   Source:
#       Cell 11R — core_repaired_v1_locked_001
#
#   Core remains unchanged:
#       9D excluded
#       Front  = 12D / 15D / 18D
#       Middle = 21D / 24D
#       Back   = 27D / 30D / 33D
#
# Secondary Back sweep:
#   Tenors:
#       27D / 30D / 33D only
#
#   VRP log:
#       [0.65, 0.70]
#
#   z-score:
#       [0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60]
#
#   z constraint:
#       z3m = z1y
#
#   RSI14 cap:
#       [73, 74, 75, 76, 77]
#
#   RV21D floor:
#       [6.5, 7.0]
#
# Candidate count:
#       2 * 7 * 5 * 2 = 140
#
# Core-gating rule:
#   If any locked Core tenor qualifies on a date:
#       Secondary is ignored.
#
#   If no locked Core tenor qualifies on a date:
#       Secondary Back candidates are tested.
#
# Evaluation views:
#   1. Program view:
#       one trade per date
#       Core first, else Secondary Back
#
#   2. Core-lock continuity view:
#       Cell 11R-style primary Core universe:
#           one trade per bucket per date
#       plus Secondary Back on Core-empty dates.
#
# This cell does NOT:
#   1. Reopen Core.
#   2. Test Secondary Front or Secondary Middle.
#   3. Lock Secondary.
#   4. Change sizing.
#   5. Implement production/intraday logic.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import itertools
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 420)
pd.set_option("display.width", 720)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL12R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

LOCKED_CORE_VERSION = "core_repaired_v1"
LOCKED_CORE_DECISION_ID = "core_repaired_v1_locked_001"

SECONDARY_SWEEP_VERSION = "secondary_back_core_gated_sweep_v1"
SECONDARY_SWEEP_FAMILY = "secondary_back_core_gated"

ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]
SECONDARY_BACK_TENORS = [27, 30, 33]

BUCKET_ORDER_MAP = {
    "Excluded": 0,
    "Front": 1,
    "Middle": 2,
    "Back": 3,
}

BUCKET_CENTER = {
    "Front": 15,
    "Middle": 21,
    "Back": 30,
}

PRIMARY_CORE_CONTINUITY_UNIVERSE = "core_primary_one_trade_per_bucket_per_date"
CORE_PROGRAM_UNIVERSE = "core_program_one_trade_per_date"

SECONDARY_INCREMENTAL_UNIVERSE = "secondary_back_incremental_core_empty_dates"
COMBINED_PROGRAM_UNIVERSE = "combined_program_core_first_else_secondary_back"
COMBINED_CONTINUITY_UNIVERSE = "combined_continuity_core_primary_plus_secondary_back"

SECONDARY_VRP_LOG_GRID = [0.65, 0.70]
SECONDARY_Z_GRID = [0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60]
SECONDARY_RSI_GRID = [73.0, 74.0, 75.0, 76.0, 77.0]
SECONDARY_RV21D_GRID = [6.5, 7.0]

TOTAL_PROGRAM_FREQUENCY_TARGET = 0.33
COMBINED_WIN_RATE_TARGET = 0.80

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def add_rank_columns(d, group_cols, suffix, bucket_center=30):
    x = d.copy()

    if x.empty:
        return x

    x[f"rank_z_1y_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    x[f"rank_z_3m_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    x[f"rank_vrp_log_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    x[f"avg_signal_rank_{suffix}"] = x[
        [
            f"rank_z_1y_{suffix}",
            f"rank_z_3m_{suffix}",
            f"rank_vrp_log_{suffix}",
        ]
    ].mean(axis=1)

    if "effective_tenor_bucket" in x.columns:
        x["bucket_center_tenor"] = x["effective_tenor_bucket"].map(BUCKET_CENTER)
        x["bucket_center_tenor"] = x["bucket_center_tenor"].fillna(bucket_center)
    else:
        x["bucket_center_tenor"] = bucket_center

    x["distance_to_bucket_center"] = (x["tenor"] - x["bucket_center_tenor"]).abs()

    return x


def select_core_universes(core_qualified_complete):
    q = core_qualified_complete.copy()

    if q.empty:
        return {
            PRIMARY_CORE_CONTINUITY_UNIVERSE: q.copy(),
            CORE_PROGRAM_UNIVERSE: q.copy(),
        }

    q = add_rank_columns(
        q,
        group_cols=["trade_date", "effective_tenor_bucket"],
        suffix="core_bucket_date",
    )

    q = add_rank_columns(
        q,
        group_cols=["trade_date"],
        suffix="core_date",
    )

    core_primary = (
        q.sort_values(
            [
                "trade_date",
                "effective_tenor_bucket_order",
                "avg_signal_rank_core_bucket_date",
                "distance_to_bucket_center",
                "rank_z_1y_core_bucket_date",
                "rank_z_3m_core_bucket_date",
                "rank_vrp_log_core_bucket_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date", "effective_tenor_bucket"], as_index=False)
        .head(1)
        .copy()
    )

    core_program = (
        q.sort_values(
            [
                "trade_date",
                "avg_signal_rank_core_date",
                "rank_z_1y_core_date",
                "rank_z_3m_core_date",
                "rank_vrp_log_core_date",
                "distance_to_bucket_center",
                "effective_tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return {
        PRIMARY_CORE_CONTINUITY_UNIVERSE: core_primary,
        CORE_PROGRAM_UNIVERSE: core_program,
    }


def select_secondary_one_per_candidate_date(secondary_qualified_complete):
    q = secondary_qualified_complete.copy()

    if q.empty:
        return q

    q["bucket_center_tenor"] = 30
    q["distance_to_bucket_center"] = (q["tenor"] - 30).abs()

    q["rank_z_1y_secondary_date"] = (
        q.groupby(["secondary_candidate_id", "trade_date"])["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    q["rank_z_3m_secondary_date"] = (
        q.groupby(["secondary_candidate_id", "trade_date"])["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    q["rank_vrp_log_secondary_date"] = (
        q.groupby(["secondary_candidate_id", "trade_date"])["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    q["avg_signal_rank_secondary_date"] = q[
        [
            "rank_z_1y_secondary_date",
            "rank_z_3m_secondary_date",
            "rank_vrp_log_secondary_date",
        ]
    ].mean(axis=1)

    selected = (
        q.sort_values(
            [
                "secondary_candidate_id",
                "trade_date",
                "avg_signal_rank_secondary_date",
                "distance_to_bucket_center",
                "rank_z_1y_secondary_date",
                "rank_z_3m_secondary_date",
                "rank_vrp_log_secondary_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["secondary_candidate_id", "trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return selected


def summarize_trade_set(df, group_cols, total_eligible_dates=None):
    d = df.copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    numeric_cols = [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "forecast_vol_pct",
        "rv21d_vol_pct",
        "threshold_model_vrp_log",
        "threshold_model_vrp_z_3m",
        "threshold_model_vrp_z_1y",
        "threshold_RSI14_max",
        "threshold_rv21d_vol_pct_min",
        "secondary_threshold_model_vrp_log",
        "secondary_threshold_model_vrp_z_3m",
        "secondary_threshold_model_vrp_z_1y",
        "secondary_threshold_RSI14_max",
        "secondary_threshold_rv21d_vol_pct_min",
    ]

    for c in numeric_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d = d.sort_values(["trade_date", "tenor"]).copy()

    grouped = d.groupby(group_cols, dropna=False, observed=False) if group_cols else [((), d)]

    rows = []

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {}

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        unique_dates = int(g["trade_date"].nunique())
        signal_date_frequency = (
            unique_dates / total_eligible_dates
            if total_eligible_dates and total_eligible_dates > 0
            else np.nan
        )

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": unique_dates,
            "signal_date_frequency": float(signal_date_frequency) if pd.notna(signal_date_frequency) else np.nan,
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        for signal_col in [
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "threshold_model_vrp_log",
            "threshold_model_vrp_z_3m",
            "threshold_model_vrp_z_1y",
            "threshold_RSI14_max",
            "threshold_rv21d_vol_pct_min",
            "secondary_threshold_model_vrp_log",
            "secondary_threshold_model_vrp_z_3m",
            "secondary_threshold_model_vrp_z_1y",
            "secondary_threshold_RSI14_max",
            "secondary_threshold_rv21d_vol_pct_min",
        ]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        rows.append(row)

    return pd.DataFrame(rows)


def attach_candidate_grid(summary_df, candidate_grid):
    if summary_df.empty:
        return summary_df

    return summary_df.merge(
        candidate_grid,
        on="secondary_candidate_id",
        how="left",
        validate="many_to_one",
    )


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


# ======================================================================================
# 2. Load inputs
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01R_unified_fds_signal_base_panel_with_rv21d_*.parquet",
    required=True,
)

locked_core_rules_long_path = latest_file(
    OUT_PROCESSED_DIR,
    "11R_core_repaired_v1_locked_rules_long_*.parquet",
    required=True,
)

locked_core_rules_wide_path = latest_file(
    OUT_PROCESSED_DIR,
    "11R_core_repaired_v1_locked_rules_wide_*.parquet",
    required=True,
)

base = pd.read_parquet(base_panel_path)
locked_core_rules_long = pd.read_parquet(locked_core_rules_long_path)
locked_core_rules_wide = pd.read_parquet(locked_core_rules_wide_path)

base["trade_date"] = pd.to_datetime(base["trade_date"], errors="coerce").dt.normalize()

for c in [
    "tenor",
    "tenor_bucket_order",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")

for c in [
    "tenor",
    "original_tenor_bucket_order",
    "effective_tenor_bucket_order",
    "model_vrp_log_min",
    "model_vrp_z_min",
    "model_vrp_z_3m_min",
    "model_vrp_z_1y_min",
    "RSI14_max",
    "rv21d_vol_pct_min",
]:
    if c in locked_core_rules_long.columns:
        locked_core_rules_long[c] = pd.to_numeric(locked_core_rules_long[c], errors="coerce")

total_eligible_signal_dates = int(base["trade_date"].nunique())

# ======================================================================================
# 3. Apply locked Core rules and build Core baselines
# ======================================================================================

core_thresholds = locked_core_rules_long.rename(columns={
    "model_vrp_log_min": "threshold_model_vrp_log",
    "model_vrp_z_3m_min": "threshold_model_vrp_z_3m",
    "model_vrp_z_1y_min": "threshold_model_vrp_z_1y",
    "RSI14_max": "threshold_RSI14_max",
    "rv21d_vol_pct_min": "threshold_rv21d_vol_pct_min",
})

core_threshold_cols = [
    "tenor",
    "original_tenor_bucket",
    "original_tenor_bucket_order",
    "effective_tenor_bucket",
    "effective_tenor_bucket_order",
    "include_tenor",
    "threshold_model_vrp_log",
    "threshold_model_vrp_z_3m",
    "threshold_model_vrp_z_1y",
    "threshold_RSI14_max",
    "threshold_rv21d_vol_pct_min",
]

base_for_join = base.copy()

base_for_join = base_for_join.rename(columns={
    "tenor_bucket": "source_tenor_bucket",
    "tenor_bucket_order": "source_tenor_bucket_order",
})

core_panel = base_for_join.merge(
    core_thresholds[core_threshold_cols],
    on="tenor",
    how="inner",
    validate="many_to_one",
)

core_signal_available = (
    core_panel["include_tenor"].astype(bool)
    & core_panel["model_vrp_log"].notna()
    & core_panel["model_vrp_z_3m"].notna()
    & core_panel["model_vrp_z_1y"].notna()
    & core_panel["RSI14"].notna()
    & core_panel["rv21d_vol_pct"].notna()
)

core_pass_mask = (
    core_signal_available
    & (core_panel["model_vrp_log"] > core_panel["threshold_model_vrp_log"])
    & (core_panel["model_vrp_z_3m"] > core_panel["threshold_model_vrp_z_3m"])
    & (core_panel["model_vrp_z_1y"] > core_panel["threshold_model_vrp_z_1y"])
    & (core_panel["RSI14"] < core_panel["threshold_RSI14_max"])
    & (core_panel["rv21d_vol_pct"] > core_panel["threshold_rv21d_vol_pct_min"])
)

core_panel["core_repaired_v1_pass"] = core_pass_mask

core_qualified_all_dates = core_panel[core_pass_mask].copy()

core_qualified_complete = core_panel[
    core_pass_mask
    & core_panel["normalized_pnl_dollars"].notna()
    & core_panel["normalized_pnl_per_day"].notna()
].copy()

core_gate_dates = sorted(core_qualified_all_dates["trade_date"].dropna().unique().tolist())
core_gate_dates_set = set(core_gate_dates)

core_empty_dates = sorted(set(base["trade_date"].dropna().unique().tolist()) - core_gate_dates_set)

core_universes = select_core_universes(core_qualified_complete)

core_primary_continuity = core_universes[PRIMARY_CORE_CONTINUITY_UNIVERSE].copy()
core_program = core_universes[CORE_PROGRAM_UNIVERSE].copy()

core_primary_continuity["source_layer"] = "Core"
core_primary_continuity["program_leg"] = "Core"
core_primary_continuity["selection_universe"] = PRIMARY_CORE_CONTINUITY_UNIVERSE
core_primary_continuity["secondary_candidate_id"] = "CORE_ONLY_BASELINE"
core_primary_continuity["secondary_candidate_subfamily"] = "core_only_baseline"

core_program["source_layer"] = "Core"
core_program["program_leg"] = "Core"
core_program["selection_universe"] = CORE_PROGRAM_UNIVERSE
core_program["secondary_candidate_id"] = "CORE_ONLY_BASELINE"
core_program["secondary_candidate_subfamily"] = "core_only_baseline"

# ======================================================================================
# 4. Build Secondary Back grid
# ======================================================================================

candidate_rows = []

idx = 1

for vrp_log, z, rsi_cap, rv_floor in itertools.product(
    SECONDARY_VRP_LOG_GRID,
    SECONDARY_Z_GRID,
    SECONDARY_RSI_GRID,
    SECONDARY_RV21D_GRID,
):
    cid = f"secondary_back_{idx:04d}"

    candidate_rows.append({
        "secondary_candidate_id": cid,
        "secondary_sweep_version": SECONDARY_SWEEP_VERSION,
        "secondary_candidate_family": SECONDARY_SWEEP_FAMILY,
        "secondary_candidate_subfamily": "secondary_back_only",
        "secondary_candidate_description": (
            f"Secondary Back only, Core-gated; log>{vrp_log:.2f}, "
            f"z3m/z1y>{z:.2f}, RSI14<{rsi_cap:.0f}, rv21d>{rv_floor:.1f}"
        ),

        "secondary_tenors": "27,30,33",
        "secondary_bucket": "Back",
        "secondary_model_vrp_log_min": float(vrp_log),
        "secondary_model_vrp_z_min": float(z),
        "secondary_model_vrp_z_3m_min": float(z),
        "secondary_model_vrp_z_1y_min": float(z),
        "secondary_RSI14_max": float(rsi_cap),
        "secondary_rv21d_vol_pct_min": float(rv_floor),

        "z3m_equals_z1y": True,
        "uses_rv21d_threshold_contract": True,
        "forecast_vol_pct_threshold_used": False,
        "core_gated": True,
        "secondary_front_tested": False,
        "secondary_middle_tested": False,
        "secondary_back_tested": True,
        "secondary_locked_here": False,
        "sizing_locked_here": False,
        "production_locked_here": False,
    })

    idx += 1

secondary_candidate_grid = pd.DataFrame(candidate_rows)

expected_candidate_count = (
    len(SECONDARY_VRP_LOG_GRID)
    * len(SECONDARY_Z_GRID)
    * len(SECONDARY_RSI_GRID)
    * len(SECONDARY_RV21D_GRID)
)

# ======================================================================================
# 5. Apply Secondary Back candidates on Core-empty dates only
# ======================================================================================

secondary_base = base_for_join[
    base_for_join["tenor"].isin(SECONDARY_BACK_TENORS)
    & base_for_join["trade_date"].isin(core_empty_dates)
].copy()

secondary_base["secondary_bucket"] = "Back"
secondary_base["effective_tenor_bucket"] = "Back"
secondary_base["effective_tenor_bucket_order"] = BUCKET_ORDER_MAP["Back"]

secondary_panel = secondary_base.merge(
    secondary_candidate_grid,
    how="cross",
)

secondary_panel = secondary_panel.rename(columns={
    "secondary_model_vrp_log_min": "secondary_threshold_model_vrp_log",
    "secondary_model_vrp_z_3m_min": "secondary_threshold_model_vrp_z_3m",
    "secondary_model_vrp_z_1y_min": "secondary_threshold_model_vrp_z_1y",
    "secondary_RSI14_max": "secondary_threshold_RSI14_max",
    "secondary_rv21d_vol_pct_min": "secondary_threshold_rv21d_vol_pct_min",
})

secondary_signal_available = (
    secondary_panel["model_vrp_log"].notna()
    & secondary_panel["model_vrp_z_3m"].notna()
    & secondary_panel["model_vrp_z_1y"].notna()
    & secondary_panel["RSI14"].notna()
    & secondary_panel["rv21d_vol_pct"].notna()
)

secondary_pass_mask = (
    secondary_signal_available
    & (secondary_panel["model_vrp_log"] > secondary_panel["secondary_threshold_model_vrp_log"])
    & (secondary_panel["model_vrp_z_3m"] > secondary_panel["secondary_threshold_model_vrp_z_3m"])
    & (secondary_panel["model_vrp_z_1y"] > secondary_panel["secondary_threshold_model_vrp_z_1y"])
    & (secondary_panel["RSI14"] < secondary_panel["secondary_threshold_RSI14_max"])
    & (secondary_panel["rv21d_vol_pct"] > secondary_panel["secondary_threshold_rv21d_vol_pct_min"])
)

secondary_panel["secondary_back_pass"] = secondary_pass_mask

secondary_qualified_all_dates = secondary_panel[secondary_pass_mask].copy()

secondary_qualified_complete = secondary_panel[
    secondary_pass_mask
    & secondary_panel["normalized_pnl_dollars"].notna()
    & secondary_panel["normalized_pnl_per_day"].notna()
].copy()

secondary_selected = select_secondary_one_per_candidate_date(secondary_qualified_complete)

secondary_selected["source_layer"] = "Secondary"
secondary_selected["program_leg"] = "Secondary_Back"
secondary_selected["selection_universe"] = SECONDARY_INCREMENTAL_UNIVERSE

# ======================================================================================
# 6. Build combined program and continuity panels
# ======================================================================================

program_frames = []
continuity_frames = []

core_program_for_candidates_base = core_program.copy()
core_continuity_for_candidates_base = core_primary_continuity.copy()

for _, cand in secondary_candidate_grid.iterrows():
    cid = cand["secondary_candidate_id"]

    sec = secondary_selected[
        secondary_selected["secondary_candidate_id"].eq(cid)
    ].copy()

    core_prog = core_program_for_candidates_base.copy()
    core_prog["secondary_candidate_id"] = cid
    core_prog["secondary_candidate_subfamily"] = cand["secondary_candidate_subfamily"]

    core_cont = core_continuity_for_candidates_base.copy()
    core_cont["secondary_candidate_id"] = cid
    core_cont["secondary_candidate_subfamily"] = cand["secondary_candidate_subfamily"]

    combined_program = pd.concat([core_prog, sec], ignore_index=True)
    combined_program["selection_universe"] = COMBINED_PROGRAM_UNIVERSE

    combined_continuity = pd.concat([core_cont, sec], ignore_index=True)
    combined_continuity["selection_universe"] = COMBINED_CONTINUITY_UNIVERSE

    program_frames.append(combined_program)
    continuity_frames.append(combined_continuity)

combined_program_panel = pd.concat(program_frames, ignore_index=True) if program_frames else pd.DataFrame()
combined_continuity_panel = pd.concat(continuity_frames, ignore_index=True) if continuity_frames else pd.DataFrame()

# Candidate-level output panels.
secondary_selected_panel = secondary_selected.copy()

# ======================================================================================
# 7. Summaries
# ======================================================================================

core_program_summary = summarize_trade_set(
    core_program.assign(selection_universe=CORE_PROGRAM_UNIVERSE),
    group_cols=["secondary_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

core_continuity_summary = summarize_trade_set(
    core_primary_continuity.assign(selection_universe=PRIMARY_CORE_CONTINUITY_UNIVERSE),
    group_cols=["secondary_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

secondary_incremental_summary = summarize_trade_set(
    secondary_selected_panel,
    group_cols=["secondary_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

secondary_incremental_by_tenor = summarize_trade_set(
    secondary_selected_panel,
    group_cols=["secondary_candidate_id", "selection_universe", "tenor"],
    total_eligible_dates=total_eligible_signal_dates,
)

secondary_incremental_by_year = (
    secondary_selected_panel.assign(year=secondary_selected_panel["trade_date"].dt.year.astype(int))
    if not secondary_selected_panel.empty
    else secondary_selected_panel.copy()
)

secondary_incremental_by_year = summarize_trade_set(
    secondary_incremental_by_year,
    group_cols=["secondary_candidate_id", "selection_universe", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)

combined_program_summary = summarize_trade_set(
    combined_program_panel,
    group_cols=["secondary_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

combined_continuity_summary = summarize_trade_set(
    combined_continuity_panel,
    group_cols=["secondary_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

combined_program_by_year = (
    combined_program_panel.assign(year=combined_program_panel["trade_date"].dt.year.astype(int))
    if not combined_program_panel.empty
    else combined_program_panel.copy()
)

combined_program_by_year = summarize_trade_set(
    combined_program_by_year,
    group_cols=["secondary_candidate_id", "selection_universe", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)

combined_continuity_by_year = (
    combined_continuity_panel.assign(year=combined_continuity_panel["trade_date"].dt.year.astype(int))
    if not combined_continuity_panel.empty
    else combined_continuity_panel.copy()
)

combined_continuity_by_year = summarize_trade_set(
    combined_continuity_by_year,
    group_cols=["secondary_candidate_id", "selection_universe", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)

secondary_incremental_summary = attach_candidate_grid(secondary_incremental_summary, secondary_candidate_grid)
secondary_incremental_by_tenor = attach_candidate_grid(secondary_incremental_by_tenor, secondary_candidate_grid)
secondary_incremental_by_year = attach_candidate_grid(secondary_incremental_by_year, secondary_candidate_grid)
combined_program_summary = attach_candidate_grid(combined_program_summary, secondary_candidate_grid)
combined_continuity_summary = attach_candidate_grid(combined_continuity_summary, secondary_candidate_grid)
combined_program_by_year = attach_candidate_grid(combined_program_by_year, secondary_candidate_grid)
combined_continuity_by_year = attach_candidate_grid(combined_continuity_by_year, secondary_candidate_grid)

# ======================================================================================
# 8. Candidate comparison
# ======================================================================================

core_program_ref = core_program_summary.iloc[0].to_dict() if len(core_program_summary) else {}
core_continuity_ref = core_continuity_summary.iloc[0].to_dict() if len(core_continuity_summary) else {}

comparison = combined_program_summary.copy()

if not comparison.empty:
    # Attach incremental Secondary metrics.
    sec_cols = [
        "secondary_candidate_id",
        "trades",
        "unique_trade_dates",
        "signal_date_frequency",
        "win_rate",
        "total_pnl",
        "avg_pnl_per_day",
        "profit_factor",
        "profit_factor_pnl_per_day",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",
        "avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026",
    ]

    sec_metrics = secondary_incremental_summary[
        [c for c in sec_cols if c in secondary_incremental_summary.columns]
    ].copy()

    sec_metrics = sec_metrics.rename(columns={
        "trades": "secondary_trades",
        "unique_trade_dates": "secondary_unique_trade_dates",
        "signal_date_frequency": "secondary_signal_date_frequency",
        "win_rate": "secondary_win_rate",
        "total_pnl": "secondary_total_pnl",
        "avg_pnl_per_day": "secondary_avg_pnl_per_day",
        "profit_factor": "secondary_profit_factor",
        "profit_factor_pnl_per_day": "secondary_profit_factor_pnl_per_day",
        "pnl_per_day_drawdown": "secondary_pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum": "secondary_worst_20_trade_pnl_per_day_sum",
        "avg_pnl_per_day_2025": "secondary_avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026": "secondary_avg_pnl_per_day_2026",
    })

    comparison = comparison.merge(
        sec_metrics,
        on="secondary_candidate_id",
        how="left",
        validate="one_to_one",
    )

    # Deltas versus Core program one-trade-per-date.
    for metric in [
        "trades",
        "unique_trade_dates",
        "signal_date_frequency",
        "win_rate",
        "total_pnl",
        "avg_pnl_per_day",
        "profit_factor",
        "profit_factor_pnl_per_day",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",
        "avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026",
    ]:
        if metric in comparison.columns and metric in core_program_ref:
            comparison[f"delta_program_{metric}_vs_core_only"] = comparison[metric] - core_program_ref[metric]

    comparison["passes_frequency_target"] = comparison["signal_date_frequency"] >= TOTAL_PROGRAM_FREQUENCY_TARGET
    comparison["passes_win_rate_target"] = comparison["win_rate"] >= COMBINED_WIN_RATE_TARGET
    comparison["passes_program_targets"] = (
        comparison["passes_frequency_target"]
        & comparison["passes_win_rate_target"]
    )

    comparison = comparison.sort_values(
        [
            "passes_program_targets",
            "signal_date_frequency",
            "win_rate",
            "avg_pnl_per_day",
            "profit_factor_pnl_per_day",
            "avg_pnl_per_day_2026",
        ],
        ascending=[False, False, False, False, False, False],
    ).reset_index(drop=True)

# Continuity comparison.
continuity_comparison = combined_continuity_summary.copy()

if not continuity_comparison.empty:
    for metric in [
        "trades",
        "unique_trade_dates",
        "signal_date_frequency",
        "win_rate",
        "total_pnl",
        "avg_pnl_per_day",
        "profit_factor",
        "profit_factor_pnl_per_day",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",
        "avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026",
    ]:
        if metric in continuity_comparison.columns and metric in core_continuity_ref:
            continuity_comparison[f"delta_continuity_{metric}_vs_core_primary"] = (
                continuity_comparison[metric] - core_continuity_ref[metric]
            )

    continuity_comparison = continuity_comparison.sort_values(
        [
            "signal_date_frequency",
            "win_rate",
            "avg_pnl_per_day",
            "profit_factor_pnl_per_day",
            "avg_pnl_per_day_2026",
        ],
        ascending=[False, False, False, False, False],
    ).reset_index(drop=True)

# ======================================================================================
# 9. Save outputs
# ======================================================================================

candidate_grid_csv_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_grid_{CELL12R_TIMESTAMP}.csv"
candidate_grid_parquet_path = OUT_PROCESSED_DIR / f"12R_secondary_back_core_gated_grid_{CELL12R_TIMESTAMP}.parquet"

core_gate_dates_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_core_gate_dates_{CELL12R_TIMESTAMP}.csv"
core_empty_dates_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_core_empty_dates_{CELL12R_TIMESTAMP}.csv"

secondary_selected_panel_path = OUT_PROCESSED_DIR / f"12R_secondary_back_core_gated_selected_secondary_panel_{CELL12R_TIMESTAMP}.parquet"
secondary_selected_panel_csv_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_selected_secondary_panel_{CELL12R_TIMESTAMP}.csv"

combined_program_panel_path = OUT_PROCESSED_DIR / f"12R_secondary_back_core_gated_combined_program_panel_{CELL12R_TIMESTAMP}.parquet"
combined_program_panel_csv_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_combined_program_panel_{CELL12R_TIMESTAMP}.csv"

combined_continuity_panel_path = OUT_PROCESSED_DIR / f"12R_secondary_back_core_gated_combined_continuity_panel_{CELL12R_TIMESTAMP}.parquet"
combined_continuity_panel_csv_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_combined_continuity_panel_{CELL12R_TIMESTAMP}.csv"

core_program_summary_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_core_program_baseline_{CELL12R_TIMESTAMP}.csv"
core_continuity_summary_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_core_continuity_baseline_{CELL12R_TIMESTAMP}.csv"

secondary_incremental_summary_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_secondary_incremental_summary_{CELL12R_TIMESTAMP}.csv"
secondary_incremental_by_tenor_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_secondary_incremental_by_tenor_{CELL12R_TIMESTAMP}.csv"
secondary_incremental_by_year_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_secondary_incremental_by_year_{CELL12R_TIMESTAMP}.csv"

combined_program_summary_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_combined_program_summary_{CELL12R_TIMESTAMP}.csv"
combined_program_by_year_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_combined_program_by_year_{CELL12R_TIMESTAMP}.csv"

combined_continuity_summary_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_combined_continuity_summary_{CELL12R_TIMESTAMP}.csv"
combined_continuity_by_year_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_combined_continuity_by_year_{CELL12R_TIMESTAMP}.csv"

comparison_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_candidate_comparison_program_view_{CELL12R_TIMESTAMP}.csv"
continuity_comparison_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_candidate_comparison_continuity_view_{CELL12R_TIMESTAMP}.csv"

validation_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_validation_{CELL12R_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"12R_secondary_back_core_gated_manifest_{CELL12R_TIMESTAMP}.json"

secondary_candidate_grid.to_csv(candidate_grid_csv_path, index=False)
secondary_candidate_grid.to_parquet(candidate_grid_parquet_path, index=False)

pd.DataFrame({"trade_date": core_gate_dates}).to_csv(core_gate_dates_path, index=False)
pd.DataFrame({"trade_date": core_empty_dates}).to_csv(core_empty_dates_path, index=False)

secondary_selected_panel.to_parquet(secondary_selected_panel_path, index=False)
secondary_selected_panel.to_csv(secondary_selected_panel_csv_path, index=False)

combined_program_panel.to_parquet(combined_program_panel_path, index=False)
combined_program_panel.to_csv(combined_program_panel_csv_path, index=False)

combined_continuity_panel.to_parquet(combined_continuity_panel_path, index=False)
combined_continuity_panel.to_csv(combined_continuity_panel_csv_path, index=False)

core_program_summary.to_csv(core_program_summary_path, index=False)
core_continuity_summary.to_csv(core_continuity_summary_path, index=False)

secondary_incremental_summary.to_csv(secondary_incremental_summary_path, index=False)
secondary_incremental_by_tenor.to_csv(secondary_incremental_by_tenor_path, index=False)
secondary_incremental_by_year.to_csv(secondary_incremental_by_year_path, index=False)

combined_program_summary.to_csv(combined_program_summary_path, index=False)
combined_program_by_year.to_csv(combined_program_by_year_path, index=False)

combined_continuity_summary.to_csv(combined_continuity_summary_path, index=False)
combined_continuity_by_year.to_csv(combined_continuity_by_year_path, index=False)

comparison.to_csv(comparison_path, index=False)
continuity_comparison.to_csv(continuity_comparison_path, index=False)

# ======================================================================================
# 10. Validation
# ======================================================================================

validation_rows = []

required_base_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

missing_base_cols = [c for c in required_base_cols if c not in base.columns]
models_found = sorted(base["model_spec"].dropna().unique().tolist()) if "model_spec" in base.columns else []

bad_candidate_count = len(secondary_candidate_grid) != expected_candidate_count

bad_z_equality = secondary_candidate_grid[
    secondary_candidate_grid["secondary_model_vrp_z_3m_min"].ne(
        secondary_candidate_grid["secondary_model_vrp_z_1y_min"]
    )
]

bad_secondary_tenor_rows = secondary_selected_panel[
    ~secondary_selected_panel["tenor"].isin(SECONDARY_BACK_TENORS)
] if not secondary_selected_panel.empty else pd.DataFrame()

bad_secondary_core_gate_dates = secondary_selected_panel[
    secondary_selected_panel["trade_date"].isin(core_gate_dates_set)
] if not secondary_selected_panel.empty else pd.DataFrame()

bad_secondary_front_middle = secondary_candidate_grid[
    secondary_candidate_grid["secondary_front_tested"].astype(bool)
    | secondary_candidate_grid["secondary_middle_tested"].astype(bool)
    | ~secondary_candidate_grid["secondary_back_tested"].astype(bool)
]

bad_forecast_threshold_flag = secondary_candidate_grid[
    secondary_candidate_grid["forecast_vol_pct_threshold_used"].astype(bool)
]

bad_rv21d_contract = secondary_candidate_grid[
    ~secondary_candidate_grid["uses_rv21d_threshold_contract"].astype(bool)
]

program_dupes = (
    combined_program_panel.groupby(["secondary_candidate_id", "trade_date"])
    .size()
    .reset_index(name="rows")
)

bad_program_multi_trade_dates = program_dupes[program_dupes["rows"].gt(1)]

secondary_dupes = (
    secondary_selected_panel.groupby(["secondary_candidate_id", "trade_date"])
    .size()
    .reset_index(name="rows")
    if not secondary_selected_panel.empty
    else pd.DataFrame(columns=["secondary_candidate_id", "trade_date", "rows"])
)

bad_secondary_multi_trade_dates = secondary_dupes[secondary_dupes["rows"].gt(1)]

core_program_dates = set(core_program["trade_date"].dropna().unique().tolist())
secondary_dates_by_candidate = (
    secondary_selected_panel.groupby("secondary_candidate_id")["trade_date"]
    .apply(lambda s: set(s.dropna().unique().tolist()))
    .to_dict()
    if not secondary_selected_panel.empty
    else {}
)

overlap_rows = []

for cid, dates in secondary_dates_by_candidate.items():
    overlap = dates.intersection(core_program_dates)
    if overlap:
        overlap_rows.append({
            "secondary_candidate_id": cid,
            "overlap_core_program_dates": len(overlap),
        })

bad_core_secondary_overlap = pd.DataFrame(overlap_rows)

bad_normalized_pnl = base[
    base["normalized_pnl_dollars"].notna()
    & base["normalized_pnl_per_day"].notna()
    & (
        (
            base["normalized_pnl_dollars"] / base["tenor"].replace(0, np.nan)
        )
        - base["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
]

comparison_candidates = sorted(comparison["secondary_candidate_id"].dropna().unique().tolist()) if not comparison.empty else []
expected_candidates = sorted(secondary_candidate_grid["secondary_candidate_id"].tolist())

add_validation(
    validation_rows,
    "base_panel_loaded",
    "PASS" if len(base) > 0 else "FAIL",
    f"rows={len(base):,}; path={base_panel_path}",
)

add_validation(
    validation_rows,
    "required_base_columns_available",
    "PASS" if not missing_base_cols else "FAIL",
    f"missing={missing_base_cols}",
)

add_validation(
    validation_rows,
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    validation_rows,
    "locked_core_rules_loaded",
    "PASS" if len(locked_core_rules_long) == 9 else "FAIL",
    f"rows={len(locked_core_rules_long):,}; path={locked_core_rules_long_path}",
)

add_validation(
    validation_rows,
    "core_gate_dates_created",
    "PASS" if len(core_gate_dates) > 0 and len(core_empty_dates) > 0 else "FAIL",
    f"core_gate_dates={len(core_gate_dates):,}; core_empty_dates={len(core_empty_dates):,}; total_dates={total_eligible_signal_dates:,}",
)

add_validation(
    validation_rows,
    "candidate_count",
    "PASS" if not bad_candidate_count else "FAIL",
    f"candidate_count={len(secondary_candidate_grid):,}; expected={expected_candidate_count:,}",
)

add_validation(
    validation_rows,
    "secondary_back_only",
    "PASS" if bad_secondary_front_middle.empty else "FAIL",
    f"bad_rows={len(bad_secondary_front_middle):,}",
)

add_validation(
    validation_rows,
    "secondary_selected_tenors_back_only",
    "PASS" if bad_secondary_tenor_rows.empty else "FAIL",
    f"bad_rows={len(bad_secondary_tenor_rows):,}",
)

add_validation(
    validation_rows,
    "secondary_only_on_core_empty_dates",
    "PASS" if bad_secondary_core_gate_dates.empty and bad_core_secondary_overlap.empty else "FAIL",
    f"secondary_on_core_gate_rows={len(bad_secondary_core_gate_dates):,}; core_secondary_overlap_candidates={len(bad_core_secondary_overlap):,}",
)

add_validation(
    validation_rows,
    "secondary_one_trade_per_candidate_date",
    "PASS" if bad_secondary_multi_trade_dates.empty else "FAIL",
    f"bad_rows={len(bad_secondary_multi_trade_dates):,}",
)

add_validation(
    validation_rows,
    "combined_program_one_trade_per_candidate_date",
    "PASS" if bad_program_multi_trade_dates.empty else "FAIL",
    f"bad_rows={len(bad_program_multi_trade_dates):,}",
)

add_validation(
    validation_rows,
    "z3m_equals_z1y_for_secondary",
    "PASS" if bad_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_z_equality):,}",
)

add_validation(
    validation_rows,
    "uses_rv21d_threshold_contract",
    "PASS" if bad_rv21d_contract.empty else "FAIL",
    f"bad_rows={len(bad_rv21d_contract):,}",
)

add_validation(
    validation_rows,
    "forecast_vol_pct_diagnostic_only",
    "PASS" if bad_forecast_threshold_flag.empty else "FAIL",
    f"bad_rows={len(bad_forecast_threshold_flag):,}",
)

add_validation(
    validation_rows,
    "all_candidates_compared_program_view",
    "PASS" if comparison_candidates == expected_candidates else "FAIL",
    f"comparison_candidates={len(comparison_candidates):,}; expected={len(expected_candidates):,}",
)

add_validation(
    validation_rows,
    "core_not_reopened",
    "PASS",
    "Locked Core rules loaded from Cell 11R and used only as gate/baseline.",
)

add_validation(
    validation_rows,
    "secondary_not_locked",
    "PASS",
    "Sweep only; no final Secondary lock.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed or added.",
)

add_validation(
    validation_rows,
    "no_production_lock",
    "PASS",
    "No production/intraday implementation performed.",
)

add_validation(
    validation_rows,
    "normalized_pnl_per_day_formula",
    "PASS" if bad_normalized_pnl.empty else "FAIL",
    f"bad_rows={len(bad_normalized_pnl):,}",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"].eq("FAIL")]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1_RV21D_repaired_v2.ipynb",
    "cell": "Cell 12R — Secondary Back sweep, Core-gated",
    "timestamp": CELL12R_TIMESTAMP,

    "locked_core_version": LOCKED_CORE_VERSION,
    "locked_core_decision_id": LOCKED_CORE_DECISION_ID,
    "secondary_sweep_version": SECONDARY_SWEEP_VERSION,

    "base_panel_path": str(base_panel_path),
    "locked_core_rules_long_path": str(locked_core_rules_long_path),
    "locked_core_rules_wide_path": str(locked_core_rules_wide_path),

    "candidate_count": int(len(secondary_candidate_grid)),
    "expected_candidate_count": int(expected_candidate_count),

    "total_eligible_signal_dates": int(total_eligible_signal_dates),
    "core_gate_dates": int(len(core_gate_dates)),
    "core_empty_dates": int(len(core_empty_dates)),

    "secondary_parameter_grid": {
        "tenors": SECONDARY_BACK_TENORS,
        "model_vrp_log": SECONDARY_VRP_LOG_GRID,
        "model_vrp_z_3m": SECONDARY_Z_GRID,
        "model_vrp_z_1y": SECONDARY_Z_GRID,
        "RSI14_max": SECONDARY_RSI_GRID,
        "rv21d_vol_pct_min": SECONDARY_RV21D_GRID,
    },

    "selection_rules": {
        "core_gate": "If any locked Core tenor qualifies on a date, Secondary is ignored.",
        "secondary_test_dates": "Core-empty dates only.",
        "secondary_selection": "At most one Secondary Back trade per candidate/date.",
        "program_view": "At most one trade per date: Core first, else Secondary Back.",
        "continuity_view": "Cell 11R-style Core primary one-trade-per-bucket/date plus Secondary Back on Core-empty dates.",
    },

    "targets": {
        "combined_win_rate_target": COMBINED_WIN_RATE_TARGET,
        "total_program_frequency_target": TOTAL_PROGRAM_FREQUENCY_TARGET,
    },

    "candidate_grid_csv_path": str(candidate_grid_csv_path),
    "candidate_grid_parquet_path": str(candidate_grid_parquet_path),
    "core_gate_dates_path": str(core_gate_dates_path),
    "core_empty_dates_path": str(core_empty_dates_path),
    "secondary_selected_panel_path": str(secondary_selected_panel_path),
    "secondary_selected_panel_csv_path": str(secondary_selected_panel_csv_path),
    "combined_program_panel_path": str(combined_program_panel_path),
    "combined_program_panel_csv_path": str(combined_program_panel_csv_path),
    "combined_continuity_panel_path": str(combined_continuity_panel_path),
    "combined_continuity_panel_csv_path": str(combined_continuity_panel_csv_path),
    "core_program_summary_path": str(core_program_summary_path),
    "core_continuity_summary_path": str(core_continuity_summary_path),
    "secondary_incremental_summary_path": str(secondary_incremental_summary_path),
    "secondary_incremental_by_tenor_path": str(secondary_incremental_by_tenor_path),
    "secondary_incremental_by_year_path": str(secondary_incremental_by_year_path),
    "combined_program_summary_path": str(combined_program_summary_path),
    "combined_program_by_year_path": str(combined_program_by_year_path),
    "combined_continuity_summary_path": str(combined_continuity_summary_path),
    "combined_continuity_by_year_path": str(combined_continuity_by_year_path),
    "comparison_path": str(comparison_path),
    "continuity_comparison_path": str(continuity_comparison_path),
    "validation_path": str(validation_path),
    "manifest_path": str(manifest_path),

    "constraints": [
        "Core remains locked and unchanged.",
        "Secondary Back only.",
        "No Secondary Front.",
        "No Secondary Middle.",
        "9D remains excluded.",
        "18D remains in Front for Core map.",
        "Secondary Back tenors are 27D/30D/33D.",
        "z3m equals z1y in this sweep.",
        "RV21D threshold contract is rv21d_vol_pct > rv21d_vol_pct_min.",
        "forecast_vol_pct is diagnostic only.",
        "No Secondary lock.",
        "No sizing changes.",
        "No production/intraday implementation.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 12R validation failed. Review validation output above.")

# ======================================================================================
# 11. Display review
# ======================================================================================

print("=" * 100)
print("Cell 12R — Secondary Back sweep, Core-gated")
print("=" * 100)
print(f"Base panel:                         {base_panel_path}")
print(f"Locked Core rules:                  {locked_core_rules_long_path}")
print(f"Secondary candidate count:          {len(secondary_candidate_grid):,}")
print(f"Total eligible signal dates:        {total_eligible_signal_dates:,}")
print(f"Core gate dates:                    {len(core_gate_dates):,}")
print(f"Core-empty dates:                   {len(core_empty_dates):,}")
print(f"Core program selected trades:       {len(core_program):,}")
print(f"Core continuity selected trades:    {len(core_primary_continuity):,}")
print(f"Secondary selected panel rows:      {len(secondary_selected_panel):,}")
print(f"Combined program panel rows:        {len(combined_program_panel):,}")
print(f"Combined continuity panel rows:     {len(combined_continuity_panel):,}")
print("Secondary tenors:                   27D / 30D / 33D")
print("Core gate:                          If Core qualifies, Secondary ignored")
print("Program view:                       one trade per date")
print("Continuity view:                    Core primary + Secondary on Core-empty dates")
print("Secondary locked here:              False")
print("Sizing locked here:                 False")
print("Production locked here:             False")

print()
print("Validation:")
display(validation)

print()
print("Secondary candidate grid:")
display(secondary_candidate_grid)

print()
print("Core program baseline — one trade per date:")
display(core_program_summary)

print()
print("Core continuity baseline — one trade per bucket/date:")
display(core_continuity_summary)

comparison_display_cols = [
    "secondary_candidate_id",
    "secondary_model_vrp_log_min",
    "secondary_model_vrp_z_min",
    "secondary_RSI14_max",
    "secondary_rv21d_vol_pct_min",

    "trades",
    "unique_trade_dates",
    "signal_date_frequency",
    "win_rate",
    "total_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "profit_factor_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",

    "secondary_trades",
    "secondary_unique_trade_dates",
    "secondary_signal_date_frequency",
    "secondary_win_rate",
    "secondary_total_pnl",
    "secondary_avg_pnl_per_day",
    "secondary_profit_factor",
    "secondary_profit_factor_pnl_per_day",
    "secondary_pnl_per_day_drawdown",
    "secondary_worst_20_trade_pnl_per_day_sum",
    "secondary_avg_pnl_per_day_2025",
    "secondary_avg_pnl_per_day_2026",

    "delta_program_unique_trade_dates_vs_core_only",
    "delta_program_signal_date_frequency_vs_core_only",
    "delta_program_win_rate_vs_core_only",
    "delta_program_total_pnl_vs_core_only",
    "delta_program_avg_pnl_per_day_vs_core_only",
    "delta_program_profit_factor_vs_core_only",
    "delta_program_profit_factor_pnl_per_day_vs_core_only",
    "delta_program_pnl_per_day_drawdown_vs_core_only",
    "delta_program_worst_20_trade_pnl_per_day_sum_vs_core_only",
    "delta_program_avg_pnl_per_day_2026_vs_core_only",

    "passes_frequency_target",
    "passes_win_rate_target",
    "passes_program_targets",
]

print()
print("Candidate comparison — PROGRAM VIEW, Core first else Secondary Back:")
display(comparison[[c for c in comparison_display_cols if c in comparison.columns]])

continuity_display_cols = [
    "secondary_candidate_id",
    "secondary_model_vrp_log_min",
    "secondary_model_vrp_z_min",
    "secondary_RSI14_max",
    "secondary_rv21d_vol_pct_min",
    "trades",
    "unique_trade_dates",
    "signal_date_frequency",
    "win_rate",
    "total_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "profit_factor_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
    "delta_continuity_trades_vs_core_primary",
    "delta_continuity_unique_trade_dates_vs_core_primary",
    "delta_continuity_signal_date_frequency_vs_core_primary",
    "delta_continuity_win_rate_vs_core_primary",
    "delta_continuity_total_pnl_vs_core_primary",
    "delta_continuity_avg_pnl_per_day_vs_core_primary",
    "delta_continuity_profit_factor_vs_core_primary",
    "delta_continuity_profit_factor_pnl_per_day_vs_core_primary",
    "delta_continuity_pnl_per_day_drawdown_vs_core_primary",
    "delta_continuity_worst_20_trade_pnl_per_day_sum_vs_core_primary",
    "delta_continuity_avg_pnl_per_day_2026_vs_core_primary",
]

print()
print("Candidate comparison — CONTINUITY VIEW, Core primary + Secondary Back:")
display(continuity_comparison[[c for c in continuity_display_cols if c in continuity_comparison.columns]])

print()
print("Secondary incremental summary:")
display(secondary_incremental_summary)

print()
print("Secondary incremental by tenor:")
display(secondary_incremental_by_tenor)

print()
print("Secondary incremental by year:")
display(secondary_incremental_by_year)

print()
print("Combined program by year:")
display(combined_program_by_year)

print()
print("Combined continuity by year:")
display(combined_continuity_by_year)

print()
print("Saved outputs:")
for p in [
    candidate_grid_csv_path,
    candidate_grid_parquet_path,
    core_gate_dates_path,
    core_empty_dates_path,
    secondary_selected_panel_path,
    secondary_selected_panel_csv_path,
    combined_program_panel_path,
    combined_program_panel_csv_path,
    combined_continuity_panel_path,
    combined_continuity_panel_csv_path,
    core_program_summary_path,
    core_continuity_summary_path,
    secondary_incremental_summary_path,
    secondary_incremental_by_tenor_path,
    secondary_incremental_by_year_path,
    combined_program_summary_path,
    combined_program_by_year_path,
    combined_continuity_summary_path,
    combined_continuity_by_year_path,
    comparison_path,
    continuity_comparison_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 12R PASSED — SECONDARY BACK CORE-GATED SWEEP COMPLETE")

In [ ]:
# ======================================================================================
# Cell 13R — Secondary Middle sweep, Core-gated, with Back shortlist
# ======================================================================================
#
# Objective:
#   Sweep Secondary Middle parameters as an incremental layer behind locked Core repaired v1.
#   Also test Secondary Middle combined with two shortlisted Secondary Back profiles from Cell 12R.
#
# Locked Core:
#   Source:
#       Cell 11R — core_repaired_v1_locked_001
#
#   Core remains unchanged:
#       9D excluded
#       Front  = 12D / 15D / 18D
#       Middle = 21D / 24D
#       Back   = 27D / 30D / 33D
#
# Secondary Middle sweep:
#   Tenors:
#       21D / 24D only
#
#   VRP log:
#       [0.55, 0.60, 0.65]
#
#   z-score:
#       [0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60]
#
#   z constraint:
#       z3m = z1y
#
#   RSI14 cap:
#       [70, 71, 72, 73, 74, 75]
#
#   RV21D floor:
#       [6.5, 7.0]
#
# Secondary Middle candidate count:
#       3 * 7 * 6 * 2 = 252
#
# Back shortlist carried from Cell 12R:
#
#   NO_BACK:
#       Middle only
#
#   BACK_FREQ_0009:
#       log > 0.65
#       z3m/z1y > 0.00
#       RSI14 < 77
#       rv21d_vol_pct > 6.5
#
#   BACK_CLEAN_0133:
#       log > 0.70
#       z3m/z1y > 0.60
#       RSI14 < 74
#       rv21d_vol_pct > 6.5
#
# Stack count:
#       252 Middle candidates * 3 Back stack variants = 756 stack candidates
#
# Core-gating rule:
#   If any locked Core tenor qualifies on a date:
#       Secondary is ignored.
#
#   If no locked Core tenor qualifies on a date:
#       test the Secondary stack.
#
# Secondary stack selection:
#   For each stack candidate and Core-empty date:
#       select at most one Secondary trade across available Middle and Back candidates.
#
# Evaluation views:
#   1. Program view:
#       one trade per date
#       Core first, else one Secondary trade
#
#   2. Continuity view:
#       Cell 11R-style Core primary one-trade-per-bucket/date
#       plus one Secondary trade on Core-empty dates
#
# This cell does NOT:
#   1. Reopen Core.
#   2. Lock Secondary Middle.
#   3. Lock Secondary Back.
#   4. Test Secondary Front.
#   5. Change sizing.
#   6. Implement production/intraday logic.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import itertools
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 520)
pd.set_option("display.width", 820)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL13R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

LOCKED_CORE_VERSION = "core_repaired_v1"
LOCKED_CORE_DECISION_ID = "core_repaired_v1_locked_001"

SECONDARY_MIDDLE_SWEEP_VERSION = "secondary_middle_core_gated_sweep_v1"
SECONDARY_STACK_SWEEP_VERSION = "secondary_middle_with_back_shortlist_core_gated_sweep_v1"

ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]
SECONDARY_MIDDLE_TENORS = [21, 24]
SECONDARY_BACK_TENORS = [27, 30, 33]

BUCKET_ORDER_MAP = {
    "Excluded": 0,
    "Front": 1,
    "Middle": 2,
    "Back": 3,
}

BUCKET_CENTER = {
    "Front": 15,
    "Middle": 21,
    "Back": 30,
}

PRIMARY_CORE_CONTINUITY_UNIVERSE = "core_primary_one_trade_per_bucket_per_date"
CORE_PROGRAM_UNIVERSE = "core_program_one_trade_per_date"

MIDDLE_INCREMENTAL_UNIVERSE = "secondary_middle_incremental_core_empty_dates"
BACK_SHORTLIST_INCREMENTAL_UNIVERSE = "secondary_back_shortlist_incremental_core_empty_dates"
STACK_SECONDARY_INCREMENTAL_UNIVERSE = "secondary_stack_incremental_core_empty_dates"

COMBINED_PROGRAM_UNIVERSE = "combined_program_core_first_else_secondary_stack"
COMBINED_CONTINUITY_UNIVERSE = "combined_continuity_core_primary_plus_secondary_stack"

SECONDARY_MIDDLE_VRP_LOG_GRID = [0.55, 0.60, 0.65]
SECONDARY_MIDDLE_Z_GRID = [0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60]
SECONDARY_MIDDLE_RSI_GRID = [70.0, 71.0, 72.0, 73.0, 74.0, 75.0]
SECONDARY_MIDDLE_RV21D_GRID = [6.5, 7.0]

TOTAL_PROGRAM_FREQUENCY_TARGET = 0.33
COMBINED_WIN_RATE_TARGET = 0.80

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def add_rank_columns(d, group_cols, suffix):
    x = d.copy()

    if x.empty:
        return x

    x[f"rank_z_1y_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    x[f"rank_z_3m_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    x[f"rank_vrp_log_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    x[f"avg_signal_rank_{suffix}"] = x[
        [
            f"rank_z_1y_{suffix}",
            f"rank_z_3m_{suffix}",
            f"rank_vrp_log_{suffix}",
        ]
    ].mean(axis=1)

    if "effective_tenor_bucket" in x.columns:
        x["bucket_center_tenor"] = x["effective_tenor_bucket"].map(BUCKET_CENTER)
        x["bucket_center_tenor"] = x["bucket_center_tenor"].fillna(30)
    else:
        x["bucket_center_tenor"] = 30

    x["distance_to_bucket_center"] = (x["tenor"] - x["bucket_center_tenor"]).abs()

    return x


def select_core_universes(core_qualified_complete):
    q = core_qualified_complete.copy()

    if q.empty:
        return {
            PRIMARY_CORE_CONTINUITY_UNIVERSE: q.copy(),
            CORE_PROGRAM_UNIVERSE: q.copy(),
        }

    q = add_rank_columns(
        q,
        group_cols=["trade_date", "effective_tenor_bucket"],
        suffix="core_bucket_date",
    )

    q = add_rank_columns(
        q,
        group_cols=["trade_date"],
        suffix="core_date",
    )

    core_primary = (
        q.sort_values(
            [
                "trade_date",
                "effective_tenor_bucket_order",
                "avg_signal_rank_core_bucket_date",
                "distance_to_bucket_center",
                "rank_z_1y_core_bucket_date",
                "rank_z_3m_core_bucket_date",
                "rank_vrp_log_core_bucket_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date", "effective_tenor_bucket"], as_index=False)
        .head(1)
        .copy()
    )

    core_program = (
        q.sort_values(
            [
                "trade_date",
                "avg_signal_rank_core_date",
                "rank_z_1y_core_date",
                "rank_z_3m_core_date",
                "rank_vrp_log_core_date",
                "distance_to_bucket_center",
                "effective_tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return {
        PRIMARY_CORE_CONTINUITY_UNIVERSE: core_primary,
        CORE_PROGRAM_UNIVERSE: core_program,
    }


def select_one_per_candidate_date(
    qualified_complete,
    candidate_col,
    group_extra_cols=None,
    suffix="secondary_date",
):
    q = qualified_complete.copy()

    if q.empty:
        return q

    group_cols = [candidate_col, "trade_date"]

    if group_extra_cols:
        group_cols = group_extra_cols + group_cols

    q = add_rank_columns(
        q,
        group_cols=group_cols,
        suffix=suffix,
    )

    selected = (
        q.sort_values(
            [
                candidate_col,
                "trade_date",
                f"avg_signal_rank_{suffix}",
                "distance_to_bucket_center",
                f"rank_z_1y_{suffix}",
                f"rank_z_3m_{suffix}",
                f"rank_vrp_log_{suffix}",
                "effective_tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True, True],
        )
        .groupby([candidate_col, "trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return selected


def select_stack_secondary_one_per_date(stack_candidates):
    q = stack_candidates.copy()

    if q.empty:
        return q

    q = add_rank_columns(
        q,
        group_cols=["secondary_stack_candidate_id", "trade_date"],
        suffix="stack_secondary_date",
    )

    selected = (
        q.sort_values(
            [
                "secondary_stack_candidate_id",
                "trade_date",
                "avg_signal_rank_stack_secondary_date",
                "distance_to_bucket_center",
                "rank_z_1y_stack_secondary_date",
                "rank_z_3m_stack_secondary_date",
                "rank_vrp_log_stack_secondary_date",
                "effective_tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True, True],
        )
        .groupby(["secondary_stack_candidate_id", "trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return selected


def summarize_trade_set(df, group_cols, total_eligible_dates=None):
    d = df.copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    numeric_cols = [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "forecast_vol_pct",
        "rv21d_vol_pct",

        "threshold_model_vrp_log",
        "threshold_model_vrp_z_3m",
        "threshold_model_vrp_z_1y",
        "threshold_RSI14_max",
        "threshold_rv21d_vol_pct_min",

        "secondary_middle_threshold_model_vrp_log",
        "secondary_middle_threshold_model_vrp_z_3m",
        "secondary_middle_threshold_model_vrp_z_1y",
        "secondary_middle_threshold_RSI14_max",
        "secondary_middle_threshold_rv21d_vol_pct_min",

        "secondary_back_threshold_model_vrp_log",
        "secondary_back_threshold_model_vrp_z_3m",
        "secondary_back_threshold_model_vrp_z_1y",
        "secondary_back_threshold_RSI14_max",
        "secondary_back_threshold_rv21d_vol_pct_min",
    ]

    for c in numeric_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d = d.sort_values(["trade_date", "tenor"]).copy()

    grouped = d.groupby(group_cols, dropna=False, observed=False) if group_cols else [((), d)]

    rows = []

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {}

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        unique_dates = int(g["trade_date"].nunique())
        signal_date_frequency = (
            unique_dates / total_eligible_dates
            if total_eligible_dates and total_eligible_dates > 0
            else np.nan
        )

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": unique_dates,
            "signal_date_frequency": float(signal_date_frequency) if pd.notna(signal_date_frequency) else np.nan,
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        for signal_col in [
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",

            "threshold_model_vrp_log",
            "threshold_model_vrp_z_3m",
            "threshold_model_vrp_z_1y",
            "threshold_RSI14_max",
            "threshold_rv21d_vol_pct_min",

            "secondary_middle_threshold_model_vrp_log",
            "secondary_middle_threshold_model_vrp_z_3m",
            "secondary_middle_threshold_model_vrp_z_1y",
            "secondary_middle_threshold_RSI14_max",
            "secondary_middle_threshold_rv21d_vol_pct_min",

            "secondary_back_threshold_model_vrp_log",
            "secondary_back_threshold_model_vrp_z_3m",
            "secondary_back_threshold_model_vrp_z_1y",
            "secondary_back_threshold_RSI14_max",
            "secondary_back_threshold_rv21d_vol_pct_min",
        ]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        if "program_leg" in g.columns:
            row["core_trade_count"] = int(g["program_leg"].eq("Core").sum())
            row["secondary_middle_trade_count"] = int(g["program_leg"].eq("Secondary_Middle").sum())
            row["secondary_back_trade_count"] = int(g["program_leg"].eq("Secondary_Back").sum())
            row["secondary_trade_count"] = int(g["program_leg"].isin(["Secondary_Middle", "Secondary_Back"]).sum())

        rows.append(row)

    return pd.DataFrame(rows)


def attach_stack_grid(summary_df, stack_grid):
    if summary_df.empty:
        return summary_df

    return summary_df.merge(
        stack_grid,
        on="secondary_stack_candidate_id",
        how="left",
        validate="many_to_one",
    )


def attach_middle_grid(summary_df, middle_grid):
    if summary_df.empty:
        return summary_df

    return summary_df.merge(
        middle_grid,
        on="secondary_middle_candidate_id",
        how="left",
        validate="many_to_one",
    )


def attach_back_grid(summary_df, back_grid):
    if summary_df.empty:
        return summary_df

    return summary_df.merge(
        back_grid,
        on="secondary_back_profile_id",
        how="left",
        validate="many_to_one",
    )


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


# ======================================================================================
# 2. Load inputs
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01R_unified_fds_signal_base_panel_with_rv21d_*.parquet",
    required=True,
)

locked_core_rules_long_path = latest_file(
    OUT_PROCESSED_DIR,
    "11R_core_repaired_v1_locked_rules_long_*.parquet",
    required=True,
)

locked_core_rules_wide_path = latest_file(
    OUT_PROCESSED_DIR,
    "11R_core_repaired_v1_locked_rules_wide_*.parquet",
    required=True,
)

base = pd.read_parquet(base_panel_path)
locked_core_rules_long = pd.read_parquet(locked_core_rules_long_path)
locked_core_rules_wide = pd.read_parquet(locked_core_rules_wide_path)

base["trade_date"] = pd.to_datetime(base["trade_date"], errors="coerce").dt.normalize()

for c in [
    "tenor",
    "tenor_bucket_order",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")

for c in [
    "tenor",
    "original_tenor_bucket_order",
    "effective_tenor_bucket_order",
    "model_vrp_log_min",
    "model_vrp_z_min",
    "model_vrp_z_3m_min",
    "model_vrp_z_1y_min",
    "RSI14_max",
    "rv21d_vol_pct_min",
]:
    if c in locked_core_rules_long.columns:
        locked_core_rules_long[c] = pd.to_numeric(locked_core_rules_long[c], errors="coerce")

total_eligible_signal_dates = int(base["trade_date"].nunique())

# ======================================================================================
# 3. Apply locked Core rules and build Core baselines
# ======================================================================================

core_thresholds = locked_core_rules_long.rename(columns={
    "model_vrp_log_min": "threshold_model_vrp_log",
    "model_vrp_z_3m_min": "threshold_model_vrp_z_3m",
    "model_vrp_z_1y_min": "threshold_model_vrp_z_1y",
    "RSI14_max": "threshold_RSI14_max",
    "rv21d_vol_pct_min": "threshold_rv21d_vol_pct_min",
})

core_threshold_cols = [
    "tenor",
    "original_tenor_bucket",
    "original_tenor_bucket_order",
    "effective_tenor_bucket",
    "effective_tenor_bucket_order",
    "include_tenor",
    "threshold_model_vrp_log",
    "threshold_model_vrp_z_3m",
    "threshold_model_vrp_z_1y",
    "threshold_RSI14_max",
    "threshold_rv21d_vol_pct_min",
]

base_for_join = base.copy()

base_for_join = base_for_join.rename(columns={
    "tenor_bucket": "source_tenor_bucket",
    "tenor_bucket_order": "source_tenor_bucket_order",
})

core_panel = base_for_join.merge(
    core_thresholds[core_threshold_cols],
    on="tenor",
    how="inner",
    validate="many_to_one",
)

core_signal_available = (
    core_panel["include_tenor"].astype(bool)
    & core_panel["model_vrp_log"].notna()
    & core_panel["model_vrp_z_3m"].notna()
    & core_panel["model_vrp_z_1y"].notna()
    & core_panel["RSI14"].notna()
    & core_panel["rv21d_vol_pct"].notna()
)

core_pass_mask = (
    core_signal_available
    & (core_panel["model_vrp_log"] > core_panel["threshold_model_vrp_log"])
    & (core_panel["model_vrp_z_3m"] > core_panel["threshold_model_vrp_z_3m"])
    & (core_panel["model_vrp_z_1y"] > core_panel["threshold_model_vrp_z_1y"])
    & (core_panel["RSI14"] < core_panel["threshold_RSI14_max"])
    & (core_panel["rv21d_vol_pct"] > core_panel["threshold_rv21d_vol_pct_min"])
)

core_panel["core_repaired_v1_pass"] = core_pass_mask

core_qualified_all_dates = core_panel[core_pass_mask].copy()

core_qualified_complete = core_panel[
    core_pass_mask
    & core_panel["normalized_pnl_dollars"].notna()
    & core_panel["normalized_pnl_per_day"].notna()
].copy()

core_gate_dates = sorted(pd.to_datetime(core_qualified_all_dates["trade_date"].dropna().unique()).tolist())
all_signal_dates = sorted(pd.to_datetime(base["trade_date"].dropna().unique()).tolist())

core_gate_dates_set = set(core_gate_dates)
core_empty_dates = sorted(set(all_signal_dates) - core_gate_dates_set)

core_universes = select_core_universes(core_qualified_complete)

core_primary_continuity = core_universes[PRIMARY_CORE_CONTINUITY_UNIVERSE].copy()
core_program = core_universes[CORE_PROGRAM_UNIVERSE].copy()

core_primary_continuity["source_layer"] = "Core"
core_primary_continuity["program_leg"] = "Core"
core_primary_continuity["selection_universe"] = PRIMARY_CORE_CONTINUITY_UNIVERSE
core_primary_continuity["secondary_stack_candidate_id"] = "CORE_ONLY_BASELINE"
core_primary_continuity["secondary_middle_candidate_id"] = "NONE"
core_primary_continuity["secondary_back_profile_id"] = "NONE"

core_program["source_layer"] = "Core"
core_program["program_leg"] = "Core"
core_program["selection_universe"] = CORE_PROGRAM_UNIVERSE
core_program["secondary_stack_candidate_id"] = "CORE_ONLY_BASELINE"
core_program["secondary_middle_candidate_id"] = "NONE"
core_program["secondary_back_profile_id"] = "NONE"

# ======================================================================================
# 4. Build Secondary Middle grid
# ======================================================================================

middle_candidate_rows = []

idx = 1

for vrp_log, z, rsi_cap, rv_floor in itertools.product(
    SECONDARY_MIDDLE_VRP_LOG_GRID,
    SECONDARY_MIDDLE_Z_GRID,
    SECONDARY_MIDDLE_RSI_GRID,
    SECONDARY_MIDDLE_RV21D_GRID,
):
    cid = f"secondary_middle_{idx:04d}"

    middle_candidate_rows.append({
        "secondary_middle_candidate_id": cid,
        "secondary_middle_sweep_version": SECONDARY_MIDDLE_SWEEP_VERSION,
        "secondary_middle_candidate_family": "secondary_middle_core_gated",
        "secondary_middle_candidate_subfamily": "secondary_middle_only",
        "secondary_middle_candidate_description": (
            f"Secondary Middle only, Core-gated; log>{vrp_log:.2f}, "
            f"z3m/z1y>{z:.2f}, RSI14<{rsi_cap:.0f}, rv21d>{rv_floor:.1f}"
        ),

        "secondary_middle_tenors": "21,24",
        "secondary_middle_bucket": "Middle",
        "secondary_middle_model_vrp_log_min": float(vrp_log),
        "secondary_middle_model_vrp_z_min": float(z),
        "secondary_middle_model_vrp_z_3m_min": float(z),
        "secondary_middle_model_vrp_z_1y_min": float(z),
        "secondary_middle_RSI14_max": float(rsi_cap),
        "secondary_middle_rv21d_vol_pct_min": float(rv_floor),

        "middle_z3m_equals_z1y": True,
        "middle_uses_rv21d_threshold_contract": True,
        "middle_forecast_vol_pct_threshold_used": False,
        "core_gated": True,
        "secondary_front_tested": False,
        "secondary_middle_tested": True,
        "secondary_back_tested_as_sweep_dimension": False,
        "secondary_middle_locked_here": False,
        "secondary_back_locked_here": False,
        "sizing_locked_here": False,
        "production_locked_here": False,
    })

    idx += 1

secondary_middle_grid = pd.DataFrame(middle_candidate_rows)

expected_middle_candidate_count = (
    len(SECONDARY_MIDDLE_VRP_LOG_GRID)
    * len(SECONDARY_MIDDLE_Z_GRID)
    * len(SECONDARY_MIDDLE_RSI_GRID)
    * len(SECONDARY_MIDDLE_RV21D_GRID)
)

# ======================================================================================
# 5. Build Secondary Back shortlist grid
# ======================================================================================

secondary_back_shortlist = pd.DataFrame([
    {
        "secondary_back_profile_id": "NO_BACK",
        "secondary_back_source_candidate_id": "NONE",
        "secondary_back_profile_description": "No Secondary Back carried; Middle-only stack.",
        "secondary_back_tenors": "",
        "secondary_back_bucket": "",
        "secondary_back_model_vrp_log_min": np.nan,
        "secondary_back_model_vrp_z_min": np.nan,
        "secondary_back_model_vrp_z_3m_min": np.nan,
        "secondary_back_model_vrp_z_1y_min": np.nan,
        "secondary_back_RSI14_max": np.nan,
        "secondary_back_rv21d_vol_pct_min": np.nan,
        "back_z3m_equals_z1y": True,
        "back_uses_rv21d_threshold_contract": True,
        "back_forecast_vol_pct_threshold_used": False,
        "secondary_back_tested": False,
        "secondary_back_locked_here": False,
    },
    {
        "secondary_back_profile_id": "BACK_FREQ_0009",
        "secondary_back_source_candidate_id": "secondary_back_0009",
        "secondary_back_profile_description": "Frequency Back candidate from Cell 12R: log>0.65, z>0.00, RSI14<77, rv21d>6.5.",
        "secondary_back_tenors": "27,30,33",
        "secondary_back_bucket": "Back",
        "secondary_back_model_vrp_log_min": 0.65,
        "secondary_back_model_vrp_z_min": 0.00,
        "secondary_back_model_vrp_z_3m_min": 0.00,
        "secondary_back_model_vrp_z_1y_min": 0.00,
        "secondary_back_RSI14_max": 77.0,
        "secondary_back_rv21d_vol_pct_min": 6.5,
        "back_z3m_equals_z1y": True,
        "back_uses_rv21d_threshold_contract": True,
        "back_forecast_vol_pct_threshold_used": False,
        "secondary_back_tested": True,
        "secondary_back_locked_here": False,
    },
    {
        "secondary_back_profile_id": "BACK_CLEAN_0133",
        "secondary_back_source_candidate_id": "secondary_back_0133",
        "secondary_back_profile_description": "Clean Back reference from Cell 12R: log>0.70, z>0.60, RSI14<74, rv21d>6.5.",
        "secondary_back_tenors": "27,30,33",
        "secondary_back_bucket": "Back",
        "secondary_back_model_vrp_log_min": 0.70,
        "secondary_back_model_vrp_z_min": 0.60,
        "secondary_back_model_vrp_z_3m_min": 0.60,
        "secondary_back_model_vrp_z_1y_min": 0.60,
        "secondary_back_RSI14_max": 74.0,
        "secondary_back_rv21d_vol_pct_min": 6.5,
        "back_z3m_equals_z1y": True,
        "back_uses_rv21d_threshold_contract": True,
        "back_forecast_vol_pct_threshold_used": False,
        "secondary_back_tested": True,
        "secondary_back_locked_here": False,
    },
])

# ======================================================================================
# 6. Build stack grid: Middle candidates × Back shortlist
# ======================================================================================

secondary_stack_grid = secondary_middle_grid.merge(
    secondary_back_shortlist,
    how="cross",
)

secondary_stack_grid = secondary_stack_grid.reset_index(drop=True)
secondary_stack_grid["secondary_stack_candidate_id"] = [
    f"secondary_stack_{i + 1:04d}" for i in range(len(secondary_stack_grid))
]

secondary_stack_grid["secondary_stack_sweep_version"] = SECONDARY_STACK_SWEEP_VERSION
secondary_stack_grid["secondary_stack_family"] = "secondary_middle_with_back_shortlist_core_gated"
secondary_stack_grid["secondary_stack_description"] = (
    secondary_stack_grid["secondary_middle_candidate_id"]
    + " + "
    + secondary_stack_grid["secondary_back_profile_id"]
)

expected_stack_candidate_count = len(secondary_middle_grid) * len(secondary_back_shortlist)

# ======================================================================================
# 7. Apply Secondary Middle candidates on Core-empty dates
# ======================================================================================

middle_base = base_for_join[
    base_for_join["tenor"].isin(SECONDARY_MIDDLE_TENORS)
    & base_for_join["trade_date"].isin(core_empty_dates)
].copy()

middle_base["effective_tenor_bucket"] = "Middle"
middle_base["effective_tenor_bucket_order"] = BUCKET_ORDER_MAP["Middle"]

middle_panel = middle_base.merge(
    secondary_middle_grid,
    how="cross",
)

middle_panel = middle_panel.rename(columns={
    "secondary_middle_model_vrp_log_min": "secondary_middle_threshold_model_vrp_log",
    "secondary_middle_model_vrp_z_3m_min": "secondary_middle_threshold_model_vrp_z_3m",
    "secondary_middle_model_vrp_z_1y_min": "secondary_middle_threshold_model_vrp_z_1y",
    "secondary_middle_RSI14_max": "secondary_middle_threshold_RSI14_max",
    "secondary_middle_rv21d_vol_pct_min": "secondary_middle_threshold_rv21d_vol_pct_min",
})

middle_signal_available = (
    middle_panel["model_vrp_log"].notna()
    & middle_panel["model_vrp_z_3m"].notna()
    & middle_panel["model_vrp_z_1y"].notna()
    & middle_panel["RSI14"].notna()
    & middle_panel["rv21d_vol_pct"].notna()
)

middle_pass_mask = (
    middle_signal_available
    & (middle_panel["model_vrp_log"] > middle_panel["secondary_middle_threshold_model_vrp_log"])
    & (middle_panel["model_vrp_z_3m"] > middle_panel["secondary_middle_threshold_model_vrp_z_3m"])
    & (middle_panel["model_vrp_z_1y"] > middle_panel["secondary_middle_threshold_model_vrp_z_1y"])
    & (middle_panel["RSI14"] < middle_panel["secondary_middle_threshold_RSI14_max"])
    & (middle_panel["rv21d_vol_pct"] > middle_panel["secondary_middle_threshold_rv21d_vol_pct_min"])
)

middle_panel["secondary_middle_pass"] = middle_pass_mask

middle_qualified_complete = middle_panel[
    middle_pass_mask
    & middle_panel["normalized_pnl_dollars"].notna()
    & middle_panel["normalized_pnl_per_day"].notna()
].copy()

middle_selected = select_one_per_candidate_date(
    middle_qualified_complete,
    candidate_col="secondary_middle_candidate_id",
    suffix="middle_secondary_date",
)

middle_selected["source_layer"] = "Secondary"
middle_selected["program_leg"] = "Secondary_Middle"
middle_selected["selection_universe"] = MIDDLE_INCREMENTAL_UNIVERSE
middle_selected["secondary_back_profile_id"] = "NONE"
middle_selected["secondary_back_source_candidate_id"] = "NONE"

# ======================================================================================
# 8. Apply Back shortlist profiles on Core-empty dates
# ======================================================================================

active_back_profiles = secondary_back_shortlist[
    secondary_back_shortlist["secondary_back_profile_id"].ne("NO_BACK")
].copy()

back_base = base_for_join[
    base_for_join["tenor"].isin(SECONDARY_BACK_TENORS)
    & base_for_join["trade_date"].isin(core_empty_dates)
].copy()

back_base["effective_tenor_bucket"] = "Back"
back_base["effective_tenor_bucket_order"] = BUCKET_ORDER_MAP["Back"]

back_panel = back_base.merge(
    active_back_profiles,
    how="cross",
)

back_panel = back_panel.rename(columns={
    "secondary_back_model_vrp_log_min": "secondary_back_threshold_model_vrp_log",
    "secondary_back_model_vrp_z_3m_min": "secondary_back_threshold_model_vrp_z_3m",
    "secondary_back_model_vrp_z_1y_min": "secondary_back_threshold_model_vrp_z_1y",
    "secondary_back_RSI14_max": "secondary_back_threshold_RSI14_max",
    "secondary_back_rv21d_vol_pct_min": "secondary_back_threshold_rv21d_vol_pct_min",
})

back_signal_available = (
    back_panel["model_vrp_log"].notna()
    & back_panel["model_vrp_z_3m"].notna()
    & back_panel["model_vrp_z_1y"].notna()
    & back_panel["RSI14"].notna()
    & back_panel["rv21d_vol_pct"].notna()
)

back_pass_mask = (
    back_signal_available
    & (back_panel["model_vrp_log"] > back_panel["secondary_back_threshold_model_vrp_log"])
    & (back_panel["model_vrp_z_3m"] > back_panel["secondary_back_threshold_model_vrp_z_3m"])
    & (back_panel["model_vrp_z_1y"] > back_panel["secondary_back_threshold_model_vrp_z_1y"])
    & (back_panel["RSI14"] < back_panel["secondary_back_threshold_RSI14_max"])
    & (back_panel["rv21d_vol_pct"] > back_panel["secondary_back_threshold_rv21d_vol_pct_min"])
)

back_panel["secondary_back_pass"] = back_pass_mask

back_qualified_complete = back_panel[
    back_pass_mask
    & back_panel["normalized_pnl_dollars"].notna()
    & back_panel["normalized_pnl_per_day"].notna()
].copy()

back_selected = select_one_per_candidate_date(
    back_qualified_complete,
    candidate_col="secondary_back_profile_id",
    suffix="back_secondary_date",
)

back_selected["source_layer"] = "Secondary"
back_selected["program_leg"] = "Secondary_Back"
back_selected["selection_universe"] = BACK_SHORTLIST_INCREMENTAL_UNIVERSE

# ======================================================================================
# 9. Build stack-level Secondary selected panel
# ======================================================================================

stack_secondary_frames = []

for _, stack in secondary_stack_grid.iterrows():
    stack_id = stack["secondary_stack_candidate_id"]
    mid_id = stack["secondary_middle_candidate_id"]
    back_profile = stack["secondary_back_profile_id"]

    mid = middle_selected[
        middle_selected["secondary_middle_candidate_id"].eq(mid_id)
    ].copy()

    mid["secondary_stack_candidate_id"] = stack_id
    mid["secondary_back_profile_id"] = back_profile
    mid["secondary_back_source_candidate_id"] = stack["secondary_back_source_candidate_id"]

    # Carry stack-level back thresholds on Middle rows as metadata columns.
    mid["secondary_back_stack_profile_id"] = back_profile
    mid["secondary_back_stack_source_candidate_id"] = stack["secondary_back_source_candidate_id"]

    frames = [mid]

    if back_profile != "NO_BACK":
        back = back_selected[
            back_selected["secondary_back_profile_id"].eq(back_profile)
        ].copy()

        back["secondary_stack_candidate_id"] = stack_id
        back["secondary_middle_candidate_id"] = mid_id
        back["secondary_back_stack_profile_id"] = back_profile
        back["secondary_back_stack_source_candidate_id"] = stack["secondary_back_source_candidate_id"]

        frames.append(back)

    stack_candidates = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    if not stack_candidates.empty:
        stack_secondary_frames.append(stack_candidates)

stack_secondary_candidates = (
    pd.concat(stack_secondary_frames, ignore_index=True)
    if stack_secondary_frames
    else pd.DataFrame()
)

stack_secondary_selected = select_stack_secondary_one_per_date(stack_secondary_candidates)

stack_secondary_selected["source_layer"] = "Secondary"
stack_secondary_selected["selection_universe"] = STACK_SECONDARY_INCREMENTAL_UNIVERSE

# ======================================================================================
# 10. Build combined program and continuity panels
# ======================================================================================

program_frames = []
continuity_frames = []

core_program_for_stacks_base = core_program.copy()
core_continuity_for_stacks_base = core_primary_continuity.copy()

for _, stack in secondary_stack_grid.iterrows():
    stack_id = stack["secondary_stack_candidate_id"]
    mid_id = stack["secondary_middle_candidate_id"]
    back_profile = stack["secondary_back_profile_id"]

    sec = stack_secondary_selected[
        stack_secondary_selected["secondary_stack_candidate_id"].eq(stack_id)
    ].copy()

    core_prog = core_program_for_stacks_base.copy()
    core_prog["secondary_stack_candidate_id"] = stack_id
    core_prog["secondary_middle_candidate_id"] = mid_id
    core_prog["secondary_back_profile_id"] = back_profile
    core_prog["secondary_back_source_candidate_id"] = stack["secondary_back_source_candidate_id"]

    core_cont = core_continuity_for_stacks_base.copy()
    core_cont["secondary_stack_candidate_id"] = stack_id
    core_cont["secondary_middle_candidate_id"] = mid_id
    core_cont["secondary_back_profile_id"] = back_profile
    core_cont["secondary_back_source_candidate_id"] = stack["secondary_back_source_candidate_id"]

    combined_program = pd.concat([core_prog, sec], ignore_index=True)
    combined_program["selection_universe"] = COMBINED_PROGRAM_UNIVERSE

    combined_continuity = pd.concat([core_cont, sec], ignore_index=True)
    combined_continuity["selection_universe"] = COMBINED_CONTINUITY_UNIVERSE

    program_frames.append(combined_program)
    continuity_frames.append(combined_continuity)

combined_program_panel = pd.concat(program_frames, ignore_index=True) if program_frames else pd.DataFrame()
combined_continuity_panel = pd.concat(continuity_frames, ignore_index=True) if continuity_frames else pd.DataFrame()

# ======================================================================================
# 11. Summaries
# ======================================================================================

core_program_summary = summarize_trade_set(
    core_program.assign(selection_universe=CORE_PROGRAM_UNIVERSE),
    group_cols=["secondary_stack_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

core_continuity_summary = summarize_trade_set(
    core_primary_continuity.assign(selection_universe=PRIMARY_CORE_CONTINUITY_UNIVERSE),
    group_cols=["secondary_stack_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

middle_incremental_summary = summarize_trade_set(
    middle_selected,
    group_cols=["secondary_middle_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)
middle_incremental_summary = attach_middle_grid(middle_incremental_summary, secondary_middle_grid)

middle_incremental_by_tenor = summarize_trade_set(
    middle_selected,
    group_cols=["secondary_middle_candidate_id", "selection_universe", "tenor"],
    total_eligible_dates=total_eligible_signal_dates,
)
middle_incremental_by_tenor = attach_middle_grid(middle_incremental_by_tenor, secondary_middle_grid)

middle_incremental_by_year_input = (
    middle_selected.assign(year=middle_selected["trade_date"].dt.year.astype(int))
    if not middle_selected.empty
    else middle_selected.copy()
)
middle_incremental_by_year = summarize_trade_set(
    middle_incremental_by_year_input,
    group_cols=["secondary_middle_candidate_id", "selection_universe", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)
middle_incremental_by_year = attach_middle_grid(middle_incremental_by_year, secondary_middle_grid)

back_shortlist_summary = summarize_trade_set(
    back_selected,
    group_cols=["secondary_back_profile_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)
back_shortlist_summary = attach_back_grid(back_shortlist_summary, active_back_profiles)

stack_secondary_summary = summarize_trade_set(
    stack_secondary_selected,
    group_cols=["secondary_stack_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)
stack_secondary_summary = attach_stack_grid(stack_secondary_summary, secondary_stack_grid)

stack_secondary_by_bucket = summarize_trade_set(
    stack_secondary_selected,
    group_cols=["secondary_stack_candidate_id", "selection_universe", "effective_tenor_bucket", "effective_tenor_bucket_order"],
    total_eligible_dates=total_eligible_signal_dates,
)
stack_secondary_by_bucket = attach_stack_grid(stack_secondary_by_bucket, secondary_stack_grid)

stack_secondary_by_tenor = summarize_trade_set(
    stack_secondary_selected,
    group_cols=["secondary_stack_candidate_id", "selection_universe", "tenor", "effective_tenor_bucket"],
    total_eligible_dates=total_eligible_signal_dates,
)
stack_secondary_by_tenor = attach_stack_grid(stack_secondary_by_tenor, secondary_stack_grid)

stack_secondary_by_year_input = (
    stack_secondary_selected.assign(year=stack_secondary_selected["trade_date"].dt.year.astype(int))
    if not stack_secondary_selected.empty
    else stack_secondary_selected.copy()
)
stack_secondary_by_year = summarize_trade_set(
    stack_secondary_by_year_input,
    group_cols=["secondary_stack_candidate_id", "selection_universe", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)
stack_secondary_by_year = attach_stack_grid(stack_secondary_by_year, secondary_stack_grid)

combined_program_summary = summarize_trade_set(
    combined_program_panel,
    group_cols=["secondary_stack_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)
combined_program_summary = attach_stack_grid(combined_program_summary, secondary_stack_grid)

combined_continuity_summary = summarize_trade_set(
    combined_continuity_panel,
    group_cols=["secondary_stack_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)
combined_continuity_summary = attach_stack_grid(combined_continuity_summary, secondary_stack_grid)

combined_program_by_year_input = (
    combined_program_panel.assign(year=combined_program_panel["trade_date"].dt.year.astype(int))
    if not combined_program_panel.empty
    else combined_program_panel.copy()
)
combined_program_by_year = summarize_trade_set(
    combined_program_by_year_input,
    group_cols=["secondary_stack_candidate_id", "selection_universe", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)
combined_program_by_year = attach_stack_grid(combined_program_by_year, secondary_stack_grid)

combined_continuity_by_year_input = (
    combined_continuity_panel.assign(year=combined_continuity_panel["trade_date"].dt.year.astype(int))
    if not combined_continuity_panel.empty
    else combined_continuity_panel.copy()
)
combined_continuity_by_year = summarize_trade_set(
    combined_continuity_by_year_input,
    group_cols=["secondary_stack_candidate_id", "selection_universe", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)
combined_continuity_by_year = attach_stack_grid(combined_continuity_by_year, secondary_stack_grid)

# ======================================================================================
# 12. Candidate comparison tables
# ======================================================================================

core_program_ref = core_program_summary.iloc[0].to_dict() if len(core_program_summary) else {}
core_continuity_ref = core_continuity_summary.iloc[0].to_dict() if len(core_continuity_summary) else {}

comparison = combined_program_summary.copy()

if not comparison.empty:
    sec_cols = [
        "secondary_stack_candidate_id",
        "trades",
        "unique_trade_dates",
        "signal_date_frequency",
        "win_rate",
        "total_pnl",
        "avg_pnl_per_day",
        "profit_factor",
        "profit_factor_pnl_per_day",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",
        "avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026",
        "secondary_middle_trade_count",
        "secondary_back_trade_count",
        "secondary_trade_count",
    ]

    sec_metrics = stack_secondary_summary[
        [c for c in sec_cols if c in stack_secondary_summary.columns]
    ].copy()

    sec_metrics = sec_metrics.rename(columns={
        "trades": "secondary_trades",
        "unique_trade_dates": "secondary_unique_trade_dates",
        "signal_date_frequency": "secondary_signal_date_frequency",
        "win_rate": "secondary_win_rate",
        "total_pnl": "secondary_total_pnl",
        "avg_pnl_per_day": "secondary_avg_pnl_per_day",
        "profit_factor": "secondary_profit_factor",
        "profit_factor_pnl_per_day": "secondary_profit_factor_pnl_per_day",
        "pnl_per_day_drawdown": "secondary_pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum": "secondary_worst_20_trade_pnl_per_day_sum",
        "avg_pnl_per_day_2025": "secondary_avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026": "secondary_avg_pnl_per_day_2026",
        "secondary_middle_trade_count": "incremental_middle_trade_count",
        "secondary_back_trade_count": "incremental_back_trade_count",
        "secondary_trade_count": "incremental_secondary_trade_count",
    })

    comparison = comparison.merge(
        sec_metrics,
        on="secondary_stack_candidate_id",
        how="left",
        validate="one_to_one",
    )

    for metric in [
        "trades",
        "unique_trade_dates",
        "signal_date_frequency",
        "win_rate",
        "total_pnl",
        "avg_pnl_per_day",
        "profit_factor",
        "profit_factor_pnl_per_day",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",
        "avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026",
    ]:
        if metric in comparison.columns and metric in core_program_ref:
            comparison[f"delta_program_{metric}_vs_core_only"] = comparison[metric] - core_program_ref[metric]

    comparison["passes_frequency_target"] = comparison["signal_date_frequency"] >= TOTAL_PROGRAM_FREQUENCY_TARGET
    comparison["passes_win_rate_target"] = comparison["win_rate"] >= COMBINED_WIN_RATE_TARGET
    comparison["passes_program_targets"] = (
        comparison["passes_frequency_target"]
        & comparison["passes_win_rate_target"]
    )

    comparison = comparison.sort_values(
        [
            "passes_program_targets",
            "signal_date_frequency",
            "win_rate",
            "avg_pnl_per_day",
            "profit_factor_pnl_per_day",
            "avg_pnl_per_day_2026",
        ],
        ascending=[False, False, False, False, False, False],
    ).reset_index(drop=True)

continuity_comparison = combined_continuity_summary.copy()

if not continuity_comparison.empty:
    for metric in [
        "trades",
        "unique_trade_dates",
        "signal_date_frequency",
        "win_rate",
        "total_pnl",
        "avg_pnl_per_day",
        "profit_factor",
        "profit_factor_pnl_per_day",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",
        "avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026",
    ]:
        if metric in continuity_comparison.columns and metric in core_continuity_ref:
            continuity_comparison[f"delta_continuity_{metric}_vs_core_primary"] = (
                continuity_comparison[metric] - core_continuity_ref[metric]
            )

    continuity_comparison = continuity_comparison.sort_values(
        [
            "signal_date_frequency",
            "win_rate",
            "avg_pnl_per_day",
            "profit_factor_pnl_per_day",
            "avg_pnl_per_day_2026",
        ],
        ascending=[False, False, False, False, False],
    ).reset_index(drop=True)

# ======================================================================================
# 13. Save outputs
# ======================================================================================

middle_grid_csv_path = OUT_AUDIT_DIR / f"13R_secondary_middle_grid_{CELL13R_TIMESTAMP}.csv"
middle_grid_parquet_path = OUT_PROCESSED_DIR / f"13R_secondary_middle_grid_{CELL13R_TIMESTAMP}.parquet"

back_shortlist_csv_path = OUT_AUDIT_DIR / f"13R_secondary_back_shortlist_{CELL13R_TIMESTAMP}.csv"
back_shortlist_parquet_path = OUT_PROCESSED_DIR / f"13R_secondary_back_shortlist_{CELL13R_TIMESTAMP}.parquet"

stack_grid_csv_path = OUT_AUDIT_DIR / f"13R_secondary_middle_back_stack_grid_{CELL13R_TIMESTAMP}.csv"
stack_grid_parquet_path = OUT_PROCESSED_DIR / f"13R_secondary_middle_back_stack_grid_{CELL13R_TIMESTAMP}.parquet"

core_gate_dates_path = OUT_AUDIT_DIR / f"13R_secondary_middle_core_gated_core_gate_dates_{CELL13R_TIMESTAMP}.csv"
core_empty_dates_path = OUT_AUDIT_DIR / f"13R_secondary_middle_core_gated_core_empty_dates_{CELL13R_TIMESTAMP}.csv"

middle_selected_panel_path = OUT_PROCESSED_DIR / f"13R_secondary_middle_selected_panel_{CELL13R_TIMESTAMP}.parquet"
middle_selected_panel_csv_path = OUT_AUDIT_DIR / f"13R_secondary_middle_selected_panel_{CELL13R_TIMESTAMP}.csv"

back_selected_panel_path = OUT_PROCESSED_DIR / f"13R_secondary_back_shortlist_selected_panel_{CELL13R_TIMESTAMP}.parquet"
back_selected_panel_csv_path = OUT_AUDIT_DIR / f"13R_secondary_back_shortlist_selected_panel_{CELL13R_TIMESTAMP}.csv"

stack_secondary_selected_panel_path = OUT_PROCESSED_DIR / f"13R_secondary_stack_selected_panel_{CELL13R_TIMESTAMP}.parquet"
stack_secondary_selected_panel_csv_path = OUT_AUDIT_DIR / f"13R_secondary_stack_selected_panel_{CELL13R_TIMESTAMP}.csv"

combined_program_panel_path = OUT_PROCESSED_DIR / f"13R_combined_program_core_first_else_secondary_stack_panel_{CELL13R_TIMESTAMP}.parquet"
combined_program_panel_csv_path = OUT_AUDIT_DIR / f"13R_combined_program_core_first_else_secondary_stack_panel_{CELL13R_TIMESTAMP}.csv"

combined_continuity_panel_path = OUT_PROCESSED_DIR / f"13R_combined_continuity_core_primary_plus_secondary_stack_panel_{CELL13R_TIMESTAMP}.parquet"
combined_continuity_panel_csv_path = OUT_AUDIT_DIR / f"13R_combined_continuity_core_primary_plus_secondary_stack_panel_{CELL13R_TIMESTAMP}.csv"

core_program_summary_path = OUT_AUDIT_DIR / f"13R_core_program_baseline_{CELL13R_TIMESTAMP}.csv"
core_continuity_summary_path = OUT_AUDIT_DIR / f"13R_core_continuity_baseline_{CELL13R_TIMESTAMP}.csv"

middle_incremental_summary_path = OUT_AUDIT_DIR / f"13R_secondary_middle_incremental_summary_{CELL13R_TIMESTAMP}.csv"
middle_incremental_by_tenor_path = OUT_AUDIT_DIR / f"13R_secondary_middle_incremental_by_tenor_{CELL13R_TIMESTAMP}.csv"
middle_incremental_by_year_path = OUT_AUDIT_DIR / f"13R_secondary_middle_incremental_by_year_{CELL13R_TIMESTAMP}.csv"

back_shortlist_summary_path = OUT_AUDIT_DIR / f"13R_secondary_back_shortlist_summary_{CELL13R_TIMESTAMP}.csv"

stack_secondary_summary_path = OUT_AUDIT_DIR / f"13R_secondary_stack_incremental_summary_{CELL13R_TIMESTAMP}.csv"
stack_secondary_by_bucket_path = OUT_AUDIT_DIR / f"13R_secondary_stack_incremental_by_bucket_{CELL13R_TIMESTAMP}.csv"
stack_secondary_by_tenor_path = OUT_AUDIT_DIR / f"13R_secondary_stack_incremental_by_tenor_{CELL13R_TIMESTAMP}.csv"
stack_secondary_by_year_path = OUT_AUDIT_DIR / f"13R_secondary_stack_incremental_by_year_{CELL13R_TIMESTAMP}.csv"

combined_program_summary_path = OUT_AUDIT_DIR / f"13R_combined_program_summary_{CELL13R_TIMESTAMP}.csv"
combined_program_by_year_path = OUT_AUDIT_DIR / f"13R_combined_program_by_year_{CELL13R_TIMESTAMP}.csv"

combined_continuity_summary_path = OUT_AUDIT_DIR / f"13R_combined_continuity_summary_{CELL13R_TIMESTAMP}.csv"
combined_continuity_by_year_path = OUT_AUDIT_DIR / f"13R_combined_continuity_by_year_{CELL13R_TIMESTAMP}.csv"

comparison_path = OUT_AUDIT_DIR / f"13R_candidate_comparison_program_view_{CELL13R_TIMESTAMP}.csv"
continuity_comparison_path = OUT_AUDIT_DIR / f"13R_candidate_comparison_continuity_view_{CELL13R_TIMESTAMP}.csv"

validation_path = OUT_AUDIT_DIR / f"13R_secondary_middle_with_back_shortlist_validation_{CELL13R_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"13R_secondary_middle_with_back_shortlist_manifest_{CELL13R_TIMESTAMP}.json"

secondary_middle_grid.to_csv(middle_grid_csv_path, index=False)
secondary_middle_grid.to_parquet(middle_grid_parquet_path, index=False)

secondary_back_shortlist.to_csv(back_shortlist_csv_path, index=False)
secondary_back_shortlist.to_parquet(back_shortlist_parquet_path, index=False)

secondary_stack_grid.to_csv(stack_grid_csv_path, index=False)
secondary_stack_grid.to_parquet(stack_grid_parquet_path, index=False)

pd.DataFrame({"trade_date": core_gate_dates}).to_csv(core_gate_dates_path, index=False)
pd.DataFrame({"trade_date": core_empty_dates}).to_csv(core_empty_dates_path, index=False)

middle_selected.to_parquet(middle_selected_panel_path, index=False)
middle_selected.to_csv(middle_selected_panel_csv_path, index=False)

back_selected.to_parquet(back_selected_panel_path, index=False)
back_selected.to_csv(back_selected_panel_csv_path, index=False)

stack_secondary_selected.to_parquet(stack_secondary_selected_panel_path, index=False)
stack_secondary_selected.to_csv(stack_secondary_selected_panel_csv_path, index=False)

combined_program_panel.to_parquet(combined_program_panel_path, index=False)
combined_program_panel.to_csv(combined_program_panel_csv_path, index=False)

combined_continuity_panel.to_parquet(combined_continuity_panel_path, index=False)
combined_continuity_panel.to_csv(combined_continuity_panel_csv_path, index=False)

core_program_summary.to_csv(core_program_summary_path, index=False)
core_continuity_summary.to_csv(core_continuity_summary_path, index=False)

middle_incremental_summary.to_csv(middle_incremental_summary_path, index=False)
middle_incremental_by_tenor.to_csv(middle_incremental_by_tenor_path, index=False)
middle_incremental_by_year.to_csv(middle_incremental_by_year_path, index=False)

back_shortlist_summary.to_csv(back_shortlist_summary_path, index=False)

stack_secondary_summary.to_csv(stack_secondary_summary_path, index=False)
stack_secondary_by_bucket.to_csv(stack_secondary_by_bucket_path, index=False)
stack_secondary_by_tenor.to_csv(stack_secondary_by_tenor_path, index=False)
stack_secondary_by_year.to_csv(stack_secondary_by_year_path, index=False)

combined_program_summary.to_csv(combined_program_summary_path, index=False)
combined_program_by_year.to_csv(combined_program_by_year_path, index=False)

combined_continuity_summary.to_csv(combined_continuity_summary_path, index=False)
combined_continuity_by_year.to_csv(combined_continuity_by_year_path, index=False)

comparison.to_csv(comparison_path, index=False)
continuity_comparison.to_csv(continuity_comparison_path, index=False)

# ======================================================================================
# 14. Validation
# ======================================================================================

validation_rows = []

required_base_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

missing_base_cols = [c for c in required_base_cols if c not in base.columns]
models_found = sorted(base["model_spec"].dropna().unique().tolist()) if "model_spec" in base.columns else []

bad_middle_candidate_count = len(secondary_middle_grid) != expected_middle_candidate_count
bad_stack_candidate_count = len(secondary_stack_grid) != expected_stack_candidate_count

bad_middle_z_equality = secondary_middle_grid[
    secondary_middle_grid["secondary_middle_model_vrp_z_3m_min"].ne(
        secondary_middle_grid["secondary_middle_model_vrp_z_1y_min"]
    )
]

bad_back_z_equality = secondary_back_shortlist[
    secondary_back_shortlist["secondary_back_profile_id"].ne("NO_BACK")
    & secondary_back_shortlist["secondary_back_model_vrp_z_3m_min"].ne(
        secondary_back_shortlist["secondary_back_model_vrp_z_1y_min"]
    )
]

bad_middle_tenor_rows = middle_selected[
    ~middle_selected["tenor"].isin(SECONDARY_MIDDLE_TENORS)
] if not middle_selected.empty else pd.DataFrame()

bad_back_tenor_rows = back_selected[
    ~back_selected["tenor"].isin(SECONDARY_BACK_TENORS)
] if not back_selected.empty else pd.DataFrame()

bad_stack_tenor_rows = stack_secondary_selected[
    ~stack_secondary_selected["tenor"].isin(SECONDARY_MIDDLE_TENORS + SECONDARY_BACK_TENORS)
] if not stack_secondary_selected.empty else pd.DataFrame()

bad_middle_core_gate_dates = middle_selected[
    middle_selected["trade_date"].isin(core_gate_dates_set)
] if not middle_selected.empty else pd.DataFrame()

bad_back_core_gate_dates = back_selected[
    back_selected["trade_date"].isin(core_gate_dates_set)
] if not back_selected.empty else pd.DataFrame()

bad_stack_core_gate_dates = stack_secondary_selected[
    stack_secondary_selected["trade_date"].isin(core_gate_dates_set)
] if not stack_secondary_selected.empty else pd.DataFrame()

stack_program_dupes = (
    combined_program_panel.groupby(["secondary_stack_candidate_id", "trade_date"])
    .size()
    .reset_index(name="rows")
)
bad_program_multi_trade_dates = stack_program_dupes[stack_program_dupes["rows"].gt(1)]

stack_secondary_dupes = (
    stack_secondary_selected.groupby(["secondary_stack_candidate_id", "trade_date"])
    .size()
    .reset_index(name="rows")
    if not stack_secondary_selected.empty
    else pd.DataFrame(columns=["secondary_stack_candidate_id", "trade_date", "rows"])
)
bad_stack_secondary_multi_trade_dates = stack_secondary_dupes[stack_secondary_dupes["rows"].gt(1)]

bad_forecast_middle_flag = secondary_middle_grid[
    secondary_middle_grid["middle_forecast_vol_pct_threshold_used"].astype(bool)
]

bad_forecast_back_flag = secondary_back_shortlist[
    secondary_back_shortlist["back_forecast_vol_pct_threshold_used"].astype(bool)
]

bad_rv21d_middle_contract = secondary_middle_grid[
    ~secondary_middle_grid["middle_uses_rv21d_threshold_contract"].astype(bool)
]

bad_rv21d_back_contract = secondary_back_shortlist[
    ~secondary_back_shortlist["back_uses_rv21d_threshold_contract"].astype(bool)
]

bad_normalized_pnl = base[
    base["normalized_pnl_dollars"].notna()
    & base["normalized_pnl_per_day"].notna()
    & (
        (
            base["normalized_pnl_dollars"] / base["tenor"].replace(0, np.nan)
        )
        - base["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
]

comparison_stack_candidates = sorted(comparison["secondary_stack_candidate_id"].dropna().unique().tolist()) if not comparison.empty else []
expected_stack_candidates = sorted(secondary_stack_grid["secondary_stack_candidate_id"].tolist())

add_validation(
    validation_rows,
    "base_panel_loaded",
    "PASS" if len(base) > 0 else "FAIL",
    f"rows={len(base):,}; path={base_panel_path}",
)

add_validation(
    validation_rows,
    "required_base_columns_available",
    "PASS" if not missing_base_cols else "FAIL",
    f"missing={missing_base_cols}",
)

add_validation(
    validation_rows,
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    validation_rows,
    "locked_core_rules_loaded",
    "PASS" if len(locked_core_rules_long) == 9 else "FAIL",
    f"rows={len(locked_core_rules_long):,}; path={locked_core_rules_long_path}",
)

add_validation(
    validation_rows,
    "core_gate_dates_created",
    "PASS" if len(core_gate_dates) > 0 and len(core_empty_dates) > 0 else "FAIL",
    f"core_gate_dates={len(core_gate_dates):,}; core_empty_dates={len(core_empty_dates):,}; total_dates={total_eligible_signal_dates:,}",
)

add_validation(
    validation_rows,
    "middle_candidate_count",
    "PASS" if not bad_middle_candidate_count else "FAIL",
    f"middle_candidate_count={len(secondary_middle_grid):,}; expected={expected_middle_candidate_count:,}",
)

add_validation(
    validation_rows,
    "stack_candidate_count",
    "PASS" if not bad_stack_candidate_count else "FAIL",
    f"stack_candidate_count={len(secondary_stack_grid):,}; expected={expected_stack_candidate_count:,}",
)

add_validation(
    validation_rows,
    "middle_tenors_only",
    "PASS" if bad_middle_tenor_rows.empty else "FAIL",
    f"bad_rows={len(bad_middle_tenor_rows):,}",
)

add_validation(
    validation_rows,
    "back_shortlist_tenors_only",
    "PASS" if bad_back_tenor_rows.empty else "FAIL",
    f"bad_rows={len(bad_back_tenor_rows):,}",
)

add_validation(
    validation_rows,
    "stack_secondary_tenors_middle_or_back_only",
    "PASS" if bad_stack_tenor_rows.empty else "FAIL",
    f"bad_rows={len(bad_stack_tenor_rows):,}",
)

add_validation(
    validation_rows,
    "secondary_only_on_core_empty_dates",
    "PASS" if bad_middle_core_gate_dates.empty and bad_back_core_gate_dates.empty and bad_stack_core_gate_dates.empty else "FAIL",
    f"middle_on_core_gate_rows={len(bad_middle_core_gate_dates):,}; back_on_core_gate_rows={len(bad_back_core_gate_dates):,}; stack_on_core_gate_rows={len(bad_stack_core_gate_dates):,}",
)

add_validation(
    validation_rows,
    "stack_secondary_one_trade_per_stack_date",
    "PASS" if bad_stack_secondary_multi_trade_dates.empty else "FAIL",
    f"bad_rows={len(bad_stack_secondary_multi_trade_dates):,}",
)

add_validation(
    validation_rows,
    "combined_program_one_trade_per_stack_date",
    "PASS" if bad_program_multi_trade_dates.empty else "FAIL",
    f"bad_rows={len(bad_program_multi_trade_dates):,}",
)

add_validation(
    validation_rows,
    "middle_z3m_equals_z1y",
    "PASS" if bad_middle_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_middle_z_equality):,}",
)

add_validation(
    validation_rows,
    "back_z3m_equals_z1y",
    "PASS" if bad_back_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_back_z_equality):,}",
)

add_validation(
    validation_rows,
    "uses_rv21d_threshold_contract",
    "PASS" if bad_rv21d_middle_contract.empty and bad_rv21d_back_contract.empty else "FAIL",
    f"middle_bad_rows={len(bad_rv21d_middle_contract):,}; back_bad_rows={len(bad_rv21d_back_contract):,}",
)

add_validation(
    validation_rows,
    "forecast_vol_pct_diagnostic_only",
    "PASS" if bad_forecast_middle_flag.empty and bad_forecast_back_flag.empty else "FAIL",
    f"middle_bad_rows={len(bad_forecast_middle_flag):,}; back_bad_rows={len(bad_forecast_back_flag):,}",
)

add_validation(
    validation_rows,
    "all_stacks_compared_program_view",
    "PASS" if comparison_stack_candidates == expected_stack_candidates else "FAIL",
    f"comparison_stacks={len(comparison_stack_candidates):,}; expected={len(expected_stack_candidates):,}",
)

add_validation(
    validation_rows,
    "core_not_reopened",
    "PASS",
    "Locked Core rules loaded from Cell 11R and used only as gate/baseline.",
)

add_validation(
    validation_rows,
    "secondary_middle_not_locked",
    "PASS",
    "Sweep only; no final Secondary Middle lock.",
)

add_validation(
    validation_rows,
    "secondary_back_not_locked",
    "PASS",
    "Back shortlist used only as research input; no final Secondary Back lock.",
)

add_validation(
    validation_rows,
    "secondary_front_not_tested",
    "PASS",
    "No Secondary Front tested in this cell.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed or added.",
)

add_validation(
    validation_rows,
    "no_production_lock",
    "PASS",
    "No production/intraday implementation performed.",
)

add_validation(
    validation_rows,
    "normalized_pnl_per_day_formula",
    "PASS" if bad_normalized_pnl.empty else "FAIL",
    f"bad_rows={len(bad_normalized_pnl):,}",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"].eq("FAIL")]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1_RV21D_repaired_v2.ipynb",
    "cell": "Cell 13R — Secondary Middle sweep, Core-gated, with Back shortlist",
    "timestamp": CELL13R_TIMESTAMP,

    "locked_core_version": LOCKED_CORE_VERSION,
    "locked_core_decision_id": LOCKED_CORE_DECISION_ID,
    "secondary_middle_sweep_version": SECONDARY_MIDDLE_SWEEP_VERSION,
    "secondary_stack_sweep_version": SECONDARY_STACK_SWEEP_VERSION,

    "base_panel_path": str(base_panel_path),
    "locked_core_rules_long_path": str(locked_core_rules_long_path),
    "locked_core_rules_wide_path": str(locked_core_rules_wide_path),

    "middle_candidate_count": int(len(secondary_middle_grid)),
    "expected_middle_candidate_count": int(expected_middle_candidate_count),
    "back_shortlist_count": int(len(secondary_back_shortlist)),
    "stack_candidate_count": int(len(secondary_stack_grid)),
    "expected_stack_candidate_count": int(expected_stack_candidate_count),

    "total_eligible_signal_dates": int(total_eligible_signal_dates),
    "core_gate_dates": int(len(core_gate_dates)),
    "core_empty_dates": int(len(core_empty_dates)),

    "secondary_middle_parameter_grid": {
        "tenors": SECONDARY_MIDDLE_TENORS,
        "model_vrp_log": SECONDARY_MIDDLE_VRP_LOG_GRID,
        "model_vrp_z_3m": SECONDARY_MIDDLE_Z_GRID,
        "model_vrp_z_1y": SECONDARY_MIDDLE_Z_GRID,
        "RSI14_max": SECONDARY_MIDDLE_RSI_GRID,
        "rv21d_vol_pct_min": SECONDARY_MIDDLE_RV21D_GRID,
    },

    "secondary_back_shortlist": secondary_back_shortlist.to_dict(orient="records"),

    "selection_rules": {
        "core_gate": "If any locked Core tenor qualifies on a date, Secondary is ignored.",
        "secondary_test_dates": "Core-empty dates only.",
        "secondary_stack_selection": "At most one Secondary trade per stack/date across available Middle and Back profiles.",
        "program_view": "At most one trade per date: Core first, else one Secondary stack trade.",
        "continuity_view": "Cell 11R-style Core primary one-trade-per-bucket/date plus one Secondary stack trade on Core-empty dates.",
    },

    "targets": {
        "combined_win_rate_target": COMBINED_WIN_RATE_TARGET,
        "total_program_frequency_target": TOTAL_PROGRAM_FREQUENCY_TARGET,
    },

    "middle_grid_csv_path": str(middle_grid_csv_path),
    "middle_grid_parquet_path": str(middle_grid_parquet_path),
    "back_shortlist_csv_path": str(back_shortlist_csv_path),
    "back_shortlist_parquet_path": str(back_shortlist_parquet_path),
    "stack_grid_csv_path": str(stack_grid_csv_path),
    "stack_grid_parquet_path": str(stack_grid_parquet_path),
    "core_gate_dates_path": str(core_gate_dates_path),
    "core_empty_dates_path": str(core_empty_dates_path),
    "middle_selected_panel_path": str(middle_selected_panel_path),
    "middle_selected_panel_csv_path": str(middle_selected_panel_csv_path),
    "back_selected_panel_path": str(back_selected_panel_path),
    "back_selected_panel_csv_path": str(back_selected_panel_csv_path),
    "stack_secondary_selected_panel_path": str(stack_secondary_selected_panel_path),
    "stack_secondary_selected_panel_csv_path": str(stack_secondary_selected_panel_csv_path),
    "combined_program_panel_path": str(combined_program_panel_path),
    "combined_program_panel_csv_path": str(combined_program_panel_csv_path),
    "combined_continuity_panel_path": str(combined_continuity_panel_path),
    "combined_continuity_panel_csv_path": str(combined_continuity_panel_csv_path),
    "core_program_summary_path": str(core_program_summary_path),
    "core_continuity_summary_path": str(core_continuity_summary_path),
    "middle_incremental_summary_path": str(middle_incremental_summary_path),
    "middle_incremental_by_tenor_path": str(middle_incremental_by_tenor_path),
    "middle_incremental_by_year_path": str(middle_incremental_by_year_path),
    "back_shortlist_summary_path": str(back_shortlist_summary_path),
    "stack_secondary_summary_path": str(stack_secondary_summary_path),
    "stack_secondary_by_bucket_path": str(stack_secondary_by_bucket_path),
    "stack_secondary_by_tenor_path": str(stack_secondary_by_tenor_path),
    "stack_secondary_by_year_path": str(stack_secondary_by_year_path),
    "combined_program_summary_path": str(combined_program_summary_path),
    "combined_program_by_year_path": str(combined_program_by_year_path),
    "combined_continuity_summary_path": str(combined_continuity_summary_path),
    "combined_continuity_by_year_path": str(combined_continuity_by_year_path),
    "comparison_path": str(comparison_path),
    "continuity_comparison_path": str(continuity_comparison_path),
    "validation_path": str(validation_path),
    "manifest_path": str(manifest_path),

    "constraints": [
        "Core remains locked and unchanged.",
        "Secondary Middle is swept.",
        "Secondary Back is carried only as a two-candidate shortlist plus NO_BACK.",
        "No Secondary Back lock.",
        "No Secondary Middle lock.",
        "No Secondary Front.",
        "9D remains excluded.",
        "18D remains in Front for Core map.",
        "Middle tenors are 21D/24D.",
        "Back shortlist tenors are 27D/30D/33D.",
        "z3m equals z1y in this sweep.",
        "RV21D threshold contract is rv21d_vol_pct > rv21d_vol_pct_min.",
        "forecast_vol_pct is diagnostic only.",
        "No sizing changes.",
        "No production/intraday implementation.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 13R validation failed. Review validation output above.")

# ======================================================================================
# 15. Display review
# ======================================================================================

print("=" * 100)
print("Cell 13R — Secondary Middle sweep, Core-gated, with Back shortlist")
print("=" * 100)
print(f"Base panel:                         {base_panel_path}")
print(f"Locked Core rules:                  {locked_core_rules_long_path}")
print(f"Middle candidate count:             {len(secondary_middle_grid):,}")
print(f"Back shortlist count:               {len(secondary_back_shortlist):,}")
print(f"Stack candidate count:              {len(secondary_stack_grid):,}")
print(f"Total eligible signal dates:        {total_eligible_signal_dates:,}")
print(f"Core gate dates:                    {len(core_gate_dates):,}")
print(f"Core-empty dates:                   {len(core_empty_dates):,}")
print(f"Core program selected trades:       {len(core_program):,}")
print(f"Core continuity selected trades:    {len(core_primary_continuity):,}")
print(f"Middle selected panel rows:         {len(middle_selected):,}")
print(f"Back selected panel rows:           {len(back_selected):,}")
print(f"Stack Secondary selected rows:      {len(stack_secondary_selected):,}")
print(f"Combined program panel rows:        {len(combined_program_panel):,}")
print(f"Combined continuity panel rows:     {len(combined_continuity_panel):,}")
print("Secondary Middle tenors:            21D / 24D")
print("Back shortlist profiles:            NO_BACK, BACK_FREQ_0009, BACK_CLEAN_0133")
print("Core gate:                          If Core qualifies, Secondary ignored")
print("Program view:                       one trade per date")
print("Continuity view:                    Core primary + one Secondary on Core-empty dates")
print("Secondary Middle locked here:       False")
print("Secondary Back locked here:         False")
print("Sizing locked here:                 False")
print("Production locked here:             False")

print()
print("Validation:")
display(validation)

print()
print("Secondary Middle candidate grid:")
display(secondary_middle_grid)

print()
print("Secondary Back shortlist:")
display(secondary_back_shortlist)

print()
print("Secondary stack grid:")
display(secondary_stack_grid)

print()
print("Core program baseline — one trade per date:")
display(core_program_summary)

print()
print("Core continuity baseline — one trade per bucket/date:")
display(core_continuity_summary)

comparison_display_cols = [
    "secondary_stack_candidate_id",
    "secondary_middle_candidate_id",
    "secondary_back_profile_id",
    "secondary_back_source_candidate_id",

    "secondary_middle_model_vrp_log_min",
    "secondary_middle_model_vrp_z_min",
    "secondary_middle_RSI14_max",
    "secondary_middle_rv21d_vol_pct_min",

    "secondary_back_model_vrp_log_min",
    "secondary_back_model_vrp_z_min",
    "secondary_back_RSI14_max",
    "secondary_back_rv21d_vol_pct_min",

    "trades",
    "unique_trade_dates",
    "signal_date_frequency",
    "win_rate",
    "total_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "profit_factor_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",

    "secondary_trades",
    "secondary_unique_trade_dates",
    "secondary_signal_date_frequency",
    "secondary_win_rate",
    "secondary_total_pnl",
    "secondary_avg_pnl_per_day",
    "secondary_profit_factor",
    "secondary_profit_factor_pnl_per_day",
    "secondary_pnl_per_day_drawdown",
    "secondary_worst_20_trade_pnl_per_day_sum",
    "secondary_avg_pnl_per_day_2025",
    "secondary_avg_pnl_per_day_2026",

    "incremental_middle_trade_count",
    "incremental_back_trade_count",
    "incremental_secondary_trade_count",

    "delta_program_unique_trade_dates_vs_core_only",
    "delta_program_signal_date_frequency_vs_core_only",
    "delta_program_win_rate_vs_core_only",
    "delta_program_total_pnl_vs_core_only",
    "delta_program_avg_pnl_per_day_vs_core_only",
    "delta_program_profit_factor_vs_core_only",
    "delta_program_profit_factor_pnl_per_day_vs_core_only",
    "delta_program_pnl_per_day_drawdown_vs_core_only",
    "delta_program_worst_20_trade_pnl_per_day_sum_vs_core_only",
    "delta_program_avg_pnl_per_day_2026_vs_core_only",

    "passes_frequency_target",
    "passes_win_rate_target",
    "passes_program_targets",
]

print()
print("Candidate comparison — PROGRAM VIEW, Core first else Secondary stack:")
display(comparison[[c for c in comparison_display_cols if c in comparison.columns]])

continuity_display_cols = [
    "secondary_stack_candidate_id",
    "secondary_middle_candidate_id",
    "secondary_back_profile_id",
    "secondary_back_source_candidate_id",

    "secondary_middle_model_vrp_log_min",
    "secondary_middle_model_vrp_z_min",
    "secondary_middle_RSI14_max",
    "secondary_middle_rv21d_vol_pct_min",

    "secondary_back_model_vrp_log_min",
    "secondary_back_model_vrp_z_min",
    "secondary_back_RSI14_max",
    "secondary_back_rv21d_vol_pct_min",

    "trades",
    "unique_trade_dates",
    "signal_date_frequency",
    "win_rate",
    "total_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "profit_factor_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",

    "core_trade_count",
    "secondary_middle_trade_count",
    "secondary_back_trade_count",
    "secondary_trade_count",

    "delta_continuity_trades_vs_core_primary",
    "delta_continuity_unique_trade_dates_vs_core_primary",
    "delta_continuity_signal_date_frequency_vs_core_primary",
    "delta_continuity_win_rate_vs_core_primary",
    "delta_continuity_total_pnl_vs_core_primary",
    "delta_continuity_avg_pnl_per_day_vs_core_primary",
    "delta_continuity_profit_factor_vs_core_primary",
    "delta_continuity_profit_factor_pnl_per_day_vs_core_primary",
    "delta_continuity_pnl_per_day_drawdown_vs_core_primary",
    "delta_continuity_worst_20_trade_pnl_per_day_sum_vs_core_primary",
    "delta_continuity_avg_pnl_per_day_2026_vs_core_primary",
]

print()
print("Candidate comparison — CONTINUITY VIEW, Core primary + Secondary stack:")
display(continuity_comparison[[c for c in continuity_display_cols if c in continuity_comparison.columns]])

print()
print("Secondary Middle incremental summary:")
display(middle_incremental_summary)

print()
print("Secondary Middle incremental by tenor:")
display(middle_incremental_by_tenor)

print()
print("Secondary Middle incremental by year:")
display(middle_incremental_by_year)

print()
print("Secondary Back shortlist summary:")
display(back_shortlist_summary)

print()
print("Stack Secondary incremental summary:")
display(stack_secondary_summary)

print()
print("Stack Secondary incremental by bucket:")
display(stack_secondary_by_bucket)

print()
print("Stack Secondary incremental by tenor:")
display(stack_secondary_by_tenor)

print()
print("Stack Secondary incremental by year:")
display(stack_secondary_by_year)

print()
print("Combined program by year:")
display(combined_program_by_year)

print()
print("Combined continuity by year:")
display(combined_continuity_by_year)

print()
print("Saved outputs:")
for p in [
    middle_grid_csv_path,
    middle_grid_parquet_path,
    back_shortlist_csv_path,
    back_shortlist_parquet_path,
    stack_grid_csv_path,
    stack_grid_parquet_path,
    core_gate_dates_path,
    core_empty_dates_path,
    middle_selected_panel_path,
    middle_selected_panel_csv_path,
    back_selected_panel_path,
    back_selected_panel_csv_path,
    stack_secondary_selected_panel_path,
    stack_secondary_selected_panel_csv_path,
    combined_program_panel_path,
    combined_program_panel_csv_path,
    combined_continuity_panel_path,
    combined_continuity_panel_csv_path,
    core_program_summary_path,
    core_continuity_summary_path,
    middle_incremental_summary_path,
    middle_incremental_by_tenor_path,
    middle_incremental_by_year_path,
    back_shortlist_summary_path,
    stack_secondary_summary_path,
    stack_secondary_by_bucket_path,
    stack_secondary_by_tenor_path,
    stack_secondary_by_year_path,
    combined_program_summary_path,
    combined_program_by_year_path,
    combined_continuity_summary_path,
    combined_continuity_by_year_path,
    comparison_path,
    continuity_comparison_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 13R PASSED — SECONDARY MIDDLE CORE-GATED SWEEP WITH BACK SHORTLIST COMPLETE")

In [ ]:
# ======================================================================================
# Cell 14R — Secondary Front conservative sweep, Core-gated, with Back/Middle shortlist
# ======================================================================================
#
# Objective:
#   Sweep conservative Secondary Front parameters as an incremental layer behind locked
#   Core repaired v1, while carrying Back/Middle shortlist profiles only as research inputs.
#
# Revised target:
#   Total signal-day frequency:        >= 25%
#   Combined program win rate:         >= 80%
#   Total Secondary win rate:          ideally >= 80%
#
# Locked Core:
#   Source:
#       Cell 11R — core_repaired_v1_locked_001
#
#   Core remains unchanged:
#       9D excluded
#       Front  = 12D / 15D / 18D
#       Middle = 21D / 24D
#       Back   = 27D / 30D / 33D
#
# Secondary Front sweep:
#   Tenors:
#       12D / 15D / 18D only
#
#   VRP log:
#       [0.55, 0.60, 0.65]
#
#   z-score:
#       [0.20, 0.30, 0.40, 0.50, 0.60]
#
#   z constraint:
#       z3m = z1y
#
#   RSI14 cap:
#       [72, 73, 74, 75]
#
#   RV21D floor:
#       [6.5, 7.0, 7.5]
#
# Front candidate count:
#       3 * 5 * 4 * 3 = 180
#
# Middle shortlist carried from Cell 13R:
#   NO_MIDDLE
#
#   MIDDLE_FREQ_0011:
#       log > 0.55
#       z3m/z1y > 0.00
#       RSI14 < 75
#       rv21d_vol_pct > 6.5
#
#   MIDDLE_CLEAN_0252:
#       log > 0.65
#       z3m/z1y > 0.60
#       RSI14 < 75
#       rv21d_vol_pct > 7.0
#
# Back shortlist carried from Cell 12R / 13R:
#   NO_BACK
#
#   BACK_FREQ_0009:
#       log > 0.65
#       z3m/z1y > 0.00
#       RSI14 < 77
#       rv21d_vol_pct > 6.5
#
#   BACK_CLEAN_0133:
#       log > 0.70
#       z3m/z1y > 0.60
#       RSI14 < 74
#       rv21d_vol_pct > 6.5
#
# Stack count:
#       180 Front candidates * 3 Middle profiles * 3 Back profiles = 1,620 stacks
#
# Core-gating rule:
#   If any locked Core tenor qualifies on a date:
#       Secondary is ignored.
#
#   If no locked Core tenor qualifies on a date:
#       test the Secondary stack.
#
# Secondary stack selection:
#   For each stack and Core-empty date:
#       select at most one Secondary trade across available Front, Middle, and Back profiles.
#
# Evaluation views:
#   1. Program view:
#       one trade per date
#       Core first, else one Secondary trade
#
#   2. Continuity view:
#       Cell 11R-style Core primary one-trade-per-bucket/date
#       plus one Secondary trade on Core-empty dates
#
# This cell does NOT:
#   1. Reopen Core.
#   2. Lock Secondary Front.
#   3. Lock Secondary Middle.
#   4. Lock Secondary Back.
#   5. Change sizing.
#   6. Implement production/intraday logic.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import itertools
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 620)
pd.set_option("display.width", 900)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL14R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

LOCKED_CORE_VERSION = "core_repaired_v1"
LOCKED_CORE_DECISION_ID = "core_repaired_v1_locked_001"

SECONDARY_FRONT_SWEEP_VERSION = "secondary_front_conservative_core_gated_sweep_v1"
SECONDARY_STACK_SWEEP_VERSION = "secondary_front_with_middle_back_shortlist_core_gated_sweep_v1"

ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]
SECONDARY_FRONT_TENORS = [12, 15, 18]
SECONDARY_MIDDLE_TENORS = [21, 24]
SECONDARY_BACK_TENORS = [27, 30, 33]

BUCKET_ORDER_MAP = {
    "Excluded": 0,
    "Front": 1,
    "Middle": 2,
    "Back": 3,
}

BUCKET_CENTER = {
    "Front": 15,
    "Middle": 21,
    "Back": 30,
}

PRIMARY_CORE_CONTINUITY_UNIVERSE = "core_primary_one_trade_per_bucket_per_date"
CORE_PROGRAM_UNIVERSE = "core_program_one_trade_per_date"

FRONT_INCREMENTAL_UNIVERSE = "secondary_front_incremental_core_empty_dates"
MIDDLE_SHORTLIST_INCREMENTAL_UNIVERSE = "secondary_middle_shortlist_incremental_core_empty_dates"
BACK_SHORTLIST_INCREMENTAL_UNIVERSE = "secondary_back_shortlist_incremental_core_empty_dates"
STACK_SECONDARY_INCREMENTAL_UNIVERSE = "secondary_stack_incremental_core_empty_dates"

COMBINED_PROGRAM_UNIVERSE = "combined_program_core_first_else_secondary_stack"
COMBINED_CONTINUITY_UNIVERSE = "combined_continuity_core_primary_plus_secondary_stack"

SECONDARY_FRONT_VRP_LOG_GRID = [0.55, 0.60, 0.65]
SECONDARY_FRONT_Z_GRID = [0.20, 0.30, 0.40, 0.50, 0.60]
SECONDARY_FRONT_RSI_GRID = [72.0, 73.0, 74.0, 75.0]
SECONDARY_FRONT_RV21D_GRID = [6.5, 7.0, 7.5]

TOTAL_PROGRAM_FREQUENCY_TARGET = 0.25
COMBINED_WIN_RATE_TARGET = 0.80
SECONDARY_WIN_RATE_IDEAL = 0.80
SECONDARY_WIN_RATE_FLOOR = 0.78

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def add_rank_columns(d, group_cols, suffix):
    x = d.copy()

    if x.empty:
        return x

    x[f"rank_z_1y_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    x[f"rank_z_3m_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    x[f"rank_vrp_log_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    x[f"avg_signal_rank_{suffix}"] = x[
        [
            f"rank_z_1y_{suffix}",
            f"rank_z_3m_{suffix}",
            f"rank_vrp_log_{suffix}",
        ]
    ].mean(axis=1)

    if "effective_tenor_bucket" in x.columns:
        x["bucket_center_tenor"] = x["effective_tenor_bucket"].map(BUCKET_CENTER)
        x["bucket_center_tenor"] = x["bucket_center_tenor"].fillna(30)
    else:
        x["bucket_center_tenor"] = 30

    x["distance_to_bucket_center"] = (x["tenor"] - x["bucket_center_tenor"]).abs()

    return x


def select_core_universes(core_qualified_complete):
    q = core_qualified_complete.copy()

    if q.empty:
        return {
            PRIMARY_CORE_CONTINUITY_UNIVERSE: q.copy(),
            CORE_PROGRAM_UNIVERSE: q.copy(),
        }

    q = add_rank_columns(
        q,
        group_cols=["trade_date", "effective_tenor_bucket"],
        suffix="core_bucket_date",
    )

    q = add_rank_columns(
        q,
        group_cols=["trade_date"],
        suffix="core_date",
    )

    core_primary = (
        q.sort_values(
            [
                "trade_date",
                "effective_tenor_bucket_order",
                "avg_signal_rank_core_bucket_date",
                "distance_to_bucket_center",
                "rank_z_1y_core_bucket_date",
                "rank_z_3m_core_bucket_date",
                "rank_vrp_log_core_bucket_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date", "effective_tenor_bucket"], as_index=False)
        .head(1)
        .copy()
    )

    core_program = (
        q.sort_values(
            [
                "trade_date",
                "avg_signal_rank_core_date",
                "rank_z_1y_core_date",
                "rank_z_3m_core_date",
                "rank_vrp_log_core_date",
                "distance_to_bucket_center",
                "effective_tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return {
        PRIMARY_CORE_CONTINUITY_UNIVERSE: core_primary,
        CORE_PROGRAM_UNIVERSE: core_program,
    }


def select_one_per_profile_date(qualified_complete, profile_col, suffix):
    q = qualified_complete.copy()

    if q.empty:
        return q

    q = add_rank_columns(
        q,
        group_cols=[profile_col, "trade_date"],
        suffix=suffix,
    )

    selected = (
        q.sort_values(
            [
                profile_col,
                "trade_date",
                f"avg_signal_rank_{suffix}",
                "distance_to_bucket_center",
                f"rank_z_1y_{suffix}",
                f"rank_z_3m_{suffix}",
                f"rank_vrp_log_{suffix}",
                "effective_tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True, True],
        )
        .groupby([profile_col, "trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return selected


def select_stack_secondary_one_per_date(stack_candidates):
    q = stack_candidates.copy()

    if q.empty:
        return q

    q = add_rank_columns(
        q,
        group_cols=["secondary_stack_candidate_id", "trade_date"],
        suffix="stack_secondary_date",
    )

    selected = (
        q.sort_values(
            [
                "secondary_stack_candidate_id",
                "trade_date",
                "avg_signal_rank_stack_secondary_date",
                "distance_to_bucket_center",
                "rank_z_1y_stack_secondary_date",
                "rank_z_3m_stack_secondary_date",
                "rank_vrp_log_stack_secondary_date",
                "effective_tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True, True],
        )
        .groupby(["secondary_stack_candidate_id", "trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return selected


def summarize_trade_set(df, group_cols, total_eligible_dates=None):
    d = df.copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    numeric_cols = [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "forecast_vol_pct",
        "rv21d_vol_pct",

        "threshold_model_vrp_log",
        "threshold_model_vrp_z_3m",
        "threshold_model_vrp_z_1y",
        "threshold_RSI14_max",
        "threshold_rv21d_vol_pct_min",

        "secondary_front_threshold_model_vrp_log",
        "secondary_front_threshold_model_vrp_z_3m",
        "secondary_front_threshold_model_vrp_z_1y",
        "secondary_front_threshold_RSI14_max",
        "secondary_front_threshold_rv21d_vol_pct_min",

        "secondary_middle_threshold_model_vrp_log",
        "secondary_middle_threshold_model_vrp_z_3m",
        "secondary_middle_threshold_model_vrp_z_1y",
        "secondary_middle_threshold_RSI14_max",
        "secondary_middle_threshold_rv21d_vol_pct_min",

        "secondary_back_threshold_model_vrp_log",
        "secondary_back_threshold_model_vrp_z_3m",
        "secondary_back_threshold_model_vrp_z_1y",
        "secondary_back_threshold_RSI14_max",
        "secondary_back_threshold_rv21d_vol_pct_min",
    ]

    for c in numeric_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d = d.sort_values(["trade_date", "tenor"]).copy()

    grouped = d.groupby(group_cols, dropna=False, observed=False) if group_cols else [((), d)]

    rows = []

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {}

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        unique_dates = int(g["trade_date"].nunique())
        signal_date_frequency = (
            unique_dates / total_eligible_dates
            if total_eligible_dates and total_eligible_dates > 0
            else np.nan
        )

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": unique_dates,
            "signal_date_frequency": float(signal_date_frequency) if pd.notna(signal_date_frequency) else np.nan,
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        for signal_col in [
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",

            "threshold_model_vrp_log",
            "threshold_model_vrp_z_3m",
            "threshold_model_vrp_z_1y",
            "threshold_RSI14_max",
            "threshold_rv21d_vol_pct_min",

            "secondary_front_threshold_model_vrp_log",
            "secondary_front_threshold_model_vrp_z_3m",
            "secondary_front_threshold_model_vrp_z_1y",
            "secondary_front_threshold_RSI14_max",
            "secondary_front_threshold_rv21d_vol_pct_min",

            "secondary_middle_threshold_model_vrp_log",
            "secondary_middle_threshold_model_vrp_z_3m",
            "secondary_middle_threshold_model_vrp_z_1y",
            "secondary_middle_threshold_RSI14_max",
            "secondary_middle_threshold_rv21d_vol_pct_min",

            "secondary_back_threshold_model_vrp_log",
            "secondary_back_threshold_model_vrp_z_3m",
            "secondary_back_threshold_model_vrp_z_1y",
            "secondary_back_threshold_RSI14_max",
            "secondary_back_threshold_rv21d_vol_pct_min",
        ]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        if "program_leg" in g.columns:
            row["core_trade_count"] = int(g["program_leg"].eq("Core").sum())
            row["secondary_front_trade_count"] = int(g["program_leg"].eq("Secondary_Front").sum())
            row["secondary_middle_trade_count"] = int(g["program_leg"].eq("Secondary_Middle").sum())
            row["secondary_back_trade_count"] = int(g["program_leg"].eq("Secondary_Back").sum())
            row["secondary_trade_count"] = int(g["program_leg"].isin([
                "Secondary_Front",
                "Secondary_Middle",
                "Secondary_Back",
            ]).sum())

        rows.append(row)

    return pd.DataFrame(rows)


def attach_stack_grid(summary_df, stack_grid):
    if summary_df.empty:
        return summary_df

    return summary_df.merge(
        stack_grid,
        on="secondary_stack_candidate_id",
        how="left",
        validate="many_to_one",
    )


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


# ======================================================================================
# 2. Load inputs
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01R_unified_fds_signal_base_panel_with_rv21d_*.parquet",
    required=True,
)

locked_core_rules_long_path = latest_file(
    OUT_PROCESSED_DIR,
    "11R_core_repaired_v1_locked_rules_long_*.parquet",
    required=True,
)

locked_core_rules_wide_path = latest_file(
    OUT_PROCESSED_DIR,
    "11R_core_repaired_v1_locked_rules_wide_*.parquet",
    required=True,
)

base = pd.read_parquet(base_panel_path)
locked_core_rules_long = pd.read_parquet(locked_core_rules_long_path)
locked_core_rules_wide = pd.read_parquet(locked_core_rules_wide_path)

base["trade_date"] = pd.to_datetime(base["trade_date"], errors="coerce").dt.normalize()

for c in [
    "tenor",
    "tenor_bucket_order",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")

for c in [
    "tenor",
    "original_tenor_bucket_order",
    "effective_tenor_bucket_order",
    "model_vrp_log_min",
    "model_vrp_z_min",
    "model_vrp_z_3m_min",
    "model_vrp_z_1y_min",
    "RSI14_max",
    "rv21d_vol_pct_min",
]:
    if c in locked_core_rules_long.columns:
        locked_core_rules_long[c] = pd.to_numeric(locked_core_rules_long[c], errors="coerce")

total_eligible_signal_dates = int(base["trade_date"].nunique())

# ======================================================================================
# 3. Apply locked Core rules and build Core baselines
# ======================================================================================

core_thresholds = locked_core_rules_long.rename(columns={
    "model_vrp_log_min": "threshold_model_vrp_log",
    "model_vrp_z_3m_min": "threshold_model_vrp_z_3m",
    "model_vrp_z_1y_min": "threshold_model_vrp_z_1y",
    "RSI14_max": "threshold_RSI14_max",
    "rv21d_vol_pct_min": "threshold_rv21d_vol_pct_min",
})

core_threshold_cols = [
    "tenor",
    "original_tenor_bucket",
    "original_tenor_bucket_order",
    "effective_tenor_bucket",
    "effective_tenor_bucket_order",
    "include_tenor",
    "threshold_model_vrp_log",
    "threshold_model_vrp_z_3m",
    "threshold_model_vrp_z_1y",
    "threshold_RSI14_max",
    "threshold_rv21d_vol_pct_min",
]

base_for_join = base.copy()

base_for_join = base_for_join.rename(columns={
    "tenor_bucket": "source_tenor_bucket",
    "tenor_bucket_order": "source_tenor_bucket_order",
})

core_panel = base_for_join.merge(
    core_thresholds[core_threshold_cols],
    on="tenor",
    how="inner",
    validate="many_to_one",
)

core_signal_available = (
    core_panel["include_tenor"].astype(bool)
    & core_panel["model_vrp_log"].notna()
    & core_panel["model_vrp_z_3m"].notna()
    & core_panel["model_vrp_z_1y"].notna()
    & core_panel["RSI14"].notna()
    & core_panel["rv21d_vol_pct"].notna()
)

core_pass_mask = (
    core_signal_available
    & (core_panel["model_vrp_log"] > core_panel["threshold_model_vrp_log"])
    & (core_panel["model_vrp_z_3m"] > core_panel["threshold_model_vrp_z_3m"])
    & (core_panel["model_vrp_z_1y"] > core_panel["threshold_model_vrp_z_1y"])
    & (core_panel["RSI14"] < core_panel["threshold_RSI14_max"])
    & (core_panel["rv21d_vol_pct"] > core_panel["threshold_rv21d_vol_pct_min"])
)

core_panel["core_repaired_v1_pass"] = core_pass_mask

core_qualified_all_dates = core_panel[core_pass_mask].copy()

core_qualified_complete = core_panel[
    core_pass_mask
    & core_panel["normalized_pnl_dollars"].notna()
    & core_panel["normalized_pnl_per_day"].notna()
].copy()

core_gate_dates = sorted(pd.to_datetime(core_qualified_all_dates["trade_date"].dropna().unique()).tolist())
all_signal_dates = sorted(pd.to_datetime(base["trade_date"].dropna().unique()).tolist())

core_gate_dates_set = set(core_gate_dates)
core_empty_dates = sorted(set(all_signal_dates) - core_gate_dates_set)

core_universes = select_core_universes(core_qualified_complete)

core_primary_continuity = core_universes[PRIMARY_CORE_CONTINUITY_UNIVERSE].copy()
core_program = core_universes[CORE_PROGRAM_UNIVERSE].copy()

core_primary_continuity["source_layer"] = "Core"
core_primary_continuity["program_leg"] = "Core"
core_primary_continuity["selection_universe"] = PRIMARY_CORE_CONTINUITY_UNIVERSE
core_primary_continuity["secondary_stack_candidate_id"] = "CORE_ONLY_BASELINE"
core_primary_continuity["secondary_front_candidate_id"] = "NONE"
core_primary_continuity["secondary_middle_profile_id"] = "NONE"
core_primary_continuity["secondary_back_profile_id"] = "NONE"

core_program["source_layer"] = "Core"
core_program["program_leg"] = "Core"
core_program["selection_universe"] = CORE_PROGRAM_UNIVERSE
core_program["secondary_stack_candidate_id"] = "CORE_ONLY_BASELINE"
core_program["secondary_front_candidate_id"] = "NONE"
core_program["secondary_middle_profile_id"] = "NONE"
core_program["secondary_back_profile_id"] = "NONE"

# ======================================================================================
# 4. Build Secondary Front conservative grid
# ======================================================================================

front_candidate_rows = []

idx = 1

for vrp_log, z, rsi_cap, rv_floor in itertools.product(
    SECONDARY_FRONT_VRP_LOG_GRID,
    SECONDARY_FRONT_Z_GRID,
    SECONDARY_FRONT_RSI_GRID,
    SECONDARY_FRONT_RV21D_GRID,
):
    cid = f"secondary_front_{idx:04d}"

    front_candidate_rows.append({
        "secondary_front_candidate_id": cid,
        "secondary_front_sweep_version": SECONDARY_FRONT_SWEEP_VERSION,
        "secondary_front_candidate_family": "secondary_front_conservative_core_gated",
        "secondary_front_candidate_subfamily": "secondary_front_only",
        "secondary_front_candidate_description": (
            f"Secondary Front conservative, Core-gated; log>{vrp_log:.2f}, "
            f"z3m/z1y>{z:.2f}, RSI14<{rsi_cap:.0f}, rv21d>{rv_floor:.1f}"
        ),

        "secondary_front_tenors": "12,15,18",
        "secondary_front_bucket": "Front",
        "secondary_front_model_vrp_log_min": float(vrp_log),
        "secondary_front_model_vrp_z_min": float(z),
        "secondary_front_model_vrp_z_3m_min": float(z),
        "secondary_front_model_vrp_z_1y_min": float(z),
        "secondary_front_RSI14_max": float(rsi_cap),
        "secondary_front_rv21d_vol_pct_min": float(rv_floor),

        "front_z3m_equals_z1y": True,
        "front_uses_rv21d_threshold_contract": True,
        "front_forecast_vol_pct_threshold_used": False,
        "core_gated": True,
        "secondary_front_tested": True,
        "secondary_middle_tested_as_shortlist": True,
        "secondary_back_tested_as_shortlist": True,
        "secondary_front_locked_here": False,
        "secondary_middle_locked_here": False,
        "secondary_back_locked_here": False,
        "sizing_locked_here": False,
        "production_locked_here": False,
    })

    idx += 1

secondary_front_grid = pd.DataFrame(front_candidate_rows)

expected_front_candidate_count = (
    len(SECONDARY_FRONT_VRP_LOG_GRID)
    * len(SECONDARY_FRONT_Z_GRID)
    * len(SECONDARY_FRONT_RSI_GRID)
    * len(SECONDARY_FRONT_RV21D_GRID)
)

# ======================================================================================
# 5. Build Middle and Back shortlists
# ======================================================================================

secondary_middle_shortlist = pd.DataFrame([
    {
        "secondary_middle_profile_id": "NO_MIDDLE",
        "secondary_middle_source_candidate_id": "NONE",
        "secondary_middle_profile_description": "No Secondary Middle carried.",
        "secondary_middle_tenors": "",
        "secondary_middle_bucket": "",
        "secondary_middle_model_vrp_log_min": np.nan,
        "secondary_middle_model_vrp_z_min": np.nan,
        "secondary_middle_model_vrp_z_3m_min": np.nan,
        "secondary_middle_model_vrp_z_1y_min": np.nan,
        "secondary_middle_RSI14_max": np.nan,
        "secondary_middle_rv21d_vol_pct_min": np.nan,
        "middle_z3m_equals_z1y": True,
        "middle_uses_rv21d_threshold_contract": True,
        "middle_forecast_vol_pct_threshold_used": False,
        "secondary_middle_tested": False,
        "secondary_middle_locked_here": False,
    },
    {
        "secondary_middle_profile_id": "MIDDLE_FREQ_0011",
        "secondary_middle_source_candidate_id": "secondary_middle_0011",
        "secondary_middle_profile_description": "Frequency Middle candidate from Cell 13R: log>0.55, z>0.00, RSI14<75, rv21d>6.5.",
        "secondary_middle_tenors": "21,24",
        "secondary_middle_bucket": "Middle",
        "secondary_middle_model_vrp_log_min": 0.55,
        "secondary_middle_model_vrp_z_min": 0.00,
        "secondary_middle_model_vrp_z_3m_min": 0.00,
        "secondary_middle_model_vrp_z_1y_min": 0.00,
        "secondary_middle_RSI14_max": 75.0,
        "secondary_middle_rv21d_vol_pct_min": 6.5,
        "middle_z3m_equals_z1y": True,
        "middle_uses_rv21d_threshold_contract": True,
        "middle_forecast_vol_pct_threshold_used": False,
        "secondary_middle_tested": True,
        "secondary_middle_locked_here": False,
    },
    {
        "secondary_middle_profile_id": "MIDDLE_CLEAN_0252",
        "secondary_middle_source_candidate_id": "secondary_middle_0252",
        "secondary_middle_profile_description": "Cleaner Middle reference from Cell 13R: log>0.65, z>0.60, RSI14<75, rv21d>7.0.",
        "secondary_middle_tenors": "21,24",
        "secondary_middle_bucket": "Middle",
        "secondary_middle_model_vrp_log_min": 0.65,
        "secondary_middle_model_vrp_z_min": 0.60,
        "secondary_middle_model_vrp_z_3m_min": 0.60,
        "secondary_middle_model_vrp_z_1y_min": 0.60,
        "secondary_middle_RSI14_max": 75.0,
        "secondary_middle_rv21d_vol_pct_min": 7.0,
        "middle_z3m_equals_z1y": True,
        "middle_uses_rv21d_threshold_contract": True,
        "middle_forecast_vol_pct_threshold_used": False,
        "secondary_middle_tested": True,
        "secondary_middle_locked_here": False,
    },
])

secondary_back_shortlist = pd.DataFrame([
    {
        "secondary_back_profile_id": "NO_BACK",
        "secondary_back_source_candidate_id": "NONE",
        "secondary_back_profile_description": "No Secondary Back carried.",
        "secondary_back_tenors": "",
        "secondary_back_bucket": "",
        "secondary_back_model_vrp_log_min": np.nan,
        "secondary_back_model_vrp_z_min": np.nan,
        "secondary_back_model_vrp_z_3m_min": np.nan,
        "secondary_back_model_vrp_z_1y_min": np.nan,
        "secondary_back_RSI14_max": np.nan,
        "secondary_back_rv21d_vol_pct_min": np.nan,
        "back_z3m_equals_z1y": True,
        "back_uses_rv21d_threshold_contract": True,
        "back_forecast_vol_pct_threshold_used": False,
        "secondary_back_tested": False,
        "secondary_back_locked_here": False,
    },
    {
        "secondary_back_profile_id": "BACK_FREQ_0009",
        "secondary_back_source_candidate_id": "secondary_back_0009",
        "secondary_back_profile_description": "Frequency Back candidate from Cell 12R/13R: log>0.65, z>0.00, RSI14<77, rv21d>6.5.",
        "secondary_back_tenors": "27,30,33",
        "secondary_back_bucket": "Back",
        "secondary_back_model_vrp_log_min": 0.65,
        "secondary_back_model_vrp_z_min": 0.00,
        "secondary_back_model_vrp_z_3m_min": 0.00,
        "secondary_back_model_vrp_z_1y_min": 0.00,
        "secondary_back_RSI14_max": 77.0,
        "secondary_back_rv21d_vol_pct_min": 6.5,
        "back_z3m_equals_z1y": True,
        "back_uses_rv21d_threshold_contract": True,
        "back_forecast_vol_pct_threshold_used": False,
        "secondary_back_tested": True,
        "secondary_back_locked_here": False,
    },
    {
        "secondary_back_profile_id": "BACK_CLEAN_0133",
        "secondary_back_source_candidate_id": "secondary_back_0133",
        "secondary_back_profile_description": "Clean Back reference from Cell 12R/13R: log>0.70, z>0.60, RSI14<74, rv21d>6.5.",
        "secondary_back_tenors": "27,30,33",
        "secondary_back_bucket": "Back",
        "secondary_back_model_vrp_log_min": 0.70,
        "secondary_back_model_vrp_z_min": 0.60,
        "secondary_back_model_vrp_z_3m_min": 0.60,
        "secondary_back_model_vrp_z_1y_min": 0.60,
        "secondary_back_RSI14_max": 74.0,
        "secondary_back_rv21d_vol_pct_min": 6.5,
        "back_z3m_equals_z1y": True,
        "back_uses_rv21d_threshold_contract": True,
        "back_forecast_vol_pct_threshold_used": False,
        "secondary_back_tested": True,
        "secondary_back_locked_here": False,
    },
])

# ======================================================================================
# 6. Build stack grid: Front candidates × Middle shortlist × Back shortlist
# ======================================================================================

secondary_stack_grid = secondary_front_grid.merge(
    secondary_middle_shortlist,
    how="cross",
).merge(
    secondary_back_shortlist,
    how="cross",
)

secondary_stack_grid = secondary_stack_grid.reset_index(drop=True)
secondary_stack_grid["secondary_stack_candidate_id"] = [
    f"secondary_stack_{i + 1:04d}" for i in range(len(secondary_stack_grid))
]

secondary_stack_grid["secondary_stack_sweep_version"] = SECONDARY_STACK_SWEEP_VERSION
secondary_stack_grid["secondary_stack_family"] = "secondary_front_with_middle_back_shortlist_core_gated"
secondary_stack_grid["secondary_stack_description"] = (
    secondary_stack_grid["secondary_front_candidate_id"]
    + " + "
    + secondary_stack_grid["secondary_middle_profile_id"]
    + " + "
    + secondary_stack_grid["secondary_back_profile_id"]
)

expected_stack_candidate_count = (
    len(secondary_front_grid)
    * len(secondary_middle_shortlist)
    * len(secondary_back_shortlist)
)

# ======================================================================================
# 7. Apply Secondary Front candidates on Core-empty dates
# ======================================================================================

front_base = base_for_join[
    base_for_join["tenor"].isin(SECONDARY_FRONT_TENORS)
    & base_for_join["trade_date"].isin(core_empty_dates)
].copy()

front_base["effective_tenor_bucket"] = "Front"
front_base["effective_tenor_bucket_order"] = BUCKET_ORDER_MAP["Front"]

front_panel = front_base.merge(
    secondary_front_grid,
    how="cross",
)

front_panel = front_panel.rename(columns={
    "secondary_front_model_vrp_log_min": "secondary_front_threshold_model_vrp_log",
    "secondary_front_model_vrp_z_3m_min": "secondary_front_threshold_model_vrp_z_3m",
    "secondary_front_model_vrp_z_1y_min": "secondary_front_threshold_model_vrp_z_1y",
    "secondary_front_RSI14_max": "secondary_front_threshold_RSI14_max",
    "secondary_front_rv21d_vol_pct_min": "secondary_front_threshold_rv21d_vol_pct_min",
})

front_signal_available = (
    front_panel["model_vrp_log"].notna()
    & front_panel["model_vrp_z_3m"].notna()
    & front_panel["model_vrp_z_1y"].notna()
    & front_panel["RSI14"].notna()
    & front_panel["rv21d_vol_pct"].notna()
)

front_pass_mask = (
    front_signal_available
    & (front_panel["model_vrp_log"] > front_panel["secondary_front_threshold_model_vrp_log"])
    & (front_panel["model_vrp_z_3m"] > front_panel["secondary_front_threshold_model_vrp_z_3m"])
    & (front_panel["model_vrp_z_1y"] > front_panel["secondary_front_threshold_model_vrp_z_1y"])
    & (front_panel["RSI14"] < front_panel["secondary_front_threshold_RSI14_max"])
    & (front_panel["rv21d_vol_pct"] > front_panel["secondary_front_threshold_rv21d_vol_pct_min"])
)

front_panel["secondary_front_pass"] = front_pass_mask

front_qualified_complete = front_panel[
    front_pass_mask
    & front_panel["normalized_pnl_dollars"].notna()
    & front_panel["normalized_pnl_per_day"].notna()
].copy()

front_selected = select_one_per_profile_date(
    front_qualified_complete,
    profile_col="secondary_front_candidate_id",
    suffix="front_secondary_date",
)

front_selected["source_layer"] = "Secondary"
front_selected["program_leg"] = "Secondary_Front"
front_selected["selection_universe"] = FRONT_INCREMENTAL_UNIVERSE

# ======================================================================================
# 8. Apply Middle shortlist profiles on Core-empty dates
# ======================================================================================

active_middle_profiles = secondary_middle_shortlist[
    secondary_middle_shortlist["secondary_middle_profile_id"].ne("NO_MIDDLE")
].copy()

middle_base = base_for_join[
    base_for_join["tenor"].isin(SECONDARY_MIDDLE_TENORS)
    & base_for_join["trade_date"].isin(core_empty_dates)
].copy()

middle_base["effective_tenor_bucket"] = "Middle"
middle_base["effective_tenor_bucket_order"] = BUCKET_ORDER_MAP["Middle"]

middle_panel = middle_base.merge(
    active_middle_profiles,
    how="cross",
)

middle_panel = middle_panel.rename(columns={
    "secondary_middle_model_vrp_log_min": "secondary_middle_threshold_model_vrp_log",
    "secondary_middle_model_vrp_z_3m_min": "secondary_middle_threshold_model_vrp_z_3m",
    "secondary_middle_model_vrp_z_1y_min": "secondary_middle_threshold_model_vrp_z_1y",
    "secondary_middle_RSI14_max": "secondary_middle_threshold_RSI14_max",
    "secondary_middle_rv21d_vol_pct_min": "secondary_middle_threshold_rv21d_vol_pct_min",
})

middle_signal_available = (
    middle_panel["model_vrp_log"].notna()
    & middle_panel["model_vrp_z_3m"].notna()
    & middle_panel["model_vrp_z_1y"].notna()
    & middle_panel["RSI14"].notna()
    & middle_panel["rv21d_vol_pct"].notna()
)

middle_pass_mask = (
    middle_signal_available
    & (middle_panel["model_vrp_log"] > middle_panel["secondary_middle_threshold_model_vrp_log"])
    & (middle_panel["model_vrp_z_3m"] > middle_panel["secondary_middle_threshold_model_vrp_z_3m"])
    & (middle_panel["model_vrp_z_1y"] > middle_panel["secondary_middle_threshold_model_vrp_z_1y"])
    & (middle_panel["RSI14"] < middle_panel["secondary_middle_threshold_RSI14_max"])
    & (middle_panel["rv21d_vol_pct"] > middle_panel["secondary_middle_threshold_rv21d_vol_pct_min"])
)

middle_panel["secondary_middle_pass"] = middle_pass_mask

middle_qualified_complete = middle_panel[
    middle_pass_mask
    & middle_panel["normalized_pnl_dollars"].notna()
    & middle_panel["normalized_pnl_per_day"].notna()
].copy()

middle_selected = select_one_per_profile_date(
    middle_qualified_complete,
    profile_col="secondary_middle_profile_id",
    suffix="middle_secondary_date",
)

middle_selected["source_layer"] = "Secondary"
middle_selected["program_leg"] = "Secondary_Middle"
middle_selected["selection_universe"] = MIDDLE_SHORTLIST_INCREMENTAL_UNIVERSE

# ======================================================================================
# 9. Apply Back shortlist profiles on Core-empty dates
# ======================================================================================

active_back_profiles = secondary_back_shortlist[
    secondary_back_shortlist["secondary_back_profile_id"].ne("NO_BACK")
].copy()

back_base = base_for_join[
    base_for_join["tenor"].isin(SECONDARY_BACK_TENORS)
    & base_for_join["trade_date"].isin(core_empty_dates)
].copy()

back_base["effective_tenor_bucket"] = "Back"
back_base["effective_tenor_bucket_order"] = BUCKET_ORDER_MAP["Back"]

back_panel = back_base.merge(
    active_back_profiles,
    how="cross",
)

back_panel = back_panel.rename(columns={
    "secondary_back_model_vrp_log_min": "secondary_back_threshold_model_vrp_log",
    "secondary_back_model_vrp_z_3m_min": "secondary_back_threshold_model_vrp_z_3m",
    "secondary_back_model_vrp_z_1y_min": "secondary_back_threshold_model_vrp_z_1y",
    "secondary_back_RSI14_max": "secondary_back_threshold_RSI14_max",
    "secondary_back_rv21d_vol_pct_min": "secondary_back_threshold_rv21d_vol_pct_min",
})

back_signal_available = (
    back_panel["model_vrp_log"].notna()
    & back_panel["model_vrp_z_3m"].notna()
    & back_panel["model_vrp_z_1y"].notna()
    & back_panel["RSI14"].notna()
    & back_panel["rv21d_vol_pct"].notna()
)

back_pass_mask = (
    back_signal_available
    & (back_panel["model_vrp_log"] > back_panel["secondary_back_threshold_model_vrp_log"])
    & (back_panel["model_vrp_z_3m"] > back_panel["secondary_back_threshold_model_vrp_z_3m"])
    & (back_panel["model_vrp_z_1y"] > back_panel["secondary_back_threshold_model_vrp_z_1y"])
    & (back_panel["RSI14"] < back_panel["secondary_back_threshold_RSI14_max"])
    & (back_panel["rv21d_vol_pct"] > back_panel["secondary_back_threshold_rv21d_vol_pct_min"])
)

back_panel["secondary_back_pass"] = back_pass_mask

back_qualified_complete = back_panel[
    back_pass_mask
    & back_panel["normalized_pnl_dollars"].notna()
    & back_panel["normalized_pnl_per_day"].notna()
].copy()

back_selected = select_one_per_profile_date(
    back_qualified_complete,
    profile_col="secondary_back_profile_id",
    suffix="back_secondary_date",
)

back_selected["source_layer"] = "Secondary"
back_selected["program_leg"] = "Secondary_Back"
back_selected["selection_universe"] = BACK_SHORTLIST_INCREMENTAL_UNIVERSE

# ======================================================================================
# 10. Build stack-level Secondary selected panel
# ======================================================================================

stack_key_cols = [
    "secondary_stack_candidate_id",
    "secondary_front_candidate_id",
    "secondary_middle_profile_id",
    "secondary_back_profile_id",
    "secondary_middle_source_candidate_id",
    "secondary_back_source_candidate_id",
]

stack_map = secondary_stack_grid[stack_key_cols].copy()

front_stack_candidates = pd.DataFrame()
middle_stack_candidates = pd.DataFrame()
back_stack_candidates = pd.DataFrame()

if not front_selected.empty:
    front_stack_candidates = front_selected.merge(
        stack_map,
        on="secondary_front_candidate_id",
        how="inner",
        validate="many_to_many",
    )

if not middle_selected.empty:
    middle_stack_candidates = middle_selected.merge(
        stack_map,
        on="secondary_middle_profile_id",
        how="inner",
        validate="many_to_many",
    )

if not back_selected.empty:
    back_stack_candidates = back_selected.merge(
        stack_map,
        on="secondary_back_profile_id",
        how="inner",
        validate="many_to_many",
    )

stack_secondary_candidates = pd.concat(
    [
        front_stack_candidates,
        middle_stack_candidates,
        back_stack_candidates,
    ],
    ignore_index=True,
)

stack_secondary_selected = select_stack_secondary_one_per_date(stack_secondary_candidates)

stack_secondary_selected["source_layer"] = "Secondary"
stack_secondary_selected["selection_universe"] = STACK_SECONDARY_INCREMENTAL_UNIVERSE

# ======================================================================================
# 11. Build combined program and continuity panels
# ======================================================================================

core_program_stack = core_program.copy().merge(
    secondary_stack_grid[stack_key_cols],
    how="cross",
)

core_program_stack["selection_universe"] = COMBINED_PROGRAM_UNIVERSE

core_continuity_stack = core_primary_continuity.copy().merge(
    secondary_stack_grid[stack_key_cols],
    how="cross",
)

core_continuity_stack["selection_universe"] = COMBINED_CONTINUITY_UNIVERSE

combined_program_panel = pd.concat(
    [
        core_program_stack,
        stack_secondary_selected.assign(selection_universe=COMBINED_PROGRAM_UNIVERSE),
    ],
    ignore_index=True,
)

combined_continuity_panel = pd.concat(
    [
        core_continuity_stack,
        stack_secondary_selected.assign(selection_universe=COMBINED_CONTINUITY_UNIVERSE),
    ],
    ignore_index=True,
)

# ======================================================================================
# 12. Summaries
# ======================================================================================

core_program_summary = summarize_trade_set(
    core_program.assign(selection_universe=CORE_PROGRAM_UNIVERSE),
    group_cols=["secondary_stack_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

core_continuity_summary = summarize_trade_set(
    core_primary_continuity.assign(selection_universe=PRIMARY_CORE_CONTINUITY_UNIVERSE),
    group_cols=["secondary_stack_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

front_incremental_summary = summarize_trade_set(
    front_selected,
    group_cols=["secondary_front_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

front_incremental_by_tenor = summarize_trade_set(
    front_selected,
    group_cols=["secondary_front_candidate_id", "selection_universe", "tenor"],
    total_eligible_dates=total_eligible_signal_dates,
)

front_incremental_by_year_input = (
    front_selected.assign(year=front_selected["trade_date"].dt.year.astype(int))
    if not front_selected.empty
    else front_selected.copy()
)

front_incremental_by_year = summarize_trade_set(
    front_incremental_by_year_input,
    group_cols=["secondary_front_candidate_id", "selection_universe", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)

middle_shortlist_summary = summarize_trade_set(
    middle_selected,
    group_cols=["secondary_middle_profile_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

back_shortlist_summary = summarize_trade_set(
    back_selected,
    group_cols=["secondary_back_profile_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

stack_secondary_summary = summarize_trade_set(
    stack_secondary_selected,
    group_cols=["secondary_stack_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)
stack_secondary_summary = attach_stack_grid(stack_secondary_summary, secondary_stack_grid)

stack_secondary_by_bucket = summarize_trade_set(
    stack_secondary_selected,
    group_cols=[
        "secondary_stack_candidate_id",
        "selection_universe",
        "effective_tenor_bucket",
        "effective_tenor_bucket_order",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)
stack_secondary_by_bucket = attach_stack_grid(stack_secondary_by_bucket, secondary_stack_grid)

stack_secondary_by_tenor = summarize_trade_set(
    stack_secondary_selected,
    group_cols=[
        "secondary_stack_candidate_id",
        "selection_universe",
        "tenor",
        "effective_tenor_bucket",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)
stack_secondary_by_tenor = attach_stack_grid(stack_secondary_by_tenor, secondary_stack_grid)

stack_secondary_by_year_input = (
    stack_secondary_selected.assign(year=stack_secondary_selected["trade_date"].dt.year.astype(int))
    if not stack_secondary_selected.empty
    else stack_secondary_selected.copy()
)

stack_secondary_by_year = summarize_trade_set(
    stack_secondary_by_year_input,
    group_cols=["secondary_stack_candidate_id", "selection_universe", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)
stack_secondary_by_year = attach_stack_grid(stack_secondary_by_year, secondary_stack_grid)

combined_program_summary = summarize_trade_set(
    combined_program_panel,
    group_cols=["secondary_stack_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)
combined_program_summary = attach_stack_grid(combined_program_summary, secondary_stack_grid)

combined_continuity_summary = summarize_trade_set(
    combined_continuity_panel,
    group_cols=["secondary_stack_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)
combined_continuity_summary = attach_stack_grid(combined_continuity_summary, secondary_stack_grid)

combined_program_by_year_input = (
    combined_program_panel.assign(year=combined_program_panel["trade_date"].dt.year.astype(int))
    if not combined_program_panel.empty
    else combined_program_panel.copy()
)

combined_program_by_year = summarize_trade_set(
    combined_program_by_year_input,
    group_cols=["secondary_stack_candidate_id", "selection_universe", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)
combined_program_by_year = attach_stack_grid(combined_program_by_year, secondary_stack_grid)

combined_continuity_by_year_input = (
    combined_continuity_panel.assign(year=combined_continuity_panel["trade_date"].dt.year.astype(int))
    if not combined_continuity_panel.empty
    else combined_continuity_panel.copy()
)

combined_continuity_by_year = summarize_trade_set(
    combined_continuity_by_year_input,
    group_cols=["secondary_stack_candidate_id", "selection_universe", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)
combined_continuity_by_year = attach_stack_grid(combined_continuity_by_year, secondary_stack_grid)

# ======================================================================================
# 13. Candidate comparison tables
# ======================================================================================

core_program_ref = core_program_summary.iloc[0].to_dict() if len(core_program_summary) else {}
core_continuity_ref = core_continuity_summary.iloc[0].to_dict() if len(core_continuity_summary) else {}

comparison = combined_program_summary.copy()

if not comparison.empty:
    sec_cols = [
        "secondary_stack_candidate_id",
        "trades",
        "unique_trade_dates",
        "signal_date_frequency",
        "win_rate",
        "total_pnl",
        "avg_pnl_per_day",
        "profit_factor",
        "profit_factor_pnl_per_day",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",
        "avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026",
        "secondary_front_trade_count",
        "secondary_middle_trade_count",
        "secondary_back_trade_count",
        "secondary_trade_count",
    ]

    sec_metrics = stack_secondary_summary[
        [c for c in sec_cols if c in stack_secondary_summary.columns]
    ].copy()

    sec_metrics = sec_metrics.rename(columns={
        "trades": "secondary_trades",
        "unique_trade_dates": "secondary_unique_trade_dates",
        "signal_date_frequency": "secondary_signal_date_frequency",
        "win_rate": "secondary_win_rate",
        "total_pnl": "secondary_total_pnl",
        "avg_pnl_per_day": "secondary_avg_pnl_per_day",
        "profit_factor": "secondary_profit_factor",
        "profit_factor_pnl_per_day": "secondary_profit_factor_pnl_per_day",
        "pnl_per_day_drawdown": "secondary_pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum": "secondary_worst_20_trade_pnl_per_day_sum",
        "avg_pnl_per_day_2025": "secondary_avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026": "secondary_avg_pnl_per_day_2026",
        "secondary_front_trade_count": "incremental_front_trade_count",
        "secondary_middle_trade_count": "incremental_middle_trade_count",
        "secondary_back_trade_count": "incremental_back_trade_count",
        "secondary_trade_count": "incremental_secondary_trade_count",
    })

    comparison = comparison.merge(
        sec_metrics,
        on="secondary_stack_candidate_id",
        how="left",
        validate="one_to_one",
    )

    for metric in [
        "trades",
        "unique_trade_dates",
        "signal_date_frequency",
        "win_rate",
        "total_pnl",
        "avg_pnl_per_day",
        "profit_factor",
        "profit_factor_pnl_per_day",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",
        "avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026",
    ]:
        if metric in comparison.columns and metric in core_program_ref:
            comparison[f"delta_program_{metric}_vs_core_only"] = comparison[metric] - core_program_ref[metric]

    comparison["passes_frequency_target_25pct"] = comparison["signal_date_frequency"] >= TOTAL_PROGRAM_FREQUENCY_TARGET
    comparison["passes_combined_win_rate_target_80pct"] = comparison["win_rate"] >= COMBINED_WIN_RATE_TARGET
    comparison["passes_secondary_win_rate_ideal_80pct"] = comparison["secondary_win_rate"] >= SECONDARY_WIN_RATE_IDEAL
    comparison["passes_secondary_win_rate_floor_78pct"] = comparison["secondary_win_rate"] >= SECONDARY_WIN_RATE_FLOOR

    comparison["passes_program_targets"] = (
        comparison["passes_frequency_target_25pct"]
        & comparison["passes_combined_win_rate_target_80pct"]
    )

    comparison["passes_full_ideal_targets"] = (
        comparison["passes_program_targets"]
        & comparison["passes_secondary_win_rate_ideal_80pct"]
    )

    comparison["passes_acceptable_secondary_floor"] = (
        comparison["passes_program_targets"]
        & comparison["passes_secondary_win_rate_floor_78pct"]
    )

    comparison = comparison.sort_values(
        [
            "passes_full_ideal_targets",
            "passes_acceptable_secondary_floor",
            "passes_program_targets",
            "secondary_win_rate",
            "signal_date_frequency",
            "win_rate",
            "avg_pnl_per_day",
            "profit_factor_pnl_per_day",
            "avg_pnl_per_day_2026",
        ],
        ascending=[False, False, False, False, False, False, False, False, False],
    ).reset_index(drop=True)

continuity_comparison = combined_continuity_summary.copy()

if not continuity_comparison.empty:
    for metric in [
        "trades",
        "unique_trade_dates",
        "signal_date_frequency",
        "win_rate",
        "total_pnl",
        "avg_pnl_per_day",
        "profit_factor",
        "profit_factor_pnl_per_day",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",
        "avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026",
    ]:
        if metric in continuity_comparison.columns and metric in core_continuity_ref:
            continuity_comparison[f"delta_continuity_{metric}_vs_core_primary"] = (
                continuity_comparison[metric] - core_continuity_ref[metric]
            )

    continuity_comparison = continuity_comparison.sort_values(
        [
            "signal_date_frequency",
            "win_rate",
            "avg_pnl_per_day",
            "profit_factor_pnl_per_day",
            "avg_pnl_per_day_2026",
        ],
        ascending=[False, False, False, False, False],
    ).reset_index(drop=True)

# ======================================================================================
# 14. Save outputs
# ======================================================================================

front_grid_csv_path = OUT_AUDIT_DIR / f"14R_secondary_front_conservative_grid_{CELL14R_TIMESTAMP}.csv"
front_grid_parquet_path = OUT_PROCESSED_DIR / f"14R_secondary_front_conservative_grid_{CELL14R_TIMESTAMP}.parquet"

middle_shortlist_csv_path = OUT_AUDIT_DIR / f"14R_secondary_middle_shortlist_{CELL14R_TIMESTAMP}.csv"
middle_shortlist_parquet_path = OUT_PROCESSED_DIR / f"14R_secondary_middle_shortlist_{CELL14R_TIMESTAMP}.parquet"

back_shortlist_csv_path = OUT_AUDIT_DIR / f"14R_secondary_back_shortlist_{CELL14R_TIMESTAMP}.csv"
back_shortlist_parquet_path = OUT_PROCESSED_DIR / f"14R_secondary_back_shortlist_{CELL14R_TIMESTAMP}.parquet"

stack_grid_csv_path = OUT_AUDIT_DIR / f"14R_secondary_front_middle_back_stack_grid_{CELL14R_TIMESTAMP}.csv"
stack_grid_parquet_path = OUT_PROCESSED_DIR / f"14R_secondary_front_middle_back_stack_grid_{CELL14R_TIMESTAMP}.parquet"

core_gate_dates_path = OUT_AUDIT_DIR / f"14R_core_gate_dates_{CELL14R_TIMESTAMP}.csv"
core_empty_dates_path = OUT_AUDIT_DIR / f"14R_core_empty_dates_{CELL14R_TIMESTAMP}.csv"

front_selected_panel_path = OUT_PROCESSED_DIR / f"14R_secondary_front_selected_panel_{CELL14R_TIMESTAMP}.parquet"
front_selected_panel_csv_path = OUT_AUDIT_DIR / f"14R_secondary_front_selected_panel_{CELL14R_TIMESTAMP}.csv"

middle_selected_panel_path = OUT_PROCESSED_DIR / f"14R_secondary_middle_shortlist_selected_panel_{CELL14R_TIMESTAMP}.parquet"
middle_selected_panel_csv_path = OUT_AUDIT_DIR / f"14R_secondary_middle_shortlist_selected_panel_{CELL14R_TIMESTAMP}.csv"

back_selected_panel_path = OUT_PROCESSED_DIR / f"14R_secondary_back_shortlist_selected_panel_{CELL14R_TIMESTAMP}.parquet"
back_selected_panel_csv_path = OUT_AUDIT_DIR / f"14R_secondary_back_shortlist_selected_panel_{CELL14R_TIMESTAMP}.csv"

stack_secondary_selected_panel_path = OUT_PROCESSED_DIR / f"14R_secondary_stack_selected_panel_{CELL14R_TIMESTAMP}.parquet"
stack_secondary_selected_panel_csv_path = OUT_AUDIT_DIR / f"14R_secondary_stack_selected_panel_{CELL14R_TIMESTAMP}.csv"

combined_program_panel_path = OUT_PROCESSED_DIR / f"14R_combined_program_core_first_else_secondary_stack_panel_{CELL14R_TIMESTAMP}.parquet"
combined_program_panel_csv_path = OUT_AUDIT_DIR / f"14R_combined_program_core_first_else_secondary_stack_panel_{CELL14R_TIMESTAMP}.csv"

combined_continuity_panel_path = OUT_PROCESSED_DIR / f"14R_combined_continuity_core_primary_plus_secondary_stack_panel_{CELL14R_TIMESTAMP}.parquet"
combined_continuity_panel_csv_path = OUT_AUDIT_DIR / f"14R_combined_continuity_core_primary_plus_secondary_stack_panel_{CELL14R_TIMESTAMP}.csv"

core_program_summary_path = OUT_AUDIT_DIR / f"14R_core_program_baseline_{CELL14R_TIMESTAMP}.csv"
core_continuity_summary_path = OUT_AUDIT_DIR / f"14R_core_continuity_baseline_{CELL14R_TIMESTAMP}.csv"

front_incremental_summary_path = OUT_AUDIT_DIR / f"14R_secondary_front_incremental_summary_{CELL14R_TIMESTAMP}.csv"
front_incremental_by_tenor_path = OUT_AUDIT_DIR / f"14R_secondary_front_incremental_by_tenor_{CELL14R_TIMESTAMP}.csv"
front_incremental_by_year_path = OUT_AUDIT_DIR / f"14R_secondary_front_incremental_by_year_{CELL14R_TIMESTAMP}.csv"

middle_shortlist_summary_path = OUT_AUDIT_DIR / f"14R_secondary_middle_shortlist_summary_{CELL14R_TIMESTAMP}.csv"
back_shortlist_summary_path = OUT_AUDIT_DIR / f"14R_secondary_back_shortlist_summary_{CELL14R_TIMESTAMP}.csv"

stack_secondary_summary_path = OUT_AUDIT_DIR / f"14R_secondary_stack_incremental_summary_{CELL14R_TIMESTAMP}.csv"
stack_secondary_by_bucket_path = OUT_AUDIT_DIR / f"14R_secondary_stack_incremental_by_bucket_{CELL14R_TIMESTAMP}.csv"
stack_secondary_by_tenor_path = OUT_AUDIT_DIR / f"14R_secondary_stack_incremental_by_tenor_{CELL14R_TIMESTAMP}.csv"
stack_secondary_by_year_path = OUT_AUDIT_DIR / f"14R_secondary_stack_incremental_by_year_{CELL14R_TIMESTAMP}.csv"

combined_program_summary_path = OUT_AUDIT_DIR / f"14R_combined_program_summary_{CELL14R_TIMESTAMP}.csv"
combined_program_by_year_path = OUT_AUDIT_DIR / f"14R_combined_program_by_year_{CELL14R_TIMESTAMP}.csv"

combined_continuity_summary_path = OUT_AUDIT_DIR / f"14R_combined_continuity_summary_{CELL14R_TIMESTAMP}.csv"
combined_continuity_by_year_path = OUT_AUDIT_DIR / f"14R_combined_continuity_by_year_{CELL14R_TIMESTAMP}.csv"

comparison_path = OUT_AUDIT_DIR / f"14R_candidate_comparison_program_view_{CELL14R_TIMESTAMP}.csv"
continuity_comparison_path = OUT_AUDIT_DIR / f"14R_candidate_comparison_continuity_view_{CELL14R_TIMESTAMP}.csv"

validation_path = OUT_AUDIT_DIR / f"14R_secondary_front_with_middle_back_shortlist_validation_{CELL14R_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"14R_secondary_front_with_middle_back_shortlist_manifest_{CELL14R_TIMESTAMP}.json"

secondary_front_grid.to_csv(front_grid_csv_path, index=False)
secondary_front_grid.to_parquet(front_grid_parquet_path, index=False)

secondary_middle_shortlist.to_csv(middle_shortlist_csv_path, index=False)
secondary_middle_shortlist.to_parquet(middle_shortlist_parquet_path, index=False)

secondary_back_shortlist.to_csv(back_shortlist_csv_path, index=False)
secondary_back_shortlist.to_parquet(back_shortlist_parquet_path, index=False)

secondary_stack_grid.to_csv(stack_grid_csv_path, index=False)
secondary_stack_grid.to_parquet(stack_grid_parquet_path, index=False)

pd.DataFrame({"trade_date": core_gate_dates}).to_csv(core_gate_dates_path, index=False)
pd.DataFrame({"trade_date": core_empty_dates}).to_csv(core_empty_dates_path, index=False)

front_selected.to_parquet(front_selected_panel_path, index=False)
front_selected.to_csv(front_selected_panel_csv_path, index=False)

middle_selected.to_parquet(middle_selected_panel_path, index=False)
middle_selected.to_csv(middle_selected_panel_csv_path, index=False)

back_selected.to_parquet(back_selected_panel_path, index=False)
back_selected.to_csv(back_selected_panel_csv_path, index=False)

stack_secondary_selected.to_parquet(stack_secondary_selected_panel_path, index=False)
stack_secondary_selected.to_csv(stack_secondary_selected_panel_csv_path, index=False)

combined_program_panel.to_parquet(combined_program_panel_path, index=False)
combined_program_panel.to_csv(combined_program_panel_csv_path, index=False)

combined_continuity_panel.to_parquet(combined_continuity_panel_path, index=False)
combined_continuity_panel.to_csv(combined_continuity_panel_csv_path, index=False)

core_program_summary.to_csv(core_program_summary_path, index=False)
core_continuity_summary.to_csv(core_continuity_summary_path, index=False)

front_incremental_summary.to_csv(front_incremental_summary_path, index=False)
front_incremental_by_tenor.to_csv(front_incremental_by_tenor_path, index=False)
front_incremental_by_year.to_csv(front_incremental_by_year_path, index=False)

middle_shortlist_summary.to_csv(middle_shortlist_summary_path, index=False)
back_shortlist_summary.to_csv(back_shortlist_summary_path, index=False)

stack_secondary_summary.to_csv(stack_secondary_summary_path, index=False)
stack_secondary_by_bucket.to_csv(stack_secondary_by_bucket_path, index=False)
stack_secondary_by_tenor.to_csv(stack_secondary_by_tenor_path, index=False)
stack_secondary_by_year.to_csv(stack_secondary_by_year_path, index=False)

combined_program_summary.to_csv(combined_program_summary_path, index=False)
combined_program_by_year.to_csv(combined_program_by_year_path, index=False)

combined_continuity_summary.to_csv(combined_continuity_summary_path, index=False)
combined_continuity_by_year.to_csv(combined_continuity_by_year_path, index=False)

comparison.to_csv(comparison_path, index=False)
continuity_comparison.to_csv(continuity_comparison_path, index=False)

# ======================================================================================
# 15. Validation
# ======================================================================================

validation_rows = []

required_base_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

missing_base_cols = [c for c in required_base_cols if c not in base.columns]
models_found = sorted(base["model_spec"].dropna().unique().tolist()) if "model_spec" in base.columns else []

bad_front_candidate_count = len(secondary_front_grid) != expected_front_candidate_count
bad_stack_candidate_count = len(secondary_stack_grid) != expected_stack_candidate_count

bad_front_z_equality = secondary_front_grid[
    secondary_front_grid["secondary_front_model_vrp_z_3m_min"].ne(
        secondary_front_grid["secondary_front_model_vrp_z_1y_min"]
    )
]

bad_middle_z_equality = secondary_middle_shortlist[
    secondary_middle_shortlist["secondary_middle_profile_id"].ne("NO_MIDDLE")
    & secondary_middle_shortlist["secondary_middle_model_vrp_z_3m_min"].ne(
        secondary_middle_shortlist["secondary_middle_model_vrp_z_1y_min"]
    )
]

bad_back_z_equality = secondary_back_shortlist[
    secondary_back_shortlist["secondary_back_profile_id"].ne("NO_BACK")
    & secondary_back_shortlist["secondary_back_model_vrp_z_3m_min"].ne(
        secondary_back_shortlist["secondary_back_model_vrp_z_1y_min"]
    )
]

bad_front_tenor_rows = front_selected[
    ~front_selected["tenor"].isin(SECONDARY_FRONT_TENORS)
] if not front_selected.empty else pd.DataFrame()

bad_middle_tenor_rows = middle_selected[
    ~middle_selected["tenor"].isin(SECONDARY_MIDDLE_TENORS)
] if not middle_selected.empty else pd.DataFrame()

bad_back_tenor_rows = back_selected[
    ~back_selected["tenor"].isin(SECONDARY_BACK_TENORS)
] if not back_selected.empty else pd.DataFrame()

bad_stack_tenor_rows = stack_secondary_selected[
    ~stack_secondary_selected["tenor"].isin(
        SECONDARY_FRONT_TENORS + SECONDARY_MIDDLE_TENORS + SECONDARY_BACK_TENORS
    )
] if not stack_secondary_selected.empty else pd.DataFrame()

bad_front_core_gate_dates = front_selected[
    front_selected["trade_date"].isin(core_gate_dates_set)
] if not front_selected.empty else pd.DataFrame()

bad_middle_core_gate_dates = middle_selected[
    middle_selected["trade_date"].isin(core_gate_dates_set)
] if not middle_selected.empty else pd.DataFrame()

bad_back_core_gate_dates = back_selected[
    back_selected["trade_date"].isin(core_gate_dates_set)
] if not back_selected.empty else pd.DataFrame()

bad_stack_core_gate_dates = stack_secondary_selected[
    stack_secondary_selected["trade_date"].isin(core_gate_dates_set)
] if not stack_secondary_selected.empty else pd.DataFrame()

stack_program_dupes = (
    combined_program_panel.groupby(["secondary_stack_candidate_id", "trade_date"])
    .size()
    .reset_index(name="rows")
)
bad_program_multi_trade_dates = stack_program_dupes[stack_program_dupes["rows"].gt(1)]

stack_secondary_dupes = (
    stack_secondary_selected.groupby(["secondary_stack_candidate_id", "trade_date"])
    .size()
    .reset_index(name="rows")
    if not stack_secondary_selected.empty
    else pd.DataFrame(columns=["secondary_stack_candidate_id", "trade_date", "rows"])
)
bad_stack_secondary_multi_trade_dates = stack_secondary_dupes[stack_secondary_dupes["rows"].gt(1)]

bad_forecast_front_flag = secondary_front_grid[
    secondary_front_grid["front_forecast_vol_pct_threshold_used"].astype(bool)
]

bad_forecast_middle_flag = secondary_middle_shortlist[
    secondary_middle_shortlist["middle_forecast_vol_pct_threshold_used"].astype(bool)
]

bad_forecast_back_flag = secondary_back_shortlist[
    secondary_back_shortlist["back_forecast_vol_pct_threshold_used"].astype(bool)
]

bad_rv21d_front_contract = secondary_front_grid[
    ~secondary_front_grid["front_uses_rv21d_threshold_contract"].astype(bool)
]

bad_rv21d_middle_contract = secondary_middle_shortlist[
    ~secondary_middle_shortlist["middle_uses_rv21d_threshold_contract"].astype(bool)
]

bad_rv21d_back_contract = secondary_back_shortlist[
    ~secondary_back_shortlist["back_uses_rv21d_threshold_contract"].astype(bool)
]

bad_normalized_pnl = base[
    base["normalized_pnl_dollars"].notna()
    & base["normalized_pnl_per_day"].notna()
    & (
        (
            base["normalized_pnl_dollars"] / base["tenor"].replace(0, np.nan)
        )
        - base["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
]

comparison_stack_candidates = sorted(comparison["secondary_stack_candidate_id"].dropna().unique().tolist()) if not comparison.empty else []
expected_stack_candidates = sorted(secondary_stack_grid["secondary_stack_candidate_id"].tolist())

add_validation(
    validation_rows,
    "base_panel_loaded",
    "PASS" if len(base) > 0 else "FAIL",
    f"rows={len(base):,}; path={base_panel_path}",
)

add_validation(
    validation_rows,
    "required_base_columns_available",
    "PASS" if not missing_base_cols else "FAIL",
    f"missing={missing_base_cols}",
)

add_validation(
    validation_rows,
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    validation_rows,
    "locked_core_rules_loaded",
    "PASS" if len(locked_core_rules_long) == 9 else "FAIL",
    f"rows={len(locked_core_rules_long):,}; path={locked_core_rules_long_path}",
)

add_validation(
    validation_rows,
    "core_gate_dates_created",
    "PASS" if len(core_gate_dates) > 0 and len(core_empty_dates) > 0 else "FAIL",
    f"core_gate_dates={len(core_gate_dates):,}; core_empty_dates={len(core_empty_dates):,}; total_dates={total_eligible_signal_dates:,}",
)

add_validation(
    validation_rows,
    "front_candidate_count",
    "PASS" if not bad_front_candidate_count else "FAIL",
    f"front_candidate_count={len(secondary_front_grid):,}; expected={expected_front_candidate_count:,}",
)

add_validation(
    validation_rows,
    "stack_candidate_count",
    "PASS" if not bad_stack_candidate_count else "FAIL",
    f"stack_candidate_count={len(secondary_stack_grid):,}; expected={expected_stack_candidate_count:,}",
)

add_validation(
    validation_rows,
    "front_tenors_only",
    "PASS" if bad_front_tenor_rows.empty else "FAIL",
    f"bad_rows={len(bad_front_tenor_rows):,}",
)

add_validation(
    validation_rows,
    "middle_shortlist_tenors_only",
    "PASS" if bad_middle_tenor_rows.empty else "FAIL",
    f"bad_rows={len(bad_middle_tenor_rows):,}",
)

add_validation(
    validation_rows,
    "back_shortlist_tenors_only",
    "PASS" if bad_back_tenor_rows.empty else "FAIL",
    f"bad_rows={len(bad_back_tenor_rows):,}",
)

add_validation(
    validation_rows,
    "stack_secondary_tenors_front_middle_or_back_only",
    "PASS" if bad_stack_tenor_rows.empty else "FAIL",
    f"bad_rows={len(bad_stack_tenor_rows):,}",
)

add_validation(
    validation_rows,
    "secondary_only_on_core_empty_dates",
    "PASS" if (
        bad_front_core_gate_dates.empty
        and bad_middle_core_gate_dates.empty
        and bad_back_core_gate_dates.empty
        and bad_stack_core_gate_dates.empty
    ) else "FAIL",
    (
        f"front_on_core_gate_rows={len(bad_front_core_gate_dates):,}; "
        f"middle_on_core_gate_rows={len(bad_middle_core_gate_dates):,}; "
        f"back_on_core_gate_rows={len(bad_back_core_gate_dates):,}; "
        f"stack_on_core_gate_rows={len(bad_stack_core_gate_dates):,}"
    ),
)

add_validation(
    validation_rows,
    "stack_secondary_one_trade_per_stack_date",
    "PASS" if bad_stack_secondary_multi_trade_dates.empty else "FAIL",
    f"bad_rows={len(bad_stack_secondary_multi_trade_dates):,}",
)

add_validation(
    validation_rows,
    "combined_program_one_trade_per_stack_date",
    "PASS" if bad_program_multi_trade_dates.empty else "FAIL",
    f"bad_rows={len(bad_program_multi_trade_dates):,}",
)

add_validation(
    validation_rows,
    "front_z3m_equals_z1y",
    "PASS" if bad_front_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_front_z_equality):,}",
)

add_validation(
    validation_rows,
    "middle_z3m_equals_z1y",
    "PASS" if bad_middle_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_middle_z_equality):,}",
)

add_validation(
    validation_rows,
    "back_z3m_equals_z1y",
    "PASS" if bad_back_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_back_z_equality):,}",
)

add_validation(
    validation_rows,
    "uses_rv21d_threshold_contract",
    "PASS" if (
        bad_rv21d_front_contract.empty
        and bad_rv21d_middle_contract.empty
        and bad_rv21d_back_contract.empty
    ) else "FAIL",
    (
        f"front_bad_rows={len(bad_rv21d_front_contract):,}; "
        f"middle_bad_rows={len(bad_rv21d_middle_contract):,}; "
        f"back_bad_rows={len(bad_rv21d_back_contract):,}"
    ),
)

add_validation(
    validation_rows,
    "forecast_vol_pct_diagnostic_only",
    "PASS" if (
        bad_forecast_front_flag.empty
        and bad_forecast_middle_flag.empty
        and bad_forecast_back_flag.empty
    ) else "FAIL",
    (
        f"front_bad_rows={len(bad_forecast_front_flag):,}; "
        f"middle_bad_rows={len(bad_forecast_middle_flag):,}; "
        f"back_bad_rows={len(bad_forecast_back_flag):,}"
    ),
)

add_validation(
    validation_rows,
    "all_stacks_compared_program_view",
    "PASS" if comparison_stack_candidates == expected_stack_candidates else "FAIL",
    f"comparison_stacks={len(comparison_stack_candidates):,}; expected={len(expected_stack_candidates):,}",
)

add_validation(
    validation_rows,
    "core_not_reopened",
    "PASS",
    "Locked Core rules loaded from Cell 11R and used only as gate/baseline.",
)

add_validation(
    validation_rows,
    "secondary_front_not_locked",
    "PASS",
    "Sweep only; no final Secondary Front lock.",
)

add_validation(
    validation_rows,
    "secondary_middle_not_locked",
    "PASS",
    "Middle shortlist used only as research input; no final Secondary Middle lock.",
)

add_validation(
    validation_rows,
    "secondary_back_not_locked",
    "PASS",
    "Back shortlist used only as research input; no final Secondary Back lock.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed or added.",
)

add_validation(
    validation_rows,
    "no_production_lock",
    "PASS",
    "No production/intraday implementation performed.",
)

add_validation(
    validation_rows,
    "normalized_pnl_per_day_formula",
    "PASS" if bad_normalized_pnl.empty else "FAIL",
    f"bad_rows={len(bad_normalized_pnl):,}",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"].eq("FAIL")]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1_RV21D_repaired_v2.ipynb",
    "cell": "Cell 14R — Secondary Front conservative sweep, Core-gated, with Back/Middle shortlist",
    "timestamp": CELL14R_TIMESTAMP,

    "locked_core_version": LOCKED_CORE_VERSION,
    "locked_core_decision_id": LOCKED_CORE_DECISION_ID,
    "secondary_front_sweep_version": SECONDARY_FRONT_SWEEP_VERSION,
    "secondary_stack_sweep_version": SECONDARY_STACK_SWEEP_VERSION,

    "base_panel_path": str(base_panel_path),
    "locked_core_rules_long_path": str(locked_core_rules_long_path),
    "locked_core_rules_wide_path": str(locked_core_rules_wide_path),

    "front_candidate_count": int(len(secondary_front_grid)),
    "expected_front_candidate_count": int(expected_front_candidate_count),
    "middle_shortlist_count": int(len(secondary_middle_shortlist)),
    "back_shortlist_count": int(len(secondary_back_shortlist)),
    "stack_candidate_count": int(len(secondary_stack_grid)),
    "expected_stack_candidate_count": int(expected_stack_candidate_count),

    "total_eligible_signal_dates": int(total_eligible_signal_dates),
    "core_gate_dates": int(len(core_gate_dates)),
    "core_empty_dates": int(len(core_empty_dates)),

    "targets": {
        "total_program_frequency_target": TOTAL_PROGRAM_FREQUENCY_TARGET,
        "combined_win_rate_target": COMBINED_WIN_RATE_TARGET,
        "secondary_win_rate_ideal": SECONDARY_WIN_RATE_IDEAL,
        "secondary_win_rate_floor": SECONDARY_WIN_RATE_FLOOR,
    },

    "secondary_front_parameter_grid": {
        "tenors": SECONDARY_FRONT_TENORS,
        "model_vrp_log": SECONDARY_FRONT_VRP_LOG_GRID,
        "model_vrp_z_3m": SECONDARY_FRONT_Z_GRID,
        "model_vrp_z_1y": SECONDARY_FRONT_Z_GRID,
        "RSI14_max": SECONDARY_FRONT_RSI_GRID,
        "rv21d_vol_pct_min": SECONDARY_FRONT_RV21D_GRID,
    },

    "secondary_middle_shortlist": secondary_middle_shortlist.to_dict(orient="records"),
    "secondary_back_shortlist": secondary_back_shortlist.to_dict(orient="records"),

    "selection_rules": {
        "core_gate": "If any locked Core tenor qualifies on a date, Secondary is ignored.",
        "secondary_test_dates": "Core-empty dates only.",
        "secondary_stack_selection": "At most one Secondary trade per stack/date across available Front, Middle, and Back profiles.",
        "program_view": "At most one trade per date: Core first, else one Secondary stack trade.",
        "continuity_view": "Cell 11R-style Core primary one-trade-per-bucket/date plus one Secondary stack trade on Core-empty dates.",
    },

    "front_grid_csv_path": str(front_grid_csv_path),
    "front_grid_parquet_path": str(front_grid_parquet_path),
    "middle_shortlist_csv_path": str(middle_shortlist_csv_path),
    "middle_shortlist_parquet_path": str(middle_shortlist_parquet_path),
    "back_shortlist_csv_path": str(back_shortlist_csv_path),
    "back_shortlist_parquet_path": str(back_shortlist_parquet_path),
    "stack_grid_csv_path": str(stack_grid_csv_path),
    "stack_grid_parquet_path": str(stack_grid_parquet_path),
    "core_gate_dates_path": str(core_gate_dates_path),
    "core_empty_dates_path": str(core_empty_dates_path),
    "front_selected_panel_path": str(front_selected_panel_path),
    "front_selected_panel_csv_path": str(front_selected_panel_csv_path),
    "middle_selected_panel_path": str(middle_selected_panel_path),
    "middle_selected_panel_csv_path": str(middle_selected_panel_csv_path),
    "back_selected_panel_path": str(back_selected_panel_path),
    "back_selected_panel_csv_path": str(back_selected_panel_csv_path),
    "stack_secondary_selected_panel_path": str(stack_secondary_selected_panel_path),
    "stack_secondary_selected_panel_csv_path": str(stack_secondary_selected_panel_csv_path),
    "combined_program_panel_path": str(combined_program_panel_path),
    "combined_program_panel_csv_path": str(combined_program_panel_csv_path),
    "combined_continuity_panel_path": str(combined_continuity_panel_path),
    "combined_continuity_panel_csv_path": str(combined_continuity_panel_csv_path),
    "core_program_summary_path": str(core_program_summary_path),
    "core_continuity_summary_path": str(core_continuity_summary_path),
    "front_incremental_summary_path": str(front_incremental_summary_path),
    "front_incremental_by_tenor_path": str(front_incremental_by_tenor_path),
    "front_incremental_by_year_path": str(front_incremental_by_year_path),
    "middle_shortlist_summary_path": str(middle_shortlist_summary_path),
    "back_shortlist_summary_path": str(back_shortlist_summary_path),
    "stack_secondary_summary_path": str(stack_secondary_summary_path),
    "stack_secondary_by_bucket_path": str(stack_secondary_by_bucket_path),
    "stack_secondary_by_tenor_path": str(stack_secondary_by_tenor_path),
    "stack_secondary_by_year_path": str(stack_secondary_by_year_path),
    "combined_program_summary_path": str(combined_program_summary_path),
    "combined_program_by_year_path": str(combined_program_by_year_path),
    "combined_continuity_summary_path": str(combined_continuity_summary_path),
    "combined_continuity_by_year_path": str(combined_continuity_by_year_path),
    "comparison_path": str(comparison_path),
    "continuity_comparison_path": str(continuity_comparison_path),
    "validation_path": str(validation_path),
    "manifest_path": str(manifest_path),

    "constraints": [
        "Core remains locked and unchanged.",
        "Secondary Front is swept conservatively.",
        "Secondary Middle is carried only as shortlist input.",
        "Secondary Back is carried only as shortlist input.",
        "No Secondary lock.",
        "9D remains excluded.",
        "18D remains in Front for Core map.",
        "Front tenors are 12D/15D/18D.",
        "Middle shortlist tenors are 21D/24D.",
        "Back shortlist tenors are 27D/30D/33D.",
        "z3m equals z1y in this sweep.",
        "RV21D threshold contract is rv21d_vol_pct > rv21d_vol_pct_min.",
        "forecast_vol_pct is diagnostic only.",
        "No sizing changes.",
        "No production/intraday implementation.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 14R validation failed. Review validation output above.")

# ======================================================================================
# 16. Display review
# ======================================================================================

print("=" * 100)
print("Cell 14R — Secondary Front conservative sweep, Core-gated, with Back/Middle shortlist")
print("=" * 100)
print(f"Base panel:                         {base_panel_path}")
print(f"Locked Core rules:                  {locked_core_rules_long_path}")
print(f"Front candidate count:              {len(secondary_front_grid):,}")
print(f"Middle shortlist count:             {len(secondary_middle_shortlist):,}")
print(f"Back shortlist count:               {len(secondary_back_shortlist):,}")
print(f"Stack candidate count:              {len(secondary_stack_grid):,}")
print(f"Total eligible signal dates:        {total_eligible_signal_dates:,}")
print(f"Core gate dates:                    {len(core_gate_dates):,}")
print(f"Core-empty dates:                   {len(core_empty_dates):,}")
print(f"Core program selected trades:       {len(core_program):,}")
print(f"Core continuity selected trades:    {len(core_primary_continuity):,}")
print(f"Front selected panel rows:          {len(front_selected):,}")
print(f"Middle selected panel rows:         {len(middle_selected):,}")
print(f"Back selected panel rows:           {len(back_selected):,}")
print(f"Stack Secondary selected rows:      {len(stack_secondary_selected):,}")
print(f"Combined program panel rows:        {len(combined_program_panel):,}")
print(f"Combined continuity panel rows:     {len(combined_continuity_panel):,}")
print("Secondary Front tenors:             12D / 15D / 18D")
print("Middle shortlist profiles:          NO_MIDDLE, MIDDLE_FREQ_0011, MIDDLE_CLEAN_0252")
print("Back shortlist profiles:            NO_BACK, BACK_FREQ_0009, BACK_CLEAN_0133")
print("Core gate:                          If Core qualifies, Secondary ignored")
print("Program view:                       one trade per date")
print("Continuity view:                    Core primary + one Secondary on Core-empty dates")
print(f"Frequency target:                   {TOTAL_PROGRAM_FREQUENCY_TARGET:.2%}")
print(f"Combined win-rate target:           {COMBINED_WIN_RATE_TARGET:.2%}")
print(f"Secondary win-rate ideal:           {SECONDARY_WIN_RATE_IDEAL:.2%}")
print("Secondary Front locked here:        False")
print("Secondary Middle locked here:       False")
print("Secondary Back locked here:         False")
print("Sizing locked here:                 False")
print("Production locked here:             False")

print()
print("Validation:")
display(validation)

print()
print("Secondary Front conservative grid:")
display(secondary_front_grid)

print()
print("Secondary Middle shortlist:")
display(secondary_middle_shortlist)

print()
print("Secondary Back shortlist:")
display(secondary_back_shortlist)

print()
print("Secondary stack grid:")
display(secondary_stack_grid)

print()
print("Core program baseline — one trade per date:")
display(core_program_summary)

print()
print("Core continuity baseline — one trade per bucket/date:")
display(core_continuity_summary)

comparison_display_cols = [
    "secondary_stack_candidate_id",
    "secondary_front_candidate_id",
    "secondary_middle_profile_id",
    "secondary_middle_source_candidate_id",
    "secondary_back_profile_id",
    "secondary_back_source_candidate_id",

    "secondary_front_model_vrp_log_min",
    "secondary_front_model_vrp_z_min",
    "secondary_front_RSI14_max",
    "secondary_front_rv21d_vol_pct_min",

    "secondary_middle_model_vrp_log_min",
    "secondary_middle_model_vrp_z_min",
    "secondary_middle_RSI14_max",
    "secondary_middle_rv21d_vol_pct_min",

    "secondary_back_model_vrp_log_min",
    "secondary_back_model_vrp_z_min",
    "secondary_back_RSI14_max",
    "secondary_back_rv21d_vol_pct_min",

    "trades",
    "unique_trade_dates",
    "signal_date_frequency",
    "win_rate",
    "total_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "profit_factor_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",

    "secondary_trades",
    "secondary_unique_trade_dates",
    "secondary_signal_date_frequency",
    "secondary_win_rate",
    "secondary_total_pnl",
    "secondary_avg_pnl_per_day",
    "secondary_profit_factor",
    "secondary_profit_factor_pnl_per_day",
    "secondary_pnl_per_day_drawdown",
    "secondary_worst_20_trade_pnl_per_day_sum",
    "secondary_avg_pnl_per_day_2025",
    "secondary_avg_pnl_per_day_2026",

    "incremental_front_trade_count",
    "incremental_middle_trade_count",
    "incremental_back_trade_count",
    "incremental_secondary_trade_count",

    "delta_program_unique_trade_dates_vs_core_only",
    "delta_program_signal_date_frequency_vs_core_only",
    "delta_program_win_rate_vs_core_only",
    "delta_program_total_pnl_vs_core_only",
    "delta_program_avg_pnl_per_day_vs_core_only",
    "delta_program_profit_factor_vs_core_only",
    "delta_program_profit_factor_pnl_per_day_vs_core_only",
    "delta_program_pnl_per_day_drawdown_vs_core_only",
    "delta_program_worst_20_trade_pnl_per_day_sum_vs_core_only",
    "delta_program_avg_pnl_per_day_2026_vs_core_only",

    "passes_frequency_target_25pct",
    "passes_combined_win_rate_target_80pct",
    "passes_secondary_win_rate_ideal_80pct",
    "passes_secondary_win_rate_floor_78pct",
    "passes_program_targets",
    "passes_full_ideal_targets",
    "passes_acceptable_secondary_floor",
]

print()
print("Candidate comparison — PROGRAM VIEW, Core first else Secondary stack:")
display(comparison[[c for c in comparison_display_cols if c in comparison.columns]])

continuity_display_cols = [
    "secondary_stack_candidate_id",
    "secondary_front_candidate_id",
    "secondary_middle_profile_id",
    "secondary_middle_source_candidate_id",
    "secondary_back_profile_id",
    "secondary_back_source_candidate_id",

    "secondary_front_model_vrp_log_min",
    "secondary_front_model_vrp_z_min",
    "secondary_front_RSI14_max",
    "secondary_front_rv21d_vol_pct_min",

    "secondary_middle_model_vrp_log_min",
    "secondary_middle_model_vrp_z_min",
    "secondary_middle_RSI14_max",
    "secondary_middle_rv21d_vol_pct_min",

    "secondary_back_model_vrp_log_min",
    "secondary_back_model_vrp_z_min",
    "secondary_back_RSI14_max",
    "secondary_back_rv21d_vol_pct_min",

    "trades",
    "unique_trade_dates",
    "signal_date_frequency",
    "win_rate",
    "total_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "profit_factor_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",

    "core_trade_count",
    "secondary_front_trade_count",
    "secondary_middle_trade_count",
    "secondary_back_trade_count",
    "secondary_trade_count",

    "delta_continuity_trades_vs_core_primary",
    "delta_continuity_unique_trade_dates_vs_core_primary",
    "delta_continuity_signal_date_frequency_vs_core_primary",
    "delta_continuity_win_rate_vs_core_primary",
    "delta_continuity_total_pnl_vs_core_primary",
    "delta_continuity_avg_pnl_per_day_vs_core_primary",
    "delta_continuity_profit_factor_vs_core_primary",
    "delta_continuity_profit_factor_pnl_per_day_vs_core_primary",
    "delta_continuity_pnl_per_day_drawdown_vs_core_primary",
    "delta_continuity_worst_20_trade_pnl_per_day_sum_vs_core_primary",
    "delta_continuity_avg_pnl_per_day_2026_vs_core_primary",
]

print()
print("Candidate comparison — CONTINUITY VIEW, Core primary + Secondary stack:")
display(continuity_comparison[[c for c in continuity_display_cols if c in continuity_comparison.columns]])

print()
print("Secondary Front incremental summary:")
display(front_incremental_summary)

print()
print("Secondary Front incremental by tenor:")
display(front_incremental_by_tenor)

print()
print("Secondary Front incremental by year:")
display(front_incremental_by_year)

print()
print("Secondary Middle shortlist summary:")
display(middle_shortlist_summary)

print()
print("Secondary Back shortlist summary:")
display(back_shortlist_summary)

print()
print("Stack Secondary incremental summary:")
display(stack_secondary_summary)

print()
print("Stack Secondary incremental by bucket:")
display(stack_secondary_by_bucket)

print()
print("Stack Secondary incremental by tenor:")
display(stack_secondary_by_tenor)

print()
print("Stack Secondary incremental by year:")
display(stack_secondary_by_year)

print()
print("Combined program by year:")
display(combined_program_by_year)

print()
print("Combined continuity by year:")
display(combined_continuity_by_year)

print()
print("Saved outputs:")
for p in [
    front_grid_csv_path,
    front_grid_parquet_path,
    middle_shortlist_csv_path,
    middle_shortlist_parquet_path,
    back_shortlist_csv_path,
    back_shortlist_parquet_path,
    stack_grid_csv_path,
    stack_grid_parquet_path,
    core_gate_dates_path,
    core_empty_dates_path,
    front_selected_panel_path,
    front_selected_panel_csv_path,
    middle_selected_panel_path,
    middle_selected_panel_csv_path,
    back_selected_panel_path,
    back_selected_panel_csv_path,
    stack_secondary_selected_panel_path,
    stack_secondary_selected_panel_csv_path,
    combined_program_panel_path,
    combined_program_panel_csv_path,
    combined_continuity_panel_path,
    combined_continuity_panel_csv_path,
    core_program_summary_path,
    core_continuity_summary_path,
    front_incremental_summary_path,
    front_incremental_by_tenor_path,
    front_incremental_by_year_path,
    middle_shortlist_summary_path,
    back_shortlist_summary_path,
    stack_secondary_summary_path,
    stack_secondary_by_bucket_path,
    stack_secondary_by_tenor_path,
    stack_secondary_by_year_path,
    combined_program_summary_path,
    combined_program_by_year_path,
    combined_continuity_summary_path,
    combined_continuity_by_year_path,
    comparison_path,
    continuity_comparison_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 14R PASSED — SECONDARY FRONT CONSERVATIVE CORE-GATED SWEEP COMPLETE")

In [ ]:
# ======================================================================================
# Cell 14R-FIX — Repair combined program stack assignment
# ======================================================================================
#
# Why this fix exists:
#   Cell 14R correctly built the Front/Middle/Back candidate panels and the stack-secondary
#   selected panel, but the combined program / continuity panels incorrectly cross-merged
#   Core rows while Core still had CORE_ONLY_BASELINE stack metadata.
#
#   Result:
#       PROGRAM VIEW rows represented Secondary-only performance, not Core + Secondary.
#       A giant NaN aggregate row appeared from duplicated Core rows.
#
# This repair:
#   1. Drops stale stack metadata from Core baseline rows.
#   2. Cross-joins clean Core rows to every stack id.
#   3. Rebuilds combined program and continuity panels.
#   4. Recomputes combined summaries and comparisons.
#
# This repair does NOT:
#   1. Change Core.
#   2. Change Secondary candidate rules.
#   3. Change stack-secondary selected trades.
#   4. Change sizing.
#   5. Lock anything.
# ======================================================================================

from datetime import datetime
import json
import numpy as np
import pandas as pd

CELL14R_FIX_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# --------------------------------------------------------------------------------------
# 1. Rebuild clean stack map
# --------------------------------------------------------------------------------------

stack_key_cols = [
    "secondary_stack_candidate_id",
    "secondary_front_candidate_id",
    "secondary_middle_profile_id",
    "secondary_back_profile_id",
    "secondary_middle_source_candidate_id",
    "secondary_back_source_candidate_id",
]

stack_map = secondary_stack_grid[stack_key_cols].copy()

expected_stack_count = int(len(secondary_stack_grid))

# --------------------------------------------------------------------------------------
# 2. Remove stale stack metadata from Core baseline rows before cross-joining
# --------------------------------------------------------------------------------------

stale_stack_cols = [
    "secondary_stack_candidate_id",
    "secondary_front_candidate_id",
    "secondary_middle_profile_id",
    "secondary_back_profile_id",
    "secondary_middle_source_candidate_id",
    "secondary_back_source_candidate_id",
]

core_program_clean = core_program.copy()
core_continuity_clean = core_primary_continuity.copy()

core_program_clean = core_program_clean.drop(
    columns=[c for c in stale_stack_cols if c in core_program_clean.columns],
    errors="ignore",
)

core_continuity_clean = core_continuity_clean.drop(
    columns=[c for c in stale_stack_cols if c in core_continuity_clean.columns],
    errors="ignore",
)

# --------------------------------------------------------------------------------------
# 3. Cross-join clean Core baseline rows to every stack candidate
# --------------------------------------------------------------------------------------

core_program_stack_fixed = core_program_clean.merge(
    stack_map,
    how="cross",
)

core_program_stack_fixed["selection_universe"] = COMBINED_PROGRAM_UNIVERSE
core_program_stack_fixed["source_layer"] = "Core"
core_program_stack_fixed["program_leg"] = "Core"

core_continuity_stack_fixed = core_continuity_clean.merge(
    stack_map,
    how="cross",
)

core_continuity_stack_fixed["selection_universe"] = COMBINED_CONTINUITY_UNIVERSE
core_continuity_stack_fixed["source_layer"] = "Core"
core_continuity_stack_fixed["program_leg"] = "Core"

# --------------------------------------------------------------------------------------
# 4. Rebuild combined panels
# --------------------------------------------------------------------------------------

stack_secondary_selected_fixed = stack_secondary_selected.copy()
stack_secondary_selected_fixed["selection_universe"] = STACK_SECONDARY_INCREMENTAL_UNIVERSE
stack_secondary_selected_fixed["source_layer"] = "Secondary"

combined_program_panel_fixed = pd.concat(
    [
        core_program_stack_fixed,
        stack_secondary_selected_fixed.assign(selection_universe=COMBINED_PROGRAM_UNIVERSE),
    ],
    ignore_index=True,
)

combined_continuity_panel_fixed = pd.concat(
    [
        core_continuity_stack_fixed,
        stack_secondary_selected_fixed.assign(selection_universe=COMBINED_CONTINUITY_UNIVERSE),
    ],
    ignore_index=True,
)

# --------------------------------------------------------------------------------------
# 5. Recompute summaries
# --------------------------------------------------------------------------------------

core_program_summary_fixed = summarize_trade_set(
    core_program.assign(selection_universe=CORE_PROGRAM_UNIVERSE),
    group_cols=["selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

core_continuity_summary_fixed = summarize_trade_set(
    core_primary_continuity.assign(selection_universe=PRIMARY_CORE_CONTINUITY_UNIVERSE),
    group_cols=["selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

stack_secondary_summary_fixed = summarize_trade_set(
    stack_secondary_selected_fixed,
    group_cols=["secondary_stack_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

stack_secondary_summary_fixed = attach_stack_grid(
    stack_secondary_summary_fixed,
    secondary_stack_grid,
)

stack_secondary_by_bucket_fixed = summarize_trade_set(
    stack_secondary_selected_fixed,
    group_cols=[
        "secondary_stack_candidate_id",
        "selection_universe",
        "effective_tenor_bucket",
        "effective_tenor_bucket_order",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

stack_secondary_by_bucket_fixed = attach_stack_grid(
    stack_secondary_by_bucket_fixed,
    secondary_stack_grid,
)

stack_secondary_by_tenor_fixed = summarize_trade_set(
    stack_secondary_selected_fixed,
    group_cols=[
        "secondary_stack_candidate_id",
        "selection_universe",
        "tenor",
        "effective_tenor_bucket",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

stack_secondary_by_tenor_fixed = attach_stack_grid(
    stack_secondary_by_tenor_fixed,
    secondary_stack_grid,
)

stack_secondary_by_year_input_fixed = (
    stack_secondary_selected_fixed.assign(year=stack_secondary_selected_fixed["trade_date"].dt.year.astype(int))
    if not stack_secondary_selected_fixed.empty
    else stack_secondary_selected_fixed.copy()
)

stack_secondary_by_year_fixed = summarize_trade_set(
    stack_secondary_by_year_input_fixed,
    group_cols=["secondary_stack_candidate_id", "selection_universe", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)

stack_secondary_by_year_fixed = attach_stack_grid(
    stack_secondary_by_year_fixed,
    secondary_stack_grid,
)

combined_program_summary_fixed = summarize_trade_set(
    combined_program_panel_fixed,
    group_cols=["secondary_stack_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

combined_program_summary_fixed = attach_stack_grid(
    combined_program_summary_fixed,
    secondary_stack_grid,
)

combined_continuity_summary_fixed = summarize_trade_set(
    combined_continuity_panel_fixed,
    group_cols=["secondary_stack_candidate_id", "selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

combined_continuity_summary_fixed = attach_stack_grid(
    combined_continuity_summary_fixed,
    secondary_stack_grid,
)

combined_program_by_year_input_fixed = combined_program_panel_fixed.assign(
    year=combined_program_panel_fixed["trade_date"].dt.year.astype(int)
)

combined_program_by_year_fixed = summarize_trade_set(
    combined_program_by_year_input_fixed,
    group_cols=["secondary_stack_candidate_id", "selection_universe", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)

combined_program_by_year_fixed = attach_stack_grid(
    combined_program_by_year_fixed,
    secondary_stack_grid,
)

combined_continuity_by_year_input_fixed = combined_continuity_panel_fixed.assign(
    year=combined_continuity_panel_fixed["trade_date"].dt.year.astype(int)
)

combined_continuity_by_year_fixed = summarize_trade_set(
    combined_continuity_by_year_input_fixed,
    group_cols=["secondary_stack_candidate_id", "selection_universe", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)

combined_continuity_by_year_fixed = attach_stack_grid(
    combined_continuity_by_year_fixed,
    secondary_stack_grid,
)

# --------------------------------------------------------------------------------------
# 6. Rebuild program comparison
# --------------------------------------------------------------------------------------

core_program_ref = core_program_summary_fixed.iloc[0].to_dict()
core_continuity_ref = core_continuity_summary_fixed.iloc[0].to_dict()

comparison_fixed = combined_program_summary_fixed.copy()

sec_cols = [
    "secondary_stack_candidate_id",
    "trades",
    "unique_trade_dates",
    "signal_date_frequency",
    "win_rate",
    "total_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "profit_factor_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
    "secondary_front_trade_count",
    "secondary_middle_trade_count",
    "secondary_back_trade_count",
    "secondary_trade_count",
]

sec_metrics = stack_secondary_summary_fixed[
    [c for c in sec_cols if c in stack_secondary_summary_fixed.columns]
].copy()

sec_metrics = sec_metrics.rename(columns={
    "trades": "secondary_trades",
    "unique_trade_dates": "secondary_unique_trade_dates",
    "signal_date_frequency": "secondary_signal_date_frequency",
    "win_rate": "secondary_win_rate",
    "total_pnl": "secondary_total_pnl",
    "avg_pnl_per_day": "secondary_avg_pnl_per_day",
    "profit_factor": "secondary_profit_factor",
    "profit_factor_pnl_per_day": "secondary_profit_factor_pnl_per_day",
    "pnl_per_day_drawdown": "secondary_pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum": "secondary_worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025": "secondary_avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026": "secondary_avg_pnl_per_day_2026",
    "secondary_front_trade_count": "incremental_front_trade_count",
    "secondary_middle_trade_count": "incremental_middle_trade_count",
    "secondary_back_trade_count": "incremental_back_trade_count",
    "secondary_trade_count": "incremental_secondary_trade_count",
})

comparison_fixed = comparison_fixed.merge(
    sec_metrics,
    on="secondary_stack_candidate_id",
    how="left",
    validate="one_to_one",
)

for metric in [
    "trades",
    "unique_trade_dates",
    "signal_date_frequency",
    "win_rate",
    "total_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "profit_factor_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
]:
    if metric in comparison_fixed.columns and metric in core_program_ref:
        comparison_fixed[f"delta_program_{metric}_vs_core_only"] = (
            comparison_fixed[metric] - core_program_ref[metric]
        )

comparison_fixed["passes_frequency_target_25pct"] = (
    comparison_fixed["signal_date_frequency"] >= TOTAL_PROGRAM_FREQUENCY_TARGET
)

comparison_fixed["passes_combined_win_rate_target_80pct"] = (
    comparison_fixed["win_rate"] >= COMBINED_WIN_RATE_TARGET
)

comparison_fixed["passes_secondary_win_rate_ideal_80pct"] = (
    comparison_fixed["secondary_win_rate"] >= SECONDARY_WIN_RATE_IDEAL
)

comparison_fixed["passes_secondary_win_rate_floor_78pct"] = (
    comparison_fixed["secondary_win_rate"] >= SECONDARY_WIN_RATE_FLOOR
)

comparison_fixed["passes_program_targets"] = (
    comparison_fixed["passes_frequency_target_25pct"]
    & comparison_fixed["passes_combined_win_rate_target_80pct"]
)

comparison_fixed["passes_full_ideal_targets"] = (
    comparison_fixed["passes_program_targets"]
    & comparison_fixed["passes_secondary_win_rate_ideal_80pct"]
)

comparison_fixed["passes_acceptable_secondary_floor"] = (
    comparison_fixed["passes_program_targets"]
    & comparison_fixed["passes_secondary_win_rate_floor_78pct"]
)

comparison_fixed = comparison_fixed.sort_values(
    [
        "passes_full_ideal_targets",
        "passes_acceptable_secondary_floor",
        "passes_program_targets",
        "secondary_win_rate",
        "signal_date_frequency",
        "win_rate",
        "avg_pnl_per_day",
        "profit_factor_pnl_per_day",
        "avg_pnl_per_day_2026",
    ],
    ascending=[False, False, False, False, False, False, False, False, False],
).reset_index(drop=True)

# --------------------------------------------------------------------------------------
# 7. Rebuild continuity comparison
# --------------------------------------------------------------------------------------

continuity_comparison_fixed = combined_continuity_summary_fixed.copy()

for metric in [
    "trades",
    "unique_trade_dates",
    "signal_date_frequency",
    "win_rate",
    "total_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "profit_factor_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
]:
    if metric in continuity_comparison_fixed.columns and metric in core_continuity_ref:
        continuity_comparison_fixed[f"delta_continuity_{metric}_vs_core_primary"] = (
            continuity_comparison_fixed[metric] - core_continuity_ref[metric]
        )

continuity_comparison_fixed = continuity_comparison_fixed.sort_values(
    [
        "signal_date_frequency",
        "win_rate",
        "avg_pnl_per_day",
        "profit_factor_pnl_per_day",
        "avg_pnl_per_day_2026",
    ],
    ascending=[False, False, False, False, False],
).reset_index(drop=True)

# --------------------------------------------------------------------------------------
# 8. Validation
# --------------------------------------------------------------------------------------

validation_rows = []

def add_fix_validation(check, status, detail):
    validation_rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })

program_stack_count = int(comparison_fixed["secondary_stack_candidate_id"].nunique())
continuity_stack_count = int(continuity_comparison_fixed["secondary_stack_candidate_id"].nunique())

program_dupes_fixed = (
    combined_program_panel_fixed
    .groupby(["secondary_stack_candidate_id", "trade_date"])
    .size()
    .reset_index(name="rows")
)

bad_program_dupes_fixed = program_dupes_fixed[program_dupes_fixed["rows"].gt(1)]

secondary_dupes_fixed = (
    stack_secondary_selected_fixed
    .groupby(["secondary_stack_candidate_id", "trade_date"])
    .size()
    .reset_index(name="rows")
)

bad_secondary_dupes_fixed = secondary_dupes_fixed[secondary_dupes_fixed["rows"].gt(1)]

bad_program_below_core_trade_count = comparison_fixed[
    comparison_fixed["trades"] < int(core_program_summary_fixed["trades"].iloc[0])
]

bad_program_below_core_dates = comparison_fixed[
    comparison_fixed["unique_trade_dates"] < int(core_program_summary_fixed["unique_trade_dates"].iloc[0])
]

bad_nan_stack_ids_program = comparison_fixed[
    comparison_fixed["secondary_stack_candidate_id"].isna()
]

bad_nan_stack_ids_continuity = continuity_comparison_fixed[
    continuity_comparison_fixed["secondary_stack_candidate_id"].isna()
]

add_fix_validation(
    "program_stack_count",
    "PASS" if program_stack_count == expected_stack_count else "FAIL",
    f"program_stack_count={program_stack_count:,}; expected={expected_stack_count:,}",
)

add_fix_validation(
    "continuity_stack_count",
    "PASS" if continuity_stack_count == expected_stack_count else "FAIL",
    f"continuity_stack_count={continuity_stack_count:,}; expected={expected_stack_count:,}",
)

add_fix_validation(
    "combined_program_one_trade_per_stack_date",
    "PASS" if bad_program_dupes_fixed.empty else "FAIL",
    f"bad_rows={len(bad_program_dupes_fixed):,}",
)

add_fix_validation(
    "stack_secondary_one_trade_per_stack_date",
    "PASS" if bad_secondary_dupes_fixed.empty else "FAIL",
    f"bad_rows={len(bad_secondary_dupes_fixed):,}",
)

add_fix_validation(
    "program_candidates_not_below_core_trade_count",
    "PASS" if bad_program_below_core_trade_count.empty else "FAIL",
    f"bad_rows={len(bad_program_below_core_trade_count):,}",
)

add_fix_validation(
    "program_candidates_not_below_core_date_count",
    "PASS" if bad_program_below_core_dates.empty else "FAIL",
    f"bad_rows={len(bad_program_below_core_dates):,}",
)

add_fix_validation(
    "no_nan_stack_ids_program",
    "PASS" if bad_nan_stack_ids_program.empty else "FAIL",
    f"bad_rows={len(bad_nan_stack_ids_program):,}",
)

add_fix_validation(
    "no_nan_stack_ids_continuity",
    "PASS" if bad_nan_stack_ids_continuity.empty else "FAIL",
    f"bad_rows={len(bad_nan_stack_ids_continuity):,}",
)

add_fix_validation(
    "core_not_reopened",
    "PASS",
    "Repair only rebuilt combined panels from existing Core and stack-secondary selected rows.",
)

add_fix_validation(
    "no_selection_logic_change",
    "PASS",
    "No candidate rules, stack-secondary selection, or thresholds changed.",
)

add_fix_validation(
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed.",
)

add_fix_validation(
    "no_lock",
    "PASS",
    "No Secondary lock, sizing lock, or production lock.",
)

validation_fixed = pd.DataFrame(validation_rows)
failures_fixed = validation_fixed[validation_fixed["status"].eq("FAIL")]

# --------------------------------------------------------------------------------------
# 9. Save repaired outputs
# --------------------------------------------------------------------------------------

combined_program_panel_fixed_path = OUT_PROCESSED_DIR / f"14R_FIX_combined_program_core_first_else_secondary_stack_panel_{CELL14R_FIX_TIMESTAMP}.parquet"
combined_program_panel_fixed_csv_path = OUT_AUDIT_DIR / f"14R_FIX_combined_program_core_first_else_secondary_stack_panel_{CELL14R_FIX_TIMESTAMP}.csv"

combined_continuity_panel_fixed_path = OUT_PROCESSED_DIR / f"14R_FIX_combined_continuity_core_primary_plus_secondary_stack_panel_{CELL14R_FIX_TIMESTAMP}.parquet"
combined_continuity_panel_fixed_csv_path = OUT_AUDIT_DIR / f"14R_FIX_combined_continuity_core_primary_plus_secondary_stack_panel_{CELL14R_FIX_TIMESTAMP}.csv"

combined_program_summary_fixed_path = OUT_AUDIT_DIR / f"14R_FIX_combined_program_summary_{CELL14R_FIX_TIMESTAMP}.csv"
combined_program_by_year_fixed_path = OUT_AUDIT_DIR / f"14R_FIX_combined_program_by_year_{CELL14R_FIX_TIMESTAMP}.csv"

combined_continuity_summary_fixed_path = OUT_AUDIT_DIR / f"14R_FIX_combined_continuity_summary_{CELL14R_FIX_TIMESTAMP}.csv"
combined_continuity_by_year_fixed_path = OUT_AUDIT_DIR / f"14R_FIX_combined_continuity_by_year_{CELL14R_FIX_TIMESTAMP}.csv"

comparison_fixed_path = OUT_AUDIT_DIR / f"14R_FIX_candidate_comparison_program_view_{CELL14R_FIX_TIMESTAMP}.csv"
continuity_comparison_fixed_path = OUT_AUDIT_DIR / f"14R_FIX_candidate_comparison_continuity_view_{CELL14R_FIX_TIMESTAMP}.csv"

stack_secondary_summary_fixed_path = OUT_AUDIT_DIR / f"14R_FIX_secondary_stack_incremental_summary_{CELL14R_FIX_TIMESTAMP}.csv"
stack_secondary_by_bucket_fixed_path = OUT_AUDIT_DIR / f"14R_FIX_secondary_stack_incremental_by_bucket_{CELL14R_FIX_TIMESTAMP}.csv"
stack_secondary_by_tenor_fixed_path = OUT_AUDIT_DIR / f"14R_FIX_secondary_stack_incremental_by_tenor_{CELL14R_FIX_TIMESTAMP}.csv"
stack_secondary_by_year_fixed_path = OUT_AUDIT_DIR / f"14R_FIX_secondary_stack_incremental_by_year_{CELL14R_FIX_TIMESTAMP}.csv"

validation_fixed_path = OUT_AUDIT_DIR / f"14R_FIX_validation_{CELL14R_FIX_TIMESTAMP}.csv"
manifest_fixed_path = OUT_AUDIT_DIR / f"14R_FIX_manifest_{CELL14R_FIX_TIMESTAMP}.json"

combined_program_panel_fixed.to_parquet(combined_program_panel_fixed_path, index=False)
combined_program_panel_fixed.to_csv(combined_program_panel_fixed_csv_path, index=False)

combined_continuity_panel_fixed.to_parquet(combined_continuity_panel_fixed_path, index=False)
combined_continuity_panel_fixed.to_csv(combined_continuity_panel_fixed_csv_path, index=False)

combined_program_summary_fixed.to_csv(combined_program_summary_fixed_path, index=False)
combined_program_by_year_fixed.to_csv(combined_program_by_year_fixed_path, index=False)

combined_continuity_summary_fixed.to_csv(combined_continuity_summary_fixed_path, index=False)
combined_continuity_by_year_fixed.to_csv(combined_continuity_by_year_fixed_path, index=False)

comparison_fixed.to_csv(comparison_fixed_path, index=False)
continuity_comparison_fixed.to_csv(continuity_comparison_fixed_path, index=False)

stack_secondary_summary_fixed.to_csv(stack_secondary_summary_fixed_path, index=False)
stack_secondary_by_bucket_fixed.to_csv(stack_secondary_by_bucket_fixed_path, index=False)
stack_secondary_by_tenor_fixed.to_csv(stack_secondary_by_tenor_fixed_path, index=False)
stack_secondary_by_year_fixed.to_csv(stack_secondary_by_year_fixed_path, index=False)

validation_fixed.to_csv(validation_fixed_path, index=False)

manifest_fixed = {
    "cell": "Cell 14R-FIX — Repair combined program stack assignment",
    "timestamp": CELL14R_FIX_TIMESTAMP,
    "repair_scope": "Rebuild combined program and continuity panels with correct stack ids assigned to Core rows.",
    "core_changed": False,
    "secondary_rules_changed": False,
    "selection_logic_changed": False,
    "sizing_changed": False,
    "lock_created": False,
    "expected_stack_count": expected_stack_count,
    "program_stack_count": program_stack_count,
    "continuity_stack_count": continuity_stack_count,
    "combined_program_panel_fixed_path": str(combined_program_panel_fixed_path),
    "combined_program_panel_fixed_csv_path": str(combined_program_panel_fixed_csv_path),
    "combined_continuity_panel_fixed_path": str(combined_continuity_panel_fixed_path),
    "combined_continuity_panel_fixed_csv_path": str(combined_continuity_panel_fixed_csv_path),
    "combined_program_summary_fixed_path": str(combined_program_summary_fixed_path),
    "combined_program_by_year_fixed_path": str(combined_program_by_year_fixed_path),
    "combined_continuity_summary_fixed_path": str(combined_continuity_summary_fixed_path),
    "combined_continuity_by_year_fixed_path": str(combined_continuity_by_year_fixed_path),
    "comparison_fixed_path": str(comparison_fixed_path),
    "continuity_comparison_fixed_path": str(continuity_comparison_fixed_path),
    "stack_secondary_summary_fixed_path": str(stack_secondary_summary_fixed_path),
    "stack_secondary_by_bucket_fixed_path": str(stack_secondary_by_bucket_fixed_path),
    "stack_secondary_by_tenor_fixed_path": str(stack_secondary_by_tenor_fixed_path),
    "stack_secondary_by_year_fixed_path": str(stack_secondary_by_year_fixed_path),
    "validation_fixed_path": str(validation_fixed_path),
    "manifest_fixed_path": str(manifest_fixed_path),
}

with open(manifest_fixed_path, "w", encoding="utf-8") as f:
    json.dump(manifest_fixed, f, indent=2, default=str)

if not failures_fixed.empty:
    display(validation_fixed)
    raise RuntimeError("Cell 14R-FIX validation failed. Review validation output above.")

# --------------------------------------------------------------------------------------
# 10. Display repaired review
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Cell 14R-FIX — Repaired combined program / continuity stack assignment")
print("=" * 100)
print(f"Expected stack count:              {expected_stack_count:,}")
print(f"Program comparison stack count:    {program_stack_count:,}")
print(f"Continuity comparison stack count: {continuity_stack_count:,}")
print(f"Core program trades:               {int(core_program_summary_fixed['trades'].iloc[0]):,}")
print(f"Core program dates:                {int(core_program_summary_fixed['unique_trade_dates'].iloc[0]):,}")
print(f"Combined program fixed rows:       {len(combined_program_panel_fixed):,}")
print(f"Combined continuity fixed rows:    {len(combined_continuity_panel_fixed):,}")

print()
print("Validation:")
display(validation_fixed)

comparison_display_cols = [
    "secondary_stack_candidate_id",
    "secondary_front_candidate_id",
    "secondary_middle_profile_id",
    "secondary_middle_source_candidate_id",
    "secondary_back_profile_id",
    "secondary_back_source_candidate_id",

    "secondary_front_model_vrp_log_min",
    "secondary_front_model_vrp_z_min",
    "secondary_front_RSI14_max",
    "secondary_front_rv21d_vol_pct_min",

    "secondary_middle_model_vrp_log_min",
    "secondary_middle_model_vrp_z_min",
    "secondary_middle_RSI14_max",
    "secondary_middle_rv21d_vol_pct_min",

    "secondary_back_model_vrp_log_min",
    "secondary_back_model_vrp_z_min",
    "secondary_back_RSI14_max",
    "secondary_back_rv21d_vol_pct_min",

    "trades",
    "unique_trade_dates",
    "signal_date_frequency",
    "win_rate",
    "total_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "profit_factor_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",

    "secondary_trades",
    "secondary_unique_trade_dates",
    "secondary_signal_date_frequency",
    "secondary_win_rate",
    "secondary_total_pnl",
    "secondary_avg_pnl_per_day",
    "secondary_profit_factor",
    "secondary_profit_factor_pnl_per_day",
    "secondary_pnl_per_day_drawdown",
    "secondary_worst_20_trade_pnl_per_day_sum",
    "secondary_avg_pnl_per_day_2025",
    "secondary_avg_pnl_per_day_2026",

    "incremental_front_trade_count",
    "incremental_middle_trade_count",
    "incremental_back_trade_count",
    "incremental_secondary_trade_count",

    "delta_program_unique_trade_dates_vs_core_only",
    "delta_program_signal_date_frequency_vs_core_only",
    "delta_program_win_rate_vs_core_only",
    "delta_program_total_pnl_vs_core_only",
    "delta_program_avg_pnl_per_day_vs_core_only",
    "delta_program_profit_factor_vs_core_only",
    "delta_program_profit_factor_pnl_per_day_vs_core_only",
    "delta_program_pnl_per_day_drawdown_vs_core_only",
    "delta_program_worst_20_trade_pnl_per_day_sum_vs_core_only",
    "delta_program_avg_pnl_per_day_2026_vs_core_only",

    "passes_frequency_target_25pct",
    "passes_combined_win_rate_target_80pct",
    "passes_secondary_win_rate_ideal_80pct",
    "passes_secondary_win_rate_floor_78pct",
    "passes_program_targets",
    "passes_full_ideal_targets",
    "passes_acceptable_secondary_floor",
]

print()
print("FIXED Candidate comparison — PROGRAM VIEW, Core first else Secondary stack:")
display(comparison_fixed[[c for c in comparison_display_cols if c in comparison_fixed.columns]])

continuity_display_cols = [
    "secondary_stack_candidate_id",
    "secondary_front_candidate_id",
    "secondary_middle_profile_id",
    "secondary_middle_source_candidate_id",
    "secondary_back_profile_id",
    "secondary_back_source_candidate_id",

    "trades",
    "unique_trade_dates",
    "signal_date_frequency",
    "win_rate",
    "total_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "profit_factor_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",

    "core_trade_count",
    "secondary_front_trade_count",
    "secondary_middle_trade_count",
    "secondary_back_trade_count",
    "secondary_trade_count",

    "delta_continuity_trades_vs_core_primary",
    "delta_continuity_unique_trade_dates_vs_core_primary",
    "delta_continuity_signal_date_frequency_vs_core_primary",
    "delta_continuity_win_rate_vs_core_primary",
    "delta_continuity_total_pnl_vs_core_primary",
    "delta_continuity_avg_pnl_per_day_vs_core_primary",
    "delta_continuity_profit_factor_vs_core_primary",
    "delta_continuity_profit_factor_pnl_per_day_vs_core_primary",
    "delta_continuity_pnl_per_day_drawdown_vs_core_primary",
    "delta_continuity_worst_20_trade_pnl_per_day_sum_vs_core_primary",
    "delta_continuity_avg_pnl_per_day_2026_vs_core_primary",
]

print()
print("FIXED Candidate comparison — CONTINUITY VIEW, Core primary + Secondary stack:")
display(continuity_comparison_fixed[[c for c in continuity_display_cols if c in continuity_comparison_fixed.columns]])

print()
print("FIXED Stack Secondary incremental summary:")
display(stack_secondary_summary_fixed)

print()
print("FIXED Combined program by year:")
display(combined_program_by_year_fixed)

print()
print("Saved repaired outputs:")
for p in [
    combined_program_panel_fixed_path,
    combined_program_panel_fixed_csv_path,
    combined_continuity_panel_fixed_path,
    combined_continuity_panel_fixed_csv_path,
    combined_program_summary_fixed_path,
    combined_program_by_year_fixed_path,
    combined_continuity_summary_fixed_path,
    combined_continuity_by_year_fixed_path,
    comparison_fixed_path,
    continuity_comparison_fixed_path,
    stack_secondary_summary_fixed_path,
    stack_secondary_by_bucket_fixed_path,
    stack_secondary_by_tenor_fixed_path,
    stack_secondary_by_year_fixed_path,
    validation_fixed_path,
    manifest_fixed_path,
]:
    print(f"  {p}")

print("\nCELL 14R-FIX PASSED — COMBINED PROGRAM STACK ASSIGNMENT REPAIRED")

In [ ]:
# ======================================================================================
# Cell 15R — Independent Secondary bucket recalibration
# ======================================================================================
#
# Objective:
#   Recalibrate Secondary Front, Middle, and Back independently behind locked Core.
#
#   This cell answers:
#       "If this bucket were the only Secondary bucket allowed behind Core,
#        how would it perform?"
#
# Core:
#   Locked Core repaired v1 remains unchanged:
#       9D excluded
#       Front  = 12D / 15D / 18D
#       Middle = 21D / 24D
#       Back   = 27D / 30D / 33D
#
# Secondary bucket tests:
#   1. Secondary Front alone:
#       tenors 12D / 15D / 18D
#
#   2. Secondary Middle alone:
#       tenors 21D / 24D
#
#   3. Secondary Back alone:
#       tenors 27D / 30D / 33D
#
# Core gate:
#   If any locked Core tenor qualifies on a date:
#       Secondary is ignored.
#
#   If no locked Core tenor qualifies:
#       test the independent Secondary bucket.
#
# Intra-bucket tenor selection:
#   If multiple tenors qualify inside the same Secondary bucket on a Core-empty date:
#       select one trade within that bucket/date by signal rank.
#
# This cell does NOT:
#   1. Select across Secondary buckets.
#   2. Build a combined Front/Middle/Back Secondary stack.
#   3. Lock Secondary parameters.
#   4. Change sizing.
#   5. Implement production/intraday logic.
#
# Final consistency screen:
#   After independent bucket scorecards are created, screen Front/Middle/Back parameter
#   triplets where:
#
#       min(Front, Back) <= Middle <= max(Front, Back)
#
#   for:
#       VRP log threshold
#       z threshold
#       RSI cap
#       RV21D floor
#
#   This consistency screen does NOT select trades across buckets.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import itertools
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 700)
pd.set_option("display.width", 950)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL15R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

LOCKED_CORE_VERSION = "core_repaired_v1"
LOCKED_CORE_DECISION_ID = "core_repaired_v1_locked_001"

SECONDARY_INDEPENDENT_SWEEP_VERSION = "secondary_independent_bucket_recalibration_v1"

ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

BUCKET_SPECS = {
    "Front": {
        "tenors": [12, 15, 18],
        "center": 15,
        "vrp_log_grid": [0.55, 0.60, 0.65],
        "z_grid": [0.20, 0.30, 0.40, 0.50, 0.60],
        "rsi_grid": [72.0, 73.0, 74.0, 75.0],
        "rv21d_grid": [6.5, 7.0, 7.5],
        "candidate_prefix": "secondary_front_ind",
        "bucket_order": 1,
    },
    "Middle": {
        "tenors": [21, 24],
        "center": 21,
        "vrp_log_grid": [0.55, 0.60, 0.65, 0.70],
        "z_grid": [0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60],
        "rsi_grid": [70.0, 71.0, 72.0, 73.0, 74.0, 75.0, 76.0],
        "rv21d_grid": [6.5, 7.0, 7.5],
        "candidate_prefix": "secondary_middle_ind",
        "bucket_order": 2,
    },
    "Back": {
        "tenors": [27, 30, 33],
        "center": 30,
        "vrp_log_grid": [0.60, 0.65, 0.70],
        "z_grid": [0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60],
        "rsi_grid": [73.0, 74.0, 75.0, 76.0, 77.0],
        "rv21d_grid": [6.5, 7.0, 7.5],
        "candidate_prefix": "secondary_back_ind",
        "bucket_order": 3,
    },
}

PRIMARY_CORE_CONTINUITY_UNIVERSE = "core_primary_one_trade_per_bucket_per_date"
CORE_PROGRAM_UNIVERSE = "core_program_one_trade_per_date"

INDEPENDENT_BUCKET_INCREMENTAL_UNIVERSE = "secondary_bucket_independent_incremental_core_empty_dates"
COMBINED_BUCKET_PROGRAM_UNIVERSE = "combined_program_core_first_else_independent_secondary_bucket"

COMBINED_WIN_RATE_TARGET = 0.80
INCREMENTAL_SECONDARY_WIN_RATE_MIN = 0.78
INCREMENTAL_SECONDARY_WIN_RATE_IDEAL = 0.80
INCREMENTAL_AVG_PNL_PER_DAY_MIN = 0.0
COMBINED_AVG_PNL_PER_DAY_MIN = 0.0
MIN_INCREMENTAL_TRADES_FOR_QUALITY_SCREEN = 10

MAX_TRIPLET_SCREEN_ROWS = 2_000_000

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)

    if matches:
        return matches[0]

    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")

    return None


def normalize_bool_series(s):
    if s.dtype == bool:
        return s.fillna(False)

    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y"])
    )


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)

    if gross_profit > 0:
        return np.inf

    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def add_rank_columns(d, group_cols, suffix):
    x = d.copy()

    if x.empty:
        return x

    x[f"rank_z_1y_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )

    x[f"rank_z_3m_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )

    x[f"rank_vrp_log_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    x[f"avg_signal_rank_{suffix}"] = x[
        [
            f"rank_z_1y_{suffix}",
            f"rank_z_3m_{suffix}",
            f"rank_vrp_log_{suffix}",
        ]
    ].mean(axis=1)

    if "bucket_center_tenor" not in x.columns:
        if "secondary_bucket" in x.columns:
            x["bucket_center_tenor"] = x["secondary_bucket"].map({
                "Front": BUCKET_SPECS["Front"]["center"],
                "Middle": BUCKET_SPECS["Middle"]["center"],
                "Back": BUCKET_SPECS["Back"]["center"],
            })
        elif "effective_tenor_bucket" in x.columns:
            x["bucket_center_tenor"] = x["effective_tenor_bucket"].map({
                "Front": BUCKET_SPECS["Front"]["center"],
                "Middle": BUCKET_SPECS["Middle"]["center"],
                "Back": BUCKET_SPECS["Back"]["center"],
            })
        else:
            x["bucket_center_tenor"] = 30

    x["bucket_center_tenor"] = pd.to_numeric(x["bucket_center_tenor"], errors="coerce").fillna(30)
    x["distance_to_bucket_center"] = (x["tenor"] - x["bucket_center_tenor"]).abs()

    return x


def select_core_universes(core_qualified_complete):
    q = core_qualified_complete.copy()

    if q.empty:
        return {
            PRIMARY_CORE_CONTINUITY_UNIVERSE: q.copy(),
            CORE_PROGRAM_UNIVERSE: q.copy(),
        }

    q = add_rank_columns(
        q,
        group_cols=["trade_date", "effective_tenor_bucket"],
        suffix="core_bucket_date",
    )

    q = add_rank_columns(
        q,
        group_cols=["trade_date"],
        suffix="core_date",
    )

    core_primary = (
        q.sort_values(
            [
                "trade_date",
                "effective_tenor_bucket_order",
                "avg_signal_rank_core_bucket_date",
                "distance_to_bucket_center",
                "rank_z_1y_core_bucket_date",
                "rank_z_3m_core_bucket_date",
                "rank_vrp_log_core_bucket_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date", "effective_tenor_bucket"], as_index=False)
        .head(1)
        .copy()
    )

    core_program = (
        q.sort_values(
            [
                "trade_date",
                "avg_signal_rank_core_date",
                "rank_z_1y_core_date",
                "rank_z_3m_core_date",
                "rank_vrp_log_core_date",
                "distance_to_bucket_center",
                "effective_tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return {
        PRIMARY_CORE_CONTINUITY_UNIVERSE: core_primary,
        CORE_PROGRAM_UNIVERSE: core_program,
    }


def select_one_trade_per_candidate_date(d, suffix):
    q = d.copy()

    if q.empty:
        return q

    q = add_rank_columns(
        q,
        group_cols=["secondary_bucket", "secondary_candidate_id", "trade_date"],
        suffix=suffix,
    )

    selected = (
        q.sort_values(
            [
                "secondary_bucket",
                "secondary_candidate_id",
                "trade_date",
                f"avg_signal_rank_{suffix}",
                "distance_to_bucket_center",
                f"rank_z_1y_{suffix}",
                f"rank_z_3m_{suffix}",
                f"rank_vrp_log_{suffix}",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True, True],
        )
        .groupby(["secondary_bucket", "secondary_candidate_id", "trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return selected


def summarize_trade_set(df, group_cols, total_eligible_dates=None):
    d = df.copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    numeric_cols = [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "forecast_vol_pct",
        "rv21d_vol_pct",

        "threshold_model_vrp_log",
        "threshold_model_vrp_z_3m",
        "threshold_model_vrp_z_1y",
        "threshold_RSI14_max",
        "threshold_rv21d_vol_pct_min",

        "secondary_threshold_model_vrp_log",
        "secondary_threshold_model_vrp_z_3m",
        "secondary_threshold_model_vrp_z_1y",
        "secondary_threshold_RSI14_max",
        "secondary_threshold_rv21d_vol_pct_min",
    ]

    for c in numeric_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d = d.sort_values(["trade_date", "tenor"]).copy()

    grouped = d.groupby(group_cols, dropna=False, observed=False) if group_cols else [((), d)]

    rows = []

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {}

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        unique_dates = int(g["trade_date"].nunique())
        signal_date_frequency = (
            unique_dates / total_eligible_dates
            if total_eligible_dates and total_eligible_dates > 0
            else np.nan
        )

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": unique_dates,
            "signal_date_frequency": float(signal_date_frequency) if pd.notna(signal_date_frequency) else np.nan,
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        for signal_col in [
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",

            "threshold_model_vrp_log",
            "threshold_model_vrp_z_3m",
            "threshold_model_vrp_z_1y",
            "threshold_RSI14_max",
            "threshold_rv21d_vol_pct_min",

            "secondary_threshold_model_vrp_log",
            "secondary_threshold_model_vrp_z_3m",
            "secondary_threshold_model_vrp_z_1y",
            "secondary_threshold_RSI14_max",
            "secondary_threshold_rv21d_vol_pct_min",
        ]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        if "program_leg" in g.columns:
            row["core_trade_count"] = int(g["program_leg"].eq("Core").sum())
            row["secondary_trade_count"] = int(g["program_leg"].str.startswith("Secondary", na=False).sum())

        rows.append(row)

    return pd.DataFrame(rows)


def build_candidate_grid(bucket_name, spec):
    rows = []

    i = 1

    for vrp_log, z, rsi_cap, rv_floor in itertools.product(
        spec["vrp_log_grid"],
        spec["z_grid"],
        spec["rsi_grid"],
        spec["rv21d_grid"],
    ):
        candidate_id = f"{spec['candidate_prefix']}_{i:04d}"

        rows.append({
            "secondary_bucket": bucket_name,
            "secondary_bucket_order": int(spec["bucket_order"]),
            "secondary_candidate_id": candidate_id,
            "secondary_independent_sweep_version": SECONDARY_INDEPENDENT_SWEEP_VERSION,
            "secondary_candidate_family": f"secondary_{bucket_name.lower()}_independent_core_gated",
            "secondary_candidate_description": (
                f"Secondary {bucket_name} independent, Core-gated; "
                f"log>{vrp_log:.2f}, z3m/z1y>{z:.2f}, "
                f"RSI14<{rsi_cap:.0f}, rv21d>{rv_floor:.1f}"
            ),

            "secondary_tenors": ",".join(str(t) for t in spec["tenors"]),
            "bucket_center_tenor": float(spec["center"]),

            "param_model_vrp_log_min": float(vrp_log),
            "param_model_vrp_z_min": float(z),
            "param_model_vrp_z_3m_min": float(z),
            "param_model_vrp_z_1y_min": float(z),
            "param_RSI14_max": float(rsi_cap),
            "param_rv21d_vol_pct_min": float(rv_floor),

            "z3m_equals_z1y": True,
            "uses_rv21d_threshold_contract": True,
            "forecast_vol_pct_threshold_used": False,
            "core_gated": True,

            "secondary_front_locked_here": False,
            "secondary_middle_locked_here": False,
            "secondary_back_locked_here": False,
            "sizing_locked_here": False,
            "production_locked_here": False,
        })

        i += 1

    return pd.DataFrame(rows)


def add_candidate_params(summary_df, candidate_grid):
    if summary_df.empty:
        return summary_df

    return summary_df.merge(
        candidate_grid,
        on=["secondary_bucket", "secondary_candidate_id"],
        how="left",
        validate="many_to_one",
    )


def rename_metric_cols(df, prefix):
    keep_cols = ["secondary_bucket", "secondary_candidate_id"]

    renamed = {}

    for c in df.columns:
        if c not in keep_cols:
            renamed[c] = f"{prefix}_{c}"

    return df.rename(columns=renamed)


def build_scorecard(bucket_name, candidate_grid, incremental_summary, combined_summary, core_ref):
    inc = incremental_summary.copy()
    comb = combined_summary.copy()

    inc_cols = [
        "secondary_bucket",
        "secondary_candidate_id",
        "trades",
        "unique_trade_dates",
        "signal_date_frequency",
        "win_rate",
        "total_pnl",
        "avg_pnl",
        "avg_pnl_per_day",
        "profit_factor",
        "profit_factor_pnl_per_day",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",
        "trades_2025",
        "avg_pnl_per_day_2025",
        "trades_2026",
        "avg_pnl_per_day_2026",
        "worst_trade_2026",
    ]

    comb_cols = [
        "secondary_bucket",
        "secondary_candidate_id",
        "trades",
        "unique_trade_dates",
        "signal_date_frequency",
        "win_rate",
        "total_pnl",
        "avg_pnl",
        "avg_pnl_per_day",
        "profit_factor",
        "profit_factor_pnl_per_day",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",
        "trades_2025",
        "avg_pnl_per_day_2025",
        "trades_2026",
        "avg_pnl_per_day_2026",
        "worst_trade_2026",
        "core_trade_count",
        "secondary_trade_count",
    ]

    inc = inc[[c for c in inc_cols if c in inc.columns]].copy()
    comb = comb[[c for c in comb_cols if c in comb.columns]].copy()

    inc = rename_metric_cols(inc, "incremental")
    comb = rename_metric_cols(comb, "combined")

    scorecard = candidate_grid.merge(
        inc,
        on=["secondary_bucket", "secondary_candidate_id"],
        how="left",
        validate="one_to_one",
    ).merge(
        comb,
        on=["secondary_bucket", "secondary_candidate_id"],
        how="left",
        validate="one_to_one",
    )

    for metric in [
        "trades",
        "unique_trade_dates",
        "signal_date_frequency",
        "win_rate",
        "total_pnl",
        "avg_pnl_per_day",
        "profit_factor",
        "profit_factor_pnl_per_day",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",
        "avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026",
    ]:
        c = f"combined_{metric}"

        if c in scorecard.columns and metric in core_ref:
            scorecard[f"delta_combined_{metric}_vs_core_only"] = scorecard[c] - core_ref[metric]

    scorecard["passes_incremental_win_rate_floor_78pct"] = (
        scorecard["incremental_win_rate"] >= INCREMENTAL_SECONDARY_WIN_RATE_MIN
    )

    scorecard["passes_incremental_win_rate_ideal_80pct"] = (
        scorecard["incremental_win_rate"] >= INCREMENTAL_SECONDARY_WIN_RATE_IDEAL
    )

    scorecard["passes_combined_win_rate_80pct"] = (
        scorecard["combined_win_rate"] >= COMBINED_WIN_RATE_TARGET
    )

    scorecard["passes_incremental_positive_avg_day"] = (
        scorecard["incremental_avg_pnl_per_day"] > INCREMENTAL_AVG_PNL_PER_DAY_MIN
    )

    scorecard["passes_combined_positive_avg_day"] = (
        scorecard["combined_avg_pnl_per_day"] > COMBINED_AVG_PNL_PER_DAY_MIN
    )

    scorecard["passes_min_trade_count"] = (
        scorecard["incremental_trades"] >= MIN_INCREMENTAL_TRADES_FOR_QUALITY_SCREEN
    )

    scorecard["passes_bucket_quality_floor"] = (
        scorecard["passes_incremental_win_rate_floor_78pct"]
        & scorecard["passes_combined_win_rate_80pct"]
        & scorecard["passes_incremental_positive_avg_day"]
        & scorecard["passes_combined_positive_avg_day"]
        & scorecard["passes_min_trade_count"]
    )

    scorecard["passes_bucket_quality_ideal"] = (
        scorecard["passes_incremental_win_rate_ideal_80pct"]
        & scorecard["passes_combined_win_rate_80pct"]
        & scorecard["passes_incremental_positive_avg_day"]
        & scorecard["passes_combined_positive_avg_day"]
        & scorecard["passes_min_trade_count"]
    )

    scorecard["bucket_score"] = (
        scorecard["passes_bucket_quality_ideal"].astype(int) * 10_000
        + scorecard["passes_bucket_quality_floor"].astype(int) * 1_000
        + scorecard["incremental_win_rate"].fillna(0) * 100
        + scorecard["combined_win_rate"].fillna(0) * 50
        + scorecard["incremental_signal_date_frequency"].fillna(0) * 25
        + np.clip(scorecard["incremental_avg_pnl_per_day"].fillna(-10_000), -10_000, 10_000) / 1_000
    )

    scorecard = scorecard.sort_values(
        [
            "passes_bucket_quality_ideal",
            "passes_bucket_quality_floor",
            "incremental_win_rate",
            "combined_win_rate",
            "incremental_signal_date_frequency",
            "incremental_avg_pnl_per_day",
            "combined_avg_pnl_per_day",
        ],
        ascending=[False, False, False, False, False, False, False],
    ).reset_index(drop=True)

    return scorecard


def run_independent_bucket_sweep(bucket_name, spec, base_for_join, core_program_clean, core_empty_dates, total_eligible_signal_dates):
    candidate_grid = build_candidate_grid(bucket_name, spec)

    bucket_base = base_for_join[
        base_for_join["tenor"].isin(spec["tenors"])
        & base_for_join["trade_date"].isin(core_empty_dates)
    ].copy()

    bucket_base["secondary_bucket"] = bucket_name
    bucket_base["secondary_bucket_order"] = int(spec["bucket_order"])
    bucket_base["bucket_center_tenor"] = float(spec["center"])

    panel = bucket_base.merge(
        candidate_grid,
        on=["secondary_bucket", "secondary_bucket_order", "bucket_center_tenor"],
        how="inner",
        validate="many_to_many",
    )

    panel = panel.rename(columns={
        "param_model_vrp_log_min": "secondary_threshold_model_vrp_log",
        "param_model_vrp_z_3m_min": "secondary_threshold_model_vrp_z_3m",
        "param_model_vrp_z_1y_min": "secondary_threshold_model_vrp_z_1y",
        "param_RSI14_max": "secondary_threshold_RSI14_max",
        "param_rv21d_vol_pct_min": "secondary_threshold_rv21d_vol_pct_min",
    })

    signal_available = (
        panel["model_vrp_log"].notna()
        & panel["model_vrp_z_3m"].notna()
        & panel["model_vrp_z_1y"].notna()
        & panel["RSI14"].notna()
        & panel["rv21d_vol_pct"].notna()
    )

    pass_mask = (
        signal_available
        & (panel["model_vrp_log"] > panel["secondary_threshold_model_vrp_log"])
        & (panel["model_vrp_z_3m"] > panel["secondary_threshold_model_vrp_z_3m"])
        & (panel["model_vrp_z_1y"] > panel["secondary_threshold_model_vrp_z_1y"])
        & (panel["RSI14"] < panel["secondary_threshold_RSI14_max"])
        & (panel["rv21d_vol_pct"] > panel["secondary_threshold_rv21d_vol_pct_min"])
    )

    panel["secondary_independent_pass"] = pass_mask

    qualified_complete = panel[
        pass_mask
        & panel["normalized_pnl_dollars"].notna()
        & panel["normalized_pnl_per_day"].notna()
    ].copy()

    selected = select_one_trade_per_candidate_date(
        qualified_complete,
        suffix=f"{bucket_name.lower()}_independent_bucket_date",
    )

    selected["source_layer"] = "Secondary"
    selected["program_leg"] = f"Secondary_{bucket_name}"
    selected["selection_universe"] = INDEPENDENT_BUCKET_INCREMENTAL_UNIVERSE

    incremental_summary = summarize_trade_set(
        selected,
        group_cols=["secondary_bucket", "secondary_candidate_id", "selection_universe"],
        total_eligible_dates=total_eligible_signal_dates,
    )

    incremental_summary = add_candidate_params(incremental_summary, candidate_grid)

    selected_by_tenor = summarize_trade_set(
        selected,
        group_cols=["secondary_bucket", "secondary_candidate_id", "selection_universe", "tenor"],
        total_eligible_dates=total_eligible_signal_dates,
    )

    selected_by_tenor = add_candidate_params(selected_by_tenor, candidate_grid)

    selected_by_year_input = (
        selected.assign(year=selected["trade_date"].dt.year.astype(int))
        if not selected.empty
        else selected.copy()
    )

    selected_by_year = summarize_trade_set(
        selected_by_year_input,
        group_cols=["secondary_bucket", "secondary_candidate_id", "selection_universe", "year"],
        total_eligible_dates=total_eligible_signal_dates,
    )

    selected_by_year = add_candidate_params(selected_by_year, candidate_grid)

    selected_by_tenor_year_input = (
        selected.assign(year=selected["trade_date"].dt.year.astype(int))
        if not selected.empty
        else selected.copy()
    )

    selected_by_tenor_year = summarize_trade_set(
        selected_by_tenor_year_input,
        group_cols=["secondary_bucket", "secondary_candidate_id", "selection_universe", "tenor", "year"],
        total_eligible_dates=total_eligible_signal_dates,
    )

    selected_by_tenor_year = add_candidate_params(selected_by_tenor_year, candidate_grid)

    worst_trades = (
        selected.sort_values(
            ["secondary_candidate_id", "normalized_pnl_per_day", "normalized_pnl_dollars"],
            ascending=[True, True, True],
        )
        .groupby(["secondary_bucket", "secondary_candidate_id"], as_index=False)
        .head(20)
        .copy()
        if not selected.empty
        else selected.copy()
    )

    core_program_bucket = core_program_clean.merge(
        candidate_grid[
            [
                "secondary_bucket",
                "secondary_candidate_id",
                "secondary_candidate_description",
                "param_model_vrp_log_min",
                "param_model_vrp_z_min",
                "param_model_vrp_z_3m_min",
                "param_model_vrp_z_1y_min",
                "param_RSI14_max",
                "param_rv21d_vol_pct_min",
            ]
        ],
        how="cross",
    )

    core_program_bucket["selection_universe"] = COMBINED_BUCKET_PROGRAM_UNIVERSE
    core_program_bucket["source_layer"] = "Core"
    core_program_bucket["program_leg"] = "Core"

    selected_for_combined = selected.copy()
    selected_for_combined["selection_universe"] = COMBINED_BUCKET_PROGRAM_UNIVERSE

    combined_panel = pd.concat(
        [
            core_program_bucket,
            selected_for_combined,
        ],
        ignore_index=True,
    )

    combined_summary = summarize_trade_set(
        combined_panel,
        group_cols=["secondary_bucket", "secondary_candidate_id", "selection_universe"],
        total_eligible_dates=total_eligible_signal_dates,
    )

    combined_summary = add_candidate_params(combined_summary, candidate_grid)

    combined_by_year_input = combined_panel.assign(
        year=combined_panel["trade_date"].dt.year.astype(int)
    )

    combined_by_year = summarize_trade_set(
        combined_by_year_input,
        group_cols=["secondary_bucket", "secondary_candidate_id", "selection_universe", "year"],
        total_eligible_dates=total_eligible_signal_dates,
    )

    combined_by_year = add_candidate_params(combined_by_year, candidate_grid)

    return {
        "bucket": bucket_name,
        "candidate_grid": candidate_grid,
        "panel": panel,
        "qualified_complete": qualified_complete,
        "selected": selected,
        "incremental_summary": incremental_summary,
        "selected_by_tenor": selected_by_tenor,
        "selected_by_year": selected_by_year,
        "selected_by_tenor_year": selected_by_tenor_year,
        "worst_trades": worst_trades,
        "combined_panel": combined_panel,
        "combined_summary": combined_summary,
        "combined_by_year": combined_by_year,
    }


def slim_for_triplet_screen(scorecard, bucket_name):
    cols = [
        "secondary_bucket",
        "secondary_candidate_id",
        "secondary_candidate_description",
        "param_model_vrp_log_min",
        "param_model_vrp_z_min",
        "param_RSI14_max",
        "param_rv21d_vol_pct_min",
        "incremental_trades",
        "incremental_unique_trade_dates",
        "incremental_signal_date_frequency",
        "incremental_win_rate",
        "incremental_total_pnl",
        "incremental_avg_pnl_per_day",
        "incremental_profit_factor_pnl_per_day",
        "incremental_pnl_per_day_drawdown",
        "incremental_worst_20_trade_pnl_per_day_sum",
        "incremental_avg_pnl_per_day_2025",
        "incremental_avg_pnl_per_day_2026",
        "combined_unique_trade_dates",
        "combined_signal_date_frequency",
        "combined_win_rate",
        "combined_total_pnl",
        "combined_avg_pnl_per_day",
        "combined_profit_factor_pnl_per_day",
        "combined_pnl_per_day_drawdown",
        "combined_worst_20_trade_pnl_per_day_sum",
        "combined_avg_pnl_per_day_2025",
        "combined_avg_pnl_per_day_2026",
        "passes_bucket_quality_floor",
        "passes_bucket_quality_ideal",
        "bucket_score",
    ]

    x = scorecard[[c for c in cols if c in scorecard.columns]].copy()

    rename_map = {
        c: f"{bucket_name.lower()}_{c}"
        for c in x.columns
        if c not in ["secondary_bucket"]
    }

    x = x.rename(columns=rename_map)

    return x


def build_parameter_consistency_screen(front_scorecard, middle_scorecard, back_scorecard):
    front_q = front_scorecard[front_scorecard["passes_bucket_quality_floor"].astype(bool)].copy()
    middle_q = middle_scorecard[middle_scorecard["passes_bucket_quality_floor"].astype(bool)].copy()
    back_q = back_scorecard[back_scorecard["passes_bucket_quality_floor"].astype(bool)].copy()

    front_q = front_q.sort_values("bucket_score", ascending=False).reset_index(drop=True)
    middle_q = middle_q.sort_values("bucket_score", ascending=False).reset_index(drop=True)
    back_q = back_q.sort_values("bucket_score", ascending=False).reset_index(drop=True)

    rows = []
    was_truncated = False

    if front_q.empty or middle_q.empty or back_q.empty:
        return pd.DataFrame(), {
            "front_quality_candidates": int(len(front_q)),
            "middle_quality_candidates": int(len(middle_q)),
            "back_quality_candidates": int(len(back_q)),
            "screen_rows": 0,
            "was_truncated": False,
            "max_rows": MAX_TRIPLET_SCREEN_ROWS,
        }

    middle_param = middle_q[
        [
            "secondary_candidate_id",
            "secondary_candidate_description",
            "param_model_vrp_log_min",
            "param_model_vrp_z_min",
            "param_RSI14_max",
            "param_rv21d_vol_pct_min",
            "incremental_trades",
            "incremental_win_rate",
            "incremental_signal_date_frequency",
            "incremental_avg_pnl_per_day",
            "incremental_profit_factor_pnl_per_day",
            "incremental_pnl_per_day_drawdown",
            "incremental_worst_20_trade_pnl_per_day_sum",
            "incremental_avg_pnl_per_day_2026",
            "combined_win_rate",
            "combined_signal_date_frequency",
            "combined_avg_pnl_per_day",
            "combined_profit_factor_pnl_per_day",
            "combined_avg_pnl_per_day_2026",
            "passes_bucket_quality_floor",
            "passes_bucket_quality_ideal",
            "bucket_score",
        ]
    ].copy()

    for f in front_q.itertuples(index=False):
        for b in back_q.itertuples(index=False):
            vrp_low = min(f.param_model_vrp_log_min, b.param_model_vrp_log_min)
            vrp_high = max(f.param_model_vrp_log_min, b.param_model_vrp_log_min)

            z_low = min(f.param_model_vrp_z_min, b.param_model_vrp_z_min)
            z_high = max(f.param_model_vrp_z_min, b.param_model_vrp_z_min)

            rsi_low = min(f.param_RSI14_max, b.param_RSI14_max)
            rsi_high = max(f.param_RSI14_max, b.param_RSI14_max)

            rv_low = min(f.param_rv21d_vol_pct_min, b.param_rv21d_vol_pct_min)
            rv_high = max(f.param_rv21d_vol_pct_min, b.param_rv21d_vol_pct_min)

            m_ok = middle_param[
                middle_param["param_model_vrp_log_min"].between(vrp_low, vrp_high, inclusive="both")
                & middle_param["param_model_vrp_z_min"].between(z_low, z_high, inclusive="both")
                & middle_param["param_RSI14_max"].between(rsi_low, rsi_high, inclusive="both")
                & middle_param["param_rv21d_vol_pct_min"].between(rv_low, rv_high, inclusive="both")
            ]

            if m_ok.empty:
                continue

            for m in m_ok.itertuples(index=False):
                min_incremental_win_rate = min(
                    f.incremental_win_rate,
                    m.incremental_win_rate,
                    b.incremental_win_rate,
                )

                min_combined_win_rate = min(
                    f.combined_win_rate,
                    m.combined_win_rate,
                    b.combined_win_rate,
                )

                avg_incremental_win_rate = np.mean([
                    f.incremental_win_rate,
                    m.incremental_win_rate,
                    b.incremental_win_rate,
                ])

                avg_combined_win_rate = np.mean([
                    f.combined_win_rate,
                    m.combined_win_rate,
                    b.combined_win_rate,
                ])

                sum_independent_incremental_frequency = (
                    f.incremental_signal_date_frequency
                    + m.incremental_signal_date_frequency
                    + b.incremental_signal_date_frequency
                )

                avg_incremental_avg_day = np.mean([
                    f.incremental_avg_pnl_per_day,
                    m.incremental_avg_pnl_per_day,
                    b.incremental_avg_pnl_per_day,
                ])

                avg_combined_avg_day = np.mean([
                    f.combined_avg_pnl_per_day,
                    m.combined_avg_pnl_per_day,
                    b.combined_avg_pnl_per_day,
                ])

                all_ideal = bool(
                    f.passes_bucket_quality_ideal
                    and m.passes_bucket_quality_ideal
                    and b.passes_bucket_quality_ideal
                )

                triplet_score = (
                    int(all_ideal) * 10_000
                    + min_incremental_win_rate * 1_000
                    + avg_incremental_win_rate * 500
                    + min_combined_win_rate * 250
                    + sum_independent_incremental_frequency * 100
                    + avg_incremental_avg_day / 100
                )

                rows.append({
                    "front_candidate_id": f.secondary_candidate_id,
                    "middle_candidate_id": m.secondary_candidate_id,
                    "back_candidate_id": b.secondary_candidate_id,

                    "front_description": f.secondary_candidate_description,
                    "middle_description": m.secondary_candidate_description,
                    "back_description": b.secondary_candidate_description,

                    "front_vrp_log": f.param_model_vrp_log_min,
                    "middle_vrp_log": m.param_model_vrp_log_min,
                    "back_vrp_log": b.param_model_vrp_log_min,

                    "front_z": f.param_model_vrp_z_min,
                    "middle_z": m.param_model_vrp_z_min,
                    "back_z": b.param_model_vrp_z_min,

                    "front_RSI14_max": f.param_RSI14_max,
                    "middle_RSI14_max": m.param_RSI14_max,
                    "back_RSI14_max": b.param_RSI14_max,

                    "front_rv21d_floor": f.param_rv21d_vol_pct_min,
                    "middle_rv21d_floor": m.param_rv21d_vol_pct_min,
                    "back_rv21d_floor": b.param_rv21d_vol_pct_min,

                    "front_incremental_trades": f.incremental_trades,
                    "middle_incremental_trades": m.incremental_trades,
                    "back_incremental_trades": b.incremental_trades,

                    "front_incremental_win_rate": f.incremental_win_rate,
                    "middle_incremental_win_rate": m.incremental_win_rate,
                    "back_incremental_win_rate": b.incremental_win_rate,

                    "front_combined_win_rate": f.combined_win_rate,
                    "middle_combined_win_rate": m.combined_win_rate,
                    "back_combined_win_rate": b.combined_win_rate,

                    "front_incremental_frequency": f.incremental_signal_date_frequency,
                    "middle_incremental_frequency": m.incremental_signal_date_frequency,
                    "back_incremental_frequency": b.incremental_signal_date_frequency,

                    "sum_independent_incremental_frequency": sum_independent_incremental_frequency,

                    "front_incremental_avg_pnl_per_day": f.incremental_avg_pnl_per_day,
                    "middle_incremental_avg_pnl_per_day": m.incremental_avg_pnl_per_day,
                    "back_incremental_avg_pnl_per_day": b.incremental_avg_pnl_per_day,

                    "front_incremental_avg_pnl_per_day_2026": f.incremental_avg_pnl_per_day_2026,
                    "middle_incremental_avg_pnl_per_day_2026": m.incremental_avg_pnl_per_day_2026,
                    "back_incremental_avg_pnl_per_day_2026": b.incremental_avg_pnl_per_day_2026,

                    "front_incremental_pf_day": f.incremental_profit_factor_pnl_per_day,
                    "middle_incremental_pf_day": m.incremental_profit_factor_pnl_per_day,
                    "back_incremental_pf_day": b.incremental_profit_factor_pnl_per_day,

                    "front_incremental_dd_day": f.incremental_pnl_per_day_drawdown,
                    "middle_incremental_dd_day": m.incremental_pnl_per_day_drawdown,
                    "back_incremental_dd_day": b.incremental_pnl_per_day_drawdown,

                    "front_incremental_worst20_day": f.incremental_worst_20_trade_pnl_per_day_sum,
                    "middle_incremental_worst20_day": m.incremental_worst_20_trade_pnl_per_day_sum,
                    "back_incremental_worst20_day": b.incremental_worst_20_trade_pnl_per_day_sum,

                    "min_incremental_win_rate": min_incremental_win_rate,
                    "avg_incremental_win_rate": avg_incremental_win_rate,
                    "min_combined_win_rate": min_combined_win_rate,
                    "avg_combined_win_rate": avg_combined_win_rate,
                    "avg_incremental_avg_pnl_per_day": avg_incremental_avg_day,
                    "avg_combined_avg_pnl_per_day": avg_combined_avg_day,

                    "front_quality_ideal": bool(f.passes_bucket_quality_ideal),
                    "middle_quality_ideal": bool(m.passes_bucket_quality_ideal),
                    "back_quality_ideal": bool(b.passes_bucket_quality_ideal),
                    "all_three_quality_ideal": all_ideal,

                    "middle_between_front_back_vrp_log": True,
                    "middle_between_front_back_z": True,
                    "middle_between_front_back_RSI14": True,
                    "middle_between_front_back_rv21d": True,
                    "middle_between_front_back_all_params": True,

                    "triplet_score": triplet_score,
                })

                if len(rows) >= MAX_TRIPLET_SCREEN_ROWS:
                    was_truncated = True
                    break

            if was_truncated:
                break

        if was_truncated:
            break

    screen = pd.DataFrame(rows)

    if not screen.empty:
        screen = screen.sort_values(
            [
                "all_three_quality_ideal",
                "min_incremental_win_rate",
                "avg_incremental_win_rate",
                "sum_independent_incremental_frequency",
                "avg_incremental_avg_pnl_per_day",
                "triplet_score",
            ],
            ascending=[False, False, False, False, False, False],
        ).reset_index(drop=True)

    meta = {
        "front_quality_candidates": int(len(front_q)),
        "middle_quality_candidates": int(len(middle_q)),
        "back_quality_candidates": int(len(back_q)),
        "screen_rows": int(len(screen)),
        "was_truncated": bool(was_truncated),
        "max_rows": int(MAX_TRIPLET_SCREEN_ROWS),
    }

    return screen, meta


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


# ======================================================================================
# 2. Load inputs
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01R_unified_fds_signal_base_panel_with_rv21d_*.parquet",
    required=True,
)

locked_core_rules_long_path = latest_file(
    OUT_PROCESSED_DIR,
    "11R_core_repaired_v1_locked_rules_long_*.parquet",
    required=True,
)

locked_core_rules_wide_path = latest_file(
    OUT_PROCESSED_DIR,
    "11R_core_repaired_v1_locked_rules_wide_*.parquet",
    required=True,
)

base = pd.read_parquet(base_panel_path)
locked_core_rules_long = pd.read_parquet(locked_core_rules_long_path)
locked_core_rules_wide = pd.read_parquet(locked_core_rules_wide_path)

base["trade_date"] = pd.to_datetime(base["trade_date"], errors="coerce").dt.normalize()

for c in [
    "tenor",
    "tenor_bucket_order",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")

for c in [
    "tenor",
    "original_tenor_bucket_order",
    "effective_tenor_bucket_order",
    "model_vrp_log_min",
    "model_vrp_z_min",
    "model_vrp_z_3m_min",
    "model_vrp_z_1y_min",
    "RSI14_max",
    "rv21d_vol_pct_min",
]:
    if c in locked_core_rules_long.columns:
        locked_core_rules_long[c] = pd.to_numeric(locked_core_rules_long[c], errors="coerce")

total_eligible_signal_dates = int(base["trade_date"].nunique())

# ======================================================================================
# 3. Apply locked Core rules and build Core baselines
# ======================================================================================

core_thresholds = locked_core_rules_long.rename(columns={
    "model_vrp_log_min": "threshold_model_vrp_log",
    "model_vrp_z_3m_min": "threshold_model_vrp_z_3m",
    "model_vrp_z_1y_min": "threshold_model_vrp_z_1y",
    "RSI14_max": "threshold_RSI14_max",
    "rv21d_vol_pct_min": "threshold_rv21d_vol_pct_min",
})

core_threshold_cols = [
    "tenor",
    "original_tenor_bucket",
    "original_tenor_bucket_order",
    "effective_tenor_bucket",
    "effective_tenor_bucket_order",
    "include_tenor",
    "threshold_model_vrp_log",
    "threshold_model_vrp_z_3m",
    "threshold_model_vrp_z_1y",
    "threshold_RSI14_max",
    "threshold_rv21d_vol_pct_min",
]

base_for_join = base.copy()

base_for_join = base_for_join.rename(columns={
    "tenor_bucket": "source_tenor_bucket",
    "tenor_bucket_order": "source_tenor_bucket_order",
})

core_panel = base_for_join.merge(
    core_thresholds[core_threshold_cols],
    on="tenor",
    how="inner",
    validate="many_to_one",
)

core_panel["include_tenor_bool"] = normalize_bool_series(core_panel["include_tenor"])

core_signal_available = (
    core_panel["include_tenor_bool"]
    & core_panel["model_vrp_log"].notna()
    & core_panel["model_vrp_z_3m"].notna()
    & core_panel["model_vrp_z_1y"].notna()
    & core_panel["RSI14"].notna()
    & core_panel["rv21d_vol_pct"].notna()
)

core_pass_mask = (
    core_signal_available
    & (core_panel["model_vrp_log"] > core_panel["threshold_model_vrp_log"])
    & (core_panel["model_vrp_z_3m"] > core_panel["threshold_model_vrp_z_3m"])
    & (core_panel["model_vrp_z_1y"] > core_panel["threshold_model_vrp_z_1y"])
    & (core_panel["RSI14"] < core_panel["threshold_RSI14_max"])
    & (core_panel["rv21d_vol_pct"] > core_panel["threshold_rv21d_vol_pct_min"])
)

core_panel["core_repaired_v1_pass"] = core_pass_mask

core_qualified_all_dates = core_panel[core_pass_mask].copy()

core_qualified_complete = core_panel[
    core_pass_mask
    & core_panel["normalized_pnl_dollars"].notna()
    & core_panel["normalized_pnl_per_day"].notna()
].copy()

core_gate_dates = sorted(pd.to_datetime(core_qualified_all_dates["trade_date"].dropna().unique()).tolist())
all_signal_dates = sorted(pd.to_datetime(base["trade_date"].dropna().unique()).tolist())

core_gate_dates_set = set(core_gate_dates)
core_empty_dates = sorted(set(all_signal_dates) - core_gate_dates_set)

core_universes = select_core_universes(core_qualified_complete)

core_primary_continuity = core_universes[PRIMARY_CORE_CONTINUITY_UNIVERSE].copy()
core_program = core_universes[CORE_PROGRAM_UNIVERSE].copy()

core_primary_continuity["source_layer"] = "Core"
core_primary_continuity["program_leg"] = "Core"
core_primary_continuity["selection_universe"] = PRIMARY_CORE_CONTINUITY_UNIVERSE

core_program["source_layer"] = "Core"
core_program["program_leg"] = "Core"
core_program["selection_universe"] = CORE_PROGRAM_UNIVERSE

stale_secondary_cols = [
    "secondary_bucket",
    "secondary_bucket_order",
    "secondary_candidate_id",
    "secondary_candidate_description",
    "param_model_vrp_log_min",
    "param_model_vrp_z_min",
    "param_model_vrp_z_3m_min",
    "param_model_vrp_z_1y_min",
    "param_RSI14_max",
    "param_rv21d_vol_pct_min",
]

core_program_clean = core_program.drop(
    columns=[c for c in stale_secondary_cols if c in core_program.columns],
    errors="ignore",
).copy()

core_summary = summarize_trade_set(
    core_program_clean.assign(selection_universe=CORE_PROGRAM_UNIVERSE),
    group_cols=["selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

core_ref = core_summary.iloc[0].to_dict() if len(core_summary) else {}

# ======================================================================================
# 4. Run independent bucket sweeps
# ======================================================================================

bucket_results = {}

for bucket_name, spec in BUCKET_SPECS.items():
    print(f"Running independent Secondary {bucket_name} sweep...")

    bucket_results[bucket_name] = run_independent_bucket_sweep(
        bucket_name=bucket_name,
        spec=spec,
        base_for_join=base_for_join,
        core_program_clean=core_program_clean,
        core_empty_dates=core_empty_dates,
        total_eligible_signal_dates=total_eligible_signal_dates,
    )

# ======================================================================================
# 5. Build scorecards
# ======================================================================================

scorecards = {}

for bucket_name, res in bucket_results.items():
    scorecards[bucket_name] = build_scorecard(
        bucket_name=bucket_name,
        candidate_grid=res["candidate_grid"],
        incremental_summary=res["incremental_summary"],
        combined_summary=res["combined_summary"],
        core_ref=core_ref,
    )

all_candidate_grids = pd.concat(
    [res["candidate_grid"] for res in bucket_results.values()],
    ignore_index=True,
)

all_incremental_selected = pd.concat(
    [res["selected"] for res in bucket_results.values()],
    ignore_index=True,
)

all_incremental_summaries = pd.concat(
    [res["incremental_summary"] for res in bucket_results.values()],
    ignore_index=True,
)

all_combined_summaries = pd.concat(
    [res["combined_summary"] for res in bucket_results.values()],
    ignore_index=True,
)

all_scorecards = pd.concat(
    [scorecards[b] for b in ["Front", "Middle", "Back"]],
    ignore_index=True,
)

# ======================================================================================
# 6. Parameter triplet consistency screen
# ======================================================================================

triplet_consistency_screen, triplet_screen_meta = build_parameter_consistency_screen(
    front_scorecard=scorecards["Front"],
    middle_scorecard=scorecards["Middle"],
    back_scorecard=scorecards["Back"],
)

# ======================================================================================
# 7. Save outputs
# ======================================================================================

paths = {}

paths["candidate_grid_all_csv"] = OUT_AUDIT_DIR / f"15R_secondary_independent_candidate_grid_all_{CELL15R_TIMESTAMP}.csv"
paths["candidate_grid_all_parquet"] = OUT_PROCESSED_DIR / f"15R_secondary_independent_candidate_grid_all_{CELL15R_TIMESTAMP}.parquet"

paths["core_program_summary_csv"] = OUT_AUDIT_DIR / f"15R_core_program_baseline_{CELL15R_TIMESTAMP}.csv"
paths["core_gate_dates_csv"] = OUT_AUDIT_DIR / f"15R_core_gate_dates_{CELL15R_TIMESTAMP}.csv"
paths["core_empty_dates_csv"] = OUT_AUDIT_DIR / f"15R_core_empty_dates_{CELL15R_TIMESTAMP}.csv"

paths["all_incremental_selected_parquet"] = OUT_PROCESSED_DIR / f"15R_secondary_independent_selected_panel_all_{CELL15R_TIMESTAMP}.parquet"
paths["all_incremental_selected_csv"] = OUT_AUDIT_DIR / f"15R_secondary_independent_selected_panel_all_{CELL15R_TIMESTAMP}.csv"

paths["all_incremental_summary_csv"] = OUT_AUDIT_DIR / f"15R_secondary_independent_incremental_summary_all_{CELL15R_TIMESTAMP}.csv"
paths["all_combined_summary_csv"] = OUT_AUDIT_DIR / f"15R_secondary_independent_combined_summary_all_{CELL15R_TIMESTAMP}.csv"
paths["all_scorecards_csv"] = OUT_AUDIT_DIR / f"15R_secondary_independent_scorecards_all_{CELL15R_TIMESTAMP}.csv"

paths["triplet_consistency_screen_csv"] = OUT_AUDIT_DIR / f"15R_secondary_parameter_triplet_consistency_screen_{CELL15R_TIMESTAMP}.csv"
paths["triplet_consistency_screen_parquet"] = OUT_PROCESSED_DIR / f"15R_secondary_parameter_triplet_consistency_screen_{CELL15R_TIMESTAMP}.parquet"

all_candidate_grids.to_csv(paths["candidate_grid_all_csv"], index=False)
all_candidate_grids.to_parquet(paths["candidate_grid_all_parquet"], index=False)

core_summary.to_csv(paths["core_program_summary_csv"], index=False)
pd.DataFrame({"trade_date": core_gate_dates}).to_csv(paths["core_gate_dates_csv"], index=False)
pd.DataFrame({"trade_date": core_empty_dates}).to_csv(paths["core_empty_dates_csv"], index=False)

all_incremental_selected.to_parquet(paths["all_incremental_selected_parquet"], index=False)
all_incremental_selected.to_csv(paths["all_incremental_selected_csv"], index=False)

all_incremental_summaries.to_csv(paths["all_incremental_summary_csv"], index=False)
all_combined_summaries.to_csv(paths["all_combined_summary_csv"], index=False)
all_scorecards.to_csv(paths["all_scorecards_csv"], index=False)

triplet_consistency_screen.to_csv(paths["triplet_consistency_screen_csv"], index=False)
triplet_consistency_screen.to_parquet(paths["triplet_consistency_screen_parquet"], index=False)

for bucket_name, res in bucket_results.items():
    bucket_lower = bucket_name.lower()

    paths[f"{bucket_lower}_candidate_grid_csv"] = OUT_AUDIT_DIR / f"15R_secondary_{bucket_lower}_candidate_grid_{CELL15R_TIMESTAMP}.csv"
    paths[f"{bucket_lower}_candidate_grid_parquet"] = OUT_PROCESSED_DIR / f"15R_secondary_{bucket_lower}_candidate_grid_{CELL15R_TIMESTAMP}.parquet"

    paths[f"{bucket_lower}_selected_parquet"] = OUT_PROCESSED_DIR / f"15R_secondary_{bucket_lower}_independent_selected_panel_{CELL15R_TIMESTAMP}.parquet"
    paths[f"{bucket_lower}_selected_csv"] = OUT_AUDIT_DIR / f"15R_secondary_{bucket_lower}_independent_selected_panel_{CELL15R_TIMESTAMP}.csv"

    paths[f"{bucket_lower}_incremental_summary_csv"] = OUT_AUDIT_DIR / f"15R_secondary_{bucket_lower}_independent_incremental_summary_{CELL15R_TIMESTAMP}.csv"
    paths[f"{bucket_lower}_incremental_by_tenor_csv"] = OUT_AUDIT_DIR / f"15R_secondary_{bucket_lower}_independent_by_tenor_{CELL15R_TIMESTAMP}.csv"
    paths[f"{bucket_lower}_incremental_by_year_csv"] = OUT_AUDIT_DIR / f"15R_secondary_{bucket_lower}_independent_by_year_{CELL15R_TIMESTAMP}.csv"
    paths[f"{bucket_lower}_incremental_by_tenor_year_csv"] = OUT_AUDIT_DIR / f"15R_secondary_{bucket_lower}_independent_by_tenor_year_{CELL15R_TIMESTAMP}.csv"
    paths[f"{bucket_lower}_worst_trades_csv"] = OUT_AUDIT_DIR / f"15R_secondary_{bucket_lower}_independent_worst_trades_{CELL15R_TIMESTAMP}.csv"

    paths[f"{bucket_lower}_combined_panel_parquet"] = OUT_PROCESSED_DIR / f"15R_secondary_{bucket_lower}_combined_core_plus_bucket_panel_{CELL15R_TIMESTAMP}.parquet"
    paths[f"{bucket_lower}_combined_summary_csv"] = OUT_AUDIT_DIR / f"15R_secondary_{bucket_lower}_combined_core_plus_bucket_summary_{CELL15R_TIMESTAMP}.csv"
    paths[f"{bucket_lower}_combined_by_year_csv"] = OUT_AUDIT_DIR / f"15R_secondary_{bucket_lower}_combined_core_plus_bucket_by_year_{CELL15R_TIMESTAMP}.csv"

    paths[f"{bucket_lower}_scorecard_csv"] = OUT_AUDIT_DIR / f"15R_secondary_{bucket_lower}_independent_scorecard_{CELL15R_TIMESTAMP}.csv"

    res["candidate_grid"].to_csv(paths[f"{bucket_lower}_candidate_grid_csv"], index=False)
    res["candidate_grid"].to_parquet(paths[f"{bucket_lower}_candidate_grid_parquet"], index=False)

    res["selected"].to_parquet(paths[f"{bucket_lower}_selected_parquet"], index=False)
    res["selected"].to_csv(paths[f"{bucket_lower}_selected_csv"], index=False)

    res["incremental_summary"].to_csv(paths[f"{bucket_lower}_incremental_summary_csv"], index=False)
    res["selected_by_tenor"].to_csv(paths[f"{bucket_lower}_incremental_by_tenor_csv"], index=False)
    res["selected_by_year"].to_csv(paths[f"{bucket_lower}_incremental_by_year_csv"], index=False)
    res["selected_by_tenor_year"].to_csv(paths[f"{bucket_lower}_incremental_by_tenor_year_csv"], index=False)
    res["worst_trades"].to_csv(paths[f"{bucket_lower}_worst_trades_csv"], index=False)

    res["combined_panel"].to_parquet(paths[f"{bucket_lower}_combined_panel_parquet"], index=False)
    res["combined_summary"].to_csv(paths[f"{bucket_lower}_combined_summary_csv"], index=False)
    res["combined_by_year"].to_csv(paths[f"{bucket_lower}_combined_by_year_csv"], index=False)

    scorecards[bucket_name].to_csv(paths[f"{bucket_lower}_scorecard_csv"], index=False)

# ======================================================================================
# 8. Validation
# ======================================================================================

validation_rows = []

required_base_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

missing_base_cols = [c for c in required_base_cols if c not in base.columns]
models_found = sorted(base["model_spec"].dropna().unique().tolist()) if "model_spec" in base.columns else []

expected_counts = {
    bucket_name: (
        len(spec["vrp_log_grid"])
        * len(spec["z_grid"])
        * len(spec["rsi_grid"])
        * len(spec["rv21d_grid"])
    )
    for bucket_name, spec in BUCKET_SPECS.items()
}

actual_counts = {
    bucket_name: int(len(bucket_results[bucket_name]["candidate_grid"]))
    for bucket_name in BUCKET_SPECS
}

bad_candidate_counts = {
    bucket_name: {
        "expected": expected_counts[bucket_name],
        "actual": actual_counts[bucket_name],
    }
    for bucket_name in BUCKET_SPECS
    if expected_counts[bucket_name] != actual_counts[bucket_name]
}

bad_z_equality = all_candidate_grids[
    all_candidate_grids["param_model_vrp_z_3m_min"].ne(
        all_candidate_grids["param_model_vrp_z_1y_min"]
    )
]

bad_forecast_threshold = all_candidate_grids[
    all_candidate_grids["forecast_vol_pct_threshold_used"].astype(bool)
]

bad_rv21d_contract = all_candidate_grids[
    ~all_candidate_grids["uses_rv21d_threshold_contract"].astype(bool)
]

bad_secondary_on_core_gate = (
    all_incremental_selected[
        all_incremental_selected["trade_date"].isin(core_gate_dates_set)
    ]
    if not all_incremental_selected.empty
    else pd.DataFrame()
)

bad_tenors = []

for bucket_name, spec in BUCKET_SPECS.items():
    selected = bucket_results[bucket_name]["selected"]

    if selected.empty:
        continue

    bad = selected[
        ~selected["tenor"].isin(spec["tenors"])
    ].copy()

    if not bad.empty:
        bad["expected_tenors"] = ",".join(str(t) for t in spec["tenors"])
        bad_tenors.append(bad)

bad_tenors_df = pd.concat(bad_tenors, ignore_index=True) if bad_tenors else pd.DataFrame()

dupes = (
    all_incremental_selected
    .groupby(["secondary_bucket", "secondary_candidate_id", "trade_date"])
    .size()
    .reset_index(name="rows")
    if not all_incremental_selected.empty
    else pd.DataFrame(columns=["secondary_bucket", "secondary_candidate_id", "trade_date", "rows"])
)

bad_dupes = dupes[dupes["rows"].gt(1)]

combined_dupes = []

for bucket_name, res in bucket_results.items():
    d = (
        res["combined_panel"]
        .groupby(["secondary_bucket", "secondary_candidate_id", "trade_date"])
        .size()
        .reset_index(name="rows")
    )

    bad = d[d["rows"].gt(1)].copy()

    if not bad.empty:
        combined_dupes.append(bad)

bad_combined_dupes = pd.concat(combined_dupes, ignore_index=True) if combined_dupes else pd.DataFrame()

bad_combined_below_core = []

for bucket_name, sc in scorecards.items():
    bad = sc[
        sc["combined_unique_trade_dates"] < int(core_summary["unique_trade_dates"].iloc[0])
    ].copy()

    if not bad.empty:
        bad_combined_below_core.append(bad)

bad_combined_below_core_df = (
    pd.concat(bad_combined_below_core, ignore_index=True)
    if bad_combined_below_core
    else pd.DataFrame()
)

bad_normalized_pnl = base[
    base["normalized_pnl_dollars"].notna()
    & base["normalized_pnl_per_day"].notna()
    & (
        (
            base["normalized_pnl_dollars"] / base["tenor"].replace(0, np.nan)
        )
        - base["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
]

if not triplet_consistency_screen.empty:
    bad_triplet_middle_rule = triplet_consistency_screen[
        ~triplet_consistency_screen["middle_between_front_back_all_params"].astype(bool)
    ]
else:
    bad_triplet_middle_rule = pd.DataFrame()

add_validation(
    validation_rows,
    "base_panel_loaded",
    "PASS" if len(base) > 0 else "FAIL",
    f"rows={len(base):,}; path={base_panel_path}",
)

add_validation(
    validation_rows,
    "required_base_columns_available",
    "PASS" if not missing_base_cols else "FAIL",
    f"missing={missing_base_cols}",
)

add_validation(
    validation_rows,
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    validation_rows,
    "locked_core_rules_loaded",
    "PASS" if len(locked_core_rules_long) == 9 else "FAIL",
    f"rows={len(locked_core_rules_long):,}; path={locked_core_rules_long_path}",
)

add_validation(
    validation_rows,
    "core_gate_dates_created",
    "PASS" if len(core_gate_dates) > 0 and len(core_empty_dates) > 0 else "FAIL",
    f"core_gate_dates={len(core_gate_dates):,}; core_empty_dates={len(core_empty_dates):,}; total_dates={total_eligible_signal_dates:,}",
)

add_validation(
    validation_rows,
    "candidate_counts",
    "PASS" if not bad_candidate_counts else "FAIL",
    f"expected={expected_counts}; actual={actual_counts}; bad={bad_candidate_counts}",
)

add_validation(
    validation_rows,
    "z3m_equals_z1y_all_candidates",
    "PASS" if bad_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_z_equality):,}",
)

add_validation(
    validation_rows,
    "rv21d_threshold_contract",
    "PASS" if bad_rv21d_contract.empty else "FAIL",
    f"bad_rows={len(bad_rv21d_contract):,}",
)

add_validation(
    validation_rows,
    "forecast_vol_pct_diagnostic_only",
    "PASS" if bad_forecast_threshold.empty else "FAIL",
    f"bad_rows={len(bad_forecast_threshold):,}",
)

add_validation(
    validation_rows,
    "secondary_only_on_core_empty_dates",
    "PASS" if bad_secondary_on_core_gate.empty else "FAIL",
    f"bad_rows={len(bad_secondary_on_core_gate):,}",
)

add_validation(
    validation_rows,
    "bucket_tenors_only",
    "PASS" if bad_tenors_df.empty else "FAIL",
    f"bad_rows={len(bad_tenors_df):,}",
)

add_validation(
    validation_rows,
    "one_trade_per_bucket_candidate_date",
    "PASS" if bad_dupes.empty else "FAIL",
    f"bad_rows={len(bad_dupes):,}",
)

add_validation(
    validation_rows,
    "combined_core_plus_bucket_one_trade_per_candidate_date",
    "PASS" if bad_combined_dupes.empty else "FAIL",
    f"bad_rows={len(bad_combined_dupes):,}",
)

add_validation(
    validation_rows,
    "combined_candidates_not_below_core_date_count",
    "PASS" if bad_combined_below_core_df.empty else "FAIL",
    f"bad_rows={len(bad_combined_below_core_df):,}",
)

add_validation(
    validation_rows,
    "triplet_screen_middle_between_front_back",
    "PASS" if bad_triplet_middle_rule.empty else "FAIL",
    f"bad_rows={len(bad_triplet_middle_rule):,}; screen_rows={len(triplet_consistency_screen):,}; meta={triplet_screen_meta}",
)

add_validation(
    validation_rows,
    "no_cross_bucket_trade_selection",
    "PASS",
    "Front, Middle, and Back are evaluated independently. No stack-level Secondary selection is performed.",
)

add_validation(
    validation_rows,
    "no_secondary_lock",
    "PASS",
    "No Secondary parameters are locked in this cell.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed or added.",
)

add_validation(
    validation_rows,
    "no_production_lock",
    "PASS",
    "No production/intraday implementation performed.",
)

add_validation(
    validation_rows,
    "normalized_pnl_per_day_formula",
    "PASS" if bad_normalized_pnl.empty else "FAIL",
    f"bad_rows={len(bad_normalized_pnl):,}",
)

validation = pd.DataFrame(validation_rows)

paths["validation_csv"] = OUT_AUDIT_DIR / f"15R_secondary_independent_bucket_recalibration_validation_{CELL15R_TIMESTAMP}.csv"
validation.to_csv(paths["validation_csv"], index=False)

# ======================================================================================
# 9. Manifest
# ======================================================================================

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1_RV21D_repaired_v2.ipynb",
    "cell": "Cell 15R — Independent Secondary bucket recalibration",
    "timestamp": CELL15R_TIMESTAMP,

    "locked_core_version": LOCKED_CORE_VERSION,
    "locked_core_decision_id": LOCKED_CORE_DECISION_ID,
    "secondary_independent_sweep_version": SECONDARY_INDEPENDENT_SWEEP_VERSION,

    "base_panel_path": str(base_panel_path),
    "locked_core_rules_long_path": str(locked_core_rules_long_path),
    "locked_core_rules_wide_path": str(locked_core_rules_wide_path),

    "total_eligible_signal_dates": int(total_eligible_signal_dates),
    "core_gate_dates": int(len(core_gate_dates)),
    "core_empty_dates": int(len(core_empty_dates)),

    "candidate_counts": actual_counts,
    "expected_candidate_counts": expected_counts,

    "bucket_specs": BUCKET_SPECS,

    "quality_screen_settings": {
        "combined_win_rate_target": COMBINED_WIN_RATE_TARGET,
        "incremental_secondary_win_rate_min": INCREMENTAL_SECONDARY_WIN_RATE_MIN,
        "incremental_secondary_win_rate_ideal": INCREMENTAL_SECONDARY_WIN_RATE_IDEAL,
        "incremental_avg_pnl_per_day_min": INCREMENTAL_AVG_PNL_PER_DAY_MIN,
        "combined_avg_pnl_per_day_min": COMBINED_AVG_PNL_PER_DAY_MIN,
        "min_incremental_trades_for_quality_screen": MIN_INCREMENTAL_TRADES_FOR_QUALITY_SCREEN,
        "max_triplet_screen_rows": MAX_TRIPLET_SCREEN_ROWS,
    },

    "triplet_screen_meta": triplet_screen_meta,

    "selection_rules": {
        "core_gate": "If any locked Core tenor qualifies on a date, Secondary is ignored.",
        "secondary_test_dates": "Core-empty dates only.",
        "bucket_independent_selection": "At most one Secondary trade per bucket/candidate/date within each independent bucket.",
        "no_cross_bucket_selection": "No Front/Middle/Back stack selection in this cell.",
        "triplet_consistency_screen": "Parameter-only screen; does not select trades across buckets.",
    },

    "paths": {k: str(v) for k, v in paths.items()},

    "constraints": [
        "Core remains locked and unchanged.",
        "Secondary Front evaluated independently.",
        "Secondary Middle evaluated independently.",
        "Secondary Back evaluated independently.",
        "No cross-bucket Secondary selection.",
        "No Secondary lock.",
        "9D remains excluded.",
        "18D remains in Front for locked Core map.",
        "z3m equals z1y in this sweep.",
        "RV21D threshold contract is rv21d_vol_pct > rv21d_vol_pct_min.",
        "forecast_vol_pct is diagnostic only.",
        "No sizing changes.",
        "No production/intraday implementation.",
    ],
}

paths["manifest_json"] = OUT_AUDIT_DIR / f"15R_secondary_independent_bucket_recalibration_manifest_{CELL15R_TIMESTAMP}.json"

with open(paths["manifest_json"], "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

failures = validation[validation["status"].eq("FAIL")]

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 15R validation failed. Review validation output above.")

# ======================================================================================
# 10. Display review
# ======================================================================================

print("=" * 100)
print("Cell 15R — Independent Secondary bucket recalibration")
print("=" * 100)
print(f"Base panel:                         {base_panel_path}")
print(f"Locked Core rules:                  {locked_core_rules_long_path}")
print(f"Total eligible signal dates:        {total_eligible_signal_dates:,}")
print(f"Core gate dates:                    {len(core_gate_dates):,}")
print(f"Core-empty dates:                   {len(core_empty_dates):,}")
print()
print("Candidate counts:")
for bucket_name in ["Front", "Middle", "Back"]:
    print(f"  {bucket_name:<8} actual={actual_counts[bucket_name]:,} expected={expected_counts[bucket_name]:,}")

print()
print("Core program baseline:")
display(core_summary)

print()
print("Validation:")
display(validation)

for bucket_name in ["Front", "Middle", "Back"]:
    print()
    print("=" * 100)
    print(f"Secondary {bucket_name} — Independent scorecard")
    print("=" * 100)

    scorecard_cols = [
        "secondary_bucket",
        "secondary_candidate_id",
        "param_model_vrp_log_min",
        "param_model_vrp_z_min",
        "param_RSI14_max",
        "param_rv21d_vol_pct_min",

        "incremental_trades",
        "incremental_unique_trade_dates",
        "incremental_signal_date_frequency",
        "incremental_win_rate",
        "incremental_total_pnl",
        "incremental_avg_pnl_per_day",
        "incremental_profit_factor_pnl_per_day",
        "incremental_pnl_per_day_drawdown",
        "incremental_worst_20_trade_pnl_per_day_sum",
        "incremental_avg_pnl_per_day_2025",
        "incremental_avg_pnl_per_day_2026",

        "combined_unique_trade_dates",
        "combined_signal_date_frequency",
        "combined_win_rate",
        "combined_total_pnl",
        "combined_avg_pnl_per_day",
        "combined_profit_factor_pnl_per_day",
        "combined_pnl_per_day_drawdown",
        "combined_worst_20_trade_pnl_per_day_sum",
        "combined_avg_pnl_per_day_2025",
        "combined_avg_pnl_per_day_2026",

        "delta_combined_unique_trade_dates_vs_core_only",
        "delta_combined_signal_date_frequency_vs_core_only",
        "delta_combined_win_rate_vs_core_only",
        "delta_combined_total_pnl_vs_core_only",
        "delta_combined_avg_pnl_per_day_vs_core_only",
        "delta_combined_profit_factor_pnl_per_day_vs_core_only",
        "delta_combined_pnl_per_day_drawdown_vs_core_only",
        "delta_combined_worst_20_trade_pnl_per_day_sum_vs_core_only",
        "delta_combined_avg_pnl_per_day_2026_vs_core_only",

        "passes_incremental_win_rate_floor_78pct",
        "passes_incremental_win_rate_ideal_80pct",
        "passes_combined_win_rate_80pct",
        "passes_bucket_quality_floor",
        "passes_bucket_quality_ideal",
        "bucket_score",
    ]

    display(scorecards[bucket_name][[c for c in scorecard_cols if c in scorecards[bucket_name].columns]])

    print()
    print(f"Secondary {bucket_name} — Incremental by tenor:")
    display(bucket_results[bucket_name]["selected_by_tenor"])

    print()
    print(f"Secondary {bucket_name} — Incremental by year:")
    display(bucket_results[bucket_name]["selected_by_year"])

    print()
    print(f"Secondary {bucket_name} — Worst trades by candidate:")
    display(bucket_results[bucket_name]["worst_trades"])

print()
print("=" * 100)
print("Parameter triplet consistency screen — Middle between/equal to Front and Back")
print("=" * 100)
print(f"Front quality candidates:       {triplet_screen_meta['front_quality_candidates']:,}")
print(f"Middle quality candidates:      {triplet_screen_meta['middle_quality_candidates']:,}")
print(f"Back quality candidates:        {triplet_screen_meta['back_quality_candidates']:,}")
print(f"Triplet screen rows:            {triplet_screen_meta['screen_rows']:,}")
print(f"Triplet screen truncated:       {triplet_screen_meta['was_truncated']}")

display(triplet_consistency_screen)

print()
print("Saved outputs:")
for k, p in paths.items():
    print(f"  {k}: {p}")

print("\nCELL 15R PASSED — INDEPENDENT SECONDARY BUCKET RECALIBRATION COMPLETE")

In [ ]:
# ======================================================================================
# Cell 16R-LITE — Secondary Middle/Back finalist grid, Front excluded
# ======================================================================================
#
# Objective:
#   Optimize Secondary using a narrow, representative finalist grid:
#       - Secondary Front fully excluded
#       - Secondary Middle shortlist
#       - Secondary Back shortlist
#
# This is a narrowed replacement for the broad Cell 16R grid.
#
# It does NOT:
#   1. Reopen Core.
#   2. Include Secondary Front.
#   3. Run the full 117 × 235 Middle/Back quality grid.
#   4. Lock Secondary.
#   5. Change sizing.
#   6. Implement production/intraday logic.
#
# Candidate structure:
#   Middle options:
#       NO_MIDDLE
#       M1: log > 0.60, z > 0.60, RSI14 < 76, rv21d > 7.0
#       M2: log > 0.65, z > 0.60, RSI14 < 76, rv21d > 7.0
#       M3: log > 0.70, z > 0.60, RSI14 < 76, rv21d > 7.0
#       M4: log > 0.65, z > 0.60, RSI14 < 75, rv21d > 7.0
#
#   Back options:
#       NO_BACK
#       B1: log > 0.70, z > 0.60, RSI14 < 76, rv21d > 6.5
#       B2: log > 0.70, z > 0.60, RSI14 < 75, rv21d > 6.5
#       B3: log > 0.70, z > 0.60, RSI14 < 76, rv21d > 7.0
#       B4: log > 0.70, z > 0.60, RSI14 < 74, rv21d > 6.5
#       B5: log > 0.65, z > 0.40, RSI14 < 76, rv21d > 6.5
#       B6: log > 0.65, z > 0.30, RSI14 < 76, rv21d > 6.5
#       B7: log > 0.65, z > 0.00, RSI14 < 77, rv21d > 6.5
#       B8: log > 0.70, z > 0.00, RSI14 < 77, rv21d > 6.5
#
# Selection modes:
#   BACK_FIRST:
#       If Back qualifies, select Back.
#       Else select Middle.
#
#   BEST_SIGNAL_RANK:
#       If both qualify, choose by signal rank:
#           model_vrp_z_1y
#           model_vrp_z_3m
#           model_vrp_log
#       tie-breakers:
#           closer to bucket center
#           Back before Middle
#           tenor
#
# Expected structural candidates:
#   5 Middle options × 9 Back options - NO_MIDDLE/NO_BACK empty pair = 44 pairs
#   44 pairs × 2 selection modes = 88 pair-mode candidates
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import itertools
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 700)
pd.set_option("display.width", 1000)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL16R_LITE_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

LOCKED_MODEL_SPEC = "unified_fds_no_min_return"
LOCKED_CORE_VERSION = "core_repaired_v1"
LOCKED_CORE_DECISION_ID = "core_repaired_v1_locked_001"

SECONDARY_MB_LITE_VERSION = "secondary_middle_back_lite_front_excluded_v1"

CORE_PROGRAM_UNIVERSE = "core_program_one_trade_per_date"
PRIMARY_CORE_CONTINUITY_UNIVERSE = "core_primary_one_trade_per_bucket_per_date"

SECONDARY_MB_LITE_INCREMENTAL_UNIVERSE = "secondary_middle_back_lite_incremental_core_empty_dates"
COMBINED_MB_LITE_PROGRAM_UNIVERSE = "combined_program_core_first_else_secondary_middle_back_lite"

SELECTION_MODES = ["BACK_FIRST", "BEST_SIGNAL_RANK"]

SECONDARY_FRONT_EXCLUDED = True
SECONDARY_ALLOWED_BUCKETS = ["Middle", "Back"]
SECONDARY_ALLOWED_TENORS = [21, 24, 27, 30, 33]
SECONDARY_EXCLUDED_FRONT_TENORS = [12, 15, 18]

COMBINED_WIN_RATE_TARGET = 0.80
SECONDARY_WIN_RATE_FLOOR = 0.78
SECONDARY_WIN_RATE_IDEAL = 0.80
SECONDARY_AVG_PNL_PER_DAY_MIN = 0.0
COMBINED_AVG_PNL_PER_DAY_MIN = 0.0
MIN_SECONDARY_TRADES = 10

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)

    if matches:
        return matches[0]

    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")

    return None


def normalize_bool_series(s):
    if s.dtype == bool:
        return s.fillna(False)

    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y"])
    )


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)

    if gross_profit > 0:
        return np.inf

    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def summarize_trade_set(df, group_cols, total_eligible_dates=None):
    d = df.copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    for c in [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "forecast_vol_pct",
        "rv21d_vol_pct",
        "threshold_model_vrp_log",
        "threshold_model_vrp_z_3m",
        "threshold_model_vrp_z_1y",
        "threshold_RSI14_max",
        "threshold_rv21d_vol_pct_min",
        "secondary_threshold_model_vrp_log",
        "secondary_threshold_model_vrp_z_3m",
        "secondary_threshold_model_vrp_z_1y",
        "secondary_threshold_RSI14_max",
        "secondary_threshold_rv21d_vol_pct_min",
    ]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d = d.sort_values(["trade_date", "tenor"]).copy()
    grouped = d.groupby(group_cols, dropna=False, observed=False) if group_cols else [((), d)]

    rows = []

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {}

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        unique_dates = int(g["trade_date"].nunique())
        signal_date_frequency = (
            unique_dates / total_eligible_dates
            if total_eligible_dates and total_eligible_dates > 0
            else np.nan
        )

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": unique_dates,
            "signal_date_frequency": float(signal_date_frequency) if pd.notna(signal_date_frequency) else np.nan,
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),

            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,

            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        if "program_leg" in g.columns:
            row["core_trade_count"] = int(g["program_leg"].eq("Core").sum())
            row["secondary_trade_count"] = int(g["program_leg"].str.startswith("Secondary", na=False).sum())
            row["secondary_middle_trade_count"] = int(g["program_leg"].eq("Secondary_Middle").sum())
            row["secondary_back_trade_count"] = int(g["program_leg"].eq("Secondary_Back").sum())

        for signal_col in [
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "threshold_model_vrp_log",
            "threshold_model_vrp_z_3m",
            "threshold_model_vrp_z_1y",
            "threshold_RSI14_max",
            "threshold_rv21d_vol_pct_min",
            "secondary_threshold_model_vrp_log",
            "secondary_threshold_model_vrp_z_3m",
            "secondary_threshold_model_vrp_z_1y",
            "secondary_threshold_RSI14_max",
            "secondary_threshold_rv21d_vol_pct_min",
        ]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        rows.append(row)

    return pd.DataFrame(rows)


def add_rank_columns(d, group_cols, suffix):
    x = d.copy()

    if x.empty:
        return x

    x[f"rank_z_1y_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )

    x[f"rank_z_3m_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )

    x[f"rank_vrp_log_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    x[f"avg_signal_rank_{suffix}"] = x[
        [
            f"rank_z_1y_{suffix}",
            f"rank_z_3m_{suffix}",
            f"rank_vrp_log_{suffix}",
        ]
    ].mean(axis=1)

    if "effective_tenor_bucket" in x.columns:
        x["bucket_center_tenor"] = x["effective_tenor_bucket"].map({
            "Front": 15,
            "Middle": 21,
            "Back": 30,
        })
    elif "secondary_bucket" in x.columns:
        x["bucket_center_tenor"] = x["secondary_bucket"].map({
            "Middle": 21,
            "Back": 30,
        })
    else:
        x["bucket_center_tenor"] = 30

    x["bucket_center_tenor"] = pd.to_numeric(x["bucket_center_tenor"], errors="coerce").fillna(30)
    x["distance_to_bucket_center"] = (x["tenor"] - x["bucket_center_tenor"]).abs()

    return x


def select_core_universes(core_qualified_complete):
    q = core_qualified_complete.copy()

    if q.empty:
        return {
            PRIMARY_CORE_CONTINUITY_UNIVERSE: q.copy(),
            CORE_PROGRAM_UNIVERSE: q.copy(),
        }

    q = add_rank_columns(
        q,
        group_cols=["trade_date", "effective_tenor_bucket"],
        suffix="core_bucket_date",
    )

    q = add_rank_columns(
        q,
        group_cols=["trade_date"],
        suffix="core_date",
    )

    core_primary = (
        q.sort_values(
            [
                "trade_date",
                "effective_tenor_bucket_order",
                "avg_signal_rank_core_bucket_date",
                "distance_to_bucket_center",
                "rank_z_1y_core_bucket_date",
                "rank_z_3m_core_bucket_date",
                "rank_vrp_log_core_bucket_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date", "effective_tenor_bucket"], as_index=False)
        .head(1)
        .copy()
    )

    core_program = (
        q.sort_values(
            [
                "trade_date",
                "avg_signal_rank_core_date",
                "rank_z_1y_core_date",
                "rank_z_3m_core_date",
                "rank_vrp_log_core_date",
                "distance_to_bucket_center",
                "effective_tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return {
        PRIMARY_CORE_CONTINUITY_UNIVERSE: core_primary,
        CORE_PROGRAM_UNIVERSE: core_program,
    }


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


def approx_match(df, col, value, tol=1e-10):
    return np.isclose(pd.to_numeric(df[col], errors="coerce"), float(value), atol=tol, rtol=0)


def find_scorecard_candidate(scorecards, bucket, log_min, z_min, rsi_max, rv_min, label):
    subset = scorecards[
        scorecards["secondary_bucket"].eq(bucket)
        & approx_match(scorecards, "param_model_vrp_log_min", log_min)
        & approx_match(scorecards, "param_model_vrp_z_min", z_min)
        & approx_match(scorecards, "param_RSI14_max", rsi_max)
        & approx_match(scorecards, "param_rv21d_vol_pct_min", rv_min)
    ].copy()

    if subset.empty:
        raise ValueError(
            f"No Cell 15R scorecard candidate found for {label}: "
            f"{bucket}, log>{log_min}, z>{z_min}, RSI<{rsi_max}, rv>{rv_min}"
        )

    if len(subset) > 1:
        subset = subset.sort_values(
            [
                "passes_bucket_quality_ideal",
                "passes_bucket_quality_floor",
                "incremental_win_rate",
                "combined_win_rate",
                "incremental_avg_pnl_per_day",
            ],
            ascending=[False, False, False, False, False],
        )

    row = subset.iloc[0].to_dict()

    return row


def build_option_table(scorecards, specs, option_type):
    rows = []

    for spec in specs:
        if spec["option_id"].startswith("NO_"):
            rows.append({
                f"{option_type}_option_id": spec["option_id"],
                f"{option_type}_candidate_id": None,
                f"{option_type}_description": spec["description"],
                f"{option_type}_is_none": True,
                f"{option_type}_param_model_vrp_log_min": np.nan,
                f"{option_type}_param_model_vrp_z_min": np.nan,
                f"{option_type}_param_RSI14_max": np.nan,
                f"{option_type}_param_rv21d_vol_pct_min": np.nan,
            })
            continue

        row = find_scorecard_candidate(
            scorecards=scorecards,
            bucket=spec["bucket"],
            log_min=spec["log_min"],
            z_min=spec["z_min"],
            rsi_max=spec["rsi_max"],
            rv_min=spec["rv_min"],
            label=spec["option_id"],
        )

        rows.append({
            f"{option_type}_option_id": spec["option_id"],
            f"{option_type}_candidate_id": row["secondary_candidate_id"],
            f"{option_type}_description": spec["description"],
            f"{option_type}_is_none": False,

            f"{option_type}_param_model_vrp_log_min": row["param_model_vrp_log_min"],
            f"{option_type}_param_model_vrp_z_min": row["param_model_vrp_z_min"],
            f"{option_type}_param_RSI14_max": row["param_RSI14_max"],
            f"{option_type}_param_rv21d_vol_pct_min": row["param_rv21d_vol_pct_min"],

            f"{option_type}_incremental_trades": row.get("incremental_trades", np.nan),
            f"{option_type}_incremental_win_rate": row.get("incremental_win_rate", np.nan),
            f"{option_type}_incremental_signal_date_frequency": row.get("incremental_signal_date_frequency", np.nan),
            f"{option_type}_incremental_avg_pnl_per_day": row.get("incremental_avg_pnl_per_day", np.nan),
            f"{option_type}_incremental_profit_factor_pnl_per_day": row.get("incremental_profit_factor_pnl_per_day", np.nan),
            f"{option_type}_incremental_avg_pnl_per_day_2026": row.get("incremental_avg_pnl_per_day_2026", np.nan),

            f"{option_type}_combined_win_rate": row.get("combined_win_rate", np.nan),
            f"{option_type}_combined_signal_date_frequency": row.get("combined_signal_date_frequency", np.nan),
            f"{option_type}_combined_avg_pnl_per_day": row.get("combined_avg_pnl_per_day", np.nan),
            f"{option_type}_combined_profit_factor_pnl_per_day": row.get("combined_profit_factor_pnl_per_day", np.nan),
            f"{option_type}_combined_avg_pnl_per_day_2026": row.get("combined_avg_pnl_per_day_2026", np.nan),

            f"{option_type}_passes_bucket_quality_floor": row.get("passes_bucket_quality_floor", False),
            f"{option_type}_passes_bucket_quality_ideal": row.get("passes_bucket_quality_ideal", False),
            f"{option_type}_bucket_score": row.get("bucket_score", np.nan),
        })

    return pd.DataFrame(rows)


def select_best_signal(pool):
    if pool.empty:
        return pool

    q = add_rank_columns(
        pool,
        group_cols=["secondary_mb_lite_pair_id", "selection_mode", "trade_date"],
        suffix="mb_lite_pair_date",
    )

    q["bucket_priority_best_signal"] = q["secondary_bucket"].map({
        "Back": 1,
        "Middle": 2,
    }).fillna(9)

    return (
        q.sort_values(
            [
                "secondary_mb_lite_pair_id",
                "selection_mode",
                "trade_date",
                "avg_signal_rank_mb_lite_pair_date",
                "distance_to_bucket_center",
                "bucket_priority_best_signal",
                "rank_z_1y_mb_lite_pair_date",
                "rank_z_3m_mb_lite_pair_date",
                "rank_vrp_log_mb_lite_pair_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True, True, True],
        )
        .groupby(["secondary_mb_lite_pair_id", "selection_mode", "trade_date"], as_index=False)
        .head(1)
        .copy()
    )


# ======================================================================================
# 2. Load inputs
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01R_unified_fds_signal_base_panel_with_rv21d_*.parquet",
    required=True,
)

locked_core_rules_long_path = latest_file(
    OUT_PROCESSED_DIR,
    "11R_core_repaired_v1_locked_rules_long_*.parquet",
    required=True,
)

locked_core_rules_wide_path = latest_file(
    OUT_PROCESSED_DIR,
    "11R_core_repaired_v1_locked_rules_wide_*.parquet",
    required=True,
)

cell15_selected_all_path = latest_file(
    OUT_PROCESSED_DIR,
    "15R_secondary_independent_selected_panel_all_*.parquet",
    required=True,
)

cell15_scorecards_all_path = latest_file(
    OUT_AUDIT_DIR,
    "15R_secondary_independent_scorecards_all_*.csv",
    required=True,
)

base = pd.read_parquet(base_panel_path)
locked_core_rules_long = pd.read_parquet(locked_core_rules_long_path)
locked_core_rules_wide = pd.read_parquet(locked_core_rules_wide_path)

cell15_selected_all = pd.read_parquet(cell15_selected_all_path)
cell15_scorecards_all = pd.read_csv(cell15_scorecards_all_path)

base["trade_date"] = pd.to_datetime(base["trade_date"], errors="coerce").dt.normalize()
cell15_selected_all["trade_date"] = pd.to_datetime(cell15_selected_all["trade_date"], errors="coerce").dt.normalize()

for c in [
    "tenor",
    "tenor_bucket_order",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")

for c in [
    "tenor",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "param_model_vrp_log_min",
    "param_model_vrp_z_min",
    "param_model_vrp_z_3m_min",
    "param_model_vrp_z_1y_min",
    "param_RSI14_max",
    "param_rv21d_vol_pct_min",
]:
    if c in cell15_selected_all.columns:
        cell15_selected_all[c] = pd.to_numeric(cell15_selected_all[c], errors="coerce")

for c in [
    "param_model_vrp_log_min",
    "param_model_vrp_z_min",
    "param_model_vrp_z_3m_min",
    "param_model_vrp_z_1y_min",
    "param_RSI14_max",
    "param_rv21d_vol_pct_min",
    "incremental_trades",
    "incremental_unique_trade_dates",
    "incremental_signal_date_frequency",
    "incremental_win_rate",
    "incremental_total_pnl",
    "incremental_avg_pnl_per_day",
    "incremental_profit_factor_pnl_per_day",
    "incremental_pnl_per_day_drawdown",
    "incremental_worst_20_trade_pnl_per_day_sum",
    "incremental_avg_pnl_per_day_2025",
    "incremental_avg_pnl_per_day_2026",
    "combined_unique_trade_dates",
    "combined_signal_date_frequency",
    "combined_win_rate",
    "combined_total_pnl",
    "combined_avg_pnl_per_day",
    "combined_profit_factor_pnl_per_day",
    "combined_pnl_per_day_drawdown",
    "combined_worst_20_trade_pnl_per_day_sum",
    "combined_avg_pnl_per_day_2025",
    "combined_avg_pnl_per_day_2026",
    "bucket_score",
]:
    if c in cell15_scorecards_all.columns:
        cell15_scorecards_all[c] = pd.to_numeric(cell15_scorecards_all[c], errors="coerce")

for c in [
    "passes_bucket_quality_floor",
    "passes_bucket_quality_ideal",
]:
    if c in cell15_scorecards_all.columns:
        cell15_scorecards_all[c] = normalize_bool_series(cell15_scorecards_all[c])

for c in [
    "tenor",
    "original_tenor_bucket_order",
    "effective_tenor_bucket_order",
    "model_vrp_log_min",
    "model_vrp_z_min",
    "model_vrp_z_3m_min",
    "model_vrp_z_1y_min",
    "RSI14_max",
    "rv21d_vol_pct_min",
]:
    if c in locked_core_rules_long.columns:
        locked_core_rules_long[c] = pd.to_numeric(locked_core_rules_long[c], errors="coerce")

total_eligible_signal_dates = int(base["trade_date"].nunique())

# ======================================================================================
# 3. Rebuild locked Core gate and Core baseline
# ======================================================================================

core_thresholds = locked_core_rules_long.rename(columns={
    "model_vrp_log_min": "threshold_model_vrp_log",
    "model_vrp_z_3m_min": "threshold_model_vrp_z_3m",
    "model_vrp_z_1y_min": "threshold_model_vrp_z_1y",
    "RSI14_max": "threshold_RSI14_max",
    "rv21d_vol_pct_min": "threshold_rv21d_vol_pct_min",
})

core_threshold_cols = [
    "tenor",
    "original_tenor_bucket",
    "original_tenor_bucket_order",
    "effective_tenor_bucket",
    "effective_tenor_bucket_order",
    "include_tenor",
    "threshold_model_vrp_log",
    "threshold_model_vrp_z_3m",
    "threshold_model_vrp_z_1y",
    "threshold_RSI14_max",
    "threshold_rv21d_vol_pct_min",
]

base_for_join = base.copy()

base_for_join = base_for_join.rename(columns={
    "tenor_bucket": "source_tenor_bucket",
    "tenor_bucket_order": "source_tenor_bucket_order",
})

core_panel = base_for_join.merge(
    core_thresholds[core_threshold_cols],
    on="tenor",
    how="inner",
    validate="many_to_one",
)

core_panel["include_tenor_bool"] = normalize_bool_series(core_panel["include_tenor"])

core_signal_available = (
    core_panel["include_tenor_bool"]
    & core_panel["model_vrp_log"].notna()
    & core_panel["model_vrp_z_3m"].notna()
    & core_panel["model_vrp_z_1y"].notna()
    & core_panel["RSI14"].notna()
    & core_panel["rv21d_vol_pct"].notna()
)

core_pass_mask = (
    core_signal_available
    & (core_panel["model_vrp_log"] > core_panel["threshold_model_vrp_log"])
    & (core_panel["model_vrp_z_3m"] > core_panel["threshold_model_vrp_z_3m"])
    & (core_panel["model_vrp_z_1y"] > core_panel["threshold_model_vrp_z_1y"])
    & (core_panel["RSI14"] < core_panel["threshold_RSI14_max"])
    & (core_panel["rv21d_vol_pct"] > core_panel["threshold_rv21d_vol_pct_min"])
)

core_panel["core_repaired_v1_pass"] = core_pass_mask

core_qualified_all_dates = core_panel[core_pass_mask].copy()

core_qualified_complete = core_panel[
    core_pass_mask
    & core_panel["normalized_pnl_dollars"].notna()
    & core_panel["normalized_pnl_per_day"].notna()
].copy()

core_gate_dates = sorted(pd.to_datetime(core_qualified_all_dates["trade_date"].dropna().unique()).tolist())
all_signal_dates = sorted(pd.to_datetime(base["trade_date"].dropna().unique()).tolist())

core_gate_dates_set = set(core_gate_dates)
core_empty_dates = sorted(set(all_signal_dates) - core_gate_dates_set)

core_universes = select_core_universes(core_qualified_complete)
core_program = core_universes[CORE_PROGRAM_UNIVERSE].copy()

core_program["source_layer"] = "Core"
core_program["program_leg"] = "Core"
core_program["selection_universe"] = CORE_PROGRAM_UNIVERSE

stale_secondary_cols = [
    "secondary_bucket",
    "secondary_bucket_order",
    "secondary_candidate_id",
    "secondary_candidate_description",
    "param_model_vrp_log_min",
    "param_model_vrp_z_min",
    "param_model_vrp_z_3m_min",
    "param_model_vrp_z_1y_min",
    "param_RSI14_max",
    "param_rv21d_vol_pct_min",
]

core_program_clean = core_program.drop(
    columns=[c for c in stale_secondary_cols if c in core_program.columns],
    errors="ignore",
).copy()

core_summary = summarize_trade_set(
    core_program_clean.assign(selection_universe=CORE_PROGRAM_UNIVERSE),
    group_cols=["selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

core_ref = core_summary.iloc[0].to_dict() if len(core_summary) else {}

# ======================================================================================
# 4. Define Middle/Back shortlist
# ======================================================================================

middle_specs = [
    {
        "option_id": "NO_MIDDLE",
        "description": "No Secondary Middle",
    },
    {
        "option_id": "M1_CLEAN_060_060_76_70",
        "bucket": "Middle",
        "description": "Middle clean: log>0.60, z>0.60, RSI<76, rv21d>7.0",
        "log_min": 0.60,
        "z_min": 0.60,
        "rsi_max": 76.0,
        "rv_min": 7.0,
    },
    {
        "option_id": "M2_CLEAN_065_060_76_70",
        "bucket": "Middle",
        "description": "Middle clean higher log: log>0.65, z>0.60, RSI<76, rv21d>7.0",
        "log_min": 0.65,
        "z_min": 0.60,
        "rsi_max": 76.0,
        "rv_min": 7.0,
    },
    {
        "option_id": "M3_TIGHT_070_060_76_70",
        "bucket": "Middle",
        "description": "Middle tight log: log>0.70, z>0.60, RSI<76, rv21d>7.0",
        "log_min": 0.70,
        "z_min": 0.60,
        "rsi_max": 76.0,
        "rv_min": 7.0,
    },
    {
        "option_id": "M4_CLEAN_065_060_75_70",
        "bucket": "Middle",
        "description": "Middle clean tighter RSI: log>0.65, z>0.60, RSI<75, rv21d>7.0",
        "log_min": 0.65,
        "z_min": 0.60,
        "rsi_max": 75.0,
        "rv_min": 7.0,
    },
]

back_specs = [
    {
        "option_id": "NO_BACK",
        "description": "No Secondary Back",
    },
    {
        "option_id": "B1_CLEAN_070_060_76_65",
        "bucket": "Back",
        "description": "Back clean: log>0.70, z>0.60, RSI<76, rv21d>6.5",
        "log_min": 0.70,
        "z_min": 0.60,
        "rsi_max": 76.0,
        "rv_min": 6.5,
    },
    {
        "option_id": "B2_CLEAN_070_060_75_65",
        "bucket": "Back",
        "description": "Back clean tighter RSI: log>0.70, z>0.60, RSI<75, rv21d>6.5",
        "log_min": 0.70,
        "z_min": 0.60,
        "rsi_max": 75.0,
        "rv_min": 6.5,
    },
    {
        "option_id": "B3_CLEAN_070_060_76_70",
        "bucket": "Back",
        "description": "Back clean higher RV floor: log>0.70, z>0.60, RSI<76, rv21d>7.0",
        "log_min": 0.70,
        "z_min": 0.60,
        "rsi_max": 76.0,
        "rv_min": 7.0,
    },
    {
        "option_id": "B4_TIGHT_070_060_74_65",
        "bucket": "Back",
        "description": "Back tight RSI: log>0.70, z>0.60, RSI<74, rv21d>6.5",
        "log_min": 0.70,
        "z_min": 0.60,
        "rsi_max": 74.0,
        "rv_min": 6.5,
    },
    {
        "option_id": "B5_MEDIUM_065_040_76_65",
        "bucket": "Back",
        "description": "Back medium frequency: log>0.65, z>0.40, RSI<76, rv21d>6.5",
        "log_min": 0.65,
        "z_min": 0.40,
        "rsi_max": 76.0,
        "rv_min": 6.5,
    },
    {
        "option_id": "B6_FREQ_065_030_76_65",
        "bucket": "Back",
        "description": "Back looser z frequency: log>0.65, z>0.30, RSI<76, rv21d>6.5",
        "log_min": 0.65,
        "z_min": 0.30,
        "rsi_max": 76.0,
        "rv_min": 6.5,
    },
    {
        "option_id": "B7_FREQ_065_000_77_65",
        "bucket": "Back",
        "description": "Back frequent reference: log>0.65, z>0.00, RSI<77, rv21d>6.5",
        "log_min": 0.65,
        "z_min": 0.00,
        "rsi_max": 77.0,
        "rv_min": 6.5,
    },
    {
        "option_id": "B8_FREQ_070_000_77_65",
        "bucket": "Back",
        "description": "Back higher log, loose z: log>0.70, z>0.00, RSI<77, rv21d>6.5",
        "log_min": 0.70,
        "z_min": 0.00,
        "rsi_max": 77.0,
        "rv_min": 6.5,
    },
]

middle_options = build_option_table(cell15_scorecards_all, middle_specs, "middle")
back_options = build_option_table(cell15_scorecards_all, back_specs, "back")

# ======================================================================================
# 5. Build lite pair grid
# ======================================================================================

pair_grid = middle_options.merge(back_options, how="cross")

pair_grid = pair_grid[
    ~(pair_grid["middle_is_none"] & pair_grid["back_is_none"])
].copy()

pair_grid = pair_grid.reset_index(drop=True)

pair_grid["secondary_mb_lite_pair_id"] = [
    f"secondary_mb_lite_pair_{i + 1:04d}" for i in range(len(pair_grid))
]

pair_grid["secondary_mb_lite_version"] = SECONDARY_MB_LITE_VERSION
pair_grid["secondary_front_excluded"] = True

pair_grid["pair_description"] = (
    pair_grid["middle_option_id"]
    + " + "
    + pair_grid["back_option_id"]
)

pair_mode_grid = pair_grid.merge(
    pd.DataFrame({"selection_mode": SELECTION_MODES}),
    how="cross",
)

pair_mode_grid["secondary_mb_lite_pair_mode_id"] = (
    pair_mode_grid["secondary_mb_lite_pair_id"]
    + "__"
    + pair_mode_grid["selection_mode"]
)

expected_pair_count = 44
expected_pair_mode_count = 88

# ======================================================================================
# 6. Prepare Cell 15R selected Middle/Back rows
# ======================================================================================

selected_cols = [
    "trade_date",
    "tenor",
    "source_tenor_bucket",
    "source_tenor_bucket_order",
    "model_spec",

    "implied_variance",
    "forecast_variance",
    "target_realized_variance",
    "implied_vol_pct",
    "forecast_vol_pct",
    "rv21d_vol_pct",

    "model_vrp_log",
    "model_vrp_ratio",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",

    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",

    "secondary_bucket",
    "secondary_bucket_order",
    "bucket_center_tenor",
    "secondary_candidate_id",
    "secondary_candidate_description",
    "secondary_tenors",

    "secondary_threshold_model_vrp_log",
    "secondary_threshold_model_vrp_z_3m",
    "secondary_threshold_model_vrp_z_1y",
    "secondary_threshold_RSI14_max",
    "secondary_threshold_rv21d_vol_pct_min",

    "param_model_vrp_log_min",
    "param_model_vrp_z_min",
    "param_model_vrp_z_3m_min",
    "param_model_vrp_z_1y_min",
    "param_RSI14_max",
    "param_rv21d_vol_pct_min",
]

selected_base = cell15_selected_all[
    [c for c in selected_cols if c in cell15_selected_all.columns]
].copy()

middle_candidate_ids = sorted(
    middle_options["middle_candidate_id"].dropna().unique().tolist()
)

back_candidate_ids = sorted(
    back_options["back_candidate_id"].dropna().unique().tolist()
)

selected_middle_base = selected_base[
    selected_base["secondary_bucket"].eq("Middle")
    & selected_base["secondary_candidate_id"].isin(middle_candidate_ids)
    & selected_base["trade_date"].isin(core_empty_dates)
].copy()

selected_back_base = selected_base[
    selected_base["secondary_bucket"].eq("Back")
    & selected_base["secondary_candidate_id"].isin(back_candidate_ids)
    & selected_base["trade_date"].isin(core_empty_dates)
].copy()

selected_middle_base = selected_middle_base.rename(columns={
    "secondary_candidate_id": "middle_candidate_id",
})

selected_back_base = selected_back_base.rename(columns={
    "secondary_candidate_id": "back_candidate_id",
})

pair_map = pair_grid[
    [
        "secondary_mb_lite_pair_id",
        "middle_option_id",
        "back_option_id",
        "middle_candidate_id",
        "back_candidate_id",
        "middle_is_none",
        "back_is_none",
    ]
].copy()

middle_pair_rows = selected_middle_base.merge(
    pair_map[
        [
            "secondary_mb_lite_pair_id",
            "middle_option_id",
            "back_option_id",
            "middle_candidate_id",
            "back_candidate_id",
        ]
    ].dropna(subset=["middle_candidate_id"]),
    on="middle_candidate_id",
    how="inner",
    validate="many_to_many",
)

middle_pair_rows["secondary_candidate_id"] = middle_pair_rows["middle_candidate_id"]
middle_pair_rows["program_leg"] = "Secondary_Middle"
middle_pair_rows["source_layer"] = "Secondary"

back_pair_rows = selected_back_base.merge(
    pair_map[
        [
            "secondary_mb_lite_pair_id",
            "middle_option_id",
            "back_option_id",
            "middle_candidate_id",
            "back_candidate_id",
        ]
    ].dropna(subset=["back_candidate_id"]),
    on="back_candidate_id",
    how="inner",
    validate="many_to_many",
)

back_pair_rows["secondary_candidate_id"] = back_pair_rows["back_candidate_id"]
back_pair_rows["program_leg"] = "Secondary_Back"
back_pair_rows["source_layer"] = "Secondary"

# ======================================================================================
# 7. Build selected Secondary rows by pair/date/mode
# ======================================================================================

key_cols = [
    "secondary_mb_lite_pair_id",
    "trade_date",
    "middle_option_id",
    "back_option_id",
]

middle_presence = middle_pair_rows[key_cols].drop_duplicates().copy()
middle_presence["middle_present"] = True

back_presence = back_pair_rows[key_cols].drop_duplicates().copy()
back_presence["back_present"] = True

presence = middle_presence.merge(
    back_presence,
    on=key_cols,
    how="outer",
)

presence["middle_present"] = presence["middle_present"].eq(True)
presence["back_present"] = presence["back_present"].eq(True)

# BACK_FIRST mode.
back_first_back_keys = presence[presence["back_present"]][key_cols].copy()
back_first_middle_keys = presence[
    presence["middle_present"] & ~presence["back_present"]
][key_cols].copy()

back_first_back = back_pair_rows.merge(
    back_first_back_keys,
    on=key_cols,
    how="inner",
    validate="one_to_one",
)

back_first_middle = middle_pair_rows.merge(
    back_first_middle_keys,
    on=key_cols,
    how="inner",
    validate="one_to_one",
)

selected_back_first = pd.concat(
    [
        back_first_back,
        back_first_middle,
    ],
    ignore_index=True,
)

selected_back_first["selection_mode"] = "BACK_FIRST"

# BEST_SIGNAL_RANK mode.
best_signal_pool = pd.concat(
    [
        middle_pair_rows,
        back_pair_rows,
    ],
    ignore_index=True,
)

best_signal_pool["selection_mode"] = "BEST_SIGNAL_RANK"

selected_best_signal = select_best_signal(best_signal_pool)

secondary_selected = pd.concat(
    [
        selected_back_first,
        selected_best_signal,
    ],
    ignore_index=True,
)

secondary_selected["secondary_mb_lite_pair_mode_id"] = (
    secondary_selected["secondary_mb_lite_pair_id"]
    + "__"
    + secondary_selected["selection_mode"]
)

secondary_selected["selection_universe"] = SECONDARY_MB_LITE_INCREMENTAL_UNIVERSE
secondary_selected["secondary_front_excluded"] = True

# Attach pair-mode metadata to make outputs self-describing.
pair_mode_meta_cols = [
    "secondary_mb_lite_pair_id",
    "selection_mode",
    "secondary_mb_lite_pair_mode_id",
    "pair_description",
    "middle_option_id",
    "back_option_id",
    "middle_candidate_id",
    "back_candidate_id",
    "middle_description",
    "back_description",
    "middle_param_model_vrp_log_min",
    "middle_param_model_vrp_z_min",
    "middle_param_RSI14_max",
    "middle_param_rv21d_vol_pct_min",
    "back_param_model_vrp_log_min",
    "back_param_model_vrp_z_min",
    "back_param_RSI14_max",
    "back_param_rv21d_vol_pct_min",
    "secondary_front_excluded",
    "secondary_mb_lite_version",
]

secondary_selected = secondary_selected.merge(
    pair_mode_grid[
        [c for c in pair_mode_meta_cols if c in pair_mode_grid.columns]
    ],
    on=[
        "secondary_mb_lite_pair_id",
        "selection_mode",
        "secondary_mb_lite_pair_mode_id",
        "middle_option_id",
        "back_option_id",
        "middle_candidate_id",
        "back_candidate_id",
    ],
    how="left",
    validate="many_to_one",
    suffixes=("", "_pair_meta"),
)

# ======================================================================================
# 8. Summaries
# ======================================================================================

secondary_summary = summarize_trade_set(
    secondary_selected,
    group_cols=[
        "secondary_mb_lite_pair_id",
        "selection_mode",
        "secondary_mb_lite_pair_mode_id",
        "selection_universe",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

secondary_by_bucket = summarize_trade_set(
    secondary_selected,
    group_cols=[
        "secondary_mb_lite_pair_id",
        "selection_mode",
        "secondary_mb_lite_pair_mode_id",
        "secondary_bucket",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

secondary_by_tenor = summarize_trade_set(
    secondary_selected,
    group_cols=[
        "secondary_mb_lite_pair_id",
        "selection_mode",
        "secondary_mb_lite_pair_mode_id",
        "secondary_bucket",
        "tenor",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

secondary_by_year_input = secondary_selected.assign(
    year=secondary_selected["trade_date"].dt.year.astype(int)
)

secondary_by_year = summarize_trade_set(
    secondary_by_year_input,
    group_cols=[
        "secondary_mb_lite_pair_id",
        "selection_mode",
        "secondary_mb_lite_pair_mode_id",
        "year",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

secondary_by_bucket_year = summarize_trade_set(
    secondary_by_year_input,
    group_cols=[
        "secondary_mb_lite_pair_id",
        "selection_mode",
        "secondary_mb_lite_pair_mode_id",
        "secondary_bucket",
        "year",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

# ======================================================================================
# 9. Combined program: Core first, else Secondary Middle/Back Lite
# ======================================================================================

core_program_pair_mode = core_program_clean.merge(
    pair_mode_grid[
        [
            "secondary_mb_lite_pair_id",
            "selection_mode",
            "secondary_mb_lite_pair_mode_id",
            "middle_option_id",
            "back_option_id",
            "middle_candidate_id",
            "back_candidate_id",
        ]
    ],
    how="cross",
)

core_program_pair_mode["source_layer"] = "Core"
core_program_pair_mode["program_leg"] = "Core"
core_program_pair_mode["selection_universe"] = COMBINED_MB_LITE_PROGRAM_UNIVERSE
core_program_pair_mode["secondary_front_excluded"] = True

secondary_for_combined = secondary_selected.copy()
secondary_for_combined["selection_universe"] = COMBINED_MB_LITE_PROGRAM_UNIVERSE

combined_program_panel = pd.concat(
    [
        core_program_pair_mode,
        secondary_for_combined,
    ],
    ignore_index=True,
)

combined_program_summary = summarize_trade_set(
    combined_program_panel,
    group_cols=[
        "secondary_mb_lite_pair_id",
        "selection_mode",
        "secondary_mb_lite_pair_mode_id",
        "selection_universe",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

combined_program_by_year_input = combined_program_panel.assign(
    year=combined_program_panel["trade_date"].dt.year.astype(int)
)

combined_program_by_year = summarize_trade_set(
    combined_program_by_year_input,
    group_cols=[
        "secondary_mb_lite_pair_id",
        "selection_mode",
        "secondary_mb_lite_pair_mode_id",
        "year",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

# ======================================================================================
# 10. Candidate comparison
# ======================================================================================

metadata_cols = [
    "secondary_mb_lite_pair_id",
    "selection_mode",
    "secondary_mb_lite_pair_mode_id",

    "pair_description",

    "middle_option_id",
    "middle_candidate_id",
    "middle_description",
    "middle_is_none",
    "middle_param_model_vrp_log_min",
    "middle_param_model_vrp_z_min",
    "middle_param_RSI14_max",
    "middle_param_rv21d_vol_pct_min",
    "middle_incremental_trades",
    "middle_incremental_win_rate",
    "middle_incremental_signal_date_frequency",
    "middle_incremental_avg_pnl_per_day",
    "middle_incremental_profit_factor_pnl_per_day",
    "middle_incremental_avg_pnl_per_day_2026",
    "middle_combined_win_rate",
    "middle_combined_avg_pnl_per_day",
    "middle_combined_avg_pnl_per_day_2026",
    "middle_passes_bucket_quality_floor",
    "middle_passes_bucket_quality_ideal",
    "middle_bucket_score",

    "back_option_id",
    "back_candidate_id",
    "back_description",
    "back_is_none",
    "back_param_model_vrp_log_min",
    "back_param_model_vrp_z_min",
    "back_param_RSI14_max",
    "back_param_rv21d_vol_pct_min",
    "back_incremental_trades",
    "back_incremental_win_rate",
    "back_incremental_signal_date_frequency",
    "back_incremental_avg_pnl_per_day",
    "back_incremental_profit_factor_pnl_per_day",
    "back_incremental_avg_pnl_per_day_2026",
    "back_combined_win_rate",
    "back_combined_avg_pnl_per_day",
    "back_combined_avg_pnl_per_day_2026",
    "back_passes_bucket_quality_floor",
    "back_passes_bucket_quality_ideal",
    "back_bucket_score",

    "secondary_front_excluded",
    "secondary_mb_lite_version",
]

metadata = pair_mode_grid[
    [c for c in metadata_cols if c in pair_mode_grid.columns]
].copy()

join_keys = [
    "secondary_mb_lite_pair_id",
    "selection_mode",
    "secondary_mb_lite_pair_mode_id",
]

secondary_metrics = secondary_summary.copy().rename(columns={
    "trades": "secondary_trades",
    "unique_trade_dates": "secondary_unique_trade_dates",
    "signal_date_frequency": "secondary_signal_date_frequency",
    "win_rate": "secondary_win_rate",
    "total_pnl": "secondary_total_pnl",
    "avg_pnl": "secondary_avg_pnl",
    "avg_pnl_per_day": "secondary_avg_pnl_per_day",
    "profit_factor": "secondary_profit_factor",
    "profit_factor_pnl_per_day": "secondary_profit_factor_pnl_per_day",
    "pnl_per_day_drawdown": "secondary_pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum": "secondary_worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025": "secondary_avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026": "secondary_avg_pnl_per_day_2026",
    "secondary_middle_trade_count": "selected_middle_trade_count",
    "secondary_back_trade_count": "selected_back_trade_count",
})

combined_metrics = combined_program_summary.copy().rename(columns={
    "trades": "combined_trades",
    "unique_trade_dates": "combined_unique_trade_dates",
    "signal_date_frequency": "combined_signal_date_frequency",
    "win_rate": "combined_win_rate",
    "total_pnl": "combined_total_pnl",
    "avg_pnl": "combined_avg_pnl",
    "avg_pnl_per_day": "combined_avg_pnl_per_day",
    "profit_factor": "combined_profit_factor",
    "profit_factor_pnl_per_day": "combined_profit_factor_pnl_per_day",
    "pnl_per_day_drawdown": "combined_pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum": "combined_worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025": "combined_avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026": "combined_avg_pnl_per_day_2026",
    "core_trade_count": "combined_core_trade_count",
    "secondary_trade_count": "combined_secondary_trade_count",
    "secondary_middle_trade_count": "combined_secondary_middle_trade_count",
    "secondary_back_trade_count": "combined_secondary_back_trade_count",
})

comparison = metadata.merge(
    secondary_metrics[[c for c in secondary_metrics.columns if c != "selection_universe"]],
    on=join_keys,
    how="left",
    validate="one_to_one",
).merge(
    combined_metrics[[c for c in combined_metrics.columns if c != "selection_universe"]],
    on=join_keys,
    how="left",
    validate="one_to_one",
)

for metric in [
    "trades",
    "unique_trade_dates",
    "signal_date_frequency",
    "win_rate",
    "total_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "profit_factor_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
]:
    combined_col = f"combined_{metric}"

    if combined_col in comparison.columns and metric in core_ref:
        comparison[f"delta_combined_{metric}_vs_core_only"] = comparison[combined_col] - core_ref[metric]

comparison["passes_combined_win_rate_80pct"] = comparison["combined_win_rate"] >= COMBINED_WIN_RATE_TARGET
comparison["passes_secondary_win_rate_floor_78pct"] = comparison["secondary_win_rate"] >= SECONDARY_WIN_RATE_FLOOR
comparison["passes_secondary_win_rate_ideal_80pct"] = comparison["secondary_win_rate"] >= SECONDARY_WIN_RATE_IDEAL
comparison["passes_secondary_positive_avg_day"] = comparison["secondary_avg_pnl_per_day"] > SECONDARY_AVG_PNL_PER_DAY_MIN
comparison["passes_combined_positive_avg_day"] = comparison["combined_avg_pnl_per_day"] > COMBINED_AVG_PNL_PER_DAY_MIN
comparison["passes_min_secondary_trades"] = comparison["secondary_trades"] >= MIN_SECONDARY_TRADES

comparison["passes_hard_screen"] = (
    comparison["passes_combined_win_rate_80pct"]
    & comparison["passes_secondary_win_rate_floor_78pct"]
    & comparison["passes_secondary_positive_avg_day"]
    & comparison["passes_combined_positive_avg_day"]
    & comparison["passes_min_secondary_trades"]
)

comparison["passes_ideal_screen"] = (
    comparison["passes_combined_win_rate_80pct"]
    & comparison["passes_secondary_win_rate_ideal_80pct"]
    & comparison["passes_secondary_positive_avg_day"]
    & comparison["passes_combined_positive_avg_day"]
    & comparison["passes_min_secondary_trades"]
)

comparison["middle_share_of_secondary_trades"] = (
    comparison["selected_middle_trade_count"] / comparison["secondary_trades"].replace(0, np.nan)
)

comparison["back_share_of_secondary_trades"] = (
    comparison["selected_back_trade_count"] / comparison["secondary_trades"].replace(0, np.nan)
)

comparison["candidate_score"] = (
    comparison["passes_ideal_screen"].astype(int) * 100_000
    + comparison["passes_hard_screen"].astype(int) * 10_000
    + comparison["secondary_win_rate"].fillna(0) * 1_000
    + comparison["combined_win_rate"].fillna(0) * 500
    + comparison["secondary_signal_date_frequency"].fillna(0) * 250
    + np.clip(comparison["secondary_avg_pnl_per_day"].fillna(-10_000), -10_000, 10_000) / 10
    + np.clip(comparison["combined_avg_pnl_per_day"].fillna(-10_000), -10_000, 10_000) / 20
    + np.clip(comparison["combined_avg_pnl_per_day_2026"].fillna(-10_000), -10_000, 10_000) / 20
)

comparison = comparison.sort_values(
    [
        "passes_ideal_screen",
        "passes_hard_screen",
        "secondary_win_rate",
        "combined_win_rate",
        "secondary_signal_date_frequency",
        "secondary_avg_pnl_per_day",
        "combined_avg_pnl_per_day",
        "combined_avg_pnl_per_day_2026",
        "secondary_avg_pnl_per_day_2026",
        "combined_profit_factor_pnl_per_day",
        "candidate_score",
    ],
    ascending=[False, False, False, False, False, False, False, False, False, False, False],
).reset_index(drop=True)

comparison["comparison_rank"] = np.arange(1, len(comparison) + 1)

preferred_pair_mode_id = comparison.iloc[0]["secondary_mb_lite_pair_mode_id"] if len(comparison) else None

preferred_secondary_panel = secondary_selected[
    secondary_selected["secondary_mb_lite_pair_mode_id"].eq(preferred_pair_mode_id)
].copy()

preferred_combined_panel = combined_program_panel[
    combined_program_panel["secondary_mb_lite_pair_mode_id"].eq(preferred_pair_mode_id)
].copy()

preferred_secondary_by_bucket = secondary_by_bucket[
    secondary_by_bucket["secondary_mb_lite_pair_mode_id"].eq(preferred_pair_mode_id)
].copy()

preferred_secondary_by_tenor = secondary_by_tenor[
    secondary_by_tenor["secondary_mb_lite_pair_mode_id"].eq(preferred_pair_mode_id)
].copy()

preferred_secondary_by_year = secondary_by_year[
    secondary_by_year["secondary_mb_lite_pair_mode_id"].eq(preferred_pair_mode_id)
].copy()

preferred_combined_by_year = combined_program_by_year[
    combined_program_by_year["secondary_mb_lite_pair_mode_id"].eq(preferred_pair_mode_id)
].copy()

worst_trades = (
    secondary_selected
    .sort_values(
        ["secondary_mb_lite_pair_mode_id", "normalized_pnl_per_day", "normalized_pnl_dollars"],
        ascending=[True, True, True],
    )
    .groupby("secondary_mb_lite_pair_mode_id", as_index=False)
    .head(20)
    .copy()
)

# ======================================================================================
# 11. Save outputs
# ======================================================================================

paths = {}

paths["middle_options_csv"] = OUT_AUDIT_DIR / f"16R_LITE_middle_options_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"
paths["back_options_csv"] = OUT_AUDIT_DIR / f"16R_LITE_back_options_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"

paths["pair_grid_csv"] = OUT_AUDIT_DIR / f"16R_LITE_middle_back_pair_grid_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"
paths["pair_mode_grid_csv"] = OUT_AUDIT_DIR / f"16R_LITE_middle_back_pair_mode_grid_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"

paths["core_program_summary_csv"] = OUT_AUDIT_DIR / f"16R_LITE_core_program_baseline_{CELL16R_LITE_TIMESTAMP}.csv"
paths["secondary_selected_panel_parquet"] = OUT_PROCESSED_DIR / f"16R_LITE_secondary_middle_back_selected_panel_front_excluded_{CELL16R_LITE_TIMESTAMP}.parquet"
paths["secondary_selected_panel_csv"] = OUT_AUDIT_DIR / f"16R_LITE_secondary_middle_back_selected_panel_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"

paths["combined_program_panel_parquet"] = OUT_PROCESSED_DIR / f"16R_LITE_combined_program_panel_front_excluded_{CELL16R_LITE_TIMESTAMP}.parquet"
paths["combined_program_panel_csv"] = OUT_AUDIT_DIR / f"16R_LITE_combined_program_panel_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"

paths["secondary_summary_csv"] = OUT_AUDIT_DIR / f"16R_LITE_secondary_incremental_summary_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"
paths["secondary_by_bucket_csv"] = OUT_AUDIT_DIR / f"16R_LITE_secondary_incremental_by_bucket_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"
paths["secondary_by_tenor_csv"] = OUT_AUDIT_DIR / f"16R_LITE_secondary_incremental_by_tenor_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"
paths["secondary_by_year_csv"] = OUT_AUDIT_DIR / f"16R_LITE_secondary_incremental_by_year_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"
paths["secondary_by_bucket_year_csv"] = OUT_AUDIT_DIR / f"16R_LITE_secondary_incremental_by_bucket_year_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"

paths["combined_program_summary_csv"] = OUT_AUDIT_DIR / f"16R_LITE_combined_program_summary_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"
paths["combined_program_by_year_csv"] = OUT_AUDIT_DIR / f"16R_LITE_combined_program_by_year_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"

paths["comparison_csv"] = OUT_AUDIT_DIR / f"16R_LITE_candidate_comparison_program_view_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"
paths["comparison_parquet"] = OUT_PROCESSED_DIR / f"16R_LITE_candidate_comparison_program_view_front_excluded_{CELL16R_LITE_TIMESTAMP}.parquet"

paths["preferred_secondary_panel_csv"] = OUT_AUDIT_DIR / f"16R_LITE_preferred_secondary_panel_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"
paths["preferred_secondary_panel_parquet"] = OUT_PROCESSED_DIR / f"16R_LITE_preferred_secondary_panel_front_excluded_{CELL16R_LITE_TIMESTAMP}.parquet"

paths["preferred_combined_panel_csv"] = OUT_AUDIT_DIR / f"16R_LITE_preferred_combined_panel_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"
paths["preferred_combined_panel_parquet"] = OUT_PROCESSED_DIR / f"16R_LITE_preferred_combined_panel_front_excluded_{CELL16R_LITE_TIMESTAMP}.parquet"

paths["preferred_secondary_by_bucket_csv"] = OUT_AUDIT_DIR / f"16R_LITE_preferred_secondary_by_bucket_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"
paths["preferred_secondary_by_tenor_csv"] = OUT_AUDIT_DIR / f"16R_LITE_preferred_secondary_by_tenor_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"
paths["preferred_secondary_by_year_csv"] = OUT_AUDIT_DIR / f"16R_LITE_preferred_secondary_by_year_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"
paths["preferred_combined_by_year_csv"] = OUT_AUDIT_DIR / f"16R_LITE_preferred_combined_by_year_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"

paths["worst_trades_csv"] = OUT_AUDIT_DIR / f"16R_LITE_worst_trades_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"

middle_options.to_csv(paths["middle_options_csv"], index=False)
back_options.to_csv(paths["back_options_csv"], index=False)

pair_grid.to_csv(paths["pair_grid_csv"], index=False)
pair_mode_grid.to_csv(paths["pair_mode_grid_csv"], index=False)

core_summary.to_csv(paths["core_program_summary_csv"], index=False)

secondary_selected.to_parquet(paths["secondary_selected_panel_parquet"], index=False)
secondary_selected.to_csv(paths["secondary_selected_panel_csv"], index=False)

combined_program_panel.to_parquet(paths["combined_program_panel_parquet"], index=False)
combined_program_panel.to_csv(paths["combined_program_panel_csv"], index=False)

secondary_summary.to_csv(paths["secondary_summary_csv"], index=False)
secondary_by_bucket.to_csv(paths["secondary_by_bucket_csv"], index=False)
secondary_by_tenor.to_csv(paths["secondary_by_tenor_csv"], index=False)
secondary_by_year.to_csv(paths["secondary_by_year_csv"], index=False)
secondary_by_bucket_year.to_csv(paths["secondary_by_bucket_year_csv"], index=False)

combined_program_summary.to_csv(paths["combined_program_summary_csv"], index=False)
combined_program_by_year.to_csv(paths["combined_program_by_year_csv"], index=False)

comparison.to_csv(paths["comparison_csv"], index=False)
comparison.to_parquet(paths["comparison_parquet"], index=False)

preferred_secondary_panel.to_csv(paths["preferred_secondary_panel_csv"], index=False)
preferred_secondary_panel.to_parquet(paths["preferred_secondary_panel_parquet"], index=False)

preferred_combined_panel.to_csv(paths["preferred_combined_panel_csv"], index=False)
preferred_combined_panel.to_parquet(paths["preferred_combined_panel_parquet"], index=False)

preferred_secondary_by_bucket.to_csv(paths["preferred_secondary_by_bucket_csv"], index=False)
preferred_secondary_by_tenor.to_csv(paths["preferred_secondary_by_tenor_csv"], index=False)
preferred_secondary_by_year.to_csv(paths["preferred_secondary_by_year_csv"], index=False)
preferred_combined_by_year.to_csv(paths["preferred_combined_by_year_csv"], index=False)

worst_trades.to_csv(paths["worst_trades_csv"], index=False)

# ======================================================================================
# 12. Validation
# ======================================================================================

validation_rows = []

required_base_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

required_cell15_selected_cols = [
    "trade_date",
    "tenor",
    "secondary_bucket",
    "secondary_candidate_id",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

missing_base_cols = [c for c in required_base_cols if c not in base.columns]
missing_cell15_selected_cols = [c for c in required_cell15_selected_cols if c not in cell15_selected_all.columns]

models_found = sorted(base["model_spec"].dropna().unique().tolist()) if "model_spec" in base.columns else []

bad_pair_count = len(pair_grid) != expected_pair_count
bad_pair_mode_count = len(pair_mode_grid) != expected_pair_mode_count

bad_front_selected = secondary_selected[
    secondary_selected["secondary_bucket"].eq("Front")
    | secondary_selected["tenor"].isin(SECONDARY_EXCLUDED_FRONT_TENORS)
]

bad_non_mb_selected = secondary_selected[
    ~secondary_selected["secondary_bucket"].isin(SECONDARY_ALLOWED_BUCKETS)
    | ~secondary_selected["tenor"].isin(SECONDARY_ALLOWED_TENORS)
]

bad_secondary_on_core_gate = secondary_selected[
    secondary_selected["trade_date"].isin(core_gate_dates_set)
]

bad_secondary_dupes = (
    secondary_selected
    .groupby(["secondary_mb_lite_pair_mode_id", "trade_date"])
    .size()
    .reset_index(name="rows")
)

bad_secondary_dupes = bad_secondary_dupes[bad_secondary_dupes["rows"].gt(1)]

bad_combined_dupes = (
    combined_program_panel
    .groupby(["secondary_mb_lite_pair_mode_id", "trade_date"])
    .size()
    .reset_index(name="rows")
)

bad_combined_dupes = bad_combined_dupes[bad_combined_dupes["rows"].gt(1)]

bad_combined_below_core_dates = comparison[
    comparison["combined_unique_trade_dates"] < int(core_summary["unique_trade_dates"].iloc[0])
]

bad_selection_modes = sorted(set(comparison["selection_mode"].dropna().unique()) - set(SELECTION_MODES))

bad_comparison_count = comparison["secondary_mb_lite_pair_mode_id"].nunique() != expected_pair_mode_count

bad_normalized_pnl = base[
    base["normalized_pnl_dollars"].notna()
    & base["normalized_pnl_per_day"].notna()
    & (
        (
            base["normalized_pnl_dollars"] / base["tenor"].replace(0, np.nan)
        )
        - base["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
]

add_validation(
    validation_rows,
    "base_panel_loaded",
    "PASS" if len(base) > 0 else "FAIL",
    f"rows={len(base):,}; path={base_panel_path}",
)

add_validation(
    validation_rows,
    "cell15_selected_loaded",
    "PASS" if len(cell15_selected_all) > 0 else "FAIL",
    f"rows={len(cell15_selected_all):,}; path={cell15_selected_all_path}",
)

add_validation(
    validation_rows,
    "cell15_scorecards_loaded",
    "PASS" if len(cell15_scorecards_all) > 0 else "FAIL",
    f"rows={len(cell15_scorecards_all):,}; path={cell15_scorecards_all_path}",
)

add_validation(
    validation_rows,
    "required_base_columns_available",
    "PASS" if not missing_base_cols else "FAIL",
    f"missing={missing_base_cols}",
)

add_validation(
    validation_rows,
    "required_cell15_selected_columns_available",
    "PASS" if not missing_cell15_selected_cols else "FAIL",
    f"missing={missing_cell15_selected_cols}",
)

add_validation(
    validation_rows,
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    validation_rows,
    "locked_core_rules_loaded",
    "PASS" if len(locked_core_rules_long) == 9 else "FAIL",
    f"rows={len(locked_core_rules_long):,}; path={locked_core_rules_long_path}",
)

add_validation(
    validation_rows,
    "core_gate_dates_created",
    "PASS" if len(core_gate_dates) > 0 and len(core_empty_dates) > 0 else "FAIL",
    f"core_gate_dates={len(core_gate_dates):,}; core_empty_dates={len(core_empty_dates):,}; total_dates={total_eligible_signal_dates:,}",
)

add_validation(
    validation_rows,
    "middle_options_count",
    "PASS" if len(middle_options) == 5 else "FAIL",
    f"middle_options={len(middle_options):,}; expected=5",
)

add_validation(
    validation_rows,
    "back_options_count",
    "PASS" if len(back_options) == 9 else "FAIL",
    f"back_options={len(back_options):,}; expected=9",
)

add_validation(
    validation_rows,
    "pair_count",
    "PASS" if not bad_pair_count else "FAIL",
    f"pair_count={len(pair_grid):,}; expected={expected_pair_count:,}",
)

add_validation(
    validation_rows,
    "pair_mode_count",
    "PASS" if not bad_pair_mode_count else "FAIL",
    f"pair_mode_count={len(pair_mode_grid):,}; expected={expected_pair_mode_count:,}",
)

add_validation(
    validation_rows,
    "all_pair_modes_compared",
    "PASS" if not bad_comparison_count else "FAIL",
    f"comparison_pair_modes={comparison['secondary_mb_lite_pair_mode_id'].nunique():,}; expected={expected_pair_mode_count:,}",
)

add_validation(
    validation_rows,
    "selection_modes_valid",
    "PASS" if not bad_selection_modes else "FAIL",
    f"bad_modes={bad_selection_modes}; modes_found={sorted(comparison['selection_mode'].dropna().unique().tolist())}",
)

add_validation(
    validation_rows,
    "secondary_front_excluded_from_selected_trades",
    "PASS" if bad_front_selected.empty else "FAIL",
    f"bad_rows={len(bad_front_selected):,}",
)

add_validation(
    validation_rows,
    "selected_trades_middle_or_back_only",
    "PASS" if bad_non_mb_selected.empty else "FAIL",
    f"bad_rows={len(bad_non_mb_selected):,}",
)

add_validation(
    validation_rows,
    "secondary_only_on_core_empty_dates",
    "PASS" if bad_secondary_on_core_gate.empty else "FAIL",
    f"bad_rows={len(bad_secondary_on_core_gate):,}",
)

add_validation(
    validation_rows,
    "one_secondary_trade_per_pair_mode_date",
    "PASS" if bad_secondary_dupes.empty else "FAIL",
    f"bad_rows={len(bad_secondary_dupes):,}",
)

add_validation(
    validation_rows,
    "combined_program_one_trade_per_pair_mode_date",
    "PASS" if bad_combined_dupes.empty else "FAIL",
    f"bad_rows={len(bad_combined_dupes):,}",
)

add_validation(
    validation_rows,
    "combined_candidates_not_below_core_date_count",
    "PASS" if bad_combined_below_core_dates.empty else "FAIL",
    f"bad_rows={len(bad_combined_below_core_dates):,}",
)

add_validation(
    validation_rows,
    "normalized_pnl_per_day_formula",
    "PASS" if bad_normalized_pnl.empty else "FAIL",
    f"bad_rows={len(bad_normalized_pnl):,}",
)

add_validation(
    validation_rows,
    "core_not_reopened",
    "PASS",
    "Locked Core rules loaded from Cell 11R and used only as gate/baseline.",
)

add_validation(
    validation_rows,
    "no_secondary_lock",
    "PASS",
    "Finalist optimization only; no Secondary lock created.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed or added.",
)

add_validation(
    validation_rows,
    "no_production_lock",
    "PASS",
    "No production/intraday implementation performed.",
)

validation = pd.DataFrame(validation_rows)

paths["validation_csv"] = OUT_AUDIT_DIR / f"16R_LITE_validation_front_excluded_{CELL16R_LITE_TIMESTAMP}.csv"
validation.to_csv(paths["validation_csv"], index=False)

failures = validation[validation["status"].eq("FAIL")]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1_RV21D_repaired_v2.ipynb",
    "cell": "Cell 16R-LITE — Secondary Middle/Back finalist grid, Front excluded",
    "timestamp": CELL16R_LITE_TIMESTAMP,

    "locked_core_version": LOCKED_CORE_VERSION,
    "locked_core_decision_id": LOCKED_CORE_DECISION_ID,
    "secondary_mb_lite_version": SECONDARY_MB_LITE_VERSION,

    "base_panel_path": str(base_panel_path),
    "locked_core_rules_long_path": str(locked_core_rules_long_path),
    "locked_core_rules_wide_path": str(locked_core_rules_wide_path),
    "cell15_selected_all_path": str(cell15_selected_all_path),
    "cell15_scorecards_all_path": str(cell15_scorecards_all_path),

    "secondary_front_excluded": True,
    "secondary_allowed_buckets": SECONDARY_ALLOWED_BUCKETS,
    "secondary_allowed_tenors": SECONDARY_ALLOWED_TENORS,
    "secondary_excluded_front_tenors": SECONDARY_EXCLUDED_FRONT_TENORS,

    "middle_options": middle_specs,
    "back_options": back_specs,

    "selection_modes": SELECTION_MODES,
    "expected_pair_count": expected_pair_count,
    "expected_pair_mode_count": expected_pair_mode_count,
    "actual_pair_count": int(len(pair_grid)),
    "actual_pair_mode_count": int(len(pair_mode_grid)),

    "total_eligible_signal_dates": int(total_eligible_signal_dates),
    "core_gate_dates": int(len(core_gate_dates)),
    "core_empty_dates": int(len(core_empty_dates)),

    "preferred_pair_mode_id": preferred_pair_mode_id,

    "screens": {
        "combined_win_rate_target": COMBINED_WIN_RATE_TARGET,
        "secondary_win_rate_floor": SECONDARY_WIN_RATE_FLOOR,
        "secondary_win_rate_ideal": SECONDARY_WIN_RATE_IDEAL,
        "secondary_avg_pnl_per_day_min": SECONDARY_AVG_PNL_PER_DAY_MIN,
        "combined_avg_pnl_per_day_min": COMBINED_AVG_PNL_PER_DAY_MIN,
        "min_secondary_trades": MIN_SECONDARY_TRADES,
    },

    "constraints": [
        "Core remains locked and unchanged.",
        "Secondary Front fully excluded.",
        "Only shortlisted Middle/Back candidates tested.",
        "No full 117 x 235 grid.",
        "No Secondary lock.",
        "No sizing changes.",
        "No production/intraday implementation.",
    ],

    "paths": {k: str(v) for k, v in paths.items()},
}

paths["manifest_json"] = OUT_AUDIT_DIR / f"16R_LITE_manifest_front_excluded_{CELL16R_LITE_TIMESTAMP}.json"

with open(paths["manifest_json"], "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 16R-LITE validation failed. Review validation output above.")

# ======================================================================================
# 13. Display review
# ======================================================================================

print("=" * 100)
print("Cell 16R-LITE — Secondary Middle/Back finalist grid, Front excluded")
print("=" * 100)
print(f"Base panel:                         {base_panel_path}")
print(f"Locked Core rules:                  {locked_core_rules_long_path}")
print(f"Cell 15R selected panel:            {cell15_selected_all_path}")
print(f"Cell 15R scorecards:                {cell15_scorecards_all_path}")
print(f"Total eligible signal dates:        {total_eligible_signal_dates:,}")
print(f"Core gate dates:                    {len(core_gate_dates):,}")
print(f"Core-empty dates:                   {len(core_empty_dates):,}")
print(f"Secondary Front excluded:           {SECONDARY_FRONT_EXCLUDED}")
print(f"Middle options:                     {len(middle_options):,}")
print(f"Back options:                       {len(back_options):,}")
print(f"Structural pairs:                   {len(pair_grid):,}")
print(f"Pair-mode candidates:               {len(pair_mode_grid):,}")
print(f"Selection modes:                    {SELECTION_MODES}")
print(f"Preferred pair-mode id:             {preferred_pair_mode_id}")
print("No Secondary lock:                  True")
print("No sizing changes:                  True")
print("No production lock:                 True")

print()
print("Validation:")
display(validation)

print()
print("Core program baseline:")
display(core_summary)

print()
print("Middle options:")
display(middle_options)

print()
print("Back options:")
display(back_options)

display_cols = [
    "comparison_rank",
    "secondary_mb_lite_pair_mode_id",
    "secondary_mb_lite_pair_id",
    "selection_mode",
    "pair_description",

    "middle_option_id",
    "middle_candidate_id",
    "middle_param_model_vrp_log_min",
    "middle_param_model_vrp_z_min",
    "middle_param_RSI14_max",
    "middle_param_rv21d_vol_pct_min",
    "middle_incremental_trades",
    "middle_incremental_win_rate",
    "middle_incremental_signal_date_frequency",
    "middle_incremental_avg_pnl_per_day",
    "middle_incremental_profit_factor_pnl_per_day",
    "middle_incremental_avg_pnl_per_day_2026",

    "back_option_id",
    "back_candidate_id",
    "back_param_model_vrp_log_min",
    "back_param_model_vrp_z_min",
    "back_param_RSI14_max",
    "back_param_rv21d_vol_pct_min",
    "back_incremental_trades",
    "back_incremental_win_rate",
    "back_incremental_signal_date_frequency",
    "back_incremental_avg_pnl_per_day",
    "back_incremental_profit_factor_pnl_per_day",
    "back_incremental_avg_pnl_per_day_2026",

    "secondary_trades",
    "secondary_unique_trade_dates",
    "secondary_signal_date_frequency",
    "secondary_win_rate",
    "secondary_total_pnl",
    "secondary_avg_pnl_per_day",
    "secondary_profit_factor_pnl_per_day",
    "secondary_pnl_per_day_drawdown",
    "secondary_worst_20_trade_pnl_per_day_sum",
    "secondary_avg_pnl_per_day_2025",
    "secondary_avg_pnl_per_day_2026",

    "selected_middle_trade_count",
    "selected_back_trade_count",
    "middle_share_of_secondary_trades",
    "back_share_of_secondary_trades",

    "combined_trades",
    "combined_unique_trade_dates",
    "combined_signal_date_frequency",
    "combined_win_rate",
    "combined_total_pnl",
    "combined_avg_pnl_per_day",
    "combined_profit_factor_pnl_per_day",
    "combined_pnl_per_day_drawdown",
    "combined_worst_20_trade_pnl_per_day_sum",
    "combined_avg_pnl_per_day_2025",
    "combined_avg_pnl_per_day_2026",

    "delta_combined_unique_trade_dates_vs_core_only",
    "delta_combined_signal_date_frequency_vs_core_only",
    "delta_combined_win_rate_vs_core_only",
    "delta_combined_total_pnl_vs_core_only",
    "delta_combined_avg_pnl_per_day_vs_core_only",
    "delta_combined_profit_factor_pnl_per_day_vs_core_only",
    "delta_combined_pnl_per_day_drawdown_vs_core_only",
    "delta_combined_worst_20_trade_pnl_per_day_sum_vs_core_only",
    "delta_combined_avg_pnl_per_day_2026_vs_core_only",

    "passes_combined_win_rate_80pct",
    "passes_secondary_win_rate_floor_78pct",
    "passes_secondary_win_rate_ideal_80pct",
    "passes_hard_screen",
    "passes_ideal_screen",
    "candidate_score",
]

print()
print("Candidate comparison — PROGRAM VIEW, Core first else Secondary Middle/Back LITE:")
display(comparison[[c for c in display_cols if c in comparison.columns]])

print()
print("Secondary incremental by selected bucket:")
display(secondary_by_bucket)

print()
print("Secondary incremental by tenor:")
display(secondary_by_tenor)

print()
print("Secondary incremental by year:")
display(secondary_by_year)

print()
print("Secondary incremental by bucket × year:")
display(secondary_by_bucket_year)

print()
print("Preferred candidate — Secondary panel:")
display(preferred_secondary_panel)

print()
print("Preferred candidate — Secondary by bucket:")
display(preferred_secondary_by_bucket)

print()
print("Preferred candidate — Secondary by tenor:")
display(preferred_secondary_by_tenor)

print()
print("Preferred candidate — Secondary by year:")
display(preferred_secondary_by_year)

print()
print("Preferred candidate — Combined by year:")
display(preferred_combined_by_year)

print()
print("Worst trades by pair-mode candidate:")
display(worst_trades)

print()
print("Saved outputs:")
for k, p in paths.items():
    print(f"  {k}: {p}")

print("\nCELL 16R-LITE PASSED — SECONDARY MIDDLE/BACK FINALIST GRID COMPLETE, FRONT EXCLUDED")

In [ ]:
# ======================================================================================
# Cell 19R — Secondary Frequency Unlock Review
# ======================================================================================
#
# Objective:
#   Reopen Secondary only and review existing Secondary candidate runs for better frequency.
#
# This cell reads prior saved candidate comparison files:
#   - Cell 17R final Middle/Back lock-readiness candidates
#   - Cell 16R-LITE Middle/Back finalist candidates, if available
#   - Cell 18R current lock decision summary, if available
#
# It does NOT:
#   - rerun any signal sweep
#   - change Core
#   - change Secondary
#   - write a new lock
#   - include sizing
#   - implement production/intraday logic
#
# Goal:
#   Identify candidates that improve frequency while preserving acceptable quality.
#
# Screens:
#   - combined_signal_date_frequency >= 18%
#   - combined_win_rate >= 80%
#   - secondary_win_rate >= 78%
#   - secondary_avg_pnl_per_day > 0
#   - combined 2026 avg/day improves versus Core-only
#
# Current provisional Secondary lock to beat:
#   secondary_final_pair_0047__BEST_SIGNAL_RANK
#   combined frequency = 15.38%
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 800)
pd.set_option("display.width", 1200)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL19R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

CURRENT_SECONDARY_LOCK_CANDIDATE_ID = "secondary_final_pair_0047__BEST_SIGNAL_RANK"
CURRENT_SECONDARY_LOCK_DECISION_ID = "secondary_middle_back_locked_001"

FREQUENCY_UNLOCK_VERSION = "secondary_frequency_unlock_review_v1"

# Candidate screens.
MIN_COMBINED_FREQUENCY = 0.18
MIN_COMBINED_WIN_RATE = 0.80
MIN_SECONDARY_WIN_RATE = 0.78
MIN_SECONDARY_AVG_DAY = 0.0
MIN_SECONDARY_TRADES = 20

# Ranking preferences.
TARGET_FREQUENCY = 0.20
TARGET_SECONDARY_WIN_RATE = 0.80

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)

    if matches:
        return matches[0]

    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")

    return None


def all_files(directory, pattern):
    return sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


def coerce_numeric_cols(df, cols):
    x = df.copy()
    for c in cols:
        if c in x.columns:
            x[c] = pd.to_numeric(x[c], errors="coerce")
    return x


def coerce_bool_cols(df, cols):
    x = df.copy()
    for c in cols:
        if c in x.columns:
            if x[c].dtype == bool:
                x[c] = x[c].fillna(False)
            else:
                x[c] = (
                    x[c]
                    .astype(str)
                    .str.strip()
                    .str.lower()
                    .isin(["true", "1", "yes", "y"])
                )
    return x


def first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def value_or_nan(row, col):
    if col is None:
        return np.nan
    return row.get(col, np.nan)


def infer_candidate_structure(row):
    middle_option = str(row.get("middle_option_id", ""))
    back_option = str(row.get("back_option_id", ""))

    middle_is_none = bool(row.get("middle_is_none", False)) or middle_option.upper().startswith("NO_MIDDLE")
    back_is_none = bool(row.get("back_is_none", False)) or back_option.upper().startswith("NO_BACK")

    if (not middle_is_none) and back_is_none:
        return "MIDDLE_ONLY"

    if middle_is_none and (not back_is_none):
        return "BACK_ONLY"

    if (not middle_is_none) and (not back_is_none):
        return "MIDDLE_BACK"

    return "UNKNOWN"


def normalize_candidate_table(df, source_run, source_path):
    x = df.copy()

    # Normalize IDs.
    pair_mode_col = first_existing_column(
        x,
        [
            "secondary_final_pair_mode_id",
            "secondary_mb_lite_pair_mode_id",
            "secondary_stack_id",
            "candidate_id",
        ],
    )

    pair_col = first_existing_column(
        x,
        [
            "secondary_final_pair_id",
            "secondary_mb_lite_pair_id",
            "secondary_stack_pair_id",
            "secondary_stack_id",
        ],
    )

    if pair_mode_col is None:
        raise ValueError(f"Cannot find pair-mode/candidate ID column for {source_run}: {source_path}")

    if pair_col is None:
        pair_col = pair_mode_col

    x["candidate_run"] = source_run
    x["candidate_source_path"] = str(source_path)
    x["normalized_candidate_id"] = x[pair_mode_col].astype(str)
    x["normalized_pair_id"] = x[pair_col].astype(str)

    # Normalize candidate structure.
    if "candidate_structure" not in x.columns:
        x["candidate_structure"] = x.apply(infer_candidate_structure, axis=1)
    else:
        x["candidate_structure"] = x["candidate_structure"].fillna(x.apply(infer_candidate_structure, axis=1))

    # Normalize option fields.
    for c in [
        "middle_option_id",
        "middle_candidate_id",
        "middle_description",
        "middle_is_none",
        "middle_param_model_vrp_log_min",
        "middle_param_model_vrp_z_min",
        "middle_param_RSI14_max",
        "middle_param_rv21d_vol_pct_min",
        "back_option_id",
        "back_candidate_id",
        "back_description",
        "back_is_none",
        "back_param_model_vrp_log_min",
        "back_param_model_vrp_z_min",
        "back_param_RSI14_max",
        "back_param_rv21d_vol_pct_min",
        "selection_mode",
        "pair_description",
    ]:
        if c not in x.columns:
            x[c] = np.nan

    if "pair_description" not in x.columns or x["pair_description"].isna().all():
        x["pair_description"] = (
            x["middle_option_id"].astype(str)
            + " + "
            + x["back_option_id"].astype(str)
        )

    x["middle_is_none"] = (
        x["middle_is_none"].astype(str).str.lower().isin(["true", "1", "yes"])
        | x["middle_option_id"].astype(str).str.upper().str.startswith("NO_MIDDLE")
    )

    x["back_is_none"] = (
        x["back_is_none"].astype(str).str.lower().isin(["true", "1", "yes"])
        | x["back_option_id"].astype(str).str.upper().str.startswith("NO_BACK")
    )

    # Normalize possible metrics.
    numeric_cols = [
        "middle_incremental_trades",
        "middle_incremental_win_rate",
        "middle_incremental_signal_date_frequency",
        "middle_incremental_total_pnl",
        "middle_incremental_avg_pnl_per_day",
        "middle_incremental_profit_factor_pnl_per_day",
        "middle_incremental_pnl_per_day_drawdown",
        "middle_incremental_worst_20_trade_pnl_per_day_sum",
        "middle_incremental_avg_pnl_per_day_2025",
        "middle_incremental_avg_pnl_per_day_2026",

        "back_incremental_trades",
        "back_incremental_win_rate",
        "back_incremental_signal_date_frequency",
        "back_incremental_total_pnl",
        "back_incremental_avg_pnl_per_day",
        "back_incremental_profit_factor_pnl_per_day",
        "back_incremental_pnl_per_day_drawdown",
        "back_incremental_worst_20_trade_pnl_per_day_sum",
        "back_incremental_avg_pnl_per_day_2025",
        "back_incremental_avg_pnl_per_day_2026",

        "secondary_trades",
        "secondary_unique_trade_dates",
        "secondary_signal_date_frequency",
        "secondary_win_rate",
        "secondary_total_pnl",
        "secondary_avg_pnl_per_day",
        "secondary_profit_factor_pnl_per_day",
        "secondary_pnl_per_day_drawdown",
        "secondary_worst_20_trade_pnl_per_day_sum",
        "secondary_avg_pnl_per_day_2025",
        "secondary_avg_pnl_per_day_2026",

        "selected_middle_trade_count",
        "selected_back_trade_count",
        "middle_share_of_secondary_trades",
        "back_share_of_secondary_trades",

        "combined_trades",
        "combined_unique_trade_dates",
        "combined_signal_date_frequency",
        "combined_win_rate",
        "combined_total_pnl",
        "combined_avg_pnl_per_day",
        "combined_profit_factor_pnl_per_day",
        "combined_pnl_per_day_drawdown",
        "combined_worst_20_trade_pnl_per_day_sum",
        "combined_avg_pnl_per_day_2025",
        "combined_avg_pnl_per_day_2026",

        "delta_combined_unique_trade_dates_vs_core_only",
        "delta_combined_signal_date_frequency_vs_core_only",
        "delta_combined_win_rate_vs_core_only",
        "delta_combined_total_pnl_vs_core_only",
        "delta_combined_avg_pnl_per_day_vs_core_only",
        "delta_combined_profit_factor_pnl_per_day_vs_core_only",
        "delta_combined_pnl_per_day_drawdown_vs_core_only",
        "delta_combined_worst_20_trade_pnl_per_day_sum_vs_core_only",
        "delta_combined_avg_pnl_per_day_2026_vs_core_only",

        "middle_param_model_vrp_log_min",
        "middle_param_model_vrp_z_min",
        "middle_param_RSI14_max",
        "middle_param_rv21d_vol_pct_min",
        "back_param_model_vrp_log_min",
        "back_param_model_vrp_z_min",
        "back_param_RSI14_max",
        "back_param_rv21d_vol_pct_min",
    ]

    x = coerce_numeric_cols(x, numeric_cols)

    bool_cols = [
        "passes_combined_win_rate_80pct",
        "passes_secondary_win_rate_80pct",
        "passes_secondary_win_rate_floor_78pct",
        "passes_secondary_win_rate_78pct_floor",
        "passes_secondary_positive_avg_day",
        "passes_combined_positive_avg_day",
        "passes_secondary_trade_count_preferred",
        "passes_2026_improves_vs_core",
        "passes_final_lock_screen_quality_only",
        "passes_final_lock_screen_strict",
        "included_buckets_standalone_quality_pass",
        "included_buckets_standalone_lock_ready",
    ]

    x = coerce_bool_cols(x, bool_cols)

    # Fill selected bucket counts if missing and candidate is bucket-only.
    if "selected_middle_trade_count" in x.columns:
        x.loc[
            x["selected_middle_trade_count"].isna() & x["candidate_structure"].eq("MIDDLE_ONLY"),
            "selected_middle_trade_count",
        ] = x.loc[
            x["selected_middle_trade_count"].isna() & x["candidate_structure"].eq("MIDDLE_ONLY"),
            "secondary_trades",
        ]

        x.loc[
            x["selected_middle_trade_count"].isna(),
            "selected_middle_trade_count",
        ] = 0

    if "selected_back_trade_count" in x.columns:
        x.loc[
            x["selected_back_trade_count"].isna() & x["candidate_structure"].eq("BACK_ONLY"),
            "selected_back_trade_count",
        ] = x.loc[
            x["selected_back_trade_count"].isna() & x["candidate_structure"].eq("BACK_ONLY"),
            "secondary_trades",
        ]

        x.loc[
            x["selected_back_trade_count"].isna(),
            "selected_back_trade_count",
        ] = 0

    # Make missing screen columns explicit.
    for c in bool_cols:
        if c not in x.columns:
            x[c] = False

    # Add current lock marker.
    x["is_current_secondary_lock"] = x["normalized_candidate_id"].eq(CURRENT_SECONDARY_LOCK_CANDIDATE_ID)

    keep_cols = [
        "candidate_run",
        "candidate_source_path",
        "normalized_candidate_id",
        "normalized_pair_id",
        "candidate_structure",
        "selection_mode",
        "pair_description",

        "middle_option_id",
        "middle_candidate_id",
        "middle_param_model_vrp_log_min",
        "middle_param_model_vrp_z_min",
        "middle_param_RSI14_max",
        "middle_param_rv21d_vol_pct_min",
        "middle_incremental_trades",
        "middle_incremental_win_rate",
        "middle_incremental_avg_pnl_per_day",
        "middle_incremental_avg_pnl_per_day_2026",

        "back_option_id",
        "back_candidate_id",
        "back_param_model_vrp_log_min",
        "back_param_model_vrp_z_min",
        "back_param_RSI14_max",
        "back_param_rv21d_vol_pct_min",
        "back_incremental_trades",
        "back_incremental_win_rate",
        "back_incremental_avg_pnl_per_day",
        "back_incremental_avg_pnl_per_day_2026",

        "secondary_trades",
        "secondary_unique_trade_dates",
        "secondary_signal_date_frequency",
        "secondary_win_rate",
        "secondary_total_pnl",
        "secondary_avg_pnl_per_day",
        "secondary_profit_factor_pnl_per_day",
        "secondary_pnl_per_day_drawdown",
        "secondary_worst_20_trade_pnl_per_day_sum",
        "secondary_avg_pnl_per_day_2025",
        "secondary_avg_pnl_per_day_2026",

        "selected_middle_trade_count",
        "selected_back_trade_count",
        "middle_share_of_secondary_trades",
        "back_share_of_secondary_trades",

        "combined_trades",
        "combined_unique_trade_dates",
        "combined_signal_date_frequency",
        "combined_win_rate",
        "combined_total_pnl",
        "combined_avg_pnl_per_day",
        "combined_profit_factor_pnl_per_day",
        "combined_pnl_per_day_drawdown",
        "combined_worst_20_trade_pnl_per_day_sum",
        "combined_avg_pnl_per_day_2025",
        "combined_avg_pnl_per_day_2026",

        "delta_combined_unique_trade_dates_vs_core_only",
        "delta_combined_signal_date_frequency_vs_core_only",
        "delta_combined_win_rate_vs_core_only",
        "delta_combined_total_pnl_vs_core_only",
        "delta_combined_avg_pnl_per_day_vs_core_only",
        "delta_combined_profit_factor_pnl_per_day_vs_core_only",
        "delta_combined_pnl_per_day_drawdown_vs_core_only",
        "delta_combined_worst_20_trade_pnl_per_day_sum_vs_core_only",
        "delta_combined_avg_pnl_per_day_2026_vs_core_only",

        "passes_combined_win_rate_80pct",
        "passes_secondary_win_rate_80pct",
        "passes_secondary_win_rate_floor_78pct",
        "passes_secondary_win_rate_78pct_floor",
        "passes_secondary_positive_avg_day",
        "passes_combined_positive_avg_day",
        "passes_secondary_trade_count_preferred",
        "passes_2026_improves_vs_core",
        "passes_final_lock_screen_quality_only",
        "passes_final_lock_screen_strict",
        "included_buckets_standalone_quality_pass",
        "included_buckets_standalone_lock_ready",

        "is_current_secondary_lock",
    ]

    return x[[c for c in keep_cols if c in x.columns]].copy()


def rank_candidates(candidates, core_2026_avg_day=None):
    x = candidates.copy()

    if core_2026_avg_day is None or pd.isna(core_2026_avg_day):
        # Use explicit delta if present; otherwise do not require.
        x["computed_2026_improves_vs_core"] = x["delta_combined_avg_pnl_per_day_2026_vs_core_only"].fillna(0) > 0
    else:
        x["computed_2026_improves_vs_core"] = x["combined_avg_pnl_per_day_2026"] > core_2026_avg_day

    x["passes_secondary_win_floor"] = (
        (x["secondary_win_rate"] >= MIN_SECONDARY_WIN_RATE)
        | x.get("passes_secondary_win_rate_floor_78pct", pd.Series(False, index=x.index)).fillna(False)
        | x.get("passes_secondary_win_rate_78pct_floor", pd.Series(False, index=x.index)).fillna(False)
    )

    x["passes_unlock_screen"] = (
        (x["combined_signal_date_frequency"] >= MIN_COMBINED_FREQUENCY)
        & (x["combined_win_rate"] >= MIN_COMBINED_WIN_RATE)
        & x["passes_secondary_win_floor"]
        & (x["secondary_avg_pnl_per_day"] > MIN_SECONDARY_AVG_DAY)
        & (x["secondary_trades"] >= MIN_SECONDARY_TRADES)
        & x["computed_2026_improves_vs_core"]
    )

    x["frequency_gap_to_20pct"] = TARGET_FREQUENCY - x["combined_signal_date_frequency"]
    x["secondary_win_gap_to_80pct"] = TARGET_SECONDARY_WIN_RATE - x["secondary_win_rate"]

    # Penalize candidates below 80% secondary win, but allow 78–80% as explicitly reviewed.
    x["secondary_win_penalty"] = np.where(
        x["secondary_win_rate"] >= TARGET_SECONDARY_WIN_RATE,
        0.0,
        (TARGET_SECONDARY_WIN_RATE - x["secondary_win_rate"]).clip(lower=0) * 10_000,
    )

    x["frequency_unlock_score"] = (
        x["passes_unlock_screen"].astype(int) * 1_000_000
        + x["combined_signal_date_frequency"].fillna(0) * 100_000
        + x["secondary_trades"].fillna(0) * 500
        + x["combined_win_rate"].fillna(0) * 10_000
        + x["secondary_win_rate"].fillna(0) * 8_000
        + np.clip(x["secondary_avg_pnl_per_day"].fillna(-10_000), -10_000, 10_000)
        + np.clip(x["combined_avg_pnl_per_day_2026"].fillna(-10_000), -10_000, 10_000) / 2
        + np.clip(x["delta_combined_avg_pnl_per_day_2026_vs_core_only"].fillna(-10_000), -10_000, 10_000)
        - x["secondary_win_penalty"]
    )

    x["balanced_unlock_score"] = (
        x["passes_unlock_screen"].astype(int) * 1_000_000
        + x["secondary_win_rate"].fillna(0) * 100_000
        + x["combined_win_rate"].fillna(0) * 50_000
        + x["combined_signal_date_frequency"].fillna(0) * 50_000
        + np.log1p(x["secondary_trades"].fillna(0)) * 1_000
        + np.clip(x["secondary_avg_pnl_per_day"].fillna(-10_000), -10_000, 10_000)
        + np.clip(x["combined_avg_pnl_per_day_2026"].fillna(-10_000), -10_000, 10_000) / 2
        - x["secondary_win_penalty"]
    )

    x["quality_score"] = (
        x["secondary_win_rate"].fillna(0) * 100_000
        + x["combined_win_rate"].fillna(0) * 50_000
        + np.clip(x["secondary_avg_pnl_per_day"].fillna(-10_000), -10_000, 10_000)
        + np.clip(x["secondary_profit_factor_pnl_per_day"].replace(np.inf, 100).fillna(0), 0, 100) * 100
        + np.log1p(x["secondary_trades"].fillna(0)) * 1_000
    )

    return x


# ======================================================================================
# 2. Load candidate comparison files
# ======================================================================================

files_loaded = []

cell17_comparison_path = latest_file(
    OUT_AUDIT_DIR,
    "17R_candidate_comparison_lock_readiness_final_*.csv",
    required=True,
)

cell17_comparison = pd.read_csv(cell17_comparison_path)
cell17_norm = normalize_candidate_table(cell17_comparison, "17R_FINAL_MB", cell17_comparison_path)
files_loaded.append(str(cell17_comparison_path))

cell16_lite_paths = all_files(
    OUT_AUDIT_DIR,
    "16R_LITE_candidate_comparison_program_view_front_excluded_*.csv",
)

cell16_norms = []

for p in cell16_lite_paths[:3]:
    tmp = pd.read_csv(p)
    cell16_norms.append(normalize_candidate_table(tmp, "16R_LITE_MB", p))
    files_loaded.append(str(p))

if cell16_norms:
    cell16_norm = pd.concat(cell16_norms, ignore_index=True)
else:
    cell16_norm = pd.DataFrame()

# Optional Cell 18 decision summary.
cell18_decision_path = latest_file(
    OUT_AUDIT_DIR,
    "18R_core_secondary_lock_decision_summary_*.csv",
    required=False,
)

cell18_decision = pd.read_csv(cell18_decision_path) if cell18_decision_path else pd.DataFrame()

if cell18_decision_path:
    files_loaded.append(str(cell18_decision_path))

# Optional Core baseline.
core_baseline_path = latest_file(
    OUT_AUDIT_DIR,
    "17R_core_program_baseline_*.csv",
    required=False,
)

core_baseline = pd.read_csv(core_baseline_path) if core_baseline_path else pd.DataFrame()

if core_baseline_path:
    files_loaded.append(str(core_baseline_path))

all_candidates = pd.concat(
    [d for d in [cell17_norm, cell16_norm] if not d.empty],
    ignore_index=True,
)

# ======================================================================================
# 3. Determine current lock and Core reference
# ======================================================================================

current_lock_rows = all_candidates[
    all_candidates["normalized_candidate_id"].eq(CURRENT_SECONDARY_LOCK_CANDIDATE_ID)
].copy()

if current_lock_rows.empty and not cell18_decision.empty:
    decision_map = dict(zip(cell18_decision["decision_item"], cell18_decision["value"]))

    synthetic_current = {
        "candidate_run": "18R_CURRENT_LOCK",
        "candidate_source_path": str(cell18_decision_path),
        "normalized_candidate_id": CURRENT_SECONDARY_LOCK_CANDIDATE_ID,
        "normalized_pair_id": CURRENT_SECONDARY_LOCK_CANDIDATE_ID,
        "candidate_structure": decision_map.get("candidate_structure", "MIDDLE_BACK"),
        "selection_mode": decision_map.get("selection_mode", "BEST_SIGNAL_RANK"),
        "pair_description": "Current locked Secondary from Cell 18R decision summary",
        "secondary_trades": float(decision_map.get("secondary_trades", np.nan)),
        "secondary_win_rate": float(decision_map.get("secondary_win_rate", np.nan)),
        "secondary_total_pnl": float(decision_map.get("secondary_total_pnl", np.nan)),
        "secondary_avg_pnl_per_day": float(decision_map.get("secondary_avg_pnl_per_day", np.nan)),
        "secondary_profit_factor_pnl_per_day": float(decision_map.get("secondary_profit_factor_pnl_per_day", np.nan)),
        "secondary_pnl_per_day_drawdown": float(decision_map.get("secondary_pnl_per_day_drawdown", np.nan)),
        "secondary_worst_20_trade_pnl_per_day_sum": float(decision_map.get("secondary_worst_20_trade_pnl_per_day_sum", np.nan)),
        "secondary_avg_pnl_per_day_2025": float(decision_map.get("secondary_avg_pnl_per_day_2025", np.nan)),
        "secondary_avg_pnl_per_day_2026": float(decision_map.get("secondary_avg_pnl_per_day_2026", np.nan)),
        "selected_middle_trade_count": float(decision_map.get("selected_middle_trade_count", np.nan)),
        "selected_back_trade_count": float(decision_map.get("selected_back_trade_count", np.nan)),
        "combined_trades": float(decision_map.get("combined_trades", np.nan)),
        "combined_unique_trade_dates": float(decision_map.get("combined_unique_trade_dates", np.nan)),
        "combined_signal_date_frequency": float(decision_map.get("combined_signal_date_frequency", np.nan)),
        "combined_win_rate": float(decision_map.get("combined_win_rate", np.nan)),
        "combined_total_pnl": float(decision_map.get("combined_total_pnl", np.nan)),
        "combined_avg_pnl_per_day": float(decision_map.get("combined_avg_pnl_per_day", np.nan)),
        "combined_profit_factor_pnl_per_day": float(decision_map.get("combined_profit_factor_pnl_per_day", np.nan)),
        "combined_pnl_per_day_drawdown": float(decision_map.get("combined_pnl_per_day_drawdown", np.nan)),
        "combined_worst_20_trade_pnl_per_day_sum": float(decision_map.get("combined_worst_20_trade_pnl_per_day_sum", np.nan)),
        "combined_avg_pnl_per_day_2025": float(decision_map.get("combined_avg_pnl_per_day_2025", np.nan)),
        "combined_avg_pnl_per_day_2026": float(decision_map.get("combined_avg_pnl_per_day_2026", np.nan)),
        "delta_combined_unique_trade_dates_vs_core_only": float(decision_map.get("delta_combined_unique_trade_dates_vs_core_only", np.nan)),
        "delta_combined_signal_date_frequency_vs_core_only": float(decision_map.get("delta_combined_signal_date_frequency_vs_core_only", np.nan)),
        "delta_combined_win_rate_vs_core_only": float(decision_map.get("delta_combined_win_rate_vs_core_only", np.nan)),
        "delta_combined_total_pnl_vs_core_only": float(decision_map.get("delta_combined_total_pnl_vs_core_only", np.nan)),
        "delta_combined_avg_pnl_per_day_vs_core_only": float(decision_map.get("delta_combined_avg_pnl_per_day_vs_core_only", np.nan)),
        "delta_combined_avg_pnl_per_day_2026_vs_core_only": float(decision_map.get("delta_combined_avg_pnl_per_day_2026_vs_core_only", np.nan)),
        "is_current_secondary_lock": True,
    }

    all_candidates = pd.concat(
        [all_candidates, pd.DataFrame([synthetic_current])],
        ignore_index=True,
    )

    current_lock_rows = all_candidates[
        all_candidates["normalized_candidate_id"].eq(CURRENT_SECONDARY_LOCK_CANDIDATE_ID)
    ].copy()

core_2026_avg_day = np.nan

if not core_baseline.empty and "avg_pnl_per_day_2026" in core_baseline.columns:
    core_2026_avg_day = float(pd.to_numeric(core_baseline["avg_pnl_per_day_2026"], errors="coerce").iloc[0])
elif not current_lock_rows.empty:
    row0 = current_lock_rows.iloc[0]
    if (
        pd.notna(row0.get("combined_avg_pnl_per_day_2026", np.nan))
        and pd.notna(row0.get("delta_combined_avg_pnl_per_day_2026_vs_core_only", np.nan))
    ):
        core_2026_avg_day = (
            float(row0["combined_avg_pnl_per_day_2026"])
            - float(row0["delta_combined_avg_pnl_per_day_2026_vs_core_only"])
        )

# ======================================================================================
# 4. Rank candidates
# ======================================================================================

ranked = rank_candidates(all_candidates, core_2026_avg_day=core_2026_avg_day)

# Remove exact duplicate normalized IDs if files overlap, prefer latest/final order:
# Keep all by default, but create a de-duped review view for clean displays.
ranked["dedupe_key"] = (
    ranked["candidate_structure"].astype(str)
    + "|"
    + ranked["selection_mode"].astype(str)
    + "|MLOG="
    + ranked["middle_param_model_vrp_log_min"].astype(str)
    + "|MZ="
    + ranked["middle_param_model_vrp_z_min"].astype(str)
    + "|BLOG="
    + ranked["back_param_model_vrp_log_min"].astype(str)
    + "|BZ="
    + ranked["back_param_model_vrp_z_min"].astype(str)
    + "|BRSI="
    + ranked["back_param_RSI14_max"].astype(str)
    + "|BRV="
    + ranked["back_param_rv21d_vol_pct_min"].astype(str)
)

# Prefer candidates from 17R for exact duplicates, then 16R.
ranked["source_preference"] = ranked["candidate_run"].map({
    "17R_FINAL_MB": 1,
    "16R_LITE_MB": 2,
    "18R_CURRENT_LOCK": 3,
}).fillna(9)

ranked_deduped = (
    ranked
    .sort_values(
        [
            "dedupe_key",
            "source_preference",
            "frequency_unlock_score",
        ],
        ascending=[True, True, False],
    )
    .groupby("dedupe_key", as_index=False)
    .head(1)
    .copy()
)

current_lock = ranked[
    ranked["is_current_secondary_lock"].eq(True)
].copy()

if current_lock.empty:
    current_lock_ref = {}
else:
    current_lock_ref = current_lock.iloc[0].to_dict()

current_freq = current_lock_ref.get("combined_signal_date_frequency", np.nan)
current_secondary_trades = current_lock_ref.get("secondary_trades", np.nan)
current_secondary_win = current_lock_ref.get("secondary_win_rate", np.nan)
current_combined_win = current_lock_ref.get("combined_win_rate", np.nan)
current_combined_2026 = current_lock_ref.get("combined_avg_pnl_per_day_2026", np.nan)

ranked_deduped["delta_frequency_vs_current_lock"] = (
    ranked_deduped["combined_signal_date_frequency"] - current_freq
)

ranked_deduped["delta_secondary_trades_vs_current_lock"] = (
    ranked_deduped["secondary_trades"] - current_secondary_trades
)

ranked_deduped["delta_secondary_win_vs_current_lock"] = (
    ranked_deduped["secondary_win_rate"] - current_secondary_win
)

ranked_deduped["delta_combined_win_vs_current_lock"] = (
    ranked_deduped["combined_win_rate"] - current_combined_win
)

ranked_deduped["delta_combined_2026_vs_current_lock"] = (
    ranked_deduped["combined_avg_pnl_per_day_2026"] - current_combined_2026
)

# Views.
unlock_candidates = ranked_deduped[
    ranked_deduped["passes_unlock_screen"].eq(True)
].copy()

frequency_view = ranked_deduped.sort_values(
    [
        "passes_unlock_screen",
        "combined_signal_date_frequency",
        "secondary_trades",
        "combined_win_rate",
        "secondary_win_rate",
        "secondary_avg_pnl_per_day",
        "combined_avg_pnl_per_day_2026",
    ],
    ascending=[False, False, False, False, False, False, False],
).copy()

balanced_view = ranked_deduped.sort_values(
    [
        "passes_unlock_screen",
        "balanced_unlock_score",
        "combined_signal_date_frequency",
        "secondary_win_rate",
        "combined_win_rate",
        "secondary_trades",
    ],
    ascending=[False, False, False, False, False, False],
).copy()

quality_view = ranked_deduped.sort_values(
    [
        "quality_score",
        "secondary_win_rate",
        "combined_win_rate",
        "secondary_avg_pnl_per_day",
        "secondary_trades",
    ],
    ascending=[False, False, False, False, False],
).copy()

# Practical finalists: prioritize candidates above 18% frequency and >=80% combined win,
# then allow secondary win 78%+ as requested.
practical_finalists = ranked_deduped[
    (ranked_deduped["combined_signal_date_frequency"] >= MIN_COMBINED_FREQUENCY)
    & (ranked_deduped["combined_win_rate"] >= MIN_COMBINED_WIN_RATE)
    & (ranked_deduped["secondary_win_rate"] >= MIN_SECONDARY_WIN_RATE)
    & (ranked_deduped["secondary_avg_pnl_per_day"] > 0)
].sort_values(
    [
        "passes_unlock_screen",
        "combined_signal_date_frequency",
        "secondary_win_rate",
        "combined_win_rate",
        "secondary_avg_pnl_per_day",
        "combined_avg_pnl_per_day_2026",
    ],
    ascending=[False, False, False, False, False, False],
).copy()

# Suggested candidate is the top balanced unlock candidate.
if not unlock_candidates.empty:
    suggested = unlock_candidates.sort_values(
        [
            "balanced_unlock_score",
            "combined_signal_date_frequency",
            "secondary_win_rate",
            "combined_win_rate",
            "secondary_avg_pnl_per_day",
        ],
        ascending=[False, False, False, False, False],
    ).head(1).copy()
elif not practical_finalists.empty:
    suggested = practical_finalists.head(1).copy()
else:
    suggested = balanced_view.head(1).copy()

# ======================================================================================
# 5. Summaries
# ======================================================================================

summary_by_structure = ranked_deduped.groupby("candidate_structure", dropna=False).agg(
    candidates=("normalized_candidate_id", "count"),
    unlock_screen_passes=("passes_unlock_screen", "sum"),
    max_combined_frequency=("combined_signal_date_frequency", "max"),
    max_secondary_trades=("secondary_trades", "max"),
    best_secondary_win_rate=("secondary_win_rate", "max"),
    best_combined_win_rate=("combined_win_rate", "max"),
    best_secondary_avg_day=("secondary_avg_pnl_per_day", "max"),
    best_combined_2026_avg_day=("combined_avg_pnl_per_day_2026", "max"),
).reset_index()

summary_by_run = ranked_deduped.groupby("candidate_run", dropna=False).agg(
    candidates=("normalized_candidate_id", "count"),
    unlock_screen_passes=("passes_unlock_screen", "sum"),
    max_combined_frequency=("combined_signal_date_frequency", "max"),
    max_secondary_trades=("secondary_trades", "max"),
    best_secondary_win_rate=("secondary_win_rate", "max"),
    best_combined_win_rate=("combined_win_rate", "max"),
    best_secondary_avg_day=("secondary_avg_pnl_per_day", "max"),
    best_combined_2026_avg_day=("combined_avg_pnl_per_day_2026", "max"),
).reset_index()

# ======================================================================================
# 6. Validation
# ======================================================================================

validation_rows = []

add_validation(
    validation_rows,
    "cell17_comparison_loaded",
    "PASS" if not cell17_comparison.empty else "FAIL",
    f"rows={len(cell17_comparison):,}; path={cell17_comparison_path}",
)

add_validation(
    validation_rows,
    "cell16_lite_comparisons_loaded_optional",
    "PASS" if cell16_lite_paths else "SKIP",
    f"files_loaded={len(cell16_lite_paths[:3])}; paths={[str(p) for p in cell16_lite_paths[:3]]}",
)

add_validation(
    validation_rows,
    "cell18_decision_loaded_optional",
    "PASS" if cell18_decision_path else "SKIP",
    f"path={cell18_decision_path}",
)

add_validation(
    validation_rows,
    "core_baseline_loaded_optional",
    "PASS" if core_baseline_path else "SKIP",
    f"path={core_baseline_path}; core_2026_avg_day={core_2026_avg_day}",
)

add_validation(
    validation_rows,
    "candidate_rows_available",
    "PASS" if len(ranked_deduped) > 0 else "FAIL",
    f"ranked_deduped_rows={len(ranked_deduped):,}; raw_rows={len(ranked):,}",
)

add_validation(
    validation_rows,
    "current_lock_candidate_found",
    "PASS" if not current_lock.empty else "FAIL",
    f"candidate={CURRENT_SECONDARY_LOCK_CANDIDATE_ID}; rows={len(current_lock):,}",
)

add_validation(
    validation_rows,
    "unlock_screen_candidates_found",
    "PASS" if len(unlock_candidates) > 0 else "WARN",
    f"unlock_candidates={len(unlock_candidates):,}; min_frequency={MIN_COMBINED_FREQUENCY:.2%}; min_combined_win={MIN_COMBINED_WIN_RATE:.2%}; min_secondary_win={MIN_SECONDARY_WIN_RATE:.2%}",
)

add_validation(
    validation_rows,
    "no_core_changes",
    "PASS",
    "Review-only cell. Core rules are not loaded for modification and no Core artifacts are written.",
)

add_validation(
    validation_rows,
    "no_secondary_lock_written",
    "PASS",
    "Review-only cell. No Secondary lock artifacts are written.",
)

add_validation(
    validation_rows,
    "no_sizing_or_production_changes",
    "PASS",
    "No sizing, production, or intraday logic is created or changed.",
)

validation = pd.DataFrame(validation_rows)

# ======================================================================================
# 7. Save review outputs
# ======================================================================================

paths = {}

paths["all_ranked_candidates_csv"] = OUT_AUDIT_DIR / f"19R_secondary_frequency_unlock_all_ranked_candidates_{CELL19R_TIMESTAMP}.csv"
paths["all_ranked_candidates_parquet"] = OUT_PROCESSED_DIR / f"19R_secondary_frequency_unlock_all_ranked_candidates_{CELL19R_TIMESTAMP}.parquet"

paths["frequency_view_csv"] = OUT_AUDIT_DIR / f"19R_secondary_frequency_unlock_frequency_view_{CELL19R_TIMESTAMP}.csv"
paths["balanced_view_csv"] = OUT_AUDIT_DIR / f"19R_secondary_frequency_unlock_balanced_view_{CELL19R_TIMESTAMP}.csv"
paths["quality_view_csv"] = OUT_AUDIT_DIR / f"19R_secondary_frequency_unlock_quality_view_{CELL19R_TIMESTAMP}.csv"
paths["practical_finalists_csv"] = OUT_AUDIT_DIR / f"19R_secondary_frequency_unlock_practical_finalists_{CELL19R_TIMESTAMP}.csv"
paths["suggested_candidate_csv"] = OUT_AUDIT_DIR / f"19R_secondary_frequency_unlock_suggested_candidate_{CELL19R_TIMESTAMP}.csv"

paths["summary_by_structure_csv"] = OUT_AUDIT_DIR / f"19R_secondary_frequency_unlock_summary_by_structure_{CELL19R_TIMESTAMP}.csv"
paths["summary_by_run_csv"] = OUT_AUDIT_DIR / f"19R_secondary_frequency_unlock_summary_by_run_{CELL19R_TIMESTAMP}.csv"

paths["validation_csv"] = OUT_AUDIT_DIR / f"19R_secondary_frequency_unlock_validation_{CELL19R_TIMESTAMP}.csv"

ranked_deduped.to_csv(paths["all_ranked_candidates_csv"], index=False)
ranked_deduped.to_parquet(paths["all_ranked_candidates_parquet"], index=False)

frequency_view.to_csv(paths["frequency_view_csv"], index=False)
balanced_view.to_csv(paths["balanced_view_csv"], index=False)
quality_view.to_csv(paths["quality_view_csv"], index=False)
practical_finalists.to_csv(paths["practical_finalists_csv"], index=False)
suggested.to_csv(paths["suggested_candidate_csv"], index=False)

summary_by_structure.to_csv(paths["summary_by_structure_csv"], index=False)
summary_by_run.to_csv(paths["summary_by_run_csv"], index=False)

validation.to_csv(paths["validation_csv"], index=False)

manifest = {
    "cell": "Cell 19R — Secondary Frequency Unlock Review",
    "timestamp": CELL19R_TIMESTAMP,
    "version": FREQUENCY_UNLOCK_VERSION,

    "objective": "Review existing Secondary candidates for higher-frequency alternatives. No new sweep or lock.",
    "current_secondary_lock_candidate_id": CURRENT_SECONDARY_LOCK_CANDIDATE_ID,
    "current_secondary_lock_decision_id": CURRENT_SECONDARY_LOCK_DECISION_ID,

    "screens": {
        "min_combined_frequency": MIN_COMBINED_FREQUENCY,
        "min_combined_win_rate": MIN_COMBINED_WIN_RATE,
        "min_secondary_win_rate": MIN_SECONDARY_WIN_RATE,
        "min_secondary_avg_pnl_per_day": MIN_SECONDARY_AVG_DAY,
        "min_secondary_trades": MIN_SECONDARY_TRADES,
        "requires_2026_improvement_vs_core": True,
    },

    "files_loaded": files_loaded,

    "core_2026_avg_day": core_2026_avg_day,

    "current_lock_summary": {
        "combined_signal_date_frequency": current_freq,
        "secondary_trades": current_secondary_trades,
        "secondary_win_rate": current_secondary_win,
        "combined_win_rate": current_combined_win,
        "combined_avg_pnl_per_day_2026": current_combined_2026,
    },

    "row_counts": {
        "raw_candidates": int(len(ranked)),
        "deduped_candidates": int(len(ranked_deduped)),
        "unlock_screen_candidates": int(len(unlock_candidates)),
        "practical_finalists": int(len(practical_finalists)),
    },

    "suggested_candidate_id": suggested["normalized_candidate_id"].iloc[0] if len(suggested) else None,

    "constraints": [
        "Review-only.",
        "No Core changes.",
        "No Secondary lock written.",
        "No sizing changes.",
        "No production/intraday implementation.",
    ],

    "paths": {k: str(v) for k, v in paths.items()},
}

paths["manifest_json"] = OUT_AUDIT_DIR / f"19R_secondary_frequency_unlock_manifest_{CELL19R_TIMESTAMP}.json"

with open(paths["manifest_json"], "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

# ======================================================================================
# 8. Display
# ======================================================================================

display_cols = [
    "candidate_run",
    "normalized_candidate_id",
    "candidate_structure",
    "selection_mode",
    "pair_description",

    "middle_option_id",
    "middle_param_model_vrp_log_min",
    "middle_param_model_vrp_z_min",
    "middle_param_RSI14_max",
    "middle_param_rv21d_vol_pct_min",

    "back_option_id",
    "back_param_model_vrp_log_min",
    "back_param_model_vrp_z_min",
    "back_param_RSI14_max",
    "back_param_rv21d_vol_pct_min",

    "secondary_trades",
    "secondary_win_rate",
    "secondary_total_pnl",
    "secondary_avg_pnl_per_day",
    "secondary_profit_factor_pnl_per_day",
    "secondary_pnl_per_day_drawdown",
    "secondary_worst_20_trade_pnl_per_day_sum",
    "secondary_avg_pnl_per_day_2025",
    "secondary_avg_pnl_per_day_2026",

    "selected_middle_trade_count",
    "selected_back_trade_count",

    "combined_unique_trade_dates",
    "combined_signal_date_frequency",
    "combined_win_rate",
    "combined_total_pnl",
    "combined_avg_pnl_per_day",
    "combined_profit_factor_pnl_per_day",
    "combined_pnl_per_day_drawdown",
    "combined_worst_20_trade_pnl_per_day_sum",
    "combined_avg_pnl_per_day_2025",
    "combined_avg_pnl_per_day_2026",

    "delta_frequency_vs_current_lock",
    "delta_secondary_trades_vs_current_lock",
    "delta_secondary_win_vs_current_lock",
    "delta_combined_win_vs_current_lock",
    "delta_combined_2026_vs_current_lock",

    "passes_unlock_screen",
    "computed_2026_improves_vs_core",
    "frequency_unlock_score",
    "balanced_unlock_score",
    "quality_score",
]

print("=" * 100)
print("Cell 19R — Secondary Frequency Unlock Review")
print("=" * 100)
print(f"Current provisional lock candidate:       {CURRENT_SECONDARY_LOCK_CANDIDATE_ID}")
print(f"Current lock combined frequency:          {current_freq:.4%}" if pd.notna(current_freq) else "Current lock combined frequency:          NaN")
print(f"Current lock Secondary trades:            {current_secondary_trades}" if pd.notna(current_secondary_trades) else "Current lock Secondary trades:            NaN")
print(f"Current lock Secondary win rate:          {current_secondary_win:.4%}" if pd.notna(current_secondary_win) else "Current lock Secondary win rate:          NaN")
print(f"Current lock combined win rate:           {current_combined_win:.4%}" if pd.notna(current_combined_win) else "Current lock combined win rate:           NaN")
print(f"Core-only 2026 avg/day reference:         {core_2026_avg_day:.6f}" if pd.notna(core_2026_avg_day) else "Core-only 2026 avg/day reference:         NaN")
print()
print(f"Raw candidate rows:                       {len(ranked):,}")
print(f"Deduped candidate rows:                   {len(ranked_deduped):,}")
print(f"Unlock-screen pass candidates:            {len(unlock_candidates):,}")
print(f"Practical finalists:                      {len(practical_finalists):,}")
print()
print("Validation:")
display(validation)

print()
print("Summary by candidate structure:")
display(summary_by_structure)

print()
print("Summary by source run:")
display(summary_by_run)

print()
print("Current provisional lock candidate:")
display(current_lock[[c for c in display_cols if c in current_lock.columns]])

print()
print("Suggested frequency-unlock candidate:")
display(suggested[[c for c in display_cols if c in suggested.columns]])

print()
print("Top frequency candidates:")
display(frequency_view[[c for c in display_cols if c in frequency_view.columns]].head(25))

print()
print("Top balanced candidates:")
display(balanced_view[[c for c in display_cols if c in balanced_view.columns]].head(25))

print()
print("Top quality candidates:")
display(quality_view[[c for c in display_cols if c in quality_view.columns]].head(25))

print()
print("Practical finalists passing unlock screen:")
display(practical_finalists[[c for c in display_cols if c in practical_finalists.columns]].head(50))

print()
print("Saved outputs:")
for k, p in paths.items():
    print(f"  {k}: {p}")

print("\nCELL 19R PASSED — SECONDARY FREQUENCY UNLOCK REVIEW COMPLETE")

In [ ]:
# ======================================================================================
# Cell 20R — Secondary Frequency Unlock Confirmation Sweep
# ======================================================================================
#
# Objective:
#   Confirm whether to replace the provisional low-frequency Secondary lock with a
#   higher-frequency Back-led Secondary configuration.
#
# Context:
#   Current provisional Secondary lock from Cell 18R:
#       secondary_final_pair_0047__BEST_SIGNAL_RANK
#       combined frequency ≈ 15.38%
#
#   Cell 19R found higher-frequency unlock candidates, with the key candidate:
#       Middle:
#           log > 0.60
#           z > 0.60
#           RSI14 < 76
#           rv21d > 7.0
#
#       Back:
#           log > 0.65
#           z > 0.00
#           RSI14 < 77
#           rv21d > 6.5
#
#       Selection:
#           BEST_SIGNAL_RANK
#
#       Combined frequency ≈ 19.42%
#
# This Cell 20R confirmation sweep tests a narrow grid around that candidate:
#
#   Secondary Front:
#       excluded
#
#   Secondary Middle:
#       log > [0.60, 0.65]
#       z > [0.50, 0.60]
#       RSI14 < 76
#       rv21d > 7.0
#
#   Secondary Back:
#       log > [0.65, 0.70]
#       z > [0.00, 0.10, 0.20]
#       RSI14 < [76, 77]
#       rv21d > 6.5
#
# Candidate structures:
#   1. Middle-only
#   2. Back-only
#   3. Middle + Back
#
# Selection modes:
#   BACK_FIRST
#   BEST_SIGNAL_RANK
#
# Confirmation screens:
#   preferred:
#       combined frequency >= 19%
#       combined win rate >= 83%
#       Secondary win rate >= 80%
#       Secondary avg/day > 0
#       combined 2026 avg/day improves vs Core-only
#
#   acceptable:
#       combined frequency >= 18%
#       combined win rate >= 80%
#       Secondary win rate >= 78%
#       Secondary avg/day > 0
#       combined 2026 avg/day improves vs Core-only
#
# This cell does NOT:
#   - change Core
#   - include Secondary Front
#   - write a new Secondary lock
#   - change sizing
#   - implement production/intraday logic
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 900)
pd.set_option("display.width", 1400)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL20R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

CORE_LOCK_VERSION = "core_repaired_v1"
CORE_LOCK_DECISION_ID = "core_repaired_v1_locked_001"

CURRENT_PROVISIONAL_SECONDARY_LOCK_ID = "secondary_final_pair_0047__BEST_SIGNAL_RANK"
CURRENT_PROVISIONAL_SECONDARY_LOCK_DECISION_ID = "secondary_middle_back_locked_001"

SECONDARY_UNLOCK_CONFIRMATION_VERSION = "secondary_frequency_unlock_confirmation_v1"

CORE_PROGRAM_UNIVERSE = "core_program_one_trade_per_date"
SECONDARY_CONFIRM_INCREMENTAL_UNIVERSE = "secondary_frequency_unlock_confirmation_core_empty_dates"
COMBINED_CONFIRM_PROGRAM_UNIVERSE = "combined_program_core_first_else_secondary_frequency_unlock_confirmation"

SECONDARY_FRONT_EXCLUDED = True
SECONDARY_ALLOWED_BUCKETS = ["Middle", "Back"]
SECONDARY_ALLOWED_TENORS = [21, 24, 27, 30, 33]
SECONDARY_EXCLUDED_FRONT_TENORS = [12, 15, 18]

SELECTION_MODES = ["BACK_FIRST", "BEST_SIGNAL_RANK"]

# Narrow confirmation grid.
MIDDLE_LOG_GRID = [0.60, 0.65]
MIDDLE_Z_GRID = [0.50, 0.60]
MIDDLE_RSI_CAP_GRID = [76.0]
MIDDLE_RV21D_FLOOR_GRID = [7.0]

BACK_LOG_GRID = [0.65, 0.70]
BACK_Z_GRID = [0.00, 0.10, 0.20]
BACK_RSI_CAP_GRID = [76.0, 77.0]
BACK_RV21D_FLOOR_GRID = [6.5]

# Screens.
PREFERRED_MIN_COMBINED_FREQUENCY = 0.19
PREFERRED_MIN_COMBINED_WIN_RATE = 0.83
PREFERRED_MIN_SECONDARY_WIN_RATE = 0.80

ACCEPTABLE_MIN_COMBINED_FREQUENCY = 0.18
ACCEPTABLE_MIN_COMBINED_WIN_RATE = 0.80
ACCEPTABLE_MIN_SECONDARY_WIN_RATE = 0.78

MIN_SECONDARY_AVG_DAY = 0.0
MIN_SECONDARY_TRADES = 20

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def normalize_bool_series(s):
    if s.dtype == bool:
        return s.fillna(False)

    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y"])
    )


def approx_match(df, col, value, tol=1e-10):
    return np.isclose(pd.to_numeric(df[col], errors="coerce"), float(value), atol=tol, rtol=0)


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    gp = float(pnl[pnl > 0].sum())
    gl = float(pnl[pnl < 0].sum())

    if gl < 0:
        return gp / abs(gl)

    if gp > 0:
        return np.inf

    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def summarize_trade_set(df, group_cols, total_eligible_dates=None):
    d = df.copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    for c in [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "forecast_vol_pct",
        "rv21d_vol_pct",
        "threshold_model_vrp_log",
        "threshold_model_vrp_z_3m",
        "threshold_model_vrp_z_1y",
        "threshold_RSI14_max",
        "threshold_rv21d_vol_pct_min",
        "secondary_threshold_model_vrp_log",
        "secondary_threshold_model_vrp_z_3m",
        "secondary_threshold_model_vrp_z_1y",
        "secondary_threshold_RSI14_max",
        "secondary_threshold_rv21d_vol_pct_min",
    ]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d = d.sort_values(["trade_date", "tenor"]).copy()

    grouped = d.groupby(group_cols, dropna=False, observed=False) if group_cols else [((), d)]

    rows = []

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {}
        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2022 = g[g["trade_date"].dt.year == 2022].copy()
        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        unique_dates = int(g["trade_date"].nunique())
        signal_date_frequency = (
            unique_dates / total_eligible_dates
            if total_eligible_dates and total_eligible_dates > 0
            else np.nan
        )

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": unique_dates,
            "signal_date_frequency": float(signal_date_frequency) if pd.notna(signal_date_frequency) else np.nan,
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),

            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,

            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_25k": int((pnl <= -25_000).sum()),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2022": int(len(y2022)),
            "pnl_2022": float(y2022["normalized_pnl_dollars"].sum()) if len(y2022) else 0.0,
            "avg_pnl_per_day_2022": float(y2022["normalized_pnl_per_day"].mean()) if len(y2022) else np.nan,
            "worst_trade_2022": float(y2022["normalized_pnl_dollars"].min()) if len(y2022) else np.nan,

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        if "program_leg" in g.columns:
            row["core_trade_count"] = int(g["program_leg"].eq("Core").sum())
            row["secondary_trade_count"] = int(g["program_leg"].str.startswith("Secondary", na=False).sum())
            row["secondary_middle_trade_count"] = int(g["program_leg"].eq("Secondary_Middle").sum())
            row["secondary_back_trade_count"] = int(g["program_leg"].eq("Secondary_Back").sum())

        for signal_col in [
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "threshold_model_vrp_log",
            "threshold_model_vrp_z_3m",
            "threshold_model_vrp_z_1y",
            "threshold_RSI14_max",
            "threshold_rv21d_vol_pct_min",
            "secondary_threshold_model_vrp_log",
            "secondary_threshold_model_vrp_z_3m",
            "secondary_threshold_model_vrp_z_1y",
            "secondary_threshold_RSI14_max",
            "secondary_threshold_rv21d_vol_pct_min",
        ]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        rows.append(row)

    return pd.DataFrame(rows)


def add_rank_columns(d, group_cols, suffix):
    x = d.copy()

    if x.empty:
        return x

    x[f"rank_z_1y_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )

    x[f"rank_z_3m_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )

    x[f"rank_vrp_log_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    x[f"avg_signal_rank_{suffix}"] = x[
        [
            f"rank_z_1y_{suffix}",
            f"rank_z_3m_{suffix}",
            f"rank_vrp_log_{suffix}",
        ]
    ].mean(axis=1)

    if "effective_tenor_bucket" in x.columns:
        x["bucket_center_tenor"] = x["effective_tenor_bucket"].map({
            "Front": 15,
            "Middle": 21,
            "Back": 30,
        })
    elif "secondary_bucket" in x.columns:
        x["bucket_center_tenor"] = x["secondary_bucket"].map({
            "Middle": 21,
            "Back": 30,
        })
    else:
        x["bucket_center_tenor"] = 30

    x["bucket_center_tenor"] = pd.to_numeric(x["bucket_center_tenor"], errors="coerce").fillna(30)
    x["distance_to_bucket_center"] = (x["tenor"] - x["bucket_center_tenor"]).abs()

    return x


def select_core_universes(core_qualified_complete):
    q = core_qualified_complete.copy()

    if q.empty:
        return {
            CORE_PROGRAM_UNIVERSE: q.copy(),
        }

    q = add_rank_columns(
        q,
        group_cols=["trade_date", "effective_tenor_bucket"],
        suffix="core_bucket_date",
    )

    q = add_rank_columns(
        q,
        group_cols=["trade_date"],
        suffix="core_date",
    )

    core_program = (
        q.sort_values(
            [
                "trade_date",
                "avg_signal_rank_core_date",
                "rank_z_1y_core_date",
                "rank_z_3m_core_date",
                "rank_vrp_log_core_date",
                "distance_to_bucket_center",
                "effective_tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return {
        CORE_PROGRAM_UNIVERSE: core_program,
    }


def find_scorecard_candidate(scorecards, bucket, log_min, z_min, rsi_max, rv_min, label):
    subset = scorecards[
        scorecards["secondary_bucket"].eq(bucket)
        & approx_match(scorecards, "param_model_vrp_log_min", log_min)
        & approx_match(scorecards, "param_model_vrp_z_min", z_min)
        & approx_match(scorecards, "param_RSI14_max", rsi_max)
        & approx_match(scorecards, "param_rv21d_vol_pct_min", rv_min)
    ].copy()

    if subset.empty:
        raise ValueError(
            f"No Cell 15R scorecard candidate found for {label}: "
            f"{bucket}, log>{log_min}, z>{z_min}, RSI<{rsi_max}, rv>{rv_min}"
        )

    sort_cols = [
        c for c in [
            "passes_bucket_quality_ideal",
            "passes_bucket_quality_floor",
            "incremental_win_rate",
            "combined_win_rate",
            "incremental_avg_pnl_per_day",
        ]
        if c in subset.columns
    ]

    if sort_cols:
        subset = subset.sort_values(
            sort_cols,
            ascending=[False] * len(sort_cols),
        )

    return subset.iloc[0].to_dict()


def build_option_table(scorecards, specs, option_type):
    rows = []

    for spec in specs:
        if spec["option_id"].startswith("NO_"):
            rows.append({
                f"{option_type}_option_id": spec["option_id"],
                f"{option_type}_candidate_id": None,
                f"{option_type}_description": spec["description"],
                f"{option_type}_is_none": True,
                f"{option_type}_param_model_vrp_log_min": np.nan,
                f"{option_type}_param_model_vrp_z_min": np.nan,
                f"{option_type}_param_RSI14_max": np.nan,
                f"{option_type}_param_rv21d_vol_pct_min": np.nan,
            })
            continue

        row = find_scorecard_candidate(
            scorecards=scorecards,
            bucket=spec["bucket"],
            log_min=spec["log_min"],
            z_min=spec["z_min"],
            rsi_max=spec["rsi_max"],
            rv_min=spec["rv_min"],
            label=spec["option_id"],
        )

        rows.append({
            f"{option_type}_option_id": spec["option_id"],
            f"{option_type}_candidate_id": row["secondary_candidate_id"],
            f"{option_type}_description": spec["description"],
            f"{option_type}_is_none": False,

            f"{option_type}_param_model_vrp_log_min": row["param_model_vrp_log_min"],
            f"{option_type}_param_model_vrp_z_min": row["param_model_vrp_z_min"],
            f"{option_type}_param_RSI14_max": row["param_RSI14_max"],
            f"{option_type}_param_rv21d_vol_pct_min": row["param_rv21d_vol_pct_min"],

            f"{option_type}_incremental_trades": row.get("incremental_trades", np.nan),
            f"{option_type}_incremental_unique_trade_dates": row.get("incremental_unique_trade_dates", np.nan),
            f"{option_type}_incremental_win_rate": row.get("incremental_win_rate", np.nan),
            f"{option_type}_incremental_signal_date_frequency": row.get("incremental_signal_date_frequency", np.nan),
            f"{option_type}_incremental_total_pnl": row.get("incremental_total_pnl", np.nan),
            f"{option_type}_incremental_avg_pnl_per_day": row.get("incremental_avg_pnl_per_day", np.nan),
            f"{option_type}_incremental_profit_factor_pnl_per_day": row.get("incremental_profit_factor_pnl_per_day", np.nan),
            f"{option_type}_incremental_pnl_per_day_drawdown": row.get("incremental_pnl_per_day_drawdown", np.nan),
            f"{option_type}_incremental_worst_20_trade_pnl_per_day_sum": row.get("incremental_worst_20_trade_pnl_per_day_sum", np.nan),
            f"{option_type}_incremental_avg_pnl_per_day_2022": row.get("incremental_avg_pnl_per_day_2022", np.nan),
            f"{option_type}_incremental_avg_pnl_per_day_2025": row.get("incremental_avg_pnl_per_day_2025", np.nan),
            f"{option_type}_incremental_avg_pnl_per_day_2026": row.get("incremental_avg_pnl_per_day_2026", np.nan),

            f"{option_type}_combined_win_rate": row.get("combined_win_rate", np.nan),
            f"{option_type}_combined_signal_date_frequency": row.get("combined_signal_date_frequency", np.nan),
            f"{option_type}_combined_total_pnl": row.get("combined_total_pnl", np.nan),
            f"{option_type}_combined_avg_pnl_per_day": row.get("combined_avg_pnl_per_day", np.nan),
            f"{option_type}_combined_profit_factor_pnl_per_day": row.get("combined_profit_factor_pnl_per_day", np.nan),
            f"{option_type}_combined_pnl_per_day_drawdown": row.get("combined_pnl_per_day_drawdown", np.nan),
            f"{option_type}_combined_worst_20_trade_pnl_per_day_sum": row.get("combined_worst_20_trade_pnl_per_day_sum", np.nan),
            f"{option_type}_combined_avg_pnl_per_day_2025": row.get("combined_avg_pnl_per_day_2025", np.nan),
            f"{option_type}_combined_avg_pnl_per_day_2026": row.get("combined_avg_pnl_per_day_2026", np.nan),

            f"{option_type}_passes_bucket_quality_floor": row.get("passes_bucket_quality_floor", False),
            f"{option_type}_passes_bucket_quality_ideal": row.get("passes_bucket_quality_ideal", False),
            f"{option_type}_bucket_score": row.get("bucket_score", np.nan),
        })

    return pd.DataFrame(rows)


def select_best_signal(pool):
    if pool.empty:
        return pool

    q = add_rank_columns(
        pool,
        group_cols=["secondary_confirm_pair_id", "selection_mode", "trade_date"],
        suffix="confirm_pair_date",
    )

    q["bucket_priority_best_signal"] = q["secondary_bucket"].map({
        "Back": 1,
        "Middle": 2,
    }).fillna(9)

    return (
        q.sort_values(
            [
                "secondary_confirm_pair_id",
                "selection_mode",
                "trade_date",
                "avg_signal_rank_confirm_pair_date",
                "distance_to_bucket_center",
                "bucket_priority_best_signal",
                "rank_z_1y_confirm_pair_date",
                "rank_z_3m_confirm_pair_date",
                "rank_vrp_log_confirm_pair_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True, True, True],
        )
        .groupby(["secondary_confirm_pair_id", "selection_mode", "trade_date"], as_index=False)
        .head(1)
        .copy()
    )


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


# ======================================================================================
# 2. Load inputs
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01R_unified_fds_signal_base_panel_with_rv21d_*.parquet",
    required=True,
)

locked_core_rules_long_path = latest_file(
    OUT_PROCESSED_DIR,
    "11R_core_repaired_v1_locked_rules_long_*.parquet",
    required=True,
)

cell15_selected_all_path = latest_file(
    OUT_PROCESSED_DIR,
    "15R_secondary_independent_selected_panel_all_*.parquet",
    required=True,
)

cell15_scorecards_all_path = latest_file(
    OUT_AUDIT_DIR,
    "15R_secondary_independent_scorecards_all_*.csv",
    required=True,
)

cell18_decision_path = latest_file(
    OUT_AUDIT_DIR,
    "18R_core_secondary_lock_decision_summary_*.csv",
    required=False,
)

cell19_suggested_path = latest_file(
    OUT_AUDIT_DIR,
    "19R_secondary_frequency_unlock_suggested_candidate_*.csv",
    required=False,
)

cell19_practical_path = latest_file(
    OUT_AUDIT_DIR,
    "19R_secondary_frequency_unlock_practical_finalists_*.csv",
    required=False,
)

base_raw = pd.read_parquet(base_panel_path)
locked_core_rules_long = pd.read_parquet(locked_core_rules_long_path)
cell15_selected_all = pd.read_parquet(cell15_selected_all_path)
cell15_scorecards_all = pd.read_csv(cell15_scorecards_all_path)

cell18_decision = pd.read_csv(cell18_decision_path) if cell18_decision_path else pd.DataFrame()
cell19_suggested = pd.read_csv(cell19_suggested_path) if cell19_suggested_path else pd.DataFrame()
cell19_practical = pd.read_csv(cell19_practical_path) if cell19_practical_path else pd.DataFrame()

base_raw_models_found = sorted(base_raw["model_spec"].dropna().unique().tolist()) if "model_spec" in base_raw.columns else []

base = base_raw.copy()
if "model_spec" in base.columns:
    base = base[base["model_spec"].eq(LOCKED_MODEL_SPEC)].copy()

cell15_selected_all = cell15_selected_all.copy()
if "model_spec" in cell15_selected_all.columns:
    cell15_selected_all = cell15_selected_all[cell15_selected_all["model_spec"].eq(LOCKED_MODEL_SPEC)].copy()

base["trade_date"] = pd.to_datetime(base["trade_date"], errors="coerce").dt.normalize()
cell15_selected_all["trade_date"] = pd.to_datetime(cell15_selected_all["trade_date"], errors="coerce").dt.normalize()

for c in [
    "tenor",
    "tenor_bucket_order",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")

for c in [
    "tenor",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "param_model_vrp_log_min",
    "param_model_vrp_z_min",
    "param_model_vrp_z_3m_min",
    "param_model_vrp_z_1y_min",
    "param_RSI14_max",
    "param_rv21d_vol_pct_min",
]:
    if c in cell15_selected_all.columns:
        cell15_selected_all[c] = pd.to_numeric(cell15_selected_all[c], errors="coerce")

for c in [
    "param_model_vrp_log_min",
    "param_model_vrp_z_min",
    "param_model_vrp_z_3m_min",
    "param_model_vrp_z_1y_min",
    "param_RSI14_max",
    "param_rv21d_vol_pct_min",
    "incremental_trades",
    "incremental_unique_trade_dates",
    "incremental_signal_date_frequency",
    "incremental_win_rate",
    "incremental_total_pnl",
    "incremental_avg_pnl_per_day",
    "incremental_profit_factor_pnl_per_day",
    "incremental_pnl_per_day_drawdown",
    "incremental_worst_20_trade_pnl_per_day_sum",
    "incremental_avg_pnl_per_day_2022",
    "incremental_avg_pnl_per_day_2025",
    "incremental_avg_pnl_per_day_2026",
    "combined_unique_trade_dates",
    "combined_signal_date_frequency",
    "combined_win_rate",
    "combined_total_pnl",
    "combined_avg_pnl_per_day",
    "combined_profit_factor_pnl_per_day",
    "combined_pnl_per_day_drawdown",
    "combined_worst_20_trade_pnl_per_day_sum",
    "combined_avg_pnl_per_day_2025",
    "combined_avg_pnl_per_day_2026",
    "bucket_score",
]:
    if c in cell15_scorecards_all.columns:
        cell15_scorecards_all[c] = pd.to_numeric(cell15_scorecards_all[c], errors="coerce")

for c in ["passes_bucket_quality_floor", "passes_bucket_quality_ideal"]:
    if c in cell15_scorecards_all.columns:
        cell15_scorecards_all[c] = normalize_bool_series(cell15_scorecards_all[c])

for c in [
    "tenor",
    "original_tenor_bucket_order",
    "effective_tenor_bucket_order",
    "model_vrp_log_min",
    "model_vrp_z_min",
    "model_vrp_z_3m_min",
    "model_vrp_z_1y_min",
    "RSI14_max",
    "rv21d_vol_pct_min",
]:
    if c in locked_core_rules_long.columns:
        locked_core_rules_long[c] = pd.to_numeric(locked_core_rules_long[c], errors="coerce")

total_eligible_signal_dates = int(base["trade_date"].nunique())

# ======================================================================================
# 3. Rebuild locked Core gate and baseline
# ======================================================================================

core_thresholds = locked_core_rules_long.rename(columns={
    "model_vrp_log_min": "threshold_model_vrp_log",
    "model_vrp_z_3m_min": "threshold_model_vrp_z_3m",
    "model_vrp_z_1y_min": "threshold_model_vrp_z_1y",
    "RSI14_max": "threshold_RSI14_max",
    "rv21d_vol_pct_min": "threshold_rv21d_vol_pct_min",
})

core_threshold_cols = [
    "tenor",
    "original_tenor_bucket",
    "original_tenor_bucket_order",
    "effective_tenor_bucket",
    "effective_tenor_bucket_order",
    "include_tenor",
    "threshold_model_vrp_log",
    "threshold_model_vrp_z_3m",
    "threshold_model_vrp_z_1y",
    "threshold_RSI14_max",
    "threshold_rv21d_vol_pct_min",
]

base_for_join = base.rename(columns={
    "tenor_bucket": "source_tenor_bucket",
    "tenor_bucket_order": "source_tenor_bucket_order",
}).copy()

core_panel = base_for_join.merge(
    core_thresholds[core_threshold_cols],
    on="tenor",
    how="inner",
    validate="many_to_one",
)

core_panel["include_tenor_bool"] = normalize_bool_series(core_panel["include_tenor"])

core_signal_available = (
    core_panel["include_tenor_bool"]
    & core_panel["model_vrp_log"].notna()
    & core_panel["model_vrp_z_3m"].notna()
    & core_panel["model_vrp_z_1y"].notna()
    & core_panel["RSI14"].notna()
    & core_panel["rv21d_vol_pct"].notna()
)

core_pass_mask = (
    core_signal_available
    & (core_panel["model_vrp_log"] > core_panel["threshold_model_vrp_log"])
    & (core_panel["model_vrp_z_3m"] > core_panel["threshold_model_vrp_z_3m"])
    & (core_panel["model_vrp_z_1y"] > core_panel["threshold_model_vrp_z_1y"])
    & (core_panel["RSI14"] < core_panel["threshold_RSI14_max"])
    & (core_panel["rv21d_vol_pct"] > core_panel["threshold_rv21d_vol_pct_min"])
)

core_panel["core_repaired_v1_pass"] = core_pass_mask

core_qualified_all_dates = core_panel[core_pass_mask].copy()

core_qualified_complete = core_panel[
    core_pass_mask
    & core_panel["normalized_pnl_dollars"].notna()
    & core_panel["normalized_pnl_per_day"].notna()
].copy()

core_gate_dates = sorted(pd.to_datetime(core_qualified_all_dates["trade_date"].dropna().unique()).tolist())
all_signal_dates = sorted(pd.to_datetime(base["trade_date"].dropna().unique()).tolist())

core_gate_dates_set = set(core_gate_dates)
core_empty_dates = sorted(set(all_signal_dates) - core_gate_dates_set)

core_universes = select_core_universes(core_qualified_complete)
core_program = core_universes[CORE_PROGRAM_UNIVERSE].copy()

core_program["source_layer"] = "Core"
core_program["program_leg"] = "Core"
core_program["selection_universe"] = CORE_PROGRAM_UNIVERSE

stale_secondary_cols = [
    "secondary_bucket",
    "secondary_bucket_order",
    "secondary_candidate_id",
    "secondary_candidate_description",
    "param_model_vrp_log_min",
    "param_model_vrp_z_min",
    "param_model_vrp_z_3m_min",
    "param_model_vrp_z_1y_min",
    "param_RSI14_max",
    "param_rv21d_vol_pct_min",
]

core_program_clean = core_program.drop(
    columns=[c for c in stale_secondary_cols if c in core_program.columns],
    errors="ignore",
).copy()

core_summary = summarize_trade_set(
    core_program_clean.assign(selection_universe=CORE_PROGRAM_UNIVERSE),
    group_cols=["selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

core_ref = core_summary.iloc[0].to_dict() if len(core_summary) else {}

# ======================================================================================
# 4. Build confirmation option grids
# ======================================================================================

middle_specs = [
    {
        "option_id": f"M_LOG{int(log_min * 100):03d}_Z{int(round(z_min * 100)):03d}_RSI76_RV70",
        "bucket": "Middle",
        "description": f"Middle confirm: log>{log_min:.2f}, z>{z_min:.2f}, RSI<76, rv21d>7.0",
        "log_min": log_min,
        "z_min": z_min,
        "rsi_max": 76.0,
        "rv_min": 7.0,
    }
    for log_min in MIDDLE_LOG_GRID
    for z_min in MIDDLE_Z_GRID
]

back_specs = [
    {
        "option_id": f"B_LOG{int(log_min * 100):03d}_Z{int(round(z_min * 100)):03d}_RSI{int(rsi_max)}_RV65",
        "bucket": "Back",
        "description": f"Back confirm: log>{log_min:.2f}, z>{z_min:.2f}, RSI<{int(rsi_max)}, rv21d>6.5",
        "log_min": log_min,
        "z_min": z_min,
        "rsi_max": rsi_max,
        "rv_min": 6.5,
    }
    for log_min in BACK_LOG_GRID
    for z_min in BACK_Z_GRID
    for rsi_max in BACK_RSI_CAP_GRID
]

middle_options = pd.concat(
    [
        pd.DataFrame([{
            "middle_option_id": "NO_MIDDLE",
            "middle_candidate_id": None,
            "middle_description": "No Secondary Middle",
            "middle_is_none": True,
            "middle_param_model_vrp_log_min": np.nan,
            "middle_param_model_vrp_z_min": np.nan,
            "middle_param_RSI14_max": np.nan,
            "middle_param_rv21d_vol_pct_min": np.nan,
        }]),
        build_option_table(cell15_scorecards_all, middle_specs, "middle"),
    ],
    ignore_index=True,
)

back_options = pd.concat(
    [
        pd.DataFrame([{
            "back_option_id": "NO_BACK",
            "back_candidate_id": None,
            "back_description": "No Secondary Back",
            "back_is_none": True,
            "back_param_model_vrp_log_min": np.nan,
            "back_param_model_vrp_z_min": np.nan,
            "back_param_RSI14_max": np.nan,
            "back_param_rv21d_vol_pct_min": np.nan,
        }]),
        build_option_table(cell15_scorecards_all, back_specs, "back"),
    ],
    ignore_index=True,
)

pair_grid = middle_options.merge(back_options, how="cross")
pair_grid = pair_grid[
    ~(pair_grid["middle_is_none"] & pair_grid["back_is_none"])
].copy()

pair_grid = pair_grid.reset_index(drop=True)

pair_grid["secondary_confirm_pair_id"] = [
    f"secondary_confirm_pair_{i + 1:04d}" for i in range(len(pair_grid))
]

pair_grid["candidate_structure"] = np.select(
    [
        pair_grid["middle_is_none"].eq(False) & pair_grid["back_is_none"].eq(True),
        pair_grid["middle_is_none"].eq(True) & pair_grid["back_is_none"].eq(False),
        pair_grid["middle_is_none"].eq(False) & pair_grid["back_is_none"].eq(False),
    ],
    [
        "MIDDLE_ONLY",
        "BACK_ONLY",
        "MIDDLE_BACK",
    ],
    default="UNKNOWN",
)

pair_grid["secondary_unlock_confirmation_version"] = SECONDARY_UNLOCK_CONFIRMATION_VERSION
pair_grid["secondary_front_excluded"] = True
pair_grid["pair_description"] = pair_grid["middle_option_id"] + " + " + pair_grid["back_option_id"]

pair_mode_grid = pair_grid.merge(
    pd.DataFrame({"selection_mode": SELECTION_MODES}),
    how="cross",
)

pair_mode_grid["secondary_confirm_pair_mode_id"] = (
    pair_mode_grid["secondary_confirm_pair_id"]
    + "__"
    + pair_mode_grid["selection_mode"]
)

expected_middle_options = 1 + len(middle_specs)
expected_back_options = 1 + len(back_specs)
expected_pair_count = expected_middle_options * expected_back_options - 1
expected_pair_mode_count = expected_pair_count * len(SELECTION_MODES)

# ======================================================================================
# 5. Prepare selected Middle/Back rows from Cell 15R
# ======================================================================================

selected_cols = [
    "trade_date",
    "tenor",
    "source_tenor_bucket",
    "source_tenor_bucket_order",
    "model_spec",

    "implied_variance",
    "forecast_variance",
    "target_realized_variance",
    "implied_vol_pct",
    "forecast_vol_pct",
    "rv21d_vol_pct",

    "model_vrp_log",
    "model_vrp_ratio",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",

    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",

    "secondary_bucket",
    "secondary_bucket_order",
    "bucket_center_tenor",
    "secondary_candidate_id",
    "secondary_candidate_description",
    "secondary_tenors",

    "secondary_threshold_model_vrp_log",
    "secondary_threshold_model_vrp_z_3m",
    "secondary_threshold_model_vrp_z_1y",
    "secondary_threshold_RSI14_max",
    "secondary_threshold_rv21d_vol_pct_min",

    "param_model_vrp_log_min",
    "param_model_vrp_z_min",
    "param_model_vrp_z_3m_min",
    "param_model_vrp_z_1y_min",
    "param_RSI14_max",
    "param_rv21d_vol_pct_min",
]

selected_base = cell15_selected_all[
    [c for c in selected_cols if c in cell15_selected_all.columns]
].copy()

middle_candidate_ids = sorted(middle_options["middle_candidate_id"].dropna().unique().tolist())
back_candidate_ids = sorted(back_options["back_candidate_id"].dropna().unique().tolist())

selected_middle_base = selected_base[
    selected_base["secondary_bucket"].eq("Middle")
    & selected_base["secondary_candidate_id"].isin(middle_candidate_ids)
    & selected_base["trade_date"].isin(core_empty_dates)
].copy()

selected_back_base = selected_base[
    selected_base["secondary_bucket"].eq("Back")
    & selected_base["secondary_candidate_id"].isin(back_candidate_ids)
    & selected_base["trade_date"].isin(core_empty_dates)
].copy()

selected_middle_base = selected_middle_base.rename(columns={
    "secondary_candidate_id": "middle_candidate_id",
})

selected_back_base = selected_back_base.rename(columns={
    "secondary_candidate_id": "back_candidate_id",
})

pair_map = pair_grid[
    [
        "secondary_confirm_pair_id",
        "candidate_structure",
        "middle_option_id",
        "back_option_id",
        "middle_candidate_id",
        "back_candidate_id",
        "middle_is_none",
        "back_is_none",
    ]
].copy()

middle_pair_rows = selected_middle_base.merge(
    pair_map[
        [
            "secondary_confirm_pair_id",
            "candidate_structure",
            "middle_option_id",
            "back_option_id",
            "middle_candidate_id",
            "back_candidate_id",
        ]
    ].dropna(subset=["middle_candidate_id"]),
    on="middle_candidate_id",
    how="inner",
    validate="many_to_many",
)

middle_pair_rows["secondary_candidate_id"] = middle_pair_rows["middle_candidate_id"]
middle_pair_rows["program_leg"] = "Secondary_Middle"
middle_pair_rows["source_layer"] = "Secondary"

back_pair_rows = selected_back_base.merge(
    pair_map[
        [
            "secondary_confirm_pair_id",
            "candidate_structure",
            "middle_option_id",
            "back_option_id",
            "middle_candidate_id",
            "back_candidate_id",
        ]
    ].dropna(subset=["back_candidate_id"]),
    on="back_candidate_id",
    how="inner",
    validate="many_to_many",
)

back_pair_rows["secondary_candidate_id"] = back_pair_rows["back_candidate_id"]
back_pair_rows["program_leg"] = "Secondary_Back"
back_pair_rows["source_layer"] = "Secondary"

# ======================================================================================
# 6. Select Secondary rows by candidate/date/mode
# ======================================================================================

key_cols = [
    "secondary_confirm_pair_id",
    "trade_date",
]

middle_presence = middle_pair_rows[key_cols].drop_duplicates().copy()
middle_presence["middle_present"] = True

back_presence = back_pair_rows[key_cols].drop_duplicates().copy()
back_presence["back_present"] = True

presence = middle_presence.merge(
    back_presence,
    on=key_cols,
    how="outer",
)

presence["middle_present"] = presence["middle_present"].eq(True)
presence["back_present"] = presence["back_present"].eq(True)

# BACK_FIRST selection.
back_first_back_keys = presence[presence["back_present"]][key_cols].copy()
back_first_middle_keys = presence[
    presence["middle_present"] & ~presence["back_present"]
][key_cols].copy()

back_first_back = back_pair_rows.merge(
    back_first_back_keys,
    on=key_cols,
    how="inner",
    validate="one_to_one",
)

back_first_middle = middle_pair_rows.merge(
    back_first_middle_keys,
    on=key_cols,
    how="inner",
    validate="one_to_one",
)

selected_back_first = pd.concat(
    [
        back_first_back,
        back_first_middle,
    ],
    ignore_index=True,
)

selected_back_first["selection_mode"] = "BACK_FIRST"

# BEST_SIGNAL_RANK selection.
best_signal_pool = pd.concat(
    [
        middle_pair_rows,
        back_pair_rows,
    ],
    ignore_index=True,
)

best_signal_pool["selection_mode"] = "BEST_SIGNAL_RANK"

selected_best_signal = select_best_signal(best_signal_pool)

secondary_selected = pd.concat(
    [
        selected_back_first,
        selected_best_signal,
    ],
    ignore_index=True,
)

secondary_selected["secondary_confirm_pair_mode_id"] = (
    secondary_selected["secondary_confirm_pair_id"]
    + "__"
    + secondary_selected["selection_mode"]
)

secondary_selected["selection_universe"] = SECONDARY_CONFIRM_INCREMENTAL_UNIVERSE
secondary_selected["secondary_front_excluded"] = True

# ======================================================================================
# 7. Summaries
# ======================================================================================

secondary_summary = summarize_trade_set(
    secondary_selected,
    group_cols=[
        "secondary_confirm_pair_id",
        "selection_mode",
        "secondary_confirm_pair_mode_id",
        "selection_universe",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

secondary_by_bucket = summarize_trade_set(
    secondary_selected,
    group_cols=[
        "secondary_confirm_pair_id",
        "selection_mode",
        "secondary_confirm_pair_mode_id",
        "secondary_bucket",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

secondary_by_tenor = summarize_trade_set(
    secondary_selected,
    group_cols=[
        "secondary_confirm_pair_id",
        "selection_mode",
        "secondary_confirm_pair_mode_id",
        "secondary_bucket",
        "tenor",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

secondary_by_year_input = secondary_selected.assign(
    year=secondary_selected["trade_date"].dt.year.astype(int)
)

secondary_by_year = summarize_trade_set(
    secondary_by_year_input,
    group_cols=[
        "secondary_confirm_pair_id",
        "selection_mode",
        "secondary_confirm_pair_mode_id",
        "year",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

secondary_by_bucket_year = summarize_trade_set(
    secondary_by_year_input,
    group_cols=[
        "secondary_confirm_pair_id",
        "selection_mode",
        "secondary_confirm_pair_mode_id",
        "secondary_bucket",
        "year",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

# ======================================================================================
# 8. Combined program: Core first, else selected Secondary
# ======================================================================================

core_program_pair_mode = core_program_clean.merge(
    pair_mode_grid[
        [
            "secondary_confirm_pair_id",
            "selection_mode",
            "secondary_confirm_pair_mode_id",
            "candidate_structure",
            "middle_option_id",
            "back_option_id",
            "middle_candidate_id",
            "back_candidate_id",
        ]
    ],
    how="cross",
)

core_program_pair_mode["source_layer"] = "Core"
core_program_pair_mode["program_leg"] = "Core"
core_program_pair_mode["selection_universe"] = COMBINED_CONFIRM_PROGRAM_UNIVERSE
core_program_pair_mode["secondary_front_excluded"] = True

secondary_for_combined = secondary_selected.copy()
secondary_for_combined["selection_universe"] = COMBINED_CONFIRM_PROGRAM_UNIVERSE

combined_program_panel = pd.concat(
    [
        core_program_pair_mode,
        secondary_for_combined,
    ],
    ignore_index=True,
)

combined_program_summary = summarize_trade_set(
    combined_program_panel,
    group_cols=[
        "secondary_confirm_pair_id",
        "selection_mode",
        "secondary_confirm_pair_mode_id",
        "selection_universe",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

combined_program_by_year_input = combined_program_panel.assign(
    year=combined_program_panel["trade_date"].dt.year.astype(int)
)

combined_program_by_year = summarize_trade_set(
    combined_program_by_year_input,
    group_cols=[
        "secondary_confirm_pair_id",
        "selection_mode",
        "secondary_confirm_pair_mode_id",
        "year",
    ],
    total_eligible_dates=total_eligible_signal_dates,
)

# ======================================================================================
# 9. Candidate comparison and confirmation screens
# ======================================================================================

metadata_cols = [
    "secondary_confirm_pair_id",
    "selection_mode",
    "secondary_confirm_pair_mode_id",
    "candidate_structure",
    "pair_description",

    "middle_option_id",
    "middle_candidate_id",
    "middle_description",
    "middle_is_none",
    "middle_param_model_vrp_log_min",
    "middle_param_model_vrp_z_min",
    "middle_param_RSI14_max",
    "middle_param_rv21d_vol_pct_min",
    "middle_incremental_trades",
    "middle_incremental_win_rate",
    "middle_incremental_signal_date_frequency",
    "middle_incremental_total_pnl",
    "middle_incremental_avg_pnl_per_day",
    "middle_incremental_profit_factor_pnl_per_day",
    "middle_incremental_pnl_per_day_drawdown",
    "middle_incremental_worst_20_trade_pnl_per_day_sum",
    "middle_incremental_avg_pnl_per_day_2025",
    "middle_incremental_avg_pnl_per_day_2026",
    "middle_passes_bucket_quality_floor",
    "middle_passes_bucket_quality_ideal",

    "back_option_id",
    "back_candidate_id",
    "back_description",
    "back_is_none",
    "back_param_model_vrp_log_min",
    "back_param_model_vrp_z_min",
    "back_param_RSI14_max",
    "back_param_rv21d_vol_pct_min",
    "back_incremental_trades",
    "back_incremental_win_rate",
    "back_incremental_signal_date_frequency",
    "back_incremental_total_pnl",
    "back_incremental_avg_pnl_per_day",
    "back_incremental_profit_factor_pnl_per_day",
    "back_incremental_pnl_per_day_drawdown",
    "back_incremental_worst_20_trade_pnl_per_day_sum",
    "back_incremental_avg_pnl_per_day_2025",
    "back_incremental_avg_pnl_per_day_2026",
    "back_passes_bucket_quality_floor",
    "back_passes_bucket_quality_ideal",

    "secondary_front_excluded",
    "secondary_unlock_confirmation_version",
]

metadata = pair_mode_grid[
    [c for c in metadata_cols if c in pair_mode_grid.columns]
].copy()

join_keys = [
    "secondary_confirm_pair_id",
    "selection_mode",
    "secondary_confirm_pair_mode_id",
]

secondary_metrics = secondary_summary.copy().rename(columns={
    "trades": "secondary_trades",
    "unique_trade_dates": "secondary_unique_trade_dates",
    "signal_date_frequency": "secondary_signal_date_frequency",
    "win_rate": "secondary_win_rate",
    "total_pnl": "secondary_total_pnl",
    "avg_pnl": "secondary_avg_pnl",
    "avg_pnl_per_day": "secondary_avg_pnl_per_day",
    "profit_factor": "secondary_profit_factor",
    "profit_factor_pnl_per_day": "secondary_profit_factor_pnl_per_day",
    "pnl_per_day_drawdown": "secondary_pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum": "secondary_worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2022": "secondary_avg_pnl_per_day_2022",
    "avg_pnl_per_day_2025": "secondary_avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026": "secondary_avg_pnl_per_day_2026",
    "secondary_middle_trade_count": "selected_middle_trade_count",
    "secondary_back_trade_count": "selected_back_trade_count",
})

combined_metrics = combined_program_summary.copy().rename(columns={
    "trades": "combined_trades",
    "unique_trade_dates": "combined_unique_trade_dates",
    "signal_date_frequency": "combined_signal_date_frequency",
    "win_rate": "combined_win_rate",
    "total_pnl": "combined_total_pnl",
    "avg_pnl": "combined_avg_pnl",
    "avg_pnl_per_day": "combined_avg_pnl_per_day",
    "profit_factor": "combined_profit_factor",
    "profit_factor_pnl_per_day": "combined_profit_factor_pnl_per_day",
    "pnl_per_day_drawdown": "combined_pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum": "combined_worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2022": "combined_avg_pnl_per_day_2022",
    "avg_pnl_per_day_2025": "combined_avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026": "combined_avg_pnl_per_day_2026",
    "core_trade_count": "combined_core_trade_count",
    "secondary_trade_count": "combined_secondary_trade_count",
    "secondary_middle_trade_count": "combined_secondary_middle_trade_count",
    "secondary_back_trade_count": "combined_secondary_back_trade_count",
})

comparison = metadata.merge(
    secondary_metrics[[c for c in secondary_metrics.columns if c != "selection_universe"]],
    on=join_keys,
    how="left",
    validate="one_to_one",
).merge(
    combined_metrics[[c for c in combined_metrics.columns if c != "selection_universe"]],
    on=join_keys,
    how="left",
    validate="one_to_one",
)

for metric in [
    "trades",
    "unique_trade_dates",
    "signal_date_frequency",
    "win_rate",
    "total_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "profit_factor_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2022",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
]:
    combined_col = f"combined_{metric}"

    if combined_col in comparison.columns and metric in core_ref:
        comparison[f"delta_combined_{metric}_vs_core_only"] = comparison[combined_col] - core_ref[metric]

comparison["middle_included"] = comparison["middle_is_none"].eq(False)
comparison["back_included"] = comparison["back_is_none"].eq(False)

comparison["middle_share_of_secondary_trades"] = (
    comparison["selected_middle_trade_count"] / comparison["secondary_trades"].replace(0, np.nan)
)

comparison["back_share_of_secondary_trades"] = (
    comparison["selected_back_trade_count"] / comparison["secondary_trades"].replace(0, np.nan)
)

comparison["passes_preferred_frequency"] = comparison["combined_signal_date_frequency"] >= PREFERRED_MIN_COMBINED_FREQUENCY
comparison["passes_preferred_combined_win"] = comparison["combined_win_rate"] >= PREFERRED_MIN_COMBINED_WIN_RATE
comparison["passes_preferred_secondary_win"] = comparison["secondary_win_rate"] >= PREFERRED_MIN_SECONDARY_WIN_RATE

comparison["passes_acceptable_frequency"] = comparison["combined_signal_date_frequency"] >= ACCEPTABLE_MIN_COMBINED_FREQUENCY
comparison["passes_acceptable_combined_win"] = comparison["combined_win_rate"] >= ACCEPTABLE_MIN_COMBINED_WIN_RATE
comparison["passes_acceptable_secondary_win"] = comparison["secondary_win_rate"] >= ACCEPTABLE_MIN_SECONDARY_WIN_RATE

comparison["passes_secondary_positive_avg_day"] = comparison["secondary_avg_pnl_per_day"] > MIN_SECONDARY_AVG_DAY
comparison["passes_secondary_trade_count"] = comparison["secondary_trades"] >= MIN_SECONDARY_TRADES

comparison["passes_2026_improves_vs_core"] = (
    comparison["combined_avg_pnl_per_day_2026"] > core_ref.get("avg_pnl_per_day_2026", np.nan)
)

comparison["passes_preferred_confirmation_screen"] = (
    comparison["passes_preferred_frequency"]
    & comparison["passes_preferred_combined_win"]
    & comparison["passes_preferred_secondary_win"]
    & comparison["passes_secondary_positive_avg_day"]
    & comparison["passes_secondary_trade_count"]
    & comparison["passes_2026_improves_vs_core"]
)

comparison["passes_acceptable_confirmation_screen"] = (
    comparison["passes_acceptable_frequency"]
    & comparison["passes_acceptable_combined_win"]
    & comparison["passes_acceptable_secondary_win"]
    & comparison["passes_secondary_positive_avg_day"]
    & comparison["passes_secondary_trade_count"]
    & comparison["passes_2026_improves_vs_core"]
)

comparison["confirmation_score"] = (
    comparison["passes_preferred_confirmation_screen"].astype(int) * 1_000_000
    + comparison["passes_acceptable_confirmation_screen"].astype(int) * 250_000
    + comparison["combined_signal_date_frequency"].fillna(0) * 100_000
    + comparison["combined_win_rate"].fillna(0) * 40_000
    + comparison["secondary_win_rate"].fillna(0) * 50_000
    + np.log1p(comparison["secondary_trades"].fillna(0)) * 2_000
    + np.clip(comparison["secondary_avg_pnl_per_day"].fillna(-10_000), -10_000, 10_000)
    + np.clip(comparison["combined_avg_pnl_per_day_2026"].fillna(-10_000), -10_000, 10_000)
)

comparison = comparison.sort_values(
    [
        "passes_preferred_confirmation_screen",
        "passes_acceptable_confirmation_screen",
        "combined_signal_date_frequency",
        "secondary_win_rate",
        "combined_win_rate",
        "secondary_avg_pnl_per_day",
        "combined_avg_pnl_per_day_2026",
        "confirmation_score",
    ],
    ascending=[False, False, False, False, False, False, False, False],
).reset_index(drop=True)

comparison["comparison_rank"] = np.arange(1, len(comparison) + 1)

preferred_candidates = comparison[
    comparison["passes_preferred_confirmation_screen"]
].copy()

acceptable_candidates = comparison[
    comparison["passes_acceptable_confirmation_screen"]
].copy()

if not preferred_candidates.empty:
    suggested = preferred_candidates.head(1).copy()
elif not acceptable_candidates.empty:
    suggested = acceptable_candidates.head(1).copy()
else:
    suggested = comparison.head(1).copy()

suggested_pair_mode_id = suggested.iloc[0]["secondary_confirm_pair_mode_id"] if len(suggested) else None

suggested_secondary_panel = secondary_selected[
    secondary_selected["secondary_confirm_pair_mode_id"].eq(suggested_pair_mode_id)
].copy()

suggested_combined_panel = combined_program_panel[
    combined_program_panel["secondary_confirm_pair_mode_id"].eq(suggested_pair_mode_id)
].copy()

suggested_secondary_by_bucket = secondary_by_bucket[
    secondary_by_bucket["secondary_confirm_pair_mode_id"].eq(suggested_pair_mode_id)
].copy()

suggested_secondary_by_tenor = secondary_by_tenor[
    secondary_by_tenor["secondary_confirm_pair_mode_id"].eq(suggested_pair_mode_id)
].copy()

suggested_secondary_by_year = secondary_by_year[
    secondary_by_year["secondary_confirm_pair_mode_id"].eq(suggested_pair_mode_id)
].copy()

suggested_secondary_by_bucket_year = secondary_by_bucket_year[
    secondary_by_bucket_year["secondary_confirm_pair_mode_id"].eq(suggested_pair_mode_id)
].copy()

suggested_combined_by_year = combined_program_by_year[
    combined_program_by_year["secondary_confirm_pair_mode_id"].eq(suggested_pair_mode_id)
].copy()

summary_by_structure = comparison.groupby("candidate_structure", dropna=False).agg(
    candidates=("secondary_confirm_pair_mode_id", "count"),
    preferred_passes=("passes_preferred_confirmation_screen", "sum"),
    acceptable_passes=("passes_acceptable_confirmation_screen", "sum"),
    max_combined_frequency=("combined_signal_date_frequency", "max"),
    max_secondary_trades=("secondary_trades", "max"),
    best_secondary_win_rate=("secondary_win_rate", "max"),
    best_combined_win_rate=("combined_win_rate", "max"),
    best_secondary_avg_day=("secondary_avg_pnl_per_day", "max"),
    best_combined_2026_avg_day=("combined_avg_pnl_per_day_2026", "max"),
).reset_index()

worst_trades = (
    secondary_selected
    .sort_values(
        ["secondary_confirm_pair_mode_id", "normalized_pnl_per_day", "normalized_pnl_dollars"],
        ascending=[True, True, True],
    )
    .groupby("secondary_confirm_pair_mode_id", as_index=False)
    .head(20)
    .copy()
)

# ======================================================================================
# 10. Validation
# ======================================================================================

validation_rows = []

required_base_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

required_cell15_selected_cols = [
    "trade_date",
    "tenor",
    "secondary_bucket",
    "secondary_candidate_id",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

missing_base_cols = [c for c in required_base_cols if c not in base.columns]
missing_cell15_selected_cols = [c for c in required_cell15_selected_cols if c not in cell15_selected_all.columns]

bad_front_selected = secondary_selected[
    secondary_selected["secondary_bucket"].eq("Front")
    | secondary_selected["tenor"].isin(SECONDARY_EXCLUDED_FRONT_TENORS)
]

bad_non_mb_selected = secondary_selected[
    ~secondary_selected["secondary_bucket"].isin(SECONDARY_ALLOWED_BUCKETS)
    | ~secondary_selected["tenor"].isin(SECONDARY_ALLOWED_TENORS)
]

bad_secondary_on_core_gate = secondary_selected[
    secondary_selected["trade_date"].isin(core_gate_dates_set)
]

bad_secondary_dupes = (
    secondary_selected
    .groupby(["secondary_confirm_pair_mode_id", "trade_date"])
    .size()
    .reset_index(name="rows")
)

bad_secondary_dupes = bad_secondary_dupes[bad_secondary_dupes["rows"].gt(1)]

bad_combined_dupes = (
    combined_program_panel
    .groupby(["secondary_confirm_pair_mode_id", "trade_date"])
    .size()
    .reset_index(name="rows")
)

bad_combined_dupes = bad_combined_dupes[bad_combined_dupes["rows"].gt(1)]

bad_combined_below_core_dates = comparison[
    comparison["combined_unique_trade_dates"] < int(core_summary["unique_trade_dates"].iloc[0])
]

bad_selection_modes = sorted(set(comparison["selection_mode"].dropna().unique()) - set(SELECTION_MODES))

bad_comparison_count = comparison["secondary_confirm_pair_mode_id"].nunique() != expected_pair_mode_count

bad_normalized_pnl = base[
    base["normalized_pnl_dollars"].notna()
    & base["normalized_pnl_per_day"].notna()
    & (
        (
            base["normalized_pnl_dollars"] / base["tenor"].replace(0, np.nan)
        )
        - base["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
]

add_validation(
    validation_rows,
    "base_panel_loaded",
    "PASS" if len(base) > 0 else "FAIL",
    f"rows={len(base):,}; raw_models_found={base_raw_models_found}; path={base_panel_path}",
)

add_validation(
    validation_rows,
    "locked_model_filtered",
    "PASS" if len(base) > 0 and sorted(base["model_spec"].dropna().unique().tolist()) == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_after_filter={sorted(base['model_spec'].dropna().unique().tolist()) if 'model_spec' in base.columns else []}",
)

add_validation(
    validation_rows,
    "cell15_selected_loaded",
    "PASS" if len(cell15_selected_all) > 0 else "FAIL",
    f"rows={len(cell15_selected_all):,}; path={cell15_selected_all_path}",
)

add_validation(
    validation_rows,
    "cell15_scorecards_loaded",
    "PASS" if len(cell15_scorecards_all) > 0 else "FAIL",
    f"rows={len(cell15_scorecards_all):,}; path={cell15_scorecards_all_path}",
)

add_validation(
    validation_rows,
    "cell19_suggested_loaded_optional",
    "PASS" if cell19_suggested_path else "SKIP",
    f"path={cell19_suggested_path}",
)

add_validation(
    validation_rows,
    "required_base_columns_available",
    "PASS" if not missing_base_cols else "FAIL",
    f"missing={missing_base_cols}",
)

add_validation(
    validation_rows,
    "required_cell15_selected_columns_available",
    "PASS" if not missing_cell15_selected_cols else "FAIL",
    f"missing={missing_cell15_selected_cols}",
)

add_validation(
    validation_rows,
    "core_gate_dates_created",
    "PASS" if len(core_gate_dates) > 0 and len(core_empty_dates) > 0 else "FAIL",
    f"core_gate_dates={len(core_gate_dates):,}; core_empty_dates={len(core_empty_dates):,}; total_dates={total_eligible_signal_dates:,}",
)

add_validation(
    validation_rows,
    "middle_options_count",
    "PASS" if len(middle_options) == expected_middle_options else "FAIL",
    f"middle_options={len(middle_options):,}; expected={expected_middle_options:,}",
)

add_validation(
    validation_rows,
    "back_options_count",
    "PASS" if len(back_options) == expected_back_options else "FAIL",
    f"back_options={len(back_options):,}; expected={expected_back_options:,}",
)

add_validation(
    validation_rows,
    "pair_count",
    "PASS" if len(pair_grid) == expected_pair_count else "FAIL",
    f"pair_count={len(pair_grid):,}; expected={expected_pair_count:,}",
)

add_validation(
    validation_rows,
    "pair_mode_count",
    "PASS" if len(pair_mode_grid) == expected_pair_mode_count else "FAIL",
    f"pair_mode_count={len(pair_mode_grid):,}; expected={expected_pair_mode_count:,}",
)

add_validation(
    validation_rows,
    "all_pair_modes_compared",
    "PASS" if not bad_comparison_count else "FAIL",
    f"comparison_pair_modes={comparison['secondary_confirm_pair_mode_id'].nunique():,}; expected={expected_pair_mode_count:,}",
)

add_validation(
    validation_rows,
    "selection_modes_valid",
    "PASS" if not bad_selection_modes else "FAIL",
    f"bad_modes={bad_selection_modes}; modes_found={sorted(comparison['selection_mode'].dropna().unique().tolist())}",
)

add_validation(
    validation_rows,
    "secondary_front_excluded_from_selected_trades",
    "PASS" if bad_front_selected.empty else "FAIL",
    f"bad_rows={len(bad_front_selected):,}",
)

add_validation(
    validation_rows,
    "selected_trades_middle_or_back_only",
    "PASS" if bad_non_mb_selected.empty else "FAIL",
    f"bad_rows={len(bad_non_mb_selected):,}",
)

add_validation(
    validation_rows,
    "secondary_only_on_core_empty_dates",
    "PASS" if bad_secondary_on_core_gate.empty else "FAIL",
    f"bad_rows={len(bad_secondary_on_core_gate):,}",
)

add_validation(
    validation_rows,
    "one_secondary_trade_per_pair_mode_date",
    "PASS" if bad_secondary_dupes.empty else "FAIL",
    f"bad_rows={len(bad_secondary_dupes):,}",
)

add_validation(
    validation_rows,
    "combined_program_one_trade_per_pair_mode_date",
    "PASS" if bad_combined_dupes.empty else "FAIL",
    f"bad_rows={len(bad_combined_dupes):,}",
)

add_validation(
    validation_rows,
    "combined_candidates_not_below_core_date_count",
    "PASS" if bad_combined_below_core_dates.empty else "FAIL",
    f"bad_rows={len(bad_combined_below_core_dates):,}",
)

add_validation(
    validation_rows,
    "normalized_pnl_per_day_formula",
    "PASS" if bad_normalized_pnl.empty else "FAIL",
    f"bad_rows={len(bad_normalized_pnl):,}",
)

add_validation(
    validation_rows,
    "preferred_confirmation_candidates_found",
    "PASS" if len(preferred_candidates) > 0 else "WARN",
    f"preferred_candidates={len(preferred_candidates):,}; acceptable_candidates={len(acceptable_candidates):,}",
)

add_validation(
    validation_rows,
    "core_not_changed",
    "PASS",
    "Locked Core rules loaded from Cell 11R and used only as gate/baseline.",
)

add_validation(
    validation_rows,
    "no_secondary_lock_written",
    "PASS",
    "Confirmation sweep only; no new Secondary lock artifacts created.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed or added.",
)

add_validation(
    validation_rows,
    "no_production_lock",
    "PASS",
    "No production/intraday implementation performed.",
)

validation = pd.DataFrame(validation_rows)

failures = validation[validation["status"].eq("FAIL")]

# ======================================================================================
# 11. Save outputs
# ======================================================================================

paths = {}

paths["middle_options_csv"] = OUT_AUDIT_DIR / f"20R_middle_options_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"
paths["back_options_csv"] = OUT_AUDIT_DIR / f"20R_back_options_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"

paths["pair_grid_csv"] = OUT_AUDIT_DIR / f"20R_pair_grid_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"
paths["pair_mode_grid_csv"] = OUT_AUDIT_DIR / f"20R_pair_mode_grid_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"

paths["core_program_summary_csv"] = OUT_AUDIT_DIR / f"20R_core_program_baseline_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"

paths["secondary_selected_panel_parquet"] = OUT_PROCESSED_DIR / f"20R_secondary_selected_panel_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.parquet"
paths["secondary_selected_panel_csv"] = OUT_AUDIT_DIR / f"20R_secondary_selected_panel_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"

paths["combined_program_panel_parquet"] = OUT_PROCESSED_DIR / f"20R_combined_program_panel_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.parquet"
paths["combined_program_panel_csv"] = OUT_AUDIT_DIR / f"20R_combined_program_panel_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"

paths["secondary_summary_csv"] = OUT_AUDIT_DIR / f"20R_secondary_incremental_summary_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"
paths["secondary_by_bucket_csv"] = OUT_AUDIT_DIR / f"20R_secondary_incremental_by_bucket_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"
paths["secondary_by_tenor_csv"] = OUT_AUDIT_DIR / f"20R_secondary_incremental_by_tenor_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"
paths["secondary_by_year_csv"] = OUT_AUDIT_DIR / f"20R_secondary_incremental_by_year_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"
paths["secondary_by_bucket_year_csv"] = OUT_AUDIT_DIR / f"20R_secondary_incremental_by_bucket_year_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"

paths["combined_program_summary_csv"] = OUT_AUDIT_DIR / f"20R_combined_program_summary_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"
paths["combined_program_by_year_csv"] = OUT_AUDIT_DIR / f"20R_combined_program_by_year_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"

paths["comparison_csv"] = OUT_AUDIT_DIR / f"20R_candidate_comparison_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"
paths["comparison_parquet"] = OUT_PROCESSED_DIR / f"20R_candidate_comparison_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.parquet"

paths["summary_by_structure_csv"] = OUT_AUDIT_DIR / f"20R_summary_by_structure_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"

paths["suggested_secondary_panel_csv"] = OUT_AUDIT_DIR / f"20R_suggested_secondary_panel_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"
paths["suggested_secondary_panel_parquet"] = OUT_PROCESSED_DIR / f"20R_suggested_secondary_panel_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.parquet"

paths["suggested_combined_panel_csv"] = OUT_AUDIT_DIR / f"20R_suggested_combined_panel_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"
paths["suggested_combined_panel_parquet"] = OUT_PROCESSED_DIR / f"20R_suggested_combined_panel_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.parquet"

paths["suggested_secondary_by_bucket_csv"] = OUT_AUDIT_DIR / f"20R_suggested_secondary_by_bucket_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"
paths["suggested_secondary_by_tenor_csv"] = OUT_AUDIT_DIR / f"20R_suggested_secondary_by_tenor_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"
paths["suggested_secondary_by_year_csv"] = OUT_AUDIT_DIR / f"20R_suggested_secondary_by_year_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"
paths["suggested_secondary_by_bucket_year_csv"] = OUT_AUDIT_DIR / f"20R_suggested_secondary_by_bucket_year_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"
paths["suggested_combined_by_year_csv"] = OUT_AUDIT_DIR / f"20R_suggested_combined_by_year_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"

paths["worst_trades_csv"] = OUT_AUDIT_DIR / f"20R_worst_trades_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"

paths["validation_csv"] = OUT_AUDIT_DIR / f"20R_validation_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.csv"

middle_options.to_csv(paths["middle_options_csv"], index=False)
back_options.to_csv(paths["back_options_csv"], index=False)

pair_grid.to_csv(paths["pair_grid_csv"], index=False)
pair_mode_grid.to_csv(paths["pair_mode_grid_csv"], index=False)

core_summary.to_csv(paths["core_program_summary_csv"], index=False)

secondary_selected.to_parquet(paths["secondary_selected_panel_parquet"], index=False)
secondary_selected.to_csv(paths["secondary_selected_panel_csv"], index=False)

combined_program_panel.to_parquet(paths["combined_program_panel_parquet"], index=False)
combined_program_panel.to_csv(paths["combined_program_panel_csv"], index=False)

secondary_summary.to_csv(paths["secondary_summary_csv"], index=False)
secondary_by_bucket.to_csv(paths["secondary_by_bucket_csv"], index=False)
secondary_by_tenor.to_csv(paths["secondary_by_tenor_csv"], index=False)
secondary_by_year.to_csv(paths["secondary_by_year_csv"], index=False)
secondary_by_bucket_year.to_csv(paths["secondary_by_bucket_year_csv"], index=False)

combined_program_summary.to_csv(paths["combined_program_summary_csv"], index=False)
combined_program_by_year.to_csv(paths["combined_program_by_year_csv"], index=False)

comparison.to_csv(paths["comparison_csv"], index=False)
comparison.to_parquet(paths["comparison_parquet"], index=False)

summary_by_structure.to_csv(paths["summary_by_structure_csv"], index=False)

suggested_secondary_panel.to_csv(paths["suggested_secondary_panel_csv"], index=False)
suggested_secondary_panel.to_parquet(paths["suggested_secondary_panel_parquet"], index=False)

suggested_combined_panel.to_csv(paths["suggested_combined_panel_csv"], index=False)
suggested_combined_panel.to_parquet(paths["suggested_combined_panel_parquet"], index=False)

suggested_secondary_by_bucket.to_csv(paths["suggested_secondary_by_bucket_csv"], index=False)
suggested_secondary_by_tenor.to_csv(paths["suggested_secondary_by_tenor_csv"], index=False)
suggested_secondary_by_year.to_csv(paths["suggested_secondary_by_year_csv"], index=False)
suggested_secondary_by_bucket_year.to_csv(paths["suggested_secondary_by_bucket_year_csv"], index=False)
suggested_combined_by_year.to_csv(paths["suggested_combined_by_year_csv"], index=False)

worst_trades.to_csv(paths["worst_trades_csv"], index=False)
validation.to_csv(paths["validation_csv"], index=False)

manifest = {
    "cell": "Cell 20R — Secondary Frequency Unlock Confirmation Sweep",
    "timestamp": CELL20R_TIMESTAMP,
    "version": SECONDARY_UNLOCK_CONFIRMATION_VERSION,

    "objective": "Confirm higher-frequency Secondary candidates around the Cell 19R frequency-unlock result.",
    "current_provisional_secondary_lock_candidate_id": CURRENT_PROVISIONAL_SECONDARY_LOCK_ID,
    "current_provisional_secondary_lock_decision_id": CURRENT_PROVISIONAL_SECONDARY_LOCK_DECISION_ID,

    "model_spec": LOCKED_MODEL_SPEC,
    "core_lock_version": CORE_LOCK_VERSION,
    "core_lock_decision_id": CORE_LOCK_DECISION_ID,

    "base_panel_path": str(base_panel_path),
    "locked_core_rules_long_path": str(locked_core_rules_long_path),
    "cell15_selected_all_path": str(cell15_selected_all_path),
    "cell15_scorecards_all_path": str(cell15_scorecards_all_path),
    "cell18_decision_path": str(cell18_decision_path) if cell18_decision_path else None,
    "cell19_suggested_path": str(cell19_suggested_path) if cell19_suggested_path else None,
    "cell19_practical_path": str(cell19_practical_path) if cell19_practical_path else None,

    "secondary_front_excluded": True,
    "secondary_allowed_buckets": SECONDARY_ALLOWED_BUCKETS,
    "secondary_allowed_tenors": SECONDARY_ALLOWED_TENORS,

    "middle_grid": {
        "log": MIDDLE_LOG_GRID,
        "z3m_equals_z1y": MIDDLE_Z_GRID,
        "RSI14_max": MIDDLE_RSI_CAP_GRID,
        "rv21d_vol_pct_min": MIDDLE_RV21D_FLOOR_GRID,
    },

    "back_grid": {
        "log": BACK_LOG_GRID,
        "z3m_equals_z1y": BACK_Z_GRID,
        "RSI14_max": BACK_RSI_CAP_GRID,
        "rv21d_vol_pct_min": BACK_RV21D_FLOOR_GRID,
    },

    "selection_modes": SELECTION_MODES,

    "expected_middle_options": expected_middle_options,
    "expected_back_options": expected_back_options,
    "expected_pair_count": expected_pair_count,
    "expected_pair_mode_count": expected_pair_mode_count,

    "actual_middle_options": int(len(middle_options)),
    "actual_back_options": int(len(back_options)),
    "actual_pair_count": int(len(pair_grid)),
    "actual_pair_mode_count": int(len(pair_mode_grid)),

    "total_eligible_signal_dates": int(total_eligible_signal_dates),
    "core_gate_dates": int(len(core_gate_dates)),
    "core_empty_dates": int(len(core_empty_dates)),

    "preferred_screen": {
        "combined_signal_date_frequency_min": PREFERRED_MIN_COMBINED_FREQUENCY,
        "combined_win_rate_min": PREFERRED_MIN_COMBINED_WIN_RATE,
        "secondary_win_rate_min": PREFERRED_MIN_SECONDARY_WIN_RATE,
        "secondary_avg_pnl_per_day_min": MIN_SECONDARY_AVG_DAY,
        "requires_2026_improvement_vs_core": True,
    },

    "acceptable_screen": {
        "combined_signal_date_frequency_min": ACCEPTABLE_MIN_COMBINED_FREQUENCY,
        "combined_win_rate_min": ACCEPTABLE_MIN_COMBINED_WIN_RATE,
        "secondary_win_rate_min": ACCEPTABLE_MIN_SECONDARY_WIN_RATE,
        "secondary_avg_pnl_per_day_min": MIN_SECONDARY_AVG_DAY,
        "requires_2026_improvement_vs_core": True,
    },

    "suggested_pair_mode_id": suggested_pair_mode_id,

    "constraints": [
        "No Core changes.",
        "No Secondary Front.",
        "No Secondary lock written.",
        "No sizing changes.",
        "No production/intraday implementation.",
    ],

    "paths": {k: str(v) for k, v in paths.items()},
}

paths["manifest_json"] = OUT_AUDIT_DIR / f"20R_manifest_frequency_unlock_confirmation_{CELL20R_TIMESTAMP}.json"

with open(paths["manifest_json"], "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 20R validation failed. Review validation output above.")

# ======================================================================================
# 12. Display
# ======================================================================================

display_cols = [
    "comparison_rank",
    "secondary_confirm_pair_mode_id",
    "secondary_confirm_pair_id",
    "candidate_structure",
    "selection_mode",
    "pair_description",

    "middle_option_id",
    "middle_candidate_id",
    "middle_param_model_vrp_log_min",
    "middle_param_model_vrp_z_min",
    "middle_param_RSI14_max",
    "middle_param_rv21d_vol_pct_min",
    "middle_incremental_trades",
    "middle_incremental_win_rate",
    "middle_incremental_avg_pnl_per_day",
    "middle_incremental_avg_pnl_per_day_2026",

    "back_option_id",
    "back_candidate_id",
    "back_param_model_vrp_log_min",
    "back_param_model_vrp_z_min",
    "back_param_RSI14_max",
    "back_param_rv21d_vol_pct_min",
    "back_incremental_trades",
    "back_incremental_win_rate",
    "back_incremental_avg_pnl_per_day",
    "back_incremental_avg_pnl_per_day_2026",

    "secondary_trades",
    "secondary_unique_trade_dates",
    "secondary_signal_date_frequency",
    "secondary_win_rate",
    "secondary_total_pnl",
    "secondary_avg_pnl_per_day",
    "secondary_profit_factor_pnl_per_day",
    "secondary_pnl_per_day_drawdown",
    "secondary_worst_20_trade_pnl_per_day_sum",
    "secondary_avg_pnl_per_day_2022",
    "secondary_avg_pnl_per_day_2025",
    "secondary_avg_pnl_per_day_2026",

    "selected_middle_trade_count",
    "selected_back_trade_count",
    "middle_share_of_secondary_trades",
    "back_share_of_secondary_trades",

    "combined_trades",
    "combined_unique_trade_dates",
    "combined_signal_date_frequency",
    "combined_win_rate",
    "combined_total_pnl",
    "combined_avg_pnl_per_day",
    "combined_profit_factor_pnl_per_day",
    "combined_pnl_per_day_drawdown",
    "combined_worst_20_trade_pnl_per_day_sum",
    "combined_avg_pnl_per_day_2022",
    "combined_avg_pnl_per_day_2025",
    "combined_avg_pnl_per_day_2026",

    "delta_combined_unique_trade_dates_vs_core_only",
    "delta_combined_signal_date_frequency_vs_core_only",
    "delta_combined_win_rate_vs_core_only",
    "delta_combined_total_pnl_vs_core_only",
    "delta_combined_avg_pnl_per_day_vs_core_only",
    "delta_combined_profit_factor_pnl_per_day_vs_core_only",
    "delta_combined_pnl_per_day_drawdown_vs_core_only",
    "delta_combined_worst_20_trade_pnl_per_day_sum_vs_core_only",
    "delta_combined_avg_pnl_per_day_2026_vs_core_only",

    "passes_preferred_confirmation_screen",
    "passes_acceptable_confirmation_screen",
    "passes_preferred_frequency",
    "passes_preferred_combined_win",
    "passes_preferred_secondary_win",
    "passes_acceptable_frequency",
    "passes_acceptable_combined_win",
    "passes_acceptable_secondary_win",
    "passes_secondary_positive_avg_day",
    "passes_secondary_trade_count",
    "passes_2026_improves_vs_core",
    "confirmation_score",
]

print("=" * 100)
print("Cell 20R — Secondary Frequency Unlock Confirmation Sweep")
print("=" * 100)
print(f"Base panel:                         {base_panel_path}")
print(f"Locked Core rules:                  {locked_core_rules_long_path}")
print(f"Cell 15R selected panel:            {cell15_selected_all_path}")
print(f"Cell 15R scorecards:                {cell15_scorecards_all_path}")
print(f"Cell 19R suggested file:            {cell19_suggested_path}")
print(f"Raw models found in base:           {base_raw_models_found}")
print(f"Total eligible signal dates:        {total_eligible_signal_dates:,}")
print(f"Core gate dates:                    {len(core_gate_dates):,}")
print(f"Core-empty dates:                   {len(core_empty_dates):,}")
print(f"Secondary Front excluded:           {SECONDARY_FRONT_EXCLUDED}")
print(f"Middle options including NO:        {len(middle_options):,}")
print(f"Back options including NO:          {len(back_options):,}")
print(f"Structural pairs:                   {len(pair_grid):,}")
print(f"Pair-mode candidates:               {len(pair_mode_grid):,}")
print(f"Selection modes:                    {SELECTION_MODES}")
print(f"Preferred-screen candidates:        {len(preferred_candidates):,}")
print(f"Acceptable-screen candidates:       {len(acceptable_candidates):,}")
print(f"Suggested pair-mode id:             {suggested_pair_mode_id}")
print("No Secondary lock written:          True")
print("No sizing changes:                  True")
print("No production lock:                 True")

print()
print("Validation:")
display(validation)

print()
print("Core program baseline:")
display(core_summary)

print()
print("Middle confirmation options:")
middle_display_cols = [
    "middle_option_id",
    "middle_candidate_id",
    "middle_param_model_vrp_log_min",
    "middle_param_model_vrp_z_min",
    "middle_param_RSI14_max",
    "middle_param_rv21d_vol_pct_min",
    "middle_incremental_trades",
    "middle_incremental_win_rate",
    "middle_incremental_avg_pnl_per_day",
    "middle_incremental_avg_pnl_per_day_2026",
    "middle_passes_bucket_quality_floor",
    "middle_passes_bucket_quality_ideal",
]
display(middle_options[[c for c in middle_display_cols if c in middle_options.columns]])

print()
print("Back confirmation options:")
back_display_cols = [
    "back_option_id",
    "back_candidate_id",
    "back_param_model_vrp_log_min",
    "back_param_model_vrp_z_min",
    "back_param_RSI14_max",
    "back_param_rv21d_vol_pct_min",
    "back_incremental_trades",
    "back_incremental_win_rate",
    "back_incremental_avg_pnl_per_day",
    "back_incremental_avg_pnl_per_day_2026",
    "back_passes_bucket_quality_floor",
    "back_passes_bucket_quality_ideal",
]
display(back_options[[c for c in back_display_cols if c in back_options.columns]])

print()
print("Candidate comparison — confirmation view:")
display(comparison[[c for c in display_cols if c in comparison.columns]])

print()
print("Preferred-screen candidates:")
display(preferred_candidates[[c for c in display_cols if c in preferred_candidates.columns]])

print()
print("Acceptable-screen candidates:")
display(acceptable_candidates[[c for c in display_cols if c in acceptable_candidates.columns]])

print()
print("Summary by candidate structure:")
display(summary_by_structure)

print()
print("Suggested candidate — Secondary by bucket:")
display(suggested_secondary_by_bucket)

print()
print("Suggested candidate — Secondary by tenor:")
display(suggested_secondary_by_tenor)

print()
print("Suggested candidate — Secondary by year:")
display(suggested_secondary_by_year)

print()
print("Suggested candidate — Secondary by bucket/year:")
display(suggested_secondary_by_bucket_year)

print()
print("Suggested candidate — Combined by year:")
display(suggested_combined_by_year)

print()
print("Suggested candidate — Secondary panel:")
display(suggested_secondary_panel)

print()
print("Worst trades by pair-mode candidate:")
display(worst_trades)

print()
print("Saved outputs:")
for k, p in paths.items():
    print(f"  {k}: {p}")

print("\nCELL 20R PASSED — SECONDARY FREQUENCY UNLOCK CONFIRMATION SWEEP COMPLETE")

In [ ]:
# ======================================================================================
# Cell 21R — Write higher-frequency Secondary lock artifacts
# ======================================================================================
#
# Objective:
#   Persist the higher-frequency Secondary lock confirmed in Cell 20R.
#
# This cell supersedes the prior provisional low-frequency Secondary lock:
#   secondary_middle_back_locked_v1
#   secondary_middle_back_locked_001
#
# New Secondary lock:
#   secondary_frequency_unlock_locked_v1
#   secondary_frequency_unlock_locked_001
#
# Source candidate:
#   secondary_confirm_pair_0028__BEST_SIGNAL_RANK
#
# Locked Core remains unchanged:
#   core_repaired_v1
#   core_repaired_v1_locked_001
#
# Locked Secondary:
#   Front:
#       excluded
#
#   Middle — 21D / 24D:
#       model_vrp_log > 0.60
#       model_vrp_z_3m > 0.60
#       model_vrp_z_1y > 0.60
#       RSI14 < 76
#       rv21d_vol_pct > 7.0
#
#   Back — 27D / 30D / 33D:
#       model_vrp_log > 0.65
#       model_vrp_z_3m > 0.00
#       model_vrp_z_1y > 0.00
#       RSI14 < 77
#       rv21d_vol_pct > 6.5
#
# Selection:
#   1. Core first.
#   2. If any Core tenor qualifies on a date, select Core and ignore Secondary.
#   3. If no Core tenor qualifies, evaluate Secondary Middle and Secondary Back.
#   4. If both Secondary buckets qualify, select by BEST_SIGNAL_RANK.
#   5. Secondary Front is excluded.
#
# This cell does NOT:
#   - rerun optimization
#   - change Core parameters
#   - include Secondary Front
#   - change sizing
#   - implement production/intraday logic
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 600)
pd.set_option("display.width", 1400)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL21R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

CORE_LOCK_VERSION = "core_repaired_v1"
CORE_LOCK_DECISION_ID = "core_repaired_v1_locked_001"

SUPERSEDED_SECONDARY_LOCK_VERSION = "secondary_middle_back_locked_v1"
SUPERSEDED_SECONDARY_LOCK_DECISION_ID = "secondary_middle_back_locked_001"

SECONDARY_LOCK_VERSION = "secondary_frequency_unlock_locked_v1"
SECONDARY_LOCK_DECISION_ID = "secondary_frequency_unlock_locked_001"

STACK_LOCK_VERSION = "core_secondary_frequency_unlock_signal_stack_locked_v1"
STACK_LOCK_DECISION_ID = "core_secondary_frequency_unlock_signal_stack_locked_001"

SECONDARY_SOURCE_CELL = "Cell 20R — Secondary Frequency Unlock Confirmation Sweep"
SECONDARY_SOURCE_CANDIDATE_ID = "secondary_confirm_pair_0028__BEST_SIGNAL_RANK"

SELECTION_RULE_TEXT = (
    "Core first. If any locked Core tenor qualifies on a date, select Core and ignore Secondary. "
    "If no Core tenor qualifies, evaluate Secondary Middle and Secondary Back. "
    "If both Secondary buckets qualify, select by BEST_SIGNAL_RANK. "
    "Secondary Front is excluded."
)

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)

    if matches:
        return matches[0]

    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")

    return None


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


def normalize_bool_series(s):
    if s.dtype == bool:
        return s.fillna(False)

    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y"])
    )


def as_float(value):
    try:
        return float(value)
    except Exception:
        return np.nan


def build_rule_row(
    signal_layer,
    signal_priority,
    lock_version,
    lock_decision_id,
    tenor,
    bucket,
    bucket_order,
    include_tenor,
    model_vrp_log_min,
    model_vrp_z_3m_min,
    model_vrp_z_1y_min,
    RSI14_max,
    rv21d_vol_pct_min,
    source_cell,
    source_candidate_id=None,
    supersedes_lock_version=None,
    supersedes_lock_decision_id=None,
    exclusion_reason=None,
):
    return {
        "stack_lock_version": STACK_LOCK_VERSION,
        "stack_lock_decision_id": STACK_LOCK_DECISION_ID,

        "model_spec": LOCKED_MODEL_SPEC,

        "signal_layer": signal_layer,
        "signal_priority": int(signal_priority),

        "lock_version": lock_version,
        "lock_decision_id": lock_decision_id,

        "supersedes_lock_version": supersedes_lock_version,
        "supersedes_lock_decision_id": supersedes_lock_decision_id,

        "tenor": int(tenor),
        "tenor_bucket": bucket,
        "tenor_bucket_order": int(bucket_order),

        "include_tenor": bool(include_tenor),

        "model_vrp_log_min": model_vrp_log_min,
        "model_vrp_z_3m_min": model_vrp_z_3m_min,
        "model_vrp_z_1y_min": model_vrp_z_1y_min,
        "RSI14_max": RSI14_max,
        "rv21d_vol_pct_min": rv21d_vol_pct_min,

        "comparison_operator_model_vrp_log": ">" if include_tenor else None,
        "comparison_operator_model_vrp_z_3m": ">" if include_tenor else None,
        "comparison_operator_model_vrp_z_1y": ">" if include_tenor else None,
        "comparison_operator_RSI14": "<" if include_tenor else None,
        "comparison_operator_rv21d_vol_pct": ">" if include_tenor else None,

        "vrp_measure": "model_vrp_log = log(implied_variance / forecast_variance)",
        "forecast_model": "har_total_simple / unified_fds_no_min_return",
        "rv_floor_contract": "rv21d_vol_pct > threshold",

        "selection_rule": SELECTION_RULE_TEXT,

        "source_cell": source_cell,
        "source_candidate_id": source_candidate_id,
        "exclusion_reason": exclusion_reason,

        "created_timestamp": CELL21R_TIMESTAMP,
    }


def make_wide_rules(rules_long):
    active = rules_long[rules_long["include_tenor"].eq(True)].copy()

    wide = (
        active
        .groupby(
            [
                "signal_layer",
                "signal_priority",
                "lock_version",
                "lock_decision_id",
                "supersedes_lock_version",
                "supersedes_lock_decision_id",
                "tenor_bucket",
                "tenor_bucket_order",
                "model_spec",
                "model_vrp_log_min",
                "model_vrp_z_3m_min",
                "model_vrp_z_1y_min",
                "RSI14_max",
                "rv21d_vol_pct_min",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            tenors=("tenor", lambda x: ",".join(str(int(v)) for v in sorted(x))),
            tenor_count=("tenor", "count"),
            selection_rule=("selection_rule", "first"),
            source_cell=("source_cell", "first"),
            source_candidate_id=("source_candidate_id", "first"),
        )
        .sort_values(["signal_priority", "tenor_bucket_order"])
        .reset_index(drop=True)
    )

    return wide


def expected_rule_tuple_df(rules_long):
    active = rules_long[rules_long["include_tenor"].eq(True)].copy()

    for c in [
        "model_vrp_log_min",
        "model_vrp_z_3m_min",
        "model_vrp_z_1y_min",
        "RSI14_max",
        "rv21d_vol_pct_min",
    ]:
        active[c] = pd.to_numeric(active[c], errors="coerce").round(10)

    return set(
        active[
            [
                "signal_layer",
                "tenor",
                "tenor_bucket",
                "model_vrp_log_min",
                "model_vrp_z_3m_min",
                "model_vrp_z_1y_min",
                "RSI14_max",
                "rv21d_vol_pct_min",
            ]
        ].itertuples(index=False, name=None)
    )


# ======================================================================================
# 2. Load source artifacts
# ======================================================================================

core_rules_long_source_path = latest_file(
    OUT_PROCESSED_DIR,
    "11R_core_repaired_v1_locked_rules_long_*.parquet",
    required=True,
)

cell20_comparison_path = latest_file(
    OUT_AUDIT_DIR,
    "20R_candidate_comparison_frequency_unlock_confirmation_*.csv",
    required=True,
)

cell20_validation_path = latest_file(
    OUT_AUDIT_DIR,
    "20R_validation_frequency_unlock_confirmation_*.csv",
    required=True,
)

cell20_manifest_path = latest_file(
    OUT_AUDIT_DIR,
    "20R_manifest_frequency_unlock_confirmation_*.json",
    required=False,
)

cell20_suggested_secondary_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "20R_suggested_secondary_panel_frequency_unlock_confirmation_*.parquet",
    required=False,
)

cell20_suggested_combined_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "20R_suggested_combined_panel_frequency_unlock_confirmation_*.parquet",
    required=False,
)

prior_secondary_lock_path = latest_file(
    OUT_PROCESSED_DIR,
    "18R_secondary_locked_rules_long_*.parquet",
    required=False,
)

prior_stack_lock_path = latest_file(
    OUT_PROCESSED_DIR,
    "18R_core_secondary_stack_locked_rules_long_*.parquet",
    required=False,
)

core_source = pd.read_parquet(core_rules_long_source_path)
cell20_comparison = pd.read_csv(cell20_comparison_path)
cell20_validation = pd.read_csv(cell20_validation_path)

prior_secondary_lock = pd.read_parquet(prior_secondary_lock_path) if prior_secondary_lock_path else pd.DataFrame()
prior_stack_lock = pd.read_parquet(prior_stack_lock_path) if prior_stack_lock_path else pd.DataFrame()

# ======================================================================================
# 3. Build locked Core rules
# ======================================================================================

core_rows = []

# Core Front — 12D / 15D / 18D
for tenor in [12, 15, 18]:
    core_rows.append(
        build_rule_row(
            signal_layer="Core",
            signal_priority=1,
            lock_version=CORE_LOCK_VERSION,
            lock_decision_id=CORE_LOCK_DECISION_ID,
            tenor=tenor,
            bucket="Front",
            bucket_order=1,
            include_tenor=True,
            model_vrp_log_min=0.60,
            model_vrp_z_3m_min=0.65,
            model_vrp_z_1y_min=0.65,
            RSI14_max=70.0,
            rv21d_vol_pct_min=8.5,
            source_cell="Cell 11R — Core repaired v1 lock",
            source_candidate_id=CORE_LOCK_DECISION_ID,
        )
    )

# Core Middle — 21D / 24D
for tenor in [21, 24]:
    core_rows.append(
        build_rule_row(
            signal_layer="Core",
            signal_priority=1,
            lock_version=CORE_LOCK_VERSION,
            lock_decision_id=CORE_LOCK_DECISION_ID,
            tenor=tenor,
            bucket="Middle",
            bucket_order=2,
            include_tenor=True,
            model_vrp_log_min=0.65,
            model_vrp_z_3m_min=0.70,
            model_vrp_z_1y_min=0.70,
            RSI14_max=70.0,
            rv21d_vol_pct_min=8.5,
            source_cell="Cell 11R — Core repaired v1 lock",
            source_candidate_id=CORE_LOCK_DECISION_ID,
        )
    )

# Core Back — 27D / 30D / 33D
for tenor in [27, 30, 33]:
    core_rows.append(
        build_rule_row(
            signal_layer="Core",
            signal_priority=1,
            lock_version=CORE_LOCK_VERSION,
            lock_decision_id=CORE_LOCK_DECISION_ID,
            tenor=tenor,
            bucket="Back",
            bucket_order=3,
            include_tenor=True,
            model_vrp_log_min=0.70,
            model_vrp_z_3m_min=0.70,
            model_vrp_z_1y_min=0.70,
            RSI14_max=70.0,
            rv21d_vol_pct_min=8.5,
            source_cell="Cell 11R — Core repaired v1 lock",
            source_candidate_id=CORE_LOCK_DECISION_ID,
        )
    )

core_locked_rules_long = pd.DataFrame(core_rows)

# ======================================================================================
# 4. Build higher-frequency locked Secondary rules
# ======================================================================================

secondary_rows = []

# Secondary Front — explicitly excluded.
for tenor in [12, 15, 18]:
    secondary_rows.append(
        build_rule_row(
            signal_layer="Secondary",
            signal_priority=2,
            lock_version=SECONDARY_LOCK_VERSION,
            lock_decision_id=SECONDARY_LOCK_DECISION_ID,
            supersedes_lock_version=SUPERSEDED_SECONDARY_LOCK_VERSION,
            supersedes_lock_decision_id=SUPERSEDED_SECONDARY_LOCK_DECISION_ID,
            tenor=tenor,
            bucket="Front",
            bucket_order=1,
            include_tenor=False,
            model_vrp_log_min=np.nan,
            model_vrp_z_3m_min=np.nan,
            model_vrp_z_1y_min=np.nan,
            RSI14_max=np.nan,
            rv21d_vol_pct_min=np.nan,
            source_cell=SECONDARY_SOURCE_CELL,
            source_candidate_id=SECONDARY_SOURCE_CANDIDATE_ID,
            exclusion_reason=(
                "Secondary Front remains excluded. Frequency unlock uses only Secondary Middle and Secondary Back."
            ),
        )
    )

# Secondary Middle — 21D / 24D.
for tenor in [21, 24]:
    secondary_rows.append(
        build_rule_row(
            signal_layer="Secondary",
            signal_priority=2,
            lock_version=SECONDARY_LOCK_VERSION,
            lock_decision_id=SECONDARY_LOCK_DECISION_ID,
            supersedes_lock_version=SUPERSEDED_SECONDARY_LOCK_VERSION,
            supersedes_lock_decision_id=SUPERSEDED_SECONDARY_LOCK_DECISION_ID,
            tenor=tenor,
            bucket="Middle",
            bucket_order=2,
            include_tenor=True,
            model_vrp_log_min=0.60,
            model_vrp_z_3m_min=0.60,
            model_vrp_z_1y_min=0.60,
            RSI14_max=76.0,
            rv21d_vol_pct_min=7.0,
            source_cell=SECONDARY_SOURCE_CELL,
            source_candidate_id=SECONDARY_SOURCE_CANDIDATE_ID,
        )
    )

# Secondary Back — 27D / 30D / 33D.
for tenor in [27, 30, 33]:
    secondary_rows.append(
        build_rule_row(
            signal_layer="Secondary",
            signal_priority=2,
            lock_version=SECONDARY_LOCK_VERSION,
            lock_decision_id=SECONDARY_LOCK_DECISION_ID,
            supersedes_lock_version=SUPERSEDED_SECONDARY_LOCK_VERSION,
            supersedes_lock_decision_id=SUPERSEDED_SECONDARY_LOCK_DECISION_ID,
            tenor=tenor,
            bucket="Back",
            bucket_order=3,
            include_tenor=True,
            model_vrp_log_min=0.65,
            model_vrp_z_3m_min=0.00,
            model_vrp_z_1y_min=0.00,
            RSI14_max=77.0,
            rv21d_vol_pct_min=6.5,
            source_cell=SECONDARY_SOURCE_CELL,
            source_candidate_id=SECONDARY_SOURCE_CANDIDATE_ID,
        )
    )

secondary_locked_rules_long = pd.DataFrame(secondary_rows)

# ======================================================================================
# 5. Combined stack rules and wide views
# ======================================================================================

stack_locked_rules_long = pd.concat(
    [
        core_locked_rules_long,
        secondary_locked_rules_long,
    ],
    ignore_index=True,
)

for c in [
    "tenor",
    "tenor_bucket_order",
    "signal_priority",
    "model_vrp_log_min",
    "model_vrp_z_3m_min",
    "model_vrp_z_1y_min",
    "RSI14_max",
    "rv21d_vol_pct_min",
]:
    stack_locked_rules_long[c] = pd.to_numeric(stack_locked_rules_long[c], errors="coerce")

stack_locked_rules_long["include_tenor"] = stack_locked_rules_long["include_tenor"].astype(bool)

core_locked_rules_long = stack_locked_rules_long[
    stack_locked_rules_long["signal_layer"].eq("Core")
].copy()

secondary_locked_rules_long = stack_locked_rules_long[
    stack_locked_rules_long["signal_layer"].eq("Secondary")
].copy()

core_locked_rules_wide = make_wide_rules(core_locked_rules_long)
secondary_locked_rules_wide = make_wide_rules(secondary_locked_rules_long)
stack_locked_rules_wide = make_wide_rules(stack_locked_rules_long)

secondary_exclusions = secondary_locked_rules_long[
    secondary_locked_rules_long["include_tenor"].eq(False)
].copy()

# ======================================================================================
# 6. Decision summary from Cell 20R source candidate
# ======================================================================================

decision_rows = []

selected20 = cell20_comparison[
    cell20_comparison["secondary_confirm_pair_mode_id"].eq(SECONDARY_SOURCE_CANDIDATE_ID)
].copy()

if not selected20.empty:
    selected20_row = selected20.iloc[0].to_dict()

    decision_rows.append({
        "decision_item": "locked_secondary_source_candidate",
        "value": SECONDARY_SOURCE_CANDIDATE_ID,
    })

    for metric in [
        "comparison_rank",
        "secondary_confirm_pair_id",
        "secondary_confirm_pair_mode_id",
        "candidate_structure",
        "selection_mode",
        "pair_description",

        "middle_option_id",
        "middle_candidate_id",
        "middle_param_model_vrp_log_min",
        "middle_param_model_vrp_z_min",
        "middle_param_RSI14_max",
        "middle_param_rv21d_vol_pct_min",
        "middle_incremental_trades",
        "middle_incremental_win_rate",
        "middle_incremental_avg_pnl_per_day",
        "middle_incremental_avg_pnl_per_day_2026",

        "back_option_id",
        "back_candidate_id",
        "back_param_model_vrp_log_min",
        "back_param_model_vrp_z_min",
        "back_param_RSI14_max",
        "back_param_rv21d_vol_pct_min",
        "back_incremental_trades",
        "back_incremental_win_rate",
        "back_incremental_avg_pnl_per_day",
        "back_incremental_avg_pnl_per_day_2026",

        "secondary_trades",
        "secondary_unique_trade_dates",
        "secondary_signal_date_frequency",
        "secondary_win_rate",
        "secondary_total_pnl",
        "secondary_avg_pnl_per_day",
        "secondary_profit_factor_pnl_per_day",
        "secondary_pnl_per_day_drawdown",
        "secondary_worst_20_trade_pnl_per_day_sum",
        "secondary_avg_pnl_per_day_2022",
        "secondary_avg_pnl_per_day_2025",
        "secondary_avg_pnl_per_day_2026",

        "selected_middle_trade_count",
        "selected_back_trade_count",
        "middle_share_of_secondary_trades",
        "back_share_of_secondary_trades",

        "combined_trades",
        "combined_unique_trade_dates",
        "combined_signal_date_frequency",
        "combined_win_rate",
        "combined_total_pnl",
        "combined_avg_pnl_per_day",
        "combined_profit_factor_pnl_per_day",
        "combined_pnl_per_day_drawdown",
        "combined_worst_20_trade_pnl_per_day_sum",
        "combined_avg_pnl_per_day_2022",
        "combined_avg_pnl_per_day_2025",
        "combined_avg_pnl_per_day_2026",

        "delta_combined_unique_trade_dates_vs_core_only",
        "delta_combined_signal_date_frequency_vs_core_only",
        "delta_combined_win_rate_vs_core_only",
        "delta_combined_total_pnl_vs_core_only",
        "delta_combined_avg_pnl_per_day_vs_core_only",
        "delta_combined_profit_factor_pnl_per_day_vs_core_only",
        "delta_combined_pnl_per_day_drawdown_vs_core_only",
        "delta_combined_worst_20_trade_pnl_per_day_sum_vs_core_only",
        "delta_combined_avg_pnl_per_day_2026_vs_core_only",

        "passes_preferred_confirmation_screen",
        "passes_acceptable_confirmation_screen",
        "passes_preferred_frequency",
        "passes_preferred_combined_win",
        "passes_preferred_secondary_win",
        "passes_acceptable_frequency",
        "passes_acceptable_combined_win",
        "passes_acceptable_secondary_win",
        "passes_secondary_positive_avg_day",
        "passes_secondary_trade_count",
        "passes_2026_improves_vs_core",
    ]:
        if metric in selected20_row:
            decision_rows.append({
                "decision_item": metric,
                "value": selected20_row[metric],
            })

decision_summary = pd.DataFrame(decision_rows)

# ======================================================================================
# 7. Validation
# ======================================================================================

validation_rows = []

expected_core_active = {
    ("Core", 12, "Front", 0.60, 0.65, 0.65, 70.0, 8.5),
    ("Core", 15, "Front", 0.60, 0.65, 0.65, 70.0, 8.5),
    ("Core", 18, "Front", 0.60, 0.65, 0.65, 70.0, 8.5),
    ("Core", 21, "Middle", 0.65, 0.70, 0.70, 70.0, 8.5),
    ("Core", 24, "Middle", 0.65, 0.70, 0.70, 70.0, 8.5),
    ("Core", 27, "Back", 0.70, 0.70, 0.70, 70.0, 8.5),
    ("Core", 30, "Back", 0.70, 0.70, 0.70, 70.0, 8.5),
    ("Core", 33, "Back", 0.70, 0.70, 0.70, 70.0, 8.5),
}

expected_secondary_active = {
    ("Secondary", 21, "Middle", 0.60, 0.60, 0.60, 76.0, 7.0),
    ("Secondary", 24, "Middle", 0.60, 0.60, 0.60, 76.0, 7.0),
    ("Secondary", 27, "Back", 0.65, 0.00, 0.00, 77.0, 6.5),
    ("Secondary", 30, "Back", 0.65, 0.00, 0.00, 77.0, 6.5),
    ("Secondary", 33, "Back", 0.65, 0.00, 0.00, 77.0, 6.5),
}

expected_all_active = expected_core_active | expected_secondary_active
active_rule_tuples = expected_rule_tuple_df(stack_locked_rules_long)

missing_active = sorted(expected_all_active - active_rule_tuples)
unexpected_active = sorted(active_rule_tuples - expected_all_active)

secondary_front_rows = secondary_locked_rules_long[
    secondary_locked_rules_long["tenor_bucket"].eq("Front")
].copy()

bad_secondary_front_included = secondary_front_rows[
    secondary_front_rows["include_tenor"].eq(True)
]

bad_core_row_count = len(core_locked_rules_long) != 8
bad_secondary_row_count = len(secondary_locked_rules_long) != 8
bad_stack_row_count = len(stack_locked_rules_long) != 16

bad_duplicate_rows = (
    stack_locked_rules_long
    .groupby(["signal_layer", "tenor"])
    .size()
    .reset_index(name="rows")
)

bad_duplicate_rows = bad_duplicate_rows[bad_duplicate_rows["rows"].gt(1)]

bad_selection_rule = not all(
    stack_locked_rules_long["selection_rule"].eq(SELECTION_RULE_TEXT)
)

# Cross-check latest Core source.
core_source_check_status = "FAIL"
core_source_check_detail = "Core source file did not pass cross-check."

source = core_source.copy()

source_rename = {
    "model_vrp_log_min": "source_model_vrp_log_min",
    "model_vrp_z_3m_min": "source_model_vrp_z_3m_min",
    "model_vrp_z_1y_min": "source_model_vrp_z_1y_min",
    "RSI14_max": "source_RSI14_max",
    "rv21d_vol_pct_min": "source_rv21d_vol_pct_min",
    "effective_tenor_bucket": "source_effective_tenor_bucket",
}

source = source.rename(columns={k: v for k, v in source_rename.items() if k in source.columns})

needed_source_cols = [
    "tenor",
    "source_effective_tenor_bucket",
    "source_model_vrp_log_min",
    "source_model_vrp_z_3m_min",
    "source_model_vrp_z_1y_min",
    "source_RSI14_max",
    "source_rv21d_vol_pct_min",
]

if all(c in source.columns for c in needed_source_cols):
    core_check = core_locked_rules_long.merge(
        source[needed_source_cols],
        on="tenor",
        how="left",
        validate="one_to_one",
    )

    core_mismatches = core_check[
        core_check["source_effective_tenor_bucket"].ne(core_check["tenor_bucket"])
        | ~np.isclose(core_check["source_model_vrp_log_min"], core_check["model_vrp_log_min"])
        | ~np.isclose(core_check["source_model_vrp_z_3m_min"], core_check["model_vrp_z_3m_min"])
        | ~np.isclose(core_check["source_model_vrp_z_1y_min"], core_check["model_vrp_z_1y_min"])
        | ~np.isclose(core_check["source_RSI14_max"], core_check["RSI14_max"])
        | ~np.isclose(core_check["source_rv21d_vol_pct_min"], core_check["rv21d_vol_pct_min"])
    ]

    core_source_check_status = "PASS" if core_mismatches.empty else "FAIL"
    core_source_check_detail = (
        f"source_path={core_rules_long_source_path}; mismatches={len(core_mismatches):,}"
    )
else:
    core_source_check_status = "FAIL"
    core_source_check_detail = (
        f"source_path={core_rules_long_source_path}; missing required columns for cross-check"
    )

# Cross-check Cell 20R source candidate.
cell20_candidate_status = "FAIL"
cell20_candidate_detail = f"Candidate {SECONDARY_SOURCE_CANDIDATE_ID} not found."

if not selected20.empty:
    selected20_row = selected20.iloc[0]

    checks = {
        "middle_log_060": np.isclose(as_float(selected20_row.get("middle_param_model_vrp_log_min", np.nan)), 0.60),
        "middle_z_060": np.isclose(as_float(selected20_row.get("middle_param_model_vrp_z_min", np.nan)), 0.60),
        "middle_rsi_76": np.isclose(as_float(selected20_row.get("middle_param_RSI14_max", np.nan)), 76.0),
        "middle_rv_70": np.isclose(as_float(selected20_row.get("middle_param_rv21d_vol_pct_min", np.nan)), 7.0),

        "back_log_065": np.isclose(as_float(selected20_row.get("back_param_model_vrp_log_min", np.nan)), 0.65),
        "back_z_000": np.isclose(as_float(selected20_row.get("back_param_model_vrp_z_min", np.nan)), 0.00),
        "back_rsi_77": np.isclose(as_float(selected20_row.get("back_param_RSI14_max", np.nan)), 77.0),
        "back_rv_65": np.isclose(as_float(selected20_row.get("back_param_rv21d_vol_pct_min", np.nan)), 6.5),

        "selection_best_signal_rank": str(selected20_row.get("selection_mode", "")) == "BEST_SIGNAL_RANK",
        "preferred_screen_passed": str(selected20_row.get("passes_preferred_confirmation_screen", "")).strip().lower() in ["true", "1", "yes"],
        "acceptable_screen_passed": str(selected20_row.get("passes_acceptable_confirmation_screen", "")).strip().lower() in ["true", "1", "yes"],
    }

    cell20_candidate_status = "PASS" if all(checks.values()) else "FAIL"
    cell20_candidate_detail = (
        f"source_path={cell20_comparison_path}; "
        f"candidate={SECONDARY_SOURCE_CANDIDATE_ID}; "
        f"checks={checks}"
    )

# Cross-check Cell 20R validation.
cell20_validation_status = "FAIL"
cell20_validation_detail = "Cell 20R validation file did not pass."

if {"check", "status"}.issubset(cell20_validation.columns):
    failures20 = cell20_validation[cell20_validation["status"].eq("FAIL")]
    cell20_validation_status = "PASS" if failures20.empty else "FAIL"
    cell20_validation_detail = (
        f"source_path={cell20_validation_path}; failures={len(failures20):,}"
    )

# Cross-check prior Secondary lock exists.
prior_lock_status = "PASS" if prior_secondary_lock_path else "WARN"
prior_lock_detail = (
    f"prior_secondary_lock_path={prior_secondary_lock_path}; "
    f"prior_stack_lock_path={prior_stack_lock_path}; "
    f"superseded_decision_id={SUPERSEDED_SECONDARY_LOCK_DECISION_ID}"
)

add_validation(
    validation_rows,
    "core_rules_row_count",
    "PASS" if not bad_core_row_count else "FAIL",
    f"rows={len(core_locked_rules_long):,}; expected=8",
)

add_validation(
    validation_rows,
    "secondary_rules_row_count",
    "PASS" if not bad_secondary_row_count else "FAIL",
    f"rows={len(secondary_locked_rules_long):,}; expected=8 including excluded Front rows",
)

add_validation(
    validation_rows,
    "stack_rules_row_count",
    "PASS" if not bad_stack_row_count else "FAIL",
    f"rows={len(stack_locked_rules_long):,}; expected=16",
)

add_validation(
    validation_rows,
    "expected_active_rules_match",
    "PASS" if not missing_active and not unexpected_active else "FAIL",
    f"missing_active={missing_active}; unexpected_active={unexpected_active}",
)

add_validation(
    validation_rows,
    "secondary_front_excluded",
    "PASS" if bad_secondary_front_included.empty and len(secondary_front_rows) == 3 else "FAIL",
    f"secondary_front_rows={len(secondary_front_rows):,}; included_front_rows={len(bad_secondary_front_included):,}",
)

add_validation(
    validation_rows,
    "no_duplicate_layer_tenor_rows",
    "PASS" if bad_duplicate_rows.empty else "FAIL",
    f"bad_rows={len(bad_duplicate_rows):,}",
)

add_validation(
    validation_rows,
    "selection_rule_populated",
    "PASS" if not bad_selection_rule else "FAIL",
    f"unique_selection_rules={stack_locked_rules_long['selection_rule'].nunique()}",
)

add_validation(
    validation_rows,
    "core_source_11R_cross_check",
    core_source_check_status,
    core_source_check_detail,
)

add_validation(
    validation_rows,
    "cell20_source_candidate_cross_check",
    cell20_candidate_status,
    cell20_candidate_detail,
)

add_validation(
    validation_rows,
    "cell20_validation_passed",
    cell20_validation_status,
    cell20_validation_detail,
)

add_validation(
    validation_rows,
    "prior_secondary_lock_superseded_reference",
    prior_lock_status,
    prior_lock_detail,
)

add_validation(
    validation_rows,
    "no_core_changes",
    "PASS",
    "Core rules remain core_repaired_v1_locked_001.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "This cell writes signal parameter artifacts only. No sizing fields are created or changed.",
)

add_validation(
    validation_rows,
    "no_production_or_intraday_implementation",
    "PASS",
    "This cell writes research/audit lock artifacts only. No production or intraday logic is implemented.",
)

validation = pd.DataFrame(validation_rows)
failures = validation[validation["status"].eq("FAIL")]

# ======================================================================================
# 8. Save artifacts
# ======================================================================================

paths = {}

paths["core_locked_rules_long_csv"] = OUT_AUDIT_DIR / f"21R_core_locked_rules_long_{CELL21R_TIMESTAMP}.csv"
paths["core_locked_rules_long_parquet"] = OUT_PROCESSED_DIR / f"21R_core_locked_rules_long_{CELL21R_TIMESTAMP}.parquet"
paths["core_locked_rules_wide_csv"] = OUT_AUDIT_DIR / f"21R_core_locked_rules_wide_{CELL21R_TIMESTAMP}.csv"
paths["core_locked_rules_wide_parquet"] = OUT_PROCESSED_DIR / f"21R_core_locked_rules_wide_{CELL21R_TIMESTAMP}.parquet"

paths["secondary_locked_rules_long_csv"] = OUT_AUDIT_DIR / f"21R_secondary_frequency_unlock_locked_rules_long_{CELL21R_TIMESTAMP}.csv"
paths["secondary_locked_rules_long_parquet"] = OUT_PROCESSED_DIR / f"21R_secondary_frequency_unlock_locked_rules_long_{CELL21R_TIMESTAMP}.parquet"
paths["secondary_locked_rules_wide_csv"] = OUT_AUDIT_DIR / f"21R_secondary_frequency_unlock_locked_rules_wide_{CELL21R_TIMESTAMP}.csv"
paths["secondary_locked_rules_wide_parquet"] = OUT_PROCESSED_DIR / f"21R_secondary_frequency_unlock_locked_rules_wide_{CELL21R_TIMESTAMP}.parquet"

paths["stack_locked_rules_long_csv"] = OUT_AUDIT_DIR / f"21R_core_secondary_frequency_unlock_locked_rules_long_{CELL21R_TIMESTAMP}.csv"
paths["stack_locked_rules_long_parquet"] = OUT_PROCESSED_DIR / f"21R_core_secondary_frequency_unlock_locked_rules_long_{CELL21R_TIMESTAMP}.parquet"
paths["stack_locked_rules_wide_csv"] = OUT_AUDIT_DIR / f"21R_core_secondary_frequency_unlock_locked_rules_wide_{CELL21R_TIMESTAMP}.csv"
paths["stack_locked_rules_wide_parquet"] = OUT_PROCESSED_DIR / f"21R_core_secondary_frequency_unlock_locked_rules_wide_{CELL21R_TIMESTAMP}.parquet"

paths["secondary_exclusions_csv"] = OUT_AUDIT_DIR / f"21R_secondary_frequency_unlock_exclusions_{CELL21R_TIMESTAMP}.csv"

paths["decision_summary_csv"] = OUT_AUDIT_DIR / f"21R_secondary_frequency_unlock_lock_decision_summary_{CELL21R_TIMESTAMP}.csv"

paths["validation_csv"] = OUT_AUDIT_DIR / f"21R_secondary_frequency_unlock_lock_validation_{CELL21R_TIMESTAMP}.csv"

core_locked_rules_long.to_csv(paths["core_locked_rules_long_csv"], index=False)
core_locked_rules_long.to_parquet(paths["core_locked_rules_long_parquet"], index=False)

core_locked_rules_wide.to_csv(paths["core_locked_rules_wide_csv"], index=False)
core_locked_rules_wide.to_parquet(paths["core_locked_rules_wide_parquet"], index=False)

secondary_locked_rules_long.to_csv(paths["secondary_locked_rules_long_csv"], index=False)
secondary_locked_rules_long.to_parquet(paths["secondary_locked_rules_long_parquet"], index=False)

secondary_locked_rules_wide.to_csv(paths["secondary_locked_rules_wide_csv"], index=False)
secondary_locked_rules_wide.to_parquet(paths["secondary_locked_rules_wide_parquet"], index=False)

stack_locked_rules_long.to_csv(paths["stack_locked_rules_long_csv"], index=False)
stack_locked_rules_long.to_parquet(paths["stack_locked_rules_long_parquet"], index=False)

stack_locked_rules_wide.to_csv(paths["stack_locked_rules_wide_csv"], index=False)
stack_locked_rules_wide.to_parquet(paths["stack_locked_rules_wide_parquet"], index=False)

secondary_exclusions.to_csv(paths["secondary_exclusions_csv"], index=False)

decision_summary.to_csv(paths["decision_summary_csv"], index=False)

validation.to_csv(paths["validation_csv"], index=False)

manifest = {
    "cell": "Cell 21R — Write higher-frequency Secondary lock artifacts",
    "timestamp": CELL21R_TIMESTAMP,

    "model_spec": LOCKED_MODEL_SPEC,

    "stack_lock_version": STACK_LOCK_VERSION,
    "stack_lock_decision_id": STACK_LOCK_DECISION_ID,

    "core_lock_version": CORE_LOCK_VERSION,
    "core_lock_decision_id": CORE_LOCK_DECISION_ID,

    "secondary_lock_version": SECONDARY_LOCK_VERSION,
    "secondary_lock_decision_id": SECONDARY_LOCK_DECISION_ID,

    "supersedes_secondary_lock_version": SUPERSEDED_SECONDARY_LOCK_VERSION,
    "supersedes_secondary_lock_decision_id": SUPERSEDED_SECONDARY_LOCK_DECISION_ID,

    "secondary_source_cell": SECONDARY_SOURCE_CELL,
    "secondary_source_candidate_id": SECONDARY_SOURCE_CANDIDATE_ID,

    "core_source_rules_path": str(core_rules_long_source_path),
    "cell20_comparison_path": str(cell20_comparison_path),
    "cell20_validation_path": str(cell20_validation_path),
    "cell20_manifest_path": str(cell20_manifest_path) if cell20_manifest_path else None,
    "cell20_suggested_secondary_panel_path": str(cell20_suggested_secondary_panel_path) if cell20_suggested_secondary_panel_path else None,
    "cell20_suggested_combined_panel_path": str(cell20_suggested_combined_panel_path) if cell20_suggested_combined_panel_path else None,
    "prior_secondary_lock_path": str(prior_secondary_lock_path) if prior_secondary_lock_path else None,
    "prior_stack_lock_path": str(prior_stack_lock_path) if prior_stack_lock_path else None,

    "locked_core": {
        "Front": {
            "tenors": [12, 15, 18],
            "model_vrp_log_min": 0.60,
            "model_vrp_z_3m_min": 0.65,
            "model_vrp_z_1y_min": 0.65,
            "RSI14_max": 70.0,
            "rv21d_vol_pct_min": 8.5,
        },
        "Middle": {
            "tenors": [21, 24],
            "model_vrp_log_min": 0.65,
            "model_vrp_z_3m_min": 0.70,
            "model_vrp_z_1y_min": 0.70,
            "RSI14_max": 70.0,
            "rv21d_vol_pct_min": 8.5,
        },
        "Back": {
            "tenors": [27, 30, 33],
            "model_vrp_log_min": 0.70,
            "model_vrp_z_3m_min": 0.70,
            "model_vrp_z_1y_min": 0.70,
            "RSI14_max": 70.0,
            "rv21d_vol_pct_min": 8.5,
        },
    },

    "locked_secondary": {
        "Front": {
            "tenors": [12, 15, 18],
            "included": False,
            "exclusion_reason": "Secondary Front remains excluded.",
        },
        "Middle": {
            "tenors": [21, 24],
            "included": True,
            "model_vrp_log_min": 0.60,
            "model_vrp_z_3m_min": 0.60,
            "model_vrp_z_1y_min": 0.60,
            "RSI14_max": 76.0,
            "rv21d_vol_pct_min": 7.0,
        },
        "Back": {
            "tenors": [27, 30, 33],
            "included": True,
            "model_vrp_log_min": 0.65,
            "model_vrp_z_3m_min": 0.00,
            "model_vrp_z_1y_min": 0.00,
            "RSI14_max": 77.0,
            "rv21d_vol_pct_min": 6.5,
        },
    },

    "selection_rule": SELECTION_RULE_TEXT,

    "cell20_source_candidate_metrics": (
        selected20.iloc[0].to_dict()
        if not selected20.empty
        else {}
    ),

    "constraints": [
        "No Core parameter changes.",
        "No Secondary Front.",
        "No sizing changes.",
        "No production/intraday implementation.",
        "This cell writes lock artifacts only.",
    ],

    "paths": {k: str(v) for k, v in paths.items()},
}

paths["manifest_json"] = OUT_AUDIT_DIR / f"21R_secondary_frequency_unlock_lock_manifest_{CELL21R_TIMESTAMP}.json"

with open(paths["manifest_json"], "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

# Add manifest to display paths.
paths["manifest_json"] = paths["manifest_json"]

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 21R validation failed. Review validation output above.")

# ======================================================================================
# 9. Display
# ======================================================================================

print("=" * 100)
print("Cell 21R — Higher-frequency Secondary lock artifacts written")
print("=" * 100)
print(f"Model spec:                         {LOCKED_MODEL_SPEC}")
print(f"Stack lock version:                 {STACK_LOCK_VERSION}")
print(f"Stack lock decision ID:             {STACK_LOCK_DECISION_ID}")
print(f"Core lock decision ID:              {CORE_LOCK_DECISION_ID}")
print(f"Secondary lock decision ID:         {SECONDARY_LOCK_DECISION_ID}")
print(f"Superseded Secondary lock ID:       {SUPERSEDED_SECONDARY_LOCK_DECISION_ID}")
print(f"Secondary source candidate:         {SECONDARY_SOURCE_CANDIDATE_ID}")
print()
print("Selection rule:")
print(SELECTION_RULE_TEXT)
print()
print("Validation:")
display(validation)

print()
print("Locked Core rules — wide:")
display(core_locked_rules_wide)

print()
print("Locked higher-frequency Secondary rules — wide:")
display(secondary_locked_rules_wide)

print()
print("Locked Secondary exclusions:")
display(secondary_exclusions)

print()
print("Combined locked stack rules — long:")
display(stack_locked_rules_long)

print()
print("Decision summary from Cell 20R source candidate:")
display(decision_summary)

print()
print("Saved outputs:")
for k, p in paths.items():
    print(f"  {k}: {p}")

print("\nCELL 21R PASSED — HIGHER-FREQUENCY SECONDARY SIGNAL PARAMETERS LOCKED INTO ARTIFACTS")

In [ ]:
# ======================================================================================
# Cell 22R — Tertiary Front Fit Review
# ======================================================================================
#
# Objective:
#   Test whether a Tertiary Front-only layer can fit after the locked Core + higher-frequency
#   Secondary signal stack.
#
# Current locked stack baseline:
#   Stack:
#       core_secondary_frequency_unlock_signal_stack_locked_v1
#       core_secondary_frequency_unlock_signal_stack_locked_001
#
#   Core:
#       core_repaired_v1_locked_001
#
#   Secondary:
#       secondary_frequency_unlock_locked_001
#
# Tertiary test layer:
#   Front only: 12D / 15D / 18D
#
# Gate:
#   Tertiary Front is evaluated only on dates where:
#       no locked Core trade was selected
#       no locked higher-frequency Secondary trade was selected
#
# Selection:
#   If multiple Tertiary Front tenors qualify on a date:
#       select by BEST_SIGNAL_RANK
#
# Tertiary Front grid:
#   model_vrp_log:
#       [0.55, 0.60, 0.65]
#
#   z threshold:
#       [0.20, 0.30, 0.40, 0.50, 0.60]
#
#   RSI14 cap:
#       [72, 73, 74, 75]
#
#   rv21d_vol_pct floor:
#       [6.5, 7.0, 7.5, 8.0]
#
# Hard acceptance screen:
#   total combined frequency improvement >= +3.0 percentage points
#   combined win rate >= 80%
#   standalone Tertiary Front win rate > 75%
#   Tertiary Front avg/day > 0
#   combined avg/day deterioration <= $75/day versus locked Core+Secondary
#   combined 2026 avg/day >= $0/day
#
# Preferred screen:
#   same hard criteria, plus:
#       combined avg/day deterioration <= $50/day
#       combined 2026 avg/day >= locked Core+Secondary 2026 avg/day
#
# Explicitly measured:
#   drawdown/day deterioration versus locked Core+Secondary
#   worst-20/day deterioration versus locked Core+Secondary
#
# This cell does NOT:
#   - change Core
#   - change Secondary
#   - write a Tertiary lock
#   - change sizing
#   - implement production/intraday logic
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 900)
pd.set_option("display.width", 1400)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL22R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

LOCKED_STACK_VERSION = "core_secondary_frequency_unlock_signal_stack_locked_v1"
LOCKED_STACK_DECISION_ID = "core_secondary_frequency_unlock_signal_stack_locked_001"

LOCKED_CORE_DECISION_ID = "core_repaired_v1_locked_001"
LOCKED_SECONDARY_DECISION_ID = "secondary_frequency_unlock_locked_001"

TERTIARY_REVIEW_VERSION = "tertiary_front_fit_review_v1"

TERTIARY_FRONT_TENORS = [12, 15, 18]
TERTIARY_BUCKET = "Front"
TERTIARY_SELECTION_MODE = "BEST_SIGNAL_RANK"

TERTIARY_LOG_GRID = [0.55, 0.60, 0.65]
TERTIARY_Z_GRID = [0.20, 0.30, 0.40, 0.50, 0.60]
TERTIARY_RSI_CAP_GRID = [72.0, 73.0, 74.0, 75.0]
TERTIARY_RV21D_FLOOR_GRID = [6.5, 7.0, 7.5, 8.0]

# User-approved hard screen.
HARD_MIN_FREQUENCY_IMPROVEMENT = 0.03
HARD_MIN_COMBINED_WIN_RATE = 0.80
HARD_MIN_TERTIARY_FRONT_WIN_RATE = 0.75  # strict greater-than
HARD_MAX_COMBINED_AVG_DAY_DETERIORATION = 75.0
HARD_MIN_COMBINED_2026_AVG_DAY = 0.0
HARD_MIN_TERTIARY_AVG_DAY = 0.0

# Preferred screen.
PREFERRED_MAX_COMBINED_AVG_DAY_DETERIORATION = 50.0

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)

    if matches:
        return matches[0]

    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")

    return None


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    gp = float(pnl[pnl > 0].sum())
    gl = float(pnl[pnl < 0].sum())

    if gl < 0:
        return gp / abs(gl)

    if gp > 0:
        return np.inf

    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def summarize_trade_set(df, group_cols, total_eligible_dates):
    d = df.copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    for c in [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "forecast_vol_pct",
        "rv21d_vol_pct",
        "tertiary_model_vrp_log_min",
        "tertiary_model_vrp_z_3m_min",
        "tertiary_model_vrp_z_1y_min",
        "tertiary_RSI14_max",
        "tertiary_rv21d_vol_pct_min",
    ]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d = d.sort_values(["trade_date", "tenor"]).copy()

    grouped = d.groupby(group_cols, dropna=False, observed=False) if group_cols else [((), d)]
    rows = []

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {}
        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2020 = g[g["trade_date"].dt.year == 2020].copy()
        y2021 = g[g["trade_date"].dt.year == 2021].copy()
        y2022 = g[g["trade_date"].dt.year == 2022].copy()
        y2023 = g[g["trade_date"].dt.year == 2023].copy()
        y2024 = g[g["trade_date"].dt.year == 2024].copy()
        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        unique_dates = int(g["trade_date"].nunique())
        signal_date_frequency = unique_dates / total_eligible_dates if total_eligible_dates > 0 else np.nan

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": unique_dates,
            "signal_date_frequency": float(signal_date_frequency),

            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),

            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,

            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_25k": int((pnl <= -25_000).sum()),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2020": int(len(y2020)),
            "pnl_2020": float(y2020["normalized_pnl_dollars"].sum()) if len(y2020) else 0.0,
            "avg_pnl_per_day_2020": float(y2020["normalized_pnl_per_day"].mean()) if len(y2020) else np.nan,
            "worst_trade_2020": float(y2020["normalized_pnl_dollars"].min()) if len(y2020) else np.nan,

            "trades_2021": int(len(y2021)),
            "pnl_2021": float(y2021["normalized_pnl_dollars"].sum()) if len(y2021) else 0.0,
            "avg_pnl_per_day_2021": float(y2021["normalized_pnl_per_day"].mean()) if len(y2021) else np.nan,
            "worst_trade_2021": float(y2021["normalized_pnl_dollars"].min()) if len(y2021) else np.nan,

            "trades_2022": int(len(y2022)),
            "pnl_2022": float(y2022["normalized_pnl_dollars"].sum()) if len(y2022) else 0.0,
            "avg_pnl_per_day_2022": float(y2022["normalized_pnl_per_day"].mean()) if len(y2022) else np.nan,
            "worst_trade_2022": float(y2022["normalized_pnl_dollars"].min()) if len(y2022) else np.nan,

            "trades_2023": int(len(y2023)),
            "pnl_2023": float(y2023["normalized_pnl_dollars"].sum()) if len(y2023) else 0.0,
            "avg_pnl_per_day_2023": float(y2023["normalized_pnl_per_day"].mean()) if len(y2023) else np.nan,
            "worst_trade_2023": float(y2023["normalized_pnl_dollars"].min()) if len(y2023) else np.nan,

            "trades_2024": int(len(y2024)),
            "pnl_2024": float(y2024["normalized_pnl_dollars"].sum()) if len(y2024) else 0.0,
            "avg_pnl_per_day_2024": float(y2024["normalized_pnl_per_day"].mean()) if len(y2024) else np.nan,
            "worst_trade_2024": float(y2024["normalized_pnl_dollars"].min()) if len(y2024) else np.nan,

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        if "program_leg" in g.columns:
            row["core_trade_count"] = int(g["program_leg"].eq("Core").sum())
            row["secondary_trade_count"] = int(g["program_leg"].str.startswith("Secondary", na=False).sum())
            row["tertiary_trade_count"] = int(g["program_leg"].eq("Tertiary_Front").sum())

        for signal_col in [
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "tertiary_model_vrp_log_min",
            "tertiary_model_vrp_z_3m_min",
            "tertiary_model_vrp_z_1y_min",
            "tertiary_RSI14_max",
            "tertiary_rv21d_vol_pct_min",
        ]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        rows.append(row)

    return pd.DataFrame(rows)


def add_best_signal_ranks(df, group_cols, suffix):
    x = df.copy()

    if x.empty:
        return x

    x[f"rank_z_1y_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )

    x[f"rank_z_3m_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )

    x[f"rank_vrp_log_{suffix}"] = (
        x.groupby(group_cols)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    x[f"avg_signal_rank_{suffix}"] = x[
        [
            f"rank_z_1y_{suffix}",
            f"rank_z_3m_{suffix}",
            f"rank_vrp_log_{suffix}",
        ]
    ].mean(axis=1)

    x["bucket_center_tenor"] = 15
    x["distance_to_bucket_center"] = (x["tenor"] - x["bucket_center_tenor"]).abs()

    return x


# ======================================================================================
# 2. Load inputs
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01R_unified_fds_signal_base_panel_with_rv21d_*.parquet",
    required=True,
)

locked_stack_rules_path = latest_file(
    OUT_PROCESSED_DIR,
    "21R_core_secondary_frequency_unlock_locked_rules_long_*.parquet",
    required=True,
)

locked_stack_validation_path = latest_file(
    OUT_AUDIT_DIR,
    "21R_secondary_frequency_unlock_lock_validation_*.csv",
    required=True,
)

locked_stack_manifest_path = latest_file(
    OUT_AUDIT_DIR,
    "21R_secondary_frequency_unlock_lock_manifest_*.json",
    required=False,
)

locked_baseline_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "20R_suggested_combined_panel_frequency_unlock_confirmation_*.parquet",
    required=True,
)

locked_baseline_summary_path = latest_file(
    OUT_AUDIT_DIR,
    "20R_combined_program_summary_frequency_unlock_confirmation_*.csv",
    required=False,
)

base_raw = pd.read_parquet(base_panel_path)
locked_stack_rules = pd.read_parquet(locked_stack_rules_path)
locked_stack_validation = pd.read_csv(locked_stack_validation_path)
locked_baseline_panel = pd.read_parquet(locked_baseline_panel_path)

locked_baseline_summary_all = (
    pd.read_csv(locked_baseline_summary_path)
    if locked_baseline_summary_path
    else pd.DataFrame()
)

base_models_found = sorted(base_raw["model_spec"].dropna().unique().tolist()) if "model_spec" in base_raw.columns else []

base = base_raw.copy()
if "model_spec" in base.columns:
    base = base[base["model_spec"].eq(LOCKED_MODEL_SPEC)].copy()

base["trade_date"] = pd.to_datetime(base["trade_date"], errors="coerce").dt.normalize()
locked_baseline_panel["trade_date"] = pd.to_datetime(locked_baseline_panel["trade_date"], errors="coerce").dt.normalize()

for c in [
    "tenor",
    "tenor_bucket_order",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")
    if c in locked_baseline_panel.columns:
        locked_baseline_panel[c] = pd.to_numeric(locked_baseline_panel[c], errors="coerce")

total_eligible_signal_dates = int(base["trade_date"].nunique())
all_signal_dates = sorted(pd.to_datetime(base["trade_date"].dropna().unique()).tolist())

# ======================================================================================
# 3. Locked Core + Secondary baseline
# ======================================================================================

locked_baseline_panel = locked_baseline_panel.copy()
locked_baseline_panel["baseline_stack_version"] = LOCKED_STACK_VERSION
locked_baseline_panel["baseline_stack_decision_id"] = LOCKED_STACK_DECISION_ID
locked_baseline_panel["candidate_layer"] = "Locked_Core_Secondary"
locked_baseline_panel["tertiary_candidate_id"] = "BASELINE_NO_TERTIARY"
locked_baseline_panel["tertiary_selection_mode"] = "NONE"

locked_baseline_trade_dates = set(pd.to_datetime(locked_baseline_panel["trade_date"].dropna().unique()).tolist())
tertiary_eligible_dates = sorted(set(all_signal_dates) - locked_baseline_trade_dates)

locked_baseline_summary = summarize_trade_set(
    locked_baseline_panel.assign(selection_universe="locked_core_secondary_baseline"),
    group_cols=["selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

if locked_baseline_summary.empty:
    raise RuntimeError("Could not summarize locked Core+Secondary baseline panel.")

baseline_ref = locked_baseline_summary.iloc[0].to_dict()

baseline_frequency = float(baseline_ref["signal_date_frequency"])
baseline_combined_win_rate = float(baseline_ref["win_rate"])
baseline_avg_day = float(baseline_ref["avg_pnl_per_day"])
baseline_2026_avg_day = float(baseline_ref["avg_pnl_per_day_2026"])
baseline_drawdown_day = float(baseline_ref["pnl_per_day_drawdown"])
baseline_worst20_day = float(baseline_ref["worst_20_trade_pnl_per_day_sum"])

# ======================================================================================
# 4. Build Tertiary Front candidate grid
# ======================================================================================

candidate_rows = []

for log_min in TERTIARY_LOG_GRID:
    for z_min in TERTIARY_Z_GRID:
        for rsi_max in TERTIARY_RSI_CAP_GRID:
            for rv_min in TERTIARY_RV21D_FLOOR_GRID:
                candidate_id = (
                    f"tertiary_front_"
                    f"LOG{int(round(log_min * 100)):03d}_"
                    f"Z{int(round(z_min * 100)):03d}_"
                    f"RSI{int(round(rsi_max)):02d}_"
                    f"RV{int(round(rv_min * 10)):02d}"
                )

                candidate_rows.append({
                    "tertiary_candidate_id": candidate_id,
                    "tertiary_review_version": TERTIARY_REVIEW_VERSION,
                    "tertiary_layer": "Tertiary_Front",
                    "tertiary_bucket": TERTIARY_BUCKET,
                    "tertiary_tenors": ",".join(str(x) for x in TERTIARY_FRONT_TENORS),
                    "tertiary_selection_mode": TERTIARY_SELECTION_MODE,
                    "tertiary_model_vrp_log_min": float(log_min),
                    "tertiary_model_vrp_z_3m_min": float(z_min),
                    "tertiary_model_vrp_z_1y_min": float(z_min),
                    "tertiary_RSI14_max": float(rsi_max),
                    "tertiary_rv21d_vol_pct_min": float(rv_min),
                    "gate": (
                        "Only evaluated on dates with no locked Core and no locked Secondary trade."
                    ),
                    "selection_rule": "BEST_SIGNAL_RANK across qualifying Tertiary Front tenors.",
                })

candidate_grid = pd.DataFrame(candidate_rows)

expected_candidate_count = (
    len(TERTIARY_LOG_GRID)
    * len(TERTIARY_Z_GRID)
    * len(TERTIARY_RSI_CAP_GRID)
    * len(TERTIARY_RV21D_FLOOR_GRID)
)

# ======================================================================================
# 5. Select Tertiary Front trades
# ======================================================================================

required_front_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rv21d_vol_pct",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

missing_front_cols = [c for c in required_front_cols if c not in base.columns]

front_base = base[
    base["tenor"].isin(TERTIARY_FRONT_TENORS)
    & base["trade_date"].isin(tertiary_eligible_dates)
    & base["model_vrp_log"].notna()
    & base["model_vrp_z_3m"].notna()
    & base["model_vrp_z_1y"].notna()
    & base["RSI14"].notna()
    & base["rv21d_vol_pct"].notna()
    & base["normalized_pnl_dollars"].notna()
    & base["normalized_pnl_per_day"].notna()
].copy()

selected_tertiary_rows = []

for _, cand in candidate_grid.iterrows():
    q = front_base[
        (front_base["model_vrp_log"] > cand["tertiary_model_vrp_log_min"])
        & (front_base["model_vrp_z_3m"] > cand["tertiary_model_vrp_z_3m_min"])
        & (front_base["model_vrp_z_1y"] > cand["tertiary_model_vrp_z_1y_min"])
        & (front_base["RSI14"] < cand["tertiary_RSI14_max"])
        & (front_base["rv21d_vol_pct"] > cand["tertiary_rv21d_vol_pct_min"])
    ].copy()

    if q.empty:
        continue

    q["tertiary_candidate_id"] = cand["tertiary_candidate_id"]
    q["tertiary_review_version"] = cand["tertiary_review_version"]
    q["tertiary_layer"] = cand["tertiary_layer"]
    q["tertiary_bucket"] = cand["tertiary_bucket"]
    q["tertiary_tenors"] = cand["tertiary_tenors"]
    q["tertiary_selection_mode"] = cand["tertiary_selection_mode"]

    q["tertiary_model_vrp_log_min"] = cand["tertiary_model_vrp_log_min"]
    q["tertiary_model_vrp_z_3m_min"] = cand["tertiary_model_vrp_z_3m_min"]
    q["tertiary_model_vrp_z_1y_min"] = cand["tertiary_model_vrp_z_1y_min"]
    q["tertiary_RSI14_max"] = cand["tertiary_RSI14_max"]
    q["tertiary_rv21d_vol_pct_min"] = cand["tertiary_rv21d_vol_pct_min"]

    q["source_layer"] = "Tertiary"
    q["program_leg"] = "Tertiary_Front"
    q["candidate_layer"] = "Tertiary_Front"
    q["selection_universe"] = "tertiary_front_incremental_after_locked_core_secondary"
    q["baseline_stack_version"] = LOCKED_STACK_VERSION
    q["baseline_stack_decision_id"] = LOCKED_STACK_DECISION_ID

    q = add_best_signal_ranks(
        q,
        group_cols=["tertiary_candidate_id", "trade_date"],
        suffix="tertiary_date",
    )

    selected = (
        q.sort_values(
            [
                "tertiary_candidate_id",
                "trade_date",
                "avg_signal_rank_tertiary_date",
                "distance_to_bucket_center",
                "rank_z_1y_tertiary_date",
                "rank_z_3m_tertiary_date",
                "rank_vrp_log_tertiary_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["tertiary_candidate_id", "trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    selected_tertiary_rows.append(selected)

if selected_tertiary_rows:
    tertiary_selected_panel = pd.concat(selected_tertiary_rows, ignore_index=True)
else:
    tertiary_selected_panel = pd.DataFrame(columns=front_base.columns.tolist() + candidate_grid.columns.tolist())

# ======================================================================================
# 6. Summaries
# ======================================================================================

tertiary_summary_raw = summarize_trade_set(
    tertiary_selected_panel,
    group_cols=["tertiary_candidate_id"],
    total_eligible_dates=total_eligible_signal_dates,
)

# Merge all candidates so zero-trade candidates are retained.
tertiary_summary = candidate_grid.merge(
    tertiary_summary_raw,
    on="tertiary_candidate_id",
    how="left",
    validate="one_to_one",
)

zero_fill_cols = [
    "trades",
    "unique_trade_dates",
    "signal_date_frequency",
    "total_pnl",
    "total_pnl_per_day_sum",
    "trades_2020",
    "trades_2021",
    "trades_2022",
    "trades_2023",
    "trades_2024",
    "trades_2025",
    "trades_2026",
    "pnl_2020",
    "pnl_2021",
    "pnl_2022",
    "pnl_2023",
    "pnl_2024",
    "pnl_2025",
    "pnl_2026",
]

for c in zero_fill_cols:
    if c in tertiary_summary.columns:
        tertiary_summary[c] = tertiary_summary[c].fillna(0)

# Build combined panel by crossing locked baseline across candidate IDs and appending tertiary selected.
baseline_cross = locked_baseline_panel.merge(
    candidate_grid[
        [
            "tertiary_candidate_id",
            "tertiary_review_version",
            "tertiary_selection_mode",
            "tertiary_model_vrp_log_min",
            "tertiary_model_vrp_z_3m_min",
            "tertiary_model_vrp_z_1y_min",
            "tertiary_RSI14_max",
            "tertiary_rv21d_vol_pct_min",
        ]
    ],
    how="cross",
)

baseline_cross["selection_universe"] = "combined_locked_core_secondary_plus_tertiary_front"
baseline_cross["candidate_layer"] = "Locked_Core_Secondary"

tertiary_for_combined = tertiary_selected_panel.copy()
tertiary_for_combined["selection_universe"] = "combined_locked_core_secondary_plus_tertiary_front"

combined_panel = pd.concat(
    [
        baseline_cross,
        tertiary_for_combined,
    ],
    ignore_index=True,
)

combined_summary_raw = summarize_trade_set(
    combined_panel,
    group_cols=["tertiary_candidate_id"],
    total_eligible_dates=total_eligible_signal_dates,
)

comparison = candidate_grid.merge(
    tertiary_summary.add_prefix("tertiary_").rename(columns={"tertiary_tertiary_candidate_id": "tertiary_candidate_id"}),
    on="tertiary_candidate_id",
    how="left",
    validate="one_to_one",
).merge(
    combined_summary_raw.add_prefix("combined_").rename(columns={"combined_tertiary_candidate_id": "tertiary_candidate_id"}),
    on="tertiary_candidate_id",
    how="left",
    validate="one_to_one",
)

# Baseline columns repeated for clear audit.
comparison["baseline_trades"] = baseline_ref["trades"]
comparison["baseline_unique_trade_dates"] = baseline_ref["unique_trade_dates"]
comparison["baseline_signal_date_frequency"] = baseline_ref["signal_date_frequency"]
comparison["baseline_win_rate"] = baseline_ref["win_rate"]
comparison["baseline_total_pnl"] = baseline_ref["total_pnl"]
comparison["baseline_avg_pnl_per_day"] = baseline_ref["avg_pnl_per_day"]
comparison["baseline_profit_factor_pnl_per_day"] = baseline_ref["profit_factor_pnl_per_day"]
comparison["baseline_pnl_per_day_drawdown"] = baseline_ref["pnl_per_day_drawdown"]
comparison["baseline_worst_20_trade_pnl_per_day_sum"] = baseline_ref["worst_20_trade_pnl_per_day_sum"]
comparison["baseline_avg_pnl_per_day_2026"] = baseline_ref["avg_pnl_per_day_2026"]

# Deltas versus locked Core+Secondary.
comparison["delta_combined_unique_trade_dates_vs_locked_stack"] = (
    comparison["combined_unique_trade_dates"] - comparison["baseline_unique_trade_dates"]
)

comparison["delta_combined_signal_date_frequency_vs_locked_stack"] = (
    comparison["combined_signal_date_frequency"] - comparison["baseline_signal_date_frequency"]
)

comparison["delta_combined_win_rate_vs_locked_stack"] = (
    comparison["combined_win_rate"] - comparison["baseline_win_rate"]
)

comparison["delta_combined_total_pnl_vs_locked_stack"] = (
    comparison["combined_total_pnl"] - comparison["baseline_total_pnl"]
)

comparison["delta_combined_avg_pnl_per_day_vs_locked_stack"] = (
    comparison["combined_avg_pnl_per_day"] - comparison["baseline_avg_pnl_per_day"]
)

comparison["combined_avg_day_deterioration_vs_locked_stack"] = (
    comparison["baseline_avg_pnl_per_day"] - comparison["combined_avg_pnl_per_day"]
)

comparison["delta_combined_profit_factor_pnl_per_day_vs_locked_stack"] = (
    comparison["combined_profit_factor_pnl_per_day"] - comparison["baseline_profit_factor_pnl_per_day"]
)

comparison["delta_combined_pnl_per_day_drawdown_vs_locked_stack"] = (
    comparison["combined_pnl_per_day_drawdown"] - comparison["baseline_pnl_per_day_drawdown"]
)

comparison["combined_pnl_per_day_drawdown_deterioration_vs_locked_stack"] = (
    comparison["baseline_pnl_per_day_drawdown"] - comparison["combined_pnl_per_day_drawdown"]
)

comparison["delta_combined_worst_20_trade_pnl_per_day_sum_vs_locked_stack"] = (
    comparison["combined_worst_20_trade_pnl_per_day_sum"] - comparison["baseline_worst_20_trade_pnl_per_day_sum"]
)

comparison["combined_worst_20_trade_pnl_per_day_deterioration_vs_locked_stack"] = (
    comparison["baseline_worst_20_trade_pnl_per_day_sum"] - comparison["combined_worst_20_trade_pnl_per_day_sum"]
)

comparison["delta_combined_avg_pnl_per_day_2026_vs_locked_stack"] = (
    comparison["combined_avg_pnl_per_day_2026"] - comparison["baseline_avg_pnl_per_day_2026"]
)

# Hard screen.
comparison["passes_frequency_improvement_3pct"] = (
    comparison["delta_combined_signal_date_frequency_vs_locked_stack"] >= HARD_MIN_FREQUENCY_IMPROVEMENT
)

comparison["passes_combined_win_80pct"] = (
    comparison["combined_win_rate"] >= HARD_MIN_COMBINED_WIN_RATE
)

comparison["passes_tertiary_front_win_gt_75pct"] = (
    comparison["tertiary_win_rate"] > HARD_MIN_TERTIARY_FRONT_WIN_RATE
)

comparison["passes_tertiary_positive_avg_day"] = (
    comparison["tertiary_avg_pnl_per_day"] > HARD_MIN_TERTIARY_AVG_DAY
)

comparison["passes_combined_avg_day_deterioration_75"] = (
    comparison["combined_avg_day_deterioration_vs_locked_stack"] <= HARD_MAX_COMBINED_AVG_DAY_DETERIORATION
)

comparison["passes_combined_2026_nonnegative"] = (
    comparison["combined_avg_pnl_per_day_2026"] >= HARD_MIN_COMBINED_2026_AVG_DAY
)

comparison["passes_hard_tertiary_front_screen"] = (
    comparison["passes_frequency_improvement_3pct"]
    & comparison["passes_combined_win_80pct"]
    & comparison["passes_tertiary_front_win_gt_75pct"]
    & comparison["passes_tertiary_positive_avg_day"]
    & comparison["passes_combined_avg_day_deterioration_75"]
    & comparison["passes_combined_2026_nonnegative"]
)

# Preferred screen.
comparison["passes_combined_avg_day_deterioration_50"] = (
    comparison["combined_avg_day_deterioration_vs_locked_stack"] <= PREFERRED_MAX_COMBINED_AVG_DAY_DETERIORATION
)

comparison["passes_combined_2026_ge_locked_stack"] = (
    comparison["combined_avg_pnl_per_day_2026"] >= comparison["baseline_avg_pnl_per_day_2026"]
)

comparison["passes_preferred_tertiary_front_screen"] = (
    comparison["passes_hard_tertiary_front_screen"]
    & comparison["passes_combined_avg_day_deterioration_50"]
    & comparison["passes_combined_2026_ge_locked_stack"]
)

# Ranking: frequency first, then quality/risk.
comparison["tertiary_front_fit_score"] = (
    comparison["passes_preferred_tertiary_front_screen"].astype(int) * 2_000_000
    + comparison["passes_hard_tertiary_front_screen"].astype(int) * 1_000_000
    + comparison["delta_combined_signal_date_frequency_vs_locked_stack"].fillna(0) * 200_000
    + comparison["combined_win_rate"].fillna(0) * 60_000
    + comparison["tertiary_win_rate"].fillna(0) * 50_000
    + np.clip(comparison["tertiary_avg_pnl_per_day"].fillna(-10_000), -10_000, 10_000)
    + np.clip(comparison["combined_avg_pnl_per_day_2026"].fillna(-10_000), -10_000, 10_000)
    - np.clip(comparison["combined_avg_day_deterioration_vs_locked_stack"].fillna(10_000), -10_000, 10_000)
    - np.clip(comparison["combined_pnl_per_day_drawdown_deterioration_vs_locked_stack"].fillna(10_000), -10_000, 10_000) / 10
    - np.clip(comparison["combined_worst_20_trade_pnl_per_day_deterioration_vs_locked_stack"].fillna(10_000), -10_000, 10_000) / 10
)

comparison = comparison.sort_values(
    [
        "passes_preferred_tertiary_front_screen",
        "passes_hard_tertiary_front_screen",
        "delta_combined_signal_date_frequency_vs_locked_stack",
        "combined_win_rate",
        "tertiary_win_rate",
        "tertiary_avg_pnl_per_day",
        "combined_avg_pnl_per_day_2026",
        "tertiary_front_fit_score",
    ],
    ascending=[False, False, False, False, False, False, False, False],
).reset_index(drop=True)

comparison["comparison_rank"] = np.arange(1, len(comparison) + 1)

hard_pass_candidates = comparison[comparison["passes_hard_tertiary_front_screen"]].copy()
preferred_pass_candidates = comparison[comparison["passes_preferred_tertiary_front_screen"]].copy()

if not preferred_pass_candidates.empty:
    suggested = preferred_pass_candidates.head(1).copy()
elif not hard_pass_candidates.empty:
    suggested = hard_pass_candidates.head(1).copy()
else:
    suggested = comparison.head(1).copy()

suggested_candidate_id = suggested["tertiary_candidate_id"].iloc[0] if len(suggested) else None

suggested_tertiary_panel = tertiary_selected_panel[
    tertiary_selected_panel["tertiary_candidate_id"].eq(suggested_candidate_id)
].copy()

suggested_combined_panel = combined_panel[
    combined_panel["tertiary_candidate_id"].eq(suggested_candidate_id)
].copy()

# Additional suggested candidate cuts.
suggested_tertiary_by_tenor = summarize_trade_set(
    suggested_tertiary_panel,
    group_cols=["tertiary_candidate_id", "tenor"],
    total_eligible_dates=total_eligible_signal_dates,
)

suggested_tertiary_by_year = summarize_trade_set(
    suggested_tertiary_panel.assign(year=suggested_tertiary_panel["trade_date"].dt.year.astype(int)) if len(suggested_tertiary_panel) else suggested_tertiary_panel,
    group_cols=["tertiary_candidate_id", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)

suggested_combined_by_year = summarize_trade_set(
    suggested_combined_panel.assign(year=suggested_combined_panel["trade_date"].dt.year.astype(int)) if len(suggested_combined_panel) else suggested_combined_panel,
    group_cols=["tertiary_candidate_id", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)

worst_tertiary_trades = (
    tertiary_selected_panel
    .sort_values(
        ["tertiary_candidate_id", "normalized_pnl_per_day", "normalized_pnl_dollars"],
        ascending=[True, True, True],
    )
    .groupby("tertiary_candidate_id", as_index=False)
    .head(20)
    .copy()
)

summary_by_log_z = comparison.groupby(
    ["tertiary_model_vrp_log_min", "tertiary_model_vrp_z_3m_min"],
    dropna=False,
).agg(
    candidates=("tertiary_candidate_id", "count"),
    preferred_passes=("passes_preferred_tertiary_front_screen", "sum"),
    hard_passes=("passes_hard_tertiary_front_screen", "sum"),
    max_frequency_improvement=("delta_combined_signal_date_frequency_vs_locked_stack", "max"),
    max_tertiary_trades=("tertiary_trades", "max"),
    best_tertiary_win_rate=("tertiary_win_rate", "max"),
    best_combined_win_rate=("combined_win_rate", "max"),
    best_tertiary_avg_day=("tertiary_avg_pnl_per_day", "max"),
    best_combined_2026_avg_day=("combined_avg_pnl_per_day_2026", "max"),
    worst_drawdown_deterioration=("combined_pnl_per_day_drawdown_deterioration_vs_locked_stack", "max"),
    worst_worst20_deterioration=("combined_worst_20_trade_pnl_per_day_deterioration_vs_locked_stack", "max"),
).reset_index()

summary_by_rsi_rv = comparison.groupby(
    ["tertiary_RSI14_max", "tertiary_rv21d_vol_pct_min"],
    dropna=False,
).agg(
    candidates=("tertiary_candidate_id", "count"),
    preferred_passes=("passes_preferred_tertiary_front_screen", "sum"),
    hard_passes=("passes_hard_tertiary_front_screen", "sum"),
    max_frequency_improvement=("delta_combined_signal_date_frequency_vs_locked_stack", "max"),
    max_tertiary_trades=("tertiary_trades", "max"),
    best_tertiary_win_rate=("tertiary_win_rate", "max"),
    best_combined_win_rate=("combined_win_rate", "max"),
    best_tertiary_avg_day=("tertiary_avg_pnl_per_day", "max"),
    best_combined_2026_avg_day=("combined_avg_pnl_per_day_2026", "max"),
    worst_drawdown_deterioration=("combined_pnl_per_day_drawdown_deterioration_vs_locked_stack", "max"),
    worst_worst20_deterioration=("combined_worst_20_trade_pnl_per_day_deterioration_vs_locked_stack", "max"),
).reset_index()

# ======================================================================================
# 7. Validation
# ======================================================================================

validation_rows = []

missing_base_cols = [c for c in required_front_cols if c not in base.columns]

stack_validation_failures = (
    locked_stack_validation[locked_stack_validation["status"].eq("FAIL")]
    if {"check", "status"}.issubset(locked_stack_validation.columns)
    else pd.DataFrame({"dummy": [1]})
)

bad_front_non_tertiary_tenors = tertiary_selected_panel[
    ~tertiary_selected_panel["tenor"].isin(TERTIARY_FRONT_TENORS)
].copy() if len(tertiary_selected_panel) else pd.DataFrame()

bad_tertiary_on_baseline_date = tertiary_selected_panel[
    tertiary_selected_panel["trade_date"].isin(locked_baseline_trade_dates)
].copy() if len(tertiary_selected_panel) else pd.DataFrame()

bad_tertiary_dupes = (
    tertiary_selected_panel
    .groupby(["tertiary_candidate_id", "trade_date"])
    .size()
    .reset_index(name="rows")
) if len(tertiary_selected_panel) else pd.DataFrame(columns=["tertiary_candidate_id", "trade_date", "rows"])

bad_tertiary_dupes = bad_tertiary_dupes[bad_tertiary_dupes["rows"].gt(1)]

bad_combined_dupes = (
    combined_panel
    .groupby(["tertiary_candidate_id", "trade_date"])
    .size()
    .reset_index(name="rows")
)

bad_combined_dupes = bad_combined_dupes[bad_combined_dupes["rows"].gt(1)]

bad_normalized_pnl = base[
    base["normalized_pnl_dollars"].notna()
    & base["normalized_pnl_per_day"].notna()
    & (
        (
            base["normalized_pnl_dollars"] / base["tenor"].replace(0, np.nan)
        )
        - base["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
].copy()

expected_baseline_dates = 317
expected_baseline_frequency = expected_baseline_dates / total_eligible_signal_dates if total_eligible_signal_dates else np.nan

add_validation(
    validation_rows,
    "base_panel_loaded",
    "PASS" if len(base) > 0 else "FAIL",
    f"rows={len(base):,}; path={base_panel_path}; models_found={base_models_found}",
)

add_validation(
    validation_rows,
    "locked_model_filtered",
    "PASS" if sorted(base["model_spec"].dropna().unique().tolist()) == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_after_filter={sorted(base['model_spec'].dropna().unique().tolist()) if 'model_spec' in base.columns else []}",
)

add_validation(
    validation_rows,
    "required_base_columns_available",
    "PASS" if not missing_base_cols else "FAIL",
    f"missing={missing_base_cols}",
)

add_validation(
    validation_rows,
    "locked_stack_rules_loaded",
    "PASS" if len(locked_stack_rules) == 16 else "FAIL",
    f"rows={len(locked_stack_rules):,}; path={locked_stack_rules_path}",
)

add_validation(
    validation_rows,
    "locked_stack_validation_passed",
    "PASS" if stack_validation_failures.empty else "FAIL",
    f"failures={len(stack_validation_failures):,}; path={locked_stack_validation_path}",
)

add_validation(
    validation_rows,
    "locked_core_secondary_baseline_loaded",
    "PASS" if len(locked_baseline_panel) > 0 else "FAIL",
    f"rows={len(locked_baseline_panel):,}; path={locked_baseline_panel_path}",
)

add_validation(
    validation_rows,
    "locked_core_secondary_baseline_frequency_matches_expected",
    "PASS" if np.isclose(baseline_frequency, expected_baseline_frequency, atol=1e-8) else "WARN",
    f"baseline_frequency={baseline_frequency:.6%}; expected_from_cell20={expected_baseline_frequency:.6%}; baseline_dates={baseline_ref['unique_trade_dates']}",
)

add_validation(
    validation_rows,
    "tertiary_candidate_count",
    "PASS" if len(candidate_grid) == expected_candidate_count else "FAIL",
    f"candidates={len(candidate_grid):,}; expected={expected_candidate_count:,}",
)

add_validation(
    validation_rows,
    "tertiary_front_only",
    "PASS" if bad_front_non_tertiary_tenors.empty else "FAIL",
    f"bad_rows={len(bad_front_non_tertiary_tenors):,}; allowed_tenors={TERTIARY_FRONT_TENORS}",
)

add_validation(
    validation_rows,
    "tertiary_only_after_locked_stack_empty_dates",
    "PASS" if bad_tertiary_on_baseline_date.empty else "FAIL",
    f"bad_rows={len(bad_tertiary_on_baseline_date):,}; baseline_dates={len(locked_baseline_trade_dates):,}; tertiary_eligible_dates={len(tertiary_eligible_dates):,}",
)

add_validation(
    validation_rows,
    "one_tertiary_trade_per_candidate_date",
    "PASS" if bad_tertiary_dupes.empty else "FAIL",
    f"bad_rows={len(bad_tertiary_dupes):,}",
)

add_validation(
    validation_rows,
    "one_combined_trade_per_candidate_date",
    "PASS" if bad_combined_dupes.empty else "FAIL",
    f"bad_rows={len(bad_combined_dupes):,}",
)

add_validation(
    validation_rows,
    "normalized_pnl_per_day_formula",
    "PASS" if bad_normalized_pnl.empty else "FAIL",
    f"bad_rows={len(bad_normalized_pnl):,}",
)

add_validation(
    validation_rows,
    "hard_screen_candidates_found",
    "PASS" if len(hard_pass_candidates) > 0 else "WARN",
    f"hard_pass_candidates={len(hard_pass_candidates):,}; preferred_pass_candidates={len(preferred_pass_candidates):,}",
)

add_validation(
    validation_rows,
    "no_core_changes",
    "PASS",
    "Review-only cell. Core rules are not modified.",
)

add_validation(
    validation_rows,
    "no_secondary_changes",
    "PASS",
    "Review-only cell. Locked Secondary rules are not modified.",
)

add_validation(
    validation_rows,
    "no_tertiary_lock_written",
    "PASS",
    "Review-only cell. No Tertiary lock artifacts are written.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields are created or changed.",
)

add_validation(
    validation_rows,
    "no_production_or_intraday_implementation",
    "PASS",
    "No production or intraday logic is implemented.",
)

validation = pd.DataFrame(validation_rows)
failures = validation[validation["status"].eq("FAIL")]

# ======================================================================================
# 8. Save outputs
# ======================================================================================

paths = {}

paths["candidate_grid_csv"] = OUT_AUDIT_DIR / f"22R_tertiary_front_candidate_grid_{CELL22R_TIMESTAMP}.csv"

paths["tertiary_selected_panel_csv"] = OUT_AUDIT_DIR / f"22R_tertiary_front_selected_panel_{CELL22R_TIMESTAMP}.csv"
paths["tertiary_selected_panel_parquet"] = OUT_PROCESSED_DIR / f"22R_tertiary_front_selected_panel_{CELL22R_TIMESTAMP}.parquet"

paths["combined_panel_csv"] = OUT_AUDIT_DIR / f"22R_combined_core_secondary_tertiary_panel_{CELL22R_TIMESTAMP}.csv"
paths["combined_panel_parquet"] = OUT_PROCESSED_DIR / f"22R_combined_core_secondary_tertiary_panel_{CELL22R_TIMESTAMP}.parquet"

paths["baseline_summary_csv"] = OUT_AUDIT_DIR / f"22R_locked_core_secondary_baseline_summary_{CELL22R_TIMESTAMP}.csv"

paths["tertiary_summary_csv"] = OUT_AUDIT_DIR / f"22R_tertiary_front_summary_{CELL22R_TIMESTAMP}.csv"
paths["comparison_csv"] = OUT_AUDIT_DIR / f"22R_tertiary_front_candidate_comparison_{CELL22R_TIMESTAMP}.csv"
paths["comparison_parquet"] = OUT_PROCESSED_DIR / f"22R_tertiary_front_candidate_comparison_{CELL22R_TIMESTAMP}.parquet"

paths["hard_pass_candidates_csv"] = OUT_AUDIT_DIR / f"22R_tertiary_front_hard_pass_candidates_{CELL22R_TIMESTAMP}.csv"
paths["preferred_pass_candidates_csv"] = OUT_AUDIT_DIR / f"22R_tertiary_front_preferred_pass_candidates_{CELL22R_TIMESTAMP}.csv"
paths["suggested_candidate_csv"] = OUT_AUDIT_DIR / f"22R_tertiary_front_suggested_candidate_{CELL22R_TIMESTAMP}.csv"

paths["suggested_tertiary_panel_csv"] = OUT_AUDIT_DIR / f"22R_suggested_tertiary_front_panel_{CELL22R_TIMESTAMP}.csv"
paths["suggested_tertiary_panel_parquet"] = OUT_PROCESSED_DIR / f"22R_suggested_tertiary_front_panel_{CELL22R_TIMESTAMP}.parquet"

paths["suggested_combined_panel_csv"] = OUT_AUDIT_DIR / f"22R_suggested_combined_core_secondary_tertiary_panel_{CELL22R_TIMESTAMP}.csv"
paths["suggested_combined_panel_parquet"] = OUT_PROCESSED_DIR / f"22R_suggested_combined_core_secondary_tertiary_panel_{CELL22R_TIMESTAMP}.parquet"

paths["suggested_tertiary_by_tenor_csv"] = OUT_AUDIT_DIR / f"22R_suggested_tertiary_front_by_tenor_{CELL22R_TIMESTAMP}.csv"
paths["suggested_tertiary_by_year_csv"] = OUT_AUDIT_DIR / f"22R_suggested_tertiary_front_by_year_{CELL22R_TIMESTAMP}.csv"
paths["suggested_combined_by_year_csv"] = OUT_AUDIT_DIR / f"22R_suggested_combined_by_year_{CELL22R_TIMESTAMP}.csv"

paths["worst_tertiary_trades_csv"] = OUT_AUDIT_DIR / f"22R_tertiary_front_worst_trades_{CELL22R_TIMESTAMP}.csv"

paths["summary_by_log_z_csv"] = OUT_AUDIT_DIR / f"22R_tertiary_front_summary_by_log_z_{CELL22R_TIMESTAMP}.csv"
paths["summary_by_rsi_rv_csv"] = OUT_AUDIT_DIR / f"22R_tertiary_front_summary_by_rsi_rv_{CELL22R_TIMESTAMP}.csv"

paths["validation_csv"] = OUT_AUDIT_DIR / f"22R_tertiary_front_validation_{CELL22R_TIMESTAMP}.csv"

candidate_grid.to_csv(paths["candidate_grid_csv"], index=False)

tertiary_selected_panel.to_csv(paths["tertiary_selected_panel_csv"], index=False)
tertiary_selected_panel.to_parquet(paths["tertiary_selected_panel_parquet"], index=False)

combined_panel.to_csv(paths["combined_panel_csv"], index=False)
combined_panel.to_parquet(paths["combined_panel_parquet"], index=False)

locked_baseline_summary.to_csv(paths["baseline_summary_csv"], index=False)

tertiary_summary.to_csv(paths["tertiary_summary_csv"], index=False)
comparison.to_csv(paths["comparison_csv"], index=False)
comparison.to_parquet(paths["comparison_parquet"], index=False)

hard_pass_candidates.to_csv(paths["hard_pass_candidates_csv"], index=False)
preferred_pass_candidates.to_csv(paths["preferred_pass_candidates_csv"], index=False)
suggested.to_csv(paths["suggested_candidate_csv"], index=False)

suggested_tertiary_panel.to_csv(paths["suggested_tertiary_panel_csv"], index=False)
suggested_tertiary_panel.to_parquet(paths["suggested_tertiary_panel_parquet"], index=False)

suggested_combined_panel.to_csv(paths["suggested_combined_panel_csv"], index=False)
suggested_combined_panel.to_parquet(paths["suggested_combined_panel_parquet"], index=False)

suggested_tertiary_by_tenor.to_csv(paths["suggested_tertiary_by_tenor_csv"], index=False)
suggested_tertiary_by_year.to_csv(paths["suggested_tertiary_by_year_csv"], index=False)
suggested_combined_by_year.to_csv(paths["suggested_combined_by_year_csv"], index=False)

worst_tertiary_trades.to_csv(paths["worst_tertiary_trades_csv"], index=False)

summary_by_log_z.to_csv(paths["summary_by_log_z_csv"], index=False)
summary_by_rsi_rv.to_csv(paths["summary_by_rsi_rv_csv"], index=False)

validation.to_csv(paths["validation_csv"], index=False)

manifest = {
    "cell": "Cell 22R — Tertiary Front Fit Review",
    "timestamp": CELL22R_TIMESTAMP,
    "version": TERTIARY_REVIEW_VERSION,

    "objective": "Review whether a Tertiary Front-only layer can fit after locked Core + higher-frequency Secondary.",
    "review_only": True,

    "locked_stack_version": LOCKED_STACK_VERSION,
    "locked_stack_decision_id": LOCKED_STACK_DECISION_ID,
    "locked_core_decision_id": LOCKED_CORE_DECISION_ID,
    "locked_secondary_decision_id": LOCKED_SECONDARY_DECISION_ID,

    "base_panel_path": str(base_panel_path),
    "locked_stack_rules_path": str(locked_stack_rules_path),
    "locked_stack_validation_path": str(locked_stack_validation_path),
    "locked_stack_manifest_path": str(locked_stack_manifest_path) if locked_stack_manifest_path else None,
    "locked_baseline_panel_path": str(locked_baseline_panel_path),
    "locked_baseline_summary_path": str(locked_baseline_summary_path) if locked_baseline_summary_path else None,

    "total_eligible_signal_dates": int(total_eligible_signal_dates),
    "locked_baseline_trade_dates": int(len(locked_baseline_trade_dates)),
    "tertiary_eligible_dates": int(len(tertiary_eligible_dates)),

    "tertiary_layer": {
        "name": "Tertiary Front",
        "tenors": TERTIARY_FRONT_TENORS,
        "gate": "Only evaluated when locked Core+Secondary has no selected trade.",
        "selection_mode": TERTIARY_SELECTION_MODE,
    },

    "candidate_grid": {
        "model_vrp_log_min": TERTIARY_LOG_GRID,
        "model_vrp_z_3m_min_equals_z_1y_min": TERTIARY_Z_GRID,
        "RSI14_max": TERTIARY_RSI_CAP_GRID,
        "rv21d_vol_pct_min": TERTIARY_RV21D_FLOOR_GRID,
        "expected_candidate_count": expected_candidate_count,
        "actual_candidate_count": int(len(candidate_grid)),
    },

    "hard_screen": {
        "frequency_improvement_min": HARD_MIN_FREQUENCY_IMPROVEMENT,
        "combined_win_rate_min": HARD_MIN_COMBINED_WIN_RATE,
        "tertiary_front_win_rate_strictly_greater_than": HARD_MIN_TERTIARY_FRONT_WIN_RATE,
        "tertiary_avg_pnl_per_day_min": HARD_MIN_TERTIARY_AVG_DAY,
        "combined_avg_day_deterioration_max": HARD_MAX_COMBINED_AVG_DAY_DETERIORATION,
        "combined_2026_avg_day_min": HARD_MIN_COMBINED_2026_AVG_DAY,
    },

    "preferred_screen": {
        "combined_avg_day_deterioration_max": PREFERRED_MAX_COMBINED_AVG_DAY_DETERIORATION,
        "combined_2026_avg_day_min": "locked Core+Secondary baseline 2026 avg/day",
    },

    "baseline_summary": baseline_ref,

    "row_counts": {
        "candidate_grid": int(len(candidate_grid)),
        "front_base_rows": int(len(front_base)),
        "tertiary_selected_rows": int(len(tertiary_selected_panel)),
        "combined_panel_rows": int(len(combined_panel)),
        "hard_pass_candidates": int(len(hard_pass_candidates)),
        "preferred_pass_candidates": int(len(preferred_pass_candidates)),
    },

    "suggested_candidate_id": suggested_candidate_id,

    "constraints": [
        "No Core changes.",
        "No Secondary changes.",
        "No Tertiary lock written.",
        "No sizing changes.",
        "No production/intraday implementation.",
    ],

    "paths": {k: str(v) for k, v in paths.items()},
}

paths["manifest_json"] = OUT_AUDIT_DIR / f"22R_tertiary_front_manifest_{CELL22R_TIMESTAMP}.json"

with open(paths["manifest_json"], "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

paths["manifest_json"] = paths["manifest_json"]

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 22R validation failed. Review validation output above.")

# ======================================================================================
# 9. Display
# ======================================================================================

display_cols = [
    "comparison_rank",
    "tertiary_candidate_id",
    "tertiary_model_vrp_log_min",
    "tertiary_model_vrp_z_3m_min",
    "tertiary_model_vrp_z_1y_min",
    "tertiary_RSI14_max",
    "tertiary_rv21d_vol_pct_min",

    "tertiary_trades",
    "tertiary_unique_trade_dates",
    "tertiary_signal_date_frequency",
    "tertiary_win_rate",
    "tertiary_total_pnl",
    "tertiary_avg_pnl_per_day",
    "tertiary_profit_factor_pnl_per_day",
    "tertiary_pnl_per_day_drawdown",
    "tertiary_worst_20_trade_pnl_per_day_sum",
    "tertiary_avg_pnl_per_day_2021",
    "tertiary_avg_pnl_per_day_2022",
    "tertiary_avg_pnl_per_day_2025",
    "tertiary_avg_pnl_per_day_2026",

    "baseline_unique_trade_dates",
    "baseline_signal_date_frequency",
    "baseline_win_rate",
    "baseline_avg_pnl_per_day",
    "baseline_pnl_per_day_drawdown",
    "baseline_worst_20_trade_pnl_per_day_sum",
    "baseline_avg_pnl_per_day_2026",

    "combined_unique_trade_dates",
    "combined_signal_date_frequency",
    "combined_win_rate",
    "combined_total_pnl",
    "combined_avg_pnl_per_day",
    "combined_profit_factor_pnl_per_day",
    "combined_pnl_per_day_drawdown",
    "combined_worst_20_trade_pnl_per_day_sum",
    "combined_avg_pnl_per_day_2021",
    "combined_avg_pnl_per_day_2022",
    "combined_avg_pnl_per_day_2025",
    "combined_avg_pnl_per_day_2026",

    "delta_combined_unique_trade_dates_vs_locked_stack",
    "delta_combined_signal_date_frequency_vs_locked_stack",
    "delta_combined_win_rate_vs_locked_stack",
    "delta_combined_total_pnl_vs_locked_stack",
    "delta_combined_avg_pnl_per_day_vs_locked_stack",
    "combined_avg_day_deterioration_vs_locked_stack",
    "delta_combined_pnl_per_day_drawdown_vs_locked_stack",
    "combined_pnl_per_day_drawdown_deterioration_vs_locked_stack",
    "delta_combined_worst_20_trade_pnl_per_day_sum_vs_locked_stack",
    "combined_worst_20_trade_pnl_per_day_deterioration_vs_locked_stack",
    "delta_combined_avg_pnl_per_day_2026_vs_locked_stack",

    "passes_frequency_improvement_3pct",
    "passes_combined_win_80pct",
    "passes_tertiary_front_win_gt_75pct",
    "passes_tertiary_positive_avg_day",
    "passes_combined_avg_day_deterioration_75",
    "passes_combined_2026_nonnegative",
    "passes_hard_tertiary_front_screen",
    "passes_combined_avg_day_deterioration_50",
    "passes_combined_2026_ge_locked_stack",
    "passes_preferred_tertiary_front_screen",
    "tertiary_front_fit_score",
]

print("=" * 100)
print("Cell 22R — Tertiary Front Fit Review")
print("=" * 100)
print(f"Base panel:                         {base_panel_path}")
print(f"Locked stack rules:                 {locked_stack_rules_path}")
print(f"Locked baseline panel:              {locked_baseline_panel_path}")
print(f"Model spec:                         {LOCKED_MODEL_SPEC}")
print(f"Total eligible signal dates:        {total_eligible_signal_dates:,}")
print(f"Locked Core+Secondary dates:        {len(locked_baseline_trade_dates):,}")
print(f"Locked Core+Secondary frequency:    {baseline_frequency:.4%}")
print(f"Tertiary eligible dates:            {len(tertiary_eligible_dates):,}")
print(f"Tertiary Front candidates:          {len(candidate_grid):,}")
print(f"Hard-screen pass candidates:        {len(hard_pass_candidates):,}")
print(f"Preferred-screen pass candidates:   {len(preferred_pass_candidates):,}")
print(f"Suggested candidate:                {suggested_candidate_id}")
print("No Tertiary lock written:           True")
print("No Core changes:                    True")
print("No Secondary changes:               True")
print("No sizing changes:                  True")
print("No production lock:                 True")

print()
print("Validation:")
display(validation)

print()
print("Locked Core+Secondary baseline:")
display(locked_baseline_summary)

print()
print("Candidate comparison — all candidates:")
display(comparison[[c for c in display_cols if c in comparison.columns]])

print()
print("Hard-screen pass candidates:")
display(hard_pass_candidates[[c for c in display_cols if c in hard_pass_candidates.columns]])

print()
print("Preferred-screen pass candidates:")
display(preferred_pass_candidates[[c for c in display_cols if c in preferred_pass_candidates.columns]])

print()
print("Suggested candidate:")
display(suggested[[c for c in display_cols if c in suggested.columns]])

print()
print("Suggested Tertiary Front by tenor:")
display(suggested_tertiary_by_tenor)

print()
print("Suggested Tertiary Front by year:")
display(suggested_tertiary_by_year)

print()
print("Suggested combined stack by year:")
display(suggested_combined_by_year)

print()
print("Summary by log/z:")
display(summary_by_log_z)

print()
print("Summary by RSI/RV:")
display(summary_by_rsi_rv)

print()
print("Worst Tertiary Front trades by candidate:")
display(worst_tertiary_trades)

print()
print("Suggested Tertiary Front selected panel:")
display(suggested_tertiary_panel)

print()
print("Saved outputs:")
for k, p in paths.items():
    print(f"  {k}: {p}")

print("\nCELL 22R PASSED — TERTIARY FRONT FIT REVIEW COMPLETE")

## Canonical Tertiary Front combined comparison

Cell 22R generates the Tertiary Front candidate grid and selected panels. Its original combined comparison is superseded. Cell 22R-FIX below is the canonical repaired combined comparison and should be used for Tertiary Front decision metrics.


In [ ]:
# ======================================================================================
# Cell 22R-FIX — Tertiary Front Combined Comparison Repair
# ======================================================================================
#
# Objective:
#   Repair Cell 22R combined comparison only.
#
# Issue repaired:
#   Cell 22R correctly selected Tertiary Front trades, but the combined summary compared
#   candidate-level Tertiary rows without correctly attaching the locked Core+Secondary
#   baseline rows to each candidate.
#
# This repair:
#   - reuses Cell 22R candidate grid
#   - reuses Cell 22R selected Tertiary Front panel
#   - reuses locked Core+Secondary baseline panel from Cell 20R
#   - rebuilds combined baseline + tertiary panels correctly by candidate
#   - recomputes screens and suggested candidates
#
# This cell does NOT:
#   - change Core
#   - change Secondary
#   - rerun Tertiary selection
#   - write a Tertiary lock
#   - change sizing
#   - implement production/intraday logic
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 900)
pd.set_option("display.width", 1400)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL22R_FIX_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

LOCKED_STACK_VERSION = "core_secondary_frequency_unlock_signal_stack_locked_v1"
LOCKED_STACK_DECISION_ID = "core_secondary_frequency_unlock_signal_stack_locked_001"

TERTIARY_REVIEW_VERSION = "tertiary_front_fit_review_v1"
TERTIARY_REPAIR_VERSION = "tertiary_front_fit_review_combined_repair_v1"

# User-approved hard screen.
HARD_MIN_FREQUENCY_IMPROVEMENT = 0.03
HARD_MIN_COMBINED_WIN_RATE = 0.80
HARD_MIN_TERTIARY_FRONT_WIN_RATE = 0.75  # strict greater-than
HARD_MAX_COMBINED_AVG_DAY_DETERIORATION = 75.0
HARD_MIN_COMBINED_2026_AVG_DAY = 0.0
HARD_MIN_TERTIARY_AVG_DAY = 0.0

# Preferred screen.
PREFERRED_MAX_COMBINED_AVG_DAY_DETERIORATION = 50.0

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)

    if matches:
        return matches[0]

    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")

    return None


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


def profit_factor_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    gp = float(pnl[pnl > 0].sum())
    gl = float(pnl[pnl < 0].sum())

    if gl < 0:
        return gp / abs(gl)

    if gp > 0:
        return np.inf

    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum().to_numpy()
    running_peak = np.maximum.accumulate(np.r_[0.0, cumulative])[:-1]
    drawdown = cumulative - running_peak

    return float(np.min(drawdown))


def worst_rolling_sum(pnl, window):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna().astype(float)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def summarize_trade_set(df, group_cols, total_eligible_dates):
    d = df.copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    for c in [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "forecast_vol_pct",
        "rv21d_vol_pct",
        "tertiary_model_vrp_log_min",
        "tertiary_model_vrp_z_3m_min",
        "tertiary_model_vrp_z_1y_min",
        "tertiary_RSI14_max",
        "tertiary_rv21d_vol_pct_min",
    ]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame(columns=group_cols)

    d = d.sort_values(["trade_date", "tenor"]).copy()

    grouped = d.groupby(group_cols, dropna=False, observed=False) if group_cols else [((), d)]
    rows = []

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {}
        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        unique_dates = int(g["trade_date"].nunique())
        signal_date_frequency = unique_dates / total_eligible_dates if total_eligible_dates > 0 else np.nan

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": unique_dates,
            "signal_date_frequency": float(signal_date_frequency),

            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),

            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,

            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_25k": int((pnl <= -25_000).sum()),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),
        })

        for year in [2020, 2021, 2022, 2023, 2024, 2025, 2026]:
            y = g[g["trade_date"].dt.year == year].copy()

            row[f"trades_{year}"] = int(len(y))
            row[f"pnl_{year}"] = float(y["normalized_pnl_dollars"].sum()) if len(y) else 0.0
            row[f"avg_pnl_per_day_{year}"] = float(y["normalized_pnl_per_day"].mean()) if len(y) else np.nan
            row[f"worst_trade_{year}"] = float(y["normalized_pnl_dollars"].min()) if len(y) else np.nan

        if "program_leg" in g.columns:
            row["core_trade_count"] = int(g["program_leg"].eq("Core").sum())
            row["secondary_trade_count"] = int(g["program_leg"].str.startswith("Secondary", na=False).sum())
            row["tertiary_trade_count"] = int(g["program_leg"].eq("Tertiary_Front").sum())

        for signal_col in [
            "rv21d_vol_pct",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "tertiary_model_vrp_log_min",
            "tertiary_model_vrp_z_3m_min",
            "tertiary_model_vrp_z_1y_min",
            "tertiary_RSI14_max",
            "tertiary_rv21d_vol_pct_min",
        ]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        rows.append(row)

    return pd.DataFrame(rows)


# ======================================================================================
# 2. Load existing Cell 22R and baseline artifacts
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01R_unified_fds_signal_base_panel_with_rv21d_*.parquet",
    required=True,
)

candidate_grid_path = latest_file(
    OUT_AUDIT_DIR,
    "22R_tertiary_front_candidate_grid_*.csv",
    required=True,
)

tertiary_selected_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "22R_tertiary_front_selected_panel_*.parquet",
    required=True,
)

locked_baseline_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "20R_suggested_combined_panel_frequency_unlock_confirmation_*.parquet",
    required=True,
)

cell22_validation_path = latest_file(
    OUT_AUDIT_DIR,
    "22R_tertiary_front_validation_*.csv",
    required=True,
)

base_raw = pd.read_parquet(base_panel_path)
candidate_grid = pd.read_csv(candidate_grid_path)
tertiary_selected_panel = pd.read_parquet(tertiary_selected_panel_path)
locked_baseline_panel = pd.read_parquet(locked_baseline_panel_path)
cell22_validation = pd.read_csv(cell22_validation_path)

base = base_raw.copy()
if "model_spec" in base.columns:
    base = base[base["model_spec"].eq(LOCKED_MODEL_SPEC)].copy()

base["trade_date"] = pd.to_datetime(base["trade_date"], errors="coerce").dt.normalize()
tertiary_selected_panel["trade_date"] = pd.to_datetime(tertiary_selected_panel["trade_date"], errors="coerce").dt.normalize()
locked_baseline_panel["trade_date"] = pd.to_datetime(locked_baseline_panel["trade_date"], errors="coerce").dt.normalize()

for d in [base, tertiary_selected_panel, locked_baseline_panel]:
    for c in [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "forecast_vol_pct",
        "rv21d_vol_pct",
    ]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

for c in [
    "tertiary_model_vrp_log_min",
    "tertiary_model_vrp_z_3m_min",
    "tertiary_model_vrp_z_1y_min",
    "tertiary_RSI14_max",
    "tertiary_rv21d_vol_pct_min",
]:
    if c in candidate_grid.columns:
        candidate_grid[c] = pd.to_numeric(candidate_grid[c], errors="coerce")
    if c in tertiary_selected_panel.columns:
        tertiary_selected_panel[c] = pd.to_numeric(tertiary_selected_panel[c], errors="coerce")

total_eligible_signal_dates = int(base["trade_date"].nunique())

# ======================================================================================
# 3. Baseline summary
# ======================================================================================

locked_baseline_panel = locked_baseline_panel.copy()
locked_baseline_panel["baseline_stack_version"] = LOCKED_STACK_VERSION
locked_baseline_panel["baseline_stack_decision_id"] = LOCKED_STACK_DECISION_ID
locked_baseline_panel["candidate_layer"] = "Locked_Core_Secondary"

locked_baseline_summary = summarize_trade_set(
    locked_baseline_panel.assign(selection_universe="locked_core_secondary_baseline"),
    group_cols=["selection_universe"],
    total_eligible_dates=total_eligible_signal_dates,
)

if locked_baseline_summary.empty:
    raise RuntimeError("Could not summarize locked Core+Secondary baseline.")

baseline_ref = locked_baseline_summary.iloc[0].to_dict()

# ======================================================================================
# 4. Tertiary standalone summary
# ======================================================================================

tertiary_summary_raw = summarize_trade_set(
    tertiary_selected_panel,
    group_cols=["tertiary_candidate_id"],
    total_eligible_dates=total_eligible_signal_dates,
)

tertiary_summary = candidate_grid.merge(
    tertiary_summary_raw,
    on="tertiary_candidate_id",
    how="left",
    validate="one_to_one",
)

zero_fill_cols = [
    "trades",
    "unique_trade_dates",
    "signal_date_frequency",
    "total_pnl",
    "total_pnl_per_day_sum",
    "trades_2020",
    "trades_2021",
    "trades_2022",
    "trades_2023",
    "trades_2024",
    "trades_2025",
    "trades_2026",
    "pnl_2020",
    "pnl_2021",
    "pnl_2022",
    "pnl_2023",
    "pnl_2024",
    "pnl_2025",
    "pnl_2026",
]

for c in zero_fill_cols:
    if c in tertiary_summary.columns:
        tertiary_summary[c] = tertiary_summary[c].fillna(0)

# ======================================================================================
# 5. Correct combined panel by candidate
# ======================================================================================

# Critical repair:
#   Drop candidate-level tertiary columns from the baseline before crossing.
#   Then assign every baseline row to every Tertiary candidate.
candidate_cols_to_drop_from_baseline = [
    c for c in candidate_grid.columns
    if c in locked_baseline_panel.columns
]

baseline_clean = locked_baseline_panel.drop(
    columns=candidate_cols_to_drop_from_baseline,
    errors="ignore",
).copy()

baseline_cross = baseline_clean.merge(
    candidate_grid,
    how="cross",
)

baseline_cross["selection_universe"] = "combined_locked_core_secondary_plus_tertiary_front_FIX"
baseline_cross["candidate_layer"] = "Locked_Core_Secondary"

tertiary_for_combined = tertiary_selected_panel.copy()
tertiary_for_combined["selection_universe"] = "combined_locked_core_secondary_plus_tertiary_front_FIX"
tertiary_for_combined["candidate_layer"] = "Tertiary_Front"

combined_panel = pd.concat(
    [
        baseline_cross,
        tertiary_for_combined,
    ],
    ignore_index=True,
    sort=False,
)

combined_summary_raw = summarize_trade_set(
    combined_panel,
    group_cols=["tertiary_candidate_id"],
    total_eligible_dates=total_eligible_signal_dates,
)

# ======================================================================================
# 6. Comparison and screens
# ======================================================================================

comparison = candidate_grid.merge(
    tertiary_summary.add_prefix("tertiary_").rename(
        columns={"tertiary_tertiary_candidate_id": "tertiary_candidate_id"}
    ),
    on="tertiary_candidate_id",
    how="left",
    validate="one_to_one",
).merge(
    combined_summary_raw.add_prefix("combined_").rename(
        columns={"combined_tertiary_candidate_id": "tertiary_candidate_id"}
    ),
    on="tertiary_candidate_id",
    how="left",
    validate="one_to_one",
)

# Baseline repeated for audit.
comparison["baseline_trades"] = baseline_ref["trades"]
comparison["baseline_unique_trade_dates"] = baseline_ref["unique_trade_dates"]
comparison["baseline_signal_date_frequency"] = baseline_ref["signal_date_frequency"]
comparison["baseline_win_rate"] = baseline_ref["win_rate"]
comparison["baseline_total_pnl"] = baseline_ref["total_pnl"]
comparison["baseline_avg_pnl_per_day"] = baseline_ref["avg_pnl_per_day"]
comparison["baseline_profit_factor_pnl_per_day"] = baseline_ref["profit_factor_pnl_per_day"]
comparison["baseline_pnl_per_day_drawdown"] = baseline_ref["pnl_per_day_drawdown"]
comparison["baseline_worst_20_trade_pnl_per_day_sum"] = baseline_ref["worst_20_trade_pnl_per_day_sum"]
comparison["baseline_avg_pnl_per_day_2026"] = baseline_ref["avg_pnl_per_day_2026"]

# Deltas versus locked Core+Secondary.
comparison["delta_combined_unique_trade_dates_vs_locked_stack"] = (
    comparison["combined_unique_trade_dates"] - comparison["baseline_unique_trade_dates"]
)

comparison["delta_combined_signal_date_frequency_vs_locked_stack"] = (
    comparison["combined_signal_date_frequency"] - comparison["baseline_signal_date_frequency"]
)

comparison["delta_combined_win_rate_vs_locked_stack"] = (
    comparison["combined_win_rate"] - comparison["baseline_win_rate"]
)

comparison["delta_combined_total_pnl_vs_locked_stack"] = (
    comparison["combined_total_pnl"] - comparison["baseline_total_pnl"]
)

comparison["delta_combined_avg_pnl_per_day_vs_locked_stack"] = (
    comparison["combined_avg_pnl_per_day"] - comparison["baseline_avg_pnl_per_day"]
)

comparison["combined_avg_day_deterioration_vs_locked_stack"] = (
    comparison["baseline_avg_pnl_per_day"] - comparison["combined_avg_pnl_per_day"]
)

comparison["delta_combined_profit_factor_pnl_per_day_vs_locked_stack"] = (
    comparison["combined_profit_factor_pnl_per_day"] - comparison["baseline_profit_factor_pnl_per_day"]
)

comparison["delta_combined_pnl_per_day_drawdown_vs_locked_stack"] = (
    comparison["combined_pnl_per_day_drawdown"] - comparison["baseline_pnl_per_day_drawdown"]
)

comparison["combined_pnl_per_day_drawdown_deterioration_vs_locked_stack"] = (
    comparison["baseline_pnl_per_day_drawdown"] - comparison["combined_pnl_per_day_drawdown"]
)

comparison["delta_combined_worst_20_trade_pnl_per_day_sum_vs_locked_stack"] = (
    comparison["combined_worst_20_trade_pnl_per_day_sum"] - comparison["baseline_worst_20_trade_pnl_per_day_sum"]
)

comparison["combined_worst_20_trade_pnl_per_day_deterioration_vs_locked_stack"] = (
    comparison["baseline_worst_20_trade_pnl_per_day_sum"] - comparison["combined_worst_20_trade_pnl_per_day_sum"]
)

comparison["delta_combined_avg_pnl_per_day_2026_vs_locked_stack"] = (
    comparison["combined_avg_pnl_per_day_2026"] - comparison["baseline_avg_pnl_per_day_2026"]
)

# Sanity column: because Tertiary is only on baseline-empty dates, this should match.
comparison["expected_combined_unique_trade_dates"] = (
    comparison["baseline_unique_trade_dates"] + comparison["tertiary_unique_trade_dates"]
)

comparison["combined_date_count_reconciliation_diff"] = (
    comparison["combined_unique_trade_dates"] - comparison["expected_combined_unique_trade_dates"]
)

# Hard screen.
comparison["passes_frequency_improvement_3pct"] = (
    comparison["delta_combined_signal_date_frequency_vs_locked_stack"] >= HARD_MIN_FREQUENCY_IMPROVEMENT
)

comparison["passes_combined_win_80pct"] = (
    comparison["combined_win_rate"] >= HARD_MIN_COMBINED_WIN_RATE
)

comparison["passes_tertiary_front_win_gt_75pct"] = (
    comparison["tertiary_win_rate"] > HARD_MIN_TERTIARY_FRONT_WIN_RATE
)

comparison["passes_tertiary_positive_avg_day"] = (
    comparison["tertiary_avg_pnl_per_day"] > HARD_MIN_TERTIARY_AVG_DAY
)

comparison["passes_combined_avg_day_deterioration_75"] = (
    comparison["combined_avg_day_deterioration_vs_locked_stack"] <= HARD_MAX_COMBINED_AVG_DAY_DETERIORATION
)

comparison["passes_combined_2026_nonnegative"] = (
    comparison["combined_avg_pnl_per_day_2026"] >= HARD_MIN_COMBINED_2026_AVG_DAY
)

comparison["passes_hard_tertiary_front_screen"] = (
    comparison["passes_frequency_improvement_3pct"]
    & comparison["passes_combined_win_80pct"]
    & comparison["passes_tertiary_front_win_gt_75pct"]
    & comparison["passes_tertiary_positive_avg_day"]
    & comparison["passes_combined_avg_day_deterioration_75"]
    & comparison["passes_combined_2026_nonnegative"]
)

# Preferred screen.
comparison["passes_combined_avg_day_deterioration_50"] = (
    comparison["combined_avg_day_deterioration_vs_locked_stack"] <= PREFERRED_MAX_COMBINED_AVG_DAY_DETERIORATION
)

comparison["passes_combined_2026_ge_locked_stack"] = (
    comparison["combined_avg_pnl_per_day_2026"] >= comparison["baseline_avg_pnl_per_day_2026"]
)

comparison["passes_preferred_tertiary_front_screen"] = (
    comparison["passes_hard_tertiary_front_screen"]
    & comparison["passes_combined_avg_day_deterioration_50"]
    & comparison["passes_combined_2026_ge_locked_stack"]
)

comparison["tertiary_front_fit_score"] = (
    comparison["passes_preferred_tertiary_front_screen"].astype(int) * 2_000_000
    + comparison["passes_hard_tertiary_front_screen"].astype(int) * 1_000_000
    + comparison["delta_combined_signal_date_frequency_vs_locked_stack"].fillna(0) * 200_000
    + comparison["combined_win_rate"].fillna(0) * 60_000
    + comparison["tertiary_win_rate"].fillna(0) * 50_000
    + np.clip(comparison["tertiary_avg_pnl_per_day"].fillna(-10_000), -10_000, 10_000)
    + np.clip(comparison["combined_avg_pnl_per_day_2026"].fillna(-10_000), -10_000, 10_000)
    - np.clip(comparison["combined_avg_day_deterioration_vs_locked_stack"].fillna(10_000), -10_000, 10_000)
    - np.clip(comparison["combined_pnl_per_day_drawdown_deterioration_vs_locked_stack"].fillna(10_000), -10_000, 10_000) / 10
    - np.clip(comparison["combined_worst_20_trade_pnl_per_day_deterioration_vs_locked_stack"].fillna(10_000), -10_000, 10_000) / 10
)

comparison = comparison.sort_values(
    [
        "passes_preferred_tertiary_front_screen",
        "passes_hard_tertiary_front_screen",
        "delta_combined_signal_date_frequency_vs_locked_stack",
        "combined_win_rate",
        "tertiary_win_rate",
        "tertiary_avg_pnl_per_day",
        "combined_avg_pnl_per_day_2026",
        "tertiary_front_fit_score",
    ],
    ascending=[False, False, False, False, False, False, False, False],
).reset_index(drop=True)

comparison["comparison_rank"] = np.arange(1, len(comparison) + 1)

hard_pass_candidates = comparison[comparison["passes_hard_tertiary_front_screen"]].copy()
preferred_pass_candidates = comparison[comparison["passes_preferred_tertiary_front_screen"]].copy()

if not preferred_pass_candidates.empty:
    suggested = preferred_pass_candidates.head(1).copy()
elif not hard_pass_candidates.empty:
    suggested = hard_pass_candidates.head(1).copy()
else:
    suggested = comparison.head(1).copy()

suggested_candidate_id = suggested["tertiary_candidate_id"].iloc[0] if len(suggested) else None

suggested_tertiary_panel = tertiary_selected_panel[
    tertiary_selected_panel["tertiary_candidate_id"].eq(suggested_candidate_id)
].copy()

suggested_combined_panel = combined_panel[
    combined_panel["tertiary_candidate_id"].eq(suggested_candidate_id)
].copy()

suggested_tertiary_by_tenor = summarize_trade_set(
    suggested_tertiary_panel,
    group_cols=["tertiary_candidate_id", "tenor"],
    total_eligible_dates=total_eligible_signal_dates,
)

suggested_tertiary_by_year = summarize_trade_set(
    suggested_tertiary_panel.assign(year=suggested_tertiary_panel["trade_date"].dt.year.astype(int)) if len(suggested_tertiary_panel) else suggested_tertiary_panel,
    group_cols=["tertiary_candidate_id", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)

suggested_combined_by_year = summarize_trade_set(
    suggested_combined_panel.assign(year=suggested_combined_panel["trade_date"].dt.year.astype(int)) if len(suggested_combined_panel) else suggested_combined_panel,
    group_cols=["tertiary_candidate_id", "year"],
    total_eligible_dates=total_eligible_signal_dates,
)

# Summary views.
summary_by_log_z = comparison.groupby(
    ["tertiary_model_vrp_log_min", "tertiary_model_vrp_z_3m_min"],
    dropna=False,
).agg(
    candidates=("tertiary_candidate_id", "count"),
    preferred_passes=("passes_preferred_tertiary_front_screen", "sum"),
    hard_passes=("passes_hard_tertiary_front_screen", "sum"),
    max_frequency_improvement=("delta_combined_signal_date_frequency_vs_locked_stack", "max"),
    max_tertiary_trades=("tertiary_trades", "max"),
    best_tertiary_win_rate=("tertiary_win_rate", "max"),
    best_combined_win_rate=("combined_win_rate", "max"),
    best_tertiary_avg_day=("tertiary_avg_pnl_per_day", "max"),
    best_combined_2026_avg_day=("combined_avg_pnl_per_day_2026", "max"),
    worst_drawdown_deterioration=("combined_pnl_per_day_drawdown_deterioration_vs_locked_stack", "max"),
    worst_worst20_deterioration=("combined_worst_20_trade_pnl_per_day_deterioration_vs_locked_stack", "max"),
).reset_index()

summary_by_rsi_rv = comparison.groupby(
    ["tertiary_RSI14_max", "tertiary_rv21d_vol_pct_min"],
    dropna=False,
).agg(
    candidates=("tertiary_candidate_id", "count"),
    preferred_passes=("passes_preferred_tertiary_front_screen", "sum"),
    hard_passes=("passes_hard_tertiary_front_screen", "sum"),
    max_frequency_improvement=("delta_combined_signal_date_frequency_vs_locked_stack", "max"),
    max_tertiary_trades=("tertiary_trades", "max"),
    best_tertiary_win_rate=("tertiary_win_rate", "max"),
    best_combined_win_rate=("combined_win_rate", "max"),
    best_tertiary_avg_day=("tertiary_avg_pnl_per_day", "max"),
    best_combined_2026_avg_day=("combined_avg_pnl_per_day_2026", "max"),
    worst_drawdown_deterioration=("combined_pnl_per_day_drawdown_deterioration_vs_locked_stack", "max"),
    worst_worst20_deterioration=("combined_worst_20_trade_pnl_per_day_deterioration_vs_locked_stack", "max"),
).reset_index()

worst_tertiary_trades = (
    tertiary_selected_panel
    .sort_values(
        ["tertiary_candidate_id", "normalized_pnl_per_day", "normalized_pnl_dollars"],
        ascending=[True, True, True],
    )
    .groupby("tertiary_candidate_id", as_index=False)
    .head(20)
    .copy()
)

# ======================================================================================
# 7. Validation
# ======================================================================================

validation_rows = []

cell22_failures = (
    cell22_validation[cell22_validation["status"].eq("FAIL")]
    if {"check", "status"}.issubset(cell22_validation.columns)
    else pd.DataFrame({"dummy": [1]})
)

bad_date_recon = comparison[
    comparison["combined_date_count_reconciliation_diff"].abs().fillna(0).gt(0)
].copy()

bad_combined_dupes = (
    combined_panel
    .groupby(["tertiary_candidate_id", "trade_date"])
    .size()
    .reset_index(name="rows")
)

bad_combined_dupes = bad_combined_dupes[bad_combined_dupes["rows"].gt(1)]

baseline_dates = int(baseline_ref["unique_trade_dates"])

add_validation(
    validation_rows,
    "cell22_validation_no_failures",
    "PASS" if cell22_failures.empty else "FAIL",
    f"failures={len(cell22_failures):,}; path={cell22_validation_path}",
)

add_validation(
    validation_rows,
    "candidate_grid_loaded",
    "PASS" if len(candidate_grid) == 240 else "FAIL",
    f"rows={len(candidate_grid):,}; path={candidate_grid_path}",
)

add_validation(
    validation_rows,
    "tertiary_selected_panel_loaded",
    "PASS" if len(tertiary_selected_panel) > 0 else "FAIL",
    f"rows={len(tertiary_selected_panel):,}; path={tertiary_selected_panel_path}",
)

add_validation(
    validation_rows,
    "locked_baseline_panel_loaded",
    "PASS" if len(locked_baseline_panel) == baseline_dates else "WARN",
    f"rows={len(locked_baseline_panel):,}; summarized_dates={baseline_dates:,}; path={locked_baseline_panel_path}",
)

add_validation(
    validation_rows,
    "baseline_cross_assigned_to_each_candidate",
    "PASS" if len(baseline_cross) == len(candidate_grid) * len(locked_baseline_panel) else "FAIL",
    f"baseline_cross_rows={len(baseline_cross):,}; expected={len(candidate_grid) * len(locked_baseline_panel):,}",
)

add_validation(
    validation_rows,
    "combined_date_count_reconciles",
    "PASS" if bad_date_recon.empty else "FAIL",
    f"bad_rows={len(bad_date_recon):,}; expected combined dates = baseline dates + tertiary dates",
)

add_validation(
    validation_rows,
    "one_combined_trade_per_candidate_date",
    "PASS" if bad_combined_dupes.empty else "FAIL",
    f"bad_rows={len(bad_combined_dupes):,}",
)

add_validation(
    validation_rows,
    "hard_screen_candidates_found",
    "PASS" if len(hard_pass_candidates) > 0 else "WARN",
    f"hard_pass_candidates={len(hard_pass_candidates):,}; preferred_pass_candidates={len(preferred_pass_candidates):,}",
)

add_validation(
    validation_rows,
    "no_core_changes",
    "PASS",
    "Repair-only cell. Core rules are not modified.",
)

add_validation(
    validation_rows,
    "no_secondary_changes",
    "PASS",
    "Repair-only cell. Locked Secondary rules are not modified.",
)

add_validation(
    validation_rows,
    "no_tertiary_lock_written",
    "PASS",
    "Repair-only cell. No Tertiary lock artifacts are written.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "No sizing fields are created or changed.",
)

add_validation(
    validation_rows,
    "no_production_or_intraday_implementation",
    "PASS",
    "No production or intraday logic is implemented.",
)

validation = pd.DataFrame(validation_rows)
failures = validation[validation["status"].eq("FAIL")]

# ======================================================================================
# 8. Save repaired outputs
# ======================================================================================

paths = {}

paths["baseline_summary_csv"] = OUT_AUDIT_DIR / f"22R_FIX_locked_core_secondary_baseline_summary_{CELL22R_FIX_TIMESTAMP}.csv"

paths["combined_panel_csv"] = OUT_AUDIT_DIR / f"22R_FIX_combined_core_secondary_tertiary_panel_{CELL22R_FIX_TIMESTAMP}.csv"
paths["combined_panel_parquet"] = OUT_PROCESSED_DIR / f"22R_FIX_combined_core_secondary_tertiary_panel_{CELL22R_FIX_TIMESTAMP}.parquet"

paths["tertiary_summary_csv"] = OUT_AUDIT_DIR / f"22R_FIX_tertiary_front_summary_{CELL22R_FIX_TIMESTAMP}.csv"

paths["comparison_csv"] = OUT_AUDIT_DIR / f"22R_FIX_tertiary_front_candidate_comparison_{CELL22R_FIX_TIMESTAMP}.csv"
paths["comparison_parquet"] = OUT_PROCESSED_DIR / f"22R_FIX_tertiary_front_candidate_comparison_{CELL22R_FIX_TIMESTAMP}.parquet"

paths["hard_pass_candidates_csv"] = OUT_AUDIT_DIR / f"22R_FIX_tertiary_front_hard_pass_candidates_{CELL22R_FIX_TIMESTAMP}.csv"
paths["preferred_pass_candidates_csv"] = OUT_AUDIT_DIR / f"22R_FIX_tertiary_front_preferred_pass_candidates_{CELL22R_FIX_TIMESTAMP}.csv"
paths["suggested_candidate_csv"] = OUT_AUDIT_DIR / f"22R_FIX_tertiary_front_suggested_candidate_{CELL22R_FIX_TIMESTAMP}.csv"

paths["suggested_tertiary_panel_csv"] = OUT_AUDIT_DIR / f"22R_FIX_suggested_tertiary_front_panel_{CELL22R_FIX_TIMESTAMP}.csv"
paths["suggested_tertiary_panel_parquet"] = OUT_PROCESSED_DIR / f"22R_FIX_suggested_tertiary_front_panel_{CELL22R_FIX_TIMESTAMP}.parquet"

paths["suggested_combined_panel_csv"] = OUT_AUDIT_DIR / f"22R_FIX_suggested_combined_core_secondary_tertiary_panel_{CELL22R_FIX_TIMESTAMP}.csv"
paths["suggested_combined_panel_parquet"] = OUT_PROCESSED_DIR / f"22R_FIX_suggested_combined_core_secondary_tertiary_panel_{CELL22R_FIX_TIMESTAMP}.parquet"

paths["suggested_tertiary_by_tenor_csv"] = OUT_AUDIT_DIR / f"22R_FIX_suggested_tertiary_front_by_tenor_{CELL22R_FIX_TIMESTAMP}.csv"
paths["suggested_tertiary_by_year_csv"] = OUT_AUDIT_DIR / f"22R_FIX_suggested_tertiary_front_by_year_{CELL22R_FIX_TIMESTAMP}.csv"
paths["suggested_combined_by_year_csv"] = OUT_AUDIT_DIR / f"22R_FIX_suggested_combined_by_year_{CELL22R_FIX_TIMESTAMP}.csv"

paths["worst_tertiary_trades_csv"] = OUT_AUDIT_DIR / f"22R_FIX_tertiary_front_worst_trades_{CELL22R_FIX_TIMESTAMP}.csv"

paths["summary_by_log_z_csv"] = OUT_AUDIT_DIR / f"22R_FIX_tertiary_front_summary_by_log_z_{CELL22R_FIX_TIMESTAMP}.csv"
paths["summary_by_rsi_rv_csv"] = OUT_AUDIT_DIR / f"22R_FIX_tertiary_front_summary_by_rsi_rv_{CELL22R_FIX_TIMESTAMP}.csv"

paths["validation_csv"] = OUT_AUDIT_DIR / f"22R_FIX_tertiary_front_validation_{CELL22R_FIX_TIMESTAMP}.csv"

locked_baseline_summary.to_csv(paths["baseline_summary_csv"], index=False)

combined_panel.to_csv(paths["combined_panel_csv"], index=False)
combined_panel.to_parquet(paths["combined_panel_parquet"], index=False)

tertiary_summary.to_csv(paths["tertiary_summary_csv"], index=False)

comparison.to_csv(paths["comparison_csv"], index=False)
comparison.to_parquet(paths["comparison_parquet"], index=False)

hard_pass_candidates.to_csv(paths["hard_pass_candidates_csv"], index=False)
preferred_pass_candidates.to_csv(paths["preferred_pass_candidates_csv"], index=False)
suggested.to_csv(paths["suggested_candidate_csv"], index=False)

suggested_tertiary_panel.to_csv(paths["suggested_tertiary_panel_csv"], index=False)
suggested_tertiary_panel.to_parquet(paths["suggested_tertiary_panel_parquet"], index=False)

suggested_combined_panel.to_csv(paths["suggested_combined_panel_csv"], index=False)
suggested_combined_panel.to_parquet(paths["suggested_combined_panel_parquet"], index=False)

suggested_tertiary_by_tenor.to_csv(paths["suggested_tertiary_by_tenor_csv"], index=False)
suggested_tertiary_by_year.to_csv(paths["suggested_tertiary_by_year_csv"], index=False)
suggested_combined_by_year.to_csv(paths["suggested_combined_by_year_csv"], index=False)

worst_tertiary_trades.to_csv(paths["worst_tertiary_trades_csv"], index=False)

summary_by_log_z.to_csv(paths["summary_by_log_z_csv"], index=False)
summary_by_rsi_rv.to_csv(paths["summary_by_rsi_rv_csv"], index=False)

validation.to_csv(paths["validation_csv"], index=False)

manifest = {
    "cell": "Cell 22R-FIX — Tertiary Front Combined Comparison Repair",
    "timestamp": CELL22R_FIX_TIMESTAMP,
    "version": TERTIARY_REPAIR_VERSION,

    "repair_reason": (
        "Cell 22R combined candidate comparison did not correctly attach locked Core+Secondary baseline rows "
        "to each Tertiary candidate before summarization."
    ),

    "base_panel_path": str(base_panel_path),
    "candidate_grid_path": str(candidate_grid_path),
    "tertiary_selected_panel_path": str(tertiary_selected_panel_path),
    "locked_baseline_panel_path": str(locked_baseline_panel_path),
    "cell22_validation_path": str(cell22_validation_path),

    "total_eligible_signal_dates": int(total_eligible_signal_dates),
    "baseline_summary": baseline_ref,

    "row_counts": {
        "candidate_grid": int(len(candidate_grid)),
        "locked_baseline_panel": int(len(locked_baseline_panel)),
        "baseline_cross": int(len(baseline_cross)),
        "tertiary_selected_panel": int(len(tertiary_selected_panel)),
        "combined_panel": int(len(combined_panel)),
        "hard_pass_candidates": int(len(hard_pass_candidates)),
        "preferred_pass_candidates": int(len(preferred_pass_candidates)),
    },

    "hard_screen": {
        "frequency_improvement_min": HARD_MIN_FREQUENCY_IMPROVEMENT,
        "combined_win_rate_min": HARD_MIN_COMBINED_WIN_RATE,
        "tertiary_front_win_rate_strictly_greater_than": HARD_MIN_TERTIARY_FRONT_WIN_RATE,
        "tertiary_avg_pnl_per_day_min": HARD_MIN_TERTIARY_AVG_DAY,
        "combined_avg_day_deterioration_max": HARD_MAX_COMBINED_AVG_DAY_DETERIORATION,
        "combined_2026_avg_day_min": HARD_MIN_COMBINED_2026_AVG_DAY,
    },

    "preferred_screen": {
        "combined_avg_day_deterioration_max": PREFERRED_MAX_COMBINED_AVG_DAY_DETERIORATION,
        "combined_2026_avg_day_min": "locked Core+Secondary baseline 2026 avg/day",
    },

    "suggested_candidate_id": suggested_candidate_id,

    "constraints": [
        "Repair-only.",
        "No Core changes.",
        "No Secondary changes.",
        "No Tertiary lock written.",
        "No sizing changes.",
        "No production/intraday implementation.",
    ],

    "paths": {k: str(v) for k, v in paths.items()},
}

paths["manifest_json"] = OUT_AUDIT_DIR / f"22R_FIX_tertiary_front_manifest_{CELL22R_FIX_TIMESTAMP}.json"

with open(paths["manifest_json"], "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

paths["manifest_json"] = paths["manifest_json"]

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 22R-FIX validation failed. Review validation output above.")

# ======================================================================================
# 9. Display
# ======================================================================================

display_cols = [
    "comparison_rank",
    "tertiary_candidate_id",
    "tertiary_model_vrp_log_min",
    "tertiary_model_vrp_z_3m_min",
    "tertiary_model_vrp_z_1y_min",
    "tertiary_RSI14_max",
    "tertiary_rv21d_vol_pct_min",

    "tertiary_trades",
    "tertiary_unique_trade_dates",
    "tertiary_signal_date_frequency",
    "tertiary_win_rate",
    "tertiary_total_pnl",
    "tertiary_avg_pnl_per_day",
    "tertiary_profit_factor_pnl_per_day",
    "tertiary_pnl_per_day_drawdown",
    "tertiary_worst_20_trade_pnl_per_day_sum",
    "tertiary_avg_pnl_per_day_2021",
    "tertiary_avg_pnl_per_day_2022",
    "tertiary_avg_pnl_per_day_2025",
    "tertiary_avg_pnl_per_day_2026",

    "baseline_unique_trade_dates",
    "baseline_signal_date_frequency",
    "baseline_win_rate",
    "baseline_avg_pnl_per_day",
    "baseline_pnl_per_day_drawdown",
    "baseline_worst_20_trade_pnl_per_day_sum",
    "baseline_avg_pnl_per_day_2026",

    "combined_unique_trade_dates",
    "expected_combined_unique_trade_dates",
    "combined_date_count_reconciliation_diff",
    "combined_signal_date_frequency",
    "combined_win_rate",
    "combined_total_pnl",
    "combined_avg_pnl_per_day",
    "combined_profit_factor_pnl_per_day",
    "combined_pnl_per_day_drawdown",
    "combined_worst_20_trade_pnl_per_day_sum",
    "combined_avg_pnl_per_day_2021",
    "combined_avg_pnl_per_day_2022",
    "combined_avg_pnl_per_day_2025",
    "combined_avg_pnl_per_day_2026",

    "delta_combined_unique_trade_dates_vs_locked_stack",
    "delta_combined_signal_date_frequency_vs_locked_stack",
    "delta_combined_win_rate_vs_locked_stack",
    "delta_combined_total_pnl_vs_locked_stack",
    "delta_combined_avg_pnl_per_day_vs_locked_stack",
    "combined_avg_day_deterioration_vs_locked_stack",
    "delta_combined_pnl_per_day_drawdown_vs_locked_stack",
    "combined_pnl_per_day_drawdown_deterioration_vs_locked_stack",
    "delta_combined_worst_20_trade_pnl_per_day_sum_vs_locked_stack",
    "combined_worst_20_trade_pnl_per_day_deterioration_vs_locked_stack",
    "delta_combined_avg_pnl_per_day_2026_vs_locked_stack",

    "passes_frequency_improvement_3pct",
    "passes_combined_win_80pct",
    "passes_tertiary_front_win_gt_75pct",
    "passes_tertiary_positive_avg_day",
    "passes_combined_avg_day_deterioration_75",
    "passes_combined_2026_nonnegative",
    "passes_hard_tertiary_front_screen",
    "passes_combined_avg_day_deterioration_50",
    "passes_combined_2026_ge_locked_stack",
    "passes_preferred_tertiary_front_screen",
    "tertiary_front_fit_score",
]

print("=" * 100)
print("Cell 22R-FIX — Tertiary Front Combined Comparison Repair")
print("=" * 100)
print(f"Base panel:                         {base_panel_path}")
print(f"Cell 22R candidate grid:            {candidate_grid_path}")
print(f"Cell 22R Tertiary selected panel:   {tertiary_selected_panel_path}")
print(f"Locked baseline panel:              {locked_baseline_panel_path}")
print(f"Total eligible signal dates:        {total_eligible_signal_dates:,}")
print(f"Locked Core+Secondary dates:        {int(baseline_ref['unique_trade_dates']):,}")
print(f"Locked Core+Secondary frequency:    {baseline_ref['signal_date_frequency']:.4%}")
print(f"Tertiary candidates:                {len(candidate_grid):,}")
print(f"Hard-screen pass candidates:        {len(hard_pass_candidates):,}")
print(f"Preferred-screen pass candidates:   {len(preferred_pass_candidates):,}")
print(f"Suggested candidate:                {suggested_candidate_id}")
print("No Tertiary lock written:           True")
print("No Core changes:                    True")
print("No Secondary changes:               True")
print("No sizing changes:                  True")
print("No production lock:                 True")

print()
print("Validation:")
display(validation)

print()
print("Locked Core+Secondary baseline:")
display(locked_baseline_summary)

print()
print("Candidate comparison — repaired combined view:")
display(comparison[[c for c in display_cols if c in comparison.columns]])

print()
print("Hard-screen pass candidates:")
display(hard_pass_candidates[[c for c in display_cols if c in hard_pass_candidates.columns]])

print()
print("Preferred-screen pass candidates:")
display(preferred_pass_candidates[[c for c in display_cols if c in preferred_pass_candidates.columns]])

print()
print("Suggested candidate:")
display(suggested[[c for c in display_cols if c in suggested.columns]])

print()
print("Suggested Tertiary Front by tenor:")
display(suggested_tertiary_by_tenor)

print()
print("Suggested Tertiary Front by year:")
display(suggested_tertiary_by_year)

print()
print("Suggested combined stack by year:")
display(suggested_combined_by_year)

print()
print("Summary by log/z:")
display(summary_by_log_z)

print()
print("Summary by RSI/RV:")
display(summary_by_rsi_rv)

print()
print("Saved outputs:")
for k, p in paths.items():
    print(f"  {k}: {p}")

print("\nCELL 22R-FIX PASSED — TERTIARY FRONT COMBINED COMPARISON REPAIRED")

In [ ]:
# ======================================================================================
# Cell 23R — Write Tertiary Front cleaner-quality lock artifacts
# ======================================================================================
#
# Objective:
#   Persist the cleaner-quality Tertiary Front lock confirmed in Cell 22R-FIX.
#
# Existing locked stack:
#   Core:
#       core_repaired_v1
#       core_repaired_v1_locked_001
#
#   Secondary:
#       secondary_frequency_unlock_locked_v1
#       secondary_frequency_unlock_locked_001
#
# New Tertiary lock:
#   tertiary_front_cleaner_quality_locked_v1
#   tertiary_front_cleaner_quality_locked_001
#
# Source candidate:
#   tertiary_front_LOG065_Z020_RSI75_RV70
#
# Tertiary Front:
#   Tenors:
#       12D / 15D / 18D
#
#   Parameters:
#       model_vrp_log > 0.65
#       model_vrp_z_3m > 0.20
#       model_vrp_z_1y > 0.20
#       RSI14 < 75
#       rv21d_vol_pct > 7.0
#
# Final selection stack:
#   1. Evaluate locked Core first.
#   2. If any Core tenor qualifies, select Core and ignore Secondary/Tertiary.
#   3. If no Core qualifies, evaluate locked Secondary Middle/Back.
#   4. If any Secondary tenor qualifies, select Secondary and ignore Tertiary.
#   5. If no Core and no Secondary qualifies, evaluate Tertiary Front.
#   6. If multiple Tertiary Front tenors qualify, select by BEST_SIGNAL_RANK.
#
# This cell does NOT:
#   - change Core
#   - change Secondary
#   - run parameter optimization
#   - change sizing
#   - implement production/intraday logic
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 700)
pd.set_option("display.width", 1400)

# ======================================================================================
# 0. Config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL23R_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

CORE_LOCK_VERSION = "core_repaired_v1"
CORE_LOCK_DECISION_ID = "core_repaired_v1_locked_001"

SECONDARY_LOCK_VERSION = "secondary_frequency_unlock_locked_v1"
SECONDARY_LOCK_DECISION_ID = "secondary_frequency_unlock_locked_001"

TERTIARY_LOCK_VERSION = "tertiary_front_cleaner_quality_locked_v1"
TERTIARY_LOCK_DECISION_ID = "tertiary_front_cleaner_quality_locked_001"

STACK_LOCK_VERSION = "core_secondary_tertiary_signal_stack_locked_v1"
STACK_LOCK_DECISION_ID = "core_secondary_tertiary_signal_stack_locked_001"

SOURCE_STACK_VERSION = "core_secondary_frequency_unlock_signal_stack_locked_v1"
SOURCE_STACK_DECISION_ID = "core_secondary_frequency_unlock_signal_stack_locked_001"

TERTIARY_SOURCE_CELL = "Cell 22R-FIX — Tertiary Front Combined Comparison Repair"
TERTIARY_SOURCE_CANDIDATE_ID = "tertiary_front_LOG065_Z020_RSI75_RV70"

FINAL_SELECTION_RULE_TEXT = (
    "Core first. If any locked Core tenor qualifies on a date, select Core and ignore Secondary/Tertiary. "
    "If no Core tenor qualifies, evaluate locked Secondary Middle/Back. "
    "If any locked Secondary tenor qualifies, select Secondary and ignore Tertiary. "
    "If no Core and no Secondary qualifies, evaluate Tertiary Front only. "
    "If multiple Tertiary Front tenors qualify, select by BEST_SIGNAL_RANK."
)

# Tertiary parameters.
TERTIARY_TENORS = [12, 15, 18]
TERTIARY_BUCKET = "Front"
TERTIARY_BUCKET_ORDER = 1
TERTIARY_MODEL_VRP_LOG_MIN = 0.65
TERTIARY_MODEL_VRP_Z_3M_MIN = 0.20
TERTIARY_MODEL_VRP_Z_1Y_MIN = 0.20
TERTIARY_RSI14_MAX = 75.0
TERTIARY_RV21D_VOL_PCT_MIN = 7.0

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(Path(directory).glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)

    if matches:
        return matches[0]

    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")

    return None


def add_validation(rows, check, status, detail):
    rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


def normalize_bool_value(x):
    if isinstance(x, bool):
        return x

    if pd.isna(x):
        return False

    return str(x).strip().lower() in ["true", "1", "yes", "y"]


def normalize_bool_series(s):
    if s.dtype == bool:
        return s.fillna(False)

    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y"])
    )


def as_float(value):
    try:
        return float(value)
    except Exception:
        return np.nan


def build_tertiary_rule_row(tenor):
    return {
        "stack_lock_version": STACK_LOCK_VERSION,
        "stack_lock_decision_id": STACK_LOCK_DECISION_ID,

        "model_spec": LOCKED_MODEL_SPEC,

        "signal_layer": "Tertiary",
        "signal_priority": 3,

        "lock_version": TERTIARY_LOCK_VERSION,
        "lock_decision_id": TERTIARY_LOCK_DECISION_ID,

        "supersedes_lock_version": None,
        "supersedes_lock_decision_id": None,

        "tenor": int(tenor),
        "tenor_bucket": TERTIARY_BUCKET,
        "tenor_bucket_order": TERTIARY_BUCKET_ORDER,

        "include_tenor": True,

        "model_vrp_log_min": TERTIARY_MODEL_VRP_LOG_MIN,
        "model_vrp_z_3m_min": TERTIARY_MODEL_VRP_Z_3M_MIN,
        "model_vrp_z_1y_min": TERTIARY_MODEL_VRP_Z_1Y_MIN,
        "RSI14_max": TERTIARY_RSI14_MAX,
        "rv21d_vol_pct_min": TERTIARY_RV21D_VOL_PCT_MIN,

        "comparison_operator_model_vrp_log": ">",
        "comparison_operator_model_vrp_z_3m": ">",
        "comparison_operator_model_vrp_z_1y": ">",
        "comparison_operator_RSI14": "<",
        "comparison_operator_rv21d_vol_pct": ">",

        "vrp_measure": "model_vrp_log = log(implied_variance / forecast_variance)",
        "forecast_model": "har_total_simple / unified_fds_no_min_return",
        "rv_floor_contract": "rv21d_vol_pct > threshold",

        "selection_rule": FINAL_SELECTION_RULE_TEXT,

        "source_cell": TERTIARY_SOURCE_CELL,
        "source_candidate_id": TERTIARY_SOURCE_CANDIDATE_ID,
        "exclusion_reason": None,

        "created_timestamp": CELL23R_TIMESTAMP,
    }


def make_wide_rules(rules_long):
    active = rules_long[rules_long["include_tenor"].eq(True)].copy()

    wide = (
        active
        .groupby(
            [
                "signal_layer",
                "signal_priority",
                "lock_version",
                "lock_decision_id",
                "tenor_bucket",
                "tenor_bucket_order",
                "model_spec",
                "model_vrp_log_min",
                "model_vrp_z_3m_min",
                "model_vrp_z_1y_min",
                "RSI14_max",
                "rv21d_vol_pct_min",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            tenors=("tenor", lambda x: ",".join(str(int(v)) for v in sorted(x))),
            tenor_count=("tenor", "count"),
            stack_lock_version=("stack_lock_version", "first"),
            stack_lock_decision_id=("stack_lock_decision_id", "first"),
            selection_rule=("selection_rule", "first"),
            source_cell=("source_cell", "first"),
            source_candidate_id=("source_candidate_id", "first"),
        )
        .sort_values(["signal_priority", "tenor_bucket_order"])
        .reset_index(drop=True)
    )

    return wide


def active_rule_tuple_set(rules_long):
    active = rules_long[rules_long["include_tenor"].eq(True)].copy()

    for c in [
        "model_vrp_log_min",
        "model_vrp_z_3m_min",
        "model_vrp_z_1y_min",
        "RSI14_max",
        "rv21d_vol_pct_min",
    ]:
        active[c] = pd.to_numeric(active[c], errors="coerce").round(10)

    return set(
        active[
            [
                "signal_layer",
                "tenor",
                "tenor_bucket",
                "model_vrp_log_min",
                "model_vrp_z_3m_min",
                "model_vrp_z_1y_min",
                "RSI14_max",
                "rv21d_vol_pct_min",
            ]
        ].itertuples(index=False, name=None)
    )


# ======================================================================================
# 2. Load source artifacts
# ======================================================================================

locked_core_secondary_stack_path = latest_file(
    OUT_PROCESSED_DIR,
    "21R_core_secondary_frequency_unlock_locked_rules_long_*.parquet",
    required=True,
)

cell22_fix_comparison_path = latest_file(
    OUT_AUDIT_DIR,
    "22R_FIX_tertiary_front_candidate_comparison_*.csv",
    required=True,
)

cell22_fix_validation_path = latest_file(
    OUT_AUDIT_DIR,
    "22R_FIX_tertiary_front_validation_*.csv",
    required=True,
)

cell22_fix_manifest_path = latest_file(
    OUT_AUDIT_DIR,
    "22R_FIX_tertiary_front_manifest_*.json",
    required=False,
)

cell22_fix_suggested_tertiary_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "22R_FIX_suggested_tertiary_front_panel_*.parquet",
    required=False,
)

cell22_fix_suggested_combined_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "22R_FIX_suggested_combined_core_secondary_tertiary_panel_*.parquet",
    required=False,
)

locked_core_secondary_stack = pd.read_parquet(locked_core_secondary_stack_path)
cell22_fix_comparison = pd.read_csv(cell22_fix_comparison_path)
cell22_fix_validation = pd.read_csv(cell22_fix_validation_path)

# ======================================================================================
# 3. Prepare unchanged Core + Secondary rows from Cell 21R source stack
# ======================================================================================

core_secondary_stack_source = locked_core_secondary_stack.copy()

required_stack_cols = [
    "signal_layer",
    "signal_priority",
    "lock_version",
    "lock_decision_id",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "include_tenor",
    "model_vrp_log_min",
    "model_vrp_z_3m_min",
    "model_vrp_z_1y_min",
    "RSI14_max",
    "rv21d_vol_pct_min",
]

missing_stack_cols = [c for c in required_stack_cols if c not in core_secondary_stack_source.columns]

if missing_stack_cols:
    raise RuntimeError(f"Missing required source stack columns: {missing_stack_cols}")

for c in [
    "tenor",
    "tenor_bucket_order",
    "signal_priority",
    "model_vrp_log_min",
    "model_vrp_z_3m_min",
    "model_vrp_z_1y_min",
    "RSI14_max",
    "rv21d_vol_pct_min",
]:
    if c in core_secondary_stack_source.columns:
        core_secondary_stack_source[c] = pd.to_numeric(core_secondary_stack_source[c], errors="coerce")

core_secondary_stack_source["include_tenor"] = normalize_bool_series(core_secondary_stack_source["include_tenor"])

# These rows remain unchanged at the signal-lock level.
core_secondary_stack_new = core_secondary_stack_source.copy()

core_secondary_stack_new["stack_lock_version"] = STACK_LOCK_VERSION
core_secondary_stack_new["stack_lock_decision_id"] = STACK_LOCK_DECISION_ID
core_secondary_stack_new["selection_rule"] = FINAL_SELECTION_RULE_TEXT
core_secondary_stack_new["created_timestamp"] = CELL23R_TIMESTAMP

# Keep original Core/Secondary lock versions and decision IDs intact.
core_locked_rules_long = core_secondary_stack_new[
    core_secondary_stack_new["signal_layer"].eq("Core")
].copy()

secondary_locked_rules_long = core_secondary_stack_new[
    core_secondary_stack_new["signal_layer"].eq("Secondary")
].copy()

# ======================================================================================
# 4. Build locked Tertiary Front rows
# ======================================================================================

tertiary_locked_rules_long = pd.DataFrame(
    [build_tertiary_rule_row(tenor) for tenor in TERTIARY_TENORS]
)

# Align columns with Core/Secondary source.
all_cols = list(dict.fromkeys(
    list(core_secondary_stack_new.columns) + list(tertiary_locked_rules_long.columns)
))

core_secondary_stack_new = core_secondary_stack_new.reindex(columns=all_cols)
tertiary_locked_rules_long = tertiary_locked_rules_long.reindex(columns=all_cols)

stack_locked_rules_long = pd.concat(
    [
        core_secondary_stack_new,
        tertiary_locked_rules_long,
    ],
    ignore_index=True,
    sort=False,
)

for c in [
    "tenor",
    "tenor_bucket_order",
    "signal_priority",
    "model_vrp_log_min",
    "model_vrp_z_3m_min",
    "model_vrp_z_1y_min",
    "RSI14_max",
    "rv21d_vol_pct_min",
]:
    if c in stack_locked_rules_long.columns:
        stack_locked_rules_long[c] = pd.to_numeric(stack_locked_rules_long[c], errors="coerce")

stack_locked_rules_long["include_tenor"] = normalize_bool_series(stack_locked_rules_long["include_tenor"])

core_locked_rules_long = stack_locked_rules_long[
    stack_locked_rules_long["signal_layer"].eq("Core")
].copy()

secondary_locked_rules_long = stack_locked_rules_long[
    stack_locked_rules_long["signal_layer"].eq("Secondary")
].copy()

tertiary_locked_rules_long = stack_locked_rules_long[
    stack_locked_rules_long["signal_layer"].eq("Tertiary")
].copy()

core_locked_rules_wide = make_wide_rules(core_locked_rules_long)
secondary_locked_rules_wide = make_wide_rules(secondary_locked_rules_long)
tertiary_locked_rules_wide = make_wide_rules(tertiary_locked_rules_long)
stack_locked_rules_wide = make_wide_rules(stack_locked_rules_long)

# ======================================================================================
# 5. Decision summary from Cell 22R-FIX source candidate
# ======================================================================================

selected22 = cell22_fix_comparison[
    cell22_fix_comparison["tertiary_candidate_id"].eq(TERTIARY_SOURCE_CANDIDATE_ID)
].copy()

decision_rows = []

decision_rows.append({
    "decision_item": "locked_tertiary_source_candidate",
    "value": TERTIARY_SOURCE_CANDIDATE_ID,
})

decision_rows.append({
    "decision_item": "tertiary_lock_version",
    "value": TERTIARY_LOCK_VERSION,
})

decision_rows.append({
    "decision_item": "tertiary_lock_decision_id",
    "value": TERTIARY_LOCK_DECISION_ID,
})

decision_rows.append({
    "decision_item": "stack_lock_version",
    "value": STACK_LOCK_VERSION,
})

decision_rows.append({
    "decision_item": "stack_lock_decision_id",
    "value": STACK_LOCK_DECISION_ID,
})

if not selected22.empty:
    selected22_row = selected22.iloc[0].to_dict()

    for metric in [
        "comparison_rank",
        "tertiary_candidate_id",
        "tertiary_model_vrp_log_min",
        "tertiary_model_vrp_z_3m_min",
        "tertiary_model_vrp_z_1y_min",
        "tertiary_RSI14_max",
        "tertiary_rv21d_vol_pct_min",

        "tertiary_trades",
        "tertiary_unique_trade_dates",
        "tertiary_signal_date_frequency",
        "tertiary_win_rate",
        "tertiary_total_pnl",
        "tertiary_avg_pnl_per_day",
        "tertiary_profit_factor_pnl_per_day",
        "tertiary_pnl_per_day_drawdown",
        "tertiary_worst_20_trade_pnl_per_day_sum",
        "tertiary_avg_pnl_per_day_2021",
        "tertiary_avg_pnl_per_day_2022",
        "tertiary_avg_pnl_per_day_2025",
        "tertiary_avg_pnl_per_day_2026",

        "baseline_unique_trade_dates",
        "baseline_signal_date_frequency",
        "baseline_win_rate",
        "baseline_avg_pnl_per_day",
        "baseline_pnl_per_day_drawdown",
        "baseline_worst_20_trade_pnl_per_day_sum",
        "baseline_avg_pnl_per_day_2026",

        "combined_unique_trade_dates",
        "combined_signal_date_frequency",
        "combined_win_rate",
        "combined_total_pnl",
        "combined_avg_pnl_per_day",
        "combined_profit_factor_pnl_per_day",
        "combined_pnl_per_day_drawdown",
        "combined_worst_20_trade_pnl_per_day_sum",
        "combined_avg_pnl_per_day_2021",
        "combined_avg_pnl_per_day_2022",
        "combined_avg_pnl_per_day_2025",
        "combined_avg_pnl_per_day_2026",

        "delta_combined_unique_trade_dates_vs_locked_stack",
        "delta_combined_signal_date_frequency_vs_locked_stack",
        "delta_combined_win_rate_vs_locked_stack",
        "delta_combined_total_pnl_vs_locked_stack",
        "delta_combined_avg_pnl_per_day_vs_locked_stack",
        "combined_avg_day_deterioration_vs_locked_stack",
        "delta_combined_pnl_per_day_drawdown_vs_locked_stack",
        "combined_pnl_per_day_drawdown_deterioration_vs_locked_stack",
        "delta_combined_worst_20_trade_pnl_per_day_sum_vs_locked_stack",
        "combined_worst_20_trade_pnl_per_day_deterioration_vs_locked_stack",
        "delta_combined_avg_pnl_per_day_2026_vs_locked_stack",

        "passes_frequency_improvement_3pct",
        "passes_combined_win_80pct",
        "passes_tertiary_front_win_gt_75pct",
        "passes_tertiary_positive_avg_day",
        "passes_combined_avg_day_deterioration_75",
        "passes_combined_2026_nonnegative",
        "passes_hard_tertiary_front_screen",
        "passes_combined_avg_day_deterioration_50",
        "passes_combined_2026_ge_locked_stack",
        "passes_preferred_tertiary_front_screen",
    ]:
        if metric in selected22_row:
            decision_rows.append({
                "decision_item": metric,
                "value": selected22_row[metric],
            })

decision_summary = pd.DataFrame(decision_rows)

# ======================================================================================
# 6. Validation
# ======================================================================================

validation_rows = []

# Expected active rules.
expected_core_active = {
    ("Core", 12, "Front", 0.60, 0.65, 0.65, 70.0, 8.5),
    ("Core", 15, "Front", 0.60, 0.65, 0.65, 70.0, 8.5),
    ("Core", 18, "Front", 0.60, 0.65, 0.65, 70.0, 8.5),
    ("Core", 21, "Middle", 0.65, 0.70, 0.70, 70.0, 8.5),
    ("Core", 24, "Middle", 0.65, 0.70, 0.70, 70.0, 8.5),
    ("Core", 27, "Back", 0.70, 0.70, 0.70, 70.0, 8.5),
    ("Core", 30, "Back", 0.70, 0.70, 0.70, 70.0, 8.5),
    ("Core", 33, "Back", 0.70, 0.70, 0.70, 70.0, 8.5),
}

expected_secondary_active = {
    ("Secondary", 21, "Middle", 0.60, 0.60, 0.60, 76.0, 7.0),
    ("Secondary", 24, "Middle", 0.60, 0.60, 0.60, 76.0, 7.0),
    ("Secondary", 27, "Back", 0.65, 0.00, 0.00, 77.0, 6.5),
    ("Secondary", 30, "Back", 0.65, 0.00, 0.00, 77.0, 6.5),
    ("Secondary", 33, "Back", 0.65, 0.00, 0.00, 77.0, 6.5),
}

expected_tertiary_active = {
    ("Tertiary", 12, "Front", 0.65, 0.20, 0.20, 75.0, 7.0),
    ("Tertiary", 15, "Front", 0.65, 0.20, 0.20, 75.0, 7.0),
    ("Tertiary", 18, "Front", 0.65, 0.20, 0.20, 75.0, 7.0),
}

expected_all_active = expected_core_active | expected_secondary_active | expected_tertiary_active
actual_active = active_rule_tuple_set(stack_locked_rules_long)

missing_active = sorted(expected_all_active - actual_active)
unexpected_active = sorted(actual_active - expected_all_active)

bad_core_count = len(core_locked_rules_long) != 8
bad_secondary_count = len(secondary_locked_rules_long) != 8
bad_tertiary_count = len(tertiary_locked_rules_long) != 3
bad_stack_count = len(stack_locked_rules_long) != 19

bad_tertiary_non_front = tertiary_locked_rules_long[
    ~tertiary_locked_rules_long["tenor"].isin(TERTIARY_TENORS)
    | ~tertiary_locked_rules_long["tenor_bucket"].eq("Front")
]

bad_tertiary_inactive = tertiary_locked_rules_long[
    ~tertiary_locked_rules_long["include_tenor"].eq(True)
]

bad_duplicate_layer_tenor = (
    stack_locked_rules_long
    .groupby(["signal_layer", "tenor"])
    .size()
    .reset_index(name="rows")
)

bad_duplicate_layer_tenor = bad_duplicate_layer_tenor[
    bad_duplicate_layer_tenor["rows"].gt(1)
]

bad_selection_rule = not all(
    stack_locked_rules_long["selection_rule"].eq(FINAL_SELECTION_RULE_TEXT)
)

# Cell 22R-FIX source candidate cross-check.
cell22_candidate_status = "FAIL"
cell22_candidate_detail = f"Candidate {TERTIARY_SOURCE_CANDIDATE_ID} not found."

if not selected22.empty:
    selected22_row = selected22.iloc[0]

    checks = {
        "log_065": np.isclose(as_float(selected22_row.get("tertiary_model_vrp_log_min", np.nan)), 0.65),
        "z3m_020": np.isclose(as_float(selected22_row.get("tertiary_model_vrp_z_3m_min", np.nan)), 0.20),
        "z1y_020": np.isclose(as_float(selected22_row.get("tertiary_model_vrp_z_1y_min", np.nan)), 0.20),
        "rsi_75": np.isclose(as_float(selected22_row.get("tertiary_RSI14_max", np.nan)), 75.0),
        "rv70": np.isclose(as_float(selected22_row.get("tertiary_rv21d_vol_pct_min", np.nan)), 7.0),
        "passes_hard_screen": normalize_bool_value(selected22_row.get("passes_hard_tertiary_front_screen", False)),
        "frequency_improvement_ge_3pp": normalize_bool_value(selected22_row.get("passes_frequency_improvement_3pct", False)),
        "combined_win_ge_80": normalize_bool_value(selected22_row.get("passes_combined_win_80pct", False)),
        "tertiary_win_gt_75": normalize_bool_value(selected22_row.get("passes_tertiary_front_win_gt_75pct", False)),
        "date_reconciliation_zero": np.isclose(as_float(selected22_row.get("combined_date_count_reconciliation_diff", np.nan)), 0.0),
    }

    cell22_candidate_status = "PASS" if all(checks.values()) else "FAIL"
    cell22_candidate_detail = (
        f"source_path={cell22_fix_comparison_path}; "
        f"candidate={TERTIARY_SOURCE_CANDIDATE_ID}; "
        f"checks={checks}"
    )

# Cell 22R-FIX validation cross-check.
if {"check", "status"}.issubset(cell22_fix_validation.columns):
    cell22_failures = cell22_fix_validation[cell22_fix_validation["status"].eq("FAIL")]
    cell22_validation_status = "PASS" if cell22_failures.empty else "FAIL"
    cell22_validation_detail = f"path={cell22_fix_validation_path}; failures={len(cell22_failures):,}"
else:
    cell22_validation_status = "FAIL"
    cell22_validation_detail = f"path={cell22_fix_validation_path}; validation file missing check/status columns"

# Core/Secondary unchanged cross-check versus Cell 21R source.
compare_cols = [
    "signal_layer",
    "tenor",
    "tenor_bucket",
    "include_tenor",
    "model_vrp_log_min",
    "model_vrp_z_3m_min",
    "model_vrp_z_1y_min",
    "RSI14_max",
    "rv21d_vol_pct_min",
    "lock_version",
    "lock_decision_id",
]

source_core_secondary_compare = core_secondary_stack_source[
    compare_cols
].copy().sort_values(["signal_layer", "tenor"]).reset_index(drop=True)

new_core_secondary_compare = core_secondary_stack_new[
    compare_cols
].copy().sort_values(["signal_layer", "tenor"]).reset_index(drop=True)

for c in [
    "model_vrp_log_min",
    "model_vrp_z_3m_min",
    "model_vrp_z_1y_min",
    "RSI14_max",
    "rv21d_vol_pct_min",
]:
    source_core_secondary_compare[c] = pd.to_numeric(source_core_secondary_compare[c], errors="coerce").round(10)
    new_core_secondary_compare[c] = pd.to_numeric(new_core_secondary_compare[c], errors="coerce").round(10)

core_secondary_unchanged = source_core_secondary_compare.equals(new_core_secondary_compare)

add_validation(
    validation_rows,
    "source_core_secondary_stack_loaded",
    "PASS" if len(core_secondary_stack_source) == 16 else "FAIL",
    f"rows={len(core_secondary_stack_source):,}; path={locked_core_secondary_stack_path}",
)

add_validation(
    validation_rows,
    "required_source_stack_columns_available",
    "PASS" if not missing_stack_cols else "FAIL",
    f"missing={missing_stack_cols}",
)

add_validation(
    validation_rows,
    "core_rules_row_count",
    "PASS" if not bad_core_count else "FAIL",
    f"rows={len(core_locked_rules_long):,}; expected=8",
)

add_validation(
    validation_rows,
    "secondary_rules_row_count",
    "PASS" if not bad_secondary_count else "FAIL",
    f"rows={len(secondary_locked_rules_long):,}; expected=8 including excluded Secondary Front rows",
)

add_validation(
    validation_rows,
    "tertiary_rules_row_count",
    "PASS" if not bad_tertiary_count else "FAIL",
    f"rows={len(tertiary_locked_rules_long):,}; expected=3 active Front rows",
)

add_validation(
    validation_rows,
    "stack_rules_row_count",
    "PASS" if not bad_stack_count else "FAIL",
    f"rows={len(stack_locked_rules_long):,}; expected=19",
)

add_validation(
    validation_rows,
    "expected_active_rules_match",
    "PASS" if not missing_active and not unexpected_active else "FAIL",
    f"missing_active={missing_active}; unexpected_active={unexpected_active}",
)

add_validation(
    validation_rows,
    "tertiary_front_only",
    "PASS" if bad_tertiary_non_front.empty else "FAIL",
    f"bad_rows={len(bad_tertiary_non_front):,}; expected_tenors={TERTIARY_TENORS}",
)

add_validation(
    validation_rows,
    "tertiary_front_active",
    "PASS" if bad_tertiary_inactive.empty else "FAIL",
    f"inactive_rows={len(bad_tertiary_inactive):,}",
)

add_validation(
    validation_rows,
    "no_duplicate_layer_tenor_rows",
    "PASS" if bad_duplicate_layer_tenor.empty else "FAIL",
    f"bad_rows={len(bad_duplicate_layer_tenor):,}",
)

add_validation(
    validation_rows,
    "selection_rule_populated",
    "PASS" if not bad_selection_rule else "FAIL",
    f"unique_selection_rules={stack_locked_rules_long['selection_rule'].nunique()}",
)

add_validation(
    validation_rows,
    "core_secondary_signal_rules_unchanged",
    "PASS" if core_secondary_unchanged else "FAIL",
    "Core and Secondary parameters/lock IDs match Cell 21R source stack.",
)

add_validation(
    validation_rows,
    "cell22_fix_source_candidate_cross_check",
    cell22_candidate_status,
    cell22_candidate_detail,
)

add_validation(
    validation_rows,
    "cell22_fix_validation_passed",
    cell22_validation_status,
    cell22_validation_detail,
)

add_validation(
    validation_rows,
    "no_parameter_optimization",
    "PASS",
    "Write-only cell. No optimization or sweep is run.",
)

add_validation(
    validation_rows,
    "no_core_changes",
    "PASS",
    "Core lock version and Core parameters remain unchanged.",
)

add_validation(
    validation_rows,
    "no_secondary_changes",
    "PASS",
    "Secondary lock version and Secondary parameters remain unchanged.",
)

add_validation(
    validation_rows,
    "no_sizing_changes",
    "PASS",
    "This cell writes signal parameter artifacts only. No sizing fields are created or changed.",
)

add_validation(
    validation_rows,
    "no_production_or_intraday_implementation",
    "PASS",
    "This cell writes research/audit lock artifacts only. No production or intraday logic is implemented.",
)

validation = pd.DataFrame(validation_rows)
failures = validation[validation["status"].eq("FAIL")]

# ======================================================================================
# 7. Save artifacts
# ======================================================================================

paths = {}

paths["core_locked_rules_long_csv"] = OUT_AUDIT_DIR / f"23R_core_locked_rules_long_{CELL23R_TIMESTAMP}.csv"
paths["core_locked_rules_long_parquet"] = OUT_PROCESSED_DIR / f"23R_core_locked_rules_long_{CELL23R_TIMESTAMP}.parquet"
paths["core_locked_rules_wide_csv"] = OUT_AUDIT_DIR / f"23R_core_locked_rules_wide_{CELL23R_TIMESTAMP}.csv"
paths["core_locked_rules_wide_parquet"] = OUT_PROCESSED_DIR / f"23R_core_locked_rules_wide_{CELL23R_TIMESTAMP}.parquet"

paths["secondary_locked_rules_long_csv"] = OUT_AUDIT_DIR / f"23R_secondary_locked_rules_long_{CELL23R_TIMESTAMP}.csv"
paths["secondary_locked_rules_long_parquet"] = OUT_PROCESSED_DIR / f"23R_secondary_locked_rules_long_{CELL23R_TIMESTAMP}.parquet"
paths["secondary_locked_rules_wide_csv"] = OUT_AUDIT_DIR / f"23R_secondary_locked_rules_wide_{CELL23R_TIMESTAMP}.csv"
paths["secondary_locked_rules_wide_parquet"] = OUT_PROCESSED_DIR / f"23R_secondary_locked_rules_wide_{CELL23R_TIMESTAMP}.parquet"

paths["tertiary_locked_rules_long_csv"] = OUT_AUDIT_DIR / f"23R_tertiary_front_cleaner_quality_locked_rules_long_{CELL23R_TIMESTAMP}.csv"
paths["tertiary_locked_rules_long_parquet"] = OUT_PROCESSED_DIR / f"23R_tertiary_front_cleaner_quality_locked_rules_long_{CELL23R_TIMESTAMP}.parquet"
paths["tertiary_locked_rules_wide_csv"] = OUT_AUDIT_DIR / f"23R_tertiary_front_cleaner_quality_locked_rules_wide_{CELL23R_TIMESTAMP}.csv"
paths["tertiary_locked_rules_wide_parquet"] = OUT_PROCESSED_DIR / f"23R_tertiary_front_cleaner_quality_locked_rules_wide_{CELL23R_TIMESTAMP}.parquet"

paths["stack_locked_rules_long_csv"] = OUT_AUDIT_DIR / f"23R_core_secondary_tertiary_locked_rules_long_{CELL23R_TIMESTAMP}.csv"
paths["stack_locked_rules_long_parquet"] = OUT_PROCESSED_DIR / f"23R_core_secondary_tertiary_locked_rules_long_{CELL23R_TIMESTAMP}.parquet"
paths["stack_locked_rules_wide_csv"] = OUT_AUDIT_DIR / f"23R_core_secondary_tertiary_locked_rules_wide_{CELL23R_TIMESTAMP}.csv"
paths["stack_locked_rules_wide_parquet"] = OUT_PROCESSED_DIR / f"23R_core_secondary_tertiary_locked_rules_wide_{CELL23R_TIMESTAMP}.parquet"

paths["decision_summary_csv"] = OUT_AUDIT_DIR / f"23R_core_secondary_tertiary_lock_decision_summary_{CELL23R_TIMESTAMP}.csv"

paths["validation_csv"] = OUT_AUDIT_DIR / f"23R_core_secondary_tertiary_lock_validation_{CELL23R_TIMESTAMP}.csv"

core_locked_rules_long.to_csv(paths["core_locked_rules_long_csv"], index=False)
core_locked_rules_long.to_parquet(paths["core_locked_rules_long_parquet"], index=False)

core_locked_rules_wide.to_csv(paths["core_locked_rules_wide_csv"], index=False)
core_locked_rules_wide.to_parquet(paths["core_locked_rules_wide_parquet"], index=False)

secondary_locked_rules_long.to_csv(paths["secondary_locked_rules_long_csv"], index=False)
secondary_locked_rules_long.to_parquet(paths["secondary_locked_rules_long_parquet"], index=False)

secondary_locked_rules_wide.to_csv(paths["secondary_locked_rules_wide_csv"], index=False)
secondary_locked_rules_wide.to_parquet(paths["secondary_locked_rules_wide_parquet"], index=False)

tertiary_locked_rules_long.to_csv(paths["tertiary_locked_rules_long_csv"], index=False)
tertiary_locked_rules_long.to_parquet(paths["tertiary_locked_rules_long_parquet"], index=False)

tertiary_locked_rules_wide.to_csv(paths["tertiary_locked_rules_wide_csv"], index=False)
tertiary_locked_rules_wide.to_parquet(paths["tertiary_locked_rules_wide_parquet"], index=False)

stack_locked_rules_long.to_csv(paths["stack_locked_rules_long_csv"], index=False)
stack_locked_rules_long.to_parquet(paths["stack_locked_rules_long_parquet"], index=False)

stack_locked_rules_wide.to_csv(paths["stack_locked_rules_wide_csv"], index=False)
stack_locked_rules_wide.to_parquet(paths["stack_locked_rules_wide_parquet"], index=False)

decision_summary.to_csv(paths["decision_summary_csv"], index=False)

validation.to_csv(paths["validation_csv"], index=False)

manifest = {
    "cell": "Cell 23R — Write Tertiary Front cleaner-quality lock artifacts",
    "timestamp": CELL23R_TIMESTAMP,

    "model_spec": LOCKED_MODEL_SPEC,

    "stack_lock_version": STACK_LOCK_VERSION,
    "stack_lock_decision_id": STACK_LOCK_DECISION_ID,

    "source_core_secondary_stack_version": SOURCE_STACK_VERSION,
    "source_core_secondary_stack_decision_id": SOURCE_STACK_DECISION_ID,

    "core_lock_version": CORE_LOCK_VERSION,
    "core_lock_decision_id": CORE_LOCK_DECISION_ID,

    "secondary_lock_version": SECONDARY_LOCK_VERSION,
    "secondary_lock_decision_id": SECONDARY_LOCK_DECISION_ID,

    "tertiary_lock_version": TERTIARY_LOCK_VERSION,
    "tertiary_lock_decision_id": TERTIARY_LOCK_DECISION_ID,

    "tertiary_source_cell": TERTIARY_SOURCE_CELL,
    "tertiary_source_candidate_id": TERTIARY_SOURCE_CANDIDATE_ID,

    "source_paths": {
        "locked_core_secondary_stack_path": str(locked_core_secondary_stack_path),
        "cell22_fix_comparison_path": str(cell22_fix_comparison_path),
        "cell22_fix_validation_path": str(cell22_fix_validation_path),
        "cell22_fix_manifest_path": str(cell22_fix_manifest_path) if cell22_fix_manifest_path else None,
        "cell22_fix_suggested_tertiary_panel_path": str(cell22_fix_suggested_tertiary_panel_path) if cell22_fix_suggested_tertiary_panel_path else None,
        "cell22_fix_suggested_combined_panel_path": str(cell22_fix_suggested_combined_panel_path) if cell22_fix_suggested_combined_panel_path else None,
    },

    "locked_core": {
        "lock_version": CORE_LOCK_VERSION,
        "lock_decision_id": CORE_LOCK_DECISION_ID,
        "status": "unchanged from Cell 21R / Cell 11R",
    },

    "locked_secondary": {
        "lock_version": SECONDARY_LOCK_VERSION,
        "lock_decision_id": SECONDARY_LOCK_DECISION_ID,
        "status": "unchanged from Cell 21R",
    },

    "locked_tertiary": {
        "lock_version": TERTIARY_LOCK_VERSION,
        "lock_decision_id": TERTIARY_LOCK_DECISION_ID,
        "source_candidate_id": TERTIARY_SOURCE_CANDIDATE_ID,
        "layer": "Tertiary Front",
        "tenors": TERTIARY_TENORS,
        "model_vrp_log_min": TERTIARY_MODEL_VRP_LOG_MIN,
        "model_vrp_z_3m_min": TERTIARY_MODEL_VRP_Z_3M_MIN,
        "model_vrp_z_1y_min": TERTIARY_MODEL_VRP_Z_1Y_MIN,
        "RSI14_max": TERTIARY_RSI14_MAX,
        "rv21d_vol_pct_min": TERTIARY_RV21D_VOL_PCT_MIN,
        "selection_mode": "BEST_SIGNAL_RANK",
        "gate": "Only evaluated if no locked Core and no locked Secondary trade qualifies.",
    },

    "final_selection_rule": FINAL_SELECTION_RULE_TEXT,

    "cell22_fix_source_candidate_metrics": (
        selected22.iloc[0].to_dict()
        if not selected22.empty
        else {}
    ),

    "row_counts": {
        "core_rules_long": int(len(core_locked_rules_long)),
        "secondary_rules_long": int(len(secondary_locked_rules_long)),
        "tertiary_rules_long": int(len(tertiary_locked_rules_long)),
        "stack_rules_long": int(len(stack_locked_rules_long)),
    },

    "constraints": [
        "No Core parameter changes.",
        "No Secondary parameter changes.",
        "No parameter optimization or sweep.",
        "No sizing changes.",
        "No production/intraday implementation.",
        "This cell writes lock artifacts only.",
    ],

    "paths": {k: str(v) for k, v in paths.items()},
}

paths["manifest_json"] = OUT_AUDIT_DIR / f"23R_core_secondary_tertiary_lock_manifest_{CELL23R_TIMESTAMP}.json"

with open(paths["manifest_json"], "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

paths["manifest_json"] = paths["manifest_json"]

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 23R validation failed. Review validation output above.")

# ======================================================================================
# 8. Display
# ======================================================================================

print("=" * 100)
print("Cell 23R — Core + Secondary + Tertiary Front lock artifacts written")
print("=" * 100)
print(f"Model spec:                         {LOCKED_MODEL_SPEC}")
print(f"Stack lock version:                 {STACK_LOCK_VERSION}")
print(f"Stack lock decision ID:             {STACK_LOCK_DECISION_ID}")
print(f"Core lock decision ID:              {CORE_LOCK_DECISION_ID}")
print(f"Secondary lock decision ID:         {SECONDARY_LOCK_DECISION_ID}")
print(f"Tertiary lock decision ID:          {TERTIARY_LOCK_DECISION_ID}")
print(f"Tertiary source candidate:          {TERTIARY_SOURCE_CANDIDATE_ID}")
print()
print("Final selection rule:")
print(FINAL_SELECTION_RULE_TEXT)
print()
print("Validation:")
display(validation)

print()
print("Locked Core rules — wide:")
display(core_locked_rules_wide)

print()
print("Locked Secondary rules — wide:")
display(secondary_locked_rules_wide)

print()
print("Locked Tertiary Front rules — wide:")
display(tertiary_locked_rules_wide)

print()
print("Combined locked Core + Secondary + Tertiary stack rules — long:")
display(stack_locked_rules_long)

print()
print("Decision summary from Cell 22R-FIX source candidate:")
display(decision_summary)

print()
print("Saved outputs:")
for k, p in paths.items():
    print(f"  {k}: {p}")

print("\nCELL 23R PASSED — CORE + SECONDARY + TERTIARY FRONT SIGNAL PARAMETERS LOCKED INTO ARTIFACTS")